# NBA Spread Prediction — Unified Colab Pipeline

Self-contained, **leak-free** spread model (player ELO + hierarchical possessions).
Runs end-to-end in Google Colab against your `basketballData` Drive folder.

Each phase **saves artifacts to disk** and can **reload them** on the next session
(set `USE_SAVED_ARTIFACTS = False` to always recompute, or `FORCE_RECOMPUTE["<phase>"] = True` for one phase).

---

## Table of contents

| Step | Section | Saved artifacts |
|------|---------|-----------------|
| — | [Configuration & Drive](#configuration--google-drive-mount) | — |
| — | [Checkpoint helpers](#checkpoint-helpers-save--load) | `checkpoints/` manifest |
| — | [Pipeline modules](#pipeline-source-inlined-modules--complete) | — |
| 0 | [Runtime toggles](#phase-0--runtime-toggles) | `checkpoints/runtime_toggles.json` |
| 1 | [Load data](#phase-1--load-data) | stints cache, `odds_dict.pkl`, `load_summary.csv` |
| 2 | [Walk-forward backtest](#phase-2--walk-forward-backtest) | `backtest_results.csv`, `state/tuning_results.json` |
| 2a | [Unified confidence diagnostics](#phase-2a--unified-confidence-diagnostics) | Inline tuning in backtest; plots in `analysis_plots/14–16_*.png` |
| 2b | [Spread diagnostics + ablation](#phase-2b--spread-diagnostics--ablation) | `diag_summary.csv`, `diag_edges.pkl`, `ablation_posthoc.json` |
| 2c | [Bankroll & confidence plots](#phase-2c--bankroll--confidence-plots) | `analysis_plots/*.png` |
| 2d | [Moneyline diagnostics](#phase-2d--moneyline-diagnostics-metawin-calibration) | `ml_diagnostics_summary.csv`, `13_ml_winprob_reliability.png` |
| 2e | [ML value tracker](#phase-2e--ml-underdog--favorite-value-tracker) | `ml_value_plays.csv` |
| 3 | [ML + Total + Score-pair](#phase-3--dedicated-ml--total--score-pair-models) | `state/win.pkl`, `state/total.pkl`, `state/score_pair.pkl`, `checkpoints/final_feats.parquet` |
| 4 | [Spread / final engines](#phase-4--spread--final-engines) | `state/meta.pkl`, `state/*.pkl` |
| 5 | [Daily prediction](#phase-5--daily-prediction) | `checkpoints/last_prediction.json` |

---

## Key improvements (2026 pipeline)
- **Specialized markets** — separate WIN vs TOTAL feature matrices; Elo de-emphasized on totals
- **Score-pair model** — independent home/away score heads → total + margin + σ
- **Gaussian O/U** — `P(Over|line)` from `N(pred_total, σ²)` with CRPS in diagnostics
- **Both-sides ML EV** — predict winner, price home *and* away, bet only if +EV else Pass
- **Multi-window form** — 3/5/10/20/season rolling + nonlinear rest buckets
- **confidence_only spread selection** — all edges visible; calibrated confidence gates bets
- **ATS classifier blend** — always mixed into cover prob before confidence calibration
- **Actionable gates** — `MIN_DISAGREEMENT_TRUST`, phantom injury, quantile width, tight spread, edge dead-zone band
- **Edge-scaled confidence** — small edges need higher confidence (`CONFIDENCE_EDGE_SCALED`)
- **Season-1 calibrator** — fit on training calib split before first simulated season
- **Total-head sanity cap** — skip unstable total refits when CV MAE > `TOTAL_HEAD_CV_SANITY_CAP`
- **Phase 2a** unified confidence tuning (prior-year only, gated ROI objective)
- **Selection comparison** — `compare_selection_strategies()` after backtest (hybrid, edge-scaled, edge-only)
- **ML calibration** (`ml_calibration`) + `MIN_ML_WIN_PCT` gate separate from spread
- **O/U** Gaussian prob gate + edge (`OU_MIN_EDGE`, `OU_MIN_PROB`)
- **Artifact schema validation** — stale `backtest_results.csv` detected on reload
- All modules inlined below — **no external repo required** on Colab


In [ ]:
# Install dependencies (Colab). Safe to re-run.
!pip install -q catboost optuna scikit-learn scipy pandas numpy tqdm matplotlib pyarrow lightgbm


## Configuration & Google Drive mount


In [ ]:
# ============================================================================
#  CONFIG  ·  Google Drive mount + paths + constants  (no data leaks)
# ============================================================================
import math, re, copy, gc, pickle, json, warnings
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/basketballData")
else:
    ROOT = Path("./basketballData")
ROOT.mkdir(parents=True, exist_ok=True)

V3_DATA_PATHS = {
    2021: ROOT / "events_2021_22_pbp_V3.csv",
    2022: ROOT / "events_2022_23_pbp_V3.csv",
    2023: ROOT / "events_2023_24_pbp_V3.csv",
    2024: ROOT / "events_2024_25_pbp_V3.csv",
}
PBP_2026_PATH    = ROOT / "[10-21-2025]-[05-18-2026]-combined-stats.csv"
MODERN_ODDS_PATH = ROOT / "all_odds.csv"
PINNACLE_LINES_PATH = ROOT / "nba_main_lines.csv"
PLAYER_LIST_PATH = ROOT / "nba_players_all.csv"
SPREAD_CSV_PATH = ROOT / "nba_betting_spread.csv"
ML_CSV_PATH = ROOT / "nba_betting_money_line.csv"
BASE_ELO = 1500.0
DEFAULT_LEAGUE_RTG = 110.0
DEFAULT_LEAGUE_XPPP = 1.10
LEAGUE_AVG_TOTAL = 225.0
OFFSEASON_REVERSION = 0.15
ASSIST_SPLIT = 0.76
USAGE_FLOOR = 0.27
HOME_PPP_BOOST = 0.002
K_MULT_HALF_LIFE = 24.4
GARBAGE_TIME_WEIGHT = 0.30
ALTITUDE_TEAMS = {"DEN", "UTA"}
GOOD_BET_EDGE = 3.5

SOS_WINDOW = 15
OPTIMAL_BET_EDGE = 2.5
SPREAD_CALIB_WINDOW = 50

# Tuning objective / CV constants
TUNING_XPPP_WEIGHT = 0.7
TUNING_POINTS_WEIGHT = 0.3
TUNING_EMBARGO_GAMES = 15
TUNING_INVALID_SCORE = 9999.0
TUNING_BOUND_EDGE_FRACTION = 0.02

META_LOSS_MAE_WEIGHT = 0.35
META_LOSS_BRIER_WEIGHT = 0.25
META_LOSS_ATS_ROI_WEIGHT = 0.50
META_LOSS_CLV_ROI_WEIGHT = 0.15

# ELO calibration / meta-anchor defaults (overridden by walk-forward tuning).
ELO_BLEND_ALPHA = 0.46
ELO_RIDGE_ALPHA = 4.66
ELO_CALIB_MIN_SAMPLES = 80
ELO_CALIB_HUBER_EPS = 1.35

# Context-aware Elo update defaults (tunable via Optuna).
CLUTCH_BOOST = 1.30
TOV_PENALTY = 0.85
FOUL_DRAW_BOOST = 1.10
VARIANCE_DAMPEN = 0.90
XPPP_ACTUAL_BLEND = 0.05
K_DEF_EVENTS = 0.15
TOV_RATE_THRESHOLD = 0.15
THREE_PA_RATE_THRESHOLD = 0.45

# Betting / meta-model Elo prominence.
ELO_UNCERTAINTY_BLEND_BOOST = 0.15
ELO_AGREEMENT_EXTRA_EDGE = 1.0
TOTAL_ELO_BETA = 0.0  # de-emphasize Elo on totals unless ablation shows lift
ELO_WIN_BLEND = 0.35

TEAM_MAP = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Clippers": "LAC", "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM", "Miami Heat": "MIA", "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN", "New Orleans Pelicans": "NOP", "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC", "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHX", "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS", "Toronto Raptors": "TOR", "Utah Jazz": "UTA",
    "Washington Wizards": "WAS",
}

names_dict: dict = {}
name_to_id: dict = {}

if PLAYER_LIST_PATH.exists():
    _names_df = pd.read_csv(PLAYER_LIST_PATH, low_memory=False)
    pid_col = "person_id" if "person_id" in _names_df.columns else "PERSON_ID"
    name_col = "display_first_last" if "display_first_last" in _names_df.columns else "DISPLAY_FIRST_LAST"
    names_dict = _names_df.set_index(pid_col)[name_col].to_dict()
    name_to_id = {v: k for k, v in names_dict.items()}

STATE_DIR = ROOT / "state"
STATE_DIR.mkdir(parents=True, exist_ok=True)

# --- Artifacts / backtest CSV schema ---
ARTIFACT_SCHEMA_VERSION = 2

# --- Task 004/017 versioned schema components ---
# Bump whichever of these changes whenever the corresponding logic changes so
# every downstream cache (stints, tuning, backtest CSVs, model artifacts) is
# automatically invalidated instead of silently reused with stale semantics.
# Bumped for Tasks 006-013 (canonical games, mixed-date parsing, garbage-time
# point preservation, zero-possession stint preservation, GAME_ID stint
# boundaries).
PREPROCESSING_SCHEMA_VERSION = 2
# Bumped for Tasks 031-032 (T-60 lineup features no longer fall back to the
# current game's own actual/first-PBP lineup; EPM prior features now require
# a versioned observation_date and expose missing flags instead of a silent
# zero). Any cache/artifact built under FEATURE_SCHEMA_VERSION < 2 encodes
# the removed leak and must not be reused.
FEATURE_SCHEMA_VERSION = 2
# Bumped for Tasks 018-029 (quote-level market_snapshots.py schema,
# open/decision(T-60)/close selection, two-sided ML/spread/total prices,
# canonical-identity game matching, devig.py, point/price CLV separation).
MARKET_SNAPSHOT_SCHEMA_VERSION = 2
VALIDATION_SCHEMA_VERSION = 1
# Freeze definition: predictions may use only data with source/ingestion
# timestamps at or before (scheduled tip - this many minutes).
DECISION_CUTOFF_MINUTES_BEFORE_TIP = 60
REQUIRED_BACKTEST_COLUMNS = (
    "EDGE",
    "EDGE_LEAN",
    "WIN_PCT",
    "ACTIONABLE",
    "DIRECTION",
    "CONFIDENCE",
    "MIN_CONFIDENCE_SCORE",
)

# --- Bet selection & staking ---
# confidence_only: |edge| >= CONFIDENCE_MIN_EDGE, then calibrated win% gates actionable bets.
BET_SELECTION_MODE = "confidence_only"  # legacy_tiers | edge_bucket | edge_bucket_ats | confidence_only
# Minimum |model-market| edge (pts) before a game is scored or bet in confidence modes.
CONFIDENCE_MIN_EDGE = 3.0
# When True, require higher confidence for small edges (see edge_scaled_min_confidence).
CONFIDENCE_EDGE_SCALED = True
MIN_EDGE_BUCKET = 5.5
EDGE_STAKE_TIERS = ((5.5, 8.0, 1.0), (8.0, 999.0, 1.5))
# Median model conf width is ~26pts; 22 was over-abstaining (~3% bet rate / 0 bets
# on early single-season walks). 28 keeps a real uncertainty filter without silence.
MAX_QUANTILE_WIDTH = 28.0
# Actionable spread gates (confidence_only and live predict).
MIN_DISAGREEMENT_TRUST = 0.85
SKIP_PHANTOM_INJURY = True
SKIP_TIGHT_SPREAD = True
# Optional dead-zone filter: (lo, hi) absolute edge pts; None disables.
# Was (7.0, 9.5) — that band had the strongest lean ATS in diagnostics; leave off.
EDGE_AVOID_BAND = None
EDGE_AVOID_BAND_MIN_ELO_AGREE = 0.5
USE_TIER_STAKE_GATES = False
WALKFORWARD_EDGE_MIN_FLOOR = 5.5  # ignored when BET_SELECTION_MODE == confidence_only

# --- Calibration ---
CALIBRATION_MODE = "legacy_stack"  # legacy_stack | simplified | beta_ats
USE_VENN_ABERS_FILTER = False
VENN_ABERS_MAX_WIDTH = 0.15
USE_SKELLAM_COVER_PROB = False
ATS_CLASSIFIER_BLEND = 0.5
ATS_CLASSIFIER_MIN_PROB = 0.524

# --- ML win-probability path ---
WIN_PROB_SOURCE = "auto"  # auto | meta_win | rolling_platt | margin_isotonic
REQUIRE_META_WIN_WHEN_FITTED = True
ML_CALIB_METHOD = "isotonic"  # platt | isotonic | none — prefer isotonic for betting calib
ML_CALIB_SCOPE = "prior_year"
# Minimum calibrated win% (0-100) to flag an ML bet (separate from spread CONFIDENCE gate).
MIN_ML_WIN_PCT = 55
# Extra EV required when betting the underdog side (decimal >= 2.0). None = use min_ev only.
ML_UNDERDOG_MIN_EV = None
# Shrink model win prob toward market implied before ML EV (0=off). Applied per bet side.
ML_MARKET_SHRINK = 0.20
# Additional shrink when the bet side is an underdog (decimal >= 2.0).
ML_UNDERDOG_EXTRA_SHRINK = 0.10
# False = pick best calibrated-EV side; True = only bet predicted winner.
ML_BET_PREDICTED_WINNER_ONLY = False
# Default ML gate: minimum expected ROI (EV) to place any moneyline bet. Pass otherwise.
ML_MIN_EV = 0.08
# |WIN_PROB - 0.5| within this band = coin flip (tight toss-up zone).
ML_COIN_FLIP_BAND = 0.03
# When True, coin-flip games may bet underdog only; False = Pass on coin flips.
ML_COIN_FLIP_UNDERDOG_ONLY = False
# In coin-flip games, underdog bets need this higher EV bar (only exceptional value).
ML_COIN_FLIP_UNDERDOG_MIN_EV = 0.10
# Default cap for walk-forward favorite decimal search (allows shorter favorites than 1.45).
ML_MAX_FAVORITE_DECIMAL_DEFAULT = 1.60

# Phase 2 backtest: logistic on raw features → isotonic for calibrated P(cover).
# Phase 2a tunes composite weights; inline backtest tuning refines each season.
CONFIDENCE_MODE = "unified"  # default | unified
CONFIDENCE_CALIB_METHOD = "isotonic"  # isotonic | platt | logistic_features | logistic_isotonic | none
CONFIDENCE_CALIB_SCOPE = "prior_year"  # prior_year (last season only) | all_prior
TUNE_CONFIDENCE_WEIGHTS_IN_BACKTEST = True  # walk-forward weight / logistic-C search each season
MIN_CONFIDENCE_SCORE = 55
# No hard upper cap by default; allows high-confidence tail when ranking is calibrated.
MAX_CONFIDENCE_SCORE = None
CONFIDENCE_SELECTION_MODE = "min_score"  # min_score | off — only gate for actionable spread bets
CONFIDENCE_STAKE_MODE = True  # scale stake by confidence_score / 100
# Year-over-year search shrink for Phase 2a per-variable weight tuning (0–1).
CONFIDENCE_WEIGHT_SHRINK = 0.55
CONFIDENCE_WEIGHT_SEARCH_SAMPLES = 120

# --- Stake sizing ---
STAKE_SIZING_MODE = "edge_scaled"  # legacy_kelly | edge_scaled | volatility_adjusted
KELLY_FRACTION_CAP = 0.25
SLATE_CORRELATION_PENALTY = 0.15
SLATE_CORRELATION_MIN_BETS = 5

# --- Feature flags ---
USE_GARBAGE_WEIGHTED_FORM = False
USE_DECAYED_SOS = False
SOS_DECAY_LAMBDA = 0.05

# --- Pythagorean expectation (rolling, pre-game only) ---
PYTHAGOREAN_EXPONENT = 13.91
PYTHAG_MIN_GAMES = 5

# --- ML market de-vig (shrink target only; does not change posted odds for EV) ---
ML_MARKET_DEVIG = True

# --- Optional CLV gate for actionable spread bets (pre-game closing line) ---
REQUIRE_NONNEGATIVE_CLV = False
MIN_CLV_POINTS = 0.0
TIGHT_SPREAD_MAX = 3.5  # diagnostics segment |market spread| <= this
# Reject total-head tuning when CV MAE exceeds this (reuse margin-only fit).
TOTAL_HEAD_CV_SANITY_CAP = 35.0

# --- Totals (O/U) ---
# Gates: |PRED_TOTAL - MARKET_TOTAL| >= OU_MIN_EDGE and optional Gaussian P(side).
OU_BET_ENABLED = True
OU_MIN_EDGE = 4.5
OU_REQUIRE_POSITIVE_CI = True
# Use N(pred_total, sigma^2) for P(Over)/P(Under) betting gates.
OU_USE_GAUSSIAN_PROB = True
# Minimum P(chosen side) to fire an O/U bet (with edge).
OU_MIN_PROB = 0.55
# Fallback sigma (points) when walk-forward RMSE is unavailable.
OU_DEFAULT_SIGMA = 12.0
# Prefer score-pair derived total when available and validated.
USE_SCORE_PAIR_TOTAL = True

# --- Model options ---
USE_LIGHTGBM_BASE = False
APPLY_SPREAD_CALIB_IN_CV = True
META_LOSS_ECE_WEIGHT = 0.10

# --- Dedicated ML / Total model training (Phase 3) ---
ML_USE_SPREAD_FEATURES = True
TOTAL_TRAIN_TARGET = "market_residual"  # market_residual | absolute
META_WIN_TRIALS_DEFAULT = 25
META_WIN_LOSS_BRIER_WEIGHT = 0.40
META_WIN_LOSS_ECE_WEIGHT = 0.20
META_WIN_LOSS_ML_ROI_WEIGHT = 0.40
TOTAL_LOSS_MAE_WEIGHT = 0.55
TOTAL_LOSS_OU_ROI_WEIGHT = 0.45
OU_CALIBRATOR_MIN_PROB = 0.524


## Checkpoint helpers (save / load)

Set `USE_SAVED_ARTIFACTS = False` in Phase 0 to force full retrain. Per-phase `FORCE_RECOMPUTE` still overrides individual steps.


In [ ]:
# ============================================================================
#  CHECKPOINT HELPERS · save / load phase artifacts under ROOT/checkpoints/
# ============================================================================
CHECKPOINT_DIR = ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
(ROOT / "analysis_plots").mkdir(parents=True, exist_ok=True)

# Master switch: True = load saved artifacts when on disk; False = always run full compute.
# If True but a file is missing, that phase still runs end-to-end automatically.
USE_SAVED_ARTIFACTS = False

# Per-phase override: True forces recompute for that phase even when USE_SAVED_ARTIFACTS=True.
FORCE_RECOMPUTE = {
    "0": False,
    "1": False,
    "2": False,
    "2a": False,
    "2b": False,
    "2c": False,
    "2d": False,
    "2e": False,
    "3": False,
    "4": False,
    "5": False,
}

ARTIFACTS = {
    "toggles": CHECKPOINT_DIR / "runtime_toggles.json",
    "odds": CHECKPOINT_DIR / "odds_dict.pkl",
    "load_summary": CHECKPOINT_DIR / "load_summary.csv",
    "phase1_manifest": CHECKPOINT_DIR / "phase1_manifest.json",
    "backtest": ROOT / "backtest_results.csv",
    "tuning": STATE_DIR / "tuning_results.json",
    "diag_summary": CHECKPOINT_DIR / "diag_summary.csv",
    "diag_edges": CHECKPOINT_DIR / "diag_edges.pkl",
    "ablation": STATE_DIR / "ablation_posthoc.json",
    "ml_summary": ROOT / "analysis_plots" / "ml_diagnostics_summary.csv",
    "ml_plays": ROOT / "ml_value_plays.csv",
    "confidence_weights": STATE_DIR / "confidence_weights_by_season.json",
    "final_feats": CHECKPOINT_DIR / "final_feats.parquet",
    "meta_win_tuning": CHECKPOINT_DIR / "meta_win_tuning.json",
    "meta_total_tuning": CHECKPOINT_DIR / "meta_total_tuning.json",
    "prediction": CHECKPOINT_DIR / "last_prediction.json",
}


def _artifact_ready(*keys: str) -> bool:
  return all(ARTIFACTS[k].exists() for k in keys)


def use_saved_artifacts(phase: str) -> bool:
    """Return True when this phase may load from disk instead of recomputing."""
    if not USE_SAVED_ARTIFACTS:
        return False
    return not FORCE_RECOMPUTE.get(phase, False)


def save_json_artifact(key: str, payload: dict) -> Path:
    path = ARTIFACTS[key]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str))
    print(f"  💾 saved → {path}")
    return path


def load_json_artifact(key: str) -> dict:
    return json.loads(ARTIFACTS[key].read_text())


def save_pickle_artifact(key: str, obj) -> Path:
    path = ARTIFACTS[key]
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  💾 saved → {path}")
    return path


def load_pickle_artifact(key: str):
    with open(ARTIFACTS[key], "rb") as f:
        return pickle.load(f)


def load_results_csv() -> pd.DataFrame:
    path = ARTIFACTS["backtest"]
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path, low_memory=False)
    if "DATE" in df.columns:
        df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    return df


def ensure_results() -> pd.DataFrame:
    """Load backtest CSV if `results` is missing or empty (safe between phase cells)."""
    global results
    try:
        empty = results is None or results.empty
    except NameError:
        empty = True
    if empty and use_saved_artifacts("2") and ARTIFACTS["backtest"].exists():
        results = load_results_csv()
        if not results.empty and "STAKE_MODERATE" not in results.columns:
            results = add_all_profile_columns(results)
        reasons = validate_backtest_df(results)
        if reasons:
            print("⚠️  Saved backtest may be stale:")
            for r in reasons:
                print(f"   · {r}")
            if is_backtest_stale(results):
                print("   → Set USE_SAVED_ARTIFACTS=False and re-run Phase 2")
    return results


print("Checkpoint dir:", CHECKPOINT_DIR)
print(f"USE_SAVED_ARTIFACTS={USE_SAVED_ARTIFACTS}  (False = always recompute)")
print("Force recompute:", {k: v for k, v in FORCE_RECOMPUTE.items() if v} or "(none)")


## Pipeline source (inlined modules — complete)


In [ ]:
# ── module: utils ────────────────────────────────────────────────────────────
"""Shared helpers."""
import math

import numpy as np
import pandas as pd



def _coerce_int(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None
    try:
        return int(float(str(x).strip()))
    except Exception:
        return None


def _parse_player_string(s):
    """Parse lineup string entries (numeric IDs or player names)."""
    if pd.isna(s) or str(s).strip() == "" or str(s).lower() == "nan":
        return []

    s_str = str(s).strip()

    if any(c.isalpha() for c in s_str) or "," in s_str:
        tokens = [t.strip() for t in s_str.replace("-", ",").split(",") if t.strip()]
        processed_ids = []
        for token in tokens:
            if token.lower() == "nan":
                continue
            if token in name_to_id:
                processed_ids.append(int(name_to_id[token]))
            else:
                processed_ids.append(abs(hash(token)) % 1000000)
        return processed_ids

    clean_str = s_str.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    for d in ["-", ",", " "]:
        if d in clean_str:
            return [int(float(t.strip())) for t in clean_str.split(d) if t.strip() and t.lower() != "nan"]
    try:
        return [int(float(clean_str))]
    except ValueError:
        return []


def _map_player_name(raw, mapping=None):
    """Resolve 'id/Name' or 'Full Name' to integer player-id."""
    mapping = mapping if mapping is not None else name_to_id
    if pd.isna(raw) or not str(raw).strip():
        return None
    s = str(raw).strip()
    if "/" in s:
        try:
            return int(s.split("/")[0])
        except ValueError:
            s = s.split("/")[-1]
    pid = mapping.get(s)
    return int(pid) if pid is not None else (hash(s) % 10**9)


def map_elo_params(optuna_params: dict) -> dict:
    """Map Optuna Elo tuner keys to PlayerRatingTracker config keys."""
    return {
        "K_OFF": optuna_params["k_off"],
        "K_DEF": optuna_params["k_def"],
        "ELO_SCALING_FACTOR": optuna_params["elo_scaling"],
        "HOME_PPP_BOOST": optuna_params["home_boost"],
        "OFFSEASON_REVERSION": optuna_params["offseason_reversion"],
        "USAGE_FLOOR": optuna_params["usage_floor"],
        "assist_split": optuna_params["assist_split"],
        "k_mult_half_life": optuna_params.get("k_mult_half_life", 15.0),
        "rd_floor": optuna_params.get("rd_floor", 30.0),
        "garbage_time_weight": optuna_params.get("garbage_time_weight", 0.5),
        "clutch_boost": optuna_params.get("clutch_boost", 1.3),
        "tov_penalty": optuna_params.get("tov_penalty", 0.85),
        "foul_draw_boost": optuna_params.get("foul_draw_boost", 1.1),
        "variance_dampen": optuna_params.get("variance_dampen", 0.9),
        "xppp_actual_blend": optuna_params.get("xppp_actual_blend", 0.05),
        "k_def_events": optuna_params.get("k_def_events", 0.15),
        "tov_rate_threshold": optuna_params.get("tov_rate_threshold", 0.15),
        "three_pa_rate_threshold": optuna_params.get("three_pa_rate_threshold", 0.45),
    }


In [ ]:
# ── module: shot_zones ───────────────────────────────────────────────────────
"""Shot zone taxonomy, coordinate normalization, and walk-forward calibration."""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
import pandas as pd

COURT_LENGTH_FT = 94.0
COURT_WIDTH_FT = 50.0
CORNER3_Y_MAX_FT = 14.0
CORNER3_DIST_MIN_FT = 22.0
SHRINK_K = 100.0

CANONICAL_ZONES = (
    "restricted", "paint", "short_mid", "midrange", "corner3", "abovebreak3", "other",
)

LEGACY_ZONE_ALIAS = {
    "ab3": "abovebreak3",
    "rim": "restricted",
    "long_mid": "midrange",
    "three": "abovebreak3",
    "mid2": "midrange",
}

DEFAULT_FG_PCT = {
    "restricted": 0.64,
    "paint": 0.50,
    "short_mid": 0.42,
    "midrange": 0.40,
    "corner3": 0.38,
    "abovebreak3": 0.35,
    "other": 0.40,
}

DEFAULT_POINT_VALUE = {
    "restricted": 2.0,
    "paint": 2.0,
    "short_mid": 2.0,
    "midrange": 2.0,
    "corner3": 3.0,
    "abovebreak3": 3.0,
    "other": 2.0,
}


def legacy_zone_alias(zone: str) -> str:
    if zone is None or (isinstance(zone, float) and pd.isna(zone)):
        return "other"
    z = str(zone).strip().lower()
    if z in CANONICAL_ZONES:
        return z
    if z in LEGACY_ZONE_ALIAS:
        return LEGACY_ZONE_ALIAS[z]
    if "corner" in z and "3" in z:
        return "corner3"
    if "restricted" in z or z == "rim":
        return "restricted"
    if "paint" in z:
        return "paint"
    if "mid" in z and "3" not in z:
        return "midrange" if "long" in z or "mid2" in z else "short_mid"
    if "3" in z or "three" in z:
        return "abovebreak3"
    return "other"


@dataclass
class ZoneCalibration:
    """Walk-forward zone make rates × shot value."""

    fg_pct: dict[str, float] = field(default_factory=lambda: dict(DEFAULT_FG_PCT))
    point_value: dict[str, float] = field(default_factory=lambda: dict(DEFAULT_POINT_VALUE))
    counts: dict[str, int] = field(default_factory=dict)

    @property
    def pps(self) -> dict[str, float]:
        out = {}
        for z in set(list(self.fg_pct.keys()) + list(DEFAULT_FG_PCT.keys())):
            zc = legacy_zone_alias(z)
            fg = self.fg_pct.get(zc, DEFAULT_FG_PCT.get(zc, 0.40))
            pv = self.point_value.get(zc, DEFAULT_POINT_VALUE.get(zc, 2.0))
            out[zc] = fg * pv
        return out

    def xpoints(self, zone: str) -> float:
        z = legacy_zone_alias(zone)
        n = self.counts.get(z, 0)
        raw_fg = self.fg_pct.get(z, DEFAULT_FG_PCT.get(z, 0.40))
        default_fg = DEFAULT_FG_PCT.get(z, 0.40)
        fg = (n * raw_fg + SHRINK_K * default_fg) / (n + SHRINK_K) if n > 0 else default_fg
        pv = self.point_value.get(z, DEFAULT_POINT_VALUE.get(z, 2.0))
        return float(fg * pv)

    def fg_pct_for_zone(self, zone: str) -> float:
        z = legacy_zone_alias(zone)
        n = self.counts.get(z, 0)
        raw_fg = self.fg_pct.get(z, DEFAULT_FG_PCT.get(z, 0.40))
        default_fg = DEFAULT_FG_PCT.get(z, 0.40)
        if n <= 0:
            return default_fg
        return float((n * raw_fg + SHRINK_K * default_fg) / (n + SHRINK_K))

    def to_flat_pps(self) -> dict[str, float]:
        """Backward-compatible flat zone_pps dict."""
        return self.pps


def normalize_court_coords(df: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """Return x_ft, y_ft from tracking columns when present."""
    x_ft = pd.Series(np.nan, index=df.index, dtype=float)
    y_ft = pd.Series(np.nan, index=df.index, dtype=float)
    cols = {c.lower(): c for c in df.columns}
    ox = cols.get("original_x") or cols.get("xlegacy") or cols.get("x_legacy")
    oy = cols.get("original_y") or cols.get("ylegacy") or cols.get("y_legacy")
    if ox and oy:
        x = pd.to_numeric(df[ox], errors="coerce")
        y = pd.to_numeric(df[oy], errors="coerce")
        # 2026 tracking: 0-100 half-court scale → feet
        if (x.abs().max(skipna=True) or 0) <= 100 and (y.abs().max(skipna=True) or 0) <= 100:
            x_ft = (x / 100.0) * COURT_LENGTH_FT
            y_ft = (y / 100.0) * COURT_WIDTH_FT
        else:
            # V3 xLegacy/yLegacy: tenths-of-foot from basket (do not use for distance)
            x_ft = x / 10.0
            y_ft = y / 10.0
    return x_ft, y_ft


def assign_shot_zone_scalar(
    dist_ft: float,
    x_ft: float = np.nan,
    y_ft: float = np.nan,
    *,
    is_3pt_text: bool = False,
    area: str | None = None,
    area_detail: str | None = None,
) -> str:
    joined = f"{area or ''} {area_detail or ''}".lower()
    if "corner" in joined and ("3" in joined or "three" in joined):
        return "corner3"
    if "restricted" in joined or joined.strip() == "rim":
        return "restricted"
    if "paint" in joined:
        return "paint"
    if "above" in joined and "break" in joined:
        return "abovebreak3"
    if "3" in joined or "three" in joined:
        return "abovebreak3"

    d = float(dist_ft) if pd.notna(dist_ft) and dist_ft >= 0 else np.nan
    if pd.notna(x_ft) and pd.notna(y_ft):
        d_coord = float(np.sqrt(x_ft ** 2 + y_ft ** 2))
        if pd.isna(d) or d < 0:
            d = d_coord
        else:
            d = min(d, d_coord) if d_coord > 0 else d

    is_3 = bool(is_3pt_text)
    if pd.notna(d):
        if d >= CORNER3_DIST_MIN_FT and (is_3 or is_3pt_text):
            if pd.notna(y_ft) and abs(y_ft) <= CORNER3_Y_MAX_FT:
                return "corner3"
            return "abovebreak3"
        if is_3:
            return "abovebreak3"
        if d <= 4:
            return "restricted"
        if d <= 10:
            return "paint"
        if d <= 16:
            return "short_mid"
        if d < CORNER3_DIST_MIN_FT:
            return "midrange"
        return "abovebreak3" if is_3 else "midrange"

    return "abovebreak3" if is_3pt_text else "other"


def assign_shot_zones_df(df: pd.DataFrame, dist: pd.Series, combo: pd.Series) -> pd.Series:
    """Vectorized zone assignment with coordinate corner-3 override."""
    x_ft, y_ft = normalize_court_coords(df)
    is_3pt = combo.str.contains(r"3pt|three|3-pt| corner ", case=False, na=False)
    if "shot_value" in df.columns:
        sv = pd.to_numeric(df["shot_value"], errors="coerce")
        is_3pt = is_3pt | (sv >= 3)

    d = pd.to_numeric(dist, errors="coerce").fillna(-1)
    has_coord = x_ft.notna() & y_ft.notna() & (x_ft.abs() + y_ft.abs() > 0)
    # Prefer official shotDistance when present (especially V3)
    d_official = d.copy()
    d_coord = np.sqrt(x_ft.fillna(0) ** 2 + y_ft.fillna(0) ** 2)
    use_coord = has_coord & ((d_official < 0) | ((d_official > 0) & (d_coord > 0) & (d_coord < d_official * 0.5)))
    d = np.where(d_official >= 0, d_official, np.where(use_coord, d_coord, d_official))
    d = np.where((d <= 0) & is_3pt, 23.0, d)
    d = np.where((d <= 0) & ~is_3pt & (d_official < 0), -1, d)

    is_3pt = is_3pt | (d >= 23.75)

    area = df["area"].fillna("").astype(str).str.lower() if "area" in df.columns else pd.Series("", index=df.index)
    area_detail = (
        df["area_detail"].fillna("").astype(str).str.lower()
        if "area_detail" in df.columns else pd.Series("", index=df.index)
    )

    corner3 = (d >= CORNER3_DIST_MIN_FT) & is_3pt & (y_ft.abs() <= CORNER3_Y_MAX_FT) & has_coord
    abovebreak3 = (d >= CORNER3_DIST_MIN_FT) & is_3pt & ~corner3
    restricted = (d >= 0) & (d <= 4) & ~is_3pt
    paint = (d > 4) & (d <= 10) & ~is_3pt
    short_mid = (d > 10) & (d <= 16) & ~is_3pt
    midrange = (d > 16) & (d < CORNER3_DIST_MIN_FT) & ~is_3pt

    zone = np.select(
        [corner3, abovebreak3, restricted, paint, short_mid, midrange, is_3pt],
        ["corner3", "abovebreak3", "restricted", "paint", "short_mid", "midrange", "abovebreak3"],
        default="other",
    )
    zones = pd.Series(zone, index=df.index)

    joined = (area + " " + area_detail + " " + combo).str.strip()
    corner_text = joined.str.contains("corner", na=False) & joined.str.contains(r"3|three|3pt", case=False, na=False)
    zones = np.where(corner_text, "corner3", zones)
    restricted_text = joined.str.contains("restricted|at rim", na=False)
    zones = np.where(restricted_text, "restricted", zones)
    paint_text = joined.str.contains("paint", na=False)
    zones = np.where(paint_text & ~corner_text, "paint", zones)

    return pd.Series(zones, index=df.index).map(legacy_zone_alias)


def compute_zone_calibration(df: pd.DataFrame) -> ZoneCalibration:
    """Leak-free zone calibration from a season of PBP (for the *next* season)."""
    if df is None or df.empty or "EVENTMSGTYPE" not in df.columns:
        return ZoneCalibration()
    is_shot = df["EVENTMSGTYPE"].isin([1, 2])
    if not is_shot.any():
        return ZoneCalibration()
    if "shot_zone" not in df.columns:
        return ZoneCalibration()

    sub = df.loc[is_shot, ["shot_zone"]].copy()
    sub["zone"] = sub["shot_zone"].map(legacy_zone_alias)
    sub["made"] = df.loc[is_shot, "is_fg_make"].astype(float)
    sub["point_value"] = np.where(sub["zone"].isin(["corner3", "abovebreak3"]), 3.0, 2.0)

    fg = sub.groupby("zone")["made"].mean()
    counts = sub.groupby("zone")["made"].count()
    pv = sub.groupby("zone")["point_value"].first()

    calib = ZoneCalibration(
        fg_pct={str(z): float(v) for z, v in fg.items()},
        point_value={str(z): float(pv.get(z, DEFAULT_POINT_VALUE.get(str(z), 2.0))) for z in fg.index},
        counts={str(z): int(counts.get(z, 0)) for z in fg.index},
    )
    return calib


def compute_zone_pps(df: pd.DataFrame) -> dict[str, float]:
    """Backward-compatible wrapper returning flat PPS by zone."""
    return compute_zone_calibration(df).to_flat_pps()


In [ ]:
# ── module: ingest ───────────────────────────────────────────────────────────
"""PBP format converters for V3 (2022-25) and 2026 combined-stats."""
import numpy as np
import pandas as pd



def _col_series(df, col_map, *names, default=np.nan):
    """Return first matching column (case-insensitive) from raw frame."""
    for name in names:
        key = name.lower()
        if key in col_map:
            return df[col_map[key]]
    if default is None:
        return None
    return pd.Series(default, index=df.index)


def _attach_shot_metadata(df, raw, col_map):
    """Normalize shot location / area columns for preprocess + shot_zones."""
    sd = _col_series(df, col_map, "shot_distance", "shotdistance")
    if sd is not None:
        df["shot_distance"] = pd.to_numeric(sd, errors="coerce")

    ox = _col_series(df, col_map, "original_x", "xlegacy", "x_legacy")
    oy = _col_series(df, col_map, "original_y", "ylegacy", "y_legacy")
    if ox is not None:
        df["original_x"] = pd.to_numeric(ox, errors="coerce")
    if oy is not None:
        df["original_y"] = pd.to_numeric(oy, errors="coerce")

    area = _col_series(df, col_map, "area")
    if area is not None:
        df["area"] = area.fillna("").astype(str)

    ad = _col_series(df, col_map, "area_detail", "areadetail", "subtype")
    if ad is not None:
        df["area_detail"] = ad.fillna("").astype(str)

    sv = _col_series(df, col_map, "shot_value", "shotvalue")
    if sv is not None:
        df["shot_value"] = pd.to_numeric(sv, errors="coerce")

    # V3 uses isFieldGoal; keep for diagnostics
    fg = _col_series(df, col_map, "isfieldgoal", "is_field_goal", default=None)
    if fg is not None:
        df["is_field_goal"] = pd.to_numeric(fg, errors="coerce").fillna(0).astype(int)

    return df


# ─────────────────────────────────────────────────────────────────────
# 2025-26 combined-stats → internal standard
# ─────────────────────────────────────────────────────────────────────
def convert_new_pbp(pbp_df, name_to_id=None):
    """Convert 2026 combined-stats CSV to internal standard columns."""
    if name_to_id is None:
        name_to_id = {}
    df = pbp_df.copy()
    col_map = {c.lower(): c for c in df.columns}

    df["GAME_ID"] = df[col_map["game_id"]]
    df["PERIOD"] = pd.to_numeric(df[col_map["period"]], errors="coerce").fillna(1).astype(int)
    df["HOME_SCORE"] = pd.to_numeric(df[col_map["home_score"]], errors="coerce").fillna(0)
    df["AWAY_SCORE"] = pd.to_numeric(df[col_map["away_score"]], errors="coerce").fillna(0)
    # Task 008: the 2025-26 combined-stats source mixes ISO ("2025-10-21") and
    # M/D/YYYY ("12/17/2025") date strings in the same column. A plain
    # `pd.to_datetime(..., errors="coerce")` call infers one format from the
    # first row and silently turns every row in the *other* format into NaT
    # (confirmed: 7 games / 4,434 rows go NaT this way). `format="mixed"`
    # parses each value independently and must never produce a NaT for a
    # value that is actually parseable.
    df["game_date"] = pd.to_datetime(df[col_map["date"]], format="mixed", errors="coerce")

    if "remaining_time" in col_map:
        df["remaining_time"] = df[col_map["remaining_time"]]

    cond = [
        (df[col_map["event_type"]] == "shot") & (df[col_map["result"]] == "made"),
        (df[col_map["event_type"]] == "shot") & (df[col_map["result"]] == "missed"),
        (df[col_map["event_type"]] == "free throw"),
        (df[col_map["event_type"]] == "rebound"),
        (df[col_map["event_type"]] == "turnover"),
        (df[col_map["event_type"]] == "foul"),
        (df[col_map["event_type"]] == "substitution"),
    ]
    df["EVENTMSGTYPE"] = np.select(cond, [1, 2, 3, 4, 5, 6, 8], default=0)
    df["EVENTMSGACTIONTYPE"] = np.where(
        (df[col_map["event_type"]] == "free throw") & (df[col_map["type"]] == "1 of 1"), 16, 0
    )

    df["points"] = pd.to_numeric(df[col_map["points"]], errors="coerce").fillna(0).astype(int)

    df["HOMEDESCRIPTION"] = np.where(
        df[col_map["team"]] == df[col_map["home_team"]], df[col_map["description"]], np.nan
    )
    df["VISITORDESCRIPTION"] = np.where(
        df[col_map["team"]] == df[col_map["away_team"]], df[col_map["description"]], np.nan
    )

    df["PLAYER1_ID"] = df[col_map["player"]].apply(lambda x: _map_player_name(x, name_to_id))
    df["PLAYER2_ID"] = df[col_map["assist"]].apply(lambda x: _map_player_name(x, name_to_id))
    df["PLAYER3_ID"] = df[col_map["block"]].apply(lambda x: _map_player_name(x, name_to_id))

    h_id_cols = [f"h{i}" for i in range(1, 6)]
    a_id_cols = [f"a{i}" for i in range(1, 6)]

    def _lineup(row, cols):
        ids = [_map_player_name(row[c], name_to_id) for c in cols if c in row.index]
        ids = sorted(int(x) for x in ids if x is not None)
        return "-".join(map(str, ids))

    df["HOME_players"] = df.apply(lambda r: _lineup(r, h_id_cols), axis=1)
    df["AWAY_players"] = df.apply(lambda r: _lineup(r, a_id_cols), axis=1)

    df = _attach_shot_metadata(df, pbp_df, col_map)

    if "official" in col_map:
        df["official"] = df[col_map["official"]]

    df = df[(df["HOME_players"] != "") & (df["AWAY_players"] != "")].reset_index(drop=True)
    return df


# ─────────────────────────────────────────────────────────────────────
# V3-format (22-23 / 23-24 / 24-25) → internal standard
# ─────────────────────────────────────────────────────────────────────
def convert_v3_pbp(pbp_df):
    """
    V3 API PBP: gameId, shotDistance, clock, xLegacy/yLegacy, actionType/subType.
    """
    df = pbp_df.copy()
    col_map = {c.lower(): c for c in df.columns}

    df["GAME_ID"] = _col_series(df, col_map, "gameid", "game_id")
    df["PERIOD"] = pd.to_numeric(_col_series(df, col_map, "period"), errors="coerce").fillna(1).astype(int)
    df["HOME_SCORE"] = pd.to_numeric(_col_series(df, col_map, "scorehome", "home_score"), errors="coerce").fillna(0)
    df["AWAY_SCORE"] = pd.to_numeric(_col_series(df, col_map, "scoreaway", "away_score"), errors="coerce").fillna(0)

    if "game_date" in col_map or "date" in col_map:
        df["game_date"] = pd.to_datetime(_col_series(df, col_map, "game_date", "date"), errors="coerce")
    else:
        df["game_date"] = df["GAME_ID"].astype(str).apply(
            lambda x: pd.to_datetime(f"20{x[1:3]}-01-01") if len(x) >= 8 else pd.NaT
        )

    if "home_team" in col_map and "away_team" in col_map:
        df["home_team"] = _col_series(df, col_map, "home_team")
        df["away_team"] = _col_series(df, col_map, "away_team")
    elif "location" in col_map and "teamtricode" in col_map:
        loc_col = col_map["location"]
        tri_col = col_map["teamtricode"]
        home_map = df[df[loc_col].astype(str).str.lower() == "h"].groupby("GAME_ID")[tri_col].first()
        away_map = df[df[loc_col].astype(str).str.lower() == "v"].groupby("GAME_ID")[tri_col].first()
        df["home_team"] = df["GAME_ID"].map(home_map)
        df["away_team"] = df["GAME_ID"].map(away_map)

    at = _col_series(df, col_map, "actiontype", "event_type").astype(str).str.lower().fillna("")
    st = _col_series(df, col_map, "subtype", "type").astype(str).str.lower().fillna("")
    sr = _col_series(df, col_map, "shotresult", "result").astype(str).str.lower().fillna("")

    is_make = at.str.contains("made shot", na=False) | ((at.str.contains("shot", na=False)) & sr.str.contains("made", na=False))
    is_miss = at.str.contains("missed shot|miss", na=False) | ((at.str.contains("shot", na=False)) & sr.str.contains("miss", na=False))

    df["EVENTMSGTYPE"] = np.select(
        [
            is_make,
            is_miss,
            at.str.contains("free throw", na=False),
            at.str.contains("rebound", na=False),
            at.str.contains("turnover", na=False),
            at.str.contains("foul", na=False),
            at.str.contains("substitution", na=False),
        ],
        [1, 2, 3, 4, 5, 6, 8],
        default=0,
    )
    df["EVENTMSGACTIONTYPE"] = 0

    desc = _col_series(df, col_map, "description").fillna("")
    loc = _col_series(df, col_map, "location").fillna("")
    df["HOMEDESCRIPTION"] = np.where(loc.astype(str).str.lower() == "h", desc, np.nan)
    df["VISITORDESCRIPTION"] = np.where(loc.astype(str).str.lower() == "v", desc, np.nan)

    df["PLAYER1_ID"] = pd.to_numeric(_col_series(df, col_map, "personid", "player1_id"), errors="coerce")
    df["PLAYER2_ID"] = pd.to_numeric(_col_series(df, col_map, "player2_id"), errors="coerce")
    df["PLAYER3_ID"] = pd.to_numeric(_col_series(df, col_map, "player3_id"), errors="coerce")

    if "home_players_on" in col_map and "away_players_on" in col_map:
        hp_col = col_map["home_players_on"]
        ap_col = col_map["away_players_on"]
        df["HOME_players"] = df[hp_col].apply(
            lambda s: "-".join(sorted(str(x) for x in _parse_player_string(s))) if isinstance(s, str) else ""
        )
        df["AWAY_players"] = df[ap_col].apply(
            lambda s: "-".join(sorted(str(x) for x in _parse_player_string(s))) if isinstance(s, str) else ""
        )
    else:
        df["HOME_players"] = ""
        df["AWAY_players"] = ""

    df["points"] = pd.to_numeric(
        _col_series(df, col_map, "pointstotal", "points"), errors="coerce"
    ).fillna(0).astype(int)

    # V3 clock → remaining_time (ISO PT format or MM:SS)
    if "remaining_time" not in df.columns:
        clock = _col_series(df, col_map, "clock", "remaining_time", default=None)
        if clock is not None and not clock.isna().all():
            def _clock_to_minutes(c):
                if pd.isna(c):
                    return 12.0
                s = str(c).strip()
                if s.startswith("PT") and "M" in s:
                    try:
                        part = s.replace("PT", "").replace("S", "")
                        mins, secs = part.split("M") if "M" in part else (part, "0")
                        return float(mins) + float(secs or 0) / 60.0
                    except (ValueError, TypeError):
                        return 12.0
                if ":" in s:
                    try:
                        mins, secs = s.split(":")
                        return float(mins) + float(secs) / 60.0
                    except (ValueError, TypeError):
                        return 12.0
                return 12.0

            df["remaining_time"] = clock.apply(_clock_to_minutes)

    df = _attach_shot_metadata(df, pbp_df, col_map)

    # subType is useful for zone hints on V3 (e.g. "3PT Jump Shot")
    if "area_detail" not in df.columns or df["area_detail"].isna().all():
        if st is not None:
            df["area_detail"] = st.fillna("").astype(str)

    df = df[(df["HOME_players"] != "") & (df["AWAY_players"] != "")].reset_index(drop=True)
    return df


In [ ]:
# ── module: preprocess ───────────────────────────────────────────────────────
"""PBP preprocessing and xPoints."""
import re

import numpy as np
import pandas as pd


# Re-export for callers
__all__ = ["preprocess_pbp", "compute_zone_pps", "compute_zone_calibration", "ZoneCalibration"]


THREE_ZONES = frozenset({"corner3", "abovebreak3"})
RIM_ZONES = frozenset({"restricted", "paint"})


def _resolve_zone_calib(zone_pps=None, zone_calib=None) -> ZoneCalibration:
    if zone_calib is not None:
        return zone_calib
    if zone_pps is not None:
        calib = ZoneCalibration()
        for z, pps in zone_pps.items():
            zc = legacy_zone_alias(z)
            pv = 3.0 if zc in THREE_ZONES else 2.0
            fg = float(pps) / pv if pv > 0 else 0.40
            calib.fg_pct[zc] = fg
            calib.point_value[zc] = pv
        return calib
    return ZoneCalibration()


def preprocess_pbp(df, compute_xpoints=True, zone_pps=None, zone_calib=None):
    """
    Preprocess play-by-play data for possession and expected points calculations.

    zone_calib : ZoneCalibration or None
        Walk-forward calibration (preferred): xPoints = fg_pct[zone] × point_value[zone].
    zone_pps : dict or None
        Deprecated flat PPS map; converted to ZoneCalibration internally.
    """
    is_home = df["HOMEDESCRIPTION"].notna() & df["VISITORDESCRIPTION"].isna()
    is_away = df["VISITORDESCRIPTION"].notna() & df["HOMEDESCRIPTION"].isna()
    is_make = (df["EVENTMSGTYPE"] == 1)
    is_miss = (df["EVENTMSGTYPE"] == 2)
    is_tov = (df["EVENTMSGTYPE"] == 5)
    is_foul = (df["EVENTMSGTYPE"] == 6)

    df["is_fg_make"] = is_make.astype(int)
    df["is_fg_miss"] = is_miss.astype(int)
    df["is_tov"] = is_tov.astype(int)

    combo_desc = df["HOMEDESCRIPTION"].fillna("") + " " + df["VISITORDESCRIPTION"].fillna("")

    calib = _resolve_zone_calib(zone_pps=zone_pps, zone_calib=zone_calib)

    if compute_xpoints:
        if "shot_distance" not in df.columns or df["shot_distance"].isna().all():
            df["shot_distance"] = combo_desc.str.extract(r"(\d+)\s*(?:'|-foot| foot| ft)").astype(float)
            df["shot_distance"] = df["shot_distance"].fillna(-1)

        if "type" not in df.columns or df["type"].isna().all():
            df["type"] = combo_desc.str.lower()

        dist = pd.to_numeric(df.get("shot_distance", pd.Series([-1] * len(df))), errors="coerce").fillna(-1)
        combo = df.get("type", pd.Series([""] * len(df))).fillna("").str.lower()

        is_3pt_text = combo.str.contains("3pt|three", na=False)
        is_rim_text = combo.str.contains("layup|dunk|tip", na=False)
        is_floater_text = combo.str.contains("floater|hook", na=False)
        is_jumper_text = combo.str.contains("jump|fadeaway|bank", na=False)

        dist = np.where((dist == -1) & is_3pt_text, 25.0, dist)
        dist = np.where((dist == -1) & is_rim_text, 1.0, dist)
        dist = np.where((dist == -1) & is_floater_text, 7.0, dist)
        dist = np.where((dist == -1) & is_jumper_text & ~is_3pt_text, 15.0, dist)
        df["shot_distance"] = dist

        df["shot_zone"] = assign_shot_zones_df(df, pd.Series(dist, index=df.index), combo)
        is_shot = is_make | is_miss

        df["xPoints"] = df["shot_zone"].map(lambda z: calib.xpoints(z))
        df.loc[~is_shot, "xPoints"] = 0.0

        df["xefg"] = df["shot_zone"].map(lambda z: calib.fg_pct_for_zone(z))
        df.loc[~is_shot, "xefg"] = 0.0

        is_three_shot = df["shot_zone"].isin(list(THREE_ZONES)) | is_3pt_text
        is_rim_shot = df["shot_zone"].isin(list(RIM_ZONES))

        df["home_xefg_added"] = np.where(is_shot & is_home, df["xefg"], 0.0)
        df["away_xefg_added"] = np.where(is_shot & is_away, df["xefg"], 0.0)
        df["home_rim_fga"] = (is_shot & is_home & is_rim_shot).astype(int)
        df["away_rim_fga"] = (is_shot & is_away & is_rim_shot).astype(int)
        df["home_three_fga"] = (is_shot & is_home & is_three_shot).astype(int)
        df["away_three_fga"] = (is_shot & is_away & is_three_shot).astype(int)
        sd = pd.to_numeric(df["shot_distance"], errors="coerce").fillna(0)
        df["home_shot_dist_sum"] = np.where(is_shot & is_home, sd, 0.0)
        df["away_shot_dist_sum"] = np.where(is_shot & is_away, sd, 0.0)
    else:
        df["xPoints"] = 0.0
        df["xefg"] = 0.0
        for c in (
            "home_xefg_added", "away_xefg_added",
            "home_rim_fga", "away_rim_fga",
            "home_three_fga", "away_three_fga",
            "home_shot_dist_sum", "away_shot_dist_sum",
        ):
            df[c] = 0.0

    df["home_pts_added"] = 0
    df["away_pts_added"] = 0

    df.loc[is_make & is_home, "home_pts_added"] = np.where(
        combo_desc[is_make & is_home].str.contains("3PT", flags=re.IGNORECASE, na=False), 3, 2)

    df.loc[is_make & is_away, "away_pts_added"] = np.where(
        combo_desc[is_make & is_away].str.contains("3PT", flags=re.IGNORECASE, na=False), 3, 2)

    ft_made = (df["EVENTMSGTYPE"] == 3) & ~combo_desc.str.contains("MISS", flags=re.IGNORECASE, na=False)

    df.loc[ft_made & is_home, "home_pts_added"] = 1
    df.loc[ft_made & is_away, "away_pts_added"] = 1

    is_shot2 = is_make | is_miss
    df["home_xpts_added"] = 0.0
    df["away_xpts_added"] = 0.0
    df.loc[is_shot2 & is_home, "home_xpts_added"] = df.loc[is_shot2 & is_home, "xPoints"]
    df.loc[is_shot2 & is_away, "away_xpts_added"] = df.loc[is_shot2 & is_away, "xPoints"]
    ft_att = (df["EVENTMSGTYPE"] == 3)
    if "GAME_ID" in df.columns and "game_date" in df.columns:
        sort_cols = ["game_date", "GAME_ID"]
        df = df.sort_values(sort_cols).reset_index(drop=True)
        game_ft = (
            df.groupby("GAME_ID", sort=False)
            .apply(lambda g: pd.Series({
                "gm_made": int(ft_made.loc[g.index].sum()),
                "gm_att": int(ft_att.loc[g.index].sum()),
            }))
            .reset_index()
        )
        game_ft["cum_made"] = game_ft["gm_made"].cumsum().shift(1, fill_value=0)
        game_ft["cum_att"] = game_ft["gm_att"].cumsum().shift(1, fill_value=0)
        game_ft["ft_pct_prior"] = np.where(
            game_ft["cum_att"] > 0,
            game_ft["cum_made"] / game_ft["cum_att"],
            0.77,
        )
        df = df.merge(game_ft[["GAME_ID", "ft_pct_prior"]], on="GAME_ID", how="left")
        df.loc[ft_att & is_home, "home_xpts_added"] = df.loc[ft_att & is_home, "ft_pct_prior"]
        df.loc[ft_att & is_away, "away_xpts_added"] = df.loc[ft_att & is_away, "ft_pct_prior"]
        df.drop(columns=["ft_pct_prior"], inplace=True)
    else:
        ft_pct = 0.77
        df.loc[ft_att & is_home, "home_xpts_added"] = ft_pct
        df.loc[ft_att & is_away, "away_xpts_added"] = ft_pct

    is_final_ft = ft_att & combo_desc.str.contains(r"1 of 1|2 of 2|3 of 3", regex=True, na=False)
    is_tech = ft_att & (df.get("EVENTMSGACTIONTYPE", pd.Series(0, index=df.index)) == 16)
    is_and1 = ft_att & (df["EVENTMSGTYPE"].shift(1) == 1) & (df["PLAYER1_ID"] == df["PLAYER1_ID"].shift(1))
    df["is_true_ft_trip"] = (is_final_ft & ~is_and1 & ~is_tech).astype(int)

    df["shot_team_state"] = pd.Series(np.nan, index=df.index, dtype="object")
    df.loc[is_miss & is_home, "shot_team_state"] = "home"
    df.loc[is_miss & is_away, "shot_team_state"] = "away"
    reset_mask = df["EVENTMSGTYPE"].isin([1, 5, 8]) | (df["PERIOD"] != df["PERIOD"].shift(1))
    df.loc[reset_mask, "shot_team_state"] = "RESET"

    df["last_shot_team"] = df["shot_team_state"].ffill(inplace=False)
    df.loc[df["last_shot_team"] == "RESET", "last_shot_team"] = np.nan

    df["home_oreb"] = ((df["EVENTMSGTYPE"] == 4) & is_home & (df["last_shot_team"] == "home")).astype(int)
    df["away_oreb"] = ((df["EVENTMSGTYPE"] == 4) & is_away & (df["last_shot_team"] == "away")).astype(int)

    df["home_dreb"] = ((df["EVENTMSGTYPE"] == 4) & is_home & (df["last_shot_team"] == "away")).astype(int)
    df["away_dreb"] = ((df["EVENTMSGTYPE"] == 4) & is_away & (df["last_shot_team"] == "home")).astype(int)

    df["home_tovs_forced"] = (is_tov & is_away).astype(int)
    df["away_tovs_forced"] = (is_tov & is_home).astype(int)
    df["home_fouls_drawn"] = (is_foul & is_away).astype(int)
    df["away_fouls_drawn"] = (is_foul & is_home).astype(int)
    if "PLAYER3_ID" in df.columns:
        df["home_blks"] = (is_miss & is_away & df["PLAYER3_ID"].notna()).astype(int)
        df["away_blks"] = (is_miss & is_home & df["PLAYER3_ID"].notna()).astype(int)
    else:
        df["home_blks"] = df["away_blks"] = 0

    df["home_fgm"] = (is_make & is_home).astype(int)
    df["away_fgm"] = (is_make & is_away).astype(int)
    df["home_fga"] = ((is_make | is_miss) & is_home).astype(int)
    df["away_fga"] = ((is_make | is_miss) & is_away).astype(int)
    df["home_tov"] = (is_tov & is_home).astype(int)
    df["away_tov"] = (is_tov & is_away).astype(int)
    _ft_att = (df["EVENTMSGTYPE"] == 3)
    df["home_fta"] = (_ft_att & is_home).astype(int)
    df["away_fta"] = (_ft_att & is_away).astype(int)
    _is_steal = combo_desc.str.contains("STEAL", case=False, na=False)
    df["home_stl"] = (_is_steal & is_home).astype(int)
    df["away_stl"] = (_is_steal & is_away).astype(int)

    if "shot_zone" in df.columns:
        is_three_shot = df["shot_zone"].isin(list(THREE_ZONES))
    else:
        is_three_shot = combo_desc.str.contains("3PT", flags=re.IGNORECASE, na=False)
    df["home_3pm"] = (is_make & is_home & is_three_shot).astype(int)
    df["away_3pm"] = (is_make & is_away & is_three_shot).astype(int)
    df["home_3pa"] = ((is_make | is_miss) & is_home & is_three_shot).astype(int)
    df["away_3pa"] = ((is_make | is_miss) & is_away & is_three_shot).astype(int)

    df["home_poss"] = ((df["is_fg_make"] + df["is_fg_miss"] + df["is_tov"]) * is_home.astype(int)
                       - df["home_oreb"]
                       + df["is_true_ft_trip"] * is_home.astype(int))

    df["away_poss"] = ((df["is_fg_make"] + df["is_fg_miss"] + df["is_tov"]) * is_away.astype(int)
                       - df["away_oreb"]
                       + df["is_true_ft_trip"] * is_away.astype(int))

    df["total_possessions"] = (df["home_poss"] + df["away_poss"])

    def _min_rem(t):
        if pd.isna(t):
            return 12.0
        parts = str(t).split(":")
        return int(parts[0]) + int(parts[1]) / 60.0 if len(parts) == 2 else 12.0

    if "remaining_time" in df.columns:
        df["min_rem"] = df["remaining_time"].apply(_min_rem)
    else:
        df["min_rem"] = 12.0

    df["abs_margin"] = (
        pd.to_numeric(df.get("HOME_SCORE", pd.Series([0] * len(df))), errors="coerce").fillna(0)
        - pd.to_numeric(df.get("AWAY_SCORE", pd.Series([0] * len(df))), errors="coerce").fillna(0)
    ).abs()

    df["garbage"] = (df["PERIOD"] == 4) & (
        (df["abs_margin"] >= 20)
        | ((df["abs_margin"] >= 15) & (df["min_rem"] <= 3.0))
    )

    # Task 010/011: `home_pts_added`/`away_pts_added` are the immutable
    # scoring-outcome columns — every downstream label, canonical game score,
    # and wager settlement must be able to trust them regardless of garbage
    # time. They are NEVER zeroed here. Only the *rating-facing* weighted
    # possession/xPoints columns are downweighted during garbage time, since
    # those exist purely to reduce Elo/form credit for garbage-time events,
    # not to change what actually happened on the scoreboard.
    #
    # A confirmed leak fixed by this change: 1,090/1,307 2025-26 games had
    # their real final score understated by ~17.8 combined points on average
    # because this same zeroing used to also apply to home_pts_added/
    # away_pts_added, which fed directly into ACTUAL_HOME/ACTUAL_AWAY labels.
    weighted_rating_cols = ["total_possessions", "home_xpts_added", "away_xpts_added"]
    df.loc[df["garbage"], weighted_rating_cols] = 0.0

    return df


In [ ]:
# ── module: stints ───────────────────────────────────────────────────────────
"""Lineup stint builder."""

import pandas as pd

def build_stints(pbp_df, assist_split=None):
    if assist_split is None:
        assist_split = ASSIST_SPLIT
    # 1. Initialize DF IMMEDIATELY
    df = pbp_df.copy()

    # 2. Check if the dataframe is empty or missing necessary columns
    if df.empty or "total_possessions" not in df.columns:
        print("⚠️ Warning: Dataframe empty or missing columns. Returning empty stints.")
        return pd.DataFrame()

    # 3. Possession Safety Check (Task 012: a slate with zero *possessions*
    # can still have real scoring points, e.g. an isolated intermediate free
    # throw in a tiny fixture/edge case; only bail out when there is truly
    # nothing to build — no possessions AND no points).
    has_pts = ("home_pts_added" in df.columns and df["home_pts_added"].sum() > 0) or (
        "away_pts_added" in df.columns and df["away_pts_added"].sum() > 0
    )
    if df["total_possessions"].sum() == 0 and not has_pts:
        print("⚠️ Warning: Dataframe contains 0 total possessions and 0 points. Check your Preprocessing.")
        return pd.DataFrame()

    # 4. Grouping Logic
    # Task 013: include GAME_ID in the stint-boundary condition. Without it,
    # the last stint of one game and the first stint of the next game could
    # silently merge into a single stint whenever both games happen to share
    # the same lineup strings and PERIOD value (e.g. two period-1 stints with
    # unresolved/empty lineups), corrupting per-game aggregation.
    df["stint_id"] = (
        (df["HOME_players"] != df["HOME_players"].shift()) |
        (df["AWAY_players"] != df["AWAY_players"].shift()) |
        (df["PERIOD"]       != df["PERIOD"].shift()) |
        (df["GAME_ID"]      != df["GAME_ID"].shift())
    ).cumsum()

    stints = df.groupby("stint_id").agg(
        GAME_ID            = ("GAME_ID",           "first"),
        game_date          = ("game_date",         "first"),
        home_team          = ("home_team",         "first"),
        away_team          = ("away_team",         "first"),
        PERIOD             = ("PERIOD",            "first"),
        HOME_players       = ("HOME_players",      "first"),
        AWAY_players       = ("AWAY_players",      "first"),
        HOME_SCORE_START   = ("HOME_SCORE",        "first"),
        AWAY_SCORE_START   = ("AWAY_SCORE",        "first"),
        home_pts           = ("home_pts_added",    "sum"),
        away_pts           = ("away_pts_added",    "sum"),
        home_xpts          = ("home_xpts_added",   "sum"),
        away_xpts          = ("away_xpts_added",   "sum"),
        possessions        = ("total_possessions", "sum"),
        home_oreb          = ("home_oreb",         "sum"),
        away_oreb          = ("away_oreb",         "sum"),
        home_dreb          = ("home_dreb",         "sum"),
        away_dreb          = ("away_dreb",         "sum"),
        home_tovs_forced   = ("home_tovs_forced",  "sum"),
        away_tovs_forced   = ("away_tovs_forced",  "sum"),
        home_fouls_drawn   = ("home_fouls_drawn",  "sum"),
        away_fouls_drawn   = ("away_fouls_drawn",  "sum"),
        home_fgm           = ("home_fgm",          "sum"),
        away_fgm           = ("away_fgm",          "sum"),
        home_fga           = ("home_fga",          "sum"),
        away_fga           = ("away_fga",          "sum"),
        home_tov           = ("home_tov",          "sum"),
        away_tov           = ("away_tov",          "sum"),
        home_fta           = ("home_fta",          "sum"),
        away_fta           = ("away_fta",          "sum"),
        home_stl           = ("home_stl",          "sum"),
        away_stl           = ("away_stl",          "sum"),
        home_blks          = ("home_blks",         "sum"),
        away_blks          = ("away_blks",         "sum"),
        home_3pm           = ("home_3pm",          "sum"),
        away_3pm           = ("away_3pm",          "sum"),
        home_3pa           = ("home_3pa",          "sum"),
        away_3pa           = ("away_3pa",          "sum"),
        home_poss          = ("home_poss",         "sum"),
        away_poss          = ("away_poss",         "sum"),
        home_xefg_sum      = ("home_xefg_added",   "sum"),
        away_xefg_sum      = ("away_xefg_added",   "sum"),
        home_rim_fga       = ("home_rim_fga",      "sum"),
        away_rim_fga       = ("away_rim_fga",      "sum"),
        home_three_fga     = ("home_three_fga",    "sum"),
        away_three_fga     = ("away_three_fga",    "sum"),
        home_shot_dist_sum = ("home_shot_dist_sum","sum"),
        away_shot_dist_sum = ("away_shot_dist_sum","sum"),
    ).reset_index()

    stints["HOME_SCORE_END"] = stints["HOME_SCORE_START"] + stints["home_pts"]
    stints["AWAY_SCORE_END"] = stints["AWAY_SCORE_START"] + stints["away_pts"]

    # 5. Build Usage Dicts (Raw counts for speed)
    home_u = {row["stint_id"]: {p: [0.0, 0.0, 0.0] for p in str(row["HOME_players"]).split("-") if p and p != "nan"}
              for _, row in stints.iterrows()}
    away_u = {row["stint_id"]: {p: [0.0, 0.0, 0.0] for p in str(row["AWAY_players"]).split("-") if p and p != "nan"}
              for _, row in stints.iterrows()}

    records = df.to_dict("records")
    for i, row in enumerate(records):
        sid  = row["stint_id"]
        p1   = str(int(row["PLAYER1_ID"])) if pd.notna(row.get("PLAYER1_ID")) else None
        p2   = str(int(row["PLAYER2_ID"])) if pd.notna(row.get("PLAYER2_ID")) else None
        is_mk = row.get("is_fg_make", 0) == 1
        is_ms = row.get("is_fg_miss", 0) == 1
        is_tv = row.get("is_tov",     0) == 1
        is_ft = row.get("is_true_ft_trip", 0) == 1

        nxt_oreb = False
        if is_ms and i + 1 < len(records):
            nx = records[i + 1]
            if nx.get("EVENTMSGTYPE") == 4 and nx.get("PLAYER1_ID") == row.get("PLAYER1_ID"):
                nxt_oreb = True

        for usage in (home_u.get(sid, {}), away_u.get(sid, {})):
            if is_mk:
                if p2 in usage and p1 in usage:
                    usage[p1][1] += 1.0
                    usage[p2][2] += 1.0
                elif p1 in usage:
                    usage[p1][0] += 1.0
            elif is_ms and not nxt_oreb:
                if p1 in usage: usage[p1][0] += 1.0
            elif is_tv or is_ft:
                if p1 in usage: usage[p1][0] += 1.0

    stints["home_usage"] = stints["stint_id"].map(home_u)
    stints["away_usage"] = stints["stint_id"].map(away_u)

    # Task 012: a stint with zero *possessions* is not necessarily a stint
    # with zero *points* — an intermediate made free throw (e.g. the first of
    # a "2 of 2" trip) scores a point but is not counted as a completed
    # possession by `is_true_ft_trip` until the final FT of the trip. Only
    # drop truly empty stints (no possessions AND no points); any stint that
    # scored real points must survive so those points are never silently
    # dropped from the canonical/summed final score.
    keep = (stints["possessions"] > 0) | (stints["home_pts"] > 0) | (stints["away_pts"] > 0)
    return stints[keep].reset_index(drop=True)


In [ ]:
# ── module: dates ────────────────────────────────────────────────────────────
"""Recover real per-game dates for V3 PBP seasons.

The V3 PBP CSVs (2021-22 .. 2024-25) contain NO date column, so
`convert_v3_pbp` stamps every game with a constant ``YYYY-01-01``.  That
collapses the whole season onto one day, which (a) breaks odds matching by
date and (b) destroys every date-derived feature (rest days, recent form,
SOS, inactivity decay) and within-season chronological ordering.

This module rebuilds a ``GAME_ID -> date`` map by aligning each season's games
against the odds/schedule file (``all_odds.csv``), which DOES carry real dates
and team names.  NBA ``GAME_ID``s increment in (approximately) schedule order,
so for any given home/away matchup the k-th occurrence in GAME_ID order is the
k-th occurrence in date order.  Games that cannot be matched to an odds row
(e.g. no line recorded) get a monotonic date interpolated from the matched
anchors so ordering and rest-day features stay sane.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

# Odds-file short names -> NBA tricodes.
ODDS_NAME_TO_TRICODE = {
    "Atlanta": "ATL", "Boston": "BOS", "Brooklyn": "BKN", "Charlotte": "CHA",
    "Chicago": "CHI", "Cleveland": "CLE", "Dallas": "DAL", "Denver": "DEN",
    "Detroit": "DET", "Golden State": "GSW", "Houston": "HOU", "Indiana": "IND",
    "LA Clippers": "LAC", "Los Angeles Clippers": "LAC", "Clippers": "LAC",
    "LA Lakers": "LAL", "Los Angeles Lakers": "LAL", "Lakers": "LAL",
    "Memphis": "MEM", "Miami": "MIA", "Milwaukee": "MIL", "Minnesota": "MIN",
    "New Orleans": "NOP", "New York": "NYK", "Knicks": "NYK",
    "Oklahoma City": "OKC", "Orlando": "ORL", "Philadelphia": "PHI", "76ers": "PHI",
    "Phoenix": "PHX", "Portland": "POR", "Sacramento": "SAC",
    "San Antonio": "SAS", "Toronto": "TOR", "Utah": "UTA", "Washington": "WAS",
}

# Full names (PBP / TEAM_MAP style) -> tricode, so we can normalise either side.
_FULLNAME_TO_TRICODE = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Clippers": "LAC", "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM", "Miami Heat": "MIA", "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN", "New Orleans Pelicans": "NOP", "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC", "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHX", "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS", "Toronto Raptors": "TOR", "Utah Jazz": "UTA",
    "Washington Wizards": "WAS",
}

_VALID_TRICODES = set(_FULLNAME_TO_TRICODE.values())


def to_tricode(name) -> str | float:
    """Normalise any team label (tricode / short / full name) to a tricode."""
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return np.nan
    s = str(name).strip()
    if s in _VALID_TRICODES:
        return s
    if s in ODDS_NAME_TO_TRICODE:
        return ODDS_NAME_TO_TRICODE[s]
    if s in _FULLNAME_TO_TRICODE:
        return _FULLNAME_TO_TRICODE[s]
    # Try the trailing word(s) – e.g. "Los Angeles Lakers" already handled above.
    return s  # leave as-is; will simply fail to match if unknown


def gameid_season_start_year(game_id) -> int | None:
    """NBA GAME_IDs encode the season start year in digits [-7:-5] of the
    zero-padded 10-char id (e.g. 0022200001 -> '22' -> 2022)."""
    s = str(game_id).strip()
    digits = "".join(ch for ch in s if ch.isdigit())
    if len(digits) < 8:
        return None
    z = digits.zfill(10)
    try:
        yy = int(z[3:5])
    except ValueError:
        return None
    return 2000 + yy


def load_odds_schedule(odds_path: str) -> pd.DataFrame:
    """Return a tidy schedule [date, home, away, season] from the odds CSV.

    Dates are taken from the canonical date embedded in the odds ``game_id``
    (``nba.g.YYYYMMDDNN``) when present, else from ``game_date``.
    """
    df = pd.read_csv(odds_path, usecols=lambda c: c in {
        "game_id", "game_date", "home_team", "away_team"})

    # Prefer the date embedded in nba.g.YYYYMMDDNN (timezone-safe canonical date)
    def _date_from_id(gid):
        s = str(gid)
        digits = "".join(ch for ch in s if ch.isdigit())
        if len(digits) >= 8:
            try:
                return pd.to_datetime(digits[:8], format="%Y%m%d")
            except (ValueError, TypeError):
                return pd.NaT
        return pd.NaT

    date_from_id = df["game_id"].apply(_date_from_id) if "game_id" in df.columns else pd.Series(pd.NaT, index=df.index)
    date_from_col = pd.to_datetime(df["game_date"].astype(str).str[:10], errors="coerce") if "game_date" in df.columns else pd.Series(pd.NaT, index=df.index)
    date = date_from_id.fillna(date_from_col)

    sched = pd.DataFrame({
        "date": date,
        "home": df["home_team"].apply(to_tricode),
        "away": df["away_team"].apply(to_tricode),
    }).dropna(subset=["date", "home", "away"])
    sched["season"] = sched["date"].dt.year + (sched["date"].dt.month >= 9).astype(int)
    sched = sched.sort_values("date").reset_index(drop=True)
    return sched


def build_gameid_date_map(game_rows: pd.DataFrame, sched: pd.DataFrame,
                          season_start_year: int) -> dict:
    """Map each GAME_ID to a real date.

    Parameters
    ----------
    game_rows : DataFrame with columns [GAME_ID, home, away] (tricodes), one
        row per game, for a single season.
    sched : output of ``load_odds_schedule`` (all seasons).
    season_start_year : NBA season start year (e.g. 2022 for the 2022-23 season).

    Returns
    -------
    dict {GAME_ID: pd.Timestamp}
    """
    season_label = season_start_year + 1  # our internal season convention
    season_sched = sched[sched["season"] == season_label].copy()

    g = game_rows.copy()
    g["home"] = g["home"].apply(to_tricode)
    g["away"] = g["away"].apply(to_tricode)
    # Numeric id for ordering / interpolation
    g["_gid_num"] = g["GAME_ID"].astype(str).str.replace(r"\D", "", regex=True)
    g["_gid_num"] = pd.to_numeric(g["_gid_num"], errors="coerce")
    g = g.sort_values("_gid_num").reset_index(drop=True)

    date_map: dict = {}

    # Align per (home, away) matchup: k-th by GAME_ID == k-th by date.
    odds_by_matchup = {
        key: sub.sort_values("date")["date"].tolist()
        for key, sub in season_sched.groupby(["home", "away"])
    }
    for (home, away), sub in g.groupby(["home", "away"]):
        odds_dates = odds_by_matchup.get((home, away), [])
        sub = sub.sort_values("_gid_num")
        for i, (_, row) in enumerate(sub.iterrows()):
            if i < len(odds_dates):
                date_map[row["GAME_ID"]] = pd.Timestamp(odds_dates[i])

    # Fill unmatched games from matched anchors using GAME_ID *rank* as the
    # x-axis (uniform spacing avoids distortion from the playoff/play-in id
    # jumps).  Inside the anchor range we interpolate; beyond it we extrapolate
    # with a robust linear fit so a partially-covered (live) season's tail still
    # gets monotonically increasing dates instead of all collapsing onto the
    # last anchor.
    g["_rank"] = g["_gid_num"].rank(method="first")
    matched = g[g["GAME_ID"].isin(date_map)].copy()
    if not matched.empty:
        matched = matched.assign(_d=matched["GAME_ID"].map(date_map))
        matched = matched.dropna(subset=["_rank", "_d"]).sort_values("_rank")
        ax = matched["_rank"].to_numpy(dtype=float)
        ay = matched["_d"].astype("int64").to_numpy()  # ns since epoch

        # Linear fit for extrapolation outside [ax.min(), ax.max()].
        if len(ax) >= 2 and ax.max() > ax.min():
            slope, intercept = np.polyfit(ax, ay, 1)
        else:
            slope, intercept = 0.0, ay[0]

        lo, hi = ax.min(), ax.max()
        for _, row in g.iterrows():
            gid = row["GAME_ID"]
            if gid in date_map:
                continue
            xi = row["_rank"]
            if np.isnan(xi):
                continue
            if lo <= xi <= hi:
                yi = np.interp(xi, ax, ay)
            else:
                yi = slope * xi + intercept            # extrapolate the tail
            date_map[gid] = pd.Timestamp(int(round(yi)))
    else:
        # No odds coverage at all -> fall back to season-start sequential dates.
        base = pd.Timestamp(f"{season_start_year}-10-20")
        ranks = g["_gid_num"].rank(method="first").fillna(0).astype(int)
        for gid, r in zip(g["GAME_ID"], ranks):
            date_map[gid] = base + pd.Timedelta(days=int(r * 82 / max(len(g), 1)))

    return date_map


def attach_real_dates(stints_df: pd.DataFrame, sched: pd.DataFrame,
                      season_start_year: int, verbose: bool = True) -> pd.DataFrame:
    """Overwrite ``game_date`` in a season's stint DataFrame with recovered dates.

    Also attaches a ``date_status`` column (Task 009 provenance): games
    matched exactly to an odds/schedule row are ``schedule_matched``; games
    whose date came from matchup-order interpolation/extrapolation are
    ``interpolated`` and should be excluded from rest/travel evaluations
    until independently verified (Rule 5 / plan Phase 1B).

    Returns a new DataFrame (sorted by the recovered date + GAME_ID + stint).
    """
    df = stints_df.copy()
    game_rows = (
        df[["GAME_ID", "home_team", "away_team"]]
        .drop_duplicates("GAME_ID")
        .rename(columns={"home_team": "home", "away_team": "away"})
        .reset_index(drop=True)
    )
    date_map = build_gameid_date_map(game_rows, sched, season_start_year)

    n_total = len(game_rows)
    matched_ids = set()
    season_label = season_start_year + 1
    season_sched = sched[sched["season"] == season_label]
    odds_matchups = {
        (h, a) for h, a in zip(season_sched["home"].map(to_tricode), season_sched["away"].map(to_tricode))
    }
    for _, row in game_rows.iterrows():
        h, a = to_tricode(row["home"]), to_tricode(row["away"])
        if row["GAME_ID"] in date_map and (h, a) in odds_matchups:
            matched_ids.add(row["GAME_ID"])
    n_exact = len(matched_ids)

    status_map = {
        gid: (DATE_STATUS_SCHEDULE_MATCHED if gid in matched_ids else DATE_STATUS_INTERPOLATED)
        for gid in date_map
    }
    df["game_date"] = df["GAME_ID"].map(date_map).fillna(df["game_date"])
    df["date_status"] = df["GAME_ID"].map(status_map).fillna(DATE_STATUS_UNRESOLVED)
    if verbose:
        print(f"  [dates] season {season_start_year}-{season_start_year+1}: "
              f"assigned dates to {n_exact}/{n_total} games via exact schedule match "
              f"({n_total - n_exact} interpolated) "
              f"(date range {df['game_date'].min()} -> {df['game_date'].max()})")
    sort_cols = [c for c in ["game_date", "GAME_ID", "stint_id"] if c in df.columns]
    return df.sort_values(sort_cols).reset_index(drop=True)


def is_valid_timestamp(ts) -> bool:
    """True when *ts* is a usable calendar timestamp (not None/NaT)."""
    if ts is None:
        return False
    try:
        t = pd.Timestamp(ts)
    except (ValueError, TypeError):
        return False
    return not pd.isna(t)


def date_from_game_id(game_id) -> pd.Timestamp:
    """Best-effort YYYYMMDD extraction from odds-style or digit-heavy ids."""
    if game_id is None:
        return pd.NaT
    digits = "".join(ch for ch in str(game_id) if ch.isdigit())
    if len(digits) < 8:
        return pd.NaT
    for start in range(0, len(digits) - 7):
        chunk = digits[start : start + 8]
        try:
            ts = pd.to_datetime(chunk, format="%Y%m%d")
            if 1990 <= ts.year <= 2040:
                return ts
        except (ValueError, TypeError):
            continue
    return pd.NaT


DATE_STATUS_AUTHORITATIVE = "authoritative"
DATE_STATUS_SCHEDULE_MATCHED = "schedule_matched"
DATE_STATUS_INTERPOLATED = "interpolated"
DATE_STATUS_UNRESOLVED = "unresolved"


@dataclass(frozen=True)
class DateResolution:
    """Provenance-bearing result of resolving one game's date.

    ``status`` is one of the four values above. ``date`` is ``None`` when
    ``status == "unresolved"`` — callers MUST fail closed for time-sensitive
    uses in that case rather than substituting an invented date (Task 009 /
    Rule 5).
    """

    date: pd.Timestamp | None
    status: str


def resolve_game_date(
    gdate,
    *,
    game_id=None,
) -> DateResolution:
    """Resolve one game's date with explicit provenance, never fabricating one.

    Resolution order:
      1. ``gdate`` parses to a real timestamp -> ``authoritative``.
      2. The ``GAME_ID`` itself encodes a plausible ``YYYYMMDD`` (e.g. the
         odds-schedule ``nba.g.YYYYMMDDNN`` convention, or embedded digits in
         an NBA-style id) -> ``schedule_matched`` (the identifier is matched
         against the game's own authoritative identity, not invented).
      3. Otherwise -> ``unresolved``, ``date=None``. Callers must not invent
         ``previous_date + 1`` or any other placeholder.

    Matchup-order interpolation (``build_gameid_date_map``) is a *separate*,
    explicitly flagged last resort for whole V3 seasons lacking any date
    column; its results are tagged ``interpolated`` by ``attach_real_dates``
    and must stay out of rest/travel evaluations until verified (Rule 5 /
    Phase 1B of the plan).
    """
    ts = pd.to_datetime(gdate, errors="coerce") if gdate is not None else pd.NaT
    if is_valid_timestamp(ts):
        return DateResolution(pd.Timestamp(ts).normalize(), DATE_STATUS_AUTHORITATIVE)

    if game_id is not None:
        from_gid = date_from_game_id(game_id)
        if is_valid_timestamp(from_gid):
            return DateResolution(pd.Timestamp(from_gid).normalize(), DATE_STATUS_SCHEDULE_MATCHED)

    return DateResolution(None, DATE_STATUS_UNRESOLVED)


def coerce_game_date(
    gdate,
    *,
    game_id=None,
    prev_date=None,
    season_start_year: int | None = None,
) -> pd.Timestamp | None:
    """Return a normalized pd.Timestamp, or ``None`` when unresolved.

    Task 009: this used to silently fabricate ``prev_date + 1 day`` (and, as a
    last resort, a fixed ``season_start_year-10-20``) whenever the real date
    was missing. That fabrication is removed: a missing/unparseable date with
    no game-identity match is now ``unresolved`` and returns ``None``, which
    every caller already treats as "skip this row" (fail closed) rather than
    silently invent chronology. ``prev_date`` and ``season_start_year`` are
    kept as accepted (now-unused) parameters for backward compatibility with
    existing call sites; they no longer influence the result.
    """
    resolution = resolve_game_date(gdate, game_id=game_id)
    return resolution.date


def to_py_date(gdate, *, game_id=None, prev_date=None):
    """Return datetime.date or None — never a fabricated date, never NaTType."""
    ts = coerce_game_date(gdate, game_id=game_id, prev_date=prev_date)
    if ts is None:
        return None
    return ts.date()


In [ ]:
# ── module: ratings ──────────────────────────────────────────────────────────
"""Player-level Elo/xPPP rating tracker."""
import math
from collections import defaultdict
from datetime import date as date_type

import numpy as np
import pandas as pd



class PlayerRatingTracker:
    """
    Glicko‑2 inspired rating tracker with separate offensive/defensive ratings.
    Maintains μ (rating), RD (uncertainty), and simplified volatility tracking.
    Context-aware stint multipliers use PBP-derived box stats passed via stint_ctx.
    """

    DEFAULTS = {
        "tau": 0.5,
        "epsilon": 1e-6,
        "default_mu": 1500.0,
        "default_rd": 350.0,
        "default_sigma": 0.06,
        "HOME_PPP_BOOST": HOME_PPP_BOOST,
        "OFFSEASON_REVERSION": OFFSEASON_REVERSION,
        "USAGE_FLOOR": USAGE_FLOOR,
        "assist_split": ASSIST_SPLIT,
        "ELO_SCALING_FACTOR": 1000,
        "K_OFF": 0.9,
        "K_DEF": 0.9,
        "k_mult_half_life": K_MULT_HALF_LIFE,
        "rd_floor": 30.0,
        "garbage_time_weight": GARBAGE_TIME_WEIGHT,
        "clutch_boost": CLUTCH_BOOST,
        "tov_penalty": TOV_PENALTY,
        "foul_draw_boost": FOUL_DRAW_BOOST,
        "variance_dampen": VARIANCE_DAMPEN,
        "xppp_actual_blend": XPPP_ACTUAL_BLEND,
        "k_def_events": K_DEF_EVENTS,
        "tov_rate_threshold": TOV_RATE_THRESHOLD,
        "three_pa_rate_threshold": THREE_PA_RATE_THRESHOLD,
    }

    def __init__(self, config=None, league_xppp=1.10):
        self.cfg = {**self.DEFAULTS, **(config or {})}
        self.league_xppp = league_xppp
        self.players = {}
        self.player_games = defaultdict(int)

    def _get(self, pid):
        pid = str(pid)
        if pid not in self.players:
            self.players[pid] = {
                "O_mu": self.cfg["default_mu"],
                "D_mu": self.cfg["default_mu"],
                "D_rim_mu": self.cfg["default_mu"],
                "D_peri_mu": self.cfg["default_mu"],
                "O_rd": self.cfg["default_rd"],
                "D_rd": self.cfg["default_rd"],
                "D_rim_rd": self.cfg["default_rd"],
                "D_peri_rd": self.cfg["default_rd"],
                "O_sigma": self.cfg["default_sigma"],
                "D_sigma": self.cfg["default_sigma"],
                "Possessions": 0,
                "luck_pts": 0.0,
                "def_events": 0.0,
                "off_tov": 0.0,
                "last_date": None,
            }
        pl = self.players[pid]
        if "D_rim_mu" not in pl:
            pl["D_rim_mu"] = pl["D_mu"]
            pl["D_peri_mu"] = pl["D_mu"]
            pl["D_rim_rd"] = pl["D_rd"]
            pl["D_peri_rd"] = pl["D_rd"]
        return pl

    @property
    def O_Elo(self):
        return {pid: data["O_mu"] for pid, data in self.players.items()}

    @property
    def D_Elo(self):
        return {pid: data["D_mu"] for pid, data in self.players.items()}

    def lineup_uncertainty(self, player_ids):
        ids = [str(x) for x in player_ids if x and str(x) != "nan"]
        if not ids:
            return 350.0, 350.0
        o_rd = np.mean([self._get(p)["O_rd"] for p in ids])
        d_rd = np.mean([self._get(p)["D_rd"] for p in ids])
        return o_rd, d_rd

    def weighted_lineup_uncertainty(self, weighted_ids):
        pairs = [(str(p), float(w)) for p, w in weighted_ids if p and w > 0]
        if not pairs:
            return 350.0, 350.0
        ws = sum(w for _, w in pairs)
        o_rd = sum(self._get(p)["O_rd"] * w for p, w in pairs) / ws
        d_rd = sum(self._get(p)["D_rd"] * w for p, w in pairs) / ws
        return o_rd, d_rd

    def lineup_rolling_rates(self, player_ids, weights=None):
        """Lineup-level luck/def-event/tov rates from accumulated player stats."""
        ids = [str(x) for x in player_ids if x and str(x) != "nan"]
        if not ids:
            return 0.0, 0.0, 0.0

        if weights:
            pairs = [(str(p), float(w)) for p, w in weights if p and w > 0]
            if pairs:
                ws = sum(w for _, w in pairs)
                luck = sum(self._get(p)["luck_pts"] * w for p, w in pairs) / ws
                defe = sum(self._get(p)["def_events"] * w for p, w in pairs) / ws
                tov = sum(self._get(p)["off_tov"] * w for p, w in pairs) / ws
                poss = sum(self._get(p)["Possessions"] * w for p, w in pairs) / ws
                if poss > 0:
                    return luck / poss, defe / poss, tov / poss
                return 0.0, 0.0, 0.0

        poss = np.mean([max(self._get(p)["Possessions"], 1.0) for p in ids])
        luck = np.mean([self._get(p)["luck_pts"] for p in ids]) / poss
        defe = np.mean([self._get(p)["def_events"] for p in ids]) / poss
        tov = np.mean([self._get(p)["off_tov"] for p in ids]) / poss
        return float(luck), float(defe), float(tov)

    def apply_inactivity_decay(self, player_ids, date):
        if date is None:
            return
        try:
            if pd.isna(date):
                return
        except (TypeError, ValueError):
            pass

        if isinstance(date, date_type):
            py_date = date
        elif is_valid_timestamp(date):
            py_date = pd.Timestamp(date).date()
        else:
            py_date = to_py_date(date)
            if py_date is None:
                return

        for pid in player_ids:
            p = self._get(pid)
            last = p.get("last_date")
            if last is not None:
                try:
                    if isinstance(last, pd.Timestamp):
                        if pd.isna(last):
                            last = None
                        else:
                            last = last.date()
                    if last is not None:
                        days = (py_date - last).days
                        if days > 60:
                            p["O_rd"] = min(350.0, p["O_rd"] * (1 + 0.1 * (days / 7)))
                            p["D_rd"] = min(350.0, p["D_rd"] * (1 + 0.1 * (days / 7)))
                except (TypeError, ValueError):
                    pass
            p["last_date"] = py_date

    def offseason_revert(self, returning_player_ids=None):
        base_rev = self.cfg["OFFSEASON_REVERSION"]
        default_rd = self.cfg["default_rd"]
        default_sigma = self.cfg["default_sigma"]

        for pid, p in self.players.items():
            if returning_player_ids is not None and pid not in returning_player_ids:
                effective_rev = min(base_rev * 2.0, 0.50)
            else:
                effective_rev = base_rev

            p["O_mu"] = 1500.0 + (p["O_mu"] - 1500.0) * (1 - effective_rev)
            p["D_mu"] = 1500.0 + (p["D_mu"] - 1500.0) * (1 - effective_rev)
            p["D_rim_mu"] = 1500.0 + (p.get("D_rim_mu", p["D_mu"]) - 1500.0) * (1 - effective_rev)
            p["D_peri_mu"] = 1500.0 + (p.get("D_peri_mu", p["D_mu"]) - 1500.0) * (1 - effective_rev)
            p["O_rd"] = default_rd
            p["D_rd"] = default_rd
            p["D_rim_rd"] = default_rd
            p["D_peri_rd"] = default_rd
            p["O_sigma"] = default_sigma
            p["D_sigma"] = default_sigma
            p["Possessions"] = 0
            p["luck_pts"] = 0.0
            p["def_events"] = 0.0
            p["off_tov"] = 0.0

        self.player_games.clear()

    def lineup_stats(self, player_ids, weights=None, opp_rim_rate=None, opp_three_rate=None):
        ids = [str(x) for x in player_ids if x and str(x) != "nan"]
        if not ids:
            return 1500.0, 1500.0, 0.0
        if weights:
            pairs = [(str(p), float(w)) for p, w in weights if p and w > 0]
            if pairs:
                off, _, exp = self.weighted_lineup_stats(pairs)
                if opp_rim_rate is not None and opp_three_rate is not None:
                    dff = self.matchup_def_rating(ids, opp_rim_rate, opp_three_rate, weights=pairs)
                else:
                    _, dff, _ = self.weighted_lineup_stats(pairs)
                return off, dff, exp
        off = np.mean([self._get(p)["O_mu"] for p in ids])
        if opp_rim_rate is not None and opp_three_rate is not None:
            dff = self.matchup_def_rating(ids, opp_rim_rate, opp_three_rate)
        else:
            dff = np.mean([self._get(p)["D_mu"] for p in ids])
        exp = np.mean([self._get(p)["Possessions"] for p in ids])
        return off, dff, exp

    def _lineup_rim_peri(self, player_ids, weights=None):
        ids = [str(x) for x in player_ids if x and str(x) != "nan"]
        if not ids:
            return 1500.0, 1500.0, 1500.0
        if weights:
            pairs = [(str(p), float(w)) for p, w in weights if p and w > 0]
            if pairs:
                ws = sum(w for _, w in pairs)
                rim = sum(self._get(p)["D_rim_mu"] * w for p, w in pairs) / ws
                peri = sum(self._get(p)["D_peri_mu"] * w for p, w in pairs) / ws
                legacy = sum(self._get(p)["D_mu"] * w for p, w in pairs) / ws
                return float(rim), float(peri), float(legacy)
        rim = np.mean([self._get(p)["D_rim_mu"] for p in ids])
        peri = np.mean([self._get(p)["D_peri_mu"] for p in ids])
        legacy = np.mean([self._get(p)["D_mu"] for p in ids])
        return float(rim), float(peri), float(legacy)

    def matchup_def_rating(self, player_ids, opp_rim_rate, opp_three_rate, weights=None):
        """Blend rim/perimeter/mid defensive Elo by opponent shot profile."""
        rim, peri, legacy = self._lineup_rim_peri(player_ids, weights)
        rim_r = float(np.clip(opp_rim_rate, 0.05, 0.65))
        peri_r = float(np.clip(opp_three_rate, 0.05, 0.65))
        mid_r = max(0.12, 1.0 - rim_r - peri_r)
        total = rim_r + peri_r + mid_r
        rw, pw, mw = rim_r / total, peri_r / total, mid_r / total
        return rw * rim + pw * peri + mw * legacy

    def weighted_lineup_stats(self, weighted_ids, opp_rim_rate=None, opp_three_rate=None):
        pairs = [(str(p), float(w)) for p, w in weighted_ids if p and w > 0]
        if not pairs:
            return 1500.0, 1500.0, 0.0
        ws = sum(w for _, w in pairs)
        off = sum(self._get(p)["O_mu"] * w for p, w in pairs) / ws
        ids = [p for p, _ in pairs]
        if opp_rim_rate is not None and opp_three_rate is not None:
            dff = self.matchup_def_rating(ids, opp_rim_rate, opp_three_rate, weights=pairs)
        else:
            dff = sum(self._get(p)["D_mu"] * w for p, w in pairs) / ws
        exp = sum(self._get(p)["Possessions"] * w for p, w in pairs) / ws
        return off, dff, exp

    def _def_credit_weights(self, stint_ctx, defender_side: str):
        """Rim/perimeter/mid credit shares for a defending side ('home'|'away')."""
        stint_ctx = stint_ctx or {}
        if defender_side == "home":
            rim = float(stint_ctx.get("away_rim_rate", 0.30))
            peri = float(stint_ctx.get("away_3pa_rate", 0.38))
        else:
            rim = float(stint_ctx.get("home_rim_rate", 0.30))
            peri = float(stint_ctx.get("home_3pa_rate", 0.38))
        mid = max(0.12, 1.0 - rim - peri)
        total = rim + peri + mid
        return rim / total, peri / total, mid / total

    def _update_def_split(self, ids, error, poss, usage, wt, stint_ctx, defender_side, abs_err=0.0):
        rw, pw, mw = self._def_credit_weights(stint_ctx, defender_side)
        self._update_ratings(ids, error * rw, poss, usage, side="def_rim", abs_err=abs_err)
        self._update_ratings(ids, error * pw, poss, usage, side="def_peri", abs_err=abs_err)
        self._update_ratings(ids, error * mw, poss, usage, side="def", abs_err=abs_err)
        for p in ids:
            pl = self._get(p)
            pl["D_mu"] = 0.5 * (pl["D_rim_mu"] + pl["D_peri_mu"])

    def _context_multiplier(self, stint_ctx, side: str) -> float:
        """Boost/penalize stint weight from PBP box context (home=A, away=B)."""
        if not stint_ctx:
            return 1.0
        m = 1.0
        if stint_ctx.get("clutch"):
            m *= self.cfg.get("clutch_boost", 1.0)
        if stint_ctx.get("garbage"):
            m *= self.cfg.get("garbage_time_weight", 0.5)

        tov_thr = self.cfg.get("tov_rate_threshold", 0.15)
        if side == "home":
            tov_rate = stint_ctx.get("home_tov_rate", 0.0)
            foul_rate = stint_ctx.get("home_foul_draw_rate", 0.0)
            three_rate = stint_ctx.get("home_3pa_rate", 0.0)
        else:
            tov_rate = stint_ctx.get("away_tov_rate", 0.0)
            foul_rate = stint_ctx.get("away_foul_draw_rate", 0.0)
            three_rate = stint_ctx.get("away_3pa_rate", 0.0)

        if tov_rate > tov_thr:
            m *= self.cfg.get("tov_penalty", 1.0)
        if foul_rate > 0.12:
            m *= self.cfg.get("foul_draw_boost", 1.0)
        if three_rate > self.cfg.get("three_pa_rate_threshold", 0.45):
            m *= self.cfg.get("variance_dampen", 1.0)
        return m

    def _accumulate_player_context(self, ids, usage, poss, stint_ctx, side: str) -> None:
        if not stint_ctx or poss <= 0:
            return
        n = max(len(ids), 1)
        base_w = 1.0 / n
        total_usage = sum(usage.values()) if usage else 0
        u_floor = self.cfg["USAGE_FLOOR"]

        if side == "home":
            luck_share = stint_ctx.get("home_luck_ppp", 0.0) * poss
            def_ev = (stint_ctx.get("home_def_events", 0.0)) * poss
            tov_ev = stint_ctx.get("home_tov_rate", 0.0) * poss
        else:
            luck_share = stint_ctx.get("away_luck_ppp", 0.0) * poss
            def_ev = stint_ctx.get("away_def_events", 0.0) * poss
            tov_ev = stint_ctx.get("away_tov_rate", 0.0) * poss

        shares = []
        for p in ids:
            if total_usage > 0:
                raw = usage.get(p, 0) / total_usage
            else:
                raw = base_w
            shares.append(max(raw, base_w * u_floor))
        s = sum(shares) or 1.0

        for i, p in enumerate(ids):
            pl = self._get(p)
            sh = shares[i] / s
            pl["luck_pts"] += luck_share * sh
            pl["def_events"] += def_ev * sh
            pl["off_tov"] += tov_ev * sh

    def _apply_def_event_credit(self, ids, usage, poss, stint_ctx, side: str, wt: float) -> None:
        """Explicit micro-credit for steals/blocks/forced turnovers."""
        if not stint_ctx or poss <= 0:
            return
        k_def_ev = self.cfg.get("k_def_events", 0.0)
        if k_def_ev <= 0:
            return
        if side == "home":
            rate = stint_ctx.get("home_def_events", 0.0)
        else:
            rate = stint_ctx.get("away_def_events", 0.0)
        if rate <= 0:
            return
        credit = k_def_ev * rate * wt
        self._update_ratings(ids, credit, poss, usage, side="def", abs_err=0.0)

    def process_stint(self, ids_A, ids_B, poss, xpts_A, xpts_B,
                      usage_A=None, usage_B=None,
                      period=1, start_A=0, start_B=0,
                      end_A=0, end_B=0, season_progress=0.5,
                      stint_ctx=None):
        ast_split = self.cfg.get("assist_split", ASSIST_SPLIT)

        def collapse_usage(u_dict):
            if not u_dict:
                return {}
            return {p: (v[0] + v[1] * ast_split + v[2] * (1 - ast_split))
                    for p, v in u_dict.items()}

        flat_usage_A = collapse_usage(usage_A)
        flat_usage_B = collapse_usage(usage_B)

        def _usage_weights(u_dict, ids):
            if not u_dict:
                n = max(len(ids), 1)
                return [(p, 1.0 / n) for p in ids]
            total = sum(u_dict.values()) or 1.0
            return [(p, u_dict.get(p, 0) / total) for p in ids]

        wA = _usage_weights(flat_usage_A, ids_A)
        wB = _usage_weights(flat_usage_B, ids_B)

        opp_rim_for_home_d = float(stint_ctx.get("away_rim_rate", 0.30) if stint_ctx else 0.30)
        opp_peri_for_home_d = float(stint_ctx.get("away_3pa_rate", 0.38) if stint_ctx else 0.38)
        opp_rim_for_away_d = float(stint_ctx.get("home_rim_rate", 0.30) if stint_ctx else 0.30)
        opp_peri_for_away_d = float(stint_ctx.get("home_3pa_rate", 0.38) if stint_ctx else 0.38)

        off_A, def_A, _ = self.lineup_stats(
            ids_A, weights=wA, opp_rim_rate=opp_rim_for_away_d, opp_three_rate=opp_peri_for_away_d,
        )
        off_B, def_B, _ = self.lineup_stats(
            ids_B, weights=wB, opp_rim_rate=opp_rim_for_home_d, opp_three_rate=opp_peri_for_home_d,
        )
        margin_A = end_A - start_A

        if stint_ctx is None:
            stint_ctx = {}
        else:
            stint_ctx = dict(stint_ctx)
        stint_ctx.setdefault("clutch", period >= 4 and abs(start_A - start_B) < 10)
        stint_ctx.setdefault("garbage", period >= 4 and abs(margin_A) >= 15)

        wt = self._weight_stint(poss, period, margin_A, season_progress, stint_ctx)
        wt_a = wt * self._context_multiplier(stint_ctx, "home")
        wt_b = wt * self._context_multiplier(stint_ctx, "away")

        scaling = self.cfg["ELO_SCALING_FACTOR"]
        h_boost = self.cfg["HOME_PPP_BOOST"]

        exp_ppp_A = self.league_xppp + h_boost + (off_A - def_B) / scaling
        exp_ppp_B = self.league_xppp - h_boost + (off_B - def_A) / scaling

        act_ppp_A = xpts_A / poss if poss > 0 else 0
        act_ppp_B = xpts_B / poss if poss > 0 else 0

        blend = self.cfg.get("xppp_actual_blend", 0.0)
        if blend > 0 and poss > 0:
            act_pts_A = stint_ctx.get("home_pts", xpts_A) / poss
            act_pts_B = stint_ctx.get("away_pts", xpts_B) / poss
            act_ppp_A = (1.0 - blend) * act_ppp_A + blend * act_pts_A
            act_ppp_B = (1.0 - blend) * act_ppp_B + blend * act_pts_B

        err_A = act_ppp_A - exp_ppp_A
        err_B = act_ppp_B - exp_ppp_B

        self._update_ratings(ids_A, err_A * wt_a, poss, flat_usage_A, side="off", abs_err=abs(err_A))
        self._update_def_split(
            ids_B, err_B * wt_b, poss, flat_usage_B, wt_b, stint_ctx, "away", abs_err=abs(err_B),
        )
        self._update_def_split(
            ids_A, -err_B * wt_a, poss, flat_usage_A, wt_a, stint_ctx, "home", abs_err=abs(err_B),
        )
        self._update_ratings(ids_B, -err_A * wt_b, poss, flat_usage_B, side="off", abs_err=abs(err_A))

        self._apply_def_event_credit(ids_A, flat_usage_A, poss, stint_ctx, "home", wt_a)
        self._apply_def_event_credit(ids_B, flat_usage_B, poss, stint_ctx, "away", wt_b)

        self._accumulate_player_context(ids_A, flat_usage_A, poss, stint_ctx, "home")
        self._accumulate_player_context(ids_B, flat_usage_B, poss, stint_ctx, "away")

    def _update_ratings(self, ids, error, poss, usage, side, abs_err=0.0):
        ids = [str(x) for x in ids if x and str(x) != "nan"]
        if not ids:
            return

        n = len(ids)
        base_w = 1.0 / n
        total_usage = sum(usage.values()) if usage else 0
        u_floor = self.cfg["USAGE_FLOOR"]
        half_life = self.cfg.get("k_mult_half_life", 15.0)
        rd_floor = self.cfg.get("rd_floor", 30.0)

        shares = []
        for p in ids:
            if total_usage > 0:
                raw = usage.get(p, 0) / total_usage
            else:
                raw = base_w
            shares.append(max(raw, base_w * u_floor))
        s = sum(shares)

        for i, p in enumerate(ids):
            pl = self._get(p)
            games = self.player_games[p]
            k_mult = max(0.5, 2.0 * math.exp(-games / max(half_life, 1.0)))
            k_base = self.cfg.get("K_OFF", 0.9) if side == "off" else self.cfg.get("K_DEF", 0.9)
            if side == "def_rim":
                rd = pl["D_rim_rd"]
            elif side == "def_peri":
                rd = pl["D_peri_rd"]
            elif side == "off":
                rd = pl["O_rd"]
            else:
                rd = pl["D_rd"]
            k_effective = k_base * (rd / 350.0)
            delta = error * k_effective * (shares[i] / s) * k_mult

            if side == "off":
                pl["O_mu"] += delta
                rd_new = pl["O_rd"] * (0.99 + 0.02 * min(abs_err, 0.15))
                pl["O_rd"] = max(rd_floor, min(350.0, rd_new))
                pl["O_sigma"] = min(0.15, pl["O_sigma"] * (1.0 + 0.05 * min(abs_err, 0.2)))
            elif side == "def_rim":
                pl["D_rim_mu"] += delta
                rd_new = pl["D_rim_rd"] * (0.99 + 0.02 * min(abs_err, 0.15))
                pl["D_rim_rd"] = max(rd_floor, min(350.0, rd_new))
            elif side == "def_peri":
                pl["D_peri_mu"] += delta
                rd_new = pl["D_peri_rd"] * (0.99 + 0.02 * min(abs_err, 0.15))
                pl["D_peri_rd"] = max(rd_floor, min(350.0, rd_new))
            else:
                pl["D_mu"] += delta
                rd_new = pl["D_rd"] * (0.99 + 0.02 * min(abs_err, 0.15))
                pl["D_rd"] = max(rd_floor, min(350.0, rd_new))
                pl["D_sigma"] = min(0.15, pl["D_sigma"] * (1.0 + 0.05 * min(abs_err, 0.2)))

            pl["Possessions"] += poss
            self.player_games[p] += 1

    def _weight_stint(self, poss, period, margin, season_progress, stint_ctx=None):
        weight = poss
        gt_w = self.cfg.get("garbage_time_weight", 0.5)
        if stint_ctx and stint_ctx.get("garbage"):
            weight *= gt_w
        elif period >= 4 and abs(margin) >= 15:
            weight *= gt_w
        if abs(margin) >= 25:
            weight *= gt_w
        weight *= (0.7 + 0.3 * season_progress)
        return weight

    def predict_game_margin(self, home_ids, away_ids, possessions,
                            weights_home=None, weights_away=None, is_home=True):
        if weights_home:
            ho, hd, _ = self.weighted_lineup_stats(weights_home)
        else:
            ho, hd, _ = self.lineup_stats(home_ids)
        if weights_away:
            ao, ad, _ = self.weighted_lineup_stats(weights_away)
        else:
            ao, ad, _ = self.lineup_stats(away_ids)

        scaling = self.cfg["ELO_SCALING_FACTOR"]
        hb = self.cfg["HOME_PPP_BOOST"] if is_home else -self.cfg["HOME_PPP_BOOST"]
        ppp_h = self.league_xppp + hb + (ho - ad) / scaling
        ppp_a = self.league_xppp - hb + (ao - hd) / scaling
        margin = (ppp_h - ppp_a) * float(possessions or 100.0)

        ho_rd, hd_rd = self.lineup_uncertainty(home_ids)
        ao_rd, ad_rd = self.lineup_uncertainty(away_ids)
        unc = (ho_rd + hd_rd + ao_rd + ad_rd) / 4.0
        return float(margin), float(unc)

    def implied_total(self, home_ids, away_ids, possessions,
                      weights_home=None, weights_away=None, league_avg_total=225.0):
        """Elo-pace implied game total."""
        if weights_home:
            ho, hd, _ = self.weighted_lineup_stats(weights_home)
        else:
            ho, hd, _ = self.lineup_stats(home_ids)
        if weights_away:
            ao, ad, _ = self.weighted_lineup_stats(weights_away)
        else:
            ao, ad, _ = self.lineup_stats(away_ids)
        scaling = self.cfg["ELO_SCALING_FACTOR"]
        hb = self.cfg["HOME_PPP_BOOST"]
        ppp_h = self.league_xppp + hb + (ho - ad) / scaling
        ppp_a = self.league_xppp - hb + (ao - hd) / scaling
        poss = float(possessions or 100.0)
        return float((ppp_h + ppp_a) * poss)

    def save_state(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump({"players": self.players, "player_games": dict(self.player_games), "cfg": self.cfg}, f)

    @classmethod
    def load_state(cls, path, league_xppp=1.10):
        import pickle
        with open(path, "rb") as f:
            data = pickle.load(f)
        tracker = cls(config=data.get("cfg"), league_xppp=league_xppp)
        tracker.players = data["players"]
        tracker.player_games = defaultdict(int, data.get("player_games", {}))
        return tracker


In [ ]:
# ── module: hierarchical ─────────────────────────────────────────────────────
"""Hierarchical possession engine."""
import itertools
from collections import defaultdict

import numpy as np



class HierarchicalPossessionEngine:
    def __init__(self, w1=0.50, w2=0.25, w3=0.15, w5=0.10,
                 k_off=2.0, k_def=1.5, league_avg_rtg=DEFAULT_LEAGUE_RTG,
                 home_boost_rtg=1.5, update_mode="points", min_poss_w5=200):
        self.update_mode = update_mode  # "points" or "xppp"
        self.min_poss_w5 = min_poss_w5
        self._base_w5 = w5
        total = w1 + w2 + w3 + w5
        self.W = {1: w1/total, 2: w2/total, 3: w3/total, 5: w5/total}
        self._w1, self._w2, self._w3 = w1, w2, w3
        self._combo_poss = defaultdict(float)

        self.K_off = {1: k_off, 2: k_off*0.50, 3: k_off*0.25, 5: k_off*0.10}
        self.K_def = {1: k_def, 2: k_def*0.50, 3: k_def*0.25, 5: k_def*0.10}

        self.lg = league_avg_rtg
        self.home_boost_rtg = home_boost_rtg
        self.off = defaultdict(float)
        self.dff = defaultdict(float)

    def _combos(self, lineup):
        ids = sorted(int(x) for x in lineup if x is not None and str(x) != "nan")
        return {
            1: [(p,) for p in ids],
            2: list(itertools.combinations(ids, 2)),
            3: list(itertools.combinations(ids, 3)),
            5: [tuple(ids)] if len(ids) == 5 else [],
        }

    def _mean(self, combos, store, single_combos=None):
        if not combos:
            return 0.0
        raw = float(np.mean([store[c] for c in combos]))
        if single_combos and len(combos) < 3:
            single = float(np.mean([store[c] for c in single_combos])) if single_combos else raw
            w = len(combos) / 3.0
            return w * raw + (1.0 - w) * single
        return raw

    def _dynamic_weights(self):
        """Increase 5-man weight when combo samples are rich."""
        w5 = self._base_w5
        if self._combo_poss:
            max5 = max((v for k, v in self._combo_poss.items() if len(k) == 5), default=0)
            if max5 >= self.min_poss_w5:
                w5 = min(0.25, self._base_w5 * 1.8)
        total = self._w1 + self._w2 + self._w3 + w5
        return {1: self._w1/total, 2: self._w2/total, 3: self._w3/total, 5: w5/total}

    def lineup_rating(self, lineup):
        cb = self._combos(lineup)
        if not cb[1]: return 0.0, 0.0
        W = self._dynamic_weights()
        singles = cb[1]
        off = sum(W[l] * self._mean(cb[l], self.off, singles) for l in W)
        dff = sum(W[l] * self._mean(cb[l], self.dff, singles) for l in W)
        return off, dff

    def predict_pts(self, off_ln, def_ln, possessions):
        cb_off = self._combos(off_ln); cb_def = self._combos(def_ln)
        W = self._dynamic_weights()
        singles_off, singles_def = cb_off[1], cb_def[1]
        os = sum(W[l] * self._mean(cb_off[l], self.off, singles_off) for l in W)
        ds = sum(W[l] * self._mean(cb_def[l], self.dff, singles_def) for l in W)
        o2 = sum(W[l] * self._mean(cb_def[l], self.off, singles_def) for l in W)
        d2 = sum(W[l] * self._mean(cb_off[l], self.dff, singles_off) for l in W)
        p = possessions / 100.0
        return (self.lg + os - ds + self.home_boost_rtg) * p, (self.lg + o2 - d2) * p, cb_off, cb_def

    def update(self, off_ln, def_ln, pts_off, pts_def, possessions, xpts_off=None, xpts_def=None):
        if possessions <= 0: return
        if self.update_mode == "xppp" and xpts_off is not None:
            pts_off = xpts_off
            pts_def = xpts_def if xpts_def is not None else pts_def
        xo, xd, cb_off, cb_def = self.predict_pts(off_ln, def_ln, possessions)

        eo, ed = pts_off - xo, pts_def - xd

        W = self._dynamic_weights()
        for l in [1, 2, 3, 5]:
            ko, kd = self.K_off[l], self.K_def[l]
            for c in cb_off[l]:
                self.off[c] += ko * eo
                self.dff[c] -= kd * ed
                self._combo_poss[c] += possessions
            for c in cb_def[l]:
                self.off[c] += ko * ed
                self.dff[c] -= kd * eo
                self._combo_poss[c] += possessions

    def offseason_revert(self):
        for store in (self.off, self.dff):
            for k in list(store): store[k] *= (1.0 - OFFSEASON_REVERSION)


    def save_state(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump(self.__dict__, f)

    @classmethod
    def load_state(cls, path):
        import pickle
        with open(path, "rb") as f:
            data = pickle.load(f)
        obj = cls.__new__(cls)
        obj.__dict__.update(data)
        return obj


In [ ]:
# ── module: trackers ─────────────────────────────────────────────────────────
"""Rolling team trackers."""

# Cell 8b – Team xPPP Tracker (rolling 40 games, previous season weight 0.5)
from collections import defaultdict, deque

class TeamXpppTracker:
    """
    Maintains rolling offensive and defensive xPPP (expected points per possession)
    for each team, using a weighted window:
      - last 40 games: weight 1.0
      - previous season games: weight 0.5
    """
    def __init__(self, window_size=40, prev_season_weight=0.5):
        self.window_size = window_size
        self.prev_season_weight = prev_season_weight
        # For each team, store deque of (game_date, season, off_xppp, def_xppp)
        self.history = defaultdict(lambda: deque(maxlen=window_size))

    def update(self, team, game_date, season, off_xppp, def_xppp):
        """
        Add a game's team-level offensive and defensive xPPP.
        off_xppp = total_offensive_xPoints / total_offensive_possessions
        def_xppp = total_defensive_xPoints_allowed / total_defensive_possessions
        """
        self.history[team].append((game_date, season, off_xppp, def_xppp))

    def get_rolling_xppp(self, team, current_season, current_date):
        """
        Returns (rolling_off_xppp, rolling_def_xppp) using weighted average.
        Games in current season: weight 1.0.
        Games in previous season: weight prev_season_weight.
        Only games before current_date are considered.
        """
        if team not in self.history:
            return DEFAULT_LEAGUE_XPPP, DEFAULT_LEAGUE_XPPP

        total_weight = 0.0
        sum_off = 0.0
        sum_def = 0.0

        for (game_date, season, off_xppp, def_xppp) in self.history[team]:
            if game_date >= current_date:
                continue   # never use future data
            if season == current_season:
                w = 1.0
            else:
                w = self.prev_season_weight   # previous season
            total_weight += w
            sum_off += off_xppp * w
            sum_def += def_xppp * w

        if total_weight == 0:
            return DEFAULT_LEAGUE_XPPP, DEFAULT_LEAGUE_XPPP

        return sum_off / total_weight, sum_def / total_weight

# Cell 11 – PaceTracker (updated)

import numpy as np
from collections import deque

class PaceTracker:
    """Rolling possession (pace) tracker.

    Tracks each team's OWN per-team possessions (~100) rather than the shared
    whole-game total, so ``get_team_pace`` and the ``pace_diff`` feature reflect
    a team's true tempo. ``get_expected_pace`` returns the expected *total* game
    possessions (~200) so the downstream scale (total head, engine-implied
    margins) is unchanged.
    """
    LEAGUE_TEAM_PACE = 100.0  # per-team possessions baseline

    def __init__(self, team_window=10, league_window=150, per_team=True):
        self.team_window = team_window
        self.league_window = league_window
        # per_team=True: track each team's own possessions (improved default).
        # per_team=False: legacy behaviour (store whole-game totals for both).
        self.per_team = per_team
        # team -> deque of that team's own per-team possessions
        self.team_history = {}
        # league pool of per-team possessions
        self.league_history = deque(maxlen=league_window)

    def _league_pace(self):
        return float(np.mean(self.league_history)) if self.league_history else self.LEAGUE_TEAM_PACE

    def get_team_pace(self, team):
        """Average per-team possessions for a team over its rolling window."""
        hist = self.team_history.get(team)
        if hist and len(hist) > 0:
            return float(np.mean(hist))
        return self.LEAGUE_TEAM_PACE

    def get_expected_pace(self, home_team, away_team):
        """Expected TOTAL game possessions (~200).

        Interaction estimate: per-team game pace ≈ home_pace + away_pace -
        league_pace (so two fast teams play faster, two slow teams slower);
        total possessions ≈ 2 × that.
        """
        h_pace = self.get_team_pace(home_team)
        a_pace = self.get_team_pace(away_team)
        lg_pace = self._league_pace()
        combined = (h_pace + a_pace) - lg_pace
        # per_team stores ~100-scale paces -> double for total game possessions;
        # legacy stores ~200-scale totals -> already total.
        return max(85.0, 2.0 * combined if self.per_team else combined)

    def update_pace(self, home_team, away_team, home_possessions, away_possessions=None):
        """Record possessions for the game.

        per_team=True: store each team's own possessions.
        per_team=False (legacy): store the whole-game total for both teams.
        Back-compatible: if ``away_possessions`` is omitted, treat the single
        value as a per-team estimate (legacy total / 2) for both teams.
        """
        if away_possessions is None:
            home_possessions = away_possessions = float(home_possessions) / 2.0
        home_possessions = float(home_possessions)
        away_possessions = float(away_possessions)
        if not self.per_team:
            total = home_possessions + away_possessions
            home_possessions = away_possessions = total
        if home_team not in self.team_history:
            self.team_history[home_team] = deque(maxlen=self.team_window)
        if away_team not in self.team_history:
            self.team_history[away_team] = deque(maxlen=self.team_window)
        self.team_history[home_team].append(home_possessions)
        self.team_history[away_team].append(away_possessions)
        self.league_history.append(home_possessions)
        self.league_history.append(away_possessions)

    def save_state(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump({
                "team_window": self.team_window,
                "league_window": self.league_window,
                "team_history": {k: list(v) for k, v in self.team_history.items()},
                "league_history": list(self.league_history),
            }, f)

    @classmethod
    def load_state(cls, path):
        import pickle
        from collections import deque
        with open(path, "rb") as f:
            data = pickle.load(f)
        obj = cls(team_window=data["team_window"], league_window=data["league_window"])
        obj.team_history = {k: deque(v, maxlen=obj.team_window) for k, v in data["team_history"].items()}
        obj.league_history = deque(data["league_history"], maxlen=obj.league_window)
        return obj


class RotationLineupTracker:
    """Rolling player possession share per team for rotation-weighted lineups."""

    def __init__(self, window_games=10, top_n=8):
        self.window_games = window_games
        self.top_n = top_n
        self.history = defaultdict(lambda: deque(maxlen=window_games))

    def update_game(self, team, player_ids, possessions_per_player):
        """Record per-player possessions from one game."""
        self.history[team].append(dict(zip(player_ids, possessions_per_player)))

    def expected_weights(self, team, fallback_ids=None):
        """Return list of (player_id, weight) for top rotation players."""
        if team not in self.history or not self.history[team]:
            if fallback_ids:
                n = max(len(fallback_ids), 1)
                return [(str(p), 1.0 / n) for p in fallback_ids if p]
            return []
        acc = defaultdict(float)
        for game in self.history[team]:
            for pid, poss in game.items():
                acc[str(pid)] += float(poss)
        total = sum(acc.values())
        if total <= 0:
            if fallback_ids:
                n = max(len(fallback_ids), 1)
                return [(str(p), 1.0 / n) for p in fallback_ids if p]
            return []
        ranked = sorted(acc.items(), key=lambda kv: kv[1], reverse=True)[: self.top_n]
        return [(pid, w / total) for pid, w in ranked]

    def save_state(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump({
                "window_games": self.window_games,
                "top_n": self.top_n,
                "history": {k: list(v) for k, v in self.history.items()},
            }, f)

    @classmethod
    def load_state(cls, path):
        import pickle
        with open(path, "rb") as f:
            data = pickle.load(f)
        obj = cls(window_games=data["window_games"], top_n=data["top_n"])
        for k, v in data["history"].items():
            obj.history[k] = deque(v, maxlen=obj.window_games)
        return obj


In [ ]:
# ── module: teamstats ────────────────────────────────────────────────────────
"""Walk-forward team box-score form tracking (no leakage).

Adds non-leaky team-level rolling features on top of the Elo/xPPP system:

  - offensive / defensive rating (points per 100 possessions)
  - 3PT makes per game and 3PT%
  - blocks per game
  - forced turnovers per game

All rolling values use only games strictly before the current game date.
"""
from __future__ import annotations

from collections import defaultdict, deque

import numpy as np
import pandas as pd

# League-average fallbacks (used before a team has any history)
_DEFAULT_FORM = {
    "off_rtg": 113.0,
    "def_rtg": 113.0,
    "fg3m": 12.0,
    "fg3pct": 0.36,
    "blk": 5.0,
    "ftov": 13.5,
    # Four-factors + extra defensive/pressure metrics
    "efg": 0.53,        # effective FG%
    "oreb_pct": 0.25,   # offensive rebound rate
    "dreb_pct": 0.75,   # defensive rebound rate
    "tov_pct": 13.0,    # own turnovers per 100 possessions
    "ftr": 0.25,        # free-throw rate (FTA / FGA)
    "stl": 7.5,         # steals per game
    "fdrawn": 20.0,     # fouls drawn per game
    "luck_adj_off": 0.0,
    "pts_for": 110.0,
    "pts_against": 110.0,
    "win": 0.5,
    "pythag_win_pct": 0.5,
    "actual_win_pct": 0.5,
    "pythag_residual": 0.0,
}

# Multi-horizon rolling windows for form features (leak-free via get_window).
FORM_ROLL_WINDOWS = (3, 5, 10, 20)
FORM_MULTIWINDOW_KEYS = (
    "off_rtg", "def_rtg", "efg", "tov_pct", "oreb_pct", "dreb_pct", "ftr", "fg3pct",
)


def pythagorean_win_pct(pts_for: float, pts_against: float, exponent: float | None = None) -> float:
    """Expected win rate from points for/against (Morey NBA exponent default)."""

    exp = float(PYTHAGOREAN_EXPONENT if exponent is None else exponent)
    pf = max(float(pts_for), 0.0)
    pa = max(float(pts_against), 0.0)
    if pf <= 0 and pa <= 0:
        return 0.5
    if pf <= 0:
        return 0.0
    if pa <= 0:
        return 1.0
    num = pf ** exp
    den = num + pa ** exp
    return float(num / den) if den > 0 else 0.5


def precompute_game_team_stats(stints_df: pd.DataFrame) -> dict:
    """
    Aggregate stint rows to one record per GAME_ID in a single vectorized pass.

    Returns dict: GAME_ID -> {act_h, act_a, tot_poss, home_poss, away_poss,
                              home_xpts, away_xpts, home_3pm, away_3pm,
                              home_3pa, away_3pa, home_blk, away_blk,
                              home_ftov, away_ftov}
    This replaces per-stint Python loops for game aggregation (big speedup).
    """
    sum_cols = {
        "home_pts": "act_h",
        "away_pts": "act_a",
        "possessions": "tot_poss",
        "home_poss": "home_poss",
        "away_poss": "away_poss",
        "home_xpts": "home_xpts",
        "away_xpts": "away_xpts",
        "home_3pm": "home_3pm",
        "away_3pm": "away_3pm",
        "home_3pa": "home_3pa",
        "away_3pa": "away_3pa",
        "home_blks": "home_blk",
        "away_blks": "away_blk",
        "home_tovs_forced": "home_ftov",
        "away_tovs_forced": "away_ftov",
        # Four-factors raw counts
        "home_oreb": "home_oreb",
        "away_oreb": "away_oreb",
        "home_dreb": "home_dreb",
        "away_dreb": "away_dreb",
        "home_fgm": "home_fgm",
        "away_fgm": "away_fgm",
        "home_fga": "home_fga",
        "away_fga": "away_fga",
        "home_tov": "home_tov",
        "away_tov": "away_tov",
        "home_fta": "home_fta",
        "away_fta": "away_fta",
        "home_fouls_drawn": "home_fdrawn",
        "away_fouls_drawn": "away_fdrawn",
        "home_stl": "home_stl",
        "away_stl": "away_stl",
        "home_xefg_sum": "home_xefg_sum",
        "away_xefg_sum": "away_xefg_sum",
        "home_rim_fga": "home_rim_fga",
        "away_rim_fga": "away_rim_fga",
        "home_three_fga": "home_three_fga",
        "away_three_fga": "away_three_fga",
        "home_shot_dist_sum": "home_shot_dist_sum",
        "away_shot_dist_sum": "away_shot_dist_sum",
    }
    present = {src: dst for src, dst in sum_cols.items() if src in stints_df.columns}
    agg = stints_df.groupby("GAME_ID", sort=False)[list(present.keys())].sum()
    agg = agg.rename(columns=present)

    # Task 014: when canonical, independently-verified final scores (Task
    # 006/007) have been attached to the stints frame, they override the
    # stint-summed act_h/act_a for *labels*. Weighted stint statistics
    # (possessions, xPoints, four-factors, etc.) are untouched and remain the
    # only source for ratings/features. Games without a canonical final
    # (e.g. seasons not yet covered by the canonical builder) keep the
    # stint-summed values as a documented fallback — never silently dropped.
    if "canonical_home_pts" in stints_df.columns and "canonical_away_pts" in stints_df.columns:
        canon = stints_df.groupby("GAME_ID", sort=False)[
            ["canonical_home_pts", "canonical_away_pts"]
        ].first()
        has_canon = canon["canonical_home_pts"].notna() & canon["canonical_away_pts"].notna()
        agg.loc[has_canon.index[has_canon], "act_h"] = canon.loc[has_canon, "canonical_home_pts"]
        agg.loc[has_canon.index[has_canon], "act_a"] = canon.loc[has_canon, "canonical_away_pts"]

    return agg.to_dict("index")


def team_game_form(game_stats: dict, side: str) -> dict:
    """Convert one game's aggregated stats into a team's box-form dict for `side`."""
    g = game_stats
    if side == "home":
        pts, opp_pts = g.get("act_h", 0.0), g.get("act_a", 0.0)
        poss = g.get("home_poss", 0.0)
        opp_poss = g.get("away_poss", 0.0)
        fg3m, fg3a = g.get("home_3pm", 0.0), g.get("home_3pa", 0.0)
        blk, ftov = g.get("home_blk", 0.0), g.get("home_ftov", 0.0)
        oreb, dreb = g.get("home_oreb", 0.0), g.get("home_dreb", 0.0)
        opp_oreb, opp_dreb = g.get("away_oreb", 0.0), g.get("away_dreb", 0.0)
        fgm, fga = g.get("home_fgm", 0.0), g.get("home_fga", 0.0)
        tov, fta = g.get("home_tov", 0.0), g.get("home_fta", 0.0)
        stl, fdrawn = g.get("home_stl", 0.0), g.get("home_fdrawn", 0.0)
    else:
        pts, opp_pts = g.get("act_a", 0.0), g.get("act_h", 0.0)
        poss = g.get("away_poss", 0.0)
        opp_poss = g.get("home_poss", 0.0)
        fg3m, fg3a = g.get("away_3pm", 0.0), g.get("away_3pa", 0.0)
        blk, ftov = g.get("away_blk", 0.0), g.get("away_ftov", 0.0)
        oreb, dreb = g.get("away_oreb", 0.0), g.get("away_dreb", 0.0)
        opp_oreb, opp_dreb = g.get("home_oreb", 0.0), g.get("home_dreb", 0.0)
        fgm, fga = g.get("away_fgm", 0.0), g.get("away_fga", 0.0)
        tov, fta = g.get("away_tov", 0.0), g.get("away_fta", 0.0)
        stl, fdrawn = g.get("away_stl", 0.0), g.get("away_fdrawn", 0.0)

    poss = poss if poss > 0 else g.get("tot_poss", 0.0) / 2.0
    opp_poss = opp_poss if opp_poss > 0 else g.get("tot_poss", 0.0) / 2.0
    xpts = g.get("home_xpts", pts) if side == "home" else g.get("away_xpts", pts)
    luck_adj = 100.0 * (xpts / poss - pts / poss) if poss > 0 else 0.0
    won = 1.0 if pts > opp_pts else 0.0
    return {
        "off_rtg": 100.0 * pts / poss if poss > 0 else _DEFAULT_FORM["off_rtg"],
        "def_rtg": 100.0 * opp_pts / opp_poss if opp_poss > 0 else _DEFAULT_FORM["def_rtg"],
        "pts_for": float(pts),
        "pts_against": float(opp_pts),
        "win": won,
        "fg3m": fg3m,
        "fg3pct": fg3m / fg3a if fg3a > 0 else _DEFAULT_FORM["fg3pct"],
        "blk": blk,
        "ftov": ftov,
        "efg": (fgm + 0.5 * fg3m) / fga if fga > 0 else _DEFAULT_FORM["efg"],
        "oreb_pct": oreb / (oreb + opp_dreb) if (oreb + opp_dreb) > 0 else _DEFAULT_FORM["oreb_pct"],
        "dreb_pct": dreb / (dreb + opp_oreb) if (dreb + opp_oreb) > 0 else _DEFAULT_FORM["dreb_pct"],
        "tov_pct": 100.0 * tov / poss if poss > 0 else _DEFAULT_FORM["tov_pct"],
        "ftr": fta / fga if fga > 0 else _DEFAULT_FORM["ftr"],
        "stl": stl,
        "fdrawn": fdrawn,
        "luck_adj_off": luck_adj,
    }


class TeamFormTracker:
    """Rolling, walk-forward team box-score form.

    Window of recent games (current season weight 1.0, previous season weighted).
    """

    def __init__(self, window=15, prev_season_weight=0.4):
        self.window = window
        self.prev_season_weight = prev_season_weight
        self.history = defaultdict(lambda: deque(maxlen=window * 2))
        # Rolling league pool of (off_rtg, def_rtg) for opponent/era adjustment.
        self.league = deque(maxlen=window * 30)

    def update(self, team, game_date, season, form: dict, opp_off=None, opp_def=None, weight: float = 1.0):
        # Store the opponent's pre-game ratings so `get` can opponent-adjust.
        w = max(0.0, float(weight))
        if w <= 0:
            return
        self.history[team].append((game_date, season, form, opp_off, opp_def, w))
        self.league.append((float(form.get("off_rtg", _DEFAULT_FORM["off_rtg"])),
                            float(form.get("def_rtg", _DEFAULT_FORM["def_rtg"]))))

    def league_avg(self):
        if not self.league:
            return _DEFAULT_FORM["off_rtg"], _DEFAULT_FORM["def_rtg"]
        arr = np.asarray(self.league, dtype=float)
        return float(arr[:, 0].mean()), float(arr[:, 1].mean())

    def get(self, team, current_season, current_date):
        base = dict(_DEFAULT_FORM)
        lg_off, lg_def = self.league_avg()
        base["off_rtg_adj"] = base["off_rtg"]
        base["def_rtg_adj"] = base["def_rtg"]
        if team not in self.history:
            return base
        acc = defaultdict(float)
        opp_off_sum = opp_def_sum = 0.0
        wsum = 0.0
        pts_for_sum = pts_against_sum = 0.0
        wins_sum = 0.0

        for gd, season, form, oo, od, *rest in self.history[team]:
            w_extra = rest[0] if rest else 1.0
            if current_date is not None and gd is not None and gd >= current_date:
                continue
            w = (1.0 if season == current_season else self.prev_season_weight) * float(w_extra)
            for k, v in form.items():
                if k in ("pts_for", "pts_against", "win"):
                    continue
                acc[k] += v * w
            pts_for_sum += float(form.get("pts_for", 0.0)) * w
            pts_against_sum += float(form.get("pts_against", 0.0)) * w
            wins_sum += float(form.get("win", 0.0)) * w
            # opponent context (fall back to league avg when missing)
            opp_off_sum += (oo if oo is not None else lg_off) * w
            opp_def_sum += (od if od is not None else lg_def) * w
            wsum += w
        if wsum == 0:
            return base
        out = {k: acc[k] / wsum for k in _DEFAULT_FORM if k not in ("pts_for", "pts_against", "win", "pythag_win_pct", "actual_win_pct", "pythag_residual")}
        mean_opp_off = opp_off_sum / wsum
        mean_opp_def = opp_def_sum / wsum
        # Schedule-adjust: credit scoring vs strong defenses / stops vs strong offenses.
        out["off_rtg_adj"] = out["off_rtg"] + (lg_def - mean_opp_def)
        out["def_rtg_adj"] = out["def_rtg"] - (mean_opp_off - lg_off)
        if wsum >= float(PYTHAG_MIN_GAMES):
            out["actual_win_pct"] = float(wins_sum / wsum)
            out["pythag_win_pct"] = pythagorean_win_pct(pts_for_sum, pts_against_sum)
            out["pythag_residual"] = out["actual_win_pct"] - out["pythag_win_pct"]
        else:
            out["actual_win_pct"] = base["actual_win_pct"]
            out["pythag_win_pct"] = base["pythag_win_pct"]
            out["pythag_residual"] = base["pythag_residual"]
        return out

    def get_window(self, team, current_season, current_date, n_games: int):
        """Mean form over the most recent n_games before current_date (leak-free)."""
        base = dict(_DEFAULT_FORM)
        if team not in self.history or n_games <= 0:
            return base
        rows = []
        for gd, season, form, oo, od, *rest in self.history[team]:
            if current_date is not None and gd is not None and gd >= current_date:
                continue
            rows.append((gd, season, form, oo, od, rest[0] if rest else 1.0))
        if not rows:
            return base
        # Most recent first
        rows = sorted(rows, key=lambda x: x[0] if x[0] is not None else 0, reverse=True)[: int(n_games)]
        acc = defaultdict(float)
        wsum = 0.0
        for gd, season, form, oo, od, w_extra in rows:
            w = (1.0 if season == current_season else self.prev_season_weight) * float(w_extra)
            for k, v in form.items():
                if k in ("pts_for", "pts_against", "win"):
                    continue
                if k in _DEFAULT_FORM:
                    acc[k] += float(v) * w
            wsum += w
        if wsum <= 0:
            return base
        out = dict(base)
        for k in _DEFAULT_FORM:
            if k in ("pts_for", "pts_against", "win", "pythag_win_pct", "actual_win_pct", "pythag_residual"):
                continue
            if k in acc:
                out[k] = acc[k] / wsum
        return out

    def multi_window_feature_dict(self, home_team, away_team, current_season, current_date,
                                   windows=None) -> dict:
        """Emit roll_N suffix features for multi-horizon form (3/5/10/20 + season)."""
        windows = windows or FORM_ROLL_WINDOWS
        keys = FORM_MULTIWINDOW_KEYS
        out = {}
        # Season ≈ full tracker window (existing get)
        h_season = self.get(home_team, current_season, current_date)
        a_season = self.get(away_team, current_season, current_date)
        for k in keys:
            hk, ak = f"h_{k}", f"a_{k}"
            out[f"{hk}_roll_season"] = float(h_season.get(k, _DEFAULT_FORM.get(k, 0.0)))
            out[f"{ak}_roll_season"] = float(a_season.get(k, _DEFAULT_FORM.get(k, 0.0)))
            out[f"{k}_diff_roll_season"] = out[f"{hk}_roll_season"] - out[f"{ak}_roll_season"]
        for n in windows:
            hf = self.get_window(home_team, current_season, current_date, n)
            af = self.get_window(away_team, current_season, current_date, n)
            suf = f"roll_{n}"
            for k in keys:
                hk, ak = f"h_{k}", f"a_{k}"
                out[f"{hk}_{suf}"] = float(hf.get(k, _DEFAULT_FORM.get(k, 0.0)))
                out[f"{ak}_{suf}"] = float(af.get(k, _DEFAULT_FORM.get(k, 0.0)))
                out[f"{k}_diff_{suf}"] = out[f"{hk}_{suf}"] - out[f"{ak}_{suf}"]
            # Recent scoring variance proxy from pts_for if present in history
            h_pts = []
            a_pts = []
            for gd, season, form, *_rest in list(self.history.get(home_team, [])):
                if current_date is not None and gd is not None and gd >= current_date:
                    continue
                h_pts.append(float(form.get("pts_for", 0.0)))
            for gd, season, form, *_rest in list(self.history.get(away_team, [])):
                if current_date is not None and gd is not None and gd >= current_date:
                    continue
                a_pts.append(float(form.get("pts_for", 0.0)))
            h_pts = h_pts[-n:] if h_pts else []
            a_pts = a_pts[-n:] if a_pts else []
            out[f"h_pts_std_{suf}"] = float(np.std(h_pts)) if len(h_pts) >= 2 else 0.0
            out[f"a_pts_std_{suf}"] = float(np.std(a_pts)) if len(a_pts) >= 2 else 0.0
        return out

    def save_state(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump({
                "window": self.window,
                "prev_season_weight": self.prev_season_weight,
                "history": {k: list(v) for k, v in self.history.items()},
                "league": list(self.league),
            }, f)

    @classmethod
    def load_state(cls, path):
        import pickle
        with open(path, "rb") as f:
            data = pickle.load(f)
        obj = cls(window=data["window"], prev_season_weight=data["prev_season_weight"])
        for k, v in data["history"].items():
            obj.history[k] = deque(v, maxlen=obj.window * 2)
        obj.league = deque(data.get("league", []), maxlen=obj.window * 30)
        return obj


def form_feature_dict(h_form: dict, a_form: dict) -> dict:
    """Build the model-facing team-form feature block from two team form dicts."""
    def _g(d, k):
        return d.get(k, _DEFAULT_FORM[k])
    return {
        "h_off_rtg": h_form["off_rtg"],
        "h_def_rtg": h_form["def_rtg"],
        "a_off_rtg": a_form["off_rtg"],
        "a_def_rtg": a_form["def_rtg"],
        # Matchup nets: home offense vs away defense, etc.
        "off_rtg_net": (h_form["off_rtg"] - a_form["def_rtg"]) - (a_form["off_rtg"] - h_form["def_rtg"]),
        "h_fg3m": h_form["fg3m"],
        "a_fg3m": a_form["fg3m"],
        "fg3m_diff": h_form["fg3m"] - a_form["fg3m"],
        "h_fg3pct": h_form["fg3pct"],
        "a_fg3pct": a_form["fg3pct"],
        "fg3pct_diff": h_form["fg3pct"] - a_form["fg3pct"],
        "h_blk": h_form["blk"],
        "a_blk": a_form["blk"],
        "blk_diff": h_form["blk"] - a_form["blk"],
        "h_ftov": h_form["ftov"],
        "a_ftov": a_form["ftov"],
        "ftov_diff": h_form["ftov"] - a_form["ftov"],
        # ── Four Factors + extra defensive/pressure metrics ──
        "h_efg": _g(h_form, "efg"), "a_efg": _g(a_form, "efg"),
        "efg_diff": _g(h_form, "efg") - _g(a_form, "efg"),
        "h_oreb_pct": _g(h_form, "oreb_pct"), "a_oreb_pct": _g(a_form, "oreb_pct"),
        "oreb_pct_diff": _g(h_form, "oreb_pct") - _g(a_form, "oreb_pct"),
        "h_dreb_pct": _g(h_form, "dreb_pct"), "a_dreb_pct": _g(a_form, "dreb_pct"),
        "dreb_pct_diff": _g(h_form, "dreb_pct") - _g(a_form, "dreb_pct"),
        "h_tov_pct": _g(h_form, "tov_pct"), "a_tov_pct": _g(a_form, "tov_pct"),
        "tov_pct_diff": _g(h_form, "tov_pct") - _g(a_form, "tov_pct"),
        "h_ftr": _g(h_form, "ftr"), "a_ftr": _g(a_form, "ftr"),
        "ftr_diff": _g(h_form, "ftr") - _g(a_form, "ftr"),
        "h_stl": _g(h_form, "stl"), "a_stl": _g(a_form, "stl"),
        "stl_diff": _g(h_form, "stl") - _g(a_form, "stl"),
        "h_fdrawn": _g(h_form, "fdrawn"), "a_fdrawn": _g(a_form, "fdrawn"),
        "fdrawn_diff": _g(h_form, "fdrawn") - _g(a_form, "fdrawn"),
        # ── Opponent/schedule-adjusted ratings ──
        "h_off_rtg_adj": h_form.get("off_rtg_adj", h_form["off_rtg"]),
        "a_off_rtg_adj": a_form.get("off_rtg_adj", a_form["off_rtg"]),
        "h_def_rtg_adj": h_form.get("def_rtg_adj", h_form["def_rtg"]),
        "a_def_rtg_adj": a_form.get("def_rtg_adj", a_form["def_rtg"]),
        # Adjusted matchup net: home off vs away def minus away off vs home def.
        "off_rtg_adj_net": (
            (h_form.get("off_rtg_adj", h_form["off_rtg"]) - a_form.get("def_rtg_adj", a_form["def_rtg"]))
            - (a_form.get("off_rtg_adj", a_form["off_rtg"]) - h_form.get("def_rtg_adj", h_form["def_rtg"]))
        ),
        "h_luck_adj_off": _g(h_form, "luck_adj_off"),
        "a_luck_adj_off": _g(a_form, "luck_adj_off"),
        "luck_adj_diff": _g(h_form, "luck_adj_off") - _g(a_form, "luck_adj_off"),
        # ── Pythagorean expectation (rolling, pre-game) ──
        "h_pythag_win_pct": _g(h_form, "pythag_win_pct"),
        "a_pythag_win_pct": _g(a_form, "pythag_win_pct"),
        "pythag_win_pct_diff": _g(h_form, "pythag_win_pct") - _g(a_form, "pythag_win_pct"),
        "h_pythag_residual": _g(h_form, "pythag_residual"),
        "a_pythag_residual": _g(a_form, "pythag_residual"),
        "pythag_residual_diff": _g(h_form, "pythag_residual") - _g(a_form, "pythag_residual"),
    }


# Original team-form features (pre-Four-Factors); kept separate so the ablation
# harness can measure the incremental value of the new box-score block.
FORM_FEATURE_COLS_BASE = [
    "h_off_rtg", "h_def_rtg", "a_off_rtg", "a_def_rtg", "off_rtg_net",
    "h_fg3m", "a_fg3m", "fg3m_diff",
    "h_fg3pct", "a_fg3pct", "fg3pct_diff",
    "h_blk", "a_blk", "blk_diff",
    "h_ftov", "a_ftov", "ftov_diff",
]

# New Four-Factors + defensive/pressure block.
FOUR_FACTOR_COLS = [
    "h_efg", "a_efg", "efg_diff",
    "h_oreb_pct", "a_oreb_pct", "oreb_pct_diff",
    "h_dreb_pct", "a_dreb_pct", "dreb_pct_diff",
    "h_tov_pct", "a_tov_pct", "tov_pct_diff",
    "h_ftr", "a_ftr", "ftr_diff",
    "h_stl", "a_stl", "stl_diff",
    "h_fdrawn", "a_fdrawn", "fdrawn_diff",
]

# Opponent/schedule-adjusted rating block.
OPP_ADJ_COLS = [
    "h_off_rtg_adj", "a_off_rtg_adj", "h_def_rtg_adj", "a_def_rtg_adj", "off_rtg_adj_net",
]

LUCK_COLS = ["h_luck_adj_off", "a_luck_adj_off", "luck_adj_diff"]

PYTHAG_COLS = [
    "h_pythag_win_pct", "a_pythag_win_pct", "pythag_win_pct_diff",
    "h_pythag_residual", "a_pythag_residual", "pythag_residual_diff",
]

FORM_FEATURE_COLS = FORM_FEATURE_COLS_BASE + FOUR_FACTOR_COLS + OPP_ADJ_COLS + LUCK_COLS + PYTHAG_COLS

# Columns produced by TeamFormTracker.multi_window_feature_dict
MULTIWINDOW_FORM_COLS = []
for _k in FORM_MULTIWINDOW_KEYS:
    MULTIWINDOW_FORM_COLS.extend([
        f"h_{_k}_roll_season", f"a_{_k}_roll_season", f"{_k}_diff_roll_season",
    ])
    for _n in FORM_ROLL_WINDOWS:
        MULTIWINDOW_FORM_COLS.extend([
            f"h_{_k}_roll_{_n}", f"a_{_k}_roll_{_n}", f"{_k}_diff_roll_{_n}",
        ])
for _n in FORM_ROLL_WINDOWS:
    MULTIWINDOW_FORM_COLS.extend([f"h_pts_std_roll_{_n}", f"a_pts_std_roll_{_n}"])


In [ ]:
# ── module: shot_quality ─────────────────────────────────────────────────────
"""Rolling pre-game shot-quality features (xEFG, rim rate, three rate)."""
from __future__ import annotations

from collections import defaultdict, deque

import numpy as np

_DEFAULT_SHOT_QUALITY = {
    "xefg": 0.54,
    "rim_rate": 0.30,
    "three_rate": 0.38,
    "avg_shot_distance": 14.0,
}


class ShotQualityTracker:
    """Strictly pre-game rolling shot diet / expected FG% by team."""

    def __init__(self, window: int = 15, prev_season_weight: float = 0.4):
        self.window = window
        self.prev_season_weight = prev_season_weight
        self._history = defaultdict(lambda: deque(maxlen=window))
        self._prior_season = defaultdict(dict)

    def _rates(self, rec: dict) -> dict:
        fga = float(rec.get("fga", 0) or 0)
        if fga < 1:
            return dict(_DEFAULT_SHOT_QUALITY)
        xefg_sum = float(rec.get("xefg_sum", 0) or 0)
        rim_fga = float(rec.get("rim_fga", 0) or 0)
        three_fga = float(rec.get("three_fga", 0) or 0)
        dist_sum = float(rec.get("shot_dist_sum", 0) or 0)
        return {
            "xefg": xefg_sum / fga,
            "rim_rate": rim_fga / fga,
            "three_rate": three_fga / fga,
            "avg_shot_distance": dist_sum / fga if dist_sum > 0 else _DEFAULT_SHOT_QUALITY["avg_shot_distance"],
        }

    def update(self, team: str, gdate, season: int, game_stats: dict) -> None:
        rec = {
            "fga": float(game_stats.get("fga", 0) or 0),
            "xefg_sum": float(game_stats.get("xefg_sum", 0) or 0),
            "rim_fga": float(game_stats.get("rim_fga", 0) or 0),
            "three_fga": float(game_stats.get("three_fga", 0) or 0),
            "shot_dist_sum": float(game_stats.get("shot_dist_sum", 0) or 0),
            "gdate": gdate,
            "season": season,
        }
        key = (team, season)
        self._history[key].append(rec)

    def _blend_prior(self, team: str, season: int, current: dict) -> dict:
        prior = self._prior_season.get((team, season))
        if not prior:
            return current
        w = self.prev_season_weight
        return {k: (1 - w) * current.get(k, _DEFAULT_SHOT_QUALITY[k]) + w * prior.get(k, _DEFAULT_SHOT_QUALITY[k])
                for k in _DEFAULT_SHOT_QUALITY}

    def on_season_boundary(self, team: str, old_season: int) -> None:
        key = (team, old_season)
        hist = self._history.get(key)
        if not hist:
            return
        totals = {"fga": 0.0, "xefg_sum": 0.0, "rim_fga": 0.0, "three_fga": 0.0, "shot_dist_sum": 0.0}
        for rec in hist:
            for k in totals:
                totals[k] += float(rec.get(k, 0) or 0)
        self._prior_season[(team, old_season + 1)] = self._rates(totals)
        self._history[key].clear()

    def get(self, team: str, season: int, gdate) -> dict:
        key = (team, season)
        hist = [r for r in self._history.get(key, []) if r.get("gdate") is not None and r["gdate"] < gdate]
        if not hist:
            prior = self._prior_season.get((team, season))
            return dict(prior or _DEFAULT_SHOT_QUALITY)
        totals = {"fga": 0.0, "xefg_sum": 0.0, "rim_fga": 0.0, "three_fga": 0.0, "shot_dist_sum": 0.0}
        for rec in hist[-self.window:]:
            for k in totals:
                totals[k] += float(rec.get(k, 0) or 0)
        current = self._rates(totals)
        return self._blend_prior(team, season, current)

    def feature_dict(self, home_team: str, away_team: str, season: int, gdate) -> dict:
        h = self.get(home_team, season, gdate)
        a = self.get(away_team, season, gdate)
        return {
            "h_xefg": h["xefg"],
            "a_xefg": a["xefg"],
            "h_rim_rate": h["rim_rate"],
            "a_rim_rate": a["rim_rate"],
            "h_three_rate": h["three_rate"],
            "a_three_rate": a["three_rate"],
            "shot_quality_edge": h["xefg"] - a["xefg"],
            "rim_rate_diff": h["rim_rate"] - a["rim_rate"],
            "three_rate_diff": h["three_rate"] - a["three_rate"],
            "avg_shot_distance_diff": h["avg_shot_distance"] - a["avg_shot_distance"],
        }


def shot_quality_stats_from_gs(gs: dict, side: str) -> dict:
    prefix = "home" if side == "home" else "away"
    return {
        "fga": float(gs.get(f"{prefix}_fga", 0) or 0),
        "xefg_sum": float(gs.get(f"{prefix}_xefg_sum", 0) or 0),
        "rim_fga": float(gs.get(f"{prefix}_rim_fga", 0) or 0),
        "three_fga": float(gs.get(f"{prefix}_three_fga", 0) or 0),
        "shot_dist_sum": float(gs.get(f"{prefix}_shot_dist_sum", 0) or 0),
    }


In [ ]:
# ── module: skellam ──────────────────────────────────────────────────────────
"""Skellam distribution cover probability from predicted scores."""
from __future__ import annotations

import numpy as np


def cover_prob_skellam(
    pred_home: float,
    pred_away: float,
    market_spread: float,
    direction: str = "Home",
) -> float:
    """P(bet side covers) using Poisson means = predicted scores."""
    mu1 = max(float(pred_home), 0.5)
    mu2 = max(float(pred_away), 0.5)
    spread = float(market_spread)
    # Home covers when actual_margin + market_spread > 0  =>  margin > -spread
    threshold = int(np.floor(-spread))
    try:
        from scipy.stats import skellam
        p_home = float(1.0 - skellam.cdf(threshold, mu1, mu2))
    except ImportError:
        diff_mean = mu1 - mu2
        diff_std = max(np.sqrt(mu1 + mu2), 1.0)
        z = (threshold + 0.5 - diff_mean) / diff_std
        p_home = float(1.0 - _norm_cdf(z))
    if direction == "Away":
        return 1.0 - p_home
    return p_home


def _norm_cdf(z: float) -> float:
    return 0.5 * (1.0 + float(np.tanh(z * 0.7978845608)))


In [ ]:
# ── module: market ───────────────────────────────────────────────────────────
"""Odds loading and betting helpers."""
from __future__ import annotations

from collections import deque

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ──────────────────────────────────────────────────────────────────────────────
# 1. LEAKAGE‑FREE ODDS RETRIEVAL (ONLY PAST DATES)
# ──────────────────────────────────────────────────────────────────────────────

_ABBR_MAP = {
    'Atlanta': 'ATL', 'Boston': 'BOS', 'Brooklyn': 'BKN', 'Charlotte': 'CHA',
    'Chicago': 'CHI', 'Cleveland': 'CLE', 'Dallas': 'DAL', 'Denver': 'DEN',
    'Detroit': 'DET', 'Golden State': 'GSW', 'Houston': 'HOU', 'Indiana': 'IND',
    'LA Clippers': 'LAC', 'Los Angeles Clippers': 'LAC', 'Clippers': 'LAC',
    'LA Lakers': 'LAL', 'Los Angeles Lakers': 'LAL', 'Lakers': 'LAL',
    'Memphis': 'MEM', 'Miami': 'MIA', 'Milwaukee': 'MIL', 'Minnesota': 'MIN',
    'New Orleans': 'NOP', 'New York': 'NYK', 'Knicks': 'NYK',
    'Oklahoma City': 'OKC', 'Orlando': 'ORL', 'Philadelphia': 'PHI', '76ers': 'PHI',
    'Phoenix': 'PHX', 'Portland': 'POR', 'Sacramento': 'SAC',
    'San Antonio': 'SAS', 'Toronto': 'TOR', 'Utah': 'UTA', 'Washington': 'WAS'
}
_TEAM_ALIASES = {
    'ATL': ['ATL', 'Atlanta', 'Atlanta Hawks'], 'BOS': ['BOS', 'Boston', 'Boston Celtics'],
    'BKN': ['BKN', 'Brooklyn', 'Brooklyn Nets'], 'CHA': ['CHA', 'Charlotte', 'Charlotte Hornets'],
    'CHI': ['CHI', 'Chicago', 'Chicago Bulls'], 'CLE': ['CLE', 'Cleveland', 'Cleveland Cavaliers'],
    'DAL': ['DAL', 'Dallas', 'Dallas Mavericks'], 'DEN': ['DEN', 'Denver', 'Denver Nuggets'],
    'DET': ['DET', 'Detroit', 'Detroit Pistons'], 'GSW': ['GSW', 'Golden State', 'Golden State Warriors'],
    'HOU': ['HOU', 'Houston', 'Houston Rockets'], 'IND': ['IND', 'Indiana', 'Indiana Pacers'],
    'LAC': ['LAC', 'LA Clippers', 'Los Angeles Clippers', 'Clippers'],
    'LAL': ['LAL', 'LA Lakers', 'Los Angeles Lakers', 'Lakers'],
    'MEM': ['MEM', 'Memphis', 'Memphis Grizzlies'], 'MIA': ['MIA', 'Miami', 'Miami Heat'],
    'MIL': ['MIL', 'Milwaukee', 'Milwaukee Bucks'], 'MIN': ['MIN', 'Minnesota', 'Minnesota Timberwolves'],
    'NOP': ['NOP', 'New Orleans', 'New Orleans Pelicans'],
    'NYK': ['NYK', 'New York', 'New York Knicks', 'Knicks'],
    'OKC': ['OKC', 'Oklahoma City', 'Oklahoma City Thunder'], 'ORL': ['ORL', 'Orlando', 'Orlando Magic'],
    'PHI': ['PHI', 'Philadelphia', 'Philadelphia 76ers', '76ers'],
    'PHX': ['PHX', 'Phoenix', 'Phoenix Suns'], 'POR': ['POR', 'Portland', 'Portland Trail Blazers'],
    'SAC': ['SAC', 'Sacramento', 'Sacramento Kings'], 'SAS': ['SAS', 'San Antonio', 'San Antonio Spurs'],
    'TOR': ['TOR', 'Toronto', 'Toronto Raptors'], 'UTA': ['UTA', 'Utah', 'Utah Jazz'],
    'WAS': ['WAS', 'Washington', 'Washington Wizards']
}


def get_game_odds(game_date, home_team, odds_dict):
    """Return spread, ML, total, public%, opening spread, line movement, closing spread.

    Task 024: also passes through the *actual* away-side ML/spread/total
    prices when the source entry provides them (``ml_away``,
    ``spread_away_points``, ``spread_home_price``, ``spread_away_price``,
    ``total_over_price``, ``total_under_price``). These are never derived by
    negating the home price — they are only populated when the underlying
    odds source (e.g. ``load_pinnacle_lines``) recorded the real two-sided
    quote. Absent that, they stay ``NaN`` (missing, not fabricated) rather
    than silently falling back to a negated home price.
    """
    out = {
        "spread": np.nan, "ml": np.nan, "total": np.nan,
        "public_home_pct": np.nan, "spread_open": np.nan, "spread_move": np.nan,
        "closing_spread": np.nan,
        "ml_away": np.nan, "spread_away_points": np.nan,
        "spread_home_price": np.nan, "spread_away_price": np.nan,
        "total_over_price": np.nan, "total_under_price": np.nan,
    }
    if not odds_dict:
        return out
    try:
        game_dt = pd.to_datetime(game_date).date()
    except Exception:
        return out
    abbr = _ABBR_MAP.get(str(home_team).strip(), str(home_team).strip())
    aliases = _TEAM_ALIASES.get(abbr, [home_team])
    for alias in aliases:
        key = (game_dt, alias)
        if key in odds_dict:
            vals = odds_dict[key]
            if isinstance(vals, dict):
                out["spread"] = vals.get("spread", np.nan)
                out["ml"] = vals.get("ml", np.nan)
                out["total"] = vals.get("total", np.nan)
                out["public_home_pct"] = vals.get("public_home_pct", np.nan)
                out["spread_open"] = vals.get("spread_open", vals.get("spread", np.nan))
                out["closing_spread"] = vals.get("closing_spread", vals.get("spread", np.nan))
                so, sc = out["spread_open"], out["spread"]
                if pd.notna(so) and pd.notna(sc):
                    out["spread_move"] = float(sc) - float(so)
                out["ml_away"] = vals.get("ml_away", np.nan)
                out["spread_away_points"] = vals.get("spread_away", np.nan)
                out["spread_home_price"] = vals.get("spread_home_price", np.nan)
                out["spread_away_price"] = vals.get("spread_away_price", np.nan)
                out["total_over_price"] = vals.get("total_over_price", np.nan)
                out["total_under_price"] = vals.get("total_under_price", np.nan)
            else:
                out["spread"], out["ml"] = vals[0], vals[1]
            break
    return out


def get_odds(game_date, home_team, odds_dict):
    """Fetch spread and moneyline for a game (backward-compatible)."""
    g = get_game_odds(game_date, home_team, odds_dict)
    return g["spread"], g["ml"]

# We'll also need a function to get closing spread for CLV tracking.
# We'll implement a separate function to fetch closing line (latest available).
def get_closing_odds(game_date, home_team, odds_dict, away_team=None):
    """Return closing spread and moneyline for *this exact game*.

    Task 025 fix: this used to silently fall back to "the latest odds row on
    or before ``game_date`` for the home team", which could — and did —
    return a *different game's* price whenever the home team's own line for
    this date/opponent was missing (e.g. it would happily return a stale
    price from that team's previous home game). That fallback is removed:
    a match now requires the home team on the exact ``game_date`` AND (when
    supplied) the opponent, matching the canonical-game-identity requirement
    of Task 025/Phase 2A ("never backfill the latest prior game for the same
    team"). No match -> ``(NaN, NaN)``, never a substituted price.
    """
    if not odds_dict:
        return np.nan, np.nan

    try:
        game_dt = pd.to_datetime(game_date).date()
        if pd.isnull(game_dt):
            return np.nan, np.nan
    except Exception:
        return np.nan, np.nan

    home_abbr = _ABBR_MAP.get(str(home_team).strip(), str(home_team).strip())
    home_aliases = set(_TEAM_ALIASES.get(home_abbr, [home_team]))

    away_aliases = None
    if away_team is not None:
        away_abbr = _ABBR_MAP.get(str(away_team).strip(), str(away_team).strip())
        away_aliases = set(_TEAM_ALIASES.get(away_abbr, [away_team]))

    for alias in home_aliases:
        key = (game_dt, alias)
        if key not in odds_dict:
            continue
        vals = odds_dict[key]
        if not isinstance(vals, dict):
            continue
        if away_aliases is not None:
            entry_away = vals.get("away_team") or vals.get("opponent")
            if entry_away is not None and entry_away not in away_aliases:
                continue  # this row is for the right team/date but wrong opponent
        spread = vals.get("closing_spread", vals.get("spread", np.nan))
        ml = vals.get("ml", np.nan)
        return spread, ml

    return np.nan, np.nan

# ──────────────────────────────────────────────────────────────────────────────
# 2. EDGE AND EXPECTED VALUE CALCULATIONS (unchanged)
# ──────────────────────────────────────────────────────────────────────────────

def american_to_decimal(odds):
    if pd.isna(odds):
        return np.nan
    if odds > 0:
        return 1 + odds / 100.0
    else:
        return 1 + 100.0 / abs(odds)

def implied_probability(odds):
    if pd.isna(odds):
        return np.nan
    if odds > 0:
        return 100.0 / (odds + 100)
    else:
        return abs(odds) / (abs(odds) + 100)


def devig_two_way(prob_a: float, prob_b: float) -> tuple[float, float]:
    """Multiplicative de-vig: normalize two implied probabilities to sum to 1."""
    if prob_a is None or prob_b is None or not np.isfinite(prob_a) or not np.isfinite(prob_b):
        return 0.5, 0.5
    total = float(prob_a) + float(prob_b)
    if total <= 1e-9:
        return 0.5, 0.5
    return float(prob_a) / total, float(prob_b) / total


def fair_probs_from_home_ml(market_ml_home) -> tuple[float, float]:
    """Fair home/away win probabilities from a single home ML quote (no vig)."""
    if pd.isna(market_ml_home):
        return 0.5, 0.5
    p_home = implied_probability(market_ml_home)
    p_away = implied_probability(-market_ml_home)
    return devig_two_way(p_home, p_away)


def fair_home_win_prob(market_ml_home) -> float:
    return fair_probs_from_home_ml(market_ml_home)[0]


def spread_edge(model_spread, market_spread):
    if pd.isna(market_spread):
        return np.nan
    return model_spread + market_spread

def moneyline_edge(model_win_prob_home, market_ml_home):
    if pd.isna(market_ml_home):
        return np.nan
    dec = american_to_decimal(market_ml_home)
    ev = (model_win_prob_home * dec) - 1.0
    return ev


UNDERDOG_DECIMAL = 2.0  # +100 American


def ml_prob_for_side(model_win_prob_home, market_ml_home, side: str) -> float:
    """Model win probability for one side, optionally shrunk toward market implied."""

    if side == "Home":
        p = float(model_win_prob_home)
        if ML_MARKET_DEVIG:
            implied = fair_home_win_prob(market_ml_home)
        else:
            implied = implied_probability(market_ml_home)
        dec = american_to_decimal(market_ml_home)
    else:
        p = 1.0 - float(model_win_prob_home)
        if ML_MARKET_DEVIG:
            implied = fair_probs_from_home_ml(market_ml_home)[1]
        else:
            implied = implied_probability(-market_ml_home)
        dec = american_to_decimal(-market_ml_home)
    shrink = float(ML_MARKET_SHRINK or 0.0)
    if dec >= UNDERDOG_DECIMAL:
        shrink = min(1.0, shrink + float(ML_UNDERDOG_EXTRA_SHRINK or 0.0))
    if shrink > 0 and pd.notna(implied):
        p = (1.0 - shrink) * p + shrink * float(implied)
    return float(np.clip(p, 0.01, 0.99))


def ml_min_ev_for_side(min_ev: float, decimal_odds: float) -> float:

    if ML_UNDERDOG_MIN_EV is not None and decimal_odds >= UNDERDOG_DECIMAL:
        return max(min_ev, float(ML_UNDERDOG_MIN_EV))
    return min_ev


def ml_predicted_winner_side(model_win_prob_home: float) -> str:
    """Side the model expects to win outright (WIN_PROB > 0.5 → Home)."""
    if pd.isna(model_win_prob_home):
        return "Pass"
    if model_win_prob_home > 0.5:
        return "Home"
    if model_win_prob_home < 0.5:
        return "Away"
    return "Pass"


def ml_is_coin_flip(model_win_prob_home: float, band: float | None = None) -> bool:
    """True when the model has no strong lean on the winner."""

    if pd.isna(model_win_prob_home):
        return False
    half_band = float(ML_COIN_FLIP_BAND if band is None else band)
    return abs(float(model_win_prob_home) - 0.5) <= half_band


def ml_underdog_bet_side(dec_home: float, dec_away: float) -> tuple[str | None, float]:
    """Return (side, decimal_odds) for the underdog, if any."""
    if dec_home >= UNDERDOG_DECIMAL and dec_home >= dec_away:
        return "Home", dec_home
    if dec_away >= UNDERDOG_DECIMAL:
        return "Away", dec_away
    return None, np.nan


def select_ml_bet(
    model_win_prob_home,
    market_ml_home,
    *,
    min_ev: float | None = None,
    max_favorite_decimal: float = 1.45,
    min_win_pct: float | None = None,
    winner_only: bool | None = None,
    coin_flip_band: float | None = None,
    coin_flip_underdog_only: bool | None = None,
    coin_flip_underdog_min_ev: float | None = None,
) -> tuple[str, float, float]:
    """Pick ML bet when calibrated EV and win% pass; Pass otherwise."""

    if pd.isna(market_ml_home) or model_win_prob_home is None or pd.isna(model_win_prob_home):
        return "Pass", np.nan, np.nan

    if min_ev is None:
        min_ev = ML_MIN_EV
    if min_win_pct is None:
        min_win_pct = float(MIN_ML_WIN_PCT)
    if winner_only is None:
        winner_only = ML_BET_PREDICTED_WINNER_ONLY
    if coin_flip_band is None:
        coin_flip_band = ML_COIN_FLIP_BAND
    if coin_flip_underdog_only is None:
        coin_flip_underdog_only = ML_COIN_FLIP_UNDERDOG_ONLY
    if coin_flip_underdog_min_ev is None:
        coin_flip_underdog_min_ev = ML_COIN_FLIP_UNDERDOG_MIN_EV

    dec_home = american_to_decimal(market_ml_home)
    dec_away = american_to_decimal(-market_ml_home)
    p_home = ml_prob_for_side(model_win_prob_home, market_ml_home, "Home")
    p_away = ml_prob_for_side(model_win_prob_home, market_ml_home, "Away")
    ev_home = (p_home * dec_home) - 1.0
    ev_away = (p_away * dec_away) - 1.0
    min_ev_home = ml_min_ev_for_side(min_ev, dec_home)
    min_ev_away = ml_min_ev_for_side(min_ev, dec_away)
    min_p = float(min_win_pct) / 100.0

    def _qualifies(side: str, ev: float, thr: float, dec: float, prob: float) -> bool:
        return (
            ev > 0
            and ev > thr
            and dec >= max_favorite_decimal
            and prob >= min_p
        )

    if winner_only:
        if ml_is_coin_flip(model_win_prob_home, coin_flip_band):
            if not coin_flip_underdog_only:
                return "Pass", np.nan, np.nan
            ud_side, ud_dec = ml_underdog_bet_side(dec_home, dec_away)
            if ud_side is not None:
                ud_ev = ev_home if ud_side == "Home" else ev_away
                ud_p = p_home if ud_side == "Home" else p_away
                ud_thr = max(
                    ml_min_ev_for_side(min_ev, ud_dec),
                    float(coin_flip_underdog_min_ev),
                )
                if _qualifies(ud_side, ud_ev, ud_thr, ud_dec, ud_p):
                    return ud_side, ud_ev, ud_dec
            return "Pass", np.nan, np.nan

        pick = ml_predicted_winner_side(model_win_prob_home)
        if pick == "Home" and _qualifies("Home", ev_home, min_ev_home, dec_home, p_home):
            return "Home", ev_home, dec_home
        if pick == "Away" and _qualifies("Away", ev_away, min_ev_away, dec_away, p_away):
            return "Away", ev_away, dec_away
        return "Pass", np.nan, np.nan

    candidates = []
    if _qualifies("Home", ev_home, min_ev_home, dec_home, p_home):
        candidates.append(("Home", ev_home, dec_home))
    if _qualifies("Away", ev_away, min_ev_away, dec_away, p_away):
        candidates.append(("Away", ev_away, dec_away))
    if not candidates:
        return "Pass", np.nan, np.nan
    return max(candidates, key=lambda x: x[1])

def market_microstructure_features(spread_move, public_home_pct, market_spread=np.nan):
    """RLM, steam, and vig-free helpers for model features."""
    sm = float(spread_move) if pd.notna(spread_move) else 0.0
    pub = float(public_home_pct) if pd.notna(public_home_pct) else 0.5
    # Reverse line movement: line moved toward home despite public on away (or vice versa)
    rlm = 0.0
    if abs(sm) >= 0.5:
        if sm > 0 and pub < 0.45:
            rlm = abs(sm)
        elif sm < 0 and pub > 0.55:
            rlm = abs(sm)
    steam = int(abs(sm) >= 1.0)
    fair_spread = market_spread  # placeholder when single-book; extend with multi-book de-vig
    return {
        "reverse_line_movement": rlm,
        "steam_flag": steam,
        "fair_spread_vigfree": fair_spread if pd.notna(fair_spread) else 0.0,
        "public_away_pct": 1.0 - pub,
    }


def spread_kelly_fraction(cover_prob: float, juice: float = -110) -> float:
    """Full Kelly fraction for ATS at given juice (e.g. -110)."""
    if cover_prob is None or not np.isfinite(cover_prob):
        return 0.0
    b = 100.0 / 110.0 if juice == -110 else abs(juice) / 100.0
    kelly = (cover_prob * b - (1.0 - cover_prob)) / b
    return float(max(0.0, kelly))


def spread_cover_prob(
    model_spread: float,
    market_spread: float,
    conf_width: float = 24.0,
    direction: str = "Home",
) -> float:
    """P(bet side covers) from Gaussian margin model.

    Uses corrected model spread vs market line; sigma scales with conformal width.
    """
    if pd.isna(model_spread) or pd.isna(market_spread):
        return 0.5
    sigma = max(float(conf_width) / 2.5, 4.0)
    z = (float(model_spread) + float(market_spread)) / sigma
    z = float(np.clip(z, -4.0, 4.0))
    try:
        from scipy.stats import norm
        p_home = float(norm.cdf(z))
    except ImportError:
        p_home = 0.5 * (1.0 + float(np.tanh(z * 0.7978845608)))
    if direction == "Away":
        return 1.0 - p_home
    return p_home


# Alias used in roadmap / docs
spread_cover_prob_gaussian = spread_cover_prob


def total_over_prob_gaussian(
    pred_total,
    market_total,
    sigma,
) -> float:
    """P(Over | L) under N(mu=pred_total, sigma^2).

    P(Over) = 1 - Phi((L - mu) / sigma)
    """
    if pd.isna(pred_total) or pd.isna(market_total) or pd.isna(sigma):
        return 0.5
    sig = float(sigma)
    if not np.isfinite(sig) or sig <= 0:
        return 0.5
    mu = float(pred_total)
    line = float(market_total)
    z = (line - mu) / sig
    z = float(np.clip(z, -6.0, 6.0))
    try:
        from scipy.stats import norm
        return float(1.0 - norm.cdf(z))
    except ImportError:
        # erf approximation of CDF
        return float(0.5 * (1.0 - np.tanh(z * 0.7978845608)))


def total_under_prob_gaussian(pred_total, market_total, sigma) -> float:
    return float(1.0 - total_over_prob_gaussian(pred_total, market_total, sigma))


def crps_gaussian(actual, mu, sigma) -> float:
    """CRPS for a Gaussian forecast N(mu, sigma^2) vs scalar actual."""
    if pd.isna(actual) or pd.isna(mu) or pd.isna(sigma):
        return np.nan
    sig = float(sigma)
    if not np.isfinite(sig) or sig <= 0:
        return np.nan
    y = float(actual)
    m = float(mu)
    z = (y - m) / sig
    try:
        from scipy.stats import norm
        # CRPS(N(m,s^2), y) = s * (z(2Phi(z)-1) + 2phi(z) - 1/sqrt(pi))
        return float(
            sig * (z * (2.0 * norm.cdf(z) - 1.0) + 2.0 * norm.pdf(z) - 1.0 / np.sqrt(np.pi))
        )
    except ImportError:
        # Fallback without scipy: absolute error scaled (not exact CRPS)
        return float(abs(y - m))


def select_ou_bet(
    pred_total,
    market_total,
    sigma=None,
    *,
    min_edge: float | None = None,
    min_prob: float | None = None,
    use_gaussian: bool | None = None,
) -> tuple[str, float, float]:
    """Pick Over / Under / Pass from edge and/or Gaussian P(side).

    Returns (direction, p_over, total_edge).
    """

    if pd.isna(pred_total) or pd.isna(market_total) or float(market_total) <= 0:
        return "Pass", 0.5, np.nan

    if min_edge is None:
        min_edge = float(OU_MIN_EDGE)
    if min_prob is None:
        min_prob = float(OU_MIN_PROB)
    if use_gaussian is None:
        use_gaussian = bool(OU_USE_GAUSSIAN_PROB)

    mu = float(pred_total)
    line = float(market_total)
    edge = mu - line
    p_over = 0.5
    if use_gaussian and sigma is not None and np.isfinite(float(sigma)) and float(sigma) > 0:
        p_over = total_over_prob_gaussian(mu, line, sigma)
    else:
        # Soft logistic proxy from edge when sigma missing
        p_over = float(1.0 / (1.0 + np.exp(-edge / 6.0)))
        p_over = float(np.clip(p_over, 0.01, 0.99))

    p_under = 1.0 - p_over
    # Require edge AND probability confidence (when gaussian on)
    if abs(edge) < float(min_edge):
        return "Pass", p_over, edge

    if p_over >= float(min_prob) and edge > 0:
        return "Over", p_over, edge
    if p_under >= float(min_prob) and edge < 0:
        return "Under", p_over, edge
    # Edge clears but prob gate fails → Pass
    if use_gaussian:
        return "Pass", p_over, edge
    # Legacy edge-only mode when gaussian disabled
    return ("Over" if edge > 0 else "Under"), p_over, edge


def explain_ml_bet(
    model_win_prob_home,
    market_ml_home,
    *,
    min_ev: float | None = None,
    **kwargs,
) -> dict:
    """Both-sides ML decision with EV breakdown for predict/simulate UX."""

    if min_ev is None:
        min_ev = ML_MIN_EV

    predicted_winner = ml_predicted_winner_side(model_win_prob_home)
    p_home_raw = float(model_win_prob_home) if model_win_prob_home is not None and pd.notna(model_win_prob_home) else np.nan
    p_away_raw = (1.0 - p_home_raw) if np.isfinite(p_home_raw) else np.nan

    if pd.isna(market_ml_home) or not np.isfinite(p_home_raw):
        return {
            "predicted_winner": predicted_winner,
            "p_home": p_home_raw,
            "p_away": p_away_raw,
            "ev_home": np.nan,
            "ev_away": np.nan,
            "ml_bet": "Pass",
            "ml_ev": np.nan,
            "ml_decimal": np.nan,
            "ml_reason": "missing_odds_or_prob",
        }

    p_home = ml_prob_for_side(p_home_raw, market_ml_home, "Home")
    p_away = ml_prob_for_side(p_home_raw, market_ml_home, "Away")
    dec_home = american_to_decimal(market_ml_home)
    dec_away = american_to_decimal(-market_ml_home)
    ev_home = (p_home * dec_home) - 1.0
    ev_away = (p_away * dec_away) - 1.0

    side, ev_side, dec_side = select_ml_bet(
        p_home_raw, market_ml_home, min_ev=min_ev, **kwargs,
    )

    if side == "Pass":
        reason = "no_side_clears_ev_and_winpct"
        if ml_is_coin_flip(p_home_raw):
            reason = "coin_flip_pass"
        elif predicted_winner == "Home" and ev_home <= float(min_ev):
            reason = "predicted_winner_home_but_ev_too_low"
        elif predicted_winner == "Away" and ev_away <= float(min_ev):
            reason = "predicted_winner_away_but_ev_too_low"
    elif side != predicted_winner and predicted_winner in ("Home", "Away"):
        reason = f"value_on_{side.lower()}_not_predicted_winner"
    elif side == predicted_winner:
        reason = f"bet_predicted_winner_{side.lower()}_with_positive_ev"
    else:
        reason = "selected"

    return {
        "predicted_winner": predicted_winner,
        "p_home": float(p_home_raw),
        "p_away": float(p_away_raw),
        "p_home_shrunk": float(p_home),
        "p_away_shrunk": float(p_away),
        "ev_home": float(ev_home),
        "ev_away": float(ev_away),
        "ml_bet": side,
        "ml_ev": float(ev_side) if pd.notna(ev_side) else np.nan,
        "ml_decimal": float(dec_side) if pd.notna(dec_side) else np.nan,
        "ml_reason": reason,
    }


def elo_meta_agreement(model_spread, market_spread, elo_margin_calibrated) -> float:
    """1.0 when Elo-calibrated margin and meta spread agree on ATS side vs market."""
    if pd.isna(market_spread) or elo_margin_calibrated is None or pd.isna(elo_margin_calibrated):
        return 1.0
    if pd.isna(model_spread):
        return 1.0
    meta_side = np.sign(float(model_spread) + float(market_spread))
    elo_side = np.sign(float(elo_margin_calibrated) + float(market_spread))
    if meta_side == 0 or elo_side == 0:
        return 1.0
    return float(meta_side == elo_side)


def bet_analysis(model_spread, market_spread, model_win_prob=None, market_ml=None,
                 min_edge_pts=2.5, min_ev=None, conf_width=None,
                 spread_move=0.0, public_home_pct=0.5, require_rlm=False,
                 confidence_calibrator=None, rating_uncertainty=None,
                 max_favorite_decimal=1.45,
                 require_elo_agreement=False, elo_margin_calibrated=None,
                 elo_agreement_extra_edge=1.0, elo_agreement=None,
                 disagreement_trust=1.0, phantom_injury_flag=False,
                 pred_home=None, pred_away=None,
                 matchup_vol_sigma=None, ats_classifier_prob=None):

    if min_ev is None:
        min_ev = ML_MIN_EV

    result = {
        "spread_edge_pts": np.nan,
        "spread_direction": "Pass",
        "spread_confidence": 0,
        "spread_stars": "No market",
        "ml_ev": np.nan,
        "ml_direction": "Pass",
        "ml_confidence": 0
    }

    if not pd.isna(market_spread):
        edge_pts = spread_edge(model_spread, market_spread)
        result["spread_edge_pts"] = edge_pts
        abs_edge = abs(edge_pts)
        if phantom_injury_flag and float(disagreement_trust or 1.0) < 0.75:
            result["spread_direction"] = "Pass"
            result["spread_stars"] = "Pass (phantom injury)"
        elif abs_edge >= min_edge_pts or min_edge_pts <= 0:
            direction = "Home" if edge_pts > 0 else "Away"
            if require_rlm:
                micro = market_microstructure_features(spread_move, public_home_pct, market_spread)
                if micro["reverse_line_movement"] < 0.25 and micro["steam_flag"] == 0:
                    direction = "Pass"
            if direction != "Pass":
                if require_elo_agreement and elo_margin_calibrated is not None and not pd.isna(elo_margin_calibrated):
                    elo_edge = float(elo_margin_calibrated) + float(market_spread)
                    if np.sign(edge_pts) != np.sign(elo_edge) and abs_edge < min_edge_pts + elo_agreement_extra_edge:
                        direction = "Pass"
                result["spread_direction"] = direction
                cw = conf_width or 24.0
                if elo_agreement is None:
                    elo_agreement = elo_meta_agreement(
                        model_spread, market_spread, elo_margin_calibrated,
                    )
                spread_cover_p = spread_cover_prob(
                    model_spread, market_spread, cw, direction=direction,
                )
                if USE_SKELLAM_COVER_PROB and pred_home is not None and pred_away is not None:
                    spread_cover_p = cover_prob_skellam(
                        pred_home, pred_away, market_spread, direction=direction,
                    )
                result["spread_cover_prob_raw"] = spread_cover_p
                ats_feats = build_confidence_features(
                    spread_edge_pts=edge_pts,
                    conf_width=cw,
                    rating_uncertainty=rating_uncertainty or 350.0,
                    elo_meta_agreement=elo_agreement,
                    spread_cover_prob=spread_cover_p,
                    matchup_vol_sigma=matchup_vol_sigma,
                    disagreement_trust=disagreement_trust,
                    phantom_injury_flag=phantom_injury_flag,
                    ats_classifier_prob=ats_classifier_prob,
                )
                ats_feats["win_prob"] = model_win_prob or 0.5
                if confidence_calibrator is not None:
                    cal = confidence_calibrator.predict("ats", ats_feats, direction=direction)
                    result["spread_confidence"] = int(round(cal["confidence_score"]))
                    result["spread_stars"] = cal["stars"]
                    result["cover_prob_calibrated"] = float(np.clip(cal["calibrated_prob"], 0.01, 0.99))
                    result["confidence_tier"] = cal["confidence_tier"]
                    result["confidence_score_raw"] = cal["confidence_score_raw"]
                else:
                    raw_score = ats_confidence_score(ats_feats)
                    prob = float(np.clip(0.35 * spread_cover_p + 0.65 * (0.524 + raw_score / 120.0), 0.01, 0.99))
                    tier = _tier_from_prob(prob)
                    result["spread_confidence"] = int(round(prob * 100))
                    result["cover_prob_calibrated"] = prob
                    result["confidence_tier"] = tier
                    result["spread_stars"] = _stars_from_tier(tier, direction)
                    result["confidence_score_raw"] = raw_score
        else:
            result["spread_direction"] = "Pass"
            result["spread_stars"] = "Pass"

    if (model_win_prob is not None) and not pd.isna(market_ml):
        side, ev_side, dec_side = select_ml_bet(
            model_win_prob,
            market_ml,
            min_ev=min_ev,
            max_favorite_decimal=max_favorite_decimal,
        )
        p_home = ml_prob_for_side(model_win_prob, market_ml, "Home")
        ev_home = (p_home * american_to_decimal(market_ml)) - 1.0
        result["ml_ev"] = ev_home
        if side != "Pass":
            result["ml_direction"] = side
            if confidence_calibrator is not None:
                cal = confidence_calibrator.predict(
                    "ml",
                    {"win_prob": model_win_prob, "ml_ev": ev_side,
                     "rating_uncertainty": rating_uncertainty or 350.0},
                    direction=side,
                )
                result["ml_confidence"] = cal["confidence_score"]
                result["ml_confidence_tier"] = cal["confidence_tier"]
            else:
                result["ml_confidence"] = min(100, int(ev_side * 1000))
        else:
            result["ml_direction"] = "Pass"

    return result

# ──────────────────────────────────────────────────────────────────────────────
# 3. FINANCIAL CALCULATOR (unchanged)
# ──────────────────────────────────────────────────────────────────────────────

def calculate_financials(df, unit_size=10.0, use_fractional_kelly=False):
    df = df.copy()
    df["SPREAD_PROFIT"] = 0.0
    win_payout = unit_size * (100 / 110)

    if use_fractional_kelly and "KELLY_FRACTION" in df.columns:
        effective_unit = unit_size * df["KELLY_FRACTION"].fillna(0)
    else:
        effective_unit = unit_size

    mask_win = df["ATS_WIN"] == 1
    mask_loss = df["ATS_LOSS"] == 1
    df.loc[mask_win, "SPREAD_PROFIT"] = win_payout * (effective_unit / unit_size)
    df.loc[mask_loss, "SPREAD_PROFIT"] = -effective_unit

    df["ML_PROFIT"] = 0.0
    df["ML_BET"] = "Pass"
    if "MARKET_ML" in df.columns and "WIN_PROB" in df.columns:
        for idx, row in df.iterrows():
            ml = row["MARKET_ML"]
            if pd.isna(ml):
                continue
            model_prob_home = row["WIN_PROB"]
            side, ev_side, dec = select_ml_bet(model_prob_home, ml)
            if side == "Pass":
                continue
            df.at[idx, "ML_BET"] = side
            home_win = row["ACTUAL_HOME"] > row["ACTUAL_AWAY"]
            won = home_win if side == "Home" else not home_win
            if won:
                df.at[idx, "ML_PROFIT"] = unit_size * (dec - 1)
            else:
                df.at[idx, "ML_PROFIT"] = -unit_size
    return df

# ──────────────────────────────────────────────────────────────────────────────
# 4. REPORTING AND GRAPHING (unchanged)
# ──────────────────────────────────────────────────────────────────────────────

def print_interval_ats_report(results_df):
    if results_df.empty:
        return None
    df = results_df[results_df["MARKET_SPREAD"].notna() & (results_df["DIRECTION"] != "Pass")].copy()
    if df.empty:
        print("  [No Vegas Market lines found or zero active bets triggered]")
        return None

    home_covers = (df["ACTUAL_MARGIN"] + df["MARKET_SPREAD"]) > 0
    away_covers = (df["ACTUAL_MARGIN"] + df["MARKET_SPREAD"]) < 0
    pushes = (df["ACTUAL_MARGIN"] + df["MARKET_SPREAD"]) == 0

    df["ATS_WIN"] = (((df["DIRECTION"] == "Home") & home_covers) |
                     ((df["DIRECTION"] == "Away") & away_covers)).astype(int)
    df["ATS_LOSS"] = (((df["DIRECTION"] == "Home") & away_covers) |
                      ((df["DIRECTION"] == "Away") & home_covers)).astype(int)
    df["ATS_PUSH"] = pushes.astype(int)

    df = calculate_financials(df, unit_size=10.0)

    bins = [-1.0, 2.999, 4.999, 7.999, float('inf')]
    labels = ["0.0 - 2.9", "3.0 - 4.9", "5.0 - 7.9", "8.0+"]
    df["EDGE_TIER"] = pd.cut(df["EDGE"], bins=bins, labels=labels, right=True)

    print("══════════════════════════════════════════════════════════════")
    print("      SPREAD BETTING PERFORMANCE ($10 Base Unit)              ")
    print("══════════════════════════════════════════════════════════════")
    for tier in labels:
        tier_df = df[df["EDGE_TIER"] == tier]
        total = len(tier_df)
        if total == 0:
            continue
        w = tier_df["ATS_WIN"].sum()
        l = tier_df["ATS_LOSS"].sum()
        p = tier_df["ATS_PUSH"].sum()
        win_pct = w / (w + l) if (w + l) > 0 else 0.0
        profit = tier_df["SPREAD_PROFIT"].sum()
        print(f" Edge {tier:>9} pts | {total:3d} bets | {w:2d}-{l:2d}-{p:1d} | {win_pct:5.1%} ATS | Net: ${profit:+.2f}")

    ml_df = df[df["ML_BET"] != "Pass"]
    if not ml_df.empty:
        ml_wins = (ml_df["ML_PROFIT"] > 0).sum()
        ml_losses = (ml_df["ML_PROFIT"] < 0).sum()
        ml_profit = ml_df["ML_PROFIT"].sum()
        ml_win_pct = ml_wins / len(ml_df)
        print("──────────────────────────────────────────────────────────────")
        print("      MONEYLINE VALUE PERFORMANCE ($10 Base Unit)             ")
        print("──────────────────────────────────────────────────────────────")
        print(f" Total ML Plays: {len(ml_df):3d} | {ml_wins:2d} Wins - {ml_losses:2d} Losses | {ml_win_pct:5.1%} Hit Rate")
        print(f" Total Net Profit: ${ml_profit:+.2f}")
    print("══════════════════════════════════════════════════════════════\n")
    return df

def plot_financial_performance(financial_df, title="Cumulative_Profit"):
    if financial_df is None or financial_df.empty:
        return
    df = financial_df.sort_values("DATE").reset_index(drop=True)
    df["CUM_SPREAD"] = df["SPREAD_PROFIT"].cumsum()
    plt.figure(figsize=(14, 7))
    plt.plot(df.index, df["CUM_SPREAD"], label="ATS Spread Profit ($10/bet)", color="#2ca02c", linewidth=2.5)
    if "ML_PROFIT" in df.columns:
        df["CUM_ML"] = df["ML_PROFIT"].cumsum()
        plt.plot(df.index, df["CUM_ML"], label="Moneyline Profit ($10/bet)", color="#1f77b4", linewidth=2.5, alpha=0.8)
    plt.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.6)
    plt.title(f"Simulation Equity Curve: {title}", fontsize=14, fontweight="bold")
    plt.xlabel("Number of Bets Placed (Chronological)", fontsize=12)
    plt.ylabel("Cumulative Profit ($)", fontsize=12)
    plt.legend(loc="upper left", fontsize=11)
    plt.grid(alpha=0.3)
    filename = f"equity_curve_{title.replace(' ', '_').lower()}.png"
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"✅ Equity curve graph saved as '{filename}'")

class SpreadCalibrator:
    """Rolling spread calibration.

    Fits ``actual_margin ≈ a + b · pred_margin`` over a rolling window, so it
    corrects both a global additive bias **and** systematic slope error (e.g.
    shrinking big favorites). Falls back to additive-bias correction before
    ``min_samples`` points accrue, matching the previous behaviour at cold start.
    """
    def __init__(self, window=50, min_samples=30, slope_clip=(0.3, 1.7), mode="linear"):
        self.window = window
        self.min_samples = min_samples
        self.slope_clip = slope_clip
        self.mode = mode   # "linear" (a + b*pred) or "additive" (legacy bias-only)
        self.preds = deque(maxlen=window)
        self.actuals = deque(maxlen=window)
        self.errors = deque(maxlen=window)
        self.abs_residuals = deque(maxlen=window)
        self.bias = 0.0
        self.a = 0.0
        self.b = 1.0

    def update(self, pred, actual):
        self.preds.append(pred)
        self.actuals.append(actual)
        self.errors.append(pred - actual)
        self.abs_residuals.append(abs(pred - actual))
        self.bias = float(np.mean(self.errors))
        if len(self.preds) >= self.min_samples:
            x = np.asarray(self.preds, dtype=float)
            y = np.asarray(self.actuals, dtype=float)
            vx = x.var()
            if vx > 1e-6:
                b = np.cov(x, y, bias=True)[0, 1] / vx
                self.b = float(np.clip(b, *self.slope_clip))
                self.a = float(y.mean() - self.b * x.mean())
            else:
                self.a, self.b = 0.0, 1.0

    def correct(self, pred):
        if self.mode == "linear" and len(self.preds) >= self.min_samples:
            return self.a + self.b * pred
        return pred - self.bias

    def predict_interval(self, pred, alpha=0.10):
        """Rolling conformal-style symmetric interval from |pred - actual| residuals."""
        if len(self.abs_residuals) >= self.min_samples:
            q = float(np.quantile(np.asarray(self.abs_residuals, dtype=float), 1.0 - alpha / 2.0))
            return pred - q, pred + q, 2.0 * q
        half = 12.0
        return pred - half, pred + half, 2.0 * half

    @classmethod
    def fit_static(cls, preds, actuals, window=50, min_samples=30, slope_clip=(0.3, 1.7), mode="linear"):
        """Offline fit on a calibration slice (no future leakage).

        .. warning::
            The fitted calibrator's ``a``/``b`` reflect the *end state*
            after replaying every ``(pred, actual)`` pair. Applying
            ``.correct()`` with that end state to every row of the same
            slice (as ``pipeline/backtest.py``'s ``static_cal`` path does)
            corrects early rows with parameters informed by later rows in
            the same slice — this is exactly the "static-calibration-to-
            rolling-inference mismatch" named in Task 054. Use
            :func:`replay_rolling_spread_calibration` instead when you need
            row-by-row corrected predictions that match how live inference
            actually calibrates (predict, *then* observe/update).
        """
        obj = cls(window=max(window, len(preds)), min_samples=min_samples,
                  slope_clip=slope_clip, mode=mode)
        for p, a in zip(preds, actuals):
            if pd.notna(p) and pd.notna(a):
                obj.update(float(p), float(a))
        return obj


def replay_rolling_spread_calibration(preds, actuals, window=50, min_samples=30,
                                       slope_clip=(0.3, 1.7), mode="linear",
                                       calibrator=None):
    """Row-by-row rolling spread calibration replay (Task 054).

    For each row, in order, this calls ``calibrator.correct(pred)`` — using
    only state accumulated from *strictly earlier* rows in this same call —
    before ``calibrator.update(pred, actual)`` observes that row's outcome.
    This is exactly the sequence live inference uses (``pipeline/simulate.py``
    calls ``.correct()`` to produce a prediction, then ``.update()`` once the
    result is known) and is the single correct way to reproduce rolling
    spread calibration inside a training-time simulation — replacing any
    "fit once on the whole slice, then re-correct every row in it with the
    final state" (``SpreadCalibrator.fit_static`` + blanket ``.correct()``)
    pattern, which lets later rows in a slice leak into earlier corrections.

    Returns
    -------
    (corrected, calibrator) : np.ndarray, SpreadCalibrator
        ``corrected[i]`` uses only rows ``< i`` from this call plus whatever
        state ``calibrator`` already had when passed in (so training
        simulation and live inference can share literally the same
        calibrator instance across a season boundary).
    """
    cal = calibrator if calibrator is not None else SpreadCalibrator(
        window=window, min_samples=min_samples, slope_clip=slope_clip, mode=mode,
    )
    corrected = np.empty(len(preds), dtype=float)
    for i, (p, a) in enumerate(zip(preds, actuals)):
        p = float(p)
        corrected[i] = cal.correct(p)
        if pd.notna(a):
            cal.update(p, float(a))
    return corrected, cal


def load_modern_odds(csv_path, date_col="game_date", home_col="home_team",
                     spread_col="spread_home_points", ml_col="money_home_odds",
                     time_col=None, game_start_hour=19):
    """Load opening lines keyed by (game_date, home_team)."""
    odds_dict = {}
    try:
        df = pd.read_csv(csv_path)
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        if time_col and time_col in df.columns:
            df["timestamp"] = pd.to_datetime(df[time_col], errors="coerce")
            df = df.sort_values(by=[date_col, "timestamp"])
        else:
            df = df.sort_values(by=[date_col])

        df["game_date_only"] = df[date_col].dt.date
        grouped = df.groupby(["game_date_only", home_col], as_index=False).first()

        for _, row in grouped.iterrows():
            dt_val = row["game_date_only"]
            if pd.isna(dt_val):
                continue
            home = str(row[home_col]).strip()
            if home == "Los Angeles Lakers":
                home = "LA Lakers"
            if home == "Los Angeles Clippers":
                home = "LA Clippers"

            s_val = row[spread_col] if pd.notna(row.get(spread_col)) else np.nan
            m_val = row[ml_col] if pd.notna(row.get(ml_col)) else np.nan

            if pd.notna(s_val) and pd.notna(m_val):
                if m_val < 0 and s_val > 0:
                    s_val = -abs(s_val)
                elif m_val > 0 and s_val < 0:
                    s_val = abs(s_val)

            entry = {"spread": float(s_val), "ml": float(m_val), "spread_open": float(s_val)}
            if pd.notna(row.get("total_over_points")):
                entry["total"] = float(row["total_over_points"])
            pub = row.get("spread_home_stake_percentage")
            if pd.notna(pub):
                entry["public_home_pct"] = float(pub) / 100.0
            odds_dict[(dt_val, home)] = entry
    except Exception as e:
        print(f"Warning loading modern odds: {e}")
    return odds_dict


def decimal_to_american(dec):
    """Convert decimal odds to American odds."""
    try:
        dec = float(dec)
    except (TypeError, ValueError):
        return np.nan
    if not np.isfinite(dec) or dec <= 1.0:
        return np.nan
    if dec >= 2.0:
        return round((dec - 1.0) * 100.0)
    return round(-100.0 / (dec - 1.0))


def load_pinnacle_lines(csv_path, schedule_df=None, match_tolerance_days=2):
    """Load Pinnacle-style lines (nba_main_lines.csv) into the odds_dict format.

    The file has columns team1/team2, decimal moneyline/spread/total odds, and a
    ``timestamp`` (no game date). We take each game's CLOSING snapshot (latest
    timestamp, ~tipoff) and convert decimal odds to American.

    If ``schedule_df`` (columns: date, home, away as tricodes) is provided, each
    odds game is matched to the real schedule game by team matchup and nearest
    date, so the authoritative game date and home/away come from the schedule.
    Otherwise we fall back to dual (closing_date, team) keys storing each team's
    own line, which still lets get_odds resolve the home team.

    Returns a dict keyed by (date, team) -> {"spread", "ml", "total"}.
    """
    out = {}
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:  # noqa: BLE001
        print(f"Warning loading pinnacle lines: {e}")
        return out
    if "timestamp" not in df.columns or df.empty:
        return out

    df["ts"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["ts"])
    key = "game_link" if "game_link" in df.columns else None

    # Closing and opening snapshots per game.
    if key:
        closing = df.sort_values("ts").groupby(key, as_index=False).last()
        opening = df.sort_values("ts").groupby(key, as_index=False).first()
        open_map = {}
        for _, orow in opening.iterrows():
            gl = orow.get("game_link")
            t1o, t2o = to_tricode(orow.get("team1")), to_tricode(orow.get("team2"))
            open_map[gl] = (t1o, t2o, orow.get("team1_spread"), orow.get("team2_spread"))
    else:
        closing = df.sort_values("ts")
        open_map = {}

    # Pre-index schedule by matchup (frozenset of tricodes) for fast matching.
    sched_idx = {}
    if schedule_df is not None and not schedule_df.empty:
        s = schedule_df.copy()
        s["home"] = s["home"].apply(to_tricode)
        s["away"] = s["away"].apply(to_tricode)
        s["date"] = pd.to_datetime(s["date"], errors="coerce")
        for _, r in s.dropna(subset=["date", "home", "away"]).iterrows():
            sched_idx.setdefault(frozenset((r["home"], r["away"])), []).append(
                (r["date"], r["home"], r["away"]))

    for _, row in closing.iterrows():
        t1 = to_tricode(row.get("team1"))
        t2 = to_tricode(row.get("team2"))
        if not isinstance(t1, str) or not isinstance(t2, str):
            continue
        ml1 = decimal_to_american(row.get("team1_moneyline"))
        ml2 = decimal_to_american(row.get("team2_moneyline"))
        spr1 = row.get("team1_spread")
        spr2 = row.get("team2_spread")
        total = row.get("over_total")
        ts_date = row["ts"].date()

        matched = None
        cand = sched_idx.get(frozenset((t1, t2)))
        if cand:
            # nearest scheduled date to the closing timestamp
            best = min(cand, key=lambda c: abs((c[0].date() - ts_date).days))
            if abs((best[0].date() - ts_date).days) <= match_tolerance_days:
                matched = best

        spr1_price, spr2_price = _f(row.get("team1_spread_odds")), _f(row.get("team2_spread_odds"))
        over_price, under_price = _f(row.get("over_total_odds")), _f(row.get("under_total_odds"))

        if matched is not None:
            gdate, home, away = matched
            d = gdate.date()
            spread_open = np.nan
            if key and row.get("game_link") in open_map:
                t1o, t2o, os1, os2 = open_map[row.get("game_link")]
                spread_open = _f(os1 if home == t1o else os2)
            # Task 024: keep BOTH teams' actual quoted prices — never derive
            # the away side by negating the home side.
            if home == t1:
                close_spread, close_ml = _f(spr1), _f(ml1)
                away_spread, away_ml = _f(spr2), _f(ml2)
                home_price, away_price = spr1_price, spr2_price
            else:
                close_spread, close_ml = _f(spr2), _f(ml2)
                away_spread, away_ml = _f(spr1), _f(ml1)
                home_price, away_price = spr2_price, spr1_price
            entry = {
                "spread": close_spread, "ml": close_ml, "total": _f(total),
                "spread_open": spread_open if pd.notna(spread_open) else close_spread,
                "closing_spread": close_spread,
                "away_team": away,
                "spread_away": away_spread, "ml_away": away_ml,
                "spread_home_price": home_price, "spread_away_price": away_price,
                "total_over_price": over_price, "total_under_price": under_price,
            }
            if pd.notna(entry["spread_open"]) and pd.notna(entry["spread"]):
                entry["spread_move"] = float(entry["spread"]) - float(entry["spread_open"])
            out[(d, home)] = entry
            # Also store the away team's own perspective for this exact game
            # so a lookup by (date, away_team) never needs to fall back to a
            # different game (Task 025).
            away_entry = {
                "spread": away_spread, "ml": away_ml, "total": _f(total),
                "spread_open": spread_open if pd.notna(spread_open) else close_spread,
                "closing_spread": away_spread,
                "away_team": home,
                "spread_away": close_spread, "ml_away": close_ml,
                "spread_home_price": away_price, "spread_away_price": home_price,
                "total_over_price": over_price, "total_under_price": under_price,
            }
            out[(d, away)] = away_entry
        else:
            spread_open = np.nan
            if key and row.get("game_link") in open_map:
                t1o, t2o, os1, os2 = open_map[row.get("game_link")]
                spread_open = _f(os1) if t1 == t1o else _f(os2)
            pairs = ((t1, spr1, ml1, t2, spr2, ml2, spr1_price, spr2_price),
                     (t2, spr2, ml2, t1, spr1, ml1, spr2_price, spr1_price))
            for team, spr, ml, opp, opp_spr, opp_ml, own_price, opp_price in pairs:
                cs, cm = _f(spr), _f(ml)
                entry = {
                    "spread": cs, "ml": cm, "total": _f(total),
                    "spread_open": spread_open if pd.notna(spread_open) else cs,
                    "closing_spread": cs,
                    "away_team": opp,
                    "spread_away": _f(opp_spr), "ml_away": _f(opp_ml),
                    "spread_home_price": own_price, "spread_away_price": opp_price,
                    "total_over_price": over_price, "total_under_price": under_price,
                }
                if pd.notna(entry["spread_open"]) and pd.notna(entry["spread"]):
                    entry["spread_move"] = float(entry["spread"]) - float(entry["spread_open"])
                out[(ts_date, team)] = entry
    return out


def _f(v):
    try:
        v = float(v)
        return v if np.isfinite(v) else np.nan
    except (TypeError, ValueError):
        return np.nan


print("✅ Leakage‑free Vegas helpers, ML/spread edge calculators, and financials loaded.")


In [ ]:
# ── module: feature_utils ────────────────────────────────────────────────────
"""Shared feature helpers (no circular imports)."""
from collections import defaultdict, deque

import numpy as np
import pandas as pd


def schedule_density(date_history, gdate):
    """Games in last 7 days and 3-in-4 nights flag."""

    if not is_valid_timestamp(gdate) or not date_history:
        return 0.0, 0
    gdate = pd.Timestamp(gdate)
    last7 = 0
    last3 = 0
    for d in date_history:
        if not is_valid_timestamp(d):
            continue
        diff = (gdate - pd.Timestamp(d)).days
        if 0 < diff <= 7:
            last7 += 1
        if 0 < diff <= 3:
            last3 += 1
    return float(last7), int(last3 >= 2)


def schedule_density_extended(date_history, gdate):
    """Extended fatigue schedule: last7, 3-in-4, 4-in-6 flags."""

    if not is_valid_timestamp(gdate) or not date_history:
        return 0.0, 0, 0
    gdate = pd.Timestamp(gdate)
    last7 = 0
    last3 = 0
    last5 = 0
    for d in date_history:
        if not is_valid_timestamp(d):
            continue
        diff = (gdate - pd.Timestamp(d)).days
        if 0 < diff <= 7:
            last7 += 1
        if 0 < diff <= 3:
            last3 += 1
        if 0 < diff <= 5:
            last5 += 1
    three_in_four = int(last3 >= 2)
    four_in_six = int(last5 >= 3)
    return float(last7), three_in_four, four_in_six


def rest_bucket_flags(rest_days: int) -> dict:
    """One-hot rest buckets: 0 / 1 / 2 / 3+ days."""
    r = int(rest_days) if rest_days is not None else 3
    return {
        "rest_0": int(r <= 0),
        "rest_1": int(r == 1),
        "rest_2": int(r == 2),
        "rest_3plus": int(r >= 3),
    }


def engine_implied_margins(elo_tracker, hier_engine,
                           ho_off, ho_def, ao_off, ao_def,
                           home_starters, away_starters, exp_poss):
    cfg = getattr(elo_tracker, "cfg", {}) or {}
    scaling = globals().get("ELO_SCALING_FACTOR", 1000) or 1000
    hb = globals().get("HOME_PPP_BOOST", 0.024)
    lx = getattr(elo_tracker, "league_xppp", 1.10)
    exp_ppp_h = lx + hb + (ho_off - ao_def) / scaling
    exp_ppp_a = lx - hb + (ao_off - ho_def) / scaling
    elo_margin = (exp_ppp_h - exp_ppp_a) * float(exp_poss or 0.0)
    try:
        hph, hpa, _, _ = hier_engine.predict_pts(home_starters, away_starters, exp_poss)
        hier_margin = float(hph - hpa)
    except Exception:
        hier_margin = 0.0
    return float(elo_margin), float(hier_margin)


def _top_lineup(weighted_ids, n=5):
    if not weighted_ids:
        return []
    ranked = sorted(weighted_ids, key=lambda x: -x[1])[:n]
    return [p for p, _ in ranked]


In [ ]:
# ── module: availability ─────────────────────────────────────────────────────
"""Injury/availability and missing rotation impact."""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Set

import numpy as np

try:
    import requests
    HAS_REQUESTS = True
except ImportError:
    HAS_REQUESTS = False

STATUS_PROB = {
    "OUT": 0.0, "DOUBTFUL": 0.10, "QUESTIONABLE": 0.50, "PROBABLE": 0.85, "GTD": 0.50,
}


@dataclass
class PlayerAvailability:
    player_id: str
    status: str
    play_prob: float


def status_to_prob(status: str) -> float:
    s = (status or "").upper()
    for key, prob in STATUS_PROB.items():
        if key in s:
            return prob
    return 1.0


def missing_rotation_share(expected_weights: Iterable[tuple], unavailable: Set[str]) -> float:
    """Fraction of expected rotation possessions missing tonight."""
    if not expected_weights:
        return 0.0
    total = sum(w for _, w in expected_weights)
    if total <= 0:
        return 0.0
    missing = sum(w for pid, w in expected_weights if str(pid) in unavailable)
    return float(missing / total)


def star_out_flag(expected_weights: Iterable[tuple], unavailable: Set[str], top_n: int = 2) -> int:
    if not expected_weights:
        return 0
    ranked = sorted(expected_weights, key=lambda x: -x[1])[:top_n]
    for pid, _ in ranked:
        if str(pid) in unavailable:
            return 1
    return 0


def filter_available_weights(expected_weights: List[tuple], unavailable: Set[str],
                             status_map: Optional[Dict[str, float]] = None) -> List[tuple]:
    """Return weights scaled by play probability."""
    out = []
    for pid, w in expected_weights:
        pid = str(pid)
        if pid in unavailable:
            continue
        prob = 1.0
        if status_map and pid in status_map:
            prob = status_map[pid]
        if prob > 0.05:
            out.append((pid, w * prob))
    return out


def fetch_espn_injuries() -> Dict[str, str]:
    """Fetch current NBA injury list from ESPN (best-effort)."""
    if not HAS_REQUESTS:
        return {}
    try:
        url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        data = r.json()
        out = {}
        for team in data.get("injuries", []):
            for inj in team.get("injuries", []):
                name = inj.get("athlete", {}).get("displayName", "")
                status = inj.get("status", "OUT")
                if name:
                    out[name.lower()] = status
        return out
    except Exception:
        return {}


def build_inactive_map_from_pbp(stints_df, game_id_col="GAME_ID") -> dict:
    """Historical proxy: players in rotation history but not in first stint = inactive."""
    return {}

def probabilistic_margin(mean_margin: float, std_margin: float, n_draws: int = 200, seed: int = 42):
    """Simple normal draws for lineup uncertainty."""
    rng = np.random.default_rng(seed)
    draws = rng.normal(mean_margin, max(std_margin, 1.0), n_draws)
    return float(np.mean(draws)), float(np.std(draws))


def expected_availability_impact(
    home_ids,
    away_ids,
    epm_tracker,
    *,
    h_star_out: float = 0,
    a_star_out: float = 0,
    h_missing: float = 0,
    a_missing: float = 0,
    as_of=None,
) -> dict:
    """Estimate offensive/defensive/pace drops from missing stars + EPM priors.

    Binary star-out alone understates impact; scale by prior strength and
    missing-rotation share. `as_of` (Task 032) is forwarded to the EPM
    tracker so a future-dated snapshot cannot influence this game's impact
    estimate.
    """
    def _side_impact(ids, star_out, missing):
        impacts = []
        if epm_tracker is not None and getattr(epm_tracker, "scaled_impact", None):
            impacts = [epm_tracker.scaled_impact(p, as_of=as_of) for p in (ids or []) if p]
        mean_abs = float(np.mean(np.abs(impacts))) if impacts else 25.0
        star = float(star_out or 0)
        miss = float(missing or 0)
        # Points of rating drop (approx ORtg/DRtg units)
        off_drop = star * (0.35 * mean_abs / 50.0) * 4.0 + miss * 3.0
        def_drop = star * (0.25 * mean_abs / 50.0) * 3.0 + miss * 2.0
        # Missing stars often slow or speed pace slightly; bias toward slight slowdown
        pace_delta = -star * 1.5 - miss * 0.8
        return off_drop, def_drop, pace_delta

    h_off, h_def, h_pace = _side_impact(home_ids, h_star_out, h_missing)
    a_off, a_def, a_pace = _side_impact(away_ids, a_star_out, a_missing)
    return {
        "h_expected_off_drop": float(h_off),
        "a_expected_off_drop": float(a_off),
        "h_expected_def_drop": float(h_def),
        "a_expected_def_drop": float(a_def),
        "h_expected_pace_delta": float(h_pace),
        "a_expected_pace_delta": float(a_pace),
        "expected_off_drop_diff": float(h_off - a_off),
        "expected_def_drop_diff": float(h_def - a_def),
        "expected_pace_delta_diff": float(h_pace - a_pace),
    }


In [ ]:
# ── module: epm_priors ───────────────────────────────────────────────────────
"""External player impact priors (EPM/RAPTOR-style CSV blend).

Task 032: every player-metric snapshot now carries an explicit
``observation_date`` (the date the external metric provider computed/
published that value). Snapshots are versioned per player -- a provider may
publish several dated EPM values for the same player over a season -- and an
as-of query (`as_of=gdate`) may only see a snapshot whose
``observation_date <= as_of``. A retrospectively updated/future metric value
must never be joinable to a game that happened before it existed (Rule 3).

When no snapshot qualifies as of the requested cutoff, callers get an
explicit "missing" signal (via `missing()` / the `*_missing` feature flags)
rather than a silent zero, so a genuinely-unavailable metric is
distinguishable from a metric whose real value happens to be zero.
"""
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


EPM_CSV = STATE_DIR / "player_epm_priors.csv"
BLEND_WEIGHT = 0.35

REQUIRED_COLUMNS = ("player_id", "epm", "observation_date")


class EpmPriorSchemaError(ValueError):
    """Raised when a raw EPM snapshot CSV is missing required versioning columns."""


class EpmPriorTracker:
    """Blend internal player Elo with external, date-versioned impact metrics."""

    def __init__(self, csv_path=None, blend: float = BLEND_WEIGHT):
        self.blend = blend
        # player_id -> sorted list of (observation_date, epm), ascending.
        self.snapshots: Dict[str, List[Tuple[pd.Timestamp, float]]] = {}
        path = Path(csv_path) if csv_path else EPM_CSV
        if path.exists():
            try:
                df = pd.read_csv(path)
                self._load_versioned(df)
            except EpmPriorSchemaError:
                raise
            except Exception:
                self.snapshots = {}

    def _load_versioned(self, df: pd.DataFrame) -> None:
        missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
        if missing:
            raise EpmPriorSchemaError(
                f"EPM prior CSV missing required versioning column(s) {missing}; "
                "every player-metric snapshot must carry an observation_date "
                "(Task 032) -- refusing to load an unversioned prior table"
            )
        d = df.copy()
        d["player_id"] = d["player_id"].astype(str)
        d["epm"] = pd.to_numeric(d["epm"], errors="coerce")
        d["observation_date"] = pd.to_datetime(d["observation_date"], errors="coerce")
        n_before = len(d)
        d = d.dropna(subset=["epm", "observation_date"])
        if len(d) < n_before:
            import warnings
            warnings.warn(
                f"EpmPriorTracker: dropped {n_before - len(d)} row(s) with "
                "unparseable epm/observation_date rather than fabricate one",
                stacklevel=2,
            )
        for pid, grp in d.groupby("player_id"):
            rows = sorted(
                (pd.Timestamp(r.observation_date), float(r.epm)) for r in grp.itertuples()
            )
            self.snapshots[pid] = rows

    @property
    def priors(self) -> Dict[str, float]:
        """Backward-compatible view: latest known value per player,
        ignoring any as-of cutoff. Prefer `get_as_of` for anything that
        touches a specific game/prediction."""
        return {pid: rows[-1][1] for pid, rows in self.snapshots.items() if rows}

    def get_as_of(self, player_id: str, as_of=None) -> Optional[float]:
        """Latest EPM snapshot with ``observation_date <= as_of``.

        `as_of=None` means "no cutoff" (latest available value) -- callers
        that have a real prediction timestamp must always pass it explicitly
        so a future snapshot cannot leak into a past game.
        """
        rows = self.snapshots.get(str(player_id))
        if not rows:
            return None
        if as_of is None:
            return rows[-1][1]
        cutoff = pd.Timestamp(as_of)
        eligible = [v for d, v in rows if d <= cutoff]
        if not eligible:
            return None
        return eligible[-1]

    def missing(self, player_id: str, as_of=None) -> bool:
        return self.get_as_of(player_id, as_of=as_of) is None

    def scaled_impact(self, player_id: str, as_of=None) -> float:
        epm = self.get_as_of(player_id, as_of=as_of)
        if epm is None:
            return 0.0
        return epm * 50.0  # scale to Elo-like units

    def blend_lineup_off(self, player_ids, internal_off: float, as_of=None) -> float:
        if not self.snapshots:
            return internal_off
        impacts = [
            self.scaled_impact(p, as_of=as_of) for p in player_ids
            if p and not self.missing(p, as_of=as_of)
        ]
        if not impacts:
            return internal_off
        ext = 1500.0 + float(np.mean(impacts))
        return (1 - self.blend) * internal_off + self.blend * ext

    def feature_dict(self, home_ids, away_ids, ho_off, ao_off, as_of=None):
        if not self.snapshots:
            return {
                "epm_prior_diff": 0.0, "epm_blend_off_diff": 0.0,
                "epm_missing_frac_home": 1.0, "epm_missing_frac_away": 1.0,
            }
        home_ids = [p for p in (home_ids or []) if p]
        away_ids = [p for p in (away_ids or []) if p]

        def _side(ids):
            vals, n_missing = [], 0
            for p in ids:
                v = self.get_as_of(p, as_of=as_of)
                if v is None:
                    n_missing += 1
                else:
                    vals.append(self.scaled_impact(p, as_of=as_of))
            missing_frac = (n_missing / len(ids)) if ids else 1.0
            mean_val = float(np.mean(vals)) if vals else 0.0
            return mean_val, missing_frac

        h_mean, h_missing_frac = _side(home_ids)
        a_mean, a_missing_frac = _side(away_ids)
        return {
            "epm_prior_diff": h_mean - a_mean,
            "epm_blend_off_diff": self.blend * (h_mean - a_mean),
            "epm_missing_frac_home": h_missing_frac,
            "epm_missing_frac_away": a_missing_frac,
        }


In [ ]:
# ── module: travel ───────────────────────────────────────────────────────────
"""Travel distance and timezone fatigue features."""
from __future__ import annotations

import math
from collections import defaultdict, deque

import numpy as np

# Arena coordinates (lat, lon) — approximate city centers
ARENA_COORDS = {
    "ATL": (33.757, -84.401), "BOS": (42.366, -71.062), "BKN": (40.683, -73.975),
    "CHA": (35.225, -80.839), "CHI": (41.881, -87.674), "CLE": (41.496, -81.688),
    "DAL": (32.790, -96.810), "DEN": (39.748, -105.007), "DET": (42.341, -83.055),
    "GSW": (37.768, -122.387), "HOU": (29.750, -95.362), "IND": (39.764, -86.155),
    "LAC": (34.043, -118.267), "LAL": (34.043, -118.267), "MEM": (35.138, -90.051),
    "MIA": (25.781, -80.188), "MIL": (43.045, -87.917), "MIN": (44.979, -93.276),
    "NOP": (29.949, -90.082), "NYK": (40.751, -73.993), "OKC": (35.463, -97.515),
    "ORL": (28.539, -81.384), "PHI": (39.901, -75.172), "PHX": (33.446, -112.071),
    "POR": (45.532, -122.667), "SAC": (38.649, -121.518), "SAS": (29.427, -98.438),
    "TOR": (43.643, -79.379), "UTA": (40.768, -111.901), "WAS": (38.898, -77.021),
}

TZ_OFFSET = {
    "ATL": -5, "BOS": -5, "BKN": -5, "CHA": -5, "CHI": -6, "CLE": -5,
    "DAL": -6, "DEN": -7, "DET": -5, "GSW": -8, "HOU": -6, "IND": -5,
    "LAC": -8, "LAL": -8, "MEM": -6, "MIA": -5, "MIL": -6, "MIN": -6,
    "NOP": -6, "NYK": -5, "OKC": -6, "ORL": -5, "PHI": -5, "PHX": -7,
    "POR": -8, "SAC": -8, "SAS": -6, "TOR": -5, "UTA": -7, "WAS": -5,
}


def haversine_miles(lat1, lon1, lat2, lon2):
    r = 3959.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


class TravelTracker:
    """Rolling travel miles and timezone shifts per team."""

    def __init__(self, window_days: int = 14):
        self.window_days = window_days
        self.history = defaultdict(lambda: deque(maxlen=30))

    def _record(self, team, gdate, lat, lon, tz):
        self.history[team].append({"date": gdate, "lat": lat, "lon": lon, "tz": tz})

    def update_game(self, home, away, gdate, prev_home=None):
        if not isinstance(gdate, np.datetime64) and hasattr(gdate, "to_pydatetime"):
            gdate = gdate
        hc = ARENA_COORDS.get(home, (40.0, -90.0))
        ac = ARENA_COORDS.get(away, (40.0, -90.0))
        self._record(home, gdate, hc[0], hc[1], TZ_OFFSET.get(home, -6))
        self._record(away, gdate, hc[0], hc[1], TZ_OFFSET.get(home, -6))

    def features(self, team, gdate, is_home=True):
        hist = self.history.get(team, deque())
        if not hist or not hasattr(gdate, "days"):
            try:
                import pandas as pd
                gdate = pd.Timestamp(gdate)
            except Exception:
                return self._zeros()
        miles_7 = miles_14 = tz_shift = trip_game = 0.0
        if len(hist) >= 1:
            prev = list(hist)[-1]
            cur = ARENA_COORDS.get(team if is_home else team, prev)
            miles_7 = haversine_miles(prev["lat"], prev["lon"], cur[0], cur[1]) if not is_home else 0
            tz_shift = abs(TZ_OFFSET.get(team, -6) - prev.get("tz", -6))
        recent = [h for h in hist if hasattr(gdate, "__sub__") and (gdate - h["date"]).days <= 7]
        trip_game = float(len(recent))
        for i in range(1, len(recent)):
            miles_14 += haversine_miles(recent[i - 1]["lat"], recent[i - 1]["lon"],
                                        recent[i]["lat"], recent[i]["lon"])
        return {
            "travel_miles_7d": miles_7,
            "travel_miles_14d": miles_14,
            "tz_shift": tz_shift,
            "road_trip_index": trip_game,
        }

    @staticmethod
    def _zeros():
        return {"travel_miles_7d": 0.0, "travel_miles_14d": 0.0, "tz_shift": 0.0, "road_trip_index": 0.0}

    def matchup_features(self, home, away, gdate):
        hf = self.features(home, gdate, is_home=True)
        af = self.features(away, gdate, is_home=False)
        return {
            "h_travel_miles_7d": hf["travel_miles_7d"],
            "a_travel_miles_7d": af["travel_miles_7d"],
            "travel_miles_diff": hf["travel_miles_7d"] - af["travel_miles_7d"],
            "h_tz_shift": hf["tz_shift"],
            "a_tz_shift": af["tz_shift"],
            "h_road_trip": af["road_trip_index"],
            "a_road_trip": af["road_trip_index"],
        }

    def save_state(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump({k: list(v) for k, v in self.history.items()}, f)

    @classmethod
    def load_state(cls, path):
        import pickle
        obj = cls()
        with open(path, "rb") as f:
            data = pickle.load(f)
        obj.history = defaultdict(lambda: deque(maxlen=30), {k: deque(v) for k, v in data.items()})
        return obj


In [ ]:
# ── module: fatigue ──────────────────────────────────────────────────────────
"""Composite fatigue index from schedule density and travel."""
from __future__ import annotations

import numpy as np


def fatigue_index(
    games_last7: float,
    travel_miles_7d: float,
    tz_shift: float,
    *,
    w_density: float = 0.35,
    w_travel: float = 0.35,
    w_tz: float = 0.30,
) -> float:
    density = float(games_last7) / 7.0
    travel = float(travel_miles_7d) / 2000.0
    tz = abs(float(tz_shift)) / 3.0
    return float(w_density * density + w_travel * travel + w_tz * tz)


def fatigue_features(h_games_last7, a_games_last7, h_travel_miles, a_travel_miles,
                     h_tz_shift, a_tz_shift) -> dict:
    h_f = fatigue_index(h_games_last7, h_travel_miles, h_tz_shift)
    a_f = fatigue_index(a_games_last7, a_travel_miles, a_tz_shift)
    return {
        "h_fatigue_index": h_f,
        "a_fatigue_index": a_f,
        "fatigue_diff": h_f - a_f,
    }


In [ ]:
# ── module: team_elo ─────────────────────────────────────────────────────────
"""Team-level Bradley-Terry style Elo for spread and total."""
from __future__ import annotations

from collections import defaultdict

import numpy as np

K_SPREAD = 20.0
K_TOTAL = 15.0
HOME_ADV = 2.5


class TeamEloTracker:
    """Separate spread and scoring strength by team."""

    def __init__(self, default: float = 1500.0):
        self.spread_mu = defaultdict(lambda: default)
        self.total_mu = defaultdict(lambda: default)
        self.games = defaultdict(int)

    def predict_spread_margin(self, home: str, away: str) -> float:
        return (self.spread_mu[home] - self.spread_mu[away]) / 25.0 + HOME_ADV

    def predict_total_adj(self, home: str, away: str) -> float:
        return (self.total_mu[home] + self.total_mu[away] - 3000.0) / 50.0

    def update(self, home: str, away: str, home_margin: float, total_pts: float,
               market_spread=None):
        exp = self.predict_spread_margin(home, away)
        err = home_margin - exp
        k = K_SPREAD / (1 + self.games[home] * 0.02)
        self.spread_mu[home] += k * err
        self.spread_mu[away] -= k * err
        exp_t = 225.0 + self.predict_total_adj(home, away)
        terr = total_pts - exp_t
        kt = K_TOTAL / (1 + self.games[home] * 0.02)
        self.total_mu[home] += kt * terr * 0.5
        self.total_mu[away] += kt * terr * 0.5
        self.games[home] += 1
        self.games[away] += 1

    def feature_dict(self, home: str, away: str):
        return {
            "team_elo_spread": self.predict_spread_margin(home, away),
            "team_elo_total_adj": self.predict_total_adj(home, away),
            "team_elo_net": (self.spread_mu[home] - self.spread_mu[away]) / 25.0,
        }

    def save_state(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump({
                "spread_mu": dict(self.spread_mu),
                "total_mu": dict(self.total_mu),
                "games": dict(self.games),
            }, f)

    @classmethod
    def load_state(cls, path):
        import pickle
        with open(path, "rb") as f:
            data = pickle.load(f)
        obj = cls()
        obj.spread_mu = defaultdict(lambda: 1500.0, data.get("spread_mu", {}))
        obj.total_mu = defaultdict(lambda: 1500.0, data.get("total_mu", {}))
        obj.games = defaultdict(int, data.get("games", {}))
        return obj


In [ ]:
# ── module: lineup_elo ───────────────────────────────────────────────────────
"""Dedicated 5-man lineup Elo with shrinkage toward player-weighted mean."""
from __future__ import annotations

from collections import defaultdict

import numpy as np

MIN_POSSESSIONS = 50
SHRINK_K = 100.0


class LineupEloTracker:
    """Stores offensive/defensive rating per sorted 5-tuple with James-Stein shrinkage."""

    def __init__(self, league_xppp: float = 1.10, scaling: float = 1000.0, home_boost: float = 0.024):
        self.league_xppp = league_xppp
        self.scaling = scaling
        self.home_boost = home_boost
        self.off = defaultdict(float)
        self.dff = defaultdict(float)
        self.poss = defaultdict(float)
        self.chemistry = defaultdict(float)

    @staticmethod
    def _key(player_ids):
        ids = sorted(str(x) for x in player_ids if x and str(x) != "nan")
        return tuple(ids[:5]) if len(ids) >= 5 else tuple(ids)

    def _player_sum(self, player_ids, player_tracker, side="off"):
        if not player_tracker:
            return 1500.0, 1500.0
        off, dff, _ = player_tracker.lineup_stats(player_ids)
        return off, dff

    def lineup_rating(self, player_ids, player_tracker=None):
        key = self._key(player_ids)
        if not key:
            return 0.0, 0.0, 0.0, 0.0
        p_off, p_def = self._player_sum(key, player_tracker)
        n = self.poss.get(key, 0.0)
        if n >= MIN_POSSESSIONS and key in self.off:
            w = n / (n + SHRINK_K)
            off = w * self.off[key] + (1 - w) * (p_off - 1500.0)
            dff = w * self.dff[key] + (1 - w) * (p_def - 1500.0)
            chem = self.chemistry.get(key, 0.0) * w
            return off, dff, chem, n
        return (p_off - 1500.0) * 0.1, (p_def - 1500.0) * 0.1, 0.0, n

    def expected_margin(self, home_ids, away_ids, possessions, player_tracker=None, is_home=True):
        ho, hd, hc, _ = self.lineup_rating(home_ids, player_tracker)
        ao, ad, ac, _ = self.lineup_rating(away_ids, player_tracker)
        hb = self.home_boost if is_home else -self.home_boost
        ppp_h = self.league_xppp + hb + (1500 + ho - (1500 + ad)) / self.scaling
        ppp_a = self.league_xppp - hb + (1500 + ao - (1500 + hd)) / self.scaling
        margin = (ppp_h - ppp_a) * possessions
        chem_net = (hc - ac) * possessions / 100.0
        return margin + chem_net, chem_net

    def update_stint(self, off_ids, def_ids, xpts_off, xpts_def, possessions,
                     player_tracker=None, is_home_offense=True):
        if possessions <= 0:
            return
        key_off = self._key(off_ids)
        if len(key_off) < 5:
            return
        exp_margin, _ = self.expected_margin(off_ids, def_ids, possessions, player_tracker, is_home_offense)
        act_margin = (xpts_off - xpts_def) * possessions / max(possessions, 1)
        err = (xpts_off / possessions - self.league_xppp) * possessions if possessions else 0
        p_off, _ = self._player_sum(key_off, player_tracker)
        residual = err - (p_off - 1500.0) / self.scaling * possessions
        k = 0.15 * possessions / (self.poss[key_off] + possessions + SHRINK_K)
        self.off[key_off] += k * err
        self.dff[key_off] += k * (-err * 0.5)
        self.chemistry[key_off] = 0.9 * self.chemistry.get(key_off, 0) + 0.1 * residual
        self.poss[key_off] += possessions

    def feature_dict(self, home_ids, away_ids, player_tracker=None):
        ho, hd, hc, hn = self.lineup_rating(home_ids, player_tracker)
        ao, ad, ac, an = self.lineup_rating(away_ids, player_tracker)
        return {
            "h_lineup5_off": ho, "h_lineup5_def": hd, "h_lineup5_chem": hc,
            "a_lineup5_off": ao, "a_lineup5_def": ad, "a_lineup5_chem": ac,
            "lineup5_net": (ho - ad) - (ao - hd),
            "lineup5_chem_diff": hc - ac,
            "lineup5_sample_min": min(hn, an),
        }

    def save_state(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump(self.__dict__, f)

    @classmethod
    def load_state(cls, path):
        import pickle
        with open(path, "rb") as f:
            data = pickle.load(f)
        obj = cls()
        obj.__dict__.update(data)
        return obj


In [ ]:
# ── module: chemistry ────────────────────────────────────────────────────────
"""Lineup chemistry: 2/3-man residuals and on/off net ratings from stints."""
from __future__ import annotations

import itertools
from collections import defaultdict

import numpy as np

MIN_DUO_POSS = 50
SHRINK = 80.0


class ChemistryTracker:
    """Tracks pairwise/triple chemistry residuals with empirical-Bayes shrinkage."""

    def __init__(self, league_xppp: float = 1.10):
        self.league_xppp = league_xppp
        self.duo_off = defaultdict(float)
        self.duo_def = defaultdict(float)
        self.duo_poss = defaultdict(float)
        self.trio_off = defaultdict(float)
        self.trio_poss = defaultdict(float)
        self.on_off = defaultdict(lambda: {"on": 0.0, "off": 0.0, "on_p": 0.0, "off_p": 0.0})

    @staticmethod
    def _duo_key(a, b):
        return tuple(sorted((str(a), str(b))))

    @staticmethod
    def _trio_key(ids):
        return tuple(sorted(str(x) for x in ids if x))

    def _shrink(self, raw, n):
        return raw * (n / (n + SHRINK)) if n > 0 else 0.0

    def update_stint(self, off_ids, def_ids, xpts_off, xpts_def, possessions,
                     player_tracker=None, expected_ppp=None):
        if possessions <= 0:
            return
        act_ppp = xpts_off / possessions
        exp = expected_ppp if expected_ppp is not None else self.league_xppp
        residual = act_ppp - exp
        ids = [str(x) for x in off_ids if x and str(x) != "nan"]
        for a, b in itertools.combinations(ids, 2):
            k = self._duo_key(a, b)
            w = 0.08 * possessions / (self.duo_poss[k] + possessions + SHRINK)
            self.duo_off[k] += w * residual
            self.duo_poss[k] += possessions
        if len(ids) >= 3:
            for trio in itertools.combinations(ids, 3):
                tk = self._trio_key(trio)
                w = 0.05 * possessions / (self.trio_poss[tk] + possessions + SHRINK)
                self.trio_off[tk] += w * residual
                self.trio_poss[tk] += possessions
        for pid in ids:
            st = self.on_off[pid]
            st["on"] += act_ppp * possessions
            st["on_p"] += possessions

    def _duo_net(self, lineup):
        ids = [str(x) for x in lineup if x and str(x) != "nan"]
        if len(ids) < 2:
            return 0.0, 0.0
        vals, weights = [], []
        for a, b in itertools.combinations(ids, 2):
            k = self._duo_key(a, b)
            n = self.duo_poss.get(k, 0)
            if n >= MIN_DUO_POSS:
                vals.append(self._shrink(self.duo_off[k], n))
                weights.append(n)
        if not vals:
            return 0.0, 0.0
        return float(np.average(vals, weights=weights)), float(sum(weights))

    def _onoff_net(self, lineup):
        nets = []
        for pid in lineup:
            st = self.on_off.get(str(pid))
            if not st or st["on_p"] < 30:
                continue
            on_ppp = st["on"] / st["on_p"]
            nets.append(on_ppp - self.league_xppp)
        return float(np.mean(nets)) if nets else 0.0

    def _trio_net(self, lineup):
        ids = [str(x) for x in lineup if x and str(x) != "nan"]
        if len(ids) < 3:
            return 0.0, 0.0
        vals, weights = [], []
        for trio in itertools.combinations(ids, 3):
            tk = self._trio_key(trio)
            n = self.trio_poss.get(tk, 0)
            if n >= MIN_DUO_POSS:
                vals.append(self._shrink(self.trio_off[tk], n))
                weights.append(n)
        if not vals:
            return 0.0, 0.0
        return float(np.average(vals, weights=weights)), float(sum(weights))

    def lineup_chemistry(self, lineup, player_tracker=None):
        duo_net, duo_w = self._duo_net(lineup)
        trio_net, _ = self._trio_net(lineup)
        onoff = self._onoff_net(lineup)
        unc = 1.0 / (1.0 + duo_w / 200.0) if duo_w else 1.0
        return duo_net * 100.0, trio_net * 100.0, onoff * 100.0, unc

    def feature_dict(self, home_lineup, away_lineup, player_tracker=None):
        h_duo, h_trio, h_on, h_unc = self.lineup_chemistry(home_lineup, player_tracker)
        a_duo, a_trio, a_on, a_unc = self.lineup_chemistry(away_lineup, player_tracker)
        return {
            "h_chem_net": h_duo + h_on,
            "a_chem_net": a_duo + a_on,
            "chem_diff": (h_duo + h_on) - (a_duo + a_on),
            "h_chem_duo_net": h_duo,
            "a_chem_duo_net": a_duo,
            "h_chem_trio_net": h_trio,
            "a_chem_trio_net": a_trio,
            "h_onoff_net": h_on,
            "a_onoff_net": a_on,
            "chem_uncertainty": h_unc + a_unc,
        }

    def usage_conflict(self, weighted_ids):
        """High-usage overlap penalty when multiple high-minute players share floor."""
        if not weighted_ids:
            return 0.0
        pairs = sorted(weighted_ids, key=lambda x: -x[1])[:4]
        if len(pairs) < 2:
            return 0.0
        top_w = pairs[0][1]
        conflict = sum(w for _, w in pairs[1:] if w > 0.15 * top_w)
        return min(1.0, conflict)

    def save_state(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump({
                "duo_off": dict(self.duo_off), "duo_poss": dict(self.duo_poss),
                "trio_off": dict(self.trio_off), "trio_poss": dict(self.trio_poss),
                "on_off": {k: dict(v) for k, v in self.on_off.items()},
                "league_xppp": self.league_xppp,
            }, f)

    @classmethod
    def load_state(cls, path):
        import pickle
        with open(path, "rb") as f:
            data = pickle.load(f)
        obj = cls(league_xppp=data.get("league_xppp", 1.10))
        obj.duo_off = defaultdict(float, data.get("duo_off", {}))
        obj.duo_poss = defaultdict(float, data.get("duo_poss", {}))
        obj.trio_off = defaultdict(float, data.get("trio_off", {}))
        obj.trio_poss = defaultdict(float, data.get("trio_poss", {}))
        obj.on_off = defaultdict(
            lambda: {"on": 0.0, "off": 0.0, "on_p": 0.0, "off_p": 0.0},
            {k: defaultdict(float, v) for k, v in data.get("on_off", {}).items()},
        )
        return obj


In [ ]:
# ── module: lineup_composite ─────────────────────────────────────────────────
"""Composite lineup rating from player Elo, chemistry, and lineup5 Elo."""
from __future__ import annotations

# lineup5 weight is fixed at 0.50 minimum per project spec
DEFAULT_WEIGHTS = {
    "lineup5": 0.50,
    "player": 0.25,
    "duo": 0.15,
    "trio": 0.10,
}


def composite_lineup_rating(
    *,
    player_off_delta: float,
    player_def_delta: float,
    lineup5_net: float,
    chem_duo_net: float,
    chem_trio_net: float = 0.0,
    weights: dict | None = None,
) -> float:
    """Single lineup-strength scalar in ~PPP-margin units."""
    w = dict(DEFAULT_WEIGHTS)
    if weights:
        w.update(weights)
    w["lineup5"] = max(0.50, float(w.get("lineup5", 0.50)))
    rem = 1.0 - w["lineup5"]
    other = w.get("player", 0.25) + w.get("duo", 0.15) + w.get("trio", 0.10)
    if other > 0:
        scale = rem / other
        w["player"] = w.get("player", 0.25) * scale
        w["duo"] = w.get("duo", 0.15) * scale
        w["trio"] = w.get("trio", 0.10) * scale

    player_component = (float(player_off_delta) - float(player_def_delta)) / 1000.0
    lineup5_component = float(lineup5_net) / 100.0
    duo_component = float(chem_duo_net) / 100.0
    trio_component = float(chem_trio_net) / 100.0

    return (
        w["player"] * player_component
        + w["duo"] * duo_component
        + w["trio"] * trio_component
        + w["lineup5"] * lineup5_component
    )


In [ ]:
# ── module: hapm ─────────────────────────────────────────────────────────────
"""Batch HAPM (sparse Ridge) priors for dyad/trio chemistry — optional meta feature."""
from __future__ import annotations

import itertools
from collections import defaultdict

import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge

SHRINK = 80.0


class HapmPriorTracker:
    """Offline-fit dyad/trio coefficients; lookup at feature time."""

    def __init__(self, alpha: float = 100.0, league_xppp: float = 1.10):
        self.alpha = alpha
        self.league_xppp = league_xppp
        self.vectorizer = DictVectorizer(sparse=True)
        self.model = Ridge(alpha=alpha, fit_intercept=True)
        self.fitted = False
        self._dyad_coef = {}
        self._trio_coef = {}

    @staticmethod
    def _parse_lineup(s) -> list[str]:
        return [p for p in str(s).split("-") if p and p != "nan"]

    def fit(self, stints_df) -> bool:
        if stints_df is None or stints_df.empty:
            return False
        rows, y, w = [], [], []
        for _, row in stints_df.iterrows():
            poss = float(row.get("possessions", 0) or 0)
            if poss < 1:
                continue
            xh = float(row.get("home_xpts", 0) or 0) / poss
            feat = {}
            home = self._parse_lineup(row.get("HOME_players", ""))
            for a, b in itertools.combinations(home, 2):
                feat[f"d:{min(a,b)}|{max(a,b)}"] = 1
            if len(home) >= 3:
                for trio in itertools.combinations(sorted(home), 3):
                    feat[f"t:{trio[0]}|{trio[1]}|{trio[2]}"] = 1
            if not feat:
                continue
            rows.append(feat)
            y.append(xh - self.league_xppp)
            w.append(poss)
        if len(rows) < 100:
            return False
        X = self.vectorizer.fit_transform(rows)
        self.model.fit(X, np.array(y), sample_weight=np.array(w))
        self.fitted = True
        names = self.vectorizer.get_feature_names_out()
        for name, coef in zip(names, self.model.coef_):
            if name.startswith("d:"):
                self._dyad_coef[name[2:]] = float(coef)
            elif name.startswith("t:"):
                self._trio_coef[name[2:]] = float(coef)
        return True

    def _lineup_prior(self, lineup) -> float:
        if not self.fitted:
            return 0.0
        ids = [str(x) for x in lineup if x and str(x) != "nan"]
        vals = []
        for a, b in itertools.combinations(ids, 2):
            k = f"{min(a,b)}|{max(a,b)}"
            if k in self._dyad_coef:
                vals.append(self._dyad_coef[k] * (SHRINK / (SHRINK + 50)))
        if len(ids) >= 3:
            for trio in itertools.combinations(sorted(ids), 3):
                k = f"{trio[0]}|{trio[1]}|{trio[2]}"
                if k in self._trio_coef:
                    vals.append(self._trio_coef[k] * (SHRINK / (SHRINK + 50)))
        return float(np.mean(vals) * 100.0) if vals else 0.0

    def feature_dict(self, home_lineup, away_lineup, game_id=None) -> dict:
        # `game_id` is accepted (and ignored) so this tracker is a drop-in
        # replacement for `PastOnlyHapmLookup.feature_dict` below at every
        # call site (`pipeline/game_features.py`), regardless of whether a
        # single static tracker or a per-game past-only lookup is in use.
        h = self._lineup_prior(home_lineup)
        a = self._lineup_prior(away_lineup)
        return {"h_hapm_net": h, "a_hapm_net": a, "hapm_net_diff": h - a}


def build_past_only_hapm_trackers(stints_df, game_dates: dict, n_blocks: int = 6, alpha: float = 100.0):
    """Sequence of past-only-fit HAPM trackers (Task 052).

    ``HapmPriorTracker.fit`` is a single batch Ridge fit; calling it once on
    an entire season and then using the same fitted tracker as a static
    lookup for every game in that season (the current production pattern in
    ``pipeline/backtest.py``) means an early-season game's HAPM feature is
    informed by dyad/trio coefficients estimated from *later* games in the
    same span — the ``HAPM-before-split`` leak named in the plan's
    "Immediate trust assessment".

    This helper performs the "past-only cross-fitting" the plan calls for:
    games are grouped into ``n_blocks`` chronological blocks (by
    ``game_dates``); a fresh tracker is fit on stints from blocks strictly
    before block ``i`` and used only for games in block ``i``. Block 0 has
    no strictly-earlier block, so it gets an unfitted tracker (returns the
    same all-zero feature dict as ``use_hapm_priors=False`` — never a
    silently invented prior).

    Parameters
    ----------
    stints_df : DataFrame with ``GAME_ID`` and the stint columns
        ``HapmPriorTracker.fit`` consumes.
    game_dates : dict mapping GAME_ID -> orderable timestamp for every game
        that will request a tracker (not just games present in stints_df).
    n_blocks : number of chronological blocks.

    Returns
    -------
    (game_id_to_tracker, block_fit_max_timestamp) : dict, dict
        ``game_id_to_tracker[gid]`` is the tracker whose ``.feature_dict``
        must be used for that game; ``block_fit_max_timestamp[gid]`` is the
        max timestamp of stints used to fit it (``None`` for block 0 —
        Rule 5: never invent a prior instead of reporting "no fit yet").
    """
    if not game_dates:
        return {}, {}
    ordered_gids = sorted(game_dates, key=lambda g: game_dates[g])
    n = len(ordered_gids)
    n_blocks = max(1, min(n_blocks, n))
    block_sizes = [n // n_blocks + (1 if i < n % n_blocks else 0) for i in range(n_blocks)]
    boundaries = [0]
    for bs in block_sizes:
        boundaries.append(boundaries[-1] + bs)
    blocks = [ordered_gids[boundaries[i]:boundaries[i + 1]] for i in range(n_blocks)]

    game_id_to_tracker: dict = {}
    block_fit_max_timestamp: dict = {}
    prior_gids: list = []
    prior_max_ts = None
    for block in blocks:
        if prior_gids:
            tracker = HapmPriorTracker(alpha=alpha)
            past_stints = stints_df[stints_df["GAME_ID"].isin(prior_gids)]
            tracker.fit(past_stints)
        else:
            tracker = HapmPriorTracker(alpha=alpha)  # unfitted: zero features
        for gid in block:
            game_id_to_tracker[gid] = tracker
            block_fit_max_timestamp[gid] = prior_max_ts
        prior_gids = prior_gids + block
        block_max = max((game_dates[g] for g in block), default=prior_max_ts)
        prior_max_ts = block_max if prior_max_ts is None else max(prior_max_ts, block_max)
    return game_id_to_tracker, block_fit_max_timestamp


class PastOnlyHapmLookup:
    """Feature-time facade over the per-game past-only-fit trackers from
    ``build_past_only_hapm_trackers`` (integration fix for the
    ``hapm_before_split_leak`` registry entry).

    ``pipeline/game_features.py`` calls ``hapm_tracker.feature_dict(...)``
    with a single tracker shared across every game in a feature-generation
    pass. That single-tracker interface is exactly the leak: one Ridge fit
    reused as a static lookup for every game in the span, including games
    strictly earlier than some of the games it was fit on. This class keeps
    the same call signature but dispatches each call to whichever
    strictly-past-only block tracker is correct for that specific
    ``game_id``, so no early game's HAPM feature can depend on a later
    game's stints.

    ``self.fitted`` is always ``True`` so ``pipeline/game_features.py``'s
    ``getattr(hapm_tracker, "fitted", False)`` guard always dispatches into
    ``feature_dict`` — per-game "no strictly-earlier block yet" handling
    (Rule 5: never invent a prior) happens inside ``feature_dict`` itself,
    which returns the same all-zero dict as ``use_hapm_priors=False``.
    """

    def __init__(self, game_id_to_tracker: dict, fit_max_timestamp: dict):
        self.game_id_to_tracker = game_id_to_tracker
        self.fit_max_timestamp = fit_max_timestamp
        self.fitted = True

    def feature_dict(self, home_lineup, away_lineup, game_id=None) -> dict:
        tracker = self.game_id_to_tracker.get(game_id)
        if tracker is None or not tracker.fitted:
            return {"h_hapm_net": 0.0, "a_hapm_net": 0.0, "hapm_net_diff": 0.0}
        return tracker.feature_dict(home_lineup, away_lineup, game_id=game_id)


def hapm_lookup_for_training(stints_df, n_blocks: int = 6, alpha: float = 100.0) -> "PastOnlyHapmLookup":
    """Build the past-only HAPM lookup for ``pipeline/backtest.py``'s
    feature-generation call sites.

    Both the base-training slice and the calibration slice fed to
    ``pipeline.features.generate_features`` are subsets of the same
    ``train_stints`` span, so a single lookup built from the full span
    covers both call sites correctly — each game still only ever resolves
    to a tracker fit on strictly-earlier games within that span.
    """
    if stints_df is None or stints_df.empty:
        return PastOnlyHapmLookup({}, {})
    game_dates = (
        stints_df[["GAME_ID", "game_date"]]
        .drop_duplicates("GAME_ID")
        .set_index("GAME_ID")["game_date"]
        .to_dict()
    )
    trackers, fit_max_ts = build_past_only_hapm_trackers(
        stints_df, game_dates, n_blocks=n_blocks, alpha=alpha,
    )
    return PastOnlyHapmLookup(trackers, fit_max_ts)


In [ ]:
# ── module: refs ─────────────────────────────────────────────────────────────
"""Referee crew tendency features (rolling from historical assignments)."""
from __future__ import annotations

from collections import defaultdict, deque

import numpy as np


class RefTracker:
    """Track foul/pace bias by ref crew id when assignment data is available."""

    def __init__(self, window: int = 50):
        self.window = window
        self.crew_pace = defaultdict(lambda: deque(maxlen=window))
        self.crew_foul = defaultdict(lambda: deque(maxlen=window))

    def update(self, crew_id: str, total_pts: float, fta: float, possessions: float):
        if not crew_id or possessions <= 0:
            return
        pace = possessions * 2  # approx game pace
        foul_rate = fta / possessions
        self.crew_pace[crew_id].append(pace)
        self.crew_foul[crew_id].append(foul_rate)

    def features(self, crew_id: str = None):
        if not crew_id or crew_id not in self.crew_pace:
            return {"ref_pace_bias": 0.0, "ref_foul_bias": 0.0}
        pace = np.mean(self.crew_pace[crew_id]) if self.crew_pace[crew_id] else 0.0
        foul = np.mean(self.crew_foul[crew_id]) if self.crew_foul[crew_id] else 0.0
        return {
            "ref_pace_bias": pace - 100.0,
            "ref_foul_bias": foul - 0.25,
        }


In [ ]:
# ── module: calibrators ──────────────────────────────────────────────────────
# Cell 10b – Rolling Isotonic Calibrator (dynamic probability calibration)
import numpy as np
from sklearn.isotonic import IsotonicRegression
from collections import deque

class RollingCalibrator:
    """
    Maintains a rolling window of (raw_prediction, actual_outcome) pairs.
    Fits an IsotonicRegression on the last `window_size` games.
    Predicts calibrated probability for new raw predictions.
    """
    def __init__(self, window_size=250, out_of_bounds='clip'):
        self.window_size = window_size
        self.out_of_bounds = out_of_bounds
        self.buffer = deque(maxlen=window_size)
        self.model = None

    def update(self, raw_prob, outcome):
        """Add a new calibration point (raw_prob in [0,1], outcome 0/1)."""
        self.buffer.append((raw_prob, outcome))
        if len(self.buffer) >= 20:   # refit only after enough samples
            self._fit()

    def _fit(self):
        if len(self.buffer) < 5:
            return
        X = np.array([p for p, _ in self.buffer]).reshape(-1, 1)
        y = np.array([o for _, o in self.buffer])
        iso = IsotonicRegression(out_of_bounds=self.out_of_bounds, increasing=True)
        iso.fit(X.ravel(), y)
        self.model = iso

    def predict(self, raw_prob):
        """Calibrate a single raw probability (clips to [0.01,0.99])."""
        if self.model is None or len(self.buffer) < 5:
            return np.clip(raw_prob, 0.01, 0.99)
        raw = np.clip(raw_prob, 0.0, 1.0)
        cal = self.model.predict([raw])[0]
        return np.clip(cal, 0.01, 0.99)
from sklearn.linear_model import LogisticRegression
from collections import deque

class RollingPlattCalibrator:
    """
    Rolling Platt scaling: fits a logistic regression on a rolling window
    of (raw_margin, outcome) to calibrate win probabilities.
    """
    def __init__(self, window_size=300, min_samples=20, cold_start_fn=None,
                 fallback_scale=12.0, variance_aware=True, ref_total=225.0):
        self.window_size = window_size
        self.min_samples = min_samples
        self.buffer = deque(maxlen=window_size)   # stores (adj_margin, outcome)
        self.model = None
        # Cold-start: before enough in-season points accrue, defer to the
        # model's trained calibrator (a single coherent path) instead of a
        # bare sigmoid. Falls back to sigmoid(margin / fallback_scale).
        self.cold_start_fn = cold_start_fn
        self.fallback_scale = fallback_scale
        # Variance-aware: a given margin is less decisive in a high-scoring
        # (higher-variance) game, so scale the margin by sqrt(total / ref_total)
        # before calibrating. ref_total ~ league-average game total.
        self.variance_aware = variance_aware
        self.ref_total = ref_total

    def _adj(self, raw_margin, total):
        if self.variance_aware and total and total > 0:
            return raw_margin / np.sqrt(max(float(total), 1.0) / self.ref_total)
        return raw_margin

    def update(self, raw_margin, outcome, total=None):
        """Add a new calibration point (raw margin, 0/1 outcome, optional total)."""
        self.buffer.append((self._adj(raw_margin, total), outcome))
        if len(self.buffer) >= self.min_samples:
            self._fit()

    def _fit(self):
        X = np.array([x for x, _ in self.buffer]).reshape(-1, 1)
        y = np.array([y for _, y in self.buffer])
        # Use strong L2 regularization to avoid overfitting
        clf = LogisticRegression(C=0.1, solver='lbfgs', max_iter=100)
        clf.fit(X, y)
        self.model = clf

    def predict(self, raw_margin, total=None):
        """Return calibrated win probability (clipped to [0.01,0.99])."""
        if self.model is None or len(self.buffer) < self.min_samples:
            if self.cold_start_fn is not None:
                try:
                    return float(np.clip(self.cold_start_fn(raw_margin), 0.01, 0.99))
                except Exception:
                    pass
            # Fallback: sigmoid with scaling
            p = 1.0 / (1.0 + np.exp(-raw_margin / self.fallback_scale))
            return np.clip(p, 0.01, 0.99)
        prob = self.model.predict_proba(np.array([[self._adj(raw_margin, total)]]))[0, 1]
        return np.clip(prob, 0.01, 0.99)


In [ ]:
# ── module: stint_context ────────────────────────────────────────────────────
"""Build stint context for context-aware Elo rating updates."""
from __future__ import annotations

import pandas as pd


def _get(row, key, default=0.0):
    if isinstance(row, dict):
        return row.get(key, default)
    return getattr(row, key, default) if hasattr(row, key) else default


def build_stint_context(row) -> dict:
    """Aggregate stint-level box stats into a context dict for process_stint."""
    poss = float(_get(row, "possessions", 1) or 1)
    if poss < 1:
        poss = 1.0

    period = int(_get(row, "PERIOD", 1) or 1)
    start_h = float(_get(row, "HOME_SCORE_START", 0) or 0)
    start_a = float(_get(row, "AWAY_SCORE_START", 0) or 0)
    margin_start = start_h - start_a

    home_pts = float(_get(row, "home_pts", 0) or 0)
    away_pts = float(_get(row, "away_pts", 0) or 0)
    home_xpts = float(_get(row, "home_xpts", 0) or 0)
    away_xpts = float(_get(row, "away_xpts", 0) or 0)

    home_tov = float(_get(row, "home_tov", 0) or 0)
    away_tov = float(_get(row, "away_tov", 0) or 0)
    home_stl = float(_get(row, "home_stl", 0) or 0)
    away_stl = float(_get(row, "away_stl", 0) or 0)
    home_blks = float(_get(row, "home_blks", 0) or 0)
    away_blks = float(_get(row, "away_blks", 0) or 0)
    home_tovs_forced = float(_get(row, "home_tovs_forced", 0) or 0)
    away_tovs_forced = float(_get(row, "away_tovs_forced", 0) or 0)
    home_fouls_drawn = float(_get(row, "home_fouls_drawn", 0) or 0)
    away_fouls_drawn = float(_get(row, "away_fouls_drawn", 0) or 0)
    home_fta = float(_get(row, "home_fta", 0) or 0)
    away_fta = float(_get(row, "away_fta", 0) or 0)
    home_fga = float(_get(row, "home_fga", 0) or 0)
    away_fga = float(_get(row, "away_fga", 0) or 0)
    home_3pa = float(_get(row, "home_3pa", 0) or 0)
    away_3pa = float(_get(row, "away_3pa", 0) or 0)
    home_rim_fga = float(_get(row, "home_rim_fga", 0) or 0)
    away_rim_fga = float(_get(row, "away_rim_fga", 0) or 0)

    clutch = period >= 4 and abs(margin_start) < 10
    garbage_flag = bool(_get(row, "garbage", False))

    return {
        "poss": poss,
        "clutch": clutch,
        "garbage": garbage_flag,
        "margin_start": margin_start,
        "home_tov_rate": home_tov / poss,
        "away_tov_rate": away_tov / poss,
        "home_def_events": (home_stl + home_blks + home_tovs_forced) / poss,
        "away_def_events": (away_stl + away_blks + away_tovs_forced) / poss,
        "home_foul_draw_rate": (home_fouls_drawn + home_fta * 0.5) / poss,
        "away_foul_draw_rate": (away_fouls_drawn + away_fta * 0.5) / poss,
        "home_3pa_rate": home_3pa / max(home_fga, 1.0),
        "away_3pa_rate": away_3pa / max(away_fga, 1.0),
        "home_rim_rate": home_rim_fga / max(home_fga, 1.0),
        "away_rim_rate": away_rim_fga / max(away_fga, 1.0),
        "home_luck_ppp": (home_pts - home_xpts) / poss,
        "away_luck_ppp": (away_pts - away_xpts) / poss,
        "home_pts": home_pts,
        "away_pts": away_pts,
        "home_xpts": home_xpts,
        "away_xpts": away_xpts,
    }


def extract_crew_id(group) -> str | None:
    """First non-null official/crew id from a game stint group."""
    for col in ("official", "crew_id", "CREW_ID"):
        if col in group.columns:
            vals = group[col].dropna()
            if len(vals):
                return str(vals.iloc[0])
    return None


In [ ]:
# ── module: game_updates ─────────────────────────────────────────────────────
"""Shared post-game tracker updates for training and simulation."""
from __future__ import annotations

from collections import defaultdict



def update_trackers_after_game(
    *,
    group,
    gs,
    game_id,
    gdate,
    home,
    away,
    home_starters,
    away_starters,
    act_h,  # Task 014: canonical final score when available (see teamstats.precompute_game_team_stats); stint-summed fallback otherwise. Never re-derive from filtered stints here.
    act_a,
    home_tot_poss,
    away_tot_poss,
    home_xppp_game,
    away_xppp_game,
    current_season,
    market_spread,
    elo_tracker,
    hier_engine,
    pace_tracker,
    team_xppp_tracker=None,
    team_form_tracker=None,
    rotation_tracker=None,
    lineup_elo_tracker=None,
    chemistry_tracker=None,
    team_elo_tracker=None,
    travel_tracker=None,
    ref_tracker=None,
    shot_quality_tracker=None,
    hapm_tracker=None,
    last_game_date=None,
    team_game_dates=None,
    team_recent_net=None,
    opponent_history=None,
    team_home_margin=None,
    team_road_margin=None,
    team_games_played=None,
    team_rosters_seen=None,
    lineup_cache=None,
    season_player_ids=None,
    seas_prog=0.5,
    ho_off=None,
    ho_def=None,
    ao_off=None,
    ao_def=None,
    h_form=None,
    a_form=None,
):
    """Apply post-game updates shared by generate_features and run_simulation."""
    if team_games_played is not None:
        team_games_played[home] += 1
        team_games_played[away] += 1
    if team_rosters_seen is not None:
        team_rosters_seen[home].update(home_starters)
        team_rosters_seen[away].update(away_starters)

    raw_margin = act_h - act_a
    if team_recent_net is not None:
        team_recent_net[home].append(raw_margin)
        team_recent_net[away].append(-raw_margin)
    if team_home_margin is not None:
        team_home_margin[home].append(raw_margin)
    if team_road_margin is not None:
        team_road_margin[away].append(-raw_margin)

    if opponent_history is not None and ho_off is not None and ao_off is not None:
        if USE_DECAYED_SOS:
            opponent_history[home].append((gdate, ao_off - ao_def))
            opponent_history[away].append((gdate, ho_off - ho_def))
        else:
            opponent_history[home].append((away, ao_off - ao_def))
            opponent_history[away].append((home, ho_off - ho_def))

    for _, row in group.iterrows():
        hp = _parse_player_string(row.get("HOME_players", ""))
        ap = _parse_player_string(row.get("AWAY_players", ""))
        p = float(row.get("possessions", 1))
        xh = float(row.get("home_xpts", 0))
        xa = float(row.get("away_xpts", 0))
        uh = row.get("home_usage", {}) or {}
        ua = row.get("away_usage", {}) or {}
        per = int(row.get("PERIOD", 1))
        ss = float(row.get("HOME_SCORE_START", 0))
        as_ = float(row.get("AWAY_SCORE_START", 0))
        se = float(row.get("HOME_SCORE_END", 0))
        ae = float(row.get("AWAY_SCORE_END", 0))
        rh = float(row.get("home_pts", 0))
        ra = float(row.get("away_pts", 0))

        if hasattr(elo_tracker, "process_stint"):
            stint_ctx = build_stint_context(row)
            elo_tracker.process_stint(
                ids_A=hp, ids_B=ap, poss=p,
                xpts_A=xh, xpts_B=xa,
                usage_A=uh, usage_B=ua,
                period=per,
                start_A=ss, start_B=as_,
                end_A=se, end_B=ae,
                season_progress=seas_prog,
                stint_ctx=stint_ctx,
            )
        if hasattr(hier_engine, "update"):
            hier_engine.update(hp, ap, rh, ra, p, xpts_off=xh, xpts_def=xa)
        if chemistry_tracker is not None:
            chemistry_tracker.update_stint(hp, ap, xh, xa, p, elo_tracker)
        if lineup_elo_tracker is not None:
            lineup_elo_tracker.update_stint(hp, ap, xh, xa, p, elo_tracker, True)

    if team_xppp_tracker is not None:
        team_xppp_tracker.update(home, gdate, current_season, home_xppp_game, away_xppp_game)
        team_xppp_tracker.update(away, gdate, current_season, away_xppp_game, home_xppp_game)

    if team_form_tracker is not None:

        form_weight = 1.0
        if USE_GARBAGE_WEIGHTED_FORM and group is not None and "garbage" in group.columns:
            if "possessions" in group.columns:
                tot_p = float(group["possessions"].sum())
                garb_p = float(group.loc[group["garbage"], "possessions"].sum()) if tot_p > 0 else 0.0
                form_weight = max(0.3, 1.0 - garb_p / tot_p) if tot_p > 0 else 1.0
            else:
                form_weight = max(0.3, 1.0 - float(group["garbage"].mean()))
        if h_form is None:
            h_form = team_form_tracker.get(home, current_season, gdate)
        if a_form is None:
            a_form = team_form_tracker.get(away, current_season, gdate)
        team_form_tracker.update(
            home, gdate, current_season, team_game_form(gs, "home"),
            opp_off=a_form.get("off_rtg"), opp_def=a_form.get("def_rtg"),
            weight=form_weight,
        )
        team_form_tracker.update(
            away, gdate, current_season, team_game_form(gs, "away"),
            opp_off=h_form.get("off_rtg"), opp_def=h_form.get("def_rtg"),
            weight=form_weight,
        )

    if team_elo_tracker is not None:
        team_elo_tracker.update(home, away, act_h - act_a, act_h + act_a, market_spread)
    if travel_tracker is not None:
        travel_tracker.update_game(home, away, gdate)

    if last_game_date is not None:
        last_game_date[home] = gdate
        last_game_date[away] = gdate
    if team_game_dates is not None:
        team_game_dates[home].append(gdate)
        team_game_dates[away].append(gdate)

    if hasattr(pace_tracker, "update_pace"):
        pace_tracker.update_pace(home, away, home_tot_poss, away_tot_poss)

    if ref_tracker is not None:
        crew_id = extract_crew_id(group)
        tot_fta = float(gs.get("home_fta", 0) or 0) + float(gs.get("away_fta", 0) or 0)
        tot_poss = float(gs.get("tot_poss", 0) or 0) or (home_tot_poss + away_tot_poss)
        if crew_id:
            ref_tracker.update(crew_id, act_h + act_a, tot_fta, tot_poss)

    if shot_quality_tracker is not None and gs:
        shot_quality_tracker.update(home, gdate, current_season, shot_quality_stats_from_gs(gs, "home"))
        shot_quality_tracker.update(away, gdate, current_season, shot_quality_stats_from_gs(gs, "away"))

    if rotation_tracker is not None:
        h_poss = defaultdict(float)
        a_poss = defaultdict(float)
        for _, row in group.iterrows():
            p = float(row.get("possessions", 0))
            for pid in _parse_player_string(row.get("HOME_players", "")):
                h_poss[str(pid)] += p
            for pid in _parse_player_string(row.get("AWAY_players", "")):
                a_poss[str(pid)] += p
        if h_poss:
            rotation_tracker.update_game(home, list(h_poss.keys()), list(h_poss.values()))
        if a_poss:
            rotation_tracker.update_game(away, list(a_poss.keys()), list(a_poss.values()))
        if season_player_ids is not None:
            for pid in h_poss:
                season_player_ids.add(str(pid))
            for pid in a_poss:
                season_player_ids.add(str(pid))

    if lineup_cache is not None:
        actual_first = group.iloc[0]
        lineup_cache[home] = _parse_player_string(actual_first.get("HOME_players", ""))
        lineup_cache[away] = _parse_player_string(actual_first.get("AWAY_players", ""))


In [ ]:
# ── module: elo_calibration ──────────────────────────────────────────────────
"""Walk-forward calibration for ELO-derived margins (unseen-data generalization)."""
from __future__ import annotations

import pickle
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import HuberRegressor
from sklearn.preprocessing import StandardScaler


# Three leak-free calibration groups (fit walk-forward, chronological).
ELO_CALIB_GROUPS = {
    "engine": [
        "elo_margin",
        "hier_margin",
        "lineup5_net",
        "team_elo_spread",
        "exp_poss",
        "elo_net",
        "elo_luck_adj_net",
        "elo_def_event_rate",
        "elo_tov_rate",
        "elo_matchup_asym",
    ],
    "uncertainty": [
        "uncertainty_diff",
        "h_rating_uncertainty",
        "a_rating_uncertainty",
        "elo_uncertainty_adj",
    ],
    "market": [
        "market_spread",
    ],
}

ELO_CALIBRATED_COLS = [
    "elo_margin_calibrated",
    "elo_hier_blend",
    "elo_vs_hier_spread",
    "elo_vs_market",
]


@dataclass
class EloCalibrationKnobs:
    """Tuned, walk-forward-safe defaults for ELO calibration + meta anchor."""

    huber_epsilon: float = 1.35
    min_samples: int = 80
    refit_g2_isotonic: bool = True
    hier_blend_elo: float = 0.6
    hier_blend_hier: float = 0.4
    elo_blend_alpha: float = 0.35
    elo_ridge_alpha: float = 3.0
    group_train_mae: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict | None) -> "EloCalibrationKnobs":
        if not d:
            return cls()
        known = {k: v for k, v in d.items() if k in cls.__dataclass_fields__}
        return cls(**known)


def _feat_vector(feat: dict, cols: list) -> np.ndarray:
    row = []
    for c in cols:
        v = feat.get(c, 0.0)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            v = 0.0
        row.append(float(v))
    return np.asarray(row, dtype=float)


def _group_matrix(df: pd.DataFrame, cols: list) -> tuple[np.ndarray, list]:
    avail = [c for c in cols if c in df.columns]
    if not avail:
        return np.zeros((len(df), 1)), avail
    return df[avail].fillna(0.0).to_numpy(dtype=float), avail


def _ensure_uncertainty_col(df: pd.DataFrame) -> pd.DataFrame:
    if "elo_uncertainty_adj" in df.columns:
        return df
    out = df.copy()
    tu = out.get("h_rating_uncertainty", pd.Series(350, index=out.index)).fillna(350) + \
         out.get("a_rating_uncertainty", pd.Series(350, index=out.index)).fillna(350)
    out["elo_uncertainty_adj"] = out.get("elo_net", 0).fillna(0) / (tu + 1e-6)
    return out


class WalkForwardEloCalibrator:
    """
    Three-group walk-forward ELO margin calibrator.

    Group 1 (engine): player/hier/lineup/team ratings → base margin.
    Group 2 (uncertainty): rating uncertainty adjustments (Huber on train,
        isotonic re-fit on calib slice after groups 1 & 3 are fixed).
    Group 3 (market): closing spread anchor on remaining residual.

    All groups use only pre-game features; calib slice is strictly later games.
    """

    def __init__(self, knobs: EloCalibrationKnobs | None = None):
        self.knobs = knobs or EloCalibrationKnobs()
        self.group_cols: dict[str, list] = {k: list(v) for k, v in ELO_CALIB_GROUPS.items()}
        self.scalers: dict[str, StandardScaler] = {
            g: StandardScaler() for g in ELO_CALIB_GROUPS
        }
        self.models: dict[str, HuberRegressor | None] = {
            g: None for g in ELO_CALIB_GROUPS
        }
        self.g2_isotonic: IsotonicRegression | None = None
        self.fitted = False
        self.coef_report: dict = {}

    def _huber(self) -> HuberRegressor:
        return HuberRegressor(
            epsilon=self.knobs.huber_epsilon,
            max_iter=300,
        )

    def _fit_group(
        self,
        df: pd.DataFrame,
        group: str,
        target: np.ndarray,
    ) -> tuple[np.ndarray, list]:
        X, cols = _group_matrix(df, self.group_cols[group])
        if len(cols) == 0 or len(target) < max(20, self.knobs.min_samples // 4):
            return np.zeros(len(df)), cols
        self.group_cols[group] = cols
        Xs = self.scalers[group].fit_transform(X)
        model = self._huber()
        model.fit(Xs, target)
        self.models[group] = model
        pred = model.predict(Xs)
        self.coef_report[group] = dict(zip(cols, model.coef_.tolist()))
        self.coef_report[f"{group}_intercept"] = float(model.intercept_)
        return pred, cols

    def _predict_group(
        self,
        df: pd.DataFrame,
        group: str,
        *,
        use_isotonic: bool = False,
    ) -> np.ndarray:
        cols = self.group_cols.get(group, [])
        if not cols or self.models.get(group) is None:
            return np.zeros(len(df))
        X, _ = _group_matrix(df, cols)
        Xs = self.scalers[group].transform(X)
        lin = self.models[group].predict(Xs)
        if group == "uncertainty" and use_isotonic and self.g2_isotonic is not None:
            return self.g2_isotonic.predict(lin)
        return lin

    def fit(
        self,
        train_df: pd.DataFrame,
        calib_df: pd.DataFrame | None = None,
    ):
        """Walk-forward fit: train groups 1→2→3, then re-fit group-2 isotonic on calib."""
        if train_df is None or train_df.empty or "actual_margin" not in train_df.columns:
            self.fitted = False
            return self

        train_df = _ensure_uncertainty_col(train_df)
        if calib_df is not None and not calib_df.empty:
            calib_df = _ensure_uncertainty_col(calib_df)

        if len(train_df) < self.knobs.min_samples:
            self.fitted = False
            return self

        y = train_df["actual_margin"].to_numpy(dtype=float)

        pred1, _ = self._fit_group(train_df, "engine", y)
        resid1 = y - pred1

        pred2, _ = self._fit_group(train_df, "uncertainty", resid1)
        resid2 = resid1 - pred2

        pred3, _ = self._fit_group(train_df, "market", resid2)

        train_mae = float(np.mean(np.abs(y - (pred1 + pred2 + pred3))))
        self.knobs.group_train_mae = {
            "engine": float(np.mean(np.abs(y - pred1))),
            "uncertainty": float(np.mean(np.abs(resid1 - pred2))),
            "market": float(np.mean(np.abs(resid2 - pred3))),
            "combined": train_mae,
        }
        self.fitted = True

        # Re-fit group-2 isotonic on calib after groups 1 & 3 are fixed (walk-forward).
        self.g2_isotonic = None
        if self.knobs.refit_g2_isotonic and calib_df is not None and not calib_df.empty:
            cols2 = self.group_cols.get("uncertainty", [])
            if self.models.get("uncertainty") is not None and cols2:
                p1_c = self._predict_group(calib_df, "engine")
                p3_c = self._predict_group(calib_df, "market")
                y_c = calib_df["actual_margin"].to_numpy(dtype=float)
                target_g2 = y_c - p1_c - p3_c
                X2, _ = _group_matrix(calib_df, cols2)
                X2s = self.scalers["uncertainty"].transform(X2)
                lin2 = self.models["uncertainty"].predict(X2s)
                if len(lin2) >= 30 and len(np.unique(np.sign(target_g2))) > 1:
                    self.g2_isotonic = IsotonicRegression(out_of_bounds="clip")
                    self.g2_isotonic.fit(lin2, target_g2)
        return self

    def predict(self, feat: dict) -> float:
        if not self.fitted:
            return float(feat.get("elo_margin", 0.0) or 0.0)
        row = _ensure_uncertainty_col(pd.DataFrame([feat]))
        return float(self.predict_batch(row)[0])

    def predict_batch(self, df: pd.DataFrame) -> np.ndarray:
        if not self.fitted or df is None or df.empty:
            if df is not None and "elo_margin" in df.columns:
                return df["elo_margin"].fillna(0.0).to_numpy(dtype=float)
            return np.zeros(len(df) if df is not None else 0)
        df = _ensure_uncertainty_col(df)
        p1 = self._predict_group(df, "engine")
        p2 = self._predict_group(df, "uncertainty", use_isotonic=True)
        p3 = self._predict_group(df, "market")
        return p1 + p2 + p3

    def predict_interval(self, feat: dict, alpha: float = 0.10) -> tuple[float, float, float]:
        pred = self.predict(feat)
        q = getattr(self, "_resid_q", 12.0)
        return pred - q, pred + q, 2.0 * q

    def update_residuals(self, residuals: list[float]):
        if len(residuals) >= 20:
            self._resid_q = float(np.quantile(np.abs(residuals), 0.90))

    def save(self, path: Path | str):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path: Path | str):
        with open(path, "rb") as f:
            return pickle.load(f)


def default_elo_calibrator_path() -> Path:
    return STATE_DIR / "elo_calibrator.pkl"


def default_elo_knobs_path() -> Path:
    return STATE_DIR / "elo_calibration_knobs.json"


def save_elo_knobs(knobs: EloCalibrationKnobs, path: Path | str | None = None):
    path = Path(path or default_elo_knobs_path())
    path.parent.mkdir(parents=True, exist_ok=True)
    import json
    path.write_text(json.dumps(knobs.to_dict(), indent=2))


def load_elo_knobs(path: Path | str | None = None) -> EloCalibrationKnobs:
    path = Path(path or default_elo_knobs_path())
    if not path.exists():
        return EloCalibrationKnobs()
    import json
    return EloCalibrationKnobs.from_dict(json.loads(path.read_text()))


def augment_elo_features(
    feat: dict,
    calibrator: WalkForwardEloCalibrator | None = None,
    knobs: EloCalibrationKnobs | None = None,
) -> dict:
    """Add calibrated ELO margin and blend features to a game feature dict."""
    out = dict(feat)
    if "elo_uncertainty_adj" not in out or pd.isna(out.get("elo_uncertainty_adj")):
        tu = float(out.get("h_rating_uncertainty", 350) + out.get("a_rating_uncertainty", 350))
        out["elo_uncertainty_adj"] = float(out.get("elo_net", 0) or 0) / (tu + 1e-6)

    k = knobs or (calibrator.knobs if calibrator is not None else EloCalibrationKnobs())
    elo_raw = float(out.get("elo_margin", 0.0) or 0.0)
    hier = float(out.get("hier_margin", 0.0) or 0.0)

    if calibrator is not None and calibrator.fitted:
        cal = calibrator.predict(out)
    else:
        unc = float(out.get("h_rating_uncertainty", 350) + out.get("a_rating_uncertainty", 350))
        w_elo = min(0.85, max(0.55, 1.0 - (unc - 200) / 800))
        hier_part = (1.0 - w_elo) * 0.5 * (elo_raw + hier)
        cal = w_elo * elo_raw + hier_part

    out["elo_margin_calibrated"] = cal
    out["elo_hier_blend"] = k.hier_blend_elo * elo_raw + k.hier_blend_hier * hier
    out["elo_vs_hier_spread"] = elo_raw - hier
    if out.get("market_spread") is not None and not pd.isna(out.get("market_spread")):
        mkt = -float(out["market_spread"])
        out["elo_vs_market"] = cal - mkt
    else:
        out["elo_vs_market"] = 0.0
    return out


def tune_elo_blend_weights(features_df: pd.DataFrame) -> dict:
    """
    Walk-forward grid search for ELO / hier blend weights (chronological tail MAE).
    Returns suggested overrides for augment + PlayerRatingTracker scaling hints.
    """
    if features_df is None or features_df.empty:
        return {}
    df = features_df.dropna(subset=["actual_margin", "elo_margin"]).copy()
    if len(df) < 100:
        return {}

    if "game_date" in df.columns:
        df = df.sort_values("game_date")
    inner_n = max(len(df) // 5, 50)
    inner_train = df.iloc[:-inner_n]
    inner_val = df.iloc[-inner_n:]
    if len(inner_val) < 30:
        inner_train, inner_val = df, df.iloc[-max(30, len(df) // 10):]

    best_mae, best = 999.0, {}
    y = inner_val["actual_margin"].values
    elo = inner_val["elo_margin"].values
    hier = inner_val.get("hier_margin", pd.Series(0, index=inner_val.index)).fillna(0).values

    for w_elo in (0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9):
        for w_hier in (0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4):
            if w_elo + w_hier > 1.01:
                continue
            pred = w_elo * elo + w_hier * hier
            mae = np.mean(np.abs(pred - y))
            if mae < best_mae:
                best_mae = mae
                best = {
                    "hier_blend_elo": w_elo,
                    "hier_blend_hier": w_hier,
                    "blend_mae": mae,
                }

    if len(inner_train) >= 50:
        elo_tr = inner_train["elo_margin"].values
        y_tr = inner_train["actual_margin"].values
        A = np.column_stack([elo_tr, np.ones(len(elo_tr))])
        coef, _, _, _ = np.linalg.lstsq(A, y_tr, rcond=None)
        best["elo_scale"] = float(coef[0])
        best["elo_bias"] = float(coef[1])
    return best


def tune_elo_calibrator(
    train_df: pd.DataFrame,
    calib_df: pd.DataFrame | None = None,
) -> EloCalibrationKnobs:
    """
    Walk-forward grid for calibrator hyperparameters (small grid, no leakage).

    Uses chronological tail of train_df as inner validation; calib_df only simulates
    the outer walk-forward calib slice for isotonic re-fit scoring.
    """
    knobs = EloCalibrationKnobs()
    if train_df is None or train_df.empty or len(train_df) < 120:
        return knobs

    df = _ensure_uncertainty_col(train_df)
    if "game_date" in df.columns:
        df = df.sort_values("game_date").reset_index(drop=True)

    inner_cut = max(len(df) // 5, 80)
    inner_train = df.iloc[:-inner_cut]
    inner_val = df.iloc[-inner_cut:]
    calib_slice = calib_df
    if calib_slice is None or calib_slice.empty:
        calib_slice = inner_val.iloc[max(0, len(inner_val) // 2):]

    best_mae = 999.0
    best_knobs = EloCalibrationKnobs()

    for huber_eps in (1.15, 1.35, 1.55, 1.75):
        for min_s in (60, 80, 100):
            if len(inner_train) < min_s:
                continue
            for refit_iso in (True, False):
                trial = EloCalibrationKnobs(
                    huber_epsilon=huber_eps,
                    min_samples=min_s,
                    refit_g2_isotonic=refit_iso,
                )
                cal = WalkForwardEloCalibrator(knobs=trial)
                cal.fit(inner_train, calib_df=calib_slice)
                if not cal.fitted:
                    continue
                pred = cal.predict_batch(inner_val)
                mae = float(np.mean(np.abs(inner_val["actual_margin"].values - pred)))
                if mae < best_mae:
                    best_mae = mae
                    best_knobs = trial
                    best_knobs.group_train_mae = {"inner_val_mae": mae}

    blend = tune_elo_blend_weights(inner_train)
    if blend:
        best_knobs.hier_blend_elo = blend.get("hier_blend_elo", best_knobs.hier_blend_elo)
        best_knobs.hier_blend_hier = blend.get("hier_blend_hier", best_knobs.hier_blend_hier)
    return best_knobs


def apply_elo_calibration_df(
    df: pd.DataFrame,
    calibrator: WalkForwardEloCalibrator | None = None,
) -> pd.DataFrame:
    """Apply walk-forward ELO calibration columns (vectorized)."""
    if df is None or df.empty:
        return df
    out = _ensure_uncertainty_col(df.copy())
    knobs = calibrator.knobs if calibrator is not None else EloCalibrationKnobs()

    if calibrator is not None and calibrator.fitted:
        cal_vals = calibrator.predict_batch(out)
    else:
        elo = out.get("elo_margin", pd.Series(0, index=out.index)).fillna(0).values
        hier = out.get("hier_margin", pd.Series(0, index=out.index)).fillna(0).values
        unc = (
            out.get("h_rating_uncertainty", pd.Series(350, index=out.index)).fillna(350).values
            + out.get("a_rating_uncertainty", pd.Series(350, index=out.index)).fillna(350).values
        )
        w_elo = np.clip(1.0 - (unc - 200) / 800, 0.55, 0.85)
        cal_vals = w_elo * elo + (1.0 - w_elo) * 0.5 * (elo + hier)

    out["elo_margin_calibrated"] = cal_vals
    elo_raw = out.get("elo_margin", pd.Series(0, index=out.index)).fillna(0)
    hier_raw = out.get("hier_margin", pd.Series(0, index=out.index)).fillna(0)
    out["elo_hier_blend"] = knobs.hier_blend_elo * elo_raw + knobs.hier_blend_hier * hier_raw
    out["elo_vs_hier_spread"] = elo_raw - hier_raw
    if "market_spread" in out.columns:
        mkt = -out["market_spread"].fillna(0.0)
        out["elo_vs_market"] = out["elo_margin_calibrated"] - mkt
    else:
        out["elo_vs_market"] = 0.0
    return out


In [ ]:
# ── module: cv ───────────────────────────────────────────────────────────────
"""Time-series cross-validation utilities (purged / embargoed)."""
from __future__ import annotations

import numpy as np
from sklearn.model_selection import BaseCrossValidator


class PurgedGroupTimeSeriesSplit(BaseCrossValidator):
    """Chronological CV with optional purge gap between train and test.

    Groups (e.g. game dates) keep all samples on the same day together.
    Suitable for StackingRegressor internal CV when dates are available.
    """

    def __init__(self, n_splits: int = 5, embargo: int = 0):
        self.n_splits = n_splits
        self.embargo = embargo

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        n = len(X)
        if groups is None:
            order = np.arange(n)
        else:
            groups = np.asarray(groups)
            uniq = np.unique(groups)
            order = np.argsort([np.where(groups == g)[0][0] for g in uniq])
            # map sample indices to sorted group order
            group_order = {g: i for i, g in enumerate(uniq[order])}
            order = np.argsort([group_order[g] for g in groups])

        indices = np.arange(n)[order]
        fold_sizes = np.full(self.n_splits, n // self.n_splits, dtype=int)
        fold_sizes[: n % self.n_splits] += 1
        current = 0
        for fold_size in fold_sizes:
            start, stop = current, current + fold_size
            test_idx = indices[start:stop]
            train_end = max(0, start - self.embargo)
            train_idx = indices[:train_end]
            current = stop
            if len(train_idx) == 0 or len(test_idx) == 0:
                continue
            yield train_idx, test_idx


class ChronologicalPartitionCV(BaseCrossValidator):
    """Date-ordered KFold partition for StackingRegressor internal CV.

    .. deprecated:: Task 049 (spread-accuracy-roadmap Stage 5)
        **Do not use this splitter in production.** Its train fold is
        ``concatenate(order[:cursor], order[cursor + fold_size:])`` — i.e.
        every fold except the first trains on rows *after* the held-out
        test block (``stack_cv_future_leakage`` in ``LEAK_REGISTRY.md``).
        It is kept only so the regression test that caught the leak
        (``tests/test_cv_past_only.py::test_chronological_partition_cv_is_documented_as_leaky``)
        has something concrete to assert against, and so any historical
        artifact fit with it can still be unpickled/inspected. Production
        code must use :class:`PastOnlyGroupCV` (via
        :class:`pipeline.oof.ManualOOFStacker`) instead, which never trains
        on a row at or after the validation block's earliest timestamp.

    StackingRegressor requires every training row to appear in exactly one
    held-out fold (``cross_val_predict`` partition constraint). Strict
    past-only purged splits cannot satisfy that on early folds, so this
    splitter sorts by ``game_date`` groups and assigns contiguous blocks
    as test folds (no shuffle). Train folds include non-test rows only —
    which, for every fold but the last, includes future rows.
    """

    def __init__(self, n_splits: int = 5):
        self.n_splits = n_splits

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        n = len(X)
        if groups is not None:
            order = np.argsort(np.asarray(groups), kind="stable")
        else:
            order = np.arange(n)
        fold_sizes = np.full(self.n_splits, n // self.n_splits, dtype=int)
        fold_sizes[: n % self.n_splits] += 1
        cursor = 0
        for fs in fold_sizes:
            test_idx = order[cursor:cursor + fs]
            train_idx = np.concatenate([order[:cursor], order[cursor + fs:]])
            cursor += fs
            yield train_idx, test_idx


class PastOnlyGroupCV(BaseCrossValidator):
    """Expanding-window, group-aware CV where every fold satisfies
    ``max(train_time) < min(validation_time))`` (Task 049).

    Replaces :class:`ChronologicalPartitionCV` for any production
    stacking/meta-learning use. Groups (e.g. ``game_date``) are sorted
    ascending and partitioned into ``n_splits + 1`` contiguous blocks of
    *unique group values* (never splitting a single date across blocks).
    Block 0 is reserved as burn-in history that is never itself a
    validation fold (there is no strictly-earlier data to train it on).
    Fold ``i`` (``1..n_splits``) trains on every row in blocks ``[0, i)``
    and validates on block ``i``.

    Because block boundaries are snapped to unique group values, the
    training block's maximum group value is strictly less than the
    validation block's minimum group value by construction — this is
    verified by ``tests/test_cv_past_only.py``.

    Rows belonging to burn-in block 0 are never covered by any validation
    fold and must be dropped from any OOF meta-training set that consumes
    this splitter (Task 051 burn-in omission) — trying to "recover" them
    with a same-block or future-trained model would reintroduce the exact
    leak this class exists to remove (Rule 5/6).
    """

    def __init__(self, n_splits: int = 5):
        self.n_splits = n_splits

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        n = len(X)
        if groups is None:
            group_vals = np.arange(n)
        else:
            group_vals = np.asarray(groups)

        uniq_sorted = np.sort(np.unique(group_vals))
        n_groups = len(uniq_sorted)
        n_blocks = min(self.n_splits + 1, n_groups) if n_groups > 0 else 0
        n_blocks = max(n_blocks, 2) if n_groups >= 2 else n_groups
        if n_blocks < 2:
            return

        block_sizes = np.full(n_blocks, n_groups // n_blocks, dtype=int)
        block_sizes[: n_groups % n_blocks] += 1
        boundaries = np.concatenate([[0], np.cumsum(block_sizes)])
        group_blocks = [
            set(uniq_sorted[boundaries[i]:boundaries[i + 1]].tolist())
            for i in range(n_blocks)
        ]

        produced = 0
        max_yield = self.n_splits
        for i in range(1, n_blocks):
            if produced >= max_yield:
                break
            train_groups = np.array(sorted(set().union(*group_blocks[:i])), dtype=group_vals.dtype)
            test_groups = np.array(sorted(group_blocks[i]), dtype=group_vals.dtype)
            train_idx = np.flatnonzero(np.isin(group_vals, train_groups))
            test_idx = np.flatnonzero(np.isin(group_vals, test_groups))
            if len(train_idx) == 0 or len(test_idx) == 0:
                continue
            yield train_idx, test_idx
            produced += 1


def bind_cv_groups(cv_splitter, groups):
    """Fix group labels on a splitter (StackingRegressor cannot take groups= in older sklearn)."""
    if groups is None:
        return cv_splitter
    bound_groups = np.asarray(groups)
    inner = cv_splitter

    class _BoundSplit(BaseCrossValidator):
        def __init__(self, base, grp):
            self._base = base
            self._groups = grp
            if hasattr(base, "n_splits"):
                self.n_splits = base.n_splits
            if hasattr(base, "embargo"):
                self.embargo = base.embargo

        def get_n_splits(self, X=None, y=None, groups=None):
            return self._base.get_n_splits(X, y, self._groups)

        def split(self, X, y=None, groups=None):
            yield from self._base.split(X, y, self._groups)

    if isinstance(cv_splitter, (PurgedGroupTimeSeriesSplit, ChronologicalPartitionCV, PastOnlyGroupCV)):
        return _BoundSplit(inner, bound_groups)
    return cv_splitter


In [ ]:
# ── module: venn_abers ───────────────────────────────────────────────────────
"""Minimal binary Venn-Abers interval from calibration scores."""
from __future__ import annotations

import numpy as np
from sklearn.isotonic import IsotonicRegression


def scores_to_interval(scores_cal: np.ndarray, labels_cal: np.ndarray, scores_test: np.ndarray):
    """Return (p0, p1, optimal_point, width) arrays for test scores."""
    scores_cal = np.asarray(scores_cal, dtype=float)
    labels_cal = np.asarray(labels_cal, dtype=int)
    scores_test = np.asarray(scores_test, dtype=float)
    p0_list, p1_list = [], []
    for s in scores_test:
        p0 = _isotonic_prob(np.append(scores_cal, s), np.append(labels_cal, 0), s)
        p1 = _isotonic_prob(np.append(scores_cal, s), np.append(labels_cal, 1), s)
        p0_list.append(p0)
        p1_list.append(p1)
    p0 = np.asarray(p0_list)
    p1 = np.asarray(p1_list)
    denom = 1.0 - p0 + p1
    opt = np.where(denom > 1e-9, p1 / denom, 0.5)
    width = p1 - p0
    return p0, p1, opt, width


def _isotonic_prob(scores, labels, query_score):
    if len(np.unique(labels)) < 2:
        return float(labels.mean())
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(scores, labels)
    return float(iso.predict([query_score])[0])


def passes_venn_abers_filter(width: float, max_width: float) -> bool:
    if width is None or not np.isfinite(width):
        return True
    return float(width) <= float(max_width)


In [ ]:
# ── module: bet_confidence ───────────────────────────────────────────────────
"""Walk-forward empirical confidence calibrators per bet type."""
from __future__ import annotations

import pickle
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm


EDGE_BUCKETS = [0, 2, 4, 6, 8, np.inf]
EDGE_LABELS = ["0-2", "2-4", "4-6", "6-8", "8+"]

DEFAULT_BUCKET_ATS_LIFT = {
    "0-2": -0.02,
    "2-4": -0.012,
    "4-6": +0.030,
    "6-8": +0.003,
    "8+": +0.058,
}

# Hand-tuned score coefficients (Phase 2 default calibration).
DEFAULT_CONFIDENCE_WEIGHTS: dict[str, float] = {
    "edge_slope": 1.5,
    "bucket_lift_scale": 40.0,
    "interval_scale": 6.0,
    "interval_cap": 28.0,
    "unc_penalty_scale": 5.0,
    "unc_center": 180.0,
    "unc_span": 400.0,
    "agree_bonus": 8.0,
    "agree_penalty": -4.0,
    "cover_scale": 35.0,
    "breakeven_cover": 0.524,
    "vol_penalty_scale": 8.0,
    "trust_scale": 12.0,
    "trust_center": 0.75,
    "phantom_penalty": 4.0,
    "ats_scale": 20.0,
    "win_prob_scale": 28.0,
    "elo_margin_scale": 1.25,
    "elo_align_bonus": 7.0,
}

# Wide multiplicative search bounds (× default) for Phase 2a first-year tuning.
DEFAULT_WEIGHT_SEARCH_MULT = (0.25, 4.0)

# Order for logistic_features / logistic_isotonic calibration vector.
# All features are oriented so higher → historically better ATS cover rate.
CONFIDENCE_FEATURE_NAMES = (
    "composite_norm",
    "abs_edge_norm",
    "spread_cover_prob",
    "ats_classifier_prob",
    "win_prob_side",
    "disagreement_trust",
    "elo_meta_agreement",
    "elo_lean_align",
    "conf_tightness",
    "unc_quality",
    "vol_quality",
    "no_phantom",
)


def lean_side_win_prob(win_prob: float, lean: str) -> float:
    wp = float(win_prob if win_prob is not None else 0.5)
    if lean == "Away":
        return 1.0 - wp
    return wp


def elo_lean_align(lean: str, elo_margin: float) -> float:
    if lean in ("Pass", None, "", "nan") or (isinstance(lean, float) and pd.isna(lean)):
        return 0.5
    em = float(elo_margin or 0)
    return 1.0 if (lean == "Home") == (em >= 0) else 0.0


def _edge_bucket(abs_edge: float) -> str:
    for lo, hi, lbl in zip(EDGE_BUCKETS[:-1], EDGE_BUCKETS[1:], EDGE_LABELS):
        if lo <= abs_edge < hi:
            return lbl
    return EDGE_LABELS[-1]


def _tier_from_prob(p: float) -> int:
    if p >= 0.62:
        return 5
    if p >= 0.58:
        return 4
    if p >= 0.555:
        return 3
    if p >= 0.535:
        return 2
    return 1


def _stars_from_tier(tier: int, direction: str = "Home") -> str:
    if direction == "Pass" or tier < 2:
        return "Pass"
    labels = {
        2: "⭐⭐ (Lean)",
        3: "⭐⭐⭐ (Good)",
        4: "⭐⭐⭐⭐ (Strong)",
        5: "⭐⭐⭐⭐⭐ (Elite)",
    }
    return labels.get(tier, "⭐⭐ (Lean)")


def empirical_bucket_ats_rates(prior_df: pd.DataFrame) -> dict[str, float]:
    """ATS win rate by |edge| bucket from prior walk-forward results."""
    if prior_df is None or prior_df.empty or "EDGE" not in prior_df.columns:
        return {}
    d = prior_df[prior_df["MARKET_SPREAD"].notna()].copy()
    if "DIRECTION" in d.columns:
        d = d[d["DIRECTION"] != "Pass"]
    elif "EDGE_LEAN" in d.columns:
        d = d[d["EDGE_LEAN"].fillna("Pass") != "Pass"]
    if d.empty:
        return {}
    cover = d["ACTUAL_MARGIN"] + d["MARKET_SPREAD"]
    if "DIRECTION" in d.columns:
        home = d["DIRECTION"] == "Home"
    else:
        home = d["EDGE_LEAN"] == "Home"
    d = d.assign(
        ats_win=((home & (cover > 0)) | (~home & (cover < 0))).astype(float),
        abs_edge=d["EDGE"].abs(),
    )
    d = d[d["ats_win"].notna()]
    rates = {}
    for lbl in EDGE_LABELS:
        lo, hi = EDGE_BUCKETS[EDGE_LABELS.index(lbl)], EDGE_BUCKETS[EDGE_LABELS.index(lbl) + 1]
        sub = d[(d["abs_edge"] >= lo) & (d["abs_edge"] < hi)]
        if len(sub) >= 15:
            rates[lbl] = float(sub["ats_win"].mean())
    return rates


def prior_year_frame(
    df: pd.DataFrame,
    season_col: str = "simulated_season_window",
) -> pd.DataFrame:
    """Return only the most recent season in a results frame."""
    if df is None or df.empty:
        return pd.DataFrame()
    col = season_col if season_col in df.columns else ("_season" if "_season" in df.columns else None)
    if col is None:
        return df.copy()
    seasons = sorted(df[col].dropna().unique())
    if not seasons:
        return pd.DataFrame()
    return df[df[col] == seasons[-1]].copy()


def weight_search_ranges(
    center: dict[str, float] | None = None,
    *,
    shrink: float = 1.0,
    wide_mult: tuple[float, float] = DEFAULT_WEIGHT_SEARCH_MULT,
) -> dict[str, tuple[float, float]]:
    """Per-variable (lo, hi) bounds; shrink toward center each tuning year."""
    center = center or DEFAULT_CONFIDENCE_WEIGHTS
    shrink = float(np.clip(shrink, 0.05, 1.0))
    lo_mult, hi_mult = wide_mult
    ranges: dict[str, tuple[float, float]] = {}
    for key, default in DEFAULT_CONFIDENCE_WEIGHTS.items():
        c = float(center.get(key, default))
        if shrink >= 0.999:
            wlo, whi = c * lo_mult, c * hi_mult
        else:
            wlo, whi = c * lo_mult, c * hi_mult
            wlo = c - (c - wlo) * shrink
            whi = c + (whi - c) * shrink
        if key.endswith("_center") or key == "breakeven_cover":
            pad = max(abs(c) * 0.15 * shrink, 0.02)
            ranges[key] = (c - pad, c + pad)
        elif key == "interval_cap":
            ranges[key] = (max(18.0, c - 6.0 * shrink), min(36.0, c + 6.0 * shrink))
        else:
            ranges[key] = (min(wlo, whi), max(wlo, whi))
    edge_floor = 0.5 * float(DEFAULT_CONFIDENCE_WEIGHTS["edge_slope"])
    lo, hi = ranges["edge_slope"]
    ranges["edge_slope"] = (max(lo, edge_floor), hi)
    return ranges


def sample_weight_candidates(
    ranges: dict[str, tuple[float, float]],
    n_samples: int,
    rng: np.random.Generator | None = None,
    *,
    include_defaults: bool = False,
) -> list[dict[str, float]]:
    rng = rng or np.random.default_rng(42)
    keys = list(DEFAULT_CONFIDENCE_WEIGHTS.keys())
    out: list[dict[str, float]] = []
    if include_defaults:
        out.append(dict(DEFAULT_CONFIDENCE_WEIGHTS))
    for _ in range(max(1, n_samples)):
        w = dict(DEFAULT_CONFIDENCE_WEIGHTS)
        for k in keys:
            lo, hi = ranges.get(k, (DEFAULT_CONFIDENCE_WEIGHTS[k], DEFAULT_CONFIDENCE_WEIGHTS[k]))
            w[k] = float(rng.uniform(lo, hi))
        edge_floor = 0.5 * float(DEFAULT_CONFIDENCE_WEIGHTS["edge_slope"])
        w["edge_slope"] = max(w["edge_slope"], edge_floor)
        out.append(w)
    return out


def shrink_weight_center(
    center: dict[str, float],
    best: dict[str, float],
    alpha: float = 0.65,
) -> dict[str, float]:
    """Blend prior center toward newly tuned weights (year-over-year refinement)."""
    merged = dict(DEFAULT_CONFIDENCE_WEIGHTS)
    merged.update(center)
    for k in DEFAULT_CONFIDENCE_WEIGHTS:
        merged[k] = float((1.0 - alpha) * merged[k] + alpha * best.get(k, merged[k]))
    return merged


def bucket_lift_map(rates: dict[str, float], breakeven: float = 110.0 / 210.0) -> dict[str, float]:
    if not rates:
        return dict(DEFAULT_BUCKET_ATS_LIFT)
    return {lbl: rates.get(lbl, breakeven) - breakeven for lbl in EDGE_LABELS}


def build_confidence_features(
    *,
    spread_edge_pts: float = 0.0,
    conf_width: float = 24.0,
    rating_uncertainty: float = 350.0,
    elo_meta_agreement: float = 1.0,
    spread_cover_prob: float = 0.5,
    matchup_vol_sigma: float | None = None,
    league_vol_sigma: float = 10.0,
    disagreement_trust: float = 1.0,
    phantom_injury_flag: bool | int = False,
    ats_classifier_prob: float | None = None,
    win_prob: float | None = None,
    elo_margin: float | None = None,
    lean: str = "Home",
    **_,
) -> dict:
    """Normalized feature dict for unified ATS confidence scoring."""
    abs_edge = abs(float(spread_edge_pts or 0))
    cw = float(conf_width or 24)
    unc = float(rating_uncertainty or 350)
    vol_sig = float(matchup_vol_sigma) if matchup_vol_sigma is not None and np.isfinite(matchup_vol_sigma) else float(league_vol_sigma)
    lg = max(float(league_vol_sigma or 10.0), 1e-6)
    vol_ratio = vol_sig / lg
    ats_p = float(ats_classifier_prob) if ats_classifier_prob is not None and np.isfinite(ats_classifier_prob) else float(spread_cover_prob or 0.5)
    em = float(elo_margin or 0.0)
    return {
        "spread_edge_pts": float(spread_edge_pts or 0),
        "edge": abs_edge,
        "abs_edge": abs_edge,
        "conf_width": cw,
        "rating_uncertainty": unc,
        "elo_meta_agreement": float(elo_meta_agreement or 0.0),
        "spread_cover_prob": float(spread_cover_prob or 0.5),
        "matchup_vol_sigma": vol_sig,
        "league_vol_sigma": lg,
        "vol_ratio": vol_ratio,
        "disagreement_trust": float(np.clip(disagreement_trust or 1.0, 0.0, 1.0)),
        "phantom_injury_flag": int(bool(phantom_injury_flag)),
        "ats_classifier_prob": ats_p,
        "win_prob_side": lean_side_win_prob(win_prob if win_prob is not None else 0.5, lean),
        "abs_elo_margin": abs(em),
        "elo_lean_align": elo_lean_align(lean, em),
    }


def build_calibrated_feature_vector(
    features: dict,
    *,
    weights: dict[str, float] | None = None,
    bucket_lifts: dict[str, float] | None = None,
) -> np.ndarray:
    """Monotone confidence-aligned vector for logistic calibration."""
    f = build_confidence_features(**features) if "abs_edge" not in features else features
    w = dict(DEFAULT_CONFIDENCE_WEIGHTS)
    if weights:
        w.update(weights)
    lifts = bucket_lifts or DEFAULT_BUCKET_ATS_LIFT

    composite = ats_confidence_score(f, lifts, w)
    cw = float(f["conf_width"])
    unc = float(f["rating_uncertainty"])
    vol_ratio = float(f.get("vol_ratio", 1.0))
    cap = float(w["interval_cap"])

    return np.array([
        composite / 120.0,
        float(f["abs_edge"]) / 12.0,
        float(f["spread_cover_prob"]),
        float(f["ats_classifier_prob"]),
        float(f["win_prob_side"]),
        float(f["disagreement_trust"]),
        float(f["elo_meta_agreement"]),
        float(f["elo_lean_align"]),
        max(0.0, (cap - min(cw, cap)) / cap),
        max(0.0, 1.0 - max(0.0, (unc - w["unc_center"]) / w["unc_span"])),
        max(0.0, min(2.0, 2.0 - vol_ratio)),
        1.0 - float(f.get("phantom_injury_flag", 0)),
    ], dtype=float)


def confidence_feature_vector(
    features: dict,
    *,
    weights: dict[str, float] | None = None,
    bucket_lifts: dict[str, float] | None = None,
) -> np.ndarray:
    """Fixed-order numeric vector for logistic_features calibration."""
    if weights is not None or bucket_lifts is not None or "composite_norm" in features:
        return build_calibrated_feature_vector(
            features, weights=weights, bucket_lifts=bucket_lifts,
        )
    f = build_confidence_features(**features) if "abs_edge" not in features else features
    return build_calibrated_feature_vector(f, weights=weights, bucket_lifts=bucket_lifts)


def ats_confidence_score(
    features: dict,
    bucket_lifts: dict[str, float] | None = None,
    weights: dict[str, float] | None = None,
) -> float:
    """Unified ATS ranking score (higher → historically better cover rate)."""
    f = build_confidence_features(**features) if "abs_edge" not in features else features
    w = dict(DEFAULT_CONFIDENCE_WEIGHTS)
    if weights:
        w.update(weights)

    abs_edge = float(f["abs_edge"])
    conf_width = float(f["conf_width"])
    unc = float(f["rating_uncertainty"])
    elo_agree = float(f["elo_meta_agreement"])
    cover_p = float(f["spread_cover_prob"])
    vol_ratio = float(f.get("vol_ratio", 1.0))
    trust = float(f.get("disagreement_trust", 1.0))
    phantom = int(f.get("phantom_injury_flag", 0))
    ats_p = float(f.get("ats_classifier_prob", cover_p))
    win_p = float(f.get("win_prob_side", 0.5))
    elo_abs = float(f.get("abs_elo_margin", 0.0))
    elo_align = float(f.get("elo_lean_align", 0.5))

    lifts = bucket_lifts or DEFAULT_BUCKET_ATS_LIFT
    bucket = _edge_bucket(abs_edge)
    bucket_bonus = lifts.get(bucket, 0.0) * w["bucket_lift_scale"]
    edge_term = abs_edge * w["edge_slope"] + bucket_bonus

    cap = w["interval_cap"]
    interval_bonus = max(0.0, (cap - min(conf_width, cap)) / cap) * w["interval_scale"]
    unc_penalty = max(0.0, (unc - w["unc_center"]) / w["unc_span"]) * w["unc_penalty_scale"]
    agree_bonus = w["agree_bonus"] if elo_agree >= 0.5 else w["agree_penalty"]
    cover_term = (cover_p - w["breakeven_cover"]) * w["cover_scale"]

    vol_penalty = max(0.0, (vol_ratio - 1.0)) * w["vol_penalty_scale"]
    trust_bonus = (trust - w["trust_center"]) * w["trust_scale"]
    phantom_penalty = w["phantom_penalty"] if phantom else 0.0
    ats_bonus = (ats_p - w["breakeven_cover"]) * w["ats_scale"]
    win_bonus = (win_p - w["breakeven_cover"]) * w["win_prob_scale"]
    elo_bonus = elo_abs * w["elo_margin_scale"] + (elo_align - 0.5) * 2.0 * w["elo_align_bonus"]

    return float(
        edge_term + interval_bonus - unc_penalty + agree_bonus + cover_term
        - vol_penalty + trust_bonus - phantom_penalty + ats_bonus + win_bonus + elo_bonus
    )


def _heuristic_calibrated_prob(raw_score: float, raw_cover: float) -> float:
    return float(np.clip(0.35 * raw_cover + 0.65 * (0.524 + raw_score / 120.0), 0.01, 0.99))


class WalkForwardBetCalibrator:
    """P(cover | confidence) per bet type; default weights in Phase 2, tuned in Phase 2a."""

    def __init__(self, method: str | None = None, confidence_weights: dict[str, float] | None = None,
                 logistic_c: float = 0.1):
        self.method = method or CONFIDENCE_CALIB_METHOD
        self.logistic_c = float(logistic_c)
        self.confidence_weights = dict(DEFAULT_CONFIDENCE_WEIGHTS)
        if confidence_weights:
            self.confidence_weights.update(confidence_weights)
        self.models: dict[str, object | None] = {"ats": None, "ml": None, "ou": None}
        self.scalers: dict[str, StandardScaler | None] = {"ats": None, "ml": None, "ou": None}
        self.bucket_lifts: dict[str, float] = dict(DEFAULT_BUCKET_ATS_LIFT)
        self._fitted = False
        self._va_ats_scores: np.ndarray | None = None
        self._va_ats_outcomes: np.ndarray | None = None
        self._calib_method_ats: str = self.method
        self._fit_season: str | None = None

    @staticmethod
    def _ats_outcome(row) -> float | None:
        if pd.isna(row.get("MARKET_SPREAD")):
            return None
        direction = row.get("DIRECTION", row.get("EDGE_LEAN", "Pass"))
        edge = float(row.get("EDGE", row.get("spread_edge_pts", 0)) or 0)
        if direction in ("Pass", "nan", "", None) or (isinstance(direction, float) and pd.isna(direction)):
            if abs(edge) < 1e-9:
                return None
            direction = "Home" if edge > 0 else "Away"
        cover = row["ACTUAL_MARGIN"] + row["MARKET_SPREAD"]
        if cover == 0:
            return None
        home = direction == "Home"
        return float((home and cover > 0) or (not home and cover < 0))

    @staticmethod
    def _ml_outcome(row) -> float | None:
        if row.get("ML_DIRECTION") == "Pass" or pd.isna(row.get("MARKET_ML")):
            return None
        home_win = row["ACTUAL_HOME"] > row["ACTUAL_AWAY"]
        if row["ML_DIRECTION"] == "Home":
            return float(home_win)
        return float(not home_win)

    @staticmethod
    def _ou_outcome(row) -> float | None:
        if row.get("OU_DIRECTION") == "Pass" or pd.isna(row.get("MARKET_TOTAL")):
            return None
        actual = row["ACTUAL_HOME"] + row["ACTUAL_AWAY"]
        if row["OU_DIRECTION"] == "Over":
            return float(actual > row["MARKET_TOTAL"])
        return float(actual < row["MARKET_TOTAL"])

    def _row_features(self, row, bet_type: str) -> dict:
        abs_edge = abs(float(row.get("EDGE", row.get("spread_edge_pts", 0)) or 0))
        vol_sig = row.get("MATCHUP_VOL_SIGMA", row.get("matchup_vol_sigma"))
        direction = row.get("DIRECTION", row.get("EDGE_LEAN", "Pass"))
        if direction in ("Pass", "nan", "", None) or (isinstance(direction, float) and pd.isna(direction)):
            direction = "Home" if float(row.get("EDGE", 0) or 0) > 0 else "Away"
        elo_m = row.get("elo_margin_calibrated", row.get("ELO_MARGIN_CALIBRATED", 0))
        return build_confidence_features(
            spread_edge_pts=float(row.get("EDGE", row.get("spread_edge_pts", 0)) or 0),
            conf_width=float(row.get("CONF_WIDTH", 24) or 24),
            rating_uncertainty=float(row.get("RATING_UNCERTAINTY", 350) or 350),
            elo_meta_agreement=float(row.get("ELO_META_AGREEMENT", 1.0) or 0.0),
            spread_cover_prob=float(
                row.get("SPREAD_COVER_PROB", row.get("SPREAD_COVER_PROB_RAW", row.get("COVER_PROB_RAW", 0.5))) or 0.5
            ),
            matchup_vol_sigma=float(vol_sig) if vol_sig is not None and pd.notna(vol_sig) else None,
            disagreement_trust=float(row.get("DISAGREEMENT_TRUST", 1.0) or 1.0),
            phantom_injury_flag=int(row.get("PHANTOM_INJURY_FLAG", 0) or 0),
            ats_classifier_prob=float(row.get("ATS_CLASSIFIER_PROB", np.nan))
            if pd.notna(row.get("ATS_CLASSIFIER_PROB", np.nan)) else None,
            win_prob=float(row.get("WIN_PROB", row.get("win_prob", 0.5)) or 0.5),
            elo_margin=float(elo_m) if elo_m is not None and pd.notna(elo_m) else 0.0,
            lean=str(direction),
        )

    def _build_score(self, row, bet_type: str) -> float:
        feat = self._row_features(row, bet_type)
        if bet_type == "ats":
            return ats_confidence_score(feat, self.bucket_lifts, self.confidence_weights)
        if bet_type == "ml":
            unc = feat["rating_uncertainty"]
            return (
                abs(feat.get("ml_ev", 0)) * 200.0
                + abs(feat.get("win_prob", 0.5) - 0.5) * 50.0
                + (700 - min(unc, 700)) / 20.0
            )
        return float(feat.get("total_edge", 0)) * 1.5

    def _enrich_ats_row(self, row) -> dict:

        r = row.to_dict() if hasattr(row, "to_dict") else dict(row)
        pred = r.get("PRED_SPREAD", r.get("RAW_PRED_MARGIN", 0))
        mkt = r.get("MARKET_SPREAD", np.nan)
        cw = float(r.get("CONF_WIDTH", 24) or 24)
        direction = r.get("DIRECTION", r.get("EDGE_LEAN", "Pass"))
        if direction in ("Pass", "nan", "", None) or (isinstance(direction, float) and pd.isna(direction)):
            direction = "Home" if float(r.get("EDGE", 0) or 0) > 0 else "Away"
        if "ELO_META_AGREEMENT" not in r or pd.isna(r.get("ELO_META_AGREEMENT")):
            r["ELO_META_AGREEMENT"] = elo_meta_agreement(
                pred, mkt, r.get("elo_margin_calibrated", r.get("ELO_MARGIN_CALIBRATED")),
            )
        if "SPREAD_COVER_PROB" not in r or pd.isna(r.get("SPREAD_COVER_PROB")):
            r["SPREAD_COVER_PROB"] = spread_cover_prob(pred, mkt, cw, direction=direction)
        return r

    def _fit_calibrator(self, bet_type: str, scores: np.ndarray, outcomes: np.ndarray,
                        feature_matrix: np.ndarray | None, method: str, logistic_c: float | None = None):
        if len(scores) < 30 or len(set(outcomes.tolist())) < 2:
            self.models[bet_type] = None
            self.scalers[bet_type] = None
            return

        y = np.asarray(outcomes, dtype=int)
        c = float(self.logistic_c if logistic_c is None else logistic_c)
        if method in ("logistic_features", "logistic_isotonic") and feature_matrix is not None:
            scaler = StandardScaler()
            X = scaler.fit_transform(feature_matrix)
            clf = LogisticRegression(C=c, solver="lbfgs", max_iter=300)
            clf.fit(X, y)
            if method == "logistic_isotonic":
                train_p = clf.predict_proba(X)[:, 1]
                iso = IsotonicRegression(out_of_bounds="clip")
                iso.fit(train_p, y)
                self.models[bet_type] = {"logistic": clf, "isotonic": iso}
            else:
                self.models[bet_type] = clf
            self.scalers[bet_type] = scaler
        elif method == "platt":
            clf = LogisticRegression(C=0.1, solver="lbfgs", max_iter=200)
            clf.fit(np.asarray(scores, dtype=float).reshape(-1, 1), y)
            self.models[bet_type] = clf
            self.scalers[bet_type] = None
        elif method == "isotonic":
            iso = IsotonicRegression(out_of_bounds="clip")
            iso.fit(np.asarray(scores, dtype=float), y)
            self.models[bet_type] = iso
            self.scalers[bet_type] = None
        else:
            self.models[bet_type] = None
            self.scalers[bet_type] = None

    def _calibration_frame(self, prior_df: pd.DataFrame, scope: str | None = None) -> pd.DataFrame:

        use_scope = scope or CONFIDENCE_CALIB_SCOPE
        if use_scope == "prior_year":
            calib = prior_year_frame(prior_df)
            return calib if not calib.empty else prior_df
        return prior_df

    def fit(
        self,
        prior_df: pd.DataFrame,
        method: str | None = None,
        scope: str | None = None,
        confidence_weights: dict[str, float] | None = None,
    ):
        if prior_df is None or prior_df.empty:
            self._fitted = False
            return self

        if confidence_weights:
            self.confidence_weights.update(confidence_weights)

        calib_df = self._calibration_frame(prior_df, scope=scope)
        if calib_df.empty:
            self._fitted = False
            return self

        season_col = "simulated_season_window" if "simulated_season_window" in calib_df.columns else "_season"
        if season_col in calib_df.columns and calib_df[season_col].notna().any():
            self._fit_season = str(calib_df[season_col].dropna().iloc[-1])

        calib_method = method or self.method
        self._calib_method_ats = calib_method
        rates = empirical_bucket_ats_rates(calib_df)
        self.bucket_lifts = bucket_lift_map(rates)

        for bet_type, outcome_fn in [
            ("ats", self._ats_outcome),
            ("ml", self._ml_outcome),
            ("ou", self._ou_outcome),
        ]:
            scores, outcomes, feat_rows = [], [], []
            for _, row in calib_df.iterrows():
                y = outcome_fn(row)
                if y is None:
                    continue
                row_dict = self._enrich_ats_row(row) if bet_type == "ats" else row
                feat = self._row_features(row_dict, bet_type)
                scores.append(self._build_score(row_dict, bet_type))
                outcomes.append(y)
                if bet_type == "ats":
                    feat_rows.append(
                        build_calibrated_feature_vector(
                            feat,
                            weights=self.confidence_weights,
                            bucket_lifts=self.bucket_lifts,
                        )
                    )

            scores_arr = np.asarray(scores, dtype=float)
            outcomes_arr = np.asarray(outcomes, dtype=int)
            feat_mat = np.vstack(feat_rows) if feat_rows else None
            use_method = calib_method if bet_type == "ats" else "isotonic"
            self._fit_calibrator(bet_type, scores_arr, outcomes_arr, feat_mat, use_method)

            if bet_type == "ats":
                self._va_ats_scores = scores_arr
                self._va_ats_outcomes = outcomes_arr

        self._fitted = True
        return self

    def _predict_prob(self, bet_type: str, raw_score: float, features: dict) -> float:
        model = self.models.get(bet_type)
        method = self._calib_method_ats if bet_type == "ats" else "isotonic"
        raw_cover = float(features.get("spread_cover_prob", 0.5) or 0.5)

        if model is None or method == "none":
            return _heuristic_calibrated_prob(raw_score, raw_cover)

        if method in ("logistic_features", "logistic_isotonic") and bet_type == "ats":
            scaler = self.scalers.get(bet_type)
            if scaler is None:
                return _heuristic_calibrated_prob(raw_score, raw_cover)
            vec = build_calibrated_feature_vector(
                features,
                weights=self.confidence_weights,
                bucket_lifts=self.bucket_lifts,
            ).reshape(1, -1)
            x = scaler.transform(vec)
            if isinstance(model, dict):
                p = float(model["logistic"].predict_proba(x)[0, 1])
                return float(model["isotonic"].predict([p])[0])
            return float(model.predict_proba(x)[0, 1])

        if method in ("platt",) or isinstance(model, LogisticRegression):
            return float(model.predict_proba(np.array([[raw_score]], dtype=float))[0, 1])

        if isinstance(model, IsotonicRegression):
            return float(model.predict([raw_score])[0])

        return _heuristic_calibrated_prob(raw_score, raw_cover)

    def predict(self, bet_type: str, features: dict, direction: str = "Home") -> dict:
        if bet_type == "ats":
            feat = build_confidence_features(**features) if "abs_edge" not in features else features
            score = ats_confidence_score(feat, self.bucket_lifts, self.confidence_weights)
        else:
            score = self._build_score(features, bet_type)

        prob = self._predict_prob(bet_type, score, features)
        prob = float(np.clip(prob, 0.01, 0.99))
        tier = _tier_from_prob(prob)
        return {
            "calibrated_prob": prob,
            "confidence_score": int(round(prob * 100)),
            "confidence_tier": tier,
            "stars": _stars_from_tier(tier, direction),
            "confidence_score_raw": score,
        }

    def venn_abers_width(self, score: float) -> float | None:
        if self._va_ats_scores is None or self._va_ats_outcomes is None:
            return None
        if len(self._va_ats_scores) < 30:
            return None
        _, _, _, width = scores_to_interval(
            self._va_ats_scores, self._va_ats_outcomes, np.array([float(score)]),
        )
        return float(width[0])

    def save(self, path: Path | str):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path: Path | str):
        with open(path, "rb") as f:
            return pickle.load(f)


def _ats_roi_from_outcomes(outcomes: np.ndarray) -> float:
    if len(outcomes) == 0:
        return -1e9
    wp = float(np.mean(outcomes))
    return float(wp * (100.0 / 110.0) - (1.0 - wp))


def _confidence_training_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Rows eligible for confidence calibration / threshold tuning (|edge| floor)."""
    if df is None or df.empty:
        return pd.DataFrame()
    m = df.copy()
    if "EDGE" not in m.columns:
        return m
    edge = pd.to_numeric(m["EDGE"], errors="coerce").fillna(0).abs()
    return m[edge >= float(CONFIDENCE_MIN_EDGE)].copy()


def build_calib_split_ats_frame(
    calib_features: pd.DataFrame,
    *,
    pred_margin_col: str = "pred_margin",
    season_label: str = "calib-split",
) -> pd.DataFrame:
    """Build ATS-shaped rows from the training calibration split (season-1 calibrator)."""
    if calib_features is None or calib_features.empty:
        return pd.DataFrame()
    df = calib_features.copy()
    mkt_col = "market_spread" if "market_spread" in df.columns else "closing_spread"
    if mkt_col not in df.columns or pred_margin_col not in df.columns:
        return pd.DataFrame()
    if "actual_margin" not in df.columns:
        if "actual_home" in df.columns and "actual_away" in df.columns:
            df["actual_margin"] = df["actual_home"] - df["actual_away"]
        else:
            return pd.DataFrame()
    mkt = pd.to_numeric(df[mkt_col], errors="coerce")
    pm = pd.to_numeric(df[pred_margin_col], errors="coerce")
    am = pd.to_numeric(df["actual_margin"], errors="coerce")
    valid = mkt.notna() & pm.notna() & am.notna()
    if not valid.any():
        return pd.DataFrame()
    out = df.loc[valid].copy()
    out["MARKET_SPREAD"] = mkt.loc[out.index].values
    out["PRED_SPREAD"] = pm.loc[out.index].values
    out["ACTUAL_MARGIN"] = am.loc[out.index].values
    out["EDGE"] = out["PRED_SPREAD"] + out["MARKET_SPREAD"]
    out["DIRECTION"] = np.where(
        out["EDGE"] > 0, "Home", np.where(out["EDGE"] < 0, "Away", "Pass"),
    )
    out["EDGE_LEAN"] = out["DIRECTION"]
    out["CONF_WIDTH"] = 24.0
    if "elo_margin_calibrated" in out.columns:
        out["ELO_MARGIN_CALIBRATED"] = out["elo_margin_calibrated"]
    out["simulated_season_window"] = season_label
    return out


def _scored_ats_bets(cal: WalkForwardBetCalibrator, df: pd.DataFrame) -> list[tuple[float, int, float]]:
    """Return (outcome, confidence_score 0-100, calibrated_prob) per scored row."""
    scored: list[tuple[float, int, float]] = []
    train_df = _confidence_training_frame(df)
    for _, row in train_df.iterrows():
        y = WalkForwardBetCalibrator._ats_outcome(row)
        if y is None:
            continue
        row_dict = cal._enrich_ats_row(row)
        raw = cal._build_score(row_dict, "ats")
        feat = cal._row_features(row_dict, "ats")
        prob = float(cal._predict_prob("ats", raw, feat))
        conf_score = int(round(prob * 100))
        scored.append((float(y), conf_score, prob))
    return scored


def _pick_confidence_band_gated_roi(
    scored: list[tuple[float, int, float]],
    *,
    min_thresholds=(55, 56, 57, 58, 59, 60),
    max_thresholds: tuple[int | None, ...] = (None, 61, 62, 63, 64, 65),
    min_bets: int = 35,
) -> tuple[float, float, float | None, int]:
    """Pick (roi, min_thr, max_thr, n_bets) maximizing flat ATS ROI in a score band."""
    best_roi, best_min, best_max, best_n = -1e9, 58, None, 0
    for lo in min_thresholds:
        for hi in max_thresholds:
            if hi is not None and hi < lo:
                continue
            sub = [
                y for y, cs, _p in scored
                if cs >= lo and (hi is None or cs <= hi)
            ]
            if len(sub) < min_bets:
                continue
            roi = _ats_roi_from_outcomes(np.asarray(sub, dtype=float))
            if roi > best_roi:
                best_roi, best_min, best_max, best_n = roi, float(lo), hi, len(sub)
    return best_roi, best_min, best_max, best_n


def _pick_threshold_gated_roi(
    scored: list[tuple[float, int, float]],
    thresholds=range(50, 71),
    min_bets: int = 35,
) -> tuple[float, float, int, int]:
    """Maximize gated ROI; tie-break by lower ECE."""

    best_roi, best_ece, best_thr, best_n = -1e9, 1e9, 58, 0
    for thr in thresholds:
        sub = [(y, p) for y, cs, p in scored if cs >= thr]
        if len(sub) < min_bets:
            continue
        y_arr = np.asarray([x[0] for x in sub], dtype=float)
        p_arr = np.asarray([x[1] for x in sub], dtype=float)
        roi = _ats_roi_from_outcomes(y_arr)
        ece = float(compute_ece(y_arr, p_arr))
        if roi > best_roi + 1e-9 or (abs(roi - best_roi) <= 1e-9 and ece < best_ece):
            best_roi, best_ece, best_thr, best_n = roi, ece, int(thr), len(sub)
    return best_roi, best_ece, best_thr, best_n


def weights_differ_from_default(weights: dict[str, float], rtol: float = 0.02) -> bool:
    for k, v in DEFAULT_CONFIDENCE_WEIGHTS.items():
        w = float(weights.get(k, v))
        if abs(w - v) > max(abs(v) * rtol, 1e-6):
            return True
    return False


def _rank_spearman(outcomes, scores) -> float:
    """Spearman rank correlation between confidence score and ATS outcome."""
    y = np.asarray(outcomes, dtype=float)
    s = np.asarray(scores, dtype=float)
    mask = np.isfinite(y) & np.isfinite(s)
    if mask.sum() < 12:
        return 0.0
    y, s = y[mask], s[mask]
    if np.std(s) < 1e-9:
        return 0.0
    r = np.corrcoef(np.argsort(np.argsort(s)), np.argsort(np.argsort(y)))[0, 1]
    return float(r) if np.isfinite(r) else 0.0


def logistic_feature_importance(cal: WalkForwardBetCalibrator) -> dict[str, float]:
    """Signed logistic coefficients mapped to human-readable feature names."""
    model = cal.models.get("ats")
    if model is None:
        return {}
    clf = model["logistic"] if isinstance(model, dict) else model
    if not hasattr(clf, "coef_"):
        return {}
    coefs = np.asarray(clf.coef_).ravel()
    names = list(CONFIDENCE_FEATURE_NAMES)
    if len(coefs) != len(names):
        return {}
    return {n: float(c) for n, c in zip(names, coefs)}


def prepare_walkforward_calibrator(
    cal: WalkForwardBetCalibrator,
    prior_df: pd.DataFrame,
    *,
    season_index: int = 1,
    method: str | None = None,
) -> WalkForwardBetCalibrator:
    """Deprecated wrapper — use fit_walkforward_confidence_calibrator."""
    fitted, _meta = fit_walkforward_confidence_calibrator(
        prior_df,
        season_index=season_index,
        weight_center=cal.confidence_weights,
        method=method,
        existing_calibrator=cal,
    )
    return fitted


def fit_walkforward_confidence_calibrator(
    prior_df: pd.DataFrame,
    *,
    train_season_label: str = "",
    test_season_label: str = "",
    season_index: int = 1,
    weight_center: dict[str, float] | None = None,
    method: str | None = None,
    calib_scope: str | None = None,
    show_progress: bool = False,
    existing_calibrator: WalkForwardBetCalibrator | None = None,
) -> tuple[WalkForwardBetCalibrator, dict]:
    """Phase 2a inline: tune + fit confidence calibrator on prior season only (leak-free)."""

    if prior_df is None or prior_df.empty:
        cal = existing_calibrator or WalkForwardBetCalibrator(method=method)
        cal._fitted = False
        return cal, {}

    train_df = _confidence_training_frame(prior_df)
    if train_df.empty:
        train_df = prior_df.copy()

    center = dict(DEFAULT_CONFIDENCE_WEIGHTS)
    if weight_center:
        center.update(weight_center)
    calib_method = method or CONFIDENCE_CALIB_METHOD
    scope = calib_scope or CONFIDENCE_CALIB_SCOPE
    cal = existing_calibrator or WalkForwardBetCalibrator(
        method=calib_method,
        confidence_weights=center,
    )
    cal.method = calib_method
    cal.confidence_weights = dict(center)

    best_cal = cal
    composite_weights = dict(center)

    if TUNE_CONFIDENCE_WEIGHTS_IN_BACKTEST:
        shrink = 1.0 if season_index <= 1 else CONFIDENCE_WEIGHT_SHRINK
        ranges = weight_search_ranges(center, shrink=shrink)
        composite_weights, best_cal, _roi, _thr, _ece, _n = tune_confidence_weights(
            train_df,
            center=center,
            ranges=ranges,
            method=calib_method,
            objective="hybrid",
            show_progress=show_progress,
            progress_desc=f"2a · weights · {train_season_label or 'prior'}",
        )
        if calib_method in ("logistic_features", "logistic_isotonic"):
            best_c, best_obj, best_cal = best_cal.logistic_c, -1e9, best_cal
            for c in (0.02, 0.05, 0.1, 0.25, 0.5, 1.0):
                trial = WalkForwardBetCalibrator(
                    method=calib_method,
                    confidence_weights=composite_weights,
                    logistic_c=c,
                )
                trial.fit(
                    train_df,
                    method=calib_method,
                    scope=scope,
                    confidence_weights=composite_weights,
                )
                scored = _scored_ats_bets(trial, train_df)
                if len(scored) < 35:
                    continue
                y = np.asarray([s[0] for s in scored], dtype=float)
                cs = [s[1] for s in scored]
                p = np.asarray([s[2] for s in scored], dtype=float)
                ece = float(compute_ece(y, p))
                sp = _rank_spearman(y, cs)
                band_roi, _bmin, _bmax, band_n = _pick_confidence_band_gated_roi(
                    scored, min_bets=35,
                )
                obj = sp + 0.5 * band_roi - ece
                if obj > best_obj:
                    best_obj, best_c, best_cal = obj, c, trial
            best_cal.logistic_c = best_c
    else:
        best_cal.fit(train_df, method=calib_method, scope=scope, confidence_weights=center)

    if not best_cal._fitted:
        best_cal.fit(train_df, method=calib_method, scope=scope, confidence_weights=composite_weights)

    feat_w = logistic_feature_importance(best_cal)
    if calib_method in ("logistic_features", "logistic_isotonic"):
        edge_coef = float(feat_w.get("abs_edge_norm", 0.0))
        if edge_coef < 0:
            calib_method = "isotonic"
            best_cal = WalkForwardBetCalibrator(
                method="isotonic",
                confidence_weights=composite_weights,
            )
            best_cal.fit(
                train_df,
                method="isotonic",
                scope=scope,
                confidence_weights=composite_weights,
            )
            feat_w = {}

    scored = _scored_ats_bets(best_cal, train_df)
    y_all = np.asarray([s[0] for s in scored], dtype=float) if scored else np.array([])
    p_all = np.asarray([s[2] for s in scored], dtype=float) if scored else np.array([])
    cs_all = [s[1] for s in scored]

    conf_thr, conf_max = walkforward_confidence_gate(
        train_df,
        default_min=MIN_CONFIDENCE_SCORE,
        default_max=MAX_CONFIDENCE_SCORE,
        bet_calibrator=best_cal if best_cal._fitted else None,
    )
    gated = [
        y for y, cs, _p in scored
        if cs >= conf_thr and (conf_max is None or cs <= conf_max)
    ]
    train_roi = _ats_roi_from_outcomes(np.asarray(gated, dtype=float)) if gated else np.nan
    train_ece = float(compute_ece(y_all, p_all)) if len(y_all) else np.nan
    train_sp = _rank_spearman(y_all, cs_all) if len(y_all) else np.nan

    meta = {
        "train_season": train_season_label,
        "test_season": test_season_label,
        "calib_method": calib_method,
        "logistic_c": float(best_cal.logistic_c),
        "train_roi": float(train_roi) if np.isfinite(train_roi) else np.nan,
        "train_ece": train_ece,
        "train_spearman": train_sp,
        "train_min_confidence": float(conf_thr),
        "train_max_confidence": float(conf_max) if conf_max is not None else None,
        "train_gated_n_bets": len(gated),
        "n_train_bets": len(scored),
        "feature_weights": feat_w,
        "suggested_weights": dict(composite_weights),
        "weights_changed": (
            weights_differ_from_default(composite_weights)
            or any(abs(v) > 1e-6 for v in feat_w.values())
        ),
        "inline_phase2a": True,
    }
    for k, v in composite_weights.items():
        meta[f"w_{k}"] = round(float(v), 4)
    for k, v in feat_w.items():
        meta[f"coef_{k}"] = round(float(v), 4)

    return best_cal, meta


def save_phase2a_walkforward_state(
    season_rows: list[dict],
    *,
    latest_center: dict[str, float] | None = None,
) -> Path:
    """Persist inline Phase 2a tuning from walk-forward backtest."""

    path = confidence_weights_path()
    path.parent.mkdir(parents=True, exist_ok=True)
    payload: dict = {
        "inline_phase2a": True,
        "calib_method": CONFIDENCE_CALIB_METHOD,
        "unified_weight_tuning": season_rows,
    }
    if season_rows:
        latest = season_rows[-1]
        payload["suggested_weights_latest"] = latest.get("suggested_weights", dict(DEFAULT_CONFIDENCE_WEIGHTS))
        payload["suggested_min_confidence"] = int(latest.get("train_min_confidence", MIN_CONFIDENCE_SCORE))
        if latest.get("train_max_confidence") is not None:
            payload["suggested_max_confidence"] = int(latest["train_max_confidence"])
        payload["suggested_logistic_c"] = latest.get("logistic_c")
        payload["feature_weights_latest"] = latest.get("feature_weights", {})
        payload["weights_changed_latest"] = bool(latest.get("weights_changed", False))
    if latest_center:
        payload["weight_center_latest"] = {k: float(v) for k, v in latest_center.items()}
    path.write_text(json.dumps(payload, indent=2, default=str))
    return path


def tune_confidence_weights(
    train_df: pd.DataFrame,
    *,
    center: dict[str, float] | None = None,
    ranges: dict[str, tuple[float, float]] | None = None,
    method: str | None = None,
    n_samples: int | None = None,
    min_bets: int = 35,
    thresholds=range(50, 71),
    show_progress: bool = True,
    progress_desc: str | None = None,
    objective: str = "roi",
) -> tuple[dict[str, float], WalkForwardBetCalibrator, float, int, float, int]:
    """Search weight floats on prior season.

    objective: roi | hybrid (rank correlation + low ECE + ROI at best threshold).
    """

    w_center = dict(DEFAULT_CONFIDENCE_WEIGHTS)
    if center:
        w_center.update(center)
    search_ranges = ranges or weight_search_ranges(w_center, shrink=1.0)
    n = n_samples or CONFIDENCE_WEIGHT_SEARCH_SAMPLES
    calib_method = method or CONFIDENCE_CALIB_METHOD

    best_weights = dict(w_center)
    best_roi, best_ece, best_thr, best_n = -1e9, 1e9, 58, 0
    best_score = -1e9
    best_cal = WalkForwardBetCalibrator(method=calib_method, confidence_weights=best_weights)

    candidates = list(sample_weight_candidates(search_ranges, n, include_defaults=True))
    cand_bar = tqdm(
        candidates,
        desc=progress_desc or "2a · weight candidates",
        disable=not show_progress,
        leave=False,
    )
    for weights in cand_bar:
        cal = WalkForwardBetCalibrator(method=calib_method, confidence_weights=weights)
        cal.fit(train_df, method=calib_method, scope="all_prior", confidence_weights=weights)
        scored = _scored_ats_bets(cal, train_df)
        if len(scored) < min_bets:
            continue
        roi, ece, thr, n_bets = _pick_threshold_gated_roi(scored, thresholds=thresholds, min_bets=min_bets)
        if objective == "hybrid":
            y = np.asarray([s[0] for s in scored], dtype=float)
            cs = [s[1] for s in scored]
            sp = _rank_spearman(y, cs)
            band_roi, _bmin, _bmax, band_n = _pick_confidence_band_gated_roi(
                scored, min_bets=min_bets,
            )
            combo = sp - 0.5 * ece + 0.5 * band_roi
            if combo > best_score:
                best_score = combo
                best_roi = band_roi
                best_ece = ece
                best_thr = int(_bmin) if band_n >= min_bets else thr
                best_n = band_n if band_n >= min_bets else n_bets
                best_weights = dict(weights)
                best_cal = cal
        elif roi > best_roi + 1e-9 or (abs(roi - best_roi) <= 1e-9 and ece < best_ece):
            best_roi, best_ece, best_thr, best_n = roi, ece, thr, n_bets
            best_weights = dict(weights)
            best_cal = cal

    best_cal = WalkForwardBetCalibrator(method=calib_method, confidence_weights=best_weights)
    best_cal.fit(train_df, method=calib_method, scope="all_prior", confidence_weights=best_weights)
    return best_weights, best_cal, best_roi, best_thr, best_ece, best_n


def yearly_confidence_weight_tuning(
    results_df: pd.DataFrame,
    *,
    season_col: str = "simulated_season_window",
    shrink: float | None = None,
    method: str | None = None,
    show_progress: bool = True,
) -> pd.DataFrame:
    """Phase 2a post-hoc replay — uses same fit_walkforward_confidence_calibrator as inline backtest."""

    df = _norm_results(results_df)
    if season_col not in df.columns:
        return pd.DataFrame()

    if "PHASE2A_INLINE" in df.columns and df["PHASE2A_INLINE"].fillna(0).astype(int).max() > 0:
        path = confidence_weights_path()
        if path.exists():
            try:
                payload = json.loads(path.read_text())
                if payload.get("inline_phase2a"):
                    print("⚡ Phase 2a ran inline during walk-forward — loading saved tuning (no re-fit).")
                rows = payload.get("unified_weight_tuning", [])
                if rows:
                    return pd.DataFrame(rows)
            except (OSError, json.JSONDecodeError):
                pass

    shrink = CONFIDENCE_WEIGHT_SHRINK if shrink is None else shrink
    seasons = sorted(df[season_col].dropna().unique())
    center = dict(DEFAULT_CONFIDENCE_WEIGHTS)
    rows = []

    season_pairs = [(seasons[i - 1], test_season) for i, test_season in enumerate(seasons) if i > 0]
    season_bar = tqdm(
        season_pairs,
        desc="2a · tune seasons",
        disable=not show_progress,
        leave=False,
    )
    for i, (train_season, test_season) in enumerate(season_bar, start=1):
        season_bar.set_postfix(train=str(train_season), test=str(test_season))
        train_df = df[df[season_col] == train_season]
        if train_df.empty:
            continue

        cal, meta = fit_walkforward_confidence_calibrator(
            train_df,
            train_season_label=str(train_season),
            test_season_label=str(test_season),
            season_index=i,
            weight_center=center,
            method=method,
            show_progress=show_progress,
        )
        composite_weights = meta.get("suggested_weights", center)

        test_df = df[df[season_col] == test_season]
        thr = int(meta.get("train_min_confidence", MIN_CONFIDENCE_SCORE))
        tmax = meta.get("train_max_confidence")
        outcomes, probs = [], []
        for _, row in test_df.iterrows():
            y = WalkForwardBetCalibrator._ats_outcome(row)
            if y is None:
                continue
            row_dict = cal._enrich_ats_row(row)
            raw = cal._build_score(row_dict, "ats")
            feat = cal._row_features(row_dict, "ats")
            prob = cal._predict_prob("ats", raw, feat)
            conf_score = int(round(prob * 100))
            if conf_score < thr or (tmax is not None and conf_score > tmax):
                continue
            probs.append(prob)
            outcomes.append(y)
        test_roi = _ats_roi_from_outcomes(np.asarray(outcomes, dtype=float)) if outcomes else np.nan
        test_ece = float(compute_ece(np.asarray(outcomes), np.asarray(probs))) if outcomes else np.nan

        row = dict(meta)
        row.update({
            "test_roi": test_roi,
            "test_ece": test_ece,
            "test_gated_n_bets": len(outcomes),
        })
        search_ranges = weight_search_ranges(center, shrink=1.0 if i == 1 else shrink)
        for k, v in composite_weights.items():
            row[f"w_{k}"] = round(float(v), 4)
            lo, hi = search_ranges.get(k, (v, v))
            row[f"range_{k}"] = f"[{lo:.3g}, {hi:.3g}]"
        rows.append(row)
        center = shrink_weight_center(center, composite_weights)

    return pd.DataFrame(rows)


def confidence_weights_path(name: str = "confidence_weights_by_season.json") -> Path:
    return STATE_DIR / name


def load_latest_confidence_weights() -> dict[str, float]:
    """Load suggested composite weights from inline Phase 2a / tuning JSON."""
    path = confidence_weights_path()
    if not path.exists():
        return dict(DEFAULT_CONFIDENCE_WEIGHTS)
    try:
        with open(path) as f:
            payload = json.load(f)
        if payload.get("inline_phase2a") and payload.get("suggested_weights_latest"):
            out = dict(DEFAULT_CONFIDENCE_WEIGHTS)
            out.update({k: float(v) for k, v in payload["suggested_weights_latest"].items()})
            return out
        suggested = payload.get("suggested_weights_latest")
        if suggested:
            out = dict(DEFAULT_CONFIDENCE_WEIGHTS)
            out.update({k: float(v) for k, v in suggested.items()})
            return out
        tuning = payload.get("unified_weight_tuning")
        if tuning:
            latest = tuning[-1]
            out = dict(DEFAULT_CONFIDENCE_WEIGHTS)
            for k, v in latest.items():
                if str(k).startswith("w_"):
                    out[str(k).replace("w_", "")] = float(v)
            return out
    except (OSError, json.JSONDecodeError, TypeError, ValueError):
        pass
    return dict(DEFAULT_CONFIDENCE_WEIGHTS)


def load_suggested_min_confidence(default: float | None = None) -> float | None:
    """Phase 2a suggested minimum confidence gate (None if not saved)."""

    path = confidence_weights_path()
    if not path.exists():
        return None
    try:
        with open(path) as f:
            payload = json.load(f)
        if "suggested_min_confidence" in payload:
            return float(payload["suggested_min_confidence"])
    except (OSError, json.JSONDecodeError, TypeError, ValueError):
        pass
    return None


def default_calibrator_path(name: str = "bet_calibrator.pkl") -> Path:
    return STATE_DIR / name


In [ ]:
# ── module: bet_selection ────────────────────────────────────────────────────
"""Bet selection helpers: edge-bucket (legacy) vs confidence-only actionable gate."""
from __future__ import annotations

import numpy as np
import pandas as pd



def uses_edge_gates(mode: str | None = None) -> bool:
    """True when |edge| bucket / quantile width filter actionable bets."""
    mode = mode or BET_SELECTION_MODE
    return mode in ("edge_bucket", "edge_bucket_ats")


def edge_bucket_min_for_stakes(mode: str | None = None) -> float | None:
    if not uses_edge_gates(mode):
        return None
    return float(MIN_EDGE_BUCKET)


def analysis_min_edge_pts(effective_threshold: float, mode: str | None = None) -> float:
    """Min edge passed to bet_analysis — 0 in confidence_only so all lines get win%."""
    if not uses_edge_gates(mode):
        return 0.0
    return float(effective_threshold)


def spread_lean_from_edge(edge_pts: float) -> str:
    if edge_pts is None or not np.isfinite(edge_pts) or abs(float(edge_pts)) < 1e-9:
        return "Pass"
    return "Home" if float(edge_pts) > 0 else "Away"


def spread_side_series(df: pd.DataFrame) -> pd.Series:
    """Model lean side for every row (EDGE_LEAN → EDGE sign → DIRECTION)."""
    if "EDGE_LEAN" in df.columns:
        lean = df["EDGE_LEAN"].fillna("Pass").astype(str)
    else:
        lean = df.get("DIRECTION", pd.Series("Pass", index=df.index)).fillna("Pass").astype(str)
    if "EDGE" in df.columns:
        edge = pd.to_numeric(df["EDGE"], errors="coerce").fillna(0)
        need = lean.isin(("Pass", "nan", "")) | lean.isna()
        inferred = np.where(edge > 0, "Home", np.where(edge < 0, "Away", "Pass"))
        lean = lean.where(~need, inferred)
    return lean


def win_pct_column(df: pd.DataFrame) -> str:
    if "WIN_PCT" in df.columns:
        return "WIN_PCT"
    return "CONFIDENCE"


def lean_spread_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Lined games with a model lean (includes sub-threshold edges)."""
    if df is None or df.empty:
        return pd.DataFrame()
    m = df[df["MARKET_SPREAD"].notna()].copy()
    side = spread_side_series(m)
    m = m[side != "Pass"].copy()
    m["_side"] = side.loc[m.index]
    return m


def actionable_spread_frame(df: pd.DataFrame, mode: str | None = None) -> pd.DataFrame:
    """Rows that count as placed spread bets under the active selection mode."""
    if df is None or df.empty:
        return pd.DataFrame()
    mode = mode or BET_SELECTION_MODE
    m = df[df["MARKET_SPREAD"].notna()].copy()
    if uses_edge_gates(mode):
        if "DIRECTION" in m.columns:
            return m[m["DIRECTION"] != "Pass"].copy()
        return lean_spread_frame(m)
    if "ACTIONABLE" in m.columns:
        return m[m["ACTIONABLE"].astype(int) == 1].copy()
    if "DIRECTION" in m.columns:
        return m[m["DIRECTION"] != "Pass"].copy()
    return lean_spread_frame(m)


def spread_bet_frame(df: pd.DataFrame, mode: str | None = None) -> pd.DataFrame:
    """Alias for actionable spread bets (metrics / diagnostics)."""
    return actionable_spread_frame(df, mode)


def passes_edge_bucket(abs_edge: float, min_edge: float | None = None) -> bool:
    if abs_edge is None or not np.isfinite(abs_edge):
        return False
    thr = MIN_EDGE_BUCKET if min_edge is None else min_edge
    return float(abs(abs_edge)) >= float(thr)


def passes_quantile_width(width: float, max_width: float | None = None) -> bool:
    if width is None or not np.isfinite(width):
        return True
    cap = MAX_QUANTILE_WIDTH if max_width is None else max_width
    return float(width) <= float(cap)


def edge_stake_multiplier(abs_edge: float, tiers=EDGE_STAKE_TIERS) -> float:
    ae = float(abs(abs_edge)) if abs_edge is not None and np.isfinite(abs_edge) else 0.0
    for lo, hi, mult in tiers:
        if lo <= ae < hi:
            return float(mult)
    return 1.0


def edge_scaled_min_confidence(abs_edge: float, base: float | None = None) -> float:
    """Higher confidence bar for small edges when CONFIDENCE_EDGE_SCALED is enabled."""
    floor = float(MIN_CONFIDENCE_SCORE if base is None else base)
    if not CONFIDENCE_EDGE_SCALED:
        return floor
    ae = float(abs(abs_edge)) if abs_edge is not None and np.isfinite(abs_edge) else 0.0
    if ae < 5.0:
        return floor + 4.0
    if ae < 8.0:
        return floor + 2.0
    return floor


def passes_edge_avoid_band(
    abs_edge: float,
    elo_meta_agreement: float | None = None,
) -> bool:
    """False when |edge| is in the configured dead-zone band without Elo agreement."""
    if EDGE_AVOID_BAND is None:
        return True
    if abs_edge is None or not np.isfinite(abs_edge):
        return True
    lo, hi = EDGE_AVOID_BAND
    ae = float(abs(abs_edge))
    if not (float(lo) <= ae < float(hi)):
        return True
    agree = float(elo_meta_agreement if elo_meta_agreement is not None else 1.0)
    return agree >= float(EDGE_AVOID_BAND_MIN_ELO_AGREE)


def passes_confidence_actionable_gates(
    *,
    lean: str,
    edge_pts: float,
    conf_score: float,
    conf_width: float,
    min_confidence: float,
    max_confidence: float | None = None,
    disagreement_trust: float = 1.0,
    phantom_injury_flag: bool = False,
    market_spread: float | None = None,
    elo_meta_agreement: float | None = None,
    min_edge: float | None = None,
) -> bool:
    """Shared actionable spread gate for simulate + live predict."""
    if lean in ("Pass", None, "", "nan"):
        return False
    min_edge_val = float(CONFIDENCE_MIN_EDGE if min_edge is None else min_edge)
    if abs(float(edge_pts or 0.0)) < min_edge_val:
        return False
    if not passes_quantile_width(conf_width):
        return False
    if float(disagreement_trust or 1.0) < float(MIN_DISAGREEMENT_TRUST):
        return False
    if SKIP_PHANTOM_INJURY and phantom_injury_flag:
        return False
    if SKIP_TIGHT_SPREAD and market_spread is not None and np.isfinite(market_spread):
        if abs(float(market_spread)) <= float(TIGHT_SPREAD_MAX):
            return False
    if not passes_edge_avoid_band(abs(edge_pts), elo_meta_agreement=elo_meta_agreement):
        return False
    if CONFIDENCE_SELECTION_MODE == "min_score":
        eff_min = edge_scaled_min_confidence(abs(edge_pts), base=min_confidence)
        if float(conf_score) < eff_min:
            return False
        if max_confidence is not None and float(conf_score) > float(max_confidence):
            return False
    return True


def apply_bet_selection_gates(
    direction: str,
    edge_pts: float,
    conf_width: float,
    *,
    mode: str | None = None,
) -> str:
    """Return direction or Pass after edge-bucket / quantile filters (skipped in confidence_only)."""
    if direction == "Pass":
        return direction
    mode = mode or BET_SELECTION_MODE
    if not uses_edge_gates(mode):
        return direction
    if mode in ("edge_bucket", "edge_bucket_ats"):
        if not passes_edge_bucket(abs(edge_pts), min_edge=MIN_EDGE_BUCKET):
            return "Pass"
        if not passes_quantile_width(conf_width, max_width=MAX_QUANTILE_WIDTH):
            return "Pass"
    return direction


def use_tier_stake_gates() -> bool:
    return bool(USE_TIER_STAKE_GATES) and BET_SELECTION_MODE == "legacy_tiers"


In [ ]:
# ── module: artifacts ────────────────────────────────────────────────────────
"""Backtest artifact schema validation, provenance, and config snapshots.

Task 003 expands the run manifest beyond the two flags previously checked so
that loading an artifact built under a different configuration, schema
version, source dataset, feature set, model version, or random seed is
rejected instead of silently reused.
"""
from __future__ import annotations

import hashlib
import json
import subprocess
from pathlib import Path
from typing import Any

import pandas as pd


__all__ = [
    "ARTIFACT_SCHEMA_VERSION",
    "REQUIRED_BACKTEST_COLUMNS",
    "SCHEMA_VERSION_KEYS",
    "backtest_config_snapshot",
    "full_config_snapshot",
    "hash_source_file",
    "git_revision",
    "build_run_manifest",
    "validate_backtest_df",
    "validate_run_manifest",
    "is_backtest_stale",
    "ArtifactMismatchError",
    "assert_manifest_compatible",
    "save_backtest_metadata",
    "load_backtest_metadata",
]


# Fields whose mismatch is always fatal (never a soft warning), because a
# mismatch means the artifact was built under different data/label/feature
# semantics and comparing it to current results would be meaningless.
SCHEMA_VERSION_KEYS = (
    "artifact_schema_version",
    "preprocessing_schema_version",
    "feature_schema_version",
    "market_snapshot_schema_version",
    "validation_schema_version",
)


def backtest_config_snapshot() -> dict[str, Any]:
    """Serializable config flags stored alongside backtest outputs."""
    return {
        "artifact_schema_version": ARTIFACT_SCHEMA_VERSION,
        "preprocessing_schema_version": PREPROCESSING_SCHEMA_VERSION,
        "feature_schema_version": FEATURE_SCHEMA_VERSION,
        "market_snapshot_schema_version": MARKET_SNAPSHOT_SCHEMA_VERSION,
        "validation_schema_version": VALIDATION_SCHEMA_VERSION,
        "BET_SELECTION_MODE": BET_SELECTION_MODE,
        "CONFIDENCE_SELECTION_MODE": CONFIDENCE_SELECTION_MODE,
        "MIN_CONFIDENCE_SCORE": MIN_CONFIDENCE_SCORE,
        "MAX_CONFIDENCE_SCORE": MAX_CONFIDENCE_SCORE,
        "MIN_ML_WIN_PCT": MIN_ML_WIN_PCT,
        "decision_cutoff_minutes_before_tip": DECISION_CUTOFF_MINUTES_BEFORE_TIP,
    }


def full_config_snapshot() -> dict[str, Any]:
    """Every JSON-serializable UPPER_CASE constant in pipeline.config.

    Used for full-provenance artifact manifests (Task 003); intentionally
    broader than `backtest_config_snapshot`, which only carries the flags
    needed for the lighter-weight `validate_backtest_df` compatibility check.
    """
    from pipeline import config as _cfg

    out: dict[str, Any] = {}
    for name in dir(_cfg):
        if not name.isupper():
            continue
        value = getattr(_cfg, name)
        try:
            json.dumps(value)
        except TypeError:
            continue
        out[name] = value
    return out


def hash_source_file(path: Path | str, *, fast: bool = True, chunk_size: int = 1 << 20) -> str | None:
    """Return a content-derived fingerprint for a source data file.

    ``fast=True`` (default) hashes ``(name, size, mtime_ns)`` rather than the
    full file body — sufficient to detect any change to large multi-hundred-MB
    raw PBP/odds CSVs without re-reading them on every run. Pass
    ``fast=False`` for a true SHA-256 content hash when the file is small
    enough (e.g. config snapshots, small fixtures) that a mismatch must be
    proven byte-for-byte.
    """
    p = Path(path)
    if not p.exists():
        return None
    stat = p.stat()
    if fast:
        h = hashlib.sha256()
        h.update(f"{p.name}:{stat.st_size}:{stat.st_mtime_ns}".encode())
        return h.hexdigest()[:24]
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def git_revision() -> str | None:
    """Best-effort short git SHA; None outside a git checkout."""
    try:
        out = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            capture_output=True, text=True, timeout=5,
            cwd=str(Path(__file__).resolve().parent.parent),
        )
        if out.returncode == 0:
            return out.stdout.strip()
    except Exception:
        pass
    return None


def build_run_manifest(
    *,
    source_paths: dict[str, Path | str] | None = None,
    game_count: int | None = None,
    quote_count: int | None = None,
    train_range: tuple[str, str] | None = None,
    calibration_range: tuple[str, str] | None = None,
    test_range: tuple[str, str] | None = None,
    feature_list: list[str] | None = None,
    model_version: str | None = None,
    seeds: dict[str, int] | None = None,
    config_overrides: dict[str, Any] | None = None,
    extra: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Build a full-provenance manifest for one backtest/training run.

    Covers: config snapshot, source file hashes, schema versions, cutoff
    definition, train/calibration/test date ranges, feature list, model
    version, random seeds, and git revision when available.
    """
    source_paths = source_paths or {}
    manifest: dict[str, Any] = {
        "config_snapshot": full_config_snapshot(),
        "config_overrides": config_overrides or {},
        "source_hashes": {
            name: hash_source_file(path) for name, path in source_paths.items()
        },
        "schema_versions": {
            "artifact_schema_version": ARTIFACT_SCHEMA_VERSION,
            "preprocessing_schema_version": PREPROCESSING_SCHEMA_VERSION,
            "feature_schema_version": FEATURE_SCHEMA_VERSION,
            "market_snapshot_schema_version": MARKET_SNAPSHOT_SCHEMA_VERSION,
            "validation_schema_version": VALIDATION_SCHEMA_VERSION,
        },
        "cutoff_definition": {
            "minutes_before_scheduled_tip": DECISION_CUTOFF_MINUTES_BEFORE_TIP,
        },
        "game_count": game_count,
        "quote_count": quote_count,
        "date_ranges": {
            "train": list(train_range) if train_range else None,
            "calibration": list(calibration_range) if calibration_range else None,
            "test": list(test_range) if test_range else None,
        },
        "feature_list": sorted(feature_list) if feature_list else None,
        "model_version": model_version,
        "seeds": seeds or {},
        "git_revision": git_revision(),
        "created_at": pd.Timestamp.utcnow().isoformat(),
    }
    if extra:
        manifest.update(extra)
    return manifest


class ArtifactMismatchError(RuntimeError):
    """Raised when a saved artifact's provenance manifest does not match the
    manifest expected by the currently running code."""


def validate_run_manifest(saved: dict[str, Any], expected: dict[str, Any]) -> list[str]:
    """Return mismatch reasons between a saved manifest and the expected one.

    Every schema-version field is compared strictly. Source hashes, feature
    list, model version, and seeds are compared when both sides provide them.
    """
    reasons: list[str] = []

    saved_schema = saved.get("schema_versions", {})
    expected_schema = expected.get("schema_versions", {})
    for key in SCHEMA_VERSION_KEYS:
        sv = saved_schema.get(key)
        ev = expected_schema.get(key)
        if sv is not None and ev is not None and sv != ev:
            reasons.append(f"{key} mismatch: saved={sv!r} current={ev!r}")

    saved_hashes = saved.get("source_hashes", {}) or {}
    expected_hashes = expected.get("source_hashes", {}) or {}
    for name, ehash in expected_hashes.items():
        shash = saved_hashes.get(name)
        if shash is not None and ehash is not None and shash != ehash:
            reasons.append(f"source hash mismatch for {name!r}: saved={shash!r} current={ehash!r}")

    if expected.get("feature_list") and saved.get("feature_list"):
        if sorted(expected["feature_list"]) != sorted(saved["feature_list"]):
            reasons.append("feature_list mismatch between saved artifact and current run")

    if expected.get("model_version") and saved.get("model_version"):
        if expected["model_version"] != saved["model_version"]:
            reasons.append(
                f"model_version mismatch: saved={saved['model_version']!r} "
                f"current={expected['model_version']!r}"
            )

    saved_cutoff = saved.get("cutoff_definition", {}).get("minutes_before_scheduled_tip")
    expected_cutoff = expected.get("cutoff_definition", {}).get("minutes_before_scheduled_tip")
    if saved_cutoff is not None and expected_cutoff is not None and saved_cutoff != expected_cutoff:
        reasons.append(f"cutoff mismatch: saved={saved_cutoff!r} current={expected_cutoff!r}")

    return reasons


def assert_manifest_compatible(saved: dict[str, Any], expected: dict[str, Any]) -> None:
    """Raise ArtifactMismatchError if `saved` is incompatible with `expected`."""
    reasons = validate_run_manifest(saved, expected)
    if reasons:
        raise ArtifactMismatchError(
            "Artifact manifest incompatible with current run:\n  - " + "\n  - ".join(reasons)
        )


def validate_backtest_df(
    df: pd.DataFrame | None,
    *,
    expected: dict[str, Any] | None = None,
) -> list[str]:
    """Return human-readable reasons the backtest CSV is incompatible with current code."""
    reasons: list[str] = []
    if df is None or df.empty:
        reasons.append("backtest results empty")
        return reasons

    missing = [c for c in REQUIRED_BACKTEST_COLUMNS if c not in df.columns]
    if missing:
        reasons.append(f"missing columns: {missing}")

    expected = expected or backtest_config_snapshot()
    meta_path = default_metadata_path()
    if meta_path.exists():
        try:
            saved = json.loads(meta_path.read_text())
            for key in ("BET_SELECTION_MODE",) + SCHEMA_VERSION_KEYS:
                if key in saved and key in expected and saved[key] != expected[key]:
                    reasons.append(f"{key} mismatch: saved={saved[key]!r} current={expected[key]!r}")
        except (json.JSONDecodeError, OSError):
            reasons.append("backtest_metadata.json unreadable")

    if BET_SELECTION_MODE == "confidence_only" and "ACTIONABLE" not in df.columns:
        reasons.append("confidence_only requires ACTIONABLE column — re-run Phase 2")

    return reasons


def is_backtest_stale(df: pd.DataFrame | None, *, expected: dict[str, Any] | None = None) -> bool:
    return bool(validate_backtest_df(df, expected=expected))


def default_metadata_path() -> Path:
    return STATE_DIR / "backtest_metadata.json"


def save_backtest_metadata(path: Path | None = None, extra: dict[str, Any] | None = None) -> Path:
    path = path or default_metadata_path()
    payload = backtest_config_snapshot()
    if extra:
        payload.update(extra)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str))
    return path


def load_backtest_metadata(path: Path | None = None) -> dict[str, Any]:
    path = path or default_metadata_path()
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text())
    except (json.JSONDecodeError, OSError):
        return {}


In [ ]:
# ── module: market_targets ───────────────────────────────────────────────────
"""Closing-line residual targets for leak-free meta-model training."""
from __future__ import annotations

import numpy as np
import pandas as pd


def closing_spread_series(df: pd.DataFrame) -> pd.Series | None:
    """Pre-tip closing line (home perspective); NaN when unavailable."""
    if df is None or df.empty:
        return None
    if "closing_spread" in df.columns:
        close = pd.to_numeric(df["closing_spread"], errors="coerce")
    elif "market_spread" in df.columns:
        close = pd.to_numeric(df["market_spread"], errors="coerce")
    else:
        return None
    if close.notna().sum() < max(20, int(0.05 * len(close))):
        return None
    return close


def margin_close_residual(actual_margin, closing_spread) -> np.ndarray:
    """ATS residual vs closing line: actual_margin + closing_spread."""
    m = np.asarray(actual_margin, dtype=float)
    c = np.asarray(closing_spread, dtype=float)
    out = m.copy()
    mask = np.isfinite(c)
    out[mask] = m[mask] + c[mask]
    return out


def residual_to_margin(residual, closing_spread) -> np.ndarray:
    """Map predicted close-residual back to predicted home margin."""
    r = np.asarray(residual, dtype=float)
    c = np.asarray(closing_spread, dtype=float)
    out = r.copy()
    mask = np.isfinite(c)
    out[mask] = r[mask] - c[mask]
    return out


def market_total_series(df: pd.DataFrame) -> pd.Series | None:
    """Pre-tip market total; NaN when unavailable."""
    if df is None or df.empty:
        return None
    if "market_total" in df.columns:
        mkt = pd.to_numeric(df["market_total"], errors="coerce")
    else:
        return None
    if mkt.notna().sum() < max(20, int(0.05 * len(mkt))):
        return None
    return mkt


def total_market_residual(actual_total, market_total) -> np.ndarray:
    """O/U residual vs market total: actual_total - market_total."""
    a = np.asarray(actual_total, dtype=float)
    m = np.asarray(market_total, dtype=float)
    out = a.copy()
    mask = np.isfinite(m)
    out[mask] = a[mask] - m[mask]
    return out


def residual_to_total(residual, market_total) -> np.ndarray:
    """Map predicted market-residual back to predicted game total."""
    r = np.asarray(residual, dtype=float)
    m = np.asarray(market_total, dtype=float)
    out = r.copy()
    mask = np.isfinite(m)
    out[mask] = r[mask] + m[mask]
    return out


def valid_market_total_fraction(df: pd.DataFrame) -> float:
    mkt = market_total_series(df)
    if mkt is None:
        return 0.0
    return float(mkt.notna().mean())


def valid_closing_fraction(df: pd.DataFrame) -> float:
    close = closing_spread_series(df)
    if close is None:
        return 0.0
    return float(close.notna().mean())


In [ ]:
# ── module: market_disagreement ──────────────────────────────────────────────
"""Walk-forward market disagreement / phantom-injury gate."""
from __future__ import annotations

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression



def _disagreement_features(
    elo_vs_close: float,
    meta_vs_close: float,
    uncertainty: float = 350.0,
    spread_move: float = 0.0,
    star_out: float = 0.0,
) -> np.ndarray:
    return np.array([
        abs(float(elo_vs_close or 0.0)),
        abs(float(meta_vs_close or 0.0)),
        abs(float(elo_vs_close or 0.0) - float(meta_vs_close or 0.0)),
        np.sign(float(elo_vs_close or 0.0)) * np.sign(float(meta_vs_close or 0.0)),
        float(uncertainty or 350.0) / 350.0,
        abs(float(spread_move or 0.0)),
        float(star_out or 0.0),
    ], dtype=float)


def phantom_injury_flag(
    elo_vs_close: float,
    meta_vs_close: float,
    h_star_out: float = 0.0,
    a_star_out: float = 0.0,
    min_elo_edge: float = 4.0,
    max_meta_edge: float = 2.5,
) -> bool:
    """Large Elo–market gap without rotation flags — market may know injury news."""
    if h_star_out > 0 or a_star_out > 0:
        return False
    elo_e = abs(float(elo_vs_close or 0.0))
    meta_e = abs(float(meta_vs_close or 0.0))
    if elo_e < min_elo_edge:
        return False
    if meta_e > max_meta_edge:
        return False
    if np.sign(elo_vs_close or 0) != np.sign(meta_vs_close or 0) and meta_e >= 1.5:
        return True
    return elo_e >= min_elo_edge + 1.0 and meta_e <= max_meta_edge


class WalkForwardMarketDisagreementModel:
    """Secondary walk-forward model for model-vs-market disagreement nights."""

    def __init__(self, min_train: int = 80, min_elo_edge: float = 3.0):
        self.min_train = min_train
        self.min_elo_edge = min_elo_edge
        self._iso = None
        self._logit = None
        self._fitted = False
        self._phantom_cover_rate = 0.5

    @property
    def fitted(self) -> bool:
        return self._fitted

    def fit(self, prior_df: pd.DataFrame) -> "WalkForwardMarketDisagreementModel":
        if prior_df is None or prior_df.empty:
            return self
        need = {"ACTUAL_MARGIN", "MARKET_SPREAD", "PRED_SPREAD"}
        if not need.issubset(prior_df.columns):
            return self

        d = prior_df[prior_df["MARKET_SPREAD"].notna()].copy()
        close = d["CLOSING_SPREAD"] if "CLOSING_SPREAD" in d.columns else d["MARKET_SPREAD"]
        d["close"] = close
        d["cover"] = d["ACTUAL_MARGIN"] + d["MARKET_SPREAD"]
        d["meta_vs_close"] = d["PRED_SPREAD"] + d["close"]

        elo_col = "ELO_MARGIN_CALIBRATED" if "ELO_MARGIN_CALIBRATED" in d.columns else None
        if elo_col is None:
            return self
        d["elo_vs_close"] = d[elo_col] + d["close"]
        d["abs_elo_edge"] = d["elo_vs_close"].abs()

        active = d[d["abs_elo_edge"] >= self.min_elo_edge]
        if len(active) < self.min_train:
            return self

        X, y, scores = [], [], []
        for _, r in active.iterrows():
            star = max(float(r.get("H_STAR_OUT", 0) or 0), float(r.get("A_STAR_OUT", 0) or 0))
            unc = float(r.get("RATING_UNCERTAINTY", 350) or 350)
            sm = float(r.get("SPREAD_MOVE", 0) or 0)
            elo_e = float(r["elo_vs_close"])
            meta_e = float(r["meta_vs_close"])
            X.append(_disagreement_features(elo_e, meta_e, unc, sm, star))
            edge = float(r["PRED_SPREAD"]) + float(r["MARKET_SPREAD"])
            home_side = edge > 0
            covered = float(r["cover"]) > 0 if home_side else float(r["cover"]) < 0
            y.append(int(covered))
            scores.append(abs(elo_e))

        X_arr = np.vstack(X)
        y_arr = np.asarray(y, dtype=int)

        try:
            self._logit = LogisticRegression(C=0.5, max_iter=300)
            self._logit.fit(X_arr, y_arr)
        except Exception:
            self._logit = None

        try:
            self._iso = IsotonicRegression(out_of_bounds="clip")
            self._iso.fit(scores, y_arr)
        except Exception:
            self._iso = None

        phantom_mask = active.apply(
            lambda r: phantom_injury_flag(
                r["elo_vs_close"], r["meta_vs_close"],
                r.get("H_STAR_OUT", 0), r.get("A_STAR_OUT", 0),
            ),
            axis=1,
        )
        if phantom_mask.sum() >= 15:
            ph = active[phantom_mask]
            ph_edge = ph["PRED_SPREAD"] + ph["MARKET_SPREAD"]
            ph_home = ph_edge > 0
            ph_cover = ph["cover"]
            ph_win = ((ph_home & (ph_cover > 0)) | (~ph_home & (ph_cover < 0))).astype(float)
            self._phantom_cover_rate = float(ph_win.mean())
        else:
            self._phantom_cover_rate = float(y_arr.mean())

        self._fitted = True
        return self

    def predict(
        self,
        *,
        elo_margin_calibrated: float,
        model_spread: float,
        closing_spread: float,
        market_spread: float | None = None,
        uncertainty: float = 350.0,
        spread_move: float = 0.0,
        h_star_out: float = 0.0,
        a_star_out: float = 0.0,
    ) -> dict:
        close = closing_spread if pd.notna(closing_spread) else market_spread
        if close is None or pd.isna(close):
            return {
                "disagreement_trust": 1.0,
                "phantom_injury_flag": False,
                "edge_bump": 0.0,
                "model_side_cover_prob": 0.524,
            }

        elo_vs = float(elo_margin_calibrated or 0.0) + float(close)
        meta_vs = float(model_spread or 0.0) + float(close)
        phantom = phantom_injury_flag(elo_vs, meta_vs, h_star_out, a_star_out)
        feats = _disagreement_features(
            elo_vs, meta_vs, uncertainty, spread_move,
            max(h_star_out, a_star_out),
        ).reshape(1, -1)

        prob = 0.524
        if self._logit is not None:
            try:
                prob = float(self._logit.predict_proba(feats)[0, 1])
            except Exception:
                pass
        elif self._iso is not None and abs(elo_vs) >= self.min_elo_edge:
            try:
                prob = float(self._iso.predict([abs(elo_vs)])[0])
            except Exception:
                pass

        trust = float(np.clip(prob / 0.524, 0.35, 1.25))
        edge_bump = 0.0
        if phantom:
            if self._phantom_cover_rate < 0.50:
                trust *= 0.65
                edge_bump = 1.5
            else:
                edge_bump = 0.5

        if abs(elo_vs) >= 6.0 and abs(elo_vs - meta_vs) >= 4.0:
            edge_bump = max(edge_bump, 1.0)
            trust *= 0.85

        return {
            "disagreement_trust": float(np.clip(trust, 0.2, 1.2)),
            "phantom_injury_flag": bool(phantom),
            "edge_bump": float(edge_bump),
            "model_side_cover_prob": float(np.clip(prob, 0.01, 0.99)),
            "elo_vs_close": elo_vs,
            "meta_vs_close": meta_vs,
        }

    def save(self, path: str | Path) -> None:
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path: str | Path) -> "WalkForwardMarketDisagreementModel":
        with open(path, "rb") as f:
            return pickle.load(f)


def default_disagreement_path() -> Path:
    return Path(STATE_DIR) / "market_disagreement.pkl"


In [ ]:
# ── module: market_snapshots ─────────────────────────────────────────────────
"""Tasks 018-023, 028, 029: a leak-proof quote-level market snapshot layer.

Replaces the game-level "one spread/one ML" dictionary in
:mod:`pipeline.market` with immutable, quote-level rows keyed by
``(GAME_ID, book, market, side, point, quote_timestamp)``. Multiple quotes for
one game remain separate rows (Task 018); every usable quote can compute
``minutes_before_tip`` against the *authoritative* tip time (Task 019, never
inferred from the quotes themselves); in-play/post-tip/ambiguous quotes are
rejected before any selection happens (Task 020); ``open_*``/``decision_*``
(T-60)/``close_*`` are three separate, deterministic selections over the
surviving rows (Tasks 021-023), with ``close_*`` fenced off from T-60 feature
building by ``assert_no_close_leak``. Task 028 keeps point CLV and
price/probability CLV in separate columns. Task 029 audits open/T-60/close
coverage by season/book and flags sources (``all_odds.csv``) whose timing
cannot be proven to be a T-60 snapshot.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

__all__ = [
    "QUOTE_COLUMNS",
    "build_quotes_frame",
    "derive_tip_utc_from_pbp",
    "attach_tip_utc",
    "reject_invalid_quotes",
    "select_open_quotes",
    "select_decision_quotes",
    "select_close_quotes",
    "assert_no_close_leak",
    "build_t60_feature_frame",
    "build_evaluation_frame",
    "point_clv",
    "price_clv",
    "match_pinnacle_quotes_to_games",
    "detect_source_horizon",
    "audit_market_coverage",
    "require_valid_t60_season",
]

# ---------------------------------------------------------------------------
# Task 018: quote-level schema
# ---------------------------------------------------------------------------

GROUP_KEY_COLS = ("GAME_ID", "book", "market", "side", "point")

QUOTE_COLUMNS = (
    "GAME_ID",          # canonical game identity (pipeline.game_results)
    "book",             # e.g. "pinnacle"
    "market",           # "moneyline" | "spread" | "total"
    "side",             # "home" | "away" | "over" | "under"
    "point",            # spread/total line; NaN for moneyline
    "price_american",
    "price_decimal",
    "quote_timestamp",  # when this price was live in the market (UTC)
    "source_timestamp", # book/feed-reported timestamp (UTC)
    "ingestion_timestamp",  # when *we* recorded it (UTC); >= source_timestamp
    "in_play",          # True if quote is known to be posted after tip
)

_REQUIRED = set(QUOTE_COLUMNS)


def build_quotes_frame(rows) -> pd.DataFrame:
    """Validate/construct the quote-level schema from an iterable of dict rows
    or an existing DataFrame. Does not deduplicate — multiple quotes for the
    same game/book/market/side/point at different timestamps are legitimate
    separate rows (Task 018 pass condition)."""
    df = pd.DataFrame(rows) if not isinstance(rows, pd.DataFrame) else rows.copy()
    missing = _REQUIRED - set(df.columns)
    if missing:
        raise ValueError(f"quotes frame missing required columns: {sorted(missing)}")
    df["quote_timestamp"] = pd.to_datetime(df["quote_timestamp"], utc=True, errors="coerce")
    df["source_timestamp"] = pd.to_datetime(df["source_timestamp"], utc=True, errors="coerce")
    df["ingestion_timestamp"] = pd.to_datetime(df["ingestion_timestamp"], utc=True, errors="coerce")
    df["in_play"] = df["in_play"].fillna(False).astype(bool)
    df["price_american"] = pd.to_numeric(df["price_american"], errors="coerce")
    df["price_decimal"] = pd.to_numeric(df["price_decimal"], errors="coerce")
    df["point"] = pd.to_numeric(df["point"], errors="coerce")
    return df.reset_index(drop=True)


# ---------------------------------------------------------------------------
# Task 019: authoritative tip UTC (never inferred from quotes)
# ---------------------------------------------------------------------------

def derive_tip_utc_from_pbp(raw_pbp_df: pd.DataFrame, game_id_col: str = "game_id",
                             time_col: str = "time_actual") -> pd.DataFrame:
    """Authoritative tip time per GAME_ID: the timestamp of the first raw PBP
    event actually recorded for that game (real wall-clock UTC "time_actual"
    column in the combined-stats source — 0% missing, independent of any
    odds/quote feed). This is *not* inferred from a market quote; it comes
    straight from the game itself, satisfying "never infer a tip time from
    the final quote" (Task 019).

    Returns a DataFrame with columns [GAME_ID, tip_utc].
    """
    if game_id_col not in raw_pbp_df.columns or time_col not in raw_pbp_df.columns:
        raise ValueError(f"raw_pbp_df must contain {game_id_col!r} and {time_col!r}")
    ts = pd.to_datetime(raw_pbp_df[time_col], utc=True, errors="coerce")
    out = (
        pd.DataFrame({"GAME_ID": raw_pbp_df[game_id_col], "_ts": ts})
        .dropna(subset=["_ts"])
        .groupby("GAME_ID", sort=False)["_ts"]
        .min()
        .reset_index()
        .rename(columns={"_ts": "tip_utc"})
    )
    return out


def attach_tip_utc(quotes_df: pd.DataFrame, tip_utc_map) -> pd.DataFrame:
    """Left-join authoritative ``tip_utc`` onto quotes by GAME_ID and compute
    ``minutes_before_tip`` for every row whose tip is known. Unmatched games
    get NaT/NaN (missing, never fabricated) rather than a guessed tip time."""
    if isinstance(tip_utc_map, dict):
        tip_df = pd.DataFrame(
            {"GAME_ID": list(tip_utc_map.keys()), "tip_utc": list(tip_utc_map.values())}
        )
    else:
        tip_df = tip_utc_map[["GAME_ID", "tip_utc"]].drop_duplicates("GAME_ID")
    tip_df = tip_df.copy()
    tip_df["tip_utc"] = pd.to_datetime(tip_df["tip_utc"], utc=True, errors="coerce")

    out = quotes_df.merge(tip_df, on="GAME_ID", how="left")
    delta = out["tip_utc"] - out["quote_timestamp"]
    out["minutes_before_tip"] = delta.dt.total_seconds() / 60.0
    return out


# ---------------------------------------------------------------------------
# Task 020: reject in-play / post-tip / ambiguous quotes
# ---------------------------------------------------------------------------

def reject_invalid_quotes(quotes_df: pd.DataFrame) -> pd.DataFrame:
    """Drop quotes that cannot legally enter any pregame snapshot:

    - ``in_play`` True (posted during/after the game);
    - unknown tip time (``tip_utc`` missing — cannot prove it was pregame);
    - ``minutes_before_tip <= 0`` (at-or-after tip);
    - flagged ``match_ambiguous`` (tied to more than one candidate game).
    """
    if "minutes_before_tip" not in quotes_df.columns:
        raise ValueError("call attach_tip_utc() before reject_invalid_quotes()")
    df = quotes_df.copy()
    keep = (~df["in_play"].astype(bool)) & df["minutes_before_tip"].notna() & (df["minutes_before_tip"] > 0)
    if "match_ambiguous" in df.columns:
        keep &= ~df["match_ambiguous"].fillna(False).astype(bool)
    return df.loc[keep].reset_index(drop=True)


# ---------------------------------------------------------------------------
# Tasks 021-023: open / decision (T-60) / close selection
# ---------------------------------------------------------------------------

def _value_cols(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c not in GROUP_KEY_COLS]


def select_open_quotes(df: pd.DataFrame, group_cols=GROUP_KEY_COLS) -> pd.DataFrame:
    """First valid pregame quote per group. Sorting is entirely by content
    (timestamp + full key + price), never by input row order, so the result
    is deterministic under a shuffled input (Task 021 pass condition)."""
    group_cols = list(group_cols)
    sort_cols = group_cols + ["quote_timestamp", "price_decimal", "price_american"]
    ordered = df.sort_values(sort_cols, kind="mergesort", na_position="last")
    out = ordered.groupby(group_cols, as_index=False, sort=False).first()
    value_cols = [c for c in out.columns if c not in group_cols]
    return out.rename(columns={c: f"open_{c}" for c in value_cols})


def select_decision_quotes(
    df: pd.DataFrame, group_cols=GROUP_KEY_COLS, cutoff_minutes: float = 60.0,
) -> pd.DataFrame:
    """Latest valid quote with ``quote_timestamp <= tip_utc - cutoff_minutes``
    per group, i.e. ``minutes_before_tip >= cutoff_minutes`` (Task 022).
    Stores quote age (minutes past the T-60 boundary) and a missingness flag
    for groups with no such quote, rather than silently omitting them."""
    group_cols = list(group_cols)
    all_keys = df[group_cols].drop_duplicates().reset_index(drop=True)

    eligible = df[df["minutes_before_tip"] >= cutoff_minutes]
    sort_cols = group_cols + ["quote_timestamp", "price_decimal", "price_american"]
    eligible = eligible.sort_values(sort_cols, kind="mergesort", na_position="last")
    picked = eligible.groupby(group_cols, as_index=False, sort=False).last()

    value_cols = [c for c in picked.columns if c not in group_cols]
    picked = picked.rename(columns={c: f"decision_{c}" for c in value_cols})
    if "decision_minutes_before_tip" in picked.columns:
        picked["decision_quote_age_minutes"] = picked["decision_minutes_before_tip"] - cutoff_minutes

    out = all_keys.merge(picked, on=group_cols, how="left")
    price_col = "decision_price_decimal" if "decision_price_decimal" in out.columns else None
    out["decision_missing"] = out[price_col].isna() if price_col else True
    return out


def select_close_quotes(df: pd.DataFrame, group_cols=GROUP_KEY_COLS) -> pd.DataFrame:
    """Last valid pre-tip quote per group — **evaluation-only** (Task 023).
    Every returned value column is prefixed ``close_`` so downstream feature
    builders can be mechanically checked for the forbidden prefix."""
    group_cols = list(group_cols)
    sort_cols = group_cols + ["quote_timestamp", "price_decimal", "price_american"]
    ordered = df.sort_values(sort_cols, kind="mergesort", na_position="last")
    out = ordered.groupby(group_cols, as_index=False, sort=False).last()
    value_cols = [c for c in out.columns if c not in group_cols]
    return out.rename(columns={c: f"close_{c}" for c in value_cols})


def assert_no_close_leak(columns) -> None:
    """Raise if any column name begins with ``close_`` (Task 023 guard: "any
    code building T-60 features must reject keys starting with close_")."""
    leaked = [c for c in columns if str(c).startswith("close_")]
    if leaked:
        raise ValueError(f"T-60 feature builder touched evaluation-only close_* columns: {leaked}")


def build_t60_feature_frame(
    quotes_df: pd.DataFrame, group_cols=GROUP_KEY_COLS, cutoff_minutes: float = 60.0,
) -> pd.DataFrame:
    """Open + decision(T-60) columns only — never close_*. Self-checks its own
    output with ``assert_no_close_leak`` before returning."""
    group_cols = list(group_cols)
    opened = select_open_quotes(quotes_df, group_cols)
    decided = select_decision_quotes(quotes_df, group_cols, cutoff_minutes)
    out = opened.merge(decided, on=group_cols, how="outer")
    assert_no_close_leak(out.columns)
    return out


def build_evaluation_frame(
    quotes_df: pd.DataFrame, group_cols=GROUP_KEY_COLS, cutoff_minutes: float = 60.0,
) -> pd.DataFrame:
    """Open + decision(T-60) + close columns, for evaluation/CLV reporting
    only. Never feed this frame's close_* columns into a T-60 feature/model
    input — use build_t60_feature_frame for that."""
    group_cols = list(group_cols)
    t60 = build_t60_feature_frame(quotes_df, group_cols, cutoff_minutes)
    closed = select_close_quotes(quotes_df, group_cols)
    return t60.merge(closed, on=group_cols, how="outer")


# ---------------------------------------------------------------------------
# Task 028: point CLV vs. price/probability CLV — separate columns
# ---------------------------------------------------------------------------

def point_clv(decision_home_point: float, close_home_point: float, side: str) -> float:
    """Bet-side point CLV in raw spread/total points (Phase 3 formulas):

    - Home/Over bet: ``decision_home_point - close_home_point``
    - Away/Under bet: ``close_home_point - decision_home_point``

    This is a *point* difference, not a return — never multiply/compound it
    with ``price_clv`` (Task 028 rule).
    """
    if pd.isna(decision_home_point) or pd.isna(close_home_point):
        return float("nan")
    side = str(side).lower()
    if side in ("home", "over"):
        return float(decision_home_point) - float(close_home_point)
    if side in ("away", "under"):
        return float(close_home_point) - float(decision_home_point)
    raise ValueError(f"unknown side for point_clv: {side!r}")


def price_clv(decision_fair_prob: float, close_fair_prob: float) -> float:
    """Fair-probability CLV for the bet side actually taken: the change in the
    market's own fair (de-vigged) probability for *your side* between decision
    and close. Positive = the market moved toward your side after you bet
    (favorable line/price movement); this is a probability-domain quantity,
    kept in its own column and never combined via a multiplicative return
    identity with ``point_clv``.
    """
    if pd.isna(decision_fair_prob) or pd.isna(close_fair_prob):
        return float("nan")
    return float(close_fair_prob) - float(decision_fair_prob)


# ---------------------------------------------------------------------------
# Task 025 support: matching a two-sided quote feed to canonical games
# (unique home+away identity; ambiguous/unmatched quotes are rejected, never
# backfilled from a different game for the same team).
# ---------------------------------------------------------------------------

def match_pinnacle_quotes_to_games(
    pinnacle_df: pd.DataFrame,
    canonical_games_df: pd.DataFrame,
    *,
    to_tricode,
    match_tolerance_days: int = 1,
    book: str = "pinnacle",
) -> pd.DataFrame:
    """Expand raw ``nba_main_lines.csv``-style rows (team1/team2, decimal
    odds, timestamp) into long-format quote rows tied to exactly one
    canonical ``GAME_ID`` (both teams must match; ambiguous matchup+date
    matches are dropped rather than guessed).

    ``canonical_games_df`` must have GAME_ID, home_team, away_team, raw_date
    (tricode home/away, per pipeline.game_results). ``to_tricode`` is injected
    (pipeline.dates.to_tricode) to avoid a hard import cycle.
    """
    games = canonical_games_df.copy()
    games["home_team"] = games["home_team"].apply(to_tricode)
    games["away_team"] = games["away_team"].apply(to_tricode)
    games["raw_date"] = pd.to_datetime(games["raw_date"], errors="coerce")

    by_matchup: dict = {}
    for _, g in games.dropna(subset=["raw_date"]).iterrows():
        key = frozenset((g["home_team"], g["away_team"]))
        by_matchup.setdefault(key, []).append((g["raw_date"], g["GAME_ID"], g["home_team"], g["away_team"]))

    df = pinnacle_df.copy()
    df["_ts"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["_ts"])
    df["_t1"] = df["team1"].apply(to_tricode)
    df["_t2"] = df["team2"].apply(to_tricode)

    rows = []
    for _, r in df.iterrows():
        t1, t2 = r["_t1"], r["_t2"]
        if not isinstance(t1, str) or not isinstance(t2, str):
            continue
        candidates = by_matchup.get(frozenset((t1, t2)), [])
        ts_date = r["_ts"].normalize()
        close_matches = [
            c for c in candidates if abs((c[0] - ts_date).days) <= match_tolerance_days
        ]
        if len(close_matches) != 1:
            continue  # zero or ambiguous -> reject (Task 020/025), never guess
        _, game_id, home_tri, _away_tri = close_matches[0]
        home_is_t1 = home_tri == t1

        def side_val(home_col, away_col):
            return (r.get(home_col), r.get(away_col)) if home_is_t1 else (r.get(away_col), r.get(home_col))

        ml_home_dec, ml_away_dec = side_val("team1_moneyline", "team2_moneyline")
        sp_home_pt, sp_away_pt = side_val("team1_spread", "team2_spread")
        sp_home_odds, sp_away_odds = side_val("team1_spread_odds", "team2_spread_odds")

        base = dict(
            GAME_ID=game_id, book=book,
            quote_timestamp=r["_ts"], source_timestamp=r["_ts"], ingestion_timestamp=r["_ts"],
            in_play=False,
        )

        def _dec_to_am(dec):
            if pd.isna(dec):
                return np.nan
            try:
                dec = float(dec)
            except (TypeError, ValueError):
                return np.nan
            if not np.isfinite(dec) or dec <= 1.0:
                return np.nan
            return (dec - 1.0) * 100.0 if dec >= 2.0 else -100.0 / (dec - 1.0)

        rows.append({**base, "market": "moneyline", "side": "home", "point": np.nan,
                     "price_decimal": ml_home_dec, "price_american": _dec_to_am(ml_home_dec)})
        rows.append({**base, "market": "moneyline", "side": "away", "point": np.nan,
                     "price_decimal": ml_away_dec, "price_american": _dec_to_am(ml_away_dec)})
        rows.append({**base, "market": "spread", "side": "home", "point": sp_home_pt,
                     "price_decimal": sp_home_odds, "price_american": _dec_to_am(sp_home_odds)})
        rows.append({**base, "market": "spread", "side": "away", "point": sp_away_pt,
                     "price_decimal": sp_away_odds, "price_american": _dec_to_am(sp_away_odds)})
        rows.append({**base, "market": "total", "side": "over", "point": r.get("over_total"),
                     "price_decimal": r.get("over_total_odds"), "price_american": _dec_to_am(r.get("over_total_odds"))})
        rows.append({**base, "market": "total", "side": "under", "point": r.get("under_total"),
                     "price_decimal": r.get("under_total_odds"), "price_american": _dec_to_am(r.get("under_total_odds"))})

    if not rows:
        return build_quotes_frame(pd.DataFrame(columns=list(QUOTE_COLUMNS)))
    return build_quotes_frame(pd.DataFrame(rows))


# ---------------------------------------------------------------------------
# Task 029: coverage audit + source-horizon labeling
# ---------------------------------------------------------------------------

def detect_source_horizon(timestamp_series: pd.Series, *, time_of_day_tolerance_seconds: float = 1.0) -> str:
    """Return ``"unknown_horizon"`` when a quote source's timestamps carry no
    real intra-day precision (e.g. ``all_odds.csv``'s ``game_date`` column is
    always stamped a fixed ``HH:MM`` regardless of actual quote time), else
    ``"timestamped"``.

    A source whose time-of-day component has essentially zero variance cannot
    be proven to represent any particular horizon (open/T-60/close) and must
    not be treated as a T-60 snapshot (Phase 2B / Task 029).
    """
    ts = pd.to_datetime(timestamp_series, errors="coerce").dropna()
    if ts.empty:
        return "unknown_horizon"
    seconds_of_day = (ts.dt.hour * 3600 + ts.dt.minute * 60 + ts.dt.second).astype(float)
    if seconds_of_day.nunique() <= 1 or seconds_of_day.std() <= time_of_day_tolerance_seconds:
        return "unknown_horizon"
    return "timestamped"


def audit_market_coverage(
    canonical_games_df: pd.DataFrame,
    quotes_df: pd.DataFrame,
    *,
    cutoff_minutes: float = 60.0,
    season_col: str = "season_type",
) -> pd.DataFrame:
    """Per (season, book) report of what fraction of canonical games have a
    usable open / T-60(decision) / close quote and an exact price for each,
    across every quote market present in ``quotes_df``.
    """
    games = canonical_games_df[["GAME_ID", season_col]].drop_duplicates("GAME_ID")
    n_games = games.groupby(season_col)["GAME_ID"].nunique()

    if quotes_df.empty:
        return pd.DataFrame(columns=[
            "season", "book", "n_games", "n_with_open", "n_with_decision_t60",
            "n_with_close", "open_coverage", "decision_t60_coverage", "close_coverage",
        ])

    rows = []
    for book, book_df in quotes_df.groupby("book"):
        opened = select_open_quotes(book_df)
        decided = select_decision_quotes(book_df, cutoff_minutes=cutoff_minutes)
        closed = select_close_quotes(book_df)

        opened_g = opened.merge(games, on="GAME_ID", how="inner")
        decided_g = decided.merge(games, on="GAME_ID", how="inner")
        closed_g = closed.merge(games, on="GAME_ID", how="inner")

        for season, n_total in n_games.items():
            n_open = opened_g.loc[opened_g[season_col] == season, "GAME_ID"].nunique()
            n_decision = decided_g.loc[
                (decided_g[season_col] == season) & (~decided_g["decision_missing"]), "GAME_ID"
            ].nunique()
            n_close = closed_g.loc[closed_g[season_col] == season, "GAME_ID"].nunique()
            rows.append({
                "season": season, "book": book, "n_games": int(n_total),
                "n_with_open": int(n_open), "n_with_decision_t60": int(n_decision),
                "n_with_close": int(n_close),
                "open_coverage": n_open / n_total if n_total else 0.0,
                "decision_t60_coverage": n_decision / n_total if n_total else 0.0,
                "close_coverage": n_close / n_total if n_total else 0.0,
            })
    return pd.DataFrame(rows)


def require_valid_t60_season(coverage_df: pd.DataFrame, season, *, min_coverage: float = 0.5) -> None:
    """Raise if ``season`` lacks valid T-60 (decision) coverage across every
    book in the report. Callers must not run market-residual evaluation for a
    season that fails this check (Task 029 / plan rule)."""
    rows = coverage_df[coverage_df["season"] == season]
    if rows.empty or rows["decision_t60_coverage"].max() < min_coverage:
        raise ValueError(
            f"season {season!r} has no book with >= {min_coverage:.0%} T-60 decision-quote "
            "coverage; market-residual/CLV evaluation is blocked for this season."
        )


In [ ]:
# ── module: devig ────────────────────────────────────────────────────────────
"""Task 026/027: de-vig candidate transforms and leakage-free method selection.

Converts a book's raw implied probabilities (``q_i = 1 / decimal_i``, one per
side of a market) into "fair" probabilities with the vig removed, via several
candidate transforms. No transform is assumed universally correct (plan
Phase 2D): the best method depends on book/market/price range/horizon, so
Task 027's selector chooses among them using **only prior-fold** OOS Brier
score / log loss / calibration error, never the fold currently being
evaluated/tested.

Every candidate function returns probabilities that are finite, in ``[0, 1]``,
sum to 1 within numerical tolerance, and preserve the input's favorite/
underdog ordering (monotonicity) — enforced by ``_check_output`` so a bad
solver root cannot silently emit a nonsensical probability vector.
"""
from __future__ import annotations

import numpy as np

__all__ = [
    "DevigError",
    "multiplicative_devig",
    "power_devig",
    "odds_ratio_devig",
    "shin_devig",
    "DEVIG_METHODS",
    "devig",
    "brier_score",
    "log_loss_score",
    "calibration_error",
    "select_devig_method_from_folds",
    "store_devig_choice",
]


class DevigError(ValueError):
    """Raised when raw implied probabilities or a devig result fail a guard."""


_EPS = 1e-9


def _validate_probs(q) -> np.ndarray:
    q = np.asarray(q, dtype=float)
    if q.ndim != 1 or q.size < 2:
        raise DevigError("q must be a 1-D array of at least 2 implied probabilities")
    if not np.all(np.isfinite(q)):
        raise DevigError("implied probabilities must be finite")
    if np.any(q <= 0) or np.any(q >= 1):
        raise DevigError("implied probabilities must lie strictly inside (0, 1)")
    return q


def _check_output(p, q: np.ndarray) -> np.ndarray:
    """Numerical guards shared by every candidate transform (Task 026)."""
    p = np.asarray(p, dtype=float)
    if not np.all(np.isfinite(p)):
        raise DevigError("devig output is not finite")
    if np.any(p < -1e-9) or np.any(p > 1 + 1e-9):
        raise DevigError("devig output outside [0, 1]")
    p = np.clip(p, 0.0, 1.0)
    total = float(p.sum())
    if not np.isclose(total, 1.0, atol=1e-6):
        raise DevigError(f"devig output does not sum to 1 (sum={total})")
    p = p / total  # exact renormalization after clipping
    # Monotonicity: a larger raw implied probability must map to a larger (or
    # equal) fair probability — the transform may not reorder favorite/dog.
    rank_in = np.argsort(np.argsort(q))
    rank_out = np.argsort(np.argsort(p))
    if not np.array_equal(rank_in, rank_out):
        raise DevigError("devig output re-ordered favorite/underdog vs. input")
    return p


def multiplicative_devig(q) -> np.ndarray:
    """p_i = q_i / sum(q)."""
    q = _validate_probs(q)
    p = q / q.sum()
    return _check_output(p, q)


def _bisect(f, lo, hi, tol=1e-10, max_iter=200):
    flo, fhi = f(lo), f(hi)
    if flo == 0.0:
        return lo
    if fhi == 0.0:
        return hi
    if flo * fhi > 0:
        return None
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        fmid = f(mid)
        if abs(fmid) < tol:
            return mid
        if flo * fmid < 0:
            hi = mid
        else:
            lo, flo = mid, fmid
    return 0.5 * (lo + hi)


def power_devig(q, k_bounds=(0.05, 20.0)) -> np.ndarray:
    """Solve sum(q_i^k) = 1 for k, then p_i = q_i^k (Shin/Jullien favorite-
    longshot power correction)."""
    q = _validate_probs(q)

    def f(k):
        return float(np.sum(q ** k) - 1.0)

    k = _bisect(f, *k_bounds)
    if k is None:
        return multiplicative_devig(q)
    p = q ** k
    return _check_output(p, q)


def odds_ratio_devig(q) -> np.ndarray:
    """Odds-ratio / logit-shift method: solve for c such that
    sum_i (c*q_i) / (1 - q_i + c*q_i) = 1."""
    q = _validate_probs(q)

    def transform(c):
        return (c * q) / (1.0 - q + c * q)

    def f(c):
        return float(transform(c).sum() - 1.0)

    # c spans orders of magnitude; bisect in log-space for stability.
    def f_log(x):
        return f(float(np.exp(x)))

    x = _bisect(f_log, -20.0, 20.0)
    if x is None:
        return multiplicative_devig(q)
    c = float(np.exp(x))
    p = transform(c)
    return _check_output(p, q)


def _shin_transform(q: np.ndarray, z: float, sigma: float) -> np.ndarray:
    disc = z ** 2 + 4.0 * (1.0 - z) * (q ** 2) / sigma
    disc = np.clip(disc, 0.0, None)
    denom = 2.0 * (1.0 - z)
    return (np.sqrt(disc) - z) / denom


def shin_devig(q) -> np.ndarray:
    """Shin (1992/1993) insider-trading/allocation model.

    Solves for the market's implied "insider fraction" z such that the
    resulting fair probabilities sum to 1. Uses the standard closed-form
    functional relationship, generalized to n outcomes (reduces to the
    textbook two-way formula when ``len(q) == 2``).
    """
    q = _validate_probs(q)
    sigma = float(q.sum())

    def f(z):
        return float(_shin_transform(q, z, sigma).sum() - 1.0)

    z = _bisect(f, 1e-9, 1.0 - 1e-6)
    if z is None:
        return multiplicative_devig(q)
    p = _shin_transform(q, z, sigma)
    return _check_output(p, q)


DEVIG_METHODS = {
    "multiplicative": multiplicative_devig,
    "power": power_devig,
    "odds_ratio": odds_ratio_devig,
    "shin": shin_devig,
}


def devig(q, method: str = "multiplicative") -> np.ndarray:
    if method not in DEVIG_METHODS:
        raise DevigError(f"unknown devig method: {method!r} (have {list(DEVIG_METHODS)})")
    return DEVIG_METHODS[method](q)


# ---------------------------------------------------------------------------
# Task 027: leakage-free method selection.
# ---------------------------------------------------------------------------

def brier_score(p_pred, outcome) -> float:
    p = np.asarray(p_pred, dtype=float)
    y = np.asarray(outcome, dtype=float)
    return float(np.mean((p - y) ** 2))


def log_loss_score(p_pred, outcome, eps: float = 1e-12) -> float:
    p = np.clip(np.asarray(p_pred, dtype=float), eps, 1 - eps)
    y = np.asarray(outcome, dtype=float)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def calibration_error(p_pred, outcome, n_bins: int = 10) -> float:
    """Mean |predicted - observed| across equal-width probability bins
    (sample-weighted expected calibration error)."""
    p = np.asarray(p_pred, dtype=float)
    y = np.asarray(outcome, dtype=float)
    if p.size == 0:
        return float("nan")
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
    err_sum = 0.0
    total = 0
    for b in range(n_bins):
        mask = idx == b
        n = int(mask.sum())
        if n == 0:
            continue
        err_sum += n * abs(float(p[mask].mean()) - float(y[mask].mean()))
        total += n
    return float(err_sum / total) if total else float("nan")


def select_devig_method_from_folds(
    folds: list[dict],
    *,
    methods: list[str] | None = None,
    global_default: str = "multiplicative",
    shrink_min_n: int = 200,
    loss_fn=brier_score,
):
    """Choose a de-vig method for each chronological fold using only the
    Brier/log-loss/calibration performance accumulated over *strictly prior*
    folds. Never looks at the fold it is choosing a method for.

    Parameters
    ----------
    folds : chronologically ordered list of ``{"q": [...], "outcome": [...]}``
        where each element of ``q`` is an array-like of raw implied
        probabilities for one game/quote and the matching element of
        ``outcome`` is the realized 0/1 result for ``q[0]``'s side (e.g. home
        win / home cover).
    shrink_min_n : minimum accumulated prior-fold sample size before trusting
        a data-driven "best method" pick over ``global_default`` (hierarchical
        shrinkage to the global default when evidence is thin).

    Returns
    -------
    (chosen, fold_losses) : ``chosen[i]`` is the method used for ``folds[i]``
    (decided from ``folds[:i]`` only); ``fold_losses[i]`` is each method's
    realized OOS loss on ``folds[i]`` (recorded for diagnostics/next-fold
    history, but ``chosen[i]`` never depends on ``fold_losses[i]``).
    """
    if methods is None:
        methods = list(DEVIG_METHODS)
    history = {m: {"loss_sum": 0.0, "n": 0} for m in methods}
    chosen: list[str] = []
    fold_losses: list[dict] = []

    for fold in folds:
        n_prior = min(v["n"] for v in history.values())
        if n_prior < shrink_min_n:
            method_for_fold = global_default
        else:
            avg_loss = {
                m: (history[m]["loss_sum"] / history[m]["n"]) for m in methods
            }
            method_for_fold = min(avg_loss, key=avg_loss.get)
        chosen.append(method_for_fold)

        losses_this_fold = {}
        for m in methods:
            preds = []
            outs = []
            for q, outcome in zip(fold["q"], fold["outcome"]):
                p = devig(q, method=m)
                preds.append(float(p[0]))
                outs.append(float(outcome))
            loss = loss_fn(preds, outs) if preds else float("nan")
            losses_this_fold[m] = loss
            if preds:
                history[m]["loss_sum"] += loss * len(preds)
                history[m]["n"] += len(preds)
        fold_losses.append(losses_this_fold)

    return chosen, fold_losses


def store_devig_choice(
    store: dict,
    key: tuple,
    *,
    method: str,
    n_obs: int,
    global_default: str = "multiplicative",
    shrink_min_n: int = 200,
) -> str:
    """Record/shrink the chosen method for one ``(book, market, horizon)`` key.

    With fewer than ``shrink_min_n`` observations the stored choice is forced
    back to ``global_default`` regardless of what looked best on the (small,
    noisy) available sample — hierarchical shrinkage per Task 027.
    """
    effective = method if n_obs >= shrink_min_n else global_default
    store[key] = {"method": effective, "n_obs": int(n_obs), "requested": method}
    return effective


In [ ]:
# ── module: bet_grading ──────────────────────────────────────────────────────
"""Tasks 037-040: centralized bet grading, exact-price profit, and CLV.

Canonical win/loss/push outcomes and exact-price P&L live here so
``pipeline/metrics.py`` and ``pipeline/stake_profiles.py`` cannot drift into
incompatible push/CLV definitions again.
"""
from __future__ import annotations

import numpy as np
import pandas as pd



def grade_spread_bet(actual_margin, market_home_spread, side: str) -> str:
    """Return ``win``, ``loss``, or ``push`` for a spread bet.

    ``actual_margin`` is home_score - away_score. ``market_home_spread`` is the
    home line (negative when home is favored). ``side`` is ``Home`` or ``Away``.
    """
    if pd.isna(actual_margin) or pd.isna(market_home_spread):
        raise ValueError("actual_margin and market_home_spread are required")
    side_l = str(side).strip().lower()
    if side_l not in ("home", "away"):
        raise ValueError(f"unknown spread side: {side!r}")
    cover = float(actual_margin) + float(market_home_spread)
    if cover == 0.0:
        return "push"
    home_covers = cover > 0.0
    if side_l == "home":
        return "win" if home_covers else "loss"
    return "win" if not home_covers else "loss"


def grade_total_bet(actual_total, market_total, side: str) -> str:
    """Return ``win``, ``loss``, or ``push`` for an over/under bet."""
    if pd.isna(actual_total) or pd.isna(market_total):
        raise ValueError("actual_total and market_total are required")
    side_l = str(side).strip().lower()
    if side_l not in ("over", "under"):
        raise ValueError(f"unknown total side: {side!r}")
    diff = float(actual_total) - float(market_total)
    if diff == 0.0:
        return "push"
    went_over = diff > 0.0
    if side_l == "over":
        return "win" if went_over else "loss"
    return "win" if not went_over else "loss"


def bet_side_point_clv(decision_home_spread, close_home_spread, side: str) -> float:
    """Bet-side point CLV (Task 038 / Phase 3).

    Home: ``decision_home_spread - close_home_spread``
    Away: ``close_home_spread - decision_home_spread``
    """
    return point_clv(decision_home_spread, close_home_spread, side)


def bet_side_price_clv(decision_fair_prob, close_fair_prob) -> float:
    """Probability-domain CLV; never compound with point CLV."""
    return price_clv(decision_fair_prob, close_fair_prob)


def _to_decimal_odds(price) -> float:
    """Accept American or decimal odds; return decimal payout multiplier."""
    if pd.isna(price):
        return float("nan")
    p = float(price)
    if p == 0.0:
        return float("nan")
    # Decimal odds are typically in (1, ~20]; American are <= -100 or >= +100.
    if abs(p) >= 100.0 or p <= -100.0:
        return float(american_to_decimal(p))
    if p > 1.0:
        return p
    return float("nan")


def exact_price_profit(stake: float, outcome: str, price) -> float:
    """Profit using the actual accepted side price (Task 039).

    ``outcome`` is ``win`` / ``loss`` / ``push``. Pushes return 0 (Task 040).
    ``price`` may be American (e.g. -110, +150) or decimal (e.g. 1.91).
    """
    stake = float(stake or 0.0)
    if stake <= 0.0:
        return 0.0
    outcome_l = str(outcome).strip().lower()
    if outcome_l == "push":
        return 0.0
    if outcome_l == "loss":
        return -stake
    if outcome_l != "win":
        raise ValueError(f"unknown outcome: {outcome!r}")
    dec = _to_decimal_odds(price)
    if not np.isfinite(dec) or dec <= 1.0:
        raise ValueError(f"invalid price for exact_price_profit: {price!r}")
    return stake * (dec - 1.0)


def hit_rate(outcomes) -> float:
    """Win rate excluding pushes from the denominator (Task 040)."""
    arr = [str(o).lower() for o in outcomes if str(o).lower() != "push"]
    if not arr:
        return float("nan")
    return float(sum(1 for o in arr if o == "win") / len(arr))


In [ ]:
# ── module: stake_profiles ───────────────────────────────────────────────────
"""Stake sizing profiles for ATS, ML, and O/U bets."""
from __future__ import annotations

import numpy as np


PROFILES = {
    "conservative": {
        "edge_buffer": 1.0,
        "kelly_fraction": 0.10,
        "max_daily_exposure": 0.05,
        "min_confidence_tier": 3,
    },
    "moderate": {
        "edge_buffer": 0.0,
        "kelly_fraction": 0.25,
        "max_daily_exposure": 0.12,
        "min_confidence_tier": 2,
    },
    "aggressive": {
        "edge_buffer": -0.5,
        "kelly_fraction": 0.40,
        "max_daily_exposure": 0.20,
        "min_confidence_tier": 2,
    },
    "edge_moderate": {
        "edge_buffer": 0.0,
        "kelly_fraction": 0.25,
        "max_daily_exposure": 0.12,
        "min_confidence_tier": 1,
    },
}


def profile_edge_threshold(base_threshold: float, profile: str) -> float:
    cfg = PROFILES[profile]
    return max(1.0, base_threshold + cfg["edge_buffer"])


def compute_stake(
    profile: str,
    *,
    direction: str,
    edge_pts: float,
    edge_threshold: float,
    cover_prob: float,
    confidence_tier: int,
    conf_width: float = 24.0,
    rating_uncertainty: float = 350.0,
    juice: float = -110,
    edge_stake_mult: float = 1.0,
    edge_bucket_min: float | None = None,
    volatility_mult: float = 1.0,
    use_tier_gates: bool | None = None,
    confidence_score: int | None = None,
) -> float:
    """Return bankroll fraction to stake (0 if no bet)."""
    if direction == "Pass":
        return 0.0
    cfg = PROFILES[profile]
    if uses_edge_gates():
        thr = profile_edge_threshold(edge_threshold, profile)
        if abs(edge_pts) < thr:
            return 0.0
        if edge_bucket_min is not None and abs(edge_pts) < edge_bucket_min:
            return 0.0
    tier_gates = use_tier_stake_gates() if use_tier_gates is None else use_tier_gates
    if tier_gates and CONFIDENCE_SELECTION_MODE != "min_score" and confidence_tier < cfg["min_confidence_tier"]:
        return 0.0

    kelly = spread_kelly_fraction(cover_prob, juice=juice)
    interval_penalty = min(1.0, 24.0 / max(conf_width, 6.0))
    unc_penalty = max(0.0, 1.0 - (rating_uncertainty - 150) / 400)

    kelly_frac = cfg["kelly_fraction"]
    if STAKE_SIZING_MODE == "edge_scaled":
        kelly_frac = min(kelly_frac, KELLY_FRACTION_CAP)
        norm_width = min(1.0, max(0.0, (float(conf_width) - 12.0) / 24.0))
        interval_penalty *= max(0.0, 1.0 - norm_width)

    stake = kelly * kelly_frac * interval_penalty * unc_penalty
    stake *= float(edge_stake_mult) * float(volatility_mult)
    if CONFIDENCE_STAKE_MODE and confidence_score is not None:
        stake *= float(np.clip(confidence_score / 100.0, 0.35, 1.0))
    return float(max(0.0, min(stake, cfg["max_daily_exposure"])))


def compute_ml_stake(
    profile: str,
    *,
    ml_direction: str,
    win_prob: float,
    market_ml: float,
    confidence_tier: int,
    max_favorite_decimal: float = 1.45,
    use_tier_gates: bool | None = None,
) -> float:
    if ml_direction == "Pass" or pd_isna(market_ml):
        return 0.0
    cfg = PROFILES[profile]
    tier_gates = use_tier_stake_gates() if use_tier_gates is None else use_tier_gates
    if tier_gates and confidence_tier < cfg["min_confidence_tier"]:
        return 0.0
    dec_home = american_to_decimal(market_ml)
    dec_away = american_to_decimal(-market_ml)
    if ml_direction == "Home":
        dec = dec_home
        p = win_prob
    else:
        dec = dec_away
        p = 1.0 - win_prob
    if dec < max_favorite_decimal:
        return 0.0
    b = dec - 1.0
    if b <= 0:
        return 0.0
    kelly = max(0.0, (p * b - (1 - p)) / b)
    kf = cfg["kelly_fraction"]
    if STAKE_SIZING_MODE == "edge_scaled":
        kf = min(kf, KELLY_FRACTION_CAP)
    return float(min(kelly * kf, cfg["max_daily_exposure"]))


def pd_isna(x):
    try:
        import pandas as pd
        return pd.isna(x)
    except Exception:
        return x is None or (isinstance(x, float) and np.isnan(x))


def apply_daily_caps(
    df,
    stake_col: str,
    date_col: str = "DATE",
    profile: str = "moderate",
    slate_correlation_penalty: float | None = None,
):
    """Cap same-day total exposure for a stake column."""
    if df is None or df.empty or stake_col not in df.columns:
        return df
    out = df.copy()
    cap = PROFILES[profile]["max_daily_exposure"]
    rho = SLATE_CORRELATION_PENALTY if slate_correlation_penalty is None else slate_correlation_penalty
    if date_col not in out.columns:
        return out
    for _, idx in out.groupby(date_col).groups.items():
        stakes = out.loc[idx, stake_col].fillna(0.0)
        total = stakes.sum()
        n_active = int((stakes > 0).sum())
        scale = 1.0
        if n_active > SLATE_CORRELATION_MIN_BETS and rho > 0:
            scale = 1.0 / (1.0 + rho * (n_active - 1))
        if total > cap and total > 0:
            out.loc[idx, stake_col] = stakes * (cap / total) * scale
        elif scale < 1.0:
            out.loc[idx, stake_col] = stakes * scale
    return out


def ats_profit(stake: float, won: bool, juice: float = -110) -> float:
    """Legacy boolean helper. Prefer ``exact_price_profit`` + ``grade_*`` for
    push-aware, exact-price grading (Tasks 037-040)."""
    if stake <= 0:
        return 0.0
    return exact_price_profit(stake, "win" if won else "loss", juice)


# ---------------------------------------------------------------------------
# Tasks 044-047: fractional Kelly, uncertainty shrink, slate sizing, bankroll
# ---------------------------------------------------------------------------

FRACTIONAL_KELLY_CANDIDATES = (0.10, 0.25, 0.50)


def kelly_fraction(p: float, decimal_odds: float) -> float:
    """Exact Kelly ``f* = (bp - q) / b`` (Task 044). Non-positive -> 0."""
    if p is None or not np.isfinite(p) or decimal_odds is None or not np.isfinite(decimal_odds):
        return 0.0
    b = float(decimal_odds) - 1.0
    if b <= 0.0:
        return 0.0
    q = 1.0 - float(p)
    f_star = (b * float(p) - q) / b
    return float(max(0.0, f_star))


def robust_fractional_kelly(
    p: float,
    decimal_odds: float,
    *,
    market_fair_p: float | None = None,
    uncertainty: float = 0.0,
    fraction: float = 0.25,
) -> float:
    """Shrink ``p`` toward market fair / lower-bound before fractional Kelly.

    Task 045: greater ``uncertainty`` must never increase the recommended stake.
    Full Kelly (fraction=1.0) is never the production default.
    """
    if fraction <= 0.0:
        return 0.0
    fraction = float(min(max(fraction, 0.0), 0.50))  # never full Kelly in production
    p = float(p) if p is not None and np.isfinite(p) else 0.0
    unc = float(max(0.0, uncertainty or 0.0))
    # Posterior lower bound: subtract uncertainty mass from p.
    p_robust = max(0.0, p - unc)
    if market_fair_p is not None and np.isfinite(market_fair_p):
        # Shrink toward market fair; more uncertainty → more shrink.
        shrink = min(1.0, unc)
        p_robust = (1.0 - shrink) * p_robust + shrink * float(market_fair_p)
    f_star = kelly_fraction(p_robust, decimal_odds)
    return float(max(0.0, f_star * fraction))


def _nearest_psd(cov: np.ndarray) -> np.ndarray:
    """Project a symmetric matrix onto the PSD cone (eigenvalue clip)."""
    cov = 0.5 * (cov + cov.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.clip(eigvals, 0.0, None)
    return eigvecs @ np.diag(eigvals) @ eigvecs.T


def optimize_slate_stakes(
    raw_stakes: np.ndarray,
    cov: np.ndarray | None = None,
    *,
    shrink: float = 0.85,
    per_bet_cap: float = 0.05,
    slate_cap: float = 0.12,
    game_caps: np.ndarray | None = None,
) -> np.ndarray:
    """Jointly size simultaneous bets with a heavily shrunk covariance (Task 046).

    ``shrink`` toward independence (identity scaled by diagonal variances).
    Hard caps: per-bet and slate total; optional per-game caps.
    """
    stakes = np.asarray(raw_stakes, dtype=float).copy()
    n = len(stakes)
    if n == 0:
        return stakes
    stakes = np.clip(stakes, 0.0, per_bet_cap)
    if cov is None:
        cov = np.eye(n)
    else:
        cov = np.asarray(cov, dtype=float)
        if cov.shape != (n, n):
            raise ValueError(f"cov shape {cov.shape} != ({n}, {n})")
        var = np.diag(cov).copy()
        var[var <= 0] = 1.0
        indep = np.diag(var)
        cov = (1.0 - shrink) * cov + shrink * indep
        cov = _nearest_psd(cov)
        # Soft risk penalty: scale down stakes whose correlated exposure is high.
        risk = np.sqrt(np.clip(np.diag(cov @ np.diag(stakes) @ cov), 0.0, None) + 1e-12)
        # Keep identity path (shrink=1) unchanged; only damp correlated books.
        if shrink < 1.0 - 1e-12:
            scale = 1.0 / (1.0 + risk)
            stakes = stakes * scale
            stakes = np.clip(stakes, 0.0, per_bet_cap)
    if game_caps is not None:
        # game_caps is a vector of group ids; cap sum within each group.
        game_caps = np.asarray(game_caps)
        for gid in np.unique(game_caps):
            mask = game_caps == gid
            total = stakes[mask].sum()
            cap = per_bet_cap  # one bet-equivalent per game by default
            if total > cap > 0:
                stakes[mask] *= cap / total
    total = stakes.sum()
    if total > slate_cap > 0:
        stakes *= slate_cap / total
    # Final PSD check for callers that inspect cov separately is in tests.
    return stakes


def simulate_bankroll(
    win_probs: np.ndarray,
    decimal_odds: np.ndarray,
    fractions: np.ndarray,
    *,
    n_paths: int = 2000,
    seed: int = 42,
    miscalibration: float = 0.0,
    correlation: float = 0.0,
) -> dict:
    """Posterior-predictive bankroll Monte Carlo (Task 047).

    Returns growth / loss / drawdown stats. Worse ``miscalibration`` or higher
    ``correlation`` must never produce a *less* conservative recommended stake
    fraction when used by ``recommend_fraction_under_risk``.
    """
    rng = np.random.default_rng(seed)
    p = np.asarray(win_probs, dtype=float)
    dec = np.asarray(decimal_odds, dtype=float)
    f = np.asarray(fractions, dtype=float)
    n = len(p)
    # Apply miscalibration: true win rate = p - miscalibration (harder).
    p_true = np.clip(p - float(miscalibration), 0.01, 0.99)
    # Correlated Bernoulli via Gaussian copula with equicorrelation.
    rho = float(np.clip(correlation, 0.0, 0.95))
    if n == 0:
        return {
            "expected_log_growth": 0.0,
            "prob_loss": 0.0,
            "p05_return": 0.0,
            "p10_return": 0.0,
            "max_drawdown": 0.0,
            "time_under_water": 0.0,
        }
    mean = np.zeros(n)
    cov = (1.0 - rho) * np.eye(n) + rho * np.ones((n, n))
    z = rng.multivariate_normal(mean, cov, size=n_paths)
    u = 0.5 * (1.0 + np.tanh(z / np.sqrt(2.0)))  # cheap normal-cdf approx
    wins = u < p_true[None, :]
    # Wealth path: start at 1.0, apply each bet sequentially within a path.
    wealth = np.ones(n_paths)
    peak = np.ones(n_paths)
    max_dd = np.zeros(n_paths)
    underwater = np.zeros(n_paths)
    log_growth = np.zeros(n_paths)
    for j in range(n):
        payoff = np.where(wins[:, j], 1.0 + f[j] * (dec[j] - 1.0), 1.0 - f[j])
        wealth *= np.maximum(payoff, 1e-12)
        peak = np.maximum(peak, wealth)
        dd = (wealth - peak) / peak
        max_dd = np.minimum(max_dd, dd)
        underwater += (wealth < 1.0).astype(float)
    log_growth = np.log(np.maximum(wealth, 1e-12))
    returns = wealth - 1.0
    return {
        "expected_log_growth": float(np.mean(log_growth)),
        "prob_loss": float(np.mean(returns < 0.0)),
        "p05_return": float(np.quantile(returns, 0.05)),
        "p10_return": float(np.quantile(returns, 0.10)),
        "max_drawdown": float(np.mean(max_dd)),
        "time_under_water": float(np.mean(underwater / max(n, 1))),
    }


def recommend_fraction_under_risk(
    win_probs: np.ndarray,
    decimal_odds: np.ndarray,
    *,
    candidates=FRACTIONAL_KELLY_CANDIDATES,
    max_drawdown_tol: float = -0.25,
    miscalibration: float = 0.0,
    correlation: float = 0.0,
    seed: int = 42,
) -> float:
    """Pick the largest fractional Kelly that still clears drawdown tolerance.

    Pre-registered warning/stop: if even 0.10 violates tolerance, return 0
    (abstain). Worse calibration / higher correlation can only shrink or hold
    the recommended fraction — never increase it.
    """
    best = 0.0
    for frac in sorted(candidates):
        fractions = np.full(len(win_probs), float(frac))
        stats = simulate_bankroll(
            win_probs, decimal_odds, fractions,
            miscalibration=miscalibration, correlation=correlation, seed=seed,
        )
        if stats["p10_return"] >= max_drawdown_tol and stats["expected_log_growth"] > 0:
            best = float(frac)
    return best


In [ ]:
# ── module: oof ──────────────────────────────────────────────────────────────
"""Manual past-only OOF stacking (Tasks 049, 050, 051).

Replaces sklearn's ``StackingRegressor`` + ``ChronologicalPartitionCV`` for
production meta-model fitting. sklearn's ``StackingRegressor`` calls
``cross_val_predict`` internally, which *requires* every training row to be
covered by exactly one held-out fold (a partition). A strictly past-only
splitter (``PastOnlyGroupCV``) cannot satisfy that for its earliest block —
there is no strictly-earlier data to train on — so sklearn's mechanism
cannot be used at all for a leak-free stack; this module implements the OOF
generation manually instead:

- Every held-out (OOF) base-learner prediction comes from a base learner
  trained only on strictly earlier rows (Task 051; ``max(train_time) <
  min(validation_time)`` for every fold, enforced by ``PastOnlyGroupCV``
  and re-checked here defensively).
- Rows in the first (burn-in) block are never covered by any fold and are
  *dropped* from the meta-learner's training set rather than imputed or
  predicted by a future-trained model (Rule 5/6; Task 051's explicit
  "omit early burn-in rows" allowance).
- Every fitted component (each base estimator's final refit, and the
  meta/final estimator) is instrumented with ``fit_max_timestamp`` — the
  maximum group/timestamp value it was trained on (Task 050/058).
- Fold-local preprocessing: callers are expected to fit any scaler/imputer
  *inside* each fold (see ``fit_transform_fn`` hook) rather than on the
  full frame beforehand; this module fits base learners fold-locally by
  construction, and exposes the same fold boundaries for callers that also
  need to cross-fit a feature (see ``cross_fit_derived_feature``).
"""
from __future__ import annotations

import copy
from dataclasses import dataclass, field

import numpy as np



class LeakageError(RuntimeError):
    """Raised when a fold or lineage check proves a leak is present."""


def _check_fold_chronology(groups, train_idx, val_idx):
    if len(train_idx) == 0 or len(val_idx) == 0:
        return
    max_train_t = groups[train_idx].max()
    min_val_t = groups[val_idx].min()
    if not (max_train_t < min_val_t):
        raise LeakageError(
            "Fold violates max(train_time) < min(validation_time): "
            f"{max_train_t!r} >= {min_val_t!r}"
        )


@dataclass
class ManualOOFStacker:
    """Past-only manual OOF stacking regressor.

    Parameters
    ----------
    estimators : list[(name, estimator)]
        Base learners with sklearn-like ``fit``/``predict``.
    final_estimator : sklearn-like estimator
        Meta-learner fit on the OOF base predictions.
    n_splits : int
        Number of past-only folds requested from ``PastOnlyGroupCV``.
    cv : splitter or None
        Defaults to ``PastOnlyGroupCV(n_splits=n_splits)``.
    """

    estimators: list
    final_estimator: object
    n_splits: int = 5
    cv: object = None

    def __post_init__(self):
        if self.cv is None:
            self.cv = PastOnlyGroupCV(n_splits=self.n_splits)
        self.estimators_: list = []
        self.final_estimator_ = None
        self.fit_max_timestamp = None
        self.oof_rows_used_ = 0
        self.n_burn_in_dropped_ = 0
        self.fold_lineage_: list = []
        self.fitted = False

    def fit(self, X, y, groups=None):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n = len(X)
        groups = np.arange(n) if groups is None else np.asarray(groups)
        if len(groups) != n:
            raise ValueError("groups must be the same length as X")

        n_est = len(self.estimators)
        oof = np.full((n, n_est), np.nan, dtype=float)
        covered = np.zeros(n, dtype=bool)

        folds = list(self.cv.split(X, y, groups))
        if not folds:
            raise LeakageError(
                "ManualOOFStacker: PastOnlyGroupCV produced zero folds — "
                "not enough distinct groups/timestamps to guarantee a "
                "past-only split. Refusing to fall back to an unsafe CV."
            )

        for train_idx, val_idx in folds:
            _check_fold_chronology(groups, train_idx, val_idx)
            self.fold_lineage_.append({
                "train_max_t": groups[train_idx].max(),
                "val_min_t": groups[val_idx].min(),
                "n_train": len(train_idx),
                "n_val": len(val_idx),
            })
            for j, (_, est) in enumerate(self.estimators):
                fold_est = copy.deepcopy(est)
                fold_est.fit(X[train_idx], y[train_idx])
                oof[val_idx, j] = fold_est.predict(X[val_idx])
            covered[val_idx] = True

        self.oof_rows_used_ = int(covered.sum())
        self.n_burn_in_dropped_ = int((~covered).sum())
        if self.oof_rows_used_ == 0:
            raise LeakageError(
                "ManualOOFStacker: no row was ever validated past-only; "
                "cannot train the meta-learner without leaking."
            )
        if np.isnan(oof[covered]).any():
            raise LeakageError(
                "ManualOOFStacker: covered rows have NaN OOF predictions — "
                "internal fold bookkeeping bug, refusing to train on gaps."
            )

        meta_X = oof[covered]
        meta_y = y[covered]
        self.final_estimator_ = copy.deepcopy(self.final_estimator)
        self.final_estimator_.fit(meta_X, meta_y)

        overall_max_t = groups.max()
        self.estimators_ = []
        for name, est in self.estimators:
            fitted = copy.deepcopy(est)
            fitted.fit(X, y)
            setattr(fitted, "fit_max_timestamp", overall_max_t)
            self.estimators_.append((name, fitted))
        setattr(self.final_estimator_, "fit_max_timestamp", overall_max_t)
        self.fit_max_timestamp = overall_max_t
        self.fitted = True
        return self

    def _base_predictions(self, X):
        X = np.asarray(X, dtype=float)
        return np.column_stack([est.predict(X) for _, est in self.estimators_])

    def predict(self, X):
        if not self.fitted:
            raise RuntimeError("ManualOOFStacker must be fit() before predict().")
        return self.final_estimator_.predict(self._base_predictions(X))


def cross_fit_derived_feature(fit_fn, predict_fn, X_feat, y_target, groups, cv=None, n_splits=5):
    """Past-only cross-fit a single derived feature (Task 052).

    Used for target-fitted engine outputs such as ``elo_stack_pred`` that
    are *features* fed into a downstream stack rather than base learners
    inside it. Returns ``(oof_values, covered_mask, fit_final)`` where
    ``fit_final`` is the artifact refit on all rows (for predict-time use,
    with ``fit_max_timestamp`` set) and ``oof_values``/``covered_mask``
    give a leak-free training column (burn-in rows are NaN/uncovered and
    must be dropped from meta-training, matching ManualOOFStacker).

    Parameters
    ----------
    fit_fn : callable(X_train, y_train) -> artifact
    predict_fn : callable(artifact, X_apply) -> np.ndarray
    """
    X_feat = np.asarray(X_feat, dtype=float)
    y_target = np.asarray(y_target, dtype=float)
    n = len(X_feat)
    groups = np.arange(n) if groups is None else np.asarray(groups)
    cv = cv or PastOnlyGroupCV(n_splits=n_splits)

    oof = np.full(n, np.nan, dtype=float)
    covered = np.zeros(n, dtype=bool)
    folds = list(cv.split(X_feat, y_target, groups))
    if not folds:
        raise LeakageError(
            "cross_fit_derived_feature: PastOnlyGroupCV produced zero folds."
        )
    for train_idx, val_idx in folds:
        _check_fold_chronology(groups, train_idx, val_idx)
        artifact = fit_fn(X_feat[train_idx], y_target[train_idx])
        oof[val_idx] = predict_fn(artifact, X_feat[val_idx])
        covered[val_idx] = True

    fit_final = fit_fn(X_feat, y_target)
    setattr(fit_final, "fit_max_timestamp", groups.max())
    return oof, covered, fit_final


In [ ]:
# ── module: model ────────────────────────────────────────────────────────────
"""Meta-model and feature engineering."""
from __future__ import annotations

import numpy as np
import pandas as pd
import optuna
from optuna.pruners import MedianPruner
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, brier_score_loss
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import TimeSeriesSplit

# ═══════════════════════════════════════════════════════════════════════════
#   ADAPTIVE SHRINKING TUNER — with plateau and exploration jitter
# ═══════════════════════════════════════════════════════════════════════════

class AdaptiveParameterBounds:
    """
    Tracks the history of best Optuna values across seasons and produces
    progressively narrower search ranges centered on the running best.
    Includes plateau (stops shrinking after N seasons) and occasional jitter
    to force exploration and avoid premature convergence.
    """
    DECAY_RATE = 0.85              # slower decay (was 0.75)
    MIN_RANGE_FRACTION = 0.25      # keep at least 25% of original range (was 10%)
    MIN_RANGE_FRACTION_TIGHT = 0.35  # when a param hits bounds repeatedly
    PLATEAU_AFTER = 5              # stop shrinking after this many seasons
    INITIAL_SHRINK = 0.50          # after first season, use ±50% of original range
    EXPLORE_JITTER = 0.05          # random ±5% jitter around the center (applied 30% of the time)

    # Optuna search names → stored config keys (Elo/Hier rename on return).
    PARAM_ALIASES = {
        "k_off": ("K_OFF",),
        "k_def": ("K_DEF",),
        "elo_scaling": ("ELO_SCALING_FACTOR",),
        "home_boost": ("HOME_PPP_BOOST",),
        "offseason_reversion": ("OFFSEASON_REVERSION",),
        "usage_floor": ("USAGE_FLOOR",),
        "k_mult_half_life": ("k_mult_half_life",),
        "rd_floor": ("rd_floor",),
        "garbage_time_weight": ("garbage_time_weight",),
        "clutch_boost": ("clutch_boost",),
        "tov_penalty": ("tov_penalty",),
        "foul_draw_boost": ("foul_draw_boost",),
        "variance_dampen": ("variance_dampen",),
        "xppp_actual_blend": ("xppp_actual_blend",),
        "k_def_events": ("k_def_events",),
        "league_rtg": ("league_avg_rtg",),
        "elo_blend_alpha": ("elo_blend_alpha",),
        "elo_ridge_alpha": ("elo_ridge_alpha",),
    }
    # Categorical / non-numeric params — never used as shrink centers.
    NON_ADAPTIVE_PARAMS = frozenset({"margin_cap", "total_elo_beta"})

    def __init__(self):
        self.history = {}          # {season_int: {"param_name": best_value, ...}}
        self.seasons_seen = 0
        self._bound_hits: dict[str, int] = {}

    @classmethod
    def _sanitize_params(cls, best_params: dict) -> dict:
        cleaned = {}
        for key, value in best_params.items():
            if key in cls.NON_ADAPTIVE_PARAMS or value is None:
                continue
            if isinstance(value, (int, float)) and not np.isfinite(value):
                continue
            cleaned[key] = value
        return cleaned

    def record_best(self, season: int, best_params: dict, param_bounds: dict | None = None):
        cleaned = self._sanitize_params(best_params)
        if season in self.history:
            self.history[season].update(cleaned)
        else:
            self.history[season] = cleaned
        self.seasons_seen = len(self.history)
        if param_bounds:
            for key, val in cleaned.items():
                if key not in param_bounds or not isinstance(val, (int, float)):
                    continue
                lo, hi = param_bounds[key]
                span = max(float(hi) - float(lo), 1e-9)
                edge = min(abs(float(val) - float(lo)), abs(float(hi) - float(val))) / span
                if edge <= 0.02:
                    self._bound_hits[key] = self._bound_hits.get(key, 0) + 1
        print(f"✅ AdaptiveParameterBounds: Recorded best params for season {season}. "
              f"Total seasons: {self.seasons_seen}.")

    def _shrink_factor(self) -> float:
        if self.seasons_seen == 0:
            return 1.0
        min_frac = self.MIN_RANGE_FRACTION
        if max(self._bound_hits.values(), default=0) >= 2:
            min_frac = max(min_frac, self.MIN_RANGE_FRACTION_TIGHT)
        if self.seasons_seen >= self.PLATEAU_AFTER:
            return min_frac
        raw = self.INITIAL_SHRINK * (self.DECAY_RATE ** (self.seasons_seen - 1))
        return max(raw, min_frac)

    def _running_best(self) -> dict:
        if not self.history:
            return {}
        latest_season = max(self.history.keys())
        return self.history[latest_season]

    def _lookup_center(self, param_name: str, best: dict,
                       original_low: float, original_high: float):
        def _as_center(value):
            if value is None:
                return None
            try:
                center = float(value)
            except (TypeError, ValueError):
                return None
            if not np.isfinite(center):
                return None
            return float(np.clip(center, original_low, original_high))

        if param_name in best:
            center = _as_center(best[param_name])
            if center is not None:
                return center
        for alt in self.PARAM_ALIASES.get(param_name, ()):
            if alt in best:
                center = _as_center(best[alt])
                if center is not None:
                    return center
        return None

    @staticmethod
    def _ensure_valid_float_range(low: float, high: float,
                                  original_low: float, original_high: float) -> tuple:
        """Guarantee low < high; fall back to full original range if needed."""
        if high > low:
            return low, high
        span = max(original_high - original_low, 1e-9)
        eps = max(span * 0.05, 1e-6)
        mid = float(np.clip((low + high) / 2.0, original_low, original_high))
        low = max(original_low, mid - eps)
        high = min(original_high, mid + eps)
        if high > low:
            return low, high
        return original_low, original_high

    def suggest_bounds_float(self, param_name: str, original_low: float, original_high: float,
                             log: bool = False) -> tuple:
        best = self._running_best()
        shrink = self._shrink_factor()
        original_span = original_high - original_low

        if self.seasons_seen == 0:
            return original_low, original_high

        center = self._lookup_center(param_name, best, original_low, original_high)
        if center is None:
            return original_low, original_high
        # Apply jitter with 30% probability to force exploration
        if self.seasons_seen > 2 and np.random.random() < 0.3:
            center = center * (1 + np.random.uniform(-self.EXPLORE_JITTER, self.EXPLORE_JITTER))
            center = float(np.clip(center, original_low, original_high))

        half_width = (original_span * shrink) / 2.0

        if log:
            import math
            center = float(np.clip(center, original_low, original_high))
            log_center = math.log(max(center, original_low + 1e-12))
            log_low = math.log(original_low)
            log_high = math.log(original_high)
            log_span = log_high - log_low
            half_log = (log_span * shrink) / 2.0
            new_low = math.exp(max(log_low, log_center - half_log))
            new_high = math.exp(min(log_high, log_center + half_log))
        else:
            new_low  = max(original_low,  center - half_width)
            new_high = min(original_high, center + half_width)

        return self._ensure_valid_float_range(
            new_low, new_high, original_low, original_high,
        )

    def suggest_bounds_int(self, param_name: str, original_low: int, original_high: int) -> tuple:
        best = self._running_best()
        shrink = self._shrink_factor()
        original_span = original_high - original_low

        if self.seasons_seen == 0:
            return original_low, original_high

        center_f = self._lookup_center(param_name, best, float(original_low), float(original_high))
        if center_f is None:
            return original_low, original_high
        center = int(round(center_f))
        # Apply jitter with 30% probability
        if self.seasons_seen > 2 and np.random.random() < 0.3:
            center = int(round(center * (1 + np.random.uniform(-self.EXPLORE_JITTER, self.EXPLORE_JITTER))))
            center = int(np.clip(center, original_low, original_high))

        half_width = max(1, int(round((original_span * shrink) / 2.0)))

        new_low  = max(original_low,  center - half_width)
        new_high = min(original_high, center + half_width)

        new_low, new_high = self._ensure_valid_float_range(
            float(new_low), float(new_high), float(original_low), float(original_high),
        )
        low_i = int(round(new_low))
        high_i = int(round(new_high))
        if high_i <= low_i:
            low_i = max(original_low, center - 1)
            high_i = min(original_high, center + 1)
        if high_i <= low_i:
            return original_low, original_high
        return low_i, high_i

    def report(self):
        print(f"\n{'═'*60}")
        print(f"  📊 AdaptiveParameterBounds Report")
        print(f"  Seasons recorded: {self.seasons_seen}")
        print(f"  Current range shrink factor: {self._shrink_factor():.3f}x of original")
        if self.history:
            latest = self._running_best()
            print("  Current best params (running anchor):")
            for k, v in latest.items():
                print(f"    {k:30s}: {v:.5f}" if isinstance(v, float) else f"    {k:30s}: {v}")
        print(f"{'═'*60}\n")

# ──────────────────────────────────────────────────────────────────────────────
# SAFE FEATURE SET – no look‑ahead, no future data
# ──────────────────────────────────────────────────────────────────────────────

def engineer_interaction_features(df):
    """
    Adds explicit non‑linear interaction features to the feature DataFrame.
    This makes them available to both Ridge and XGBoost.
    """
    df_eng = df.copy()

    # Quality × Pace
    df_eng['elo_pace_interaction'] = df_eng['elo_net'] * df_eng['exp_poss']

    # Quality × Rest differential
    rest_diff = df_eng['h_rest'] - df_eng['a_rest']
    df_eng['elo_rest_interaction'] = df_eng['elo_net'] * rest_diff

    # Form (rolling xPPP net) × Pace
    df_eng['form_pace_interaction'] = df_eng['roll_net_xppp'] * df_eng['exp_poss']

    # Uncertainty‑adjusted quality
    total_uncertainty = df_eng['h_rating_uncertainty'] + df_eng['a_rating_uncertainty']
    df_eng['elo_uncertainty_adj'] = df_eng['elo_net'] / (total_uncertainty + 1e-6)

    # Team net rating (pts/100) × pace — blends box-score form with tempo
    if 'off_rtg_net' in df_eng.columns:
        df_eng['rtg_pace_interaction'] = df_eng['off_rtg_net'] * df_eng['exp_poss']

    # Extended engine margin features (Phase 3)
    if 'elo_margin' in df_eng.columns and 'exp_poss' in df_eng.columns:
        df_eng['elo_margin_per100'] = df_eng['elo_margin'] / (df_eng['exp_poss'] + 1e-6) * 100.0
    if 'elo_margin' in df_eng.columns and 'hier_margin' in df_eng.columns:
        df_eng['engine_margin_spread'] = df_eng['elo_margin'] - df_eng['hier_margin']
    if 'elo_margin' in df_eng.columns and 'pace_diff' in df_eng.columns:
        df_eng['elo_margin_pace'] = df_eng['elo_margin'] * df_eng['pace_diff']

    if 'elo_margin_calibrated' in df_eng.columns and 'market_spread' in df_eng.columns:
        df_eng['elo_edge_pts'] = df_eng['elo_margin_calibrated'] + df_eng['market_spread'].fillna(0)
        meta_col = next(
            (c for c in ('pred_margin', 'elo_hier_blend', 'hier_margin', 'elo_margin')
             if c in df_eng.columns),
            None,
        )
        if meta_col is not None:
            meta_edge = df_eng[meta_col] + df_eng['market_spread'].fillna(0)
            df_eng['elo_meta_agreement'] = (
                np.sign(df_eng['elo_edge_pts']) == np.sign(meta_edge)
            ).astype(float)

    return df_eng

# In Cell 10, update SAFE_FEATURE_COLS


# Already-computed game-context features (previously built but never fed to the model).
CONTEXT_FEATURE_COLS = [
    "h_experience", "a_experience", "days_since_season_start",
    "h_new_starters", "a_new_starters",
]

# Direct engine-implied margins so the stack can weight the engines directly.
ENGINE_MARGIN_COLS = [
    "elo_margin", "hier_margin", "elo_margin_per100", "engine_margin_spread", "elo_margin_pace",
    "elo_margin_calibrated", "elo_hier_blend", "elo_vs_hier_spread", "elo_vs_market",
    "elo_luck_adj_net", "elo_def_event_rate", "elo_tov_rate", "elo_matchup_asym",
    "elo_margin_z", "elo_consistency", "elo_edge_pts",
]

ELO_INTERACTION_COLS = [
    "elo_pace_interaction", "elo_rest_interaction", "elo_uncertainty_adj", "elo_margin_pace",
    "elo_meta_agreement",
]

# Market context features (available pre-tip; not used for spread edge directly).
MARKET_TOTAL_COLS = ["market_total", "market_total_minus_league"]
MARKET_LINE_COLS = ["spread_move", "public_home_pct", "market_fair_win_prob"]

# Elo-only inputs for the two-stage stack ridge head (expanded for calibration).
ELO_STACK_FEATURES = [
    "elo_margin", "elo_margin_calibrated", "elo_hier_blend", "elo_vs_hier_spread",
    "elo_net", "elo_diff_off", "elo_diff_def", "exp_poss", "elo_pace_interaction",
    "lineup5_net", "team_elo_spread", "team_elo_net",
    "uncertainty_diff", "elo_uncertainty_adj", "h_rating_uncertainty",
    "elo_vs_market", "elo_luck_adj_net", "elo_def_event_rate", "elo_tov_rate",
    "elo_matchup_asym", "elo_margin_z", "elo_consistency",
]

ELO_WIN_FEATURES = [
    "elo_net", "uncertainty_diff", "elo_margin_calibrated", "elo_vs_market",
    "elo_matchup_asym", "elo_consistency", "elo_margin_z", "h_rating_uncertainty",
    "a_rating_uncertainty",
]

# Schedule-density / fatigue features derived from game dates.
SCHEDULE_FEATURE_COLS = [
    "h_games_last7", "a_games_last7", "games_last7_diff", "h_3in4", "a_3in4",
    "h_4in6", "a_4in6",
    "h_rest_0", "h_rest_1", "h_rest_2", "h_rest_3plus",
    "a_rest_0", "a_rest_1", "a_rest_2", "a_rest_3plus",
    "h_travel_rest_interaction", "a_travel_rest_interaction", "travel_rest_interaction_diff",
]

AVAILABILITY_IMPACT_COLS = [
    "h_expected_off_drop", "a_expected_off_drop",
    "h_expected_def_drop", "a_expected_def_drop",
    "h_expected_pace_delta", "a_expected_pace_delta",
    "expected_off_drop_diff", "expected_def_drop_diff", "expected_pace_delta_diff",
]

# Team-specific home-court edge (rolling venue-split margins).
HCA_FEATURE_COLS = ["h_home_edge", "a_road_edge", "hca_net"]

# Core (engine + tempo + interaction + SOS) features, excluding the toggleable
# groups below.  Kept as its own list so the ablation harness can rebuild subsets.
CORE_FEATURE_COLS = [
    "elo_net", "hier_net",
    "h_elo_off", "h_elo_def", "a_elo_off", "a_elo_def",
    "elo_diff_off", "elo_diff_def",
    "h_hier_off", "h_hier_def", "a_hier_off", "a_hier_def",
    "exp_poss", "h_rest", "a_rest", "h_b2b", "a_b2b",
    "is_altitude", "h_season_phase", "a_season_phase",
    "h_rating_uncertainty", "a_rating_uncertainty", "uncertainty_diff",
    "h_roll_off_xppp", "h_roll_def_xppp", "a_roll_off_xppp", "a_roll_def_xppp",
    "roll_net_xppp",
    "elo_pace_interaction", "elo_rest_interaction", "form_pace_interaction", "elo_uncertainty_adj",
    "rtg_pace_interaction",
    "h_recent_net", "a_recent_net", "recent_diff",
    "pace_diff", "pace_abs_diff", "pace_interaction",
    "h_sos", "a_sos", "sos_diff",
    "elo_margin_per100", "engine_margin_spread", "elo_margin_pace",
    "elo_margin_calibrated", "elo_hier_blend", "elo_vs_hier_spread", "elo_vs_market",
]

# Extended features from roadmap modules (chemistry, lineup Elo, travel, team Elo, market micro).
SHOT_QUALITY_COLS = [
    "h_xefg", "a_xefg", "shot_quality_edge",
    "h_rim_rate", "a_rim_rate", "rim_rate_diff",
    "h_three_rate", "a_three_rate", "three_rate_diff",
    "avg_shot_distance_diff",
    "h_elo_def_rim", "a_elo_def_rim", "h_elo_def_peri", "a_elo_def_peri",
    "rim_def_diff", "peri_def_diff",
]

LINEUP_COMPOSITE_COLS = [
    "h_lineup_composite", "a_lineup_composite", "lineup_composite_diff",
    "h_chem_duo_net", "a_chem_duo_net", "h_chem_trio_net", "a_chem_trio_net",
]

HAPM_COLS = ["h_hapm_net", "a_hapm_net", "hapm_net_diff"]

EXTENDED_FEATURE_COLS = [
    "h_lineup5_off", "h_lineup5_def", "h_lineup5_chem", "a_lineup5_off", "a_lineup5_def", "a_lineup5_chem",
    "lineup5_net", "lineup5_chem_diff", "lineup5_sample_min",
    "h_chem_net", "a_chem_net", "chem_diff", "h_onoff_net", "a_onoff_net", "chem_uncertainty", "usage_conflict",
    *LINEUP_COMPOSITE_COLS,
    *SHOT_QUALITY_COLS,
    *HAPM_COLS,
    "team_elo_spread", "team_elo_total_adj", "team_elo_net",
    "h_travel_miles_7d", "a_travel_miles_7d", "travel_miles_diff", "h_tz_shift", "a_tz_shift",
    "h_road_trip", "a_road_trip",
    "h_fatigue_index", "a_fatigue_index", "fatigue_diff",
    "reverse_line_movement", "steam_flag", "fair_spread_vigfree", "public_away_pct",
    "h_star_out", "a_star_out", "epm_prior_diff", "epm_blend_off_diff",
    "ref_pace_bias", "ref_foul_bias",
]

MARKET_MICRO_COLS = ["reverse_line_movement", "steam_flag", "fair_spread_vigfree", "public_away_pct"]

SAFE_FEATURE_COLS = [
    *CORE_FEATURE_COLS,
    *FORM_FEATURE_COLS,
    *MULTIWINDOW_FORM_COLS,
    *CONTEXT_FEATURE_COLS,
    *SCHEDULE_FEATURE_COLS,
    *HCA_FEATURE_COLS,
    *ENGINE_MARGIN_COLS,
    *MARKET_TOTAL_COLS,
    *MARKET_LINE_COLS,
    *EXTENDED_FEATURE_COLS,
    *AVAILABILITY_IMPACT_COLS,
    "elo_stack_pred",
]

# Moneyline: strength, Elo, efficiency, HCA, availability, rest, market movement.
# Do NOT share identical matrix with totals.
WIN_FEATURE_COLS = [
    "elo_net", "hier_net", "team_elo_spread", "team_elo_net",
    "elo_margin", "elo_margin_calibrated", "elo_hier_blend", "elo_vs_market",
    "elo_diff_off", "elo_diff_def", "elo_matchup_asym", "elo_consistency", "elo_margin_z",
    "h_elo_off", "h_elo_def", "a_elo_off", "a_elo_def",
    "h_off_rtg", "h_def_rtg", "a_off_rtg", "a_def_rtg", "off_rtg_net",
    "h_off_rtg_adj", "a_off_rtg_adj", "h_def_rtg_adj", "a_def_rtg_adj", "off_rtg_adj_net",
    "h_home_edge", "a_road_edge", "hca_net", "is_altitude",
    "h_star_out", "a_star_out", "lineup5_net", "lineup_composite_diff",
    "h_hapm_net", "a_hapm_net", "hapm_net_diff", "epm_prior_diff",
    *AVAILABILITY_IMPACT_COLS,
    "h_rest", "a_rest", "h_b2b", "a_b2b",
    "h_rest_0", "h_rest_1", "h_rest_2", "h_rest_3plus",
    "a_rest_0", "a_rest_1", "a_rest_2", "a_rest_3plus",
    "h_3in4", "a_3in4", "travel_miles_diff", "h_fatigue_index", "a_fatigue_index", "fatigue_diff",
    "h_travel_rest_interaction", "a_travel_rest_interaction",
    "spread_move", "public_home_pct", "market_fair_win_prob",
    "fair_spread_vigfree", "reverse_line_movement", "steam_flag",
    "uncertainty_diff", "h_rating_uncertainty", "a_rating_uncertainty",
    "h_recent_net", "a_recent_net", "recent_diff",
    "sos_diff",
]

# Totals / scoring: pace, efficiency pairs, shot quality, fatigue — de-emphasize Elo.
TOTAL_FEATURE_COLS = [
    "exp_poss", "pace_diff", "pace_abs_diff", "pace_interaction",
    "h_rest", "a_rest", "h_b2b", "a_b2b",
    "h_rest_0", "h_rest_1", "h_rest_2", "h_rest_3plus",
    "a_rest_0", "a_rest_1", "a_rest_2", "a_rest_3plus",
    "h_3in4", "a_3in4", "h_4in6", "a_4in6",
    "h_travel_rest_interaction", "a_travel_rest_interaction", "travel_rest_interaction_diff",
    "market_total", "market_total_minus_league",
    "ref_pace_bias", "ref_foul_bias",
    "rtg_pace_interaction", "form_pace_interaction",
    "h_fatigue_index", "a_fatigue_index", "fatigue_diff",
    "h_roll_off_xppp", "a_roll_off_xppp", "h_roll_def_xppp", "a_roll_def_xppp",
    "roll_net_xppp",
    "h_off_rtg", "a_off_rtg", "h_def_rtg", "a_def_rtg",
    "h_off_rtg_adj", "a_off_rtg_adj", "h_def_rtg_adj", "a_def_rtg_adj",
    "h_efg", "a_efg", "efg_diff",
    "h_ftr", "a_ftr", "ftr_diff",
    "h_tov_pct", "a_tov_pct", "tov_pct_diff",
    "h_oreb_pct", "a_oreb_pct", "h_dreb_pct", "a_dreb_pct",
    "h_xefg", "a_xefg", "shot_quality_edge",
    "h_rim_rate", "a_rim_rate", "rim_rate_diff",
    "h_three_rate", "a_three_rate", "three_rate_diff",
    "h_sos", "a_sos", "sos_diff",
    "h_pace", "a_pace",
    "h_expected_off_drop", "a_expected_off_drop",
    "h_expected_pace_delta", "a_expected_pace_delta",
    "expected_off_drop_diff", "expected_pace_delta_diff",
    # Light multi-window pace/efficiency (totals care about short-term scoring form)
    "h_off_rtg_roll_3", "a_off_rtg_roll_3", "off_rtg_diff_roll_3",
    "h_off_rtg_roll_5", "a_off_rtg_roll_5", "off_rtg_diff_roll_5",
    "h_off_rtg_roll_10", "a_off_rtg_roll_10",
    "h_def_rtg_roll_5", "a_def_rtg_roll_5",
    "h_efg_roll_5", "a_efg_roll_5",
    "h_pts_std_roll_5", "a_pts_std_roll_5",
    "h_pts_std_roll_10", "a_pts_std_roll_10",
]

WIN_SPREAD_DERIVED_COLS = ["meta_pred_margin", "spread_quantile_width", "ats_classifier_prob"]


def total_feature_cols(df: pd.DataFrame | None = None) -> list[str]:
    """Pace/total-focused feature subset (columns present in df when provided)."""
    base = list(dict.fromkeys(TOTAL_FEATURE_COLS + MARKET_TOTAL_COLS))
    if df is None:
        return base
    return [c for c in base if c in df.columns]


def win_feature_cols(df: pd.DataFrame | None = None, use_spread_features: bool | None = None) -> list[str]:
    """Feature columns for MetaWinModel — market-specific, not full SAFE set."""
    use = ML_USE_SPREAD_FEATURES if use_spread_features is None else use_spread_features
    cols = list(WIN_FEATURE_COLS)
    if use:
        extra = WIN_SPREAD_DERIVED_COLS
        if df is not None:
            extra = [c for c in extra if c in df.columns]
        cols.extend([c for c in extra if c not in cols])
    if df is None:
        return cols
    present = [c for c in cols if c in df.columns]
    # Fall back to SAFE intersection if curated list too sparse on older frames
    if len(present) < 15:
        return [c for c in SAFE_FEATURE_COLS if c in df.columns]
    return present

# Default values (kept for compatibility)
DEFAULT_XGB_PARAMS = {
    'max_depth': 2,
    'learning_rate': 0.01,
    'n_estimators': 300,
    'min_child_weight': 100,
    'colsample_bytree': 0.4,
    'subsample': 0.5,
    'reg_alpha': 10.0,
    'reg_lambda': 10.0,
    'gamma': 5.0,
    'objective': 'reg:pseudohubererror',   # Huber loss for robustness
    'huber_slope': 1.0,
    'eval_metric': 'mae',
    'early_stopping_rounds': 50,
    'n_jobs': -1,
    'random_state': 42
}
DEFAULT_RIDGE_ALPHA = 50.0
DEFAULT_BLEND_WEIGHT = 0.70   # kept only for signature compatibility

# ──────────────────────────────────────────────────────────────────────────────
# STACKED META MODEL – Ridge + XGBoost Residuals with Early Stopping
# ──────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, HuberRegressor, LogisticRegression
from sklearn.ensemble import StackingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import mean_absolute_error, brier_score_loss
from sklearn.model_selection import TimeSeriesSplit
from catboost import CatBoostRegressor
import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import KFold

MARGIN_CAP_CHOICES = [20.0, 25.0, 30.0, 35.0, 40.0, 45.0, None]


class _ScaledEstimator:
    """Bundles a fitted scaler + estimator as one object so cross-fit
    helpers (``pipeline.oof.cross_fit_derived_feature``) can attach
    ``fit_max_timestamp`` to a single artifact (Task 052)."""

    def __init__(self, scaler, model):
        self.scaler = scaler
        self.model = model

    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))


def elo_implied_total_from_row(row) -> float | None:
    """Pace-adjusted total from team offensive/defensive Elo ratings."""
    if isinstance(row, dict):
        exp_poss = row.get("exp_poss")
        ho = float(row.get("h_elo_off") or 1500)
        hd = float(row.get("h_elo_def") or 1500)
        ao = float(row.get("a_elo_off") or 1500)
        ad = float(row.get("a_elo_def") or 1500)
    else:
        exp_poss = row.get("exp_poss")
        ho = float(row.get("h_elo_off") or 1500)
        hd = float(row.get("h_elo_def") or 1500)
        ao = float(row.get("a_elo_off") or 1500)
        ad = float(row.get("a_elo_def") or 1500)
    if exp_poss is None or pd.isna(exp_poss):
        return None
    scaling = 1000.0
    hb = 0.002
    ppp_h = DEFAULT_LEAGUE_XPPP + hb + (ho - ad) / scaling
    ppp_a = DEFAULT_LEAGUE_XPPP - hb + (ao - hd) / scaling
    return float((ppp_h + ppp_a) * float(exp_poss))


def _stack_cv_splitter(fast_mode=False, use_purged_cv=True, n_splits=None):
    """Internal CV for ManualOOFStacker OOF meta-features (Task 049).

    ``use_purged_cv=True`` (default, production) returns ``PastOnlyGroupCV``
    — every fold satisfies ``max(train_time) < min(validation_time)``.
    ``ChronologicalPartitionCV`` (leaky: later folds train on future rows,
    see ``LEAK_REGISTRY.md::stack_cv_future_leakage``) is no longer used by
    any production path; it is retained in ``pipeline/cv.py`` only as the
    subject of its own regression test.
    """
    if n_splits is None:
        n_splits = 3 if fast_mode else 5
    if use_purged_cv:
        return PastOnlyGroupCV(n_splits=n_splits)
    return KFold(n_splits=n_splits, shuffle=True, random_state=42)


def _stack_fit_groups(X_df):
    """Chronological group labels for purged stack CV (game dates)."""
    if X_df is not None and "game_date" in X_df.columns:
        return X_df["game_date"].values
    return None


def _outer_game_cv_splits(df, n_splits, embargo=None):
    """Chronological outer CV grouped by game date (no cross-game leakage)."""
    embargo = TUNING_EMBARGO_GAMES if embargo is None else embargo
    groups = _stack_fit_groups(df)
    if groups is None and "GAME_ID" in df.columns:
        groups = df["GAME_ID"].values
    cv = PurgedGroupTimeSeriesSplit(n_splits=n_splits, embargo=embargo)
    yield from cv.split(df, groups=groups)


# ------------------------------------------------------------
# MetaScoreModel – Parallel Stacking with Huber meta‑learner
# ------------------------------------------------------------
class MetaScoreModel:
    def __init__(self, ridge_alpha=10.0, cb_params=None, huber_epsilon=1.0,
                 use_isotonic_calibration=False, cv_splitter=None,
                 margin_cap=30.0, total_mode="model", league_avg_total=225.0,
                 feature_cols=None, use_elo_stack=True,
                 prediction_mode="absolute", residual_alpha=0.5,
                 use_purged_cv=True, segment=None, ou_min_edge=3.0,
                 elo_blend_alpha=0.35, elo_ridge_alpha=3.0,
                 dynamic_elo_blend=True, total_elo_beta=None,
                 elo_win_blend=None, train_target="close_residual",
                 use_quantile_heads=True, quantile_alphas=(0.1, 0.9),
                 use_lightgbm_base=False):
        """
        Parameters
        ----------
        ridge_alpha : float
            Regularisation strength for Ridge base estimator.
        cb_params : dict or None
            Parameters for CatBoostRegressor (depth, iterations, etc.).
        huber_epsilon : float
            Epsilon parameter for the HuberRegressor meta‑learner.
            Controls the transition point from quadratic to linear loss.
        use_isotonic_calibration : bool
            If True, use IsotonicRegression for win‑prob calibration;
            otherwise, use LogisticRegression (Platt scaling).
        margin_cap : float or None
            Symmetric cap applied to the training margin target. None disables
            capping (rely on the Huber meta-learner for robustness). Widened
            from the legacy 20.0 so big favourites aren't shrunk toward the mean.
        total_mode : str
            "model" trains a second head to predict game total (pace/scoring
            aware); "fixed" reverts to the legacy constant ``league_avg_total``;
            "external" skips the embedded total head (use ``MetaTotalModel``).
        league_avg_total : float
            Fallback total used when total_mode == "fixed" or the total head is
            unavailable.
        feature_cols : list or None
            Feature columns the model consumes. Defaults to SAFE_FEATURE_COLS;
            the ablation harness passes subsets here.
        """
        self.ridge_alpha = ridge_alpha
        self.huber_epsilon = huber_epsilon
        self.use_isotonic = use_isotonic_calibration
        self.margin_cap = margin_cap
        self.total_mode = total_mode
        self.league_avg_total = league_avg_total
        self.use_elo_stack = use_elo_stack
        self.prediction_mode = prediction_mode  # absolute | residual | blend
        self.residual_alpha = residual_alpha
        self.train_target = train_target  # close_residual | absolute
        self.use_quantile_heads = use_quantile_heads
        self.quantile_alphas = tuple(quantile_alphas)
        self.use_lightgbm_base = use_lightgbm_base
        self.segment = segment
        self.ou_min_edge = ou_min_edge
        self.elo_blend_alpha = elo_blend_alpha
        self.elo_ridge_alpha = elo_ridge_alpha
        self.dynamic_elo_blend = dynamic_elo_blend
        self.total_elo_beta = TOTAL_ELO_BETA if total_elo_beta is None else total_elo_beta
        self.quantile_models = {}
        self.elo_ridge = None
        self.elo_scaler = None
        self._elo_cols = []

        # Default CatBoost params (safe, low depth to avoid overfitting)
        default_cb = {
            'depth': 4,
            'iterations': 300,
            'learning_rate': 0.05,
            'l2_leaf_reg': 3.0,
            'loss_function': 'Huber:delta=1.5',
            'verbose': 0,
            'random_seed': 42,
            'thread_count': 1,
        }
        if cb_params is not None:
            default_cb.update(cb_params)
        self.cb_params = default_cb

        # Base estimators
        base_models = [
            ('ridge', Ridge(alpha=self.ridge_alpha)),
            ('catboost', CatBoostRegressor(**self.cb_params))
        ]
        if self.use_lightgbm_base:
            try:
                from lightgbm import LGBMRegressor
                base_models.append((
                    'lightgbm',
                    LGBMRegressor(
                        n_estimators=200, max_depth=4, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1,
                    ),
                ))
            except ImportError:
                pass

        # Final meta‑learner with Huber loss
        final_estimator = HuberRegressor(epsilon=self.huber_epsilon)

        # Manual past-only OOF stacking (Task 049/050/051). sklearn's
        # StackingRegressor requires its internal CV to be a *partition*
        # (every sample in exactly one held-out fold), which a strictly
        # past-only splitter cannot satisfy for its earliest block — so we
        # never hand a past-only splitter to StackingRegressor. Instead
        # ManualOOFStacker generates OOF base predictions manually, dropping
        # burn-in rows with no valid past-only fold from meta-training
        # (Task 051) and stamping fit_max_timestamp on every fitted
        # component (Task 050/058).
        self.use_purged_cv = use_purged_cv
        if cv_splitter is None:
            cv_splitter = _stack_cv_splitter(use_purged_cv=use_purged_cv)
        self.stack = ManualOOFStacker(
            estimators=base_models,
            final_estimator=final_estimator,
            cv=cv_splitter,
        )

        # Second head for game TOTAL (separate estimator instances so the two
        # stacks never share fitted state).
        total_base = [
            ('ridge', Ridge(alpha=self.ridge_alpha)),
            ('catboost', CatBoostRegressor(**self.cb_params)),
        ]
        self.total_stack = ManualOOFStacker(
            estimators=total_base,
            final_estimator=HuberRegressor(epsilon=self.huber_epsilon),
            cv=_stack_cv_splitter(use_purged_cv=use_purged_cv),
        )
        self._total_head_ok = False

        # In tune_meta_model (when building the stack for tuning), do the same.

        self.scaler = StandardScaler()
        self.calibrator = None
        self.fitted = False
        base_feats = list(feature_cols) if feature_cols else SAFE_FEATURE_COLS.copy()
        if use_elo_stack and "elo_stack_pred" not in base_feats:
            base_feats.append("elo_stack_pred")
        self.features = base_feats

    def _elo_matrix(self, X_df):
        cols = [c for c in ELO_STACK_FEATURES if c in X_df.columns]
        self._elo_cols = cols
        if not cols:
            return np.zeros((len(X_df), 1))
        return X_df[cols].fillna(0).to_numpy(dtype=float)

    def _augment_elo_stack(self, X_df):
        """Add elo_stack_pred column from the Elo-only ridge head."""
        out = X_df.copy()
        if self.use_elo_stack and self.elo_ridge is not None and self._elo_cols:
            Xe = self._elo_matrix(out)
            out["elo_stack_pred"] = self.elo_ridge.predict(self.elo_scaler.transform(Xe))
        else:
            out["elo_stack_pred"] = 0.0
        return out

    def _get_features(self, X_df):
        """Extract and clean the safe feature set."""
        missing = [c for c in self.features if c not in X_df.columns]
        if missing:
            X_df = X_df.copy()
            for col in missing:
                X_df[col] = 0.0
        return X_df[self.features].fillna(0)

    def _effective_train_target(self, X_df) -> str:
        if self.train_target != "close_residual":
            return "absolute"
        close = closing_spread_series(X_df)
        if close is None or valid_closing_fraction(X_df) < 0.5:
            return "absolute"
        return "close_residual"

    def _build_stack_target(self, y_margin, X_df):
        mode = self._effective_train_target(X_df)
        if mode == "close_residual":
            close = closing_spread_series(X_df)
            return margin_close_residual(y_margin, close.values), mode
        return np.asarray(y_margin, dtype=float), mode

    def _stack_output_to_margin(self, stack_pred, X_df, mode=None):
        if mode is None:
            mode = self._effective_train_target(X_df)
        pred = np.asarray(stack_pred, dtype=float)
        if mode != "close_residual":
            return pred
        close = closing_spread_series(X_df)
        if close is None:
            return pred
        return residual_to_margin(pred, close.values)

    def _closing_from_feat(self, feat_dict):
        for key in ("closing_spread", "market_spread"):
            val = feat_dict.get(key)
            if val is not None and not pd.isna(val):
                return float(val)
        return np.nan

    def _fit_quantile_heads(self, X_scaled, y_target):
        self.quantile_models = {}
        if not self.use_quantile_heads:
            return
        for alpha in self.quantile_alphas:
            key = f"q{int(alpha * 100)}"
            qparams = dict(self.cb_params)
            qparams.update({
                "loss_function": f"Quantile:alpha={alpha}",
                "iterations": min(int(qparams.get("iterations", 300)), 250),
                "depth": min(int(qparams.get("depth", 4)), 4),
                "allow_writing_files": False,
            })
            model = CatBoostRegressor(**qparams)
            try:
                model.fit(X_scaled, y_target)
                self.quantile_models[key] = model
            except Exception as e:  # noqa: BLE001
                print(f"⚠️ quantile head {key} fit failed ({e})")

    def fit(self, X_df, y_home, y_away, calib_df=None):
        """
        Fit the stacking ensemble on the base set, then optionally calibrate
        win probabilities on a held‑out calibration set.
        """
        y_margin = np.asarray(y_home - y_away, dtype=float)
        y_stack, train_mode = self._build_stack_target(y_margin, X_df)
        self._active_train_mode = train_mode
        if self.margin_cap is not None:
            y_margin_target = np.clip(y_stack, -self.margin_cap, self.margin_cap)
        else:
            y_margin_target = y_stack

        X_work = X_df.copy()
        stack_groups = _stack_fit_groups(X_df)
        elo_groups = stack_groups if stack_groups is not None else np.arange(len(X_df))
        if self.use_elo_stack:
            Xe = self._elo_matrix(X_work)
            if Xe.shape[1] > 0:
                # Cross-fit elo_stack_pred past-only (Task 052): the Elo-only
                # ridge head is itself target-fitted, so feeding an in-sample
                # (fit-on-everything) prediction into the margin stack's
                # training rows would leak the target into a feature. Each
                # training row's elo_stack_pred instead comes from a ridge
                # fit only on strictly earlier rows; the ridge refit on all
                # rows (self.elo_scaler/self.elo_ridge) is used only at
                # predict() time, when every row being scored is later than
                # every row it was fit on.
                def _elo_fit(Xtr, ytr):
                    sc = StandardScaler()
                    Xs = sc.fit_transform(Xtr)
                    m = Ridge(alpha=self.elo_ridge_alpha)
                    m.fit(Xs, ytr)
                    return _ScaledEstimator(sc, m)

                def _elo_predict(artifact, Xap):
                    return artifact.predict(Xap)

                elo_oof, elo_covered, elo_artifact_final = cross_fit_derived_feature(
                    _elo_fit, _elo_predict, Xe, y_margin_target, elo_groups,
                    cv=_stack_cv_splitter(use_purged_cv=self.use_purged_cv),
                )
                self.elo_scaler, self.elo_ridge = elo_artifact_final.scaler, elo_artifact_final.model
                self.elo_ridge.fit_max_timestamp = elo_artifact_final.fit_max_timestamp
                self.elo_scaler.fit_max_timestamp = elo_artifact_final.fit_max_timestamp
                self._elo_stack_covered = elo_covered
                X_work = X_work.copy()
                X_work["elo_stack_pred"] = np.where(elo_covered, elo_oof, 0.0)
            else:
                X_work["elo_stack_pred"] = 0.0
                self._elo_stack_covered = np.ones(len(X_work), dtype=bool)
        else:
            self._elo_stack_covered = np.ones(len(X_work), dtype=bool)

        X = self._get_features(X_work)
        X_scaled = self.scaler.fit_transform(X)
        self.stack.fit(X_scaled, y_margin_target, groups=stack_groups)

        self._fit_quantile_heads(X_scaled, y_margin_target)

        # Fit the TOTAL head (pace/scoring aware) on the same scaled features.
        if self.total_mode == "model":
            y_total = np.asarray(y_home + y_away, dtype=float)
            try:
                self.total_stack.fit(X_scaled, y_total, groups=stack_groups)
                self._total_head_ok = True
            except Exception as e:  # noqa: BLE001
                print(f"⚠️ total head fit failed ({e}); falling back to fixed total.")
                self._total_head_ok = False

        # --- Calibration (if calibration set provided) ---
        if calib_df is not None:
            raw_preds = self._predict_raw(calib_df)['pred_margin']
            raw_preds = raw_preds.reshape(-1, 1)
            y_win_cal = (calib_df['actual_home'] > calib_df['actual_away']).astype(int)

            if self.use_isotonic:
                self.calibrator = IsotonicRegression(out_of_bounds='clip')
                self.calibrator.fit(raw_preds.ravel(), y_win_cal)
                print(f"✅ Isotonic calibration fitted on {len(raw_preds)} samples.")
            else:
                # Platt scaling (logistic) – use strong L2 to avoid overfitting
                self.calibrator = LogisticRegression(C=0.1, solver='lbfgs', max_iter=100)
                self.calibrator.fit(raw_preds, y_win_cal)
                print(f"✅ Platt calibration (logistic) fitted on {len(raw_preds)} samples.")

        self.fitted = True
        print(f"✅ Stacked ensemble fitted with {len(self.stack.estimators_)} base models.")

    def _predict_raw(self, X_df):
        """Return raw predicted margin (before calibration)."""
        X_work = self._augment_elo_stack(X_df)
        X = self._get_features(X_work)
        X_scaled = self.scaler.transform(X)
        stack_out = self.stack.predict(X_scaled)
        mode = getattr(self, "_active_train_mode", self._effective_train_target(X_df))
        margin = self._stack_output_to_margin(stack_out, X_df, mode=mode)
        out = {"pred_margin": margin, "pred_stack": stack_out}
        if self.quantile_models:
            q_margins = {}
            for key, qm in self.quantile_models.items():
                q_stack = qm.predict(X_scaled)
                q_margins[key] = self._stack_output_to_margin(q_stack, X_df, mode=mode)
            if "q10" in q_margins and "q90" in q_margins:
                out["spread_q10"] = q_margins["q10"]
                out["spread_q90"] = q_margins["q90"]
                out["spread_quantile_width"] = q_margins["q90"] - q_margins["q10"]
        return out

    def _apply_prediction_mode(self, raw_margin, feat_dict):
        """Blend absolute prediction with market-residual mode (predict-time only)."""
        if getattr(self, "_active_train_mode", "absolute") == "close_residual":
            return raw_margin
        market_spread = feat_dict.get("market_spread") or feat_dict.get("closing_spread")
        if self.prediction_mode == "absolute" or market_spread is None or pd.isna(market_spread):
            return raw_margin
        market_prior = -float(market_spread)
        if self.prediction_mode == "residual":
            return market_prior + self.residual_alpha * (raw_margin - market_prior)
        # blend
        return (1 - self.residual_alpha) * market_prior + self.residual_alpha * raw_margin

    def _dynamic_elo_blend_alpha(self, row: dict) -> float:
        """Increase Elo anchor weight when lineup uncertainty is high."""
        base = float(self.elo_blend_alpha)
        if not self.dynamic_elo_blend:
            return base
        unc = float(row.get("uncertainty_diff", 0) or 0)
        boost = ELO_UNCERTAINTY_BLEND_BOOST * max(0.0, (unc - 50.0) / 300.0)
        return float(np.clip(base + boost, 0.0, 0.65))

    def _elo_implied_total(self, row: dict) -> float | None:
        return elo_implied_total_from_row(row)

    def predict(self, feat_dict):
        """
        Predict for a single game given a feature dictionary.
        Returns: dict with pred_home, pred_away, pred_margin, pred_total, win_prob.
        """
        if not self.fitted:
            raise RuntimeError("Model must be fitted before predicting.")

        row = dict(feat_dict)
        X_work = self._augment_elo_stack(pd.DataFrame([row]))
        X_scaled = self.scaler.transform(self._get_features(X_work))
        raw_out = self._predict_raw(pd.DataFrame([row]))
        raw_margin = float(raw_out["pred_margin"][0] if hasattr(raw_out["pred_margin"], "__len__") else raw_out["pred_margin"])
        raw_margin = self._apply_prediction_mode(raw_margin, row)

        spread_q10 = spread_q90 = spread_q_width = None
        if "spread_q10" in raw_out:
            spread_q10 = float(raw_out["spread_q10"][0])
            spread_q90 = float(raw_out["spread_q90"][0])
            spread_q_width = float(raw_out["spread_quantile_width"][0])

        # Blend meta prediction toward calibrated ELO anchor (dynamic weight)
        elo_anchor = row.get("elo_margin_calibrated") or row.get("elo_stack_pred") or row.get("elo_margin")
        blend_a = self._dynamic_elo_blend_alpha(row)
        if elo_anchor is not None and not pd.isna(elo_anchor) and blend_a > 0:
            raw_margin = (1.0 - blend_a) * raw_margin + blend_a * float(elo_anchor)

        # Calibrate win probability
        if self.calibrator is not None:
            if self.use_isotonic:
                win_prob = self.calibrator.predict([raw_margin])[0]
            else:
                win_prob = self.calibrator.predict_proba(np.array([[raw_margin]]))[0, 1]
        else:
            # Fallback sigmoid (no calibration)
            win_prob = 1.0 / (1.0 + np.exp(-raw_margin / 12.0))
            win_prob = np.clip(win_prob, 0.01, 0.99)

        # Predicted total: learned head when available, else legacy fixed total.
        if self.total_mode == "model" and self._total_head_ok and getattr(self, "total_stack", None) is not None:
            try:
                pred_total = float(self.total_stack.predict(X_scaled)[0])
            except Exception:
                pred_total = self.league_avg_total
        else:
            pred_total = self.league_avg_total
        elo_total = self._elo_implied_total(row)
        if elo_total is not None and self.total_elo_beta > 0:
            pred_total = (1.0 - self.total_elo_beta) * pred_total + self.total_elo_beta * elo_total
        pred_home = (pred_total + raw_margin) / 2.0
        pred_away = (pred_total - raw_margin) / 2.0

        return {
            'pred_home': float(pred_home),
            'pred_away': float(pred_away),
            'pred_margin': float(raw_margin),
            'pred_total': float(pred_home + pred_away),
            'win_prob': float(win_prob),
            'ou_direction': self._ou_signal(feat_dict, pred_total),
            'spread_q10': spread_q10,
            'spread_q90': spread_q90,
            'spread_quantile_width': spread_q_width,
        }

    def _ou_signal(self, feat_dict, pred_total):
        """Over/Under bet direction when edge exceeds threshold."""
        mkt = feat_dict.get("market_total")
        if mkt is None or pd.isna(mkt) or mkt == 0:
            return "Pass"
        diff = pred_total - float(mkt)
        if abs(diff) >= self.ou_min_edge:
            return "Over" if diff > 0 else "Under"
        return "Pass"

    def calibrate_prob(self, raw_margin):
        """Calibrated win probability from a raw margin using the trained
        calibrator. Used as the cold-start for the in-season rolling calibrator
        so a single coherent calibration path is used throughout."""
        if self.calibrator is not None:
            try:
                if self.use_isotonic:
                    p = self.calibrator.predict([raw_margin])[0]
                else:
                    p = self.calibrator.predict_proba(np.array([[raw_margin]]))[0, 1]
                return float(np.clip(p, 0.01, 0.99))
            except Exception:
                pass
        return float(np.clip(1.0 / (1.0 + np.exp(-raw_margin / 12.0)), 0.01, 0.99))

    def save(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path):
        import pickle
        with open(path, "rb") as f:
            return pickle.load(f)


# ------------------------------------------------------------
# Hyperparameter Tuning for MetaScoreModel
# ------------------------------------------------------------
def _fit_elo_stack_cv(X_tr, y_tr, elo_cols, alpha=5.0):
    """Fit elo-only ridge on a train fold; returns (scaler, ridge, avail_cols)."""
    avail = [c for c in elo_cols if c in X_tr.columns]
    if not avail:
        return None, None, avail
    Xe = X_tr[avail].fillna(0).to_numpy(dtype=float)
    y_arr = np.asarray(y_tr, dtype=float)
    if len(y_arr) != len(Xe):
        raise ValueError(
            f"ELO stack CV: X/y length mismatch ({len(Xe)} vs {len(y_arr)})"
        )
    scaler_e = StandardScaler()
    Xe_s = scaler_e.fit_transform(Xe)
    elo_ridge = Ridge(alpha=alpha)
    elo_ridge.fit(Xe_s, y_arr)
    return scaler_e, elo_ridge, avail


def _apply_elo_stack_cv(X_df, scaler_e, elo_ridge, avail):
    """Apply a train-fitted elo ridge head (no refit on val/test)."""
    X_work = X_df.copy()
    if not avail or scaler_e is None or elo_ridge is None:
        X_work["elo_stack_pred"] = 0.0
        return X_work
    Xe = X_work[avail].fillna(0).to_numpy(dtype=float)
    Xe_s = scaler_e.transform(Xe)
    X_work["elo_stack_pred"] = elo_ridge.predict(Xe_s)
    return X_work


def _augment_elo_stack_cv(X_df, y_margin_target, elo_cols, alpha=5.0):
    """Fit elo ridge on X_df and add elo_stack_pred (single-split helper)."""
    scaler_e, elo_ridge, avail = _fit_elo_stack_cv(
        X_df, y_margin_target, elo_cols, alpha=alpha,
    )
    return _apply_elo_stack_cv(X_df, scaler_e, elo_ridge, avail)


def _margin_tuning_objective(
    df, X_base, trial, b, EMBARGO, use_elo_stack=True, fast_mode=False,
    train_target="close_residual",
):
    """Shared margin-model Optuna objective — margin/probability-first
    (Task 055: no ROI/edge-grid/CLV terms; overfit penalty penalizes
    val_mae exceeding train_mae)."""
    close_all = closing_spread_series(df)
    use_close_residual = (
        train_target == "close_residual"
        and close_all is not None
        and valid_closing_fraction(df) >= 0.5
    )

    cb_iter_lo, cb_iter_hi = (200, 400) if fast_mode else (300, 500)
    outer_splits = 2 if fast_mode else 3
    inner_splits = 3 if fast_mode else 5
    stack_jobs = 1 if fast_mode else -1

    if b is not None:
        lo, hi = b.suggest_bounds_float('ridge_alpha', 0.01, 200.0, log=True)
        ridge_alpha = trial.suggest_float('ridge_alpha', lo, hi, log=True)
        lo, hi = b.suggest_bounds_int('cb_depth', 2, 7)
        cb_depth = trial.suggest_int('cb_depth', lo, hi)
        lo, hi = b.suggest_bounds_int('cb_iter', cb_iter_lo, cb_iter_hi)
        lo = max(cb_iter_lo, (lo // 100) * 100)
        hi = min(cb_iter_hi, max(lo + 100, ((hi + 99) // 100) * 100))
        if hi <= lo:
            lo, hi = cb_iter_lo, min(cb_iter_lo + 100, cb_iter_hi)
        cb_iter = trial.suggest_int('cb_iter', lo, hi, step=100)
        lo, hi = b.suggest_bounds_float('cb_lr', 0.01, 0.1, log=True)
        cb_lr = trial.suggest_float('cb_lr', lo, hi, log=True)
        lo, hi = b.suggest_bounds_float('cb_l2', 1.0, 10.0, log=True)
        cb_l2 = trial.suggest_float('cb_l2', lo, hi, log=True)
        lo, hi = b.suggest_bounds_float('huber_epsilon', 1.01, 4.0)
        huber_epsilon = trial.suggest_float('huber_epsilon', lo, hi)
        lo, hi = b.suggest_bounds_float('elo_blend_alpha', 0.0, 0.65)
        elo_blend_alpha = trial.suggest_float('elo_blend_alpha', lo, hi)
        lo, hi = b.suggest_bounds_float('elo_ridge_alpha', 1.0, 12.0, log=True)
        elo_ridge_alpha = trial.suggest_float('elo_ridge_alpha', lo, hi, log=True)
    else:
        ridge_alpha = trial.suggest_float('ridge_alpha', 0.1, 100.0, log=True)
        cb_depth = trial.suggest_int('cb_depth', 2, 7)
        cb_iter = trial.suggest_int('cb_iter', cb_iter_lo, cb_iter_hi, step=100)
        cb_lr = trial.suggest_float('cb_lr', 0.01, 0.1, log=True)
        cb_l2 = trial.suggest_float('cb_l2', 1.0, 10.0, log=True)
        huber_epsilon = trial.suggest_float('huber_epsilon', 1.01, 4.0)
        elo_blend_alpha = trial.suggest_float('elo_blend_alpha', 0.0, 0.65)
        elo_ridge_alpha = trial.suggest_float('elo_ridge_alpha', 1.0, 12.0, log=True)

    margin_cap_choice = trial.suggest_categorical('margin_cap', MARGIN_CAP_CHOICES)

    base_models = [
        ('ridge', Ridge(alpha=ridge_alpha)),
        ('catboost', CatBoostRegressor(
            depth=cb_depth, iterations=cb_iter, learning_rate=cb_lr,
            l2_leaf_reg=cb_l2, loss_function='Huber:delta=1.5',
            verbose=0, random_seed=42, thread_count=1,
            allow_writing_files=False,
        ))
    ]
    stack = ManualOOFStacker(
        estimators=base_models,
        final_estimator=HuberRegressor(epsilon=huber_epsilon),
        cv=_stack_cv_splitter(fast_mode=fast_mode, n_splits=inner_splits),
    )

    y_margin = df['actual_margin']
    y_home = df['actual_home']
    y_away = df['actual_away']
    scores = []
    fold_idx = 0
    outer_splits_list = list(_outer_game_cv_splits(df, outer_splits, EMBARGO))
    n_outer = len(outer_splits_list)

    for train_idx, val_idx in outer_splits_list:
        if len(val_idx) < 20:
            continue

        ym_tr = y_margin.iloc[train_idx]
        ym_val = y_margin.iloc[val_idx]
        if use_close_residual:
            ym_tr_fit = margin_close_residual(ym_tr.values, close_all.iloc[train_idx].values)
            if margin_cap_choice is not None:
                ym_tr_fit = np.clip(ym_tr_fit, -margin_cap_choice, margin_cap_choice)
        elif margin_cap_choice is not None:
            ym_tr_fit = ym_tr.clip(-margin_cap_choice, margin_cap_choice)
        else:
            ym_tr_fit = ym_tr

        X_tr = X_base.iloc[train_idx].copy()
        X_val = X_base.iloc[val_idx].copy()
        if use_elo_stack:
            elo_sc, elo_ridge, elo_avail = _fit_elo_stack_cv(
                X_tr, np.asarray(ym_tr_fit, dtype=float), ELO_STACK_FEATURES, alpha=elo_ridge_alpha,
            )
            X_tr = _apply_elo_stack_cv(X_tr, elo_sc, elo_ridge, elo_avail)
            X_val = _apply_elo_stack_cv(X_val, elo_sc, elo_ridge, elo_avail)

        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)
        print(
            f"    trial {trial.number + 1}: outer fold {fold_idx + 1}/{n_outer} "
            f"(train={len(train_idx)}, val={len(val_idx)}, cb_iter={cb_iter})...",
            flush=True,
        )
        groups_tr = _stack_fit_groups(df.iloc[train_idx])
        stack.fit(X_tr_scaled, ym_tr_fit, groups=groups_tr)

        val_stack = stack.predict(X_val_scaled)
        if use_close_residual:
            val_preds = residual_to_margin(val_stack, close_all.iloc[val_idx].values)
        else:
            val_preds = val_stack
        if APPLY_SPREAD_CALIB_IN_CV:
            sc = SpreadCalibrator(window=max(50, len(train_idx)), min_samples=min(30, len(train_idx) // 3))
            train_raw = stack.predict(X_tr_scaled)
            if use_close_residual:
                train_raw = residual_to_margin(train_raw, close_all.iloc[train_idx].values)
            for p, a in zip(train_raw, ym_tr.values):
                if np.isfinite(p) and np.isfinite(a):
                    sc.update(float(p), float(a))
            val_preds = np.array([sc.correct(float(p)) for p in val_preds], dtype=float)
        anchor = None
        for col in ("elo_margin_calibrated", "elo_stack_pred", "elo_margin"):
            if col in df.columns:
                anchor = df[col].iloc[val_idx].values.astype(float)
                break
            if col in X_val.columns:
                anchor = X_val[col].values.astype(float)
                break
        if anchor is not None and elo_blend_alpha > 0:
            mask = np.isfinite(anchor)
            val_preds = val_preds.copy()
            val_preds[mask] = (
                (1.0 - elo_blend_alpha) * val_preds[mask]
                + elo_blend_alpha * anchor[mask]
            )
        val_probs = np.clip(1.0 / (1.0 + np.exp(-val_preds / 12.0)), 0.01, 0.99)
        val_mae = mean_absolute_error(ym_val, val_preds)
        brier = brier_score_loss((ym_val > 0).astype(int), val_probs)
        val_ece = compute_ece((ym_val > 0).astype(int), val_probs)
        train_stack = stack.predict(X_tr_scaled)
        if use_close_residual:
            train_preds = residual_to_margin(train_stack, close_all.iloc[train_idx].values)
        else:
            train_preds = train_stack
        train_mae = mean_absolute_error(ym_tr, train_preds)
        # Task 055: penalize validation error *exceeding* training error
        # (overfitting), not the reverse. The prior sign — max(0, train_mae
        # - val_mae) — rewarded configurations where validation error was
        # *lower* than training error and never penalized true overfitting
        # (val_mae >> train_mae), which is backwards.
        overfit_penalty = max(0.0, (val_mae - train_mae) - 1.0) * 2.0

        # Task 055: margin-model tuning must be a proper-scoring-rule
        # objective on margin/probability quality only. ATS ROI, the
        # best-of-edge-grid search, and the CLV term are removed — they
        # (a) require betting columns to be present (market_spread), which
        # the pass condition explicitly forbids depending on, and (b) let
        # the tuner chase a noisy, threshold-selected backtest ROI number
        # instead of the agreed margin-first objective (Phase 4A of the
        # plan). Betting-policy tuning is a separate, later step (Task 056)
        # on policy-tuning-role rows only, never mixed into this loss.
        combined = (
            META_LOSS_MAE_WEIGHT * val_mae
            + META_LOSS_BRIER_WEIGHT * brier
            + META_LOSS_ECE_WEIGHT * (val_ece if np.isfinite(val_ece) else 0.0)
            + overfit_penalty
        )
        scores.append(combined)
        trial.report(combined, step=fold_idx)
        fold_idx += 1
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores) if scores else TUNING_INVALID_SCORE


def meta_params_for_bounds(best_params: dict) -> dict:
    """Flatten cached MetaScoreModel params for AdaptiveParameterBounds."""
    out = dict(best_params)
    cb = out.pop("cb_params", None) or {}
    if cb:
        for src, dst in (
            ("depth", "cb_depth"),
            ("iterations", "cb_iter"),
            ("learning_rate", "cb_lr"),
            ("l2_leaf_reg", "cb_l2"),
        ):
            if out.get(dst) is None:
                val = cb.get(src)
                if val is not None:
                    out[dst] = val
    return AdaptiveParameterBounds._sanitize_params(out)


def tune_margin_model(train_features_df, n_trials=25, season=None,
                      bounds: 'AdaptiveParameterBounds' = None,
                      feature_cols=None, use_elo_stack=True,
                      fast_mode=None, train_target="close_residual"):
    """Tune margin stack with ROI-first objective (matches production fit).

    fast_mode: lighter CV + fewer CatBoost trees during search. Auto-enabled
    when n_trials <= 5 (smoke tests). Full backtests should use fast_mode=False.
    """
    df = train_features_df.dropna(subset=['actual_margin']).reset_index(drop=True)
    cols = list(feature_cols) if feature_cols else SAFE_FEATURE_COLS
    cols = [c for c in cols if c in df.columns and c != 'elo_stack_pred']
    X_base = df[cols].fillna(0)
    EMBARGO = TUNING_EMBARGO_GAMES
    b = bounds
    if fast_mode is None:
        fast_mode = n_trials <= 5

    if n_trials <= 0:
        print("⏭️ Skipping margin tuning (n_trials=0) — using default hyperparameters.")
        return {
            'ridge_alpha': 7.72,
            'huber_epsilon': 1.42,
            'margin_cap': 35.0,
            'elo_blend_alpha': ELO_BLEND_ALPHA,
            'elo_ridge_alpha': ELO_RIDGE_ALPHA,
            'cb_params': {
                'depth': 2,
                'iterations': 500,
                'learning_rate': 0.014,
                'l2_leaf_reg': 7.16,
                'thread_count': 1,
            },
        }

    def objective(trial):
        return _margin_tuning_objective(
            df, X_base, trial, b, EMBARGO,
            use_elo_stack=use_elo_stack, fast_mode=fast_mode,
            train_target=train_target,
        )

    mode_label = "FAST" if fast_mode else "FULL"
    outer = 2 if fast_mode else 3
    inner = 3 if fast_mode else 5
    print(
        f"⏳ Tuning margin model ({mode_label}: {n_trials} trials, "
        f"{outer}×{inner} CV, ~{n_trials * outer} stack fits)...",
        flush=True,
    )

    def _trial_callback(study, trial):
        if trial.value is not None:
            print(
                f"  ✓ trial {trial.number + 1}/{n_trials} score={trial.value:.4f}",
                flush=True,
            )

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42, multivariate=True),
        pruner=MedianPruner(
            n_startup_trials=min(5, max(1, n_trials)),
            n_warmup_steps=1,
        ),
    )
    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True,
        callbacks=[_trial_callback],
    )
    print(f"✅ Margin tuning complete. Best score: {study.best_value:.4f}")
    best = study.best_params
    if b is not None and season is not None:
        b.record_best(season, meta_params_for_bounds({
            'ridge_alpha': best['ridge_alpha'],
            'huber_epsilon': best['huber_epsilon'],
            'margin_cap': best.get('margin_cap'),
            'elo_blend_alpha': best.get('elo_blend_alpha'),
            'elo_ridge_alpha': best.get('elo_ridge_alpha'),
            'cb_params': {
                'depth': best['cb_depth'],
                'iterations': best['cb_iter'],
                'learning_rate': best['cb_lr'],
                'l2_leaf_reg': best['cb_l2'],
            },
        }))
        b.report()
    return {
        'ridge_alpha': best['ridge_alpha'],
        'huber_epsilon': best['huber_epsilon'],
        'margin_cap': best.get('margin_cap', 30.0),
        'elo_blend_alpha': best.get('elo_blend_alpha', 0.35),
        'elo_ridge_alpha': best.get('elo_ridge_alpha', 3.0),
        'cb_params': {
            'depth': best['cb_depth'],
            'iterations': best['cb_iter'],
            'learning_rate': best['cb_lr'],
            'l2_leaf_reg': best['cb_l2'],
            'thread_count': 1,
        }
    }


def tune_total_model(train_features_df, n_trials=20, bounds=None, feature_cols=None,
                     use_elo_stack=True, fast_mode=None, season=None,
                     train_target: str | None = None):
    """Tune standalone total model with MAE + O/U ROI objective.

    fast_mode: lighter outer/inner CV and fewer CatBoost trees during search.
    Auto-enabled when n_trials <= 5 (smoke tests).
    """
    if fast_mode is None:
        fast_mode = n_trials <= 5
    if train_target is None:
        train_target = TOTAL_TRAIN_TARGET

    df = train_features_df.dropna(subset=['actual_total']).reset_index(drop=True)
    cols = list(feature_cols) if feature_cols else total_feature_cols(df)
    cols = [c for c in cols if c in df.columns and c != 'elo_stack_pred']
    X_base = df[cols].fillna(0)
    y_total = df['actual_total']
    mkt_all = market_total_series(df)
    use_market_residual = (
        train_target == "market_residual"
        and mkt_all is not None
        and valid_market_total_fraction(df) >= 0.5
    )
    EMBARGO = TUNING_EMBARGO_GAMES
    cb_iter_lo, cb_iter_hi = (200, 400) if fast_mode else (200, 800)
    outer_splits = 2 if fast_mode else 3
    inner_splits = 3 if fast_mode else 5
    stack_jobs = 1 if fast_mode else -1
    b = bounds

    def objective(trial):
        if b is not None:
            lo, hi = b.suggest_bounds_float('ridge_alpha', 0.1, 100.0, log=True)
            ridge_alpha = trial.suggest_float('ridge_alpha', lo, hi, log=True)
            lo, hi = b.suggest_bounds_int('cb_depth', 2, 6)
            cb_depth = trial.suggest_int('cb_depth', lo, hi)
            lo, hi = b.suggest_bounds_int('cb_iter', cb_iter_lo, cb_iter_hi)
            lo = max(cb_iter_lo, (lo // 100) * 100)
            hi = min(cb_iter_hi, max(lo + 100, ((hi + 99) // 100) * 100))
            if hi <= lo:
                lo, hi = cb_iter_lo, min(cb_iter_lo + 100, cb_iter_hi)
            cb_iter = trial.suggest_int('cb_iter', lo, hi, step=100)
        else:
            ridge_alpha = trial.suggest_float('ridge_alpha', 0.1, 100.0, log=True)
            cb_depth = trial.suggest_int('cb_depth', 2, 6)
            cb_iter = trial.suggest_int('cb_iter', cb_iter_lo, cb_iter_hi, step=100)
        total_elo_beta = trial.suggest_float('total_elo_beta', 0.0, 0.5)
        huber_eps = trial.suggest_float('huber_epsilon', 1.0, 2.0)

        fold_scores = []
        fold_idx = 0
        outer_splits_list = list(_outer_game_cv_splits(df, outer_splits, EMBARGO))
        n_outer = len(outer_splits_list)
        for train_idx, val_idx in outer_splits_list:
            if len(val_idx) < 20:
                continue
            X_tr = X_base.iloc[train_idx].copy()
            X_val = X_base.iloc[val_idx].copy()
            y_tr = y_total.iloc[train_idx]
            y_val = y_total.iloc[val_idx]
            if use_market_residual:
                mkt_tr = mkt_all.iloc[train_idx].values
                mkt_val = mkt_all.iloc[val_idx].values
                y_tr_fit = total_market_residual(y_tr.values, mkt_tr)
                target_mode = "market_residual"
            else:
                y_tr_fit = y_tr.values
                mkt_val = mkt_all.iloc[val_idx].values if mkt_all is not None else None
                target_mode = "absolute"

            scaler = StandardScaler()
            X_tr_s = scaler.fit_transform(X_tr)
            X_val_s = scaler.transform(X_val)
            print(
                f"    total trial {trial.number + 1}: fold {fold_idx + 1}/{n_outer} "
                f"(train={len(train_idx)}, val={len(val_idx)}, cb_iter={cb_iter})...",
                flush=True,
            )
            groups_tr = _stack_fit_groups(df.iloc[train_idx])
            tot_stack = ManualOOFStacker(
                estimators=[
                    ('ridge', Ridge(alpha=ridge_alpha)),
                    ('catboost', CatBoostRegressor(
                        depth=cb_depth, iterations=cb_iter, learning_rate=0.05,
                        verbose=0, random_seed=42, thread_count=1,
                        allow_writing_files=False,
                    )),
                ],
                final_estimator=HuberRegressor(epsilon=huber_eps),
                cv=_stack_cv_splitter(fast_mode=fast_mode, n_splits=inner_splits),
            )
            tot_stack.fit(X_tr_s, y_tr_fit, groups=groups_tr)
            pred = tot_stack.predict(X_val_s)
            if target_mode == "market_residual" and mkt_val is not None:
                pred = residual_to_total(pred, mkt_val)
            if total_elo_beta > 0:
                val_rows = df.iloc[val_idx]
                elo_totals = np.array([
                    elo_implied_total_from_row(r.to_dict()) or np.nan
                    for _, r in val_rows.iterrows()
                ])
                mask = ~np.isnan(elo_totals)
                if mask.any():
                    pred = pred.copy()
                    pred[mask] = (
                        (1.0 - total_elo_beta) * pred[mask]
                        + total_elo_beta * elo_totals[mask]
                    )
            val_mae = mean_absolute_error(y_val, pred)

            ou_roi = 0.0
            if mkt_val is not None and np.isfinite(mkt_val).sum() >= 15:
                mkt_v = np.asarray(mkt_val, dtype=float)
                edge = pred - mkt_v
                best_r = -1.0
                for thr in (3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0):
                    active = np.abs(edge) >= thr
                    if active.sum() < 10:
                        continue
                    actual = y_val.values
                    over_win = (edge > 0) & (actual > mkt_v)
                    under_win = (edge < 0) & (actual < mkt_v)
                    win = over_win | under_win
                    wp = win[active].mean()
                    roi = wp * (100.0 / 110.0) - (1.0 - wp)
                    best_r = max(best_r, roi)
                ou_roi = best_r if best_r > -1 else 0.0

            combined = (
                TOTAL_LOSS_MAE_WEIGHT * val_mae
                - TOTAL_LOSS_OU_ROI_WEIGHT * ou_roi
            )
            fold_scores.append(combined)
            fold_idx += 1
        return np.mean(fold_scores) if fold_scores else TUNING_INVALID_SCORE

    mode_label = "FAST" if fast_mode else "FULL"
    target_label = "residual" if use_market_residual else "absolute"
    print(f"  ⏳ Total-model tuning ({mode_label}, {n_trials} trials, {target_label})...", flush=True)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    best = study.best_params
    if b is not None and season is not None:
        b.record_best(season, meta_params_for_bounds({
            'ridge_alpha': best['ridge_alpha'],
            'total_elo_beta': best.get('total_elo_beta', TOTAL_ELO_BETA),
            'huber_epsilon': best.get('huber_epsilon', 1.35),
            'cb_params': {
                'depth': best['cb_depth'],
                'iterations': best['cb_iter'],
            },
        }))
    return {
        'ridge_alpha': best['ridge_alpha'],
        'total_elo_beta': best.get('total_elo_beta', TOTAL_ELO_BETA),
        'huber_epsilon': best.get('huber_epsilon', 1.35),
        'train_target': "market_residual" if use_market_residual else "absolute",
        'feature_cols': cols,
        'cb_params': {
            'depth': best['cb_depth'],
            'iterations': best['cb_iter'],
            'learning_rate': 0.05,
            'thread_count': 1,
        },
        '_cv_score': float(study.best_value),
    }


class MetaTotalModel:
    """Standalone game-total regressor (decoupled from MetaScoreModel margin head)."""

    def __init__(
        self,
        ridge_alpha=10.0,
        cb_params=None,
        huber_epsilon=1.35,
        feature_cols=None,
        total_elo_beta=None,
        train_target="absolute",
        use_quantile_heads=False,
        league_avg_total=225.0,
    ):
        self.ridge_alpha = ridge_alpha
        self.huber_epsilon = huber_epsilon
        self.total_elo_beta = TOTAL_ELO_BETA if total_elo_beta is None else total_elo_beta
        self.train_target = train_target
        self.use_quantile_heads = use_quantile_heads
        self.league_avg_total = league_avg_total
        default_cb = {
            'depth': 4,
            'iterations': 300,
            'learning_rate': 0.05,
            'verbose': 0,
            'random_seed': 42,
            'thread_count': 1,
            'allow_writing_files': False,
        }
        if cb_params:
            default_cb.update(cb_params)
        self.cb_params = default_cb
        self.feature_cols = list(feature_cols) if feature_cols else None
        self.stack = None
        self.scaler = StandardScaler()
        self.quantile_models = {}
        self.fitted = False
        self._active_train_mode = "absolute"
        self._fit_cols_ = []

    def _resolve_cols(self, df: pd.DataFrame) -> list[str]:
        if self.feature_cols:
            return [c for c in self.feature_cols if c in df.columns]
        return total_feature_cols(df)

    def _effective_train_target(self, df: pd.DataFrame) -> str:
        if self.train_target != "market_residual":
            return "absolute"
        mkt = market_total_series(df)
        if mkt is None or valid_market_total_fraction(df) < 0.5:
            return "absolute"
        return "market_residual"

    def _build_target(self, df: pd.DataFrame):
        y_total = np.asarray(df["actual_total"], dtype=float)
        mode = self._effective_train_target(df)
        if mode == "market_residual":
            mkt = market_total_series(df)
            return total_market_residual(y_total, mkt.values), mode
        return y_total, mode

    def fit(self, X_df, y_home=None, y_away=None, calib_df=None):
        df = X_df.copy()
        if "actual_total" not in df.columns and y_home is not None and y_away is not None:
            df["actual_total"] = np.asarray(y_home, dtype=float) + np.asarray(y_away, dtype=float)
        cols = self._resolve_cols(df)
        self._fit_cols_ = cols
        y_target, mode = self._build_target(df)
        self._active_train_mode = mode

        X = df[cols].fillna(0)
        X_scaled = self.scaler.fit_transform(X)
        self.stack = ManualOOFStacker(
            estimators=[
                ('ridge', Ridge(alpha=self.ridge_alpha)),
                ('catboost', CatBoostRegressor(**self.cb_params)),
            ],
            final_estimator=HuberRegressor(epsilon=self.huber_epsilon),
            cv=_stack_cv_splitter(use_purged_cv=True),
        )
        groups = _stack_fit_groups(df)
        self.stack.fit(X_scaled, y_target, groups=groups)

        if self.use_quantile_heads:
            from sklearn.ensemble import GradientBoostingRegressor
            for alpha, key in ((0.1, "q10"), (0.9, "q90")):
                qm = GradientBoostingRegressor(loss="quantile", alpha=alpha, n_estimators=80, max_depth=3)
                qm.fit(X_scaled, y_target)
                self.quantile_models[key] = qm

        self.fitted = True
        return self

    def _predict_raw_total(self, feat_dict: dict) -> dict:
        row = pd.DataFrame([feat_dict])
        cols = self._fit_cols_ or self._resolve_cols(row)
        X = row[cols].fillna(0)
        for c in cols:
            if c not in X.columns:
                X[c] = 0.0
        X_scaled = self.scaler.transform(X[cols])
        raw = float(self.stack.predict(X_scaled)[0])
        mode = self._active_train_mode
        mkt = feat_dict.get("market_total")
        if mode == "market_residual" and mkt is not None and np.isfinite(float(mkt)):
            pred_total = float(residual_to_total(np.array([raw]), np.array([float(mkt)]))[0])
        else:
            pred_total = raw
        elo_total = elo_implied_total_from_row(feat_dict)
        if elo_total is not None and self.total_elo_beta > 0:
            pred_total = (1.0 - self.total_elo_beta) * pred_total + self.total_elo_beta * elo_total
        out = {"pred_total": float(pred_total), "pred_total_raw": float(raw)}
        if self.quantile_models:
            for key, qm in self.quantile_models.items():
                q_raw = float(qm.predict(X_scaled)[0])
                if mode == "market_residual" and mkt is not None and np.isfinite(float(mkt)):
                    q_val = float(residual_to_total(np.array([q_raw]), np.array([float(mkt)]))[0])
                else:
                    q_val = q_raw
                out[f"total_{key}"] = q_val
            if "total_q10" in out and "total_q90" in out:
                out["total_quantile_width"] = out["total_q90"] - out["total_q10"]
        return out

    def predict_total(self, feat_dict: dict) -> float:
        if not self.fitted:
            raise RuntimeError("MetaTotalModel must be fitted first.")
        return self._predict_raw_total(feat_dict)["pred_total"]

    def save(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path):
        import pickle
        with open(path, "rb") as f:
            return pickle.load(f)


class MetaScorePairModel:
    """Independent home/away score regressors → total, margin, O/U distribution inputs."""

    def __init__(
        self,
        ridge_alpha=10.0,
        cb_params=None,
        huber_epsilon=1.35,
        feature_cols=None,
        train_target="absolute",
        use_quantile_heads=False,
        sigma_floor=8.0,
    ):
        self.ridge_alpha = ridge_alpha
        self.huber_epsilon = huber_epsilon
        self.train_target = train_target
        self.use_quantile_heads = use_quantile_heads
        self.sigma_floor = sigma_floor
        default_cb = {
            "depth": 4,
            "iterations": 300,
            "learning_rate": 0.05,
            "verbose": 0,
            "random_seed": 42,
            "thread_count": 1,
            "allow_writing_files": False,
        }
        if cb_params:
            default_cb.update(cb_params)
        self.cb_params = default_cb
        self.feature_cols = list(feature_cols) if feature_cols else None
        self.home_stack = None
        self.away_stack = None
        self.scaler = StandardScaler()
        self.fitted = False
        self._fit_cols_ = []
        self._rmse_home = float(sigma_floor)
        self._rmse_away = float(sigma_floor)
        self._rmse_total = float(sigma_floor) * np.sqrt(2.0)

    def _resolve_cols(self, df: pd.DataFrame) -> list[str]:
        if self.feature_cols:
            return [c for c in self.feature_cols if c in df.columns]
        return total_feature_cols(df)

    def _make_stack(self):
        return ManualOOFStacker(
            estimators=[
                ("ridge", Ridge(alpha=self.ridge_alpha)),
                ("catboost", CatBoostRegressor(**self.cb_params)),
            ],
            final_estimator=HuberRegressor(epsilon=self.huber_epsilon),
            cv=_stack_cv_splitter(use_purged_cv=True),
        )

    def fit(self, X_df, y_home=None, y_away=None, calib_df=None):
        df = X_df.copy()
        if y_home is None:
            y_home = df["actual_home"] if "actual_home" in df.columns else None
        if y_away is None:
            y_away = df["actual_away"] if "actual_away" in df.columns else None
        if y_home is None or y_away is None:
            raise ValueError("MetaScorePairModel requires actual_home/actual_away or y_home/y_away")
        y_h = np.asarray(y_home, dtype=float)
        y_a = np.asarray(y_away, dtype=float)
        cols = self._resolve_cols(df)
        self._fit_cols_ = cols
        X = df[cols].fillna(0)
        X_scaled = self.scaler.fit_transform(X)

        groups = _stack_fit_groups(df)
        self.home_stack = self._make_stack()
        self.away_stack = self._make_stack()
        for stack, y in ((self.home_stack, y_h), (self.away_stack, y_a)):
            stack.fit(X_scaled, y, groups=groups)

        # In-sample residual SD as initial sigma (overwritten by walk-forward when available)
        pred_h = self.home_stack.predict(X_scaled)
        pred_a = self.away_stack.predict(X_scaled)
        self._rmse_home = float(max(self.sigma_floor, np.sqrt(np.mean((pred_h - y_h) ** 2))))
        self._rmse_away = float(max(self.sigma_floor, np.sqrt(np.mean((pred_a - y_a) ** 2))))
        pred_t = pred_h + pred_a
        y_t = y_h + y_a
        self._rmse_total = float(max(self.sigma_floor, np.sqrt(np.mean((pred_t - y_t) ** 2))))
        self.fitted = True
        return self

    def predict_scores(self, feat_dict: dict) -> dict:
        if not self.fitted:
            raise RuntimeError("MetaScorePairModel must be fitted first.")
        row = pd.DataFrame([feat_dict])
        cols = self._fit_cols_ or self._resolve_cols(row)
        for c in cols:
            if c not in row.columns:
                row[c] = 0.0
        X_scaled = self.scaler.transform(row[cols].fillna(0))
        pred_home = float(self.home_stack.predict(X_scaled)[0])
        pred_away = float(self.away_stack.predict(X_scaled)[0])
        pred_total = pred_home + pred_away
        pred_margin = pred_home - pred_away
        sigma_total = float(self._rmse_total)
        out = {
            "pred_home": pred_home,
            "pred_away": pred_away,
            "pred_home_pts": pred_home,
            "pred_away_pts": pred_away,
            "pred_total": pred_total,
            "pred_margin": pred_margin,
            "sigma_total": sigma_total,
            "sigma_home": float(self._rmse_home),
            "sigma_away": float(self._rmse_away),
        }
        mkt = feat_dict.get("market_total")
        if mkt is not None and np.isfinite(float(mkt)):
            out["p_over"] = total_over_prob_gaussian(pred_total, float(mkt), sigma_total)
            out["p_under"] = 1.0 - out["p_over"]
        return out

    def _predict_raw_total(self, feat_dict: dict) -> dict:
        return self.predict_scores(feat_dict)

    def predict_total(self, feat_dict: dict) -> float:
        return self.predict_scores(feat_dict)["pred_total"]

    def set_walkforward_rmse(self, rmse_home=None, rmse_away=None, rmse_total=None):
        if rmse_home is not None and np.isfinite(rmse_home):
            self._rmse_home = float(max(self.sigma_floor, rmse_home))
        if rmse_away is not None and np.isfinite(rmse_away):
            self._rmse_away = float(max(self.sigma_floor, rmse_away))
        if rmse_total is not None and np.isfinite(rmse_total):
            self._rmse_total = float(max(self.sigma_floor, rmse_total))

    def save(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path):
        import pickle
        with open(path, "rb") as f:
            return pickle.load(f)


def _meta_win_tuning_objective(
    trial,
    df,
    X_base,
    y_win,
    pred_margin_series,
    *,
    fast_mode,
    outer_splits,
    inner_splits,
    use_spread_features,
):

    C = trial.suggest_float("C", 0.01, 1.0, log=True)
    elo_win_blend = trial.suggest_float("elo_win_blend", 0.0, 0.6)
    calib_method = trial.suggest_categorical("calib_method", ["isotonic", "none"])

    scores = []
    EMBARGO = TUNING_EMBARGO_GAMES
    for train_idx, val_idx in _outer_game_cv_splits(df, outer_splits, EMBARGO):
        if len(val_idx) < 30:
            continue
        train_df = df.iloc[train_idx].copy()
        val_df = df.iloc[val_idx].copy()
        cols = win_feature_cols(train_df, use_spread_features=use_spread_features)
        if pred_margin_series is not None:
            train_df["meta_pred_margin"] = pred_margin_series.iloc[train_idx].values
            val_df["meta_pred_margin"] = pred_margin_series.iloc[val_idx].values
        X_tr = train_df[cols].fillna(0)
        X_val = val_df[cols].fillna(0)
        y_tr = y_win.iloc[train_idx].astype(int)
        y_val = y_win.iloc[val_idx].astype(int)

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        lr = LogisticRegression(C=C, solver="lbfgs", max_iter=200)
        lr.fit(X_tr_s, y_tr)

        elo_cols = [c for c in ELO_WIN_FEATURES if c in train_df.columns]
        p_val = lr.predict_proba(scaler.transform(X_val))[:, 1]
        if elo_cols:
            elo_sc = StandardScaler()
            Xe_tr = elo_sc.fit_transform(train_df[elo_cols].fillna(0))
            Xe_val = elo_sc.transform(val_df[elo_cols].fillna(0))
            elo_lr = LogisticRegression(C=0.1, solver="lbfgs", max_iter=200)
            elo_lr.fit(Xe_tr, y_tr)
            p_elo = elo_lr.predict_proba(Xe_val)[:, 1]
            p_val = (1.0 - elo_win_blend) * p_val + elo_win_blend * p_elo

        if calib_method == "isotonic" and len(train_idx) >= 80:
            iso = IsotonicRegression(out_of_bounds="clip")
            p_tr = lr.predict_proba(X_tr_s)[:, 1]
            iso.fit(p_tr, y_tr.values)
            p_val = iso.predict(p_val)

        brier = brier_score_loss(y_val, p_val)
        ece = compute_ece(y_val.values, p_val)

        ml_roi = 0.0
        if "market_ml" in val_df.columns:
            best_r = -1.0
            for ev_thr in (0.03, 0.05, 0.08, 0.10):
                rois = []
                for idx, row in val_df.iterrows():
                    mkt_ml = row.get("market_ml")
                    if mkt_ml is None or pd.isna(mkt_ml):
                        continue
                    i_loc = val_df.index.get_loc(idx)
                    p = float(p_val[i_loc])
                    fair = fair_home_win_prob(float(mkt_ml))
                    if fair is None:
                        continue
                    ev_home = p - fair
                    ev_away = (1 - p) - (1 - fair)
                    if ev_home >= ev_thr and p >= 0.5:
                        rois.append(float(y_val.loc[idx]))
                    elif ev_away >= ev_thr and p < 0.5:
                        rois.append(float(1 - y_val.loc[idx]))
                if len(rois) >= 10:
                    wp = np.mean(rois)
                    roi = wp * (100.0 / 110.0) - (1.0 - wp)
                    best_r = max(best_r, roi)
            ml_roi = best_r if best_r > -1 else 0.0

        combined = (
            META_WIN_LOSS_BRIER_WEIGHT * brier
            + META_WIN_LOSS_ECE_WEIGHT * (ece if np.isfinite(ece) else 0.0)
            - META_WIN_LOSS_ML_ROI_WEIGHT * ml_roi
        )
        scores.append(combined)
    return np.mean(scores) if scores else TUNING_INVALID_SCORE


def tune_meta_win_model(
    train_features_df,
    n_trials=25,
    season=None,
    bounds=None,
    feature_cols=None,
    pred_margin_col="pred_margin",
    fast_mode=None,
    use_spread_features: bool | None = None,
):
    """Tune MetaWinModel hyperparameters with Brier + ECE + ML ROI objective."""
    if fast_mode is None:
        fast_mode = n_trials <= 5

    df = train_features_df.dropna(subset=["actual_home", "actual_away"]).reset_index(drop=True)
    y_win = (df["actual_home"] > df["actual_away"]).astype(int)
    pred_margin_series = df[pred_margin_col] if pred_margin_col in df.columns else None
    outer_splits = 2 if fast_mode else 3
    inner_splits = 3 if fast_mode else 5

    if n_trials <= 0:
        return {
            "C": 0.1,
            "elo_win_blend": ELO_WIN_BLEND,
            "calib_method": "isotonic",
            "feature_cols": win_feature_cols(df, use_spread_features=use_spread_features),
        }

    cols_placeholder = win_feature_cols(df, use_spread_features=use_spread_features)
    X_base = df[cols_placeholder].fillna(0)

    def objective(trial):
        return _meta_win_tuning_objective(
            trial, df, X_base, y_win, pred_margin_series,
            fast_mode=fast_mode,
            outer_splits=outer_splits,
            inner_splits=inner_splits,
            use_spread_features=use_spread_features,
        )

    mode_label = "FAST" if fast_mode else "FULL"
    print(f"  ⏳ MetaWin tuning ({mode_label}, {n_trials} trials)...", flush=True)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    best = study.best_params
    return {
        "C": best["C"],
        "elo_win_blend": best.get("elo_win_blend", ELO_WIN_BLEND),
        "calib_method": best.get("calib_method", "isotonic"),
        "feature_cols": win_feature_cols(df, use_spread_features=use_spread_features),
        "_cv_score": float(study.best_value),
    }


def tune_meta_model(train_features_df, n_trials=25, season=None,
                    bounds: 'AdaptiveParameterBounds' = None,
                    use_isotonic_calib=False, feature_cols=None):
    """Alias for tune_margin_model (backward compatible)."""
    return tune_margin_model(
        train_features_df, n_trials=n_trials, season=season,
        bounds=bounds, feature_cols=feature_cols, use_elo_stack=True,
    )


class MetaWinModel:
    """Separate ML winner classifier with optional Elo-only head blend."""

    def __init__(self, C=0.1, feature_cols=None, elo_win_blend=None, calib_method="isotonic"):
        self.C = C
        self.feature_cols = list(feature_cols) if feature_cols else win_feature_cols()
        self.elo_win_blend = ELO_WIN_BLEND if elo_win_blend is None else elo_win_blend
        self.calib_method = calib_method
        self.model = LogisticRegression(C=C, solver='lbfgs', max_iter=200)
        self.scaler = StandardScaler()
        self.elo_model = LogisticRegression(C=0.1, solver='lbfgs', max_iter=200)
        self.elo_scaler = StandardScaler()
        self._elo_cols_ = []
        self.elo_fitted = False
        self.calibrator = None
        self.fitted = False

    def _elo_matrix(self, X_df):
        cols = [c for c in ELO_WIN_FEATURES if c in X_df.columns]
        self._elo_cols_ = cols
        if not cols:
            return None
        return X_df[cols].fillna(0)

    def fit(self, X_df, y_win, calib_df=None, pred_margin_col='pred_margin'):
        cols = [c for c in self.feature_cols if c in X_df.columns]
        self._fit_cols_ = list(cols)
        self._uses_pred_margin_ = pred_margin_col in X_df.columns
        X = X_df[cols].fillna(0)
        if self._uses_pred_margin_:
            X = X.copy()
            X['meta_pred_margin'] = X_df[pred_margin_col].fillna(0)
        X_scaled = self.scaler.fit_transform(X)
        self.model.fit(X_scaled, y_win.astype(int))

        Xe = self._elo_matrix(X_df)
        if Xe is not None and Xe.shape[1] > 0:
            Xe_s = self.elo_scaler.fit_transform(Xe)
            self.elo_model.fit(Xe_s, y_win.astype(int))
            self.elo_fitted = True

        if calib_df is not None and self.calib_method != "none":
            Xc = calib_df[self._fit_cols_].fillna(0)
            if self._uses_pred_margin_ and pred_margin_col in calib_df.columns:
                Xc = Xc.copy()
                Xc['meta_pred_margin'] = calib_df[pred_margin_col].fillna(0)
            raw = self._blend_raw_proba(Xc, calib_df, pred_margin_col)
            y_cal = (calib_df['actual_home'] > calib_df['actual_away']).astype(int)
            if self.calib_method == "platt":
                self.calibrator = LogisticRegression(C=0.1, solver='lbfgs', max_iter=100)
                self.calibrator.fit(raw.reshape(-1, 1), y_cal)
            else:
                self.calibrator = IsotonicRegression(out_of_bounds='clip')
                self.calibrator.fit(raw, y_cal)
        self.fitted = True

    def _blend_raw_proba(self, X, X_df, pred_margin_col='pred_margin'):
        p = self.model.predict_proba(self.scaler.transform(X))[:, 1]
        if not self.elo_fitted:
            return p
        Xe = self._elo_matrix(X_df)
        if Xe is None:
            return p
        p_elo = self.elo_model.predict_proba(self.elo_scaler.transform(Xe))[:, 1]
        w = self.elo_win_blend
        return (1.0 - w) * p + w * p_elo

    def predict_proba(self, feat_dict, pred_margin=None):
        if not self.fitted:
            raise RuntimeError("MetaWinModel must be fitted first.")
        row = pd.DataFrame([feat_dict])
        cols = getattr(self, '_fit_cols_', [c for c in self.feature_cols if c in row.columns])
        X = row[cols].fillna(0)
        if getattr(self, '_uses_pred_margin_', pred_margin is not None):
            X = X.copy()
            X['meta_pred_margin'] = 0.0 if pred_margin is None else pred_margin
        p = float(self._blend_raw_proba(X, row, pred_margin_col='meta_pred_margin')[0])
        if self.calibrator is not None:
            if isinstance(self.calibrator, LogisticRegression):
                p = float(self.calibrator.predict_proba(np.array([[p]]))[0, 1])
            else:
                p = float(self.calibrator.predict([p])[0])
        return float(np.clip(p, 0.01, 0.99))

    def save(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path):
        import pickle
        with open(path, "rb") as f:
            return pickle.load(f)


def build_feature_row(raw_feat: dict) -> dict:
    """Apply interaction features to a single pre-game feature dict (train/serve parity)."""
    df = pd.DataFrame([raw_feat])
    return engineer_interaction_features(df).iloc[0].to_dict()


In [ ]:
# ── module: metrics ──────────────────────────────────────────────────────────
"""Backtest metrics and betting edge grid search."""
from __future__ import annotations

import numpy as np
import pandas as pd

BREAKEVEN_ATS = 110.0 / 210.0


def variance_aware_edge_threshold(
    base_threshold: float,
    quantile_width: float,
    *,
    base_width: float = 20.0,
    scale: float = 0.05,
    max_bump: float = 2.5,
) -> float:
    """Require higher edge when quantile spread uncertainty is wide (#19)."""
    if not np.isfinite(quantile_width) or quantile_width <= base_width:
        return float(base_threshold)
    bump = min(max_bump, scale * (float(quantile_width) - base_width))
    return float(base_threshold) + bump


def _active_spread_bets(df):
    """Rows with a market line and an actionable spread bet under the active mode."""
    return spread_bet_frame(df)


def compute_clv(model_spread, bet_spread, closing_spread, side: str | None = None):
    """Bet-side point CLV (Task 038).

    Preferred: pass ``side`` (``Home``/``Away``) with ``bet_spread`` as the
    decision/T-60 home line and ``closing_spread`` as the close home line.

    Legacy callers pass ``(model_spread, bet_spread, closing_spread)`` without
    ``side``; the bet side is then inferred from ``sign(model + bet)``.
    Point CLV is never a multiplicative return — use ``PRICE_CLV`` separately.
    """

    if pd.isna(bet_spread) or pd.isna(closing_spread):
        return np.nan
    if side is None:
        if pd.isna(model_spread):
            return np.nan
        edge = float(model_spread) + float(bet_spread)
        side = "Home" if edge >= 0 else "Away"
    return bet_side_point_clv(bet_spread, closing_spread, side)


def add_clv_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add point CLV and (when available) price CLV columns.

    Point CLV uses decision vs close home spreads and the bet DIRECTION.
    Price/probability CLV is kept in a separate column and never compounded
    with point CLV via a multiplicative return identity (Task 028/038).
    """

    out = df.copy()
    if "CLOSING_SPREAD" not in out.columns:
        out["CLOSING_SPREAD"] = out.get("MARKET_SPREAD", np.nan)
    if "MARKET_SPREAD" in out.columns and "PRED_SPREAD" in out.columns and "ACTUAL_MARGIN" in out.columns:
        out["MARKET_RESIDUAL"] = out["ACTUAL_MARGIN"] + out["CLOSING_SPREAD"]
        out["MODEL_VS_CLOSE"] = out["PRED_SPREAD"] + out["CLOSING_SPREAD"]
    if all(c in out.columns for c in ("MARKET_SPREAD", "CLOSING_SPREAD")):
        decision = out["MARKET_SPREAD"]  # decision/T-60 line when wired; legacy alias today
        close = out["CLOSING_SPREAD"]
        if "DIRECTION" in out.columns:
            sides = out["DIRECTION"].fillna("Pass")
        elif "EDGE" in out.columns:
            sides = np.where(out["EDGE"] > 0, "Home", np.where(out["EDGE"] < 0, "Away", "Pass"))
        else:
            sides = pd.Series(["Pass"] * len(out), index=out.index)
        point_vals = []
        for dec, clo, side in zip(decision, close, sides):
            if str(side) in ("Pass", "nan") or pd.isna(dec) or pd.isna(clo):
                point_vals.append(np.nan)
            else:
                point_vals.append(bet_side_point_clv(dec, clo, side))
        out["CLV"] = point_vals
        out["POINT_CLV"] = out["CLV"]
        if "DECISION_FAIR_PROB" in out.columns and "CLOSE_FAIR_PROB" in out.columns:
            out["PRICE_CLV"] = [
                bet_side_price_clv(d, c)
                for d, c in zip(out["DECISION_FAIR_PROB"], out["CLOSE_FAIR_PROB"])
            ]
    return out


def bootstrap_ci(values, n_boot: int = 1000, alpha: float = 0.05, stat="mean", seed: int = 42):
    """Bootstrap confidence interval for mean or rate.

    NOTE: when ``stat="mean"`` on per-bet profits this is a *mean-profit*
    interval, NOT an ROI interval. Use ``bootstrap_roi_ci`` for
    ``sum(profit)/sum(stake)`` with date/week block resampling (Task 042).
    """
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) < 5:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    stats = []
    for _ in range(n_boot):
        sample = rng.choice(arr, size=len(arr), replace=True)
        stats.append(sample.mean() if stat == "mean" else np.mean(sample > 0))
    lo = float(np.quantile(stats, alpha / 2))
    hi = float(np.quantile(stats, 1 - alpha / 2))
    return float(np.mean(stats)), lo, hi


def bootstrap_roi_ci(
    profits,
    stakes,
    *,
    block_ids=None,
    n_boot: int = 1000,
    alpha: float = 0.05,
    seed: int = 42,
):
    """Block-bootstrap CI for ``sum(profit)/sum(stake)`` (Task 042).

    ``block_ids`` groups bets into date/week blocks resampled as wholes.
    Returns ``(roi_point, lo, hi)``. Distinct from ``bootstrap_ci`` mean-profit.
    """
    profits = np.asarray(profits, dtype=float)
    stakes = np.asarray(stakes, dtype=float)
    mask = np.isfinite(profits) & np.isfinite(stakes) & (stakes > 0)
    profits, stakes = profits[mask], stakes[mask]
    if len(profits) < 5:
        return np.nan, np.nan, np.nan
    if block_ids is None:
        block_ids = np.arange(len(profits))
    else:
        block_ids = np.asarray(block_ids)[mask]
    unique_blocks = np.unique(block_ids)
    if len(unique_blocks) < 3:
        # Fall back to bet-level only when too few blocks — still label as ROI.
        unique_blocks = np.arange(len(profits))
        block_ids = unique_blocks
    rng = np.random.default_rng(seed)
    rois = []
    for _ in range(n_boot):
        sampled = rng.choice(unique_blocks, size=len(unique_blocks), replace=True)
        p_sum = 0.0
        s_sum = 0.0
        for b in sampled:
            sel = block_ids == b
            p_sum += float(profits[sel].sum())
            s_sum += float(stakes[sel].sum())
        if s_sum > 0:
            rois.append(p_sum / s_sum)
    if not rois:
        return np.nan, np.nan, np.nan
    point = float(np.sum(profits) / np.sum(stakes))
    lo = float(np.quantile(rois, alpha / 2))
    hi = float(np.quantile(rois, 1 - alpha / 2))
    return point, lo, hi


def assert_actionable_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Reject inconsistent actionable-bet rows (Task 043).

    Requires agreement among ACTIONABLE, DIRECTION, stake availability, and
    market presence for rows marked actionable.
    """
    if df is None or df.empty:
        return df
    required = {"DIRECTION", "MARKET_SPREAD"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"actionable frame missing columns: {sorted(missing)}")
    out = df.copy()
    if "ACTIONABLE" not in out.columns:
        out["ACTIONABLE"] = out["DIRECTION"].ne("Pass") & out["MARKET_SPREAD"].notna()
    bad = []
    for i, r in out.iterrows():
        actionable = bool(r.get("ACTIONABLE"))
        direction = r.get("DIRECTION", "Pass")
        has_market = pd.notna(r.get("MARKET_SPREAD"))
        if actionable:
            if direction in (None, "Pass", "nan") or not has_market:
                bad.append(i)
        if direction not in (None, "Pass", "nan", "Home", "Away"):
            bad.append(i)
        stake_cols = [c for c in out.columns if c.startswith("STAKE_")]
        for sc in stake_cols:
            stake = r.get(sc, 0) or 0
            if float(stake) > 0 and (direction in (None, "Pass") or not has_market):
                bad.append(i)
    if bad:
        raise ValueError(
            f"actionable-frame consistency failed for {len(set(bad))} rows "
            f"(e.g. index {next(iter(set(bad)))})"
        )
    return out


def ats_win_series(df):
    """Boolean series: active non-push spread bets that covered (Task 040).

    Pushes are excluded (NaN), not counted as losses.
    """

    d = _active_spread_bets(df)
    if d.empty:
        return pd.Series(dtype=bool)
    outcomes = [
        grade_spread_bet(m, s, side)
        for m, s, side in zip(d["ACTUAL_MARGIN"], d["MARKET_SPREAD"], d["DIRECTION"])
    ]
    result = pd.Series(
        [np.nan if o == "push" else (o == "win") for o in outcomes],
        index=d.index,
        dtype=float,
    )
    return result.dropna().astype(bool)


def clv_weighted_roi(df):
    """ROI weighted by positive CLV (rewards beating the close)."""
    d = add_clv_columns(_active_spread_bets(df))
    if d.empty or "CLV" not in d.columns:
        return np.nan
    wins = ats_win_series(d)
    if len(wins) == 0:
        return np.nan
    weights = np.clip(d.loc[wins.index, "CLV"].fillna(0) + 1.0, 0.5, 2.0)
    wp = (wins.astype(float) * weights).sum() / weights.sum()
    return wp * (100.0 / 110.0) - (1 - wp)


def sharpe_ratio(profits, eps: float = 1e-9) -> float:
    """Per-bet Sharpe (mean / std)."""
    arr = np.asarray(profits, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) < 5:
        return float("nan")
    sd = float(arr.std(ddof=1)) if len(arr) > 1 else 0.0
    if sd < eps:
        return float("nan")
    return float(arr.mean() / sd)


def max_drawdown(cumulative: np.ndarray) -> float:
    if len(cumulative) == 0:
        return float("nan")
    peak = np.maximum.accumulate(cumulative)
    dd = cumulative - peak
    return float(dd.min())


def walkforward_edge_threshold(
    prior_df,
    edges=(2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0),
    default=3.5,
    min_bets=40,
    optimize="bucket_roi",
    require_positive_ci=True,
    min_floor=None,
):
    """Pick ATS edge threshold from prior seasons.

    optimize: "roi" | "clv_roi" | "ats" | "bucket_roi"
    bucket_roi: maximize ROI with penalty for bets landing in the weak 2–4 pt bucket.
    require_positive_ci: reject thresholds whose bootstrap ROI 95% CI lower bound < break-even
    """
    if prior_df is None or prior_df.empty or "EDGE" not in prior_df.columns:
        return default
    d = prior_df[prior_df["MARKET_SPREAD"].notna()].copy()
    if d.empty:
        return default
    best, best_score = default, -1e9
    cover = d["ACTUAL_MARGIN"] + d["MARKET_SPREAD"]
    d = add_clv_columns(d)
    for t in edges:
        active = d["EDGE"].abs() >= t
        bets = d[active]
        if len(bets) < min_bets:
            continue
        home = bets["EDGE"] > 0
        c = cover[active]
        win = (home & (c > 0)) | (~home & (c < 0))
        wp = win.mean()
        roi = wp * (100.0 / 110.0) - (1 - wp)
        if optimize == "ats":
            score = wp
        elif optimize == "clv_roi":
            score = clv_weighted_roi(bets)
            if np.isnan(score):
                score = roi
        elif optimize == "bucket_roi":
            abs_e = bets["EDGE"].abs()
            weak_share = (abs_e < 4.0).mean()
            core = abs_e >= 4.0
            if core.sum() >= max(20, min_bets // 2):
                core_home = home[core]
                core_c = c[core]
                core_win = (core_home & (core_c > 0)) | (~core_home & (core_c < 0))
                core_wp = core_win.mean()
                core_roi = core_wp * (100.0 / 110.0) - (1 - core_wp)
                score = core_roi - 0.08 * weak_share
            else:
                score = roi - 0.10 * weak_share
        else:
            score = roi
        if require_positive_ci:
            wins_arr = win.astype(float).values
            _, lo, _ = bootstrap_ci(wins_arr, n_boot=500)
            if np.isfinite(lo) and lo < BREAKEVEN_ATS:
                continue
        if score > best_score or (abs(score - best_score) < 1e-9 and t > best):
            best_score, best = score, t
    if min_floor is not None and best < min_floor:
        best = float(min_floor)
    return best


def walkforward_confidence_gate(
    prior_df,
    thresholds_min=None,
    thresholds_max: tuple[int | None, ...] = (None, 61, 62, 63, 64, 65),
    default_min: float | None = None,
    default_max: float | None = None,
    min_bets: int = 80,
    bet_calibrator=None,
    min_gated_bets: int = 100,
) -> tuple[float, float | None]:
    """Walk-forward min/max WIN_PCT band for actionable spread bets."""

    if default_min is None:
        default_min = float(MIN_CONFIDENCE_SCORE)
    if default_max is None:
        default_max = MAX_CONFIDENCE_SCORE
    if thresholds_min is None:
        thresholds_min = (55, 56, 57, 58, 59, 60)

    if prior_df is None or prior_df.empty:
        return float(default_min), default_max

    d = actionable_spread_frame(prior_df)
    if d.empty:
        # Early seasons may not have ACTIONABLE populated consistently; fallback to leans.
        d = lean_spread_frame(prior_df)
    if d.empty:
        return float(default_min), default_max

    if bet_calibrator is not None and getattr(bet_calibrator, "_fitted", False):
        scored = _scored_ats_bets(bet_calibrator, d)
        if len(scored) >= min_bets:
            roi, best_min, best_max, best_n = _pick_confidence_band_gated_roi(
                scored,
                min_thresholds=thresholds_min,
                max_thresholds=thresholds_max,
                min_bets=min_bets,
            )
            if best_n >= min_bets and roi > -1e8:
                best_min = max(float(best_min), float(MIN_CONFIDENCE_SCORE))
                if best_max is not None and default_max is not None:
                    best_max = min(float(best_max), float(default_max))
                gated_n = sum(
                    1 for _y, cs, _p in scored
                    if cs >= best_min and (best_max is None or cs <= best_max)
                )
                if gated_n < min_gated_bets:
                    return float(MIN_CONFIDENCE_SCORE), default_max
                return best_min, best_max

    # Fallback: min-only on stored WIN_PCT / CONFIDENCE
    best_min = walkforward_min_confidence(
        prior_df,
        thresholds=thresholds_min,
        default=default_min,
        min_bets=min_bets,
        require_positive_ci=False,
        bet_calibrator=None,
    )
    best_min = max(float(best_min), float(MIN_CONFIDENCE_SCORE))
    if best_min > 60:
        best_min = float(MIN_CONFIDENCE_SCORE)
    return best_min, default_max


def walkforward_min_confidence(
    prior_df,
    thresholds=(50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70),
    default: float = 64,
    min_bets: int = 40,
    require_positive_ci: bool = True,
    bet_calibrator=None,
):
    """Pick minimum WIN_PCT / CONFIDENCE for ATS leans from prior seasons only.

    When *bet_calibrator* is fitted, leans are re-scored through the same isotonic
    path used at simulation time (avoids tuning on stale raw WIN_PCT).
    """

    if prior_df is None or prior_df.empty:
        return float(default)

    d = lean_spread_frame(prior_df)
    if d.empty:
        return float(MIN_CONFIDENCE_SCORE if default is None else default)

    if bet_calibrator is not None and getattr(bet_calibrator, "_fitted", False):

        scored = _scored_ats_bets(bet_calibrator, d)
        if len(scored) >= min_bets:
            _, _, best_thr, best_n = _pick_threshold_gated_roi(
                scored, thresholds=thresholds, min_bets=min_bets,
            )
            if best_n >= min_bets:
                return float(best_thr)
        if scored:
            best_thr = float(default)
            best_roi = -1e9
            for thr in thresholds:
                sub = [y for y, cs, _p in scored if cs >= thr]
                if len(sub) < min_bets:
                    continue
                wp = float(np.mean(sub))
                roi = wp * (100.0 / 110.0) - (1.0 - wp)
                if roi > best_roi:
                    best_roi, best_thr = roi, float(thr)
            if best_roi > -1e8:
                return best_thr

    score_col = win_pct_column(d)
    if score_col not in d.columns:
        return float(default)

    cover = d["ACTUAL_MARGIN"] + d["MARKET_SPREAD"]
    side = d["_side"]
    win = (side.eq("Home") & (cover > 0)) | (side.eq("Away") & (cover < 0))
    d = d.assign(_win=win.astype(float))

    best_thr = float(default)
    best_roi = -1e9
    for thr in thresholds:
        bets = d[pd.to_numeric(d[score_col], errors="coerce") >= thr]
        if len(bets) < min_bets:
            continue
        wp = float(bets["_win"].mean())
        roi = wp * (100.0 / 110.0) - (1 - wp)
        if require_positive_ci:
            _, lo, _ = bootstrap_ci(bets["_win"].astype(float).values, n_boot=500)
            if np.isfinite(lo) and lo < BREAKEVEN_ATS:
                continue
        if roi > best_roi:
            best_roi, best_thr = roi, float(thr)
    return best_thr


def apply_edge_threshold(df, threshold):
    """Re-mark spread DIRECTION/EDGE selection at a given edge threshold."""
    df = df.copy()
    e = df["EDGE"]
    active = e.abs() >= threshold
    df["DIRECTION"] = np.where(active, np.where(e > 0, "Home", "Away"), "Pass")
    df["EDGE_THRESHOLD"] = threshold
    return df


def grid_search_bet_edge(results_df, edges=(1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 5.5, 6.0)):
    """Find ATS edge threshold that maximizes ROI; re-derive DIRECTION from EDGE each step."""
    best_edge, best_roi = 2.5, -1.0
    rows = []
    df = results_df.copy()
    if "MARKET_SPREAD" not in df.columns:
        return best_edge, pd.DataFrame(rows)
    has_odds = df["MARKET_SPREAD"].notna()
    for edge in edges:
        sim = apply_edge_threshold(df[has_odds], edge)
        bets = sim[sim["DIRECTION"] != "Pass"]
        if len(bets) < 20:
            continue
        wins = ats_win_series(bets)
        win_pct = wins.mean()
        roi = win_pct * (100 / 110) - (1 - win_pct)
        rows.append({"edge": edge, "n_bets": len(bets), "win_pct": win_pct, "roi": roi})
        if roi > best_roi:
            best_roi, best_edge = roi, edge
    return best_edge, pd.DataFrame(rows)


def compare_selection_strategies(
    results_df,
    *,
    min_conf: float | None = None,
    confidence_min_edge: float | None = None,
    edge_thresholds: tuple[float, ...] = (5.5, 6.0),
) -> pd.DataFrame:
    """Apples-to-apples ROI table on one backtest export (no re-simulation)."""

    if results_df is None or results_df.empty or "MARKET_SPREAD" not in results_df.columns:
        return pd.DataFrame()

    min_conf = float(MIN_CONFIDENCE_SCORE if min_conf is None else min_conf)
    confidence_min_edge = float(CONFIDENCE_MIN_EDGE if confidence_min_edge is None else confidence_min_edge)
    df = results_df[results_df["MARKET_SPREAD"].notna()].copy()
    conf_col = "CONFIDENCE" if "CONFIDENCE" in df.columns else "WIN_PCT"
    edge_s = pd.to_numeric(df.get("EDGE"), errors="coerce").abs()
    rows = []

    def _row(mode: str, bets: pd.DataFrame) -> dict:
        if bets.empty:
            return {"mode": mode, "n_bets": 0, "win_pct": np.nan, "roi": np.nan}
        wins = ats_win_series(bets)
        if wins.empty:
            return {"mode": mode, "n_bets": 0, "win_pct": np.nan, "roi": np.nan}
        wp = float(wins.mean())
        return {
            "mode": mode,
            "n_bets": int(len(bets)),
            "win_pct": wp,
            "roi": float(wp * (100.0 / 110.0) - (1.0 - wp)),
        }

    if conf_col in df.columns:
        hybrid = df[
            (df["DIRECTION"] != "Pass")
            & (edge_s >= confidence_min_edge)
            & (pd.to_numeric(df[conf_col], errors="coerce") >= min_conf)
        ]
        rows.append(_row(
            f"confidence_hybrid|edge|>={confidence_min_edge:g}&conf>={min_conf:g}",
            hybrid,
        ))
        scaled_thr = edge_scaled_min_confidence(confidence_min_edge, base=min_conf)
        edge_scaled = df[
            (df["DIRECTION"] != "Pass")
            & (edge_s >= confidence_min_edge)
            & (
                pd.to_numeric(df[conf_col], errors="coerce")
                >= edge_s.map(lambda ae: edge_scaled_min_confidence(ae, base=min_conf))
            )
        ]
        rows.append(_row(
            f"confidence_edge_scaled|edge|>={confidence_min_edge:g}&scaled_conf>={scaled_thr:g}+",
            edge_scaled,
        ))

    for thr in edge_thresholds:
        sim = apply_edge_threshold(df, thr)
        edge_bets = sim[sim["DIRECTION"] != "Pass"]
        rows.append(_row(f"edge_only|edge|>={thr:g}", edge_bets))

    return pd.DataFrame(rows)


def walkforward_favorite_decimal(
    prior_df,
    candidates=(1.35, 1.40, 1.45, 1.50, 1.55, 1.60, 1.65, 1.70),
    default=None,
    min_bets=30,
):
    """Pick max favorite decimal odds for ML bets from prior seasons."""
    if default is None:
        default = ML_MAX_FAVORITE_DECIMAL_DEFAULT
    if prior_df is None or prior_df.empty or "ML_DIRECTION" not in prior_df.columns:
        return default
    best, best_roi = default, -1e9
    for dec in candidates:
        bets = prior_df[prior_df["ML_DIRECTION"] != "Pass"].copy()
        if bets.empty:
            continue
        profits = []
        for _, r in bets.iterrows():
            ml = r["MARKET_ML"]
            if pd.isna(ml):
                continue
            d_home = 1 + ml / 100.0 if ml > 0 else 1 + 100.0 / abs(ml)
            d_away = 1 + (-ml) / 100.0 if -ml > 0 else 1 + 100.0 / abs(-ml)
            d = d_home if r["ML_DIRECTION"] == "Home" else d_away
            if d < dec:
                continue
            home_win = r["ACTUAL_HOME"] > r["ACTUAL_AWAY"]
            won = home_win if r["ML_DIRECTION"] == "Home" else not home_win
            profits.append((d - 1) if won else -1.0)
        if len(profits) < min_bets:
            continue
        wp = np.mean([p > 0 for p in profits])
        roi = np.mean(profits)
        if roi > best_roi:
            best_roi, best = roi, dec
    return best


def apply_ou_threshold(df, threshold):
    """Re-mark O/U direction from TOTAL_EDGE at a given points threshold."""
    df = df.copy()
    if "TOTAL_EDGE" not in df.columns:
        if {"PRED_TOTAL", "MARKET_TOTAL"}.issubset(df.columns):
            df["TOTAL_EDGE"] = df["PRED_TOTAL"] - df["MARKET_TOTAL"]
        else:
            df["OU_DIRECTION"] = "Pass"
            return df
    te = pd.to_numeric(df["TOTAL_EDGE"], errors="coerce")
    mkt = pd.to_numeric(df["MARKET_TOTAL"], errors="coerce")
    has_total = mkt.notna() & (mkt > 0)
    active = has_total & (te.abs() >= float(threshold))
    direction = pd.Series("Pass", index=df.index, dtype=object)
    direction.loc[active & (te > 0)] = "Over"
    direction.loc[active & (te < 0)] = "Under"
    df["OU_DIRECTION"] = direction
    if {"ACTUAL_HOME", "ACTUAL_AWAY", "MARKET_TOTAL"}.issubset(df.columns):
        actual_total = df["ACTUAL_HOME"] + df["ACTUAL_AWAY"]
        mkt = pd.to_numeric(df["MARKET_TOTAL"], errors="coerce")
        ou_win = []
        for tot, line, direction in zip(actual_total, mkt, df["OU_DIRECTION"]):
            if direction == "Pass" or pd.isna(tot) or pd.isna(line):
                ou_win.append(np.nan)
                continue
            outcome = grade_total_bet(tot, line, direction)
            if outcome == "push":
                ou_win.append(np.nan)  # excluded from hit-rate (Task 040)
            else:
                ou_win.append(1.0 if outcome == "win" else 0.0)
        df["OU_WIN"] = ou_win
    df["OU_MIN_EDGE"] = threshold
    return df


def walkforward_ou_edge(
    prior_df,
    edges=(3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0),
    default=4.5,
    min_bets=40,
    require_positive_ci: bool = True,
):
    """Pick O/U edge threshold from prior seasons (re-derive all O/U bets each step)."""

    if default is None:
        default = OU_MIN_EDGE
    if require_positive_ci is None:
        require_positive_ci = OU_REQUIRE_POSITIVE_CI
    if prior_df is None or prior_df.empty:
        return float(default)

    base = prior_df[prior_df["MARKET_TOTAL"].notna()].copy()
    if base.empty:
        return float(default)
    if "TOTAL_EDGE" not in base.columns and {"PRED_TOTAL", "MARKET_TOTAL"}.issubset(base.columns):
        base["TOTAL_EDGE"] = base["PRED_TOTAL"] - base["MARKET_TOTAL"]

    best, best_roi = float(default), -1e9
    for t in edges:
        sim = apply_ou_threshold(base, t)
        bets = sim[sim["OU_DIRECTION"] != "Pass"]
        if len(bets) < min_bets or "OU_WIN" not in bets.columns:
            continue
        wins = bets["OU_WIN"].astype(float)
        wp = float(wins.mean())
        roi = wp * (100.0 / 110.0) - (1 - wp)
        if require_positive_ci:
            _, lo, _ = bootstrap_ci(wins.values, n_boot=500)
            if np.isfinite(lo) and lo < BREAKEVEN_ATS:
                continue
        if roi > best_roi:
            best_roi, best = roi, float(t)
    return best


def compute_stake_profits(df, profile: str = "moderate"):
    """Add per-profile stake and profit columns from backtest results.

    Task 041: always apply slate/daily caps to stakes *before* computing
    profit. Task 040: pushes return zero profit and are not treated as losses.
    """

    if df is None or df.empty:
        return df
    out = df.copy()
    stake_col = f"STAKE_{profile.upper()}"
    profit_col = f"PROFIT_{profile.upper()}"

    thr = float(out["EDGE_THRESHOLD"].iloc[0]) if "EDGE_THRESHOLD" in out.columns else 2.5
    edge_bucket_min = edge_bucket_min_for_stakes()

    # Fast path: stakes already present — still must cap before profit.
    if stake_col in out.columns:
        out = apply_daily_caps(out, stake_col, profile=profile)
        profits = []
        for _, r in out.iterrows():
            stake = float(r.get(stake_col, 0) or 0)
            direction = r.get("DIRECTION", "Pass")
            juice = r.get("SPREAD_PRICE", r.get("JUICE", -110))
            if juice is None or (isinstance(juice, float) and np.isnan(juice)):
                juice = -110
            if stake <= 0 or direction in (None, "Pass") or pd.isna(r.get("MARKET_SPREAD")):
                profits.append(0.0)
                continue
            outcome = grade_spread_bet(r["ACTUAL_MARGIN"], r["MARKET_SPREAD"], direction)
            profits.append(exact_price_profit(stake, outcome, juice))
        out[profit_col] = profits
        return out

    stakes = []
    for _, r in out.iterrows():
        direction = r.get("DIRECTION", "Pass")
        edge = float(r.get("EDGE", 0) or 0)
        cover_prob = float(r.get("COVER_PROB_CALIBRATED", r.get("WIN_PROB", 0.5)) or 0.5)
        tier = int(r.get("CONFIDENCE_TIER", 2) or 2)
        conf_score = r.get("CONFIDENCE")
        conf_score = int(conf_score) if pd.notna(conf_score) else None
        edge_mult = edge_stake_multiplier(abs(edge)) if uses_edge_gates() else 1.0
        vol_mult = float(r.get("VOL_STAKE_MULT", 1.0) or 1.0)
        juice = r.get("SPREAD_PRICE", r.get("JUICE", -110))
        if juice is None or (isinstance(juice, float) and np.isnan(juice)):
            juice = -110
        stake = compute_stake(
            profile,
            direction=direction,
            edge_pts=edge,
            edge_threshold=thr,
            cover_prob=cover_prob,
            confidence_tier=tier,
            conf_width=float(r.get("CONF_WIDTH", 24) or 24),
            rating_uncertainty=float(r.get("RATING_UNCERTAINTY", 350) or 350),
            juice=float(juice) if abs(float(juice)) >= 100 else -110,
            edge_stake_mult=edge_mult,
            edge_bucket_min=edge_bucket_min,
            volatility_mult=vol_mult,
            confidence_score=conf_score,
        )
        stakes.append(stake)
    out[stake_col] = stakes
    # Caps BEFORE profit (Task 041) — never the reverse.
    out = apply_daily_caps(out, stake_col, profile=profile)
    profits = []
    for _, r in out.iterrows():
        stake = float(r.get(stake_col, 0) or 0)
        direction = r.get("DIRECTION", "Pass")
        juice = r.get("SPREAD_PRICE", r.get("JUICE", -110))
        if juice is None or (isinstance(juice, float) and np.isnan(juice)):
            juice = -110
        if stake <= 0 or direction in (None, "Pass") or pd.isna(r.get("MARKET_SPREAD")):
            profits.append(0.0)
            continue
        outcome = grade_spread_bet(r["ACTUAL_MARGIN"], r["MARKET_SPREAD"], direction)
        profits.append(exact_price_profit(stake, outcome, juice))
    out[profit_col] = profits
    return out


def benchmark_betting_roi(df, profile: str = "moderate", n_boot: int = 1000):
    """Print ROI, drawdown, and bootstrap CI for a stake profile."""

    if df is None or df.empty:
        print(f"No results for profile {profile}.")
        return {}
    prof_col = f"PROFIT_{profile.upper()}"
    stake_col = f"STAKE_{profile.upper()}"
    if prof_col not in df.columns:
        df = compute_stake_profits(df, profile=profile)
    active = df[df[stake_col] > 0]
    if active.empty:
        print(f"Profile {profile}: no active bets.")
        return {"roi": np.nan, "n_bets": 0}
    profits = active[prof_col].values
    stakes = active[stake_col].values
    staked = stakes.sum()
    total_profit = profits.sum()
    roi = total_profit / staked if staked > 0 else np.nan
    cum = np.cumsum(profits)
    dd = float((cum - np.maximum.accumulate(cum)).min()) if len(cum) else 0.0
    # Hit rate excludes zero-profit pushes (profit==0 with stake>0 may also be
    # floating noise; grade-aware path already returns 0 for pushes).
    graded_wins = profits > 0
    graded_decided = profits != 0
    wp = float(graded_wins[graded_decided].mean()) if graded_decided.any() else np.nan
    sharpe = sharpe_ratio(profits)
    # Task 042: label mean-profit CI separately from true ROI CI.
    mean_profit, mean_lo, mean_hi = bootstrap_ci(profits, n_boot=n_boot, stat="mean")
    date_col = "DATE" if "DATE" in active.columns else ("game_date" if "game_date" in active.columns else None)
    block_ids = active[date_col].astype(str).values if date_col else None
    roi_point, roi_lo, roi_hi = bootstrap_roi_ci(
        profits, stakes, block_ids=block_ids, n_boot=n_boot,
    )
    print(f"\n--- Betting ROI ({profile}) ---")
    print(f"  Active bets: {len(active)}  win% (excl. pushes): {wp:.1%}" if np.isfinite(wp) else f"  Active bets: {len(active)}")
    print(f"  ROI: {roi:+.1%}  total profit (units): {total_profit:+.3f}")
    print(f"  Max drawdown (units): {dd:.3f}")
    if np.isfinite(sharpe):
        print(f"  Sharpe (per bet): {sharpe:.2f}")
    if np.isfinite(mean_lo):
        print(f"  Mean profit/bet 95% CI (NOT ROI): [{mean_lo:+.3f}, {mean_hi:+.3f}]")
    if np.isfinite(roi_lo):
        print(f"  ROI 95% CI (block bootstrap): [{roi_lo:+.1%}, {roi_hi:+.1%}]")
    if "CONFIDENCE_TIER" in active.columns:
        print("  By confidence tier:")
        for tier in sorted(active["CONFIDENCE_TIER"].dropna().unique()):
            sub = active[active["CONFIDENCE_TIER"] == tier]
            if len(sub) >= 5:
                sub_roi = sub[prof_col].sum() / sub[stake_col].sum()
                print(f"    tier {int(tier)}: n={len(sub)} roi={sub_roi:+.1%}")
    return {"roi": float(roi), "n_bets": len(active), "win_pct": float(wp),
            "max_drawdown": dd, "sharpe": sharpe,
            "mean_profit_ci_lo": mean_lo, "mean_profit_ci_hi": mean_hi,
            "roi_ci_lo": roi_lo, "roi_ci_hi": roi_hi,
            "roi_point": roi_point}


def add_all_profile_columns(df):
    """Add stake/profit columns for all three profiles."""
    out = df
    for p in ("conservative", "moderate", "aggressive"):
        out = compute_stake_profits(out, profile=p)
    return out


def portfolio_kelly_fractions(df, max_daily_exposure: float = 0.15, fractional: float = 0.25):
    """Cap correlated same-day Kelly stakes."""
    if df is None or df.empty or "KELLY_FRACTION" not in df.columns:
        return df
    out = df.copy()
    if "DATE" not in out.columns:
        out["ADJ_KELLY"] = out["KELLY_FRACTION"] * fractional
        return out
    out["ADJ_KELLY"] = 0.0
    for _, g in out.groupby("DATE"):
        k = g["KELLY_FRACTION"].fillna(0) * fractional
        total = k.sum()
        if total > max_daily_exposure:
            k = k * (max_daily_exposure / total)
        out.loc[g.index, "ADJ_KELLY"] = k
    return out


def monthly_holdout_metrics(results_df, season_col="simulated_season_window"):
    """Spread MAE and ATS by calendar month within each season."""
    if results_df is None or results_df.empty:
        return pd.DataFrame()
    df = results_df.copy()
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    df["month"] = df["DATE"].dt.to_period("M").astype(str)
    rows = []
    for (season, month), g in df.groupby([season_col, "month"], dropna=False):
        mae = (g["PRED_SPREAD"] - g["ACTUAL_MARGIN"]).abs().mean()
        wins = ats_win_series(g)
        rows.append({
            "season": season, "month": month, "n_games": len(g),
            "spread_mae": mae, "ats_pct": wins.mean() if len(wins) else np.nan,
            "n_bets": len(wins),
        })
    return pd.DataFrame(rows)


def benchmark_results(results_df, n_boot: int = 1000):
    """Print spread MAE, ATS, CLV, bootstrap CIs."""
    if results_df.empty:
        print("No results to benchmark.")
        return
    df = add_clv_columns(results_df)
    mae = (df["PRED_SPREAD"] - df["ACTUAL_MARGIN"]).abs().mean()
    active = _active_spread_bets(df)
    if len(active):
        wins = ats_win_series(df)
        ats = wins.mean()
        _, lo, hi = bootstrap_ci(wins.astype(float), n_boot=n_boot)
    else:
        ats = lo = hi = float("nan")
    ml_acc = df.get("MODEL_ML_CORRECT", pd.Series(dtype=float)).mean()
    total_mae = df.get("TOTAL_ERR", pd.Series(dtype=float)).abs().mean()
    clv_mean = df.get("CLV", pd.Series(dtype=float)).mean()
    close_mae = np.nan
    if "CLOSING_SPREAD" in df.columns and df["CLOSING_SPREAD"].notna().any():
        mkt = df[df["CLOSING_SPREAD"].notna()]
        close_mae = (-mkt["CLOSING_SPREAD"] - mkt["ACTUAL_MARGIN"]).abs().mean()
    print(f"Spread MAE: {mae:.2f}")
    if np.isfinite(close_mae):
        print(f"Closing line MAE: {close_mae:.2f}  (model vs close: {mae - close_mae:+.2f})")
    print(f"ATS win% (active bets): {ats:.1%} ({len(active)} bets)")
    if np.isfinite(lo):
        print(f"  95% CI: [{lo:.1%}, {hi:.1%}]  (breakeven {BREAKEVEN_ATS:.1%})")
    if np.isfinite(clv_mean):
        print(f"Mean CLV (pts): {clv_mean:+.3f}")
    print(f"ML winner accuracy (diag): {ml_acc:.1%}")
    print(f"Total MAE (diag): {total_mae:.2f}")


def print_accuracy_layers(results_df) -> None:
    """Spread MAE vs all-leans ATS vs actionable ATS (model vs policy quality)."""

    if results_df is None or results_df.empty:
        print("No results for accuracy layers.")
        return
    spread_mae = (
        pd.to_numeric(results_df["PRED_SPREAD"], errors="coerce")
        - pd.to_numeric(results_df["ACTUAL_MARGIN"], errors="coerce")
    ).abs().mean()
    leans = lean_spread_frame(results_df)
    act = actionable_spread_frame(results_df)
    lean_w = float(ats_win_series(leans).mean()) if len(leans) else float("nan")
    act_w = float(ats_win_series(act).mean()) if len(act) else float("nan")
    lean_roi = float(lean_w * (100.0 / 110.0) - (1.0 - lean_w)) if np.isfinite(lean_w) else float("nan")
    act_roi = float(act_w * (100.0 / 110.0) - (1.0 - act_w)) if np.isfinite(act_w) else float("nan")
    print("\nAccuracy layers (spread model vs selection policy):")
    print(f"  1. Spread MAE:        {spread_mae:.2f}")
    print(f"  2. All-leans ATS:     {lean_w:.1%}  ({len(leans):,} games)  flat ROI {lean_roi:+.1%}")
    print(f"  3. Actionable ATS:    {act_w:.1%}  ({len(act):,} bets)   flat ROI {act_roi:+.1%}")


def compute_metrics(results_df: pd.DataFrame, season_col="simulated_season_window") -> pd.DataFrame:
    """Per-season + overall metric table for one backtest result frame."""
    if results_df is None or results_df.empty:
        return pd.DataFrame()
    df = results_df.copy()
    df["_spread_ae"] = (df["PRED_SPREAD"] - df["ACTUAL_MARGIN"]).abs()
    if "RAW_PRED_MARGIN" in df.columns:
        df["_raw_spread_ae"] = (df["RAW_PRED_MARGIN"] - df["ACTUAL_MARGIN"]).abs()
    if "PRED_TOTAL" in df and "ACTUAL_HOME" in df:
        df["_total_ae"] = (df["PRED_TOTAL"] - (df["ACTUAL_HOME"] + df["ACTUAL_AWAY"])).abs()
    else:
        df["_total_ae"] = np.nan
    home_win = (df["ACTUAL_MARGIN"] > 0).astype(int)
    df["_brier"] = (df.get("WIN_PROB", 0.5) - home_win) ** 2

    def _one(g):
        active = g[g["DIRECTION"] != "Pass"]
        if len(active):
            wins = (
                ((active["DIRECTION"] == "Home") & (active["ACTUAL_MARGIN"] + active["MARKET_SPREAD"] > 0))
                | ((active["DIRECTION"] == "Away") & (active["ACTUAL_MARGIN"] + active["MARKET_SPREAD"] < 0))
            )
            ats = wins.mean()
            roi = ats * (100.0 / 110.0) - (1 - ats)
            n_bets = int(len(active))
        else:
            ats = roi = np.nan
            n_bets = 0
        raw_mae = g["_raw_spread_ae"].mean() if "_raw_spread_ae" in g.columns else np.nan
        return pd.Series({
            "n_games": int(len(g)),
            "spread_mae": g["_spread_ae"].mean(),
            "raw_spread_mae": raw_mae,
            "total_mae": g["_total_ae"].mean(),
            "ats_pct": ats,
            "roi": roi,
            "n_bets": n_bets,
            "brier": g["_brier"].mean(),
        })

    rows = []
    for s, g in df.groupby(season_col):
        rec = {"season": s}
        rec.update(_one(g).to_dict())
        rows.append(rec)
    overall = {"season": "ALL"}
    overall.update(_one(df).to_dict())
    rows.append(overall)
    return pd.DataFrame(rows)


In [ ]:
# ── module: calibration_metrics ──────────────────────────────────────────────
"""Calibration quality metrics for probabilistic betting models."""
from __future__ import annotations

import numpy as np
import pandas as pd


def compute_brier(y_true, y_prob) -> float:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_prob, dtype=float)
    mask = np.isfinite(y) & np.isfinite(p)
    if mask.sum() == 0:
        return float("nan")
    return float(np.mean((p[mask] - y[mask]) ** 2))


def compute_ece(y_true, y_prob, n_bins: int = 10) -> float:
    """Expected Calibration Error (equal-width bins on predicted probability)."""
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_prob, dtype=float)
    mask = np.isfinite(y) & np.isfinite(p)
    y, p = y[mask], p[mask]
    if len(y) < n_bins:
        return float("nan")
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            m = (p >= lo) & (p <= hi)
        else:
            m = (p >= lo) & (p < hi)
        if not m.any():
            continue
        ece += m.mean() * abs(float(y[m].mean()) - float(p[m].mean()))
    return float(ece)


def reliability_bins(y_true, y_prob, n_bins: int = 10) -> pd.DataFrame:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_prob, dtype=float)
    mask = np.isfinite(y) & np.isfinite(p)
    y, p = y[mask], p[mask]
    if len(y) == 0:
        return pd.DataFrame()
    edges = np.linspace(0, 1, n_bins + 1)
    rows = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        if i == n_bins - 1:
            m = (p >= lo) & (p <= hi)
        else:
            m = (p >= lo) & (p < hi)
        if not m.any():
            continue
        rows.append({
            "bin_lo": lo,
            "bin_hi": hi,
            "pred_mean": float(p[m].mean()),
            "obs_rate": float(y[m].mean()),
            "n": int(m.sum()),
        })
    return pd.DataFrame(rows)


In [ ]:
# ── module: ml_calibration ───────────────────────────────────────────────────
"""Walk-forward calibration for MetaWin home win probability (prior season only)."""
from __future__ import annotations

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression



def _home_win_series(df: pd.DataFrame) -> pd.Series:
    if "ACTUAL_HOME" not in df.columns or "ACTUAL_AWAY" not in df.columns:
        return pd.Series(dtype=float)
    return (df["ACTUAL_HOME"].astype(float) > df["ACTUAL_AWAY"].astype(float)).astype(int)


def prior_year_ml_frame(prior_df: pd.DataFrame, season_col: str = "simulated_season_window") -> pd.DataFrame:
    """Last simulated season only (walk-forward calib slice)."""
    if prior_df is None or prior_df.empty:
        return pd.DataFrame()
    if season_col not in prior_df.columns:
        return prior_df
    seasons = sorted(prior_df[season_col].dropna().unique())
    if not seasons:
        return prior_df
    return prior_df[prior_df[season_col] == seasons[-1]].copy()


class WalkForwardMLCalibrator:
    """Map raw MetaWin P(home) → calibrated P(home) using prior-season outcomes."""

    def __init__(self, method: str | None = None):
        self.method = (method or ML_CALIB_METHOD or "platt").lower()
        self.model = None
        self._fitted = False
        self._fit_season: str | None = None

    def fit(
        self,
        prior_df: pd.DataFrame,
        *,
        prob_col: str = "WIN_PROB_RAW",
        scope: str = "prior_year",
        season_col: str = "simulated_season_window",
        min_samples: int = 80,
    ) -> "WalkForwardMLCalibrator":
        df = prior_year_ml_frame(prior_df, season_col=season_col) if scope == "prior_year" else prior_df
        if df is None or df.empty:
            self._fitted = False
            return self

        col = prob_col if prob_col in df.columns else "WIN_PROB"
        if col not in df.columns:
            self._fitted = False
            return self

        y = _home_win_series(df)
        p = pd.to_numeric(df[col], errors="coerce")
        mask = p.notna() & y.notna()
        p = p[mask].astype(float).values
        y = y[mask].astype(int).values
        if len(p) < min_samples or len(np.unique(y)) < 2:
            self._fitted = False
            return self

        if season_col in df.columns and df[season_col].notna().any():
            self._fit_season = str(df[season_col].dropna().iloc[-1])

        if self.method == "isotonic":
            iso = IsotonicRegression(out_of_bounds="clip")
            iso.fit(p, y)
            self.model = iso
        elif self.method in ("platt", "logistic"):
            clf = LogisticRegression(C=0.1, solver="lbfgs", max_iter=200)
            clf.fit(p.reshape(-1, 1), y)
            self.model = clf
        else:
            self.model = None
            self._fitted = False
            return self

        self._fitted = True
        return self

    def transform(self, p_home: float) -> float:
        if not self._fitted or self.model is None or p_home is None or not np.isfinite(p_home):
            return float(np.clip(p_home, 0.01, 0.99))
        p = float(np.clip(p_home, 0.01, 0.99))
        if isinstance(self.model, IsotonicRegression):
            out = float(self.model.predict([p])[0])
        else:
            out = float(self.model.predict_proba(np.array([[p]], dtype=float))[0, 1])
        return float(np.clip(out, 0.01, 0.99))

    def save(self, path: Path | str):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path: Path | str) -> "WalkForwardMLCalibrator":
        with open(path, "rb") as f:
            return pickle.load(f)


def default_ml_calibrator_path() -> Path:
    return STATE_DIR / "ml_calibrator.pkl"


In [ ]:
# ── module: game_features ────────────────────────────────────────────────────
"""Shared pre-game feature builder for training, simulation, and live prediction."""
from __future__ import annotations

from collections import defaultdict, deque

import numpy as np
import pandas as pd



def _compute_sos(history, ref_date=None) -> float:
    """Mean opponent net rating; optional exponential decay when USE_DECAYED_SOS."""
    if not history:
        return 0.0

    weighted_sum = 0.0
    weight_total = 0.0
    for entry in history:
        if len(entry) < 2:
            continue
        first, net = entry[0], float(entry[1])
        w = 1.0
        if USE_DECAYED_SOS and ref_date is not None and hasattr(first, "year"):
            try:
                days = max(0, (pd.Timestamp(ref_date) - pd.Timestamp(first)).days)
                w = float(np.exp(-SOS_DECAY_LAMBDA * days))
            except Exception:
                w = 1.0
        weighted_sum += net * w
        weight_total += w
    if weight_total <= 0:
        return float(np.mean([float(e[1]) for e in history if len(e) >= 2]))
    return float(weighted_sum / weight_total)


def build_game_features(
    *,
    game_id,
    gdate,
    home_team,
    away_team,
    home_starters,
    away_starters,
    elo_tracker,
    hier_engine,
    pace_tracker,
    odds_dict=None,
    team_xppp_tracker=None,
    team_form_tracker=None,
    rotation_tracker=None,
    lineup_elo_tracker=None,
    chemistry_tracker=None,
    team_elo_tracker=None,
    travel_tracker=None,
    epm_tracker=None,
    ref_tracker=None,
    shot_quality_tracker=None,
    hapm_tracker=None,
    last_date=None,
    team_game_dates=None,
    team_recent_net=None,
    opponent_history=None,
    team_home_margin=None,
    team_road_margin=None,
    team_games_played=None,
    team_rosters_seen=None,
    season_start_date=None,
    inactive_ids=None,
    crew_id=None,
    gs=None,
    elo_calibrator=None,
):
    """Build a single-game feature dict matching generate_features / run_simulation."""
    last_date = last_date or {}
    team_game_dates = team_game_dates or defaultdict(lambda: deque(maxlen=20))
    team_recent_net = team_recent_net or defaultdict(lambda: deque(maxlen=10))
    opponent_history = opponent_history or defaultdict(lambda: deque(maxlen=SOS_WINDOW))
    team_home_margin = team_home_margin or defaultdict(lambda: deque(maxlen=20))
    team_road_margin = team_road_margin or defaultdict(lambda: deque(maxlen=20))
    team_games_played = team_games_played or defaultdict(int)
    team_rosters_seen = team_rosters_seen or defaultdict(set)
    inactive_ids = set(str(x) for x in (inactive_ids or []))

    prev_hint = last_date.get(home_team) or last_date.get(away_team)
    season_yr = None
    if is_valid_timestamp(season_start_date):
        season_yr = season_start_date.year
    coerced = coerce_game_date(
        gdate, game_id=game_id, prev_date=prev_hint, season_start_year=season_yr,
    )
    if coerced is not None:
        gdate = coerced
    elif not is_valid_timestamp(gdate):
        gdate = pd.Timestamp.now().normalize()
    else:
        gdate = pd.Timestamp(gdate).normalize()

    current_season = gdate.year + (1 if gdate.month >= 9 else 0)

    h_weights = rotation_tracker.expected_weights(home_team, home_starters) if rotation_tracker else []
    a_weights = rotation_tracker.expected_weights(away_team, away_starters) if rotation_tracker else []

    if inactive_ids:
        h_weights = filter_available_weights(h_weights, inactive_ids)
        a_weights = filter_available_weights(a_weights, inactive_ids)

    home_lineup = _top_lineup(h_weights) or home_starters
    away_lineup = _top_lineup(a_weights) or away_starters

    if hasattr(elo_tracker, "apply_inactivity_decay"):
        elo_tracker.apply_inactivity_decay(home_lineup + away_lineup, gdate.date())

    def _prev_ts(team):
        v = last_date.get(team)
        if is_valid_timestamp(v):
            return pd.Timestamp(v)
        return gdate - pd.Timedelta(days=7)

    h_rest = min((gdate - _prev_ts(home_team)).days, 14)
    a_rest = min((gdate - _prev_ts(away_team)).days, 14)

    sq_h = sq_a = None
    if shot_quality_tracker is not None:
        sq_h = shot_quality_tracker.get(home_team, current_season, gdate)
        sq_a = shot_quality_tracker.get(away_team, current_season, gdate)
    opp_rim_for_home_d = float(sq_a["rim_rate"]) if sq_a else 0.30
    opp_peri_for_home_d = float(sq_a["three_rate"]) if sq_a else 0.38
    opp_rim_for_away_d = float(sq_h["rim_rate"]) if sq_h else 0.30
    opp_peri_for_away_d = float(sq_h["three_rate"]) if sq_h else 0.38

    if h_weights:
        ho_off, ho_def, ho_exp = elo_tracker.weighted_lineup_stats(
            h_weights, opp_rim_rate=opp_rim_for_home_d, opp_three_rate=opp_peri_for_home_d,
        )
        h_o_rd, h_d_rd = elo_tracker.weighted_lineup_uncertainty(h_weights)
    else:
        ho_off, ho_def, ho_exp = elo_tracker.lineup_stats(
            home_starters, opp_rim_rate=opp_rim_for_home_d, opp_three_rate=opp_peri_for_home_d,
        )
        h_o_rd, h_d_rd = elo_tracker.lineup_uncertainty(home_starters)
    if a_weights:
        ao_off, ao_def, ao_exp = elo_tracker.weighted_lineup_stats(
            a_weights, opp_rim_rate=opp_rim_for_away_d, opp_three_rate=opp_peri_for_away_d,
        )
        a_o_rd, a_d_rd = elo_tracker.weighted_lineup_uncertainty(a_weights)
    else:
        ao_off, ao_def, ao_exp = elo_tracker.lineup_stats(
            away_starters, opp_rim_rate=opp_rim_for_away_d, opp_three_rate=opp_peri_for_away_d,
        )
        a_o_rd, a_d_rd = elo_tracker.lineup_uncertainty(away_starters)

    h_rim_def, h_peri_def, _ = elo_tracker._lineup_rim_peri(home_lineup, h_weights or None)
    a_rim_def, a_peri_def, _ = elo_tracker._lineup_rim_peri(away_lineup, a_weights or None)

    if epm_tracker is not None:
        # Task 032: `as_of=gdate` -- a future EPM snapshot must never be
        # joinable to this (earlier) game's T-60 features.
        ho_off = epm_tracker.blend_lineup_off(home_lineup, ho_off, as_of=gdate)
        ao_off = epm_tracker.blend_lineup_off(away_lineup, ao_off, as_of=gdate)

    h_hoff, h_hdef = hier_engine.lineup_rating(home_lineup)
    a_hoff, a_hdef = hier_engine.lineup_rating(away_lineup)

    if team_xppp_tracker is not None:
        h_roll_off, h_roll_def = team_xppp_tracker.get_rolling_xppp(home_team, current_season, gdate)
        a_roll_off, a_roll_def = team_xppp_tracker.get_rolling_xppp(away_team, current_season, gdate)
    else:
        h_roll_off = h_roll_def = a_roll_off = a_roll_def = DEFAULT_LEAGUE_XPPP

    pred_poss = pace_tracker.get_expected_pace(home_team, away_team)
    h_pace = pace_tracker.get_team_pace(home_team) if hasattr(pace_tracker, "get_team_pace") else pred_poss
    a_pace = pace_tracker.get_team_pace(away_team) if hasattr(pace_tracker, "get_team_pace") else pred_poss
    pace_diff = h_pace - a_pace

    h_games_last7, h_3in4, h_4in6 = schedule_density_extended(team_game_dates[home_team], gdate)
    a_games_last7, a_3in4, a_4in6 = schedule_density_extended(team_game_dates[away_team], gdate)

    h_new_starters = len(set(home_starters) - team_rosters_seen[home_team]) / max(len(home_starters), 1)
    a_new_starters = len(set(away_starters) - team_rosters_seen[away_team]) / max(len(away_starters), 1)

    h_missing_rotation = missing_rotation_share(h_weights, inactive_ids)
    a_missing_rotation = missing_rotation_share(a_weights, inactive_ids)
    h_star_out = star_out_flag(h_weights, inactive_ids)
    a_star_out = star_out_flag(a_weights, inactive_ids)

    days_since_start = (gdate - season_start_date).days if season_start_date is not None else 50
    h_season_phase = team_games_played[home_team] / 82.0
    a_season_phase = team_games_played[away_team] / 82.0

    go = get_game_odds(gdate, home_team, odds_dict) if odds_dict else {}
    market_spread = go.get("spread", np.nan)
    market_ml = go.get("ml", np.nan)
    market_fair_win_prob = fair_home_win_prob(market_ml) if pd.notna(market_ml) else 0.5
    market_total = go.get("total", np.nan)
    closing_spread = go.get("closing_spread", market_spread)
    market_total_minus_league = float(market_total) - LEAGUE_AVG_TOTAL if pd.notna(market_total) else 0.0
    spread_move = go.get("spread_move", 0.0) if pd.notna(go.get("spread_move", np.nan)) else 0.0
    public_home_pct = go.get("public_home_pct", 0.0) if pd.notna(go.get("public_home_pct", np.nan)) else 0.0
    micro = market_microstructure_features(spread_move, public_home_pct, market_spread)

    h_recent = np.mean(team_recent_net[home_team]) if team_recent_net[home_team] else 0.0
    a_recent = np.mean(team_recent_net[away_team]) if team_recent_net[away_team] else 0.0

    h_sos = _compute_sos(opponent_history[home_team], gdate)
    a_sos = _compute_sos(opponent_history[away_team], gdate)

    h_home_edge = (np.mean(team_home_margin[home_team]) - np.mean(team_road_margin[home_team])) \
        if team_home_margin[home_team] and team_road_margin[home_team] else 0.0
    a_road_edge = (np.mean(team_road_margin[away_team]) - np.mean(team_home_margin[away_team])) \
        if team_home_margin[away_team] and team_road_margin[away_team] else 0.0

    if team_form_tracker is not None:
        h_form = team_form_tracker.get(home_team, current_season, gdate)
        a_form = team_form_tracker.get(away_team, current_season, gdate)
        form_feats = form_feature_dict(h_form, a_form)
        if hasattr(team_form_tracker, "multi_window_feature_dict"):
            form_feats.update(
                team_form_tracker.multi_window_feature_dict(
                    home_team, away_team, current_season, gdate,
                )
            )
    else:
        h_form, a_form = dict(_DEFAULT_FORM), dict(_DEFAULT_FORM)
        form_feats = form_feature_dict(h_form, a_form)

    elo_margin, hier_margin = engine_implied_margins(
        elo_tracker, hier_engine, ho_off, ho_def, ao_off, ao_def,
        home_lineup, away_lineup, pred_poss)

    h_luck, h_defev, h_tov = elo_tracker.lineup_rolling_rates(home_lineup, h_weights or None)
    a_luck, a_defev, a_tov = elo_tracker.lineup_rolling_rates(away_lineup, a_weights or None)
    elo_luck_adj_net = (h_luck - a_luck) * 100.0
    elo_def_event_rate = h_defev - a_defev
    elo_tov_rate = h_tov - a_tov
    elo_matchup_asym = (ho_off - ao_def) - (ao_off - ho_def)
    mean_unc = (h_o_rd + h_d_rd + a_o_rd + a_d_rd) / 4.0
    elo_consistency = 1.0 / (mean_unc + 1e-6)
    elo_margin_z = elo_margin / max(pred_poss, 1.0)

    feat = {
        "GAME_ID": game_id,
        "game_date": gdate,
        "home_team": home_team,
        "away_team": away_team,
        "elo_margin": elo_margin,
        "hier_margin": hier_margin,
        "h_elo_off": ho_off, "h_elo_def": ho_def,
        "a_elo_off": ao_off, "a_elo_def": ao_def,
        "h_elo_def_rim": h_rim_def, "h_elo_def_peri": h_peri_def,
        "a_elo_def_rim": a_rim_def, "a_elo_def_peri": a_peri_def,
        "rim_def_diff": h_rim_def - a_rim_def,
        "peri_def_diff": h_peri_def - a_peri_def,
        "elo_diff_off": ho_off - ao_off, "elo_diff_def": ho_def - ao_def,
        "elo_net": (ho_off - ao_def) - (ao_off - ho_def),
        "elo_matchup_asym": elo_matchup_asym,
        "elo_luck_adj_net": elo_luck_adj_net,
        "elo_def_event_rate": elo_def_event_rate,
        "elo_tov_rate": elo_tov_rate,
        "elo_consistency": elo_consistency,
        "elo_margin_z": elo_margin_z,
        "h_hier_off": h_hoff, "h_hier_def": h_hdef,
        "a_hier_off": a_hoff, "a_hier_def": a_hdef,
        "hier_net": (h_hoff - a_hdef) - (a_hoff - h_hdef),
        "exp_poss": pred_poss,
        "h_rest": h_rest, "a_rest": a_rest,
        "h_b2b": int(h_rest <= 1), "a_b2b": int(a_rest <= 1),
        "h_rest_0": rest_bucket_flags(h_rest)["rest_0"],
        "h_rest_1": rest_bucket_flags(h_rest)["rest_1"],
        "h_rest_2": rest_bucket_flags(h_rest)["rest_2"],
        "h_rest_3plus": rest_bucket_flags(h_rest)["rest_3plus"],
        "a_rest_0": rest_bucket_flags(a_rest)["rest_0"],
        "a_rest_1": rest_bucket_flags(a_rest)["rest_1"],
        "a_rest_2": rest_bucket_flags(a_rest)["rest_2"],
        "a_rest_3plus": rest_bucket_flags(a_rest)["rest_3plus"],
        "h_games_last7": h_games_last7, "a_games_last7": a_games_last7,
        "games_last7_diff": h_games_last7 - a_games_last7,
        "h_3in4": h_3in4, "a_3in4": a_3in4,
        "h_4in6": h_4in6, "a_4in6": a_4in6,
        "is_altitude": int(home_team in ALTITUDE_TEAMS),
        "h_experience": ho_exp, "a_experience": ao_exp,
        "days_since_season_start": days_since_start,
        "h_season_phase": h_season_phase, "a_season_phase": a_season_phase,
        "h_new_starters": h_new_starters, "a_new_starters": a_new_starters,
        "h_missing_rotation": h_missing_rotation, "a_missing_rotation": a_missing_rotation,
        "h_star_out": h_star_out, "a_star_out": a_star_out,
        "h_rating_uncertainty": h_o_rd + h_d_rd,
        "a_rating_uncertainty": a_o_rd + a_d_rd,
        "uncertainty_diff": (h_o_rd + h_d_rd) - (a_o_rd + a_d_rd),
        "h_roll_off_xppp": h_roll_off, "h_roll_def_xppp": h_roll_def,
        "a_roll_off_xppp": a_roll_off, "a_roll_def_xppp": a_roll_def,
        "roll_net_xppp": (h_roll_off - a_roll_def) - (a_roll_off - h_roll_def),
        "market_spread": market_spread, "market_ml": market_ml,
        "market_fair_win_prob": market_fair_win_prob,
        "closing_spread": closing_spread,
        "market_total": market_total if pd.notna(market_total) else 0.0,
        "market_total_minus_league": market_total_minus_league,
        "spread_move": spread_move, "public_home_pct": public_home_pct,
        "h_recent_net": h_recent, "a_recent_net": a_recent, "recent_diff": h_recent - a_recent,
        "pace_diff": pace_diff, "pace_abs_diff": abs(pace_diff),
        "pace_interaction": pace_diff * ((ho_off - ao_def) - (ao_off - ho_def)),
        "h_pace": float(h_pace), "a_pace": float(a_pace),
        "h_sos": h_sos, "a_sos": a_sos, "sos_diff": h_sos - a_sos,
        "h_home_edge": h_home_edge, "a_road_edge": a_road_edge,
        "hca_net": h_home_edge - a_road_edge,
        **form_feats,
        **micro,
    }

    if lineup_elo_tracker is not None:
        feat.update(lineup_elo_tracker.feature_dict(home_lineup, away_lineup, elo_tracker))
    if chemistry_tracker is not None:
        chem_feats = chemistry_tracker.feature_dict(home_lineup, away_lineup, elo_tracker)
        feat.update(chem_feats)
        feat["usage_conflict"] = chemistry_tracker.usage_conflict(h_weights) + chemistry_tracker.usage_conflict(a_weights)
        lineup5_net = feat.get("lineup5_net", 0.0)
        feat["h_lineup_composite"] = composite_lineup_rating(
            player_off_delta=ho_off - ao_def,
            player_def_delta=ho_def - ao_off,
            lineup5_net=lineup5_net,
            chem_duo_net=chem_feats.get("h_chem_duo_net", 0.0),
            chem_trio_net=chem_feats.get("h_chem_trio_net", 0.0),
        )
        feat["a_lineup_composite"] = composite_lineup_rating(
            player_off_delta=ao_off - ho_def,
            player_def_delta=ao_def - ho_off,
            lineup5_net=-lineup5_net,
            chem_duo_net=chem_feats.get("a_chem_duo_net", 0.0),
            chem_trio_net=chem_feats.get("a_chem_trio_net", 0.0),
        )
        feat["lineup_composite_diff"] = feat["h_lineup_composite"] - feat["a_lineup_composite"]
    if shot_quality_tracker is not None:
        feat.update(shot_quality_tracker.feature_dict(home_team, away_team, current_season, gdate))
    if hapm_tracker is not None and getattr(hapm_tracker, "fitted", False):
        feat.update(hapm_tracker.feature_dict(home_lineup, away_lineup, game_id=game_id))
    else:
        feat.setdefault("hapm_net_diff", 0.0)
    if team_elo_tracker is not None:
        feat.update(team_elo_tracker.feature_dict(home_team, away_team))
    if travel_tracker is not None:
        feat.update(travel_tracker.matchup_features(home_team, away_team, gdate))
        tm = feat
        feat.update(fatigue_features(
            feat.get("h_games_last7", 0), feat.get("a_games_last7", 0),
            tm.get("h_travel_miles_7d", 0), tm.get("a_travel_miles_7d", 0),
            tm.get("h_tz_shift", 0), tm.get("a_tz_shift", 0),
        ))
    else:
        feat.update(fatigue_features(
            feat.get("h_games_last7", 0), feat.get("a_games_last7", 0), 0, 0, 0, 0,
        ))
    # Travel × rest interactions (nonlinear fatigue)
    h_travel = float(feat.get("h_travel_miles_7d", 0) or 0)
    a_travel = float(feat.get("a_travel_miles_7d", 0) or 0)
    feat["h_travel_rest_interaction"] = h_travel * float(feat.get("h_rest_0", 0) + 0.5 * feat.get("h_rest_1", 0))
    feat["a_travel_rest_interaction"] = a_travel * float(feat.get("a_rest_0", 0) + 0.5 * feat.get("a_rest_1", 0))
    feat["travel_rest_interaction_diff"] = (
        feat["h_travel_rest_interaction"] - feat["a_travel_rest_interaction"]
    )
    if epm_tracker is not None:
        # Task 032: `as_of=gdate` enforces the same future-snapshot guard
        # for the emitted `epm_*` feature columns.
        feat.update(epm_tracker.feature_dict(home_lineup, away_lineup, ho_off, ao_off, as_of=gdate))
        # Quantitative availability impact from EPM priors + star-out
        feat.update(expected_availability_impact(
            home_lineup, away_lineup, epm_tracker,
            h_star_out=feat.get("h_star_out", 0),
            a_star_out=feat.get("a_star_out", 0),
            h_missing=feat.get("h_missing_rotation", 0),
            a_missing=feat.get("a_missing_rotation", 0),
            as_of=gdate,
        ))
    else:
        feat.update({
            "epm_prior_diff": 0.0, "epm_blend_off_diff": 0.0,
            "epm_missing_frac_home": 1.0, "epm_missing_frac_away": 1.0,
            "h_expected_off_drop": 0.0, "a_expected_off_drop": 0.0,
            "h_expected_def_drop": 0.0, "a_expected_def_drop": 0.0,
            "h_expected_pace_delta": 0.0, "a_expected_pace_delta": 0.0,
            "expected_off_drop_diff": 0.0, "expected_def_drop_diff": 0.0,
            "expected_pace_delta_diff": 0.0,
        })
    if ref_tracker is not None:
        feat.update(ref_tracker.features(crew_id))
    else:
        feat.update({"ref_pace_bias": 0.0, "ref_foul_bias": 0.0})

    return augment_elo_features(feat, elo_calibrator)


In [ ]:
# ── module: features ─────────────────────────────────────────────────────────
"""Feature generation for training and inference."""
from collections import defaultdict, deque

import numpy as np
import pandas as pd
from tqdm.auto import tqdm



def generate_features(
    stints_df: pd.DataFrame,
    hier_engine,
    elo_tracker,
    pace_tracker,
    odds_dict: dict = None,
    update_engines: bool = True,
    team_xppp_tracker=None,
    team_form_tracker=None,
    rotation_tracker=None,
    sos_window: int = None,
    lineup_elo_tracker=None,
    chemistry_tracker=None,
    team_elo_tracker=None,
    travel_tracker=None,
    epm_tracker=None,
    ref_tracker=None,
    shot_quality_tracker=None,
    hapm_tracker=None,
    elo_calibrator=None,
):
    """
    Generate a feature DataFrame for all games in stints_df, ensuring NO data leakage.

    Uses build_game_features() for train/serve parity, then applies post-game updates.
    """
    df = stints_df.copy()
    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    # Task 015: enforce chronological iteration explicitly by decision
    # timestamp (game_date) then GAME_ID, regardless of input row order, so
    # a shuffled `stints_df` produces identical ordered predictions/tracker
    # state as an already-sorted one.
    df = df.sort_values(["game_date", "GAME_ID", "stint_id"]).reset_index(drop=True)
    _per_game_dates = df.drop_duplicates("GAME_ID")["game_date"]
    assert _per_game_dates.is_monotonic_increasing, (
        "generate_features: per-game decision timestamps are not monotonic "
        "nondecreasing after sort — chronological iteration is violated"
    )

    rows = []
    last_game_date = {}
    team_game_dates = defaultdict(lambda: deque(maxlen=20))
    season_year = None
    season_start_date = None
    team_games_played = defaultdict(int)
    team_rosters_seen = defaultdict(set)

    RECENT_WINDOW = 10
    team_recent_net = defaultdict(lambda: deque(maxlen=RECENT_WINDOW))
    team_home_margin = defaultdict(lambda: deque(maxlen=20))
    team_road_margin = defaultdict(lambda: deque(maxlen=20))

    if sos_window is None:
        sos_window = SOS_WINDOW
    opponent_history = defaultdict(lambda: deque(maxlen=sos_window))

    lineup_cache = {}
    season_player_ids = set()
    total_games = len(df["GAME_ID"].unique()) if update_engines else 1
    game_stats_map = precompute_game_team_stats(df)

    last_valid_date = None
    skipped_games = 0

    for game_id, group in tqdm(df.groupby("GAME_ID", sort=False), desc="Building features"):
        raw_gdate = group["game_date"].iloc[0]
        season_yr = season_start_date.year if is_valid_timestamp(season_start_date) else None
        gdate = coerce_game_date(
            raw_gdate, game_id=game_id, prev_date=last_valid_date, season_start_year=season_yr,
        )
        if gdate is None:
            skipped_games += 1
            continue
        last_valid_date = gdate
        home_raw = group["home_team"].iloc[0]
        away_raw = group["away_team"].iloc[0]
        home = TEAM_MAP.get(home_raw, home_raw)
        away = TEAM_MAP.get(away_raw, away_raw)

        if isinstance(gdate, pd.Timestamp):
            current_season = gdate.year + (1 if gdate.month >= 9 else 0)
        else:
            current_season = 2025

        if update_engines:
            if season_year is None:
                season_year = gdate.year
            elif gdate.month > 8 and gdate.year > season_year:
                returning_ids = set(season_player_ids) if season_player_ids else None
                hier_engine.offseason_revert()
                elo_tracker.offseason_revert(returning_player_ids=returning_ids)
                season_player_ids = set()
                season_year = gdate.year
                season_start_date = gdate
                team_games_played.clear()
                team_rosters_seen.clear()
                lineup_cache.clear()
                team_recent_net.clear()
                opponent_history.clear()
                if shot_quality_tracker is not None:
                    for t in list(team_rosters_seen.keys()):
                        shot_quality_tracker.on_season_boundary(t, current_season - 1)

            if season_start_date is None:
                season_start_date = gdate

        # Task 031: `home_starters`/`away_starters` used for T-60 feature
        # generation below must never be derived from *this* game's own
        # actual/current PBP lineup (Rule 4). `lineup_cache` holds only the
        # actual roster observed in a *previously completed* game (updated
        # post-game in `update_trackers_after_game`); if a team has no prior
        # game in this cache yet (first game of a season), the pre-game
        # roster is genuinely unknown and must be left empty (Rule 5) rather
        # than fabricated from this game's own PBP. Trackers downstream
        # (elo_tracker.lineup_stats, rotation_tracker.expected_weights, ...)
        # already fall back to league-average/empty-weight behavior for an
        # empty roster.
        home_starters = lineup_cache.get(home, [])
        away_starters = lineup_cache.get(away, [])
        # `actual_home_starters`/`actual_away_starters` are this game's real
        # PBP-observed lineup. They are POSTGAME-only information: used to
        # update rosters-seen bookkeeping and `lineup_cache` for *future*
        # games, never fed into this game's own T-60 feature row.
        actual_home_starters = _parse_player_string(group.iloc[0].get("HOME_players", ""))
        actual_away_starters = _parse_player_string(group.iloc[0].get("AWAY_players", ""))

        # Task 014: `act_h`/`act_a` come from `precompute_game_team_stats`,
        # which prefers the canonical, independently-verified final score
        # (Task 006/007) over stint-summed points whenever it is available.
        # This is the single source of truth for `actual_home`/`actual_away`
        # labels below — never re-sum home_pts/away_pts from filtered stints
        # here.
        gs = game_stats_map.get(game_id, {})
        act_h = float(gs.get("act_h", 0.0))
        act_a = float(gs.get("act_a", 0.0))
        tot_poss = float(gs.get("tot_poss", 0.0)) or 100.0
        home_tot_poss = float(gs.get("home_poss", 0.0)) or tot_poss / 2
        away_tot_poss = float(gs.get("away_poss", 0.0)) or tot_poss / 2
        home_tot_xpts = float(gs.get("home_xpts", 0.0))
        away_tot_xpts = float(gs.get("away_xpts", 0.0))
        home_xppp_game = home_tot_xpts / home_tot_poss if home_tot_poss > 0 else DEFAULT_LEAGUE_XPPP
        away_xppp_game = away_tot_xpts / away_tot_poss if away_tot_poss > 0 else DEFAULT_LEAGUE_XPPP

        row_feat = build_game_features(
            game_id=game_id,
            gdate=gdate,
            home_team=home,
            away_team=away,
            home_starters=home_starters,
            away_starters=away_starters,
            elo_tracker=elo_tracker,
            hier_engine=hier_engine,
            pace_tracker=pace_tracker,
            odds_dict=odds_dict,
            team_xppp_tracker=team_xppp_tracker,
            team_form_tracker=team_form_tracker,
            rotation_tracker=rotation_tracker,
            lineup_elo_tracker=lineup_elo_tracker,
            chemistry_tracker=chemistry_tracker,
            team_elo_tracker=team_elo_tracker,
            travel_tracker=travel_tracker,
            epm_tracker=epm_tracker,
            ref_tracker=ref_tracker,
            shot_quality_tracker=shot_quality_tracker,
            hapm_tracker=hapm_tracker,
            last_date=last_game_date,
            team_game_dates=team_game_dates,
            team_recent_net=team_recent_net,
            opponent_history=opponent_history,
            team_home_margin=team_home_margin,
            team_road_margin=team_road_margin,
            team_games_played=team_games_played,
            team_rosters_seen=team_rosters_seen,
            season_start_date=season_start_date,
            gs=gs,
            elo_calibrator=elo_calibrator,
        )

        raw_margin = act_h - act_a
        row_feat.update({
            "actual_home": act_h,
            "actual_away": act_a,
            "actual_margin": raw_margin,
            "actual_margin_capped": np.clip(raw_margin, -20.0, 20.0),
            "actual_total": act_h + act_a,
            "home_win": int(act_h > act_a),
        })
        rows.append(row_feat)

        if update_engines:
            seas_prog = len(rows) / max(total_games, 1)
            market_spread = row_feat.get("market_spread", np.nan)
            update_trackers_after_game(
                group=group,
                gs=gs,
                game_id=game_id,
                gdate=gdate,
                home=home,
                away=away,
                home_starters=actual_home_starters,
                away_starters=actual_away_starters,
                act_h=act_h,
                act_a=act_a,
                home_tot_poss=home_tot_poss,
                away_tot_poss=away_tot_poss,
                home_xppp_game=home_xppp_game,
                away_xppp_game=away_xppp_game,
                current_season=current_season,
                market_spread=market_spread,
                elo_tracker=elo_tracker,
                hier_engine=hier_engine,
                pace_tracker=pace_tracker,
                team_xppp_tracker=team_xppp_tracker,
                team_form_tracker=team_form_tracker,
                rotation_tracker=rotation_tracker,
                lineup_elo_tracker=lineup_elo_tracker,
                chemistry_tracker=chemistry_tracker,
                team_elo_tracker=team_elo_tracker,
                travel_tracker=travel_tracker,
                ref_tracker=ref_tracker,
                shot_quality_tracker=shot_quality_tracker,
                hapm_tracker=hapm_tracker,
                last_game_date=last_game_date,
                team_game_dates=team_game_dates,
                team_recent_net=team_recent_net,
                opponent_history=opponent_history,
                team_home_margin=team_home_margin,
                team_road_margin=team_road_margin,
                team_games_played=team_games_played,
                team_rosters_seen=team_rosters_seen,
                lineup_cache=lineup_cache,
                season_player_ids=season_player_ids,
                seas_prog=seas_prog,
                ho_off=row_feat.get("h_elo_off"),
                ho_def=row_feat.get("h_elo_def"),
                ao_off=row_feat.get("a_elo_off"),
                ao_def=row_feat.get("a_elo_def"),
            )

    if skipped_games:
        print(f"  ℹ️ features: skipped {skipped_games} games with unrecoverable dates")

    return pd.DataFrame(rows)


In [ ]:
# ── module: tuning ───────────────────────────────────────────────────────────
"""Hyperparameter tuning."""
from __future__ import annotations

import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error


# Original Optuna search bounds (AdaptiveParameterBounds shrinks around running best).
ELO_PARAM_BOUNDS = {
    "k_off": (0.05, 1.5),
    "k_def": (0.05, 1.5),
    "elo_scaling": (200, 1200),
    "home_boost": (0.0001, 0.007),
    "offseason_reversion": (0.1, 0.30),
    "usage_floor": (0.20, 0.32),
    "assist_split": (0.60, 0.95),
    "k_mult_half_life": (10.0, 35.0),
    "rd_floor": (25.0, 50.0),
    "garbage_time_weight": (0.0, 1.0),
    "clutch_boost": (1.0, 1.6),
    "tov_penalty": (0.7, 1.0),
    "foul_draw_boost": (1.0, 1.3),
    "variance_dampen": (0.5, 1.0),
    "xppp_actual_blend": (0.0, 0.2),
    "k_def_events": (0.05, 0.35),
    "tov_rate_threshold": (0.10, 0.25),
    "three_pa_rate_threshold": (0.35, 0.55),
}

HIER_PARAM_BOUNDS = {
    "w1": (0.1, 1.0),
    "w2": (0.1, 0.5),
    "w3": (0.1, 0.5),
    "w5": (0.1, 1.0),
    "k_off": (0.05, 0.9),
    "k_def": (0.01, 0.90),
    "league_rtg": (109.5, 114.0),
}


def _blend_mae(points_mae: float, xppp_mae: float) -> float:
    return TUNING_XPPP_WEIGHT * xppp_mae + TUNING_POINTS_WEIGHT * points_mae


def _params_at_bounds(
    trial_params: dict,
    bounds_spec: dict,
    edge_frac: float = TUNING_BOUND_EDGE_FRACTION,
) -> list[str]:
    """Return param names whose values sit within edge_frac of original bounds."""
    at_edge = []
    for name, spec in bounds_spec.items():
        if name not in trial_params:
            continue
        val = trial_params[name]
        if isinstance(spec[0], int):
            lo, hi = float(spec[0]), float(spec[1])
        else:
            lo, hi = spec
        span = hi - lo
        if span <= 0:
            continue
        if val <= lo + edge_frac * span or val >= hi - edge_frac * span:
            at_edge.append(name)
    return at_edge


def _suggest_float_param(trial, bounds, name, lo, hi, log=False):
    if bounds is not None:
        lo, hi = bounds.suggest_bounds_float(name, lo, hi, log=log)
    return trial.suggest_float(name, lo, hi, log=log)


def _suggest_int_param(trial, bounds, name, lo, hi):
    if bounds is not None:
        lo, hi = bounds.suggest_bounds_int(name, lo, hi)
    return trial.suggest_int(name, lo, hi)


def _ensure_season_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "game_date" in df.columns:
        df = df.sort_values(["game_date", "GAME_ID", "stint_id"]).reset_index(drop=True)
    if "season" not in df.columns and "game_date" in df.columns:
        dt = pd.to_datetime(df["game_date"])
        df["season"] = dt.dt.year + (dt.dt.month >= 9).astype(int)
    return df


def _prepare_stint_records(df: pd.DataFrame) -> list[dict]:
    """Pre-parse player strings and numeric fields once per tuning objective."""
    records = []
    cols = df.columns
    has_season = "season" in cols
    for row in df.itertuples(index=False):
        poss = float(getattr(row, "possessions", 1) or 1)
        if not (np.isfinite(poss) and poss >= 1):
            continue
        hp = _parse_player_string(row.HOME_players)
        ap = _parse_player_string(row.AWAY_players)
        rec = {
            "hp": hp,
            "ap": ap,
            "poss": poss,
            "home_xpts": float(getattr(row, "home_xpts", 0) or 0),
            "away_xpts": float(getattr(row, "away_xpts", 0) or 0),
            "home_pts": float(getattr(row, "home_pts", 0) or 0),
            "away_pts": float(getattr(row, "away_pts", 0) or 0),
            "home_usage": getattr(row, "home_usage", {}),
            "away_usage": getattr(row, "away_usage", {}),
            "period": int(getattr(row, "PERIOD", 1) or 1),
            "start_A": float(getattr(row, "HOME_SCORE_START", 0) or 0),
            "start_B": float(getattr(row, "AWAY_SCORE_START", 0) or 0),
            "end_A": float(getattr(row, "HOME_SCORE_END", 0) or 0),
            "end_B": float(getattr(row, "AWAY_SCORE_END", 0) or 0),
            "stint_ctx": build_stint_context(row),
        }
        if has_season:
            rec["season"] = getattr(row, "season")
        if hasattr(row, "GAME_ID"):
            rec["game_id"] = getattr(row, "GAME_ID")
        records.append(rec)
    return records


def _group_records_by_season(records: list[dict]) -> dict:
    grouped = {}
    for rec in records:
        grouped.setdefault(rec["season"], []).append(rec)
    return grouped


def _elo_warmup(tracker, records: list[dict]) -> None:
    for rec in records:
        tracker.process_stint(
            ids_A=rec["hp"], ids_B=rec["ap"], poss=rec["poss"],
            xpts_A=rec["home_xpts"], xpts_B=rec["away_xpts"],
            usage_A=rec["home_usage"], usage_B=rec["away_usage"],
            period=rec["period"],
            start_A=rec["start_A"], start_B=rec["start_B"],
            end_A=rec["end_A"], end_B=rec["end_B"],
            season_progress=0.5,
            stint_ctx=rec.get("stint_ctx"),
        )


def _game_margin_from_tracker(tracker, hp, ap, poss, league_xppp, home_boost, elo_scaling):
    ho_off, ho_def, _ = tracker.lineup_stats(hp)
    ao_off, ao_def, _ = tracker.lineup_stats(ap)
    pred_ppp_h = league_xppp + home_boost + (ho_off - ao_def) / elo_scaling
    pred_ppp_a = league_xppp - home_boost + (ao_off - ho_def) / elo_scaling
    return (pred_ppp_h - pred_ppp_a) * poss


def _elo_eval_fold(
    tracker, records: list[dict], league_xppp: float,
    home_boost: float, elo_scaling: int,
) -> tuple[float, float]:
    """Return (blend_mae, ats_miss_rate) for validation records."""
    yt, yp, xppp_yt, xppp_yp = [], [], [], []
    by_game: dict = {}
    for rec in records:
        gid = rec.get("game_id", 0)
        by_game.setdefault(gid, []).append(rec)

    ats_miss = 0.5
    game_margins_pred, game_margins_act = [], []
    for recs in by_game.values():
        hp, ap = recs[0]["hp"], recs[0]["ap"]
        poss = sum(r["poss"] for r in recs)
        act_margin = sum(r["home_pts"] for r in recs) - sum(r["away_pts"] for r in recs)
        pred_margin = _game_margin_from_tracker(
            tracker, hp, ap, poss, league_xppp, home_boost, elo_scaling,
        )
        game_margins_pred.append(pred_margin)
        game_margins_act.append(act_margin)

        for rec in recs:
            ho_off, ho_def, _ = tracker.lineup_stats(rec["hp"])
            ao_off, ao_def, _ = tracker.lineup_stats(rec["ap"])
            pred_ppp_h = league_xppp + home_boost + (ho_off - ao_def) / elo_scaling
            pred_ppp_a = league_xppp - home_boost + (ao_off - ho_def) / elo_scaling
            tracker.process_stint(
                ids_A=rec["hp"], ids_B=rec["ap"], poss=rec["poss"],
                xpts_A=rec["home_xpts"], xpts_B=rec["away_xpts"],
                usage_A=rec["home_usage"], usage_B=rec["away_usage"],
                period=rec["period"],
                start_A=rec["start_A"], start_B=rec["start_B"],
                end_A=rec["end_A"], end_B=rec["end_B"],
                season_progress=0.5,
                stint_ctx=rec.get("stint_ctx"),
            )
            poss_i = rec["poss"]
            yt.extend([rec["home_pts"], rec["away_pts"]])
            yp.extend([pred_ppp_h * poss_i, pred_ppp_a * poss_i])
            xppp_yt.extend([rec["home_xpts"], rec["away_xpts"]])
            xppp_yp.extend([pred_ppp_h * poss_i, pred_ppp_a * poss_i])

    if game_margins_pred:
        hits = sum(
            1 for p, a in zip(game_margins_pred, game_margins_act)
            if (p > 0) == (a > 0) or (p == 0 and a == 0)
        )
        ats_miss = 1.0 - hits / len(game_margins_pred)

    mae = _fold_mae_from_predictions(yt, yp, xppp_yt, xppp_yp)
    return mae, ats_miss


def _fold_mae_from_predictions(
    yt: list[float], yp: list[float],
    xppp_yt: list[float], xppp_yp: list[float],
) -> float:
    arr_t, arr_p = np.array(yt), np.array(yp)
    arr_x_t, arr_x_p = np.array(xppp_yt), np.array(xppp_yp)
    mask = np.isfinite(arr_t) & np.isfinite(arr_p)
    mask_x = np.isfinite(arr_x_t) & np.isfinite(arr_x_p)
    if mask.sum() > 10 and mask_x.sum() > 10:
        return _blend_mae(
            mean_absolute_error(arr_t[mask], arr_p[mask]),
            mean_absolute_error(arr_x_t[mask_x], arr_x_p[mask_x]),
        )
    return TUNING_INVALID_SCORE


MIN_SINGLE_SEASON_TUNING_RECORDS = 50

_ELO_CONFIG_KEYS = (
    "K_OFF", "K_DEF", "ELO_SCALING_FACTOR", "HOME_PPP_BOOST", "OFFSEASON_REVERSION",
    "USAGE_FLOOR", "assist_split", "k_mult_half_life", "rd_floor", "garbage_time_weight",
    "clutch_boost", "tov_penalty", "foul_draw_boost", "variance_dampen",
    "xppp_actual_blend", "k_def_events", "tov_rate_threshold", "three_pa_rate_threshold",
)


def default_elo_config() -> dict:
    """Pipeline default Elo config (used when CV tuning has no valid folds)."""
    d = PlayerRatingTracker.DEFAULTS
    return {k: d[k] for k in _ELO_CONFIG_KEYS}


def default_hier_config() -> dict:
    """Pipeline default hierarchical weights."""
    return {
        "w1": 0.50,
        "w2": 0.25,
        "w3": 0.15,
        "w5": 0.10,
        "k_off": 2.0,
        "k_def": 1.5,
        "league_avg_rtg": float(DEFAULT_LEAGUE_RTG),
    }


def _season_walkforward_mae(
    seasons: list,
    by_season: dict,
    fold_fn,
) -> float:
    fold_maes = []
    if len(seasons) >= 2:
        for i in range(1, len(seasons)):
            train_recs = []
            for s in seasons[:i]:
                train_recs.extend(by_season.get(s, []))
            val_recs = by_season.get(seasons[i], [])
            if not train_recs or not val_recs:
                fold_maes.append(TUNING_INVALID_SCORE)
                continue
            fold_maes.append(fold_fn(train_recs, val_recs))
    elif len(seasons) == 1:
        # One training season: chronological 70/30 holdout within that season.
        recs = by_season.get(seasons[0], [])
        if len(recs) >= MIN_SINGLE_SEASON_TUNING_RECORDS:
            split = int(len(recs) * 0.7)
            train_recs = recs[:split]
            val_recs = recs[split:]
            if train_recs and val_recs:
                fold_maes.append(fold_fn(train_recs, val_recs))
    if not fold_maes:
        if len(seasons) == 1:
            n = len(by_season.get(seasons[0], []))
            print(
                f"  ⚠ Tuning CV: single season with {n} stints "
                f"(need ≥{MIN_SINGLE_SEASON_TUNING_RECORDS} for holdout).",
            )
        elif len(seasons) < 2:
            print("  ⚠ Tuning CV: no seasons in training data.")
        return TUNING_INVALID_SCORE
    return float(np.mean(fold_maes))


def _hier_warmup(engine, records: list[dict]) -> None:
    for rec in records:
        engine.update(rec["hp"], rec["ap"], rec["home_pts"], rec["away_pts"], rec["poss"])


def _hier_eval_fold(engine, records: list[dict]) -> float:
    yt, yp, xppp_yt, xppp_yp = [], [], [], []
    for rec in records:
        poss = rec["poss"]
        xh, xa, _, _ = engine.predict_pts(rec["hp"], rec["ap"], poss)
        xppp_h = rec["home_xpts"] / poss if poss else 0.0
        xppp_a = rec["away_xpts"] / poss if poss else 0.0
        if np.isfinite(xh) and np.isfinite(xa):
            yt.extend([rec["home_pts"], rec["away_pts"]])
            yp.extend([xh, xa])
        xppp_yt.extend([xppp_h, xppp_a])
        xppp_yp.extend([xh / poss if poss else 0.0, xa / poss if poss else 0.0])
        engine.update(rec["hp"], rec["ap"], rec["home_pts"], rec["away_pts"], poss)
    if len(yt) <= 10:
        return TUNING_INVALID_SCORE
    points_mae = mean_absolute_error(yt, yp)
    arr_x_t, arr_x_p = np.array(xppp_yt), np.array(xppp_yp)
    mask_x = np.isfinite(arr_x_t) & np.isfinite(arr_x_p)
    xppp_mae = (
        mean_absolute_error(arr_x_t[mask_x], arr_x_p[mask_x])
        if mask_x.sum() > 10 else points_mae
    )
    return _blend_mae(points_mae, xppp_mae)


def _run_optuna_study(objective, n_trials, label, bounds_spec=None):
    def callback(study, trial):
        print(f"\nTrial {trial.number}:")
        for key, value in trial.params.items():
            if isinstance(value, float):
                print(f"  {key}: {value:.4f}")
            else:
                print(f"  {key}: {value}")
        if trial.value is not None:
            print(f"  CV MAE: {trial.value:.4f}")
        if bounds_spec and trial.params:
            at_edge = _params_at_bounds(trial.params, bounds_spec)
            if at_edge:
                print(f"  ⚠ at bound edge: {', '.join(at_edge)}")

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True, callbacks=[callback])

    print("\n" + "=" * 50)
    print(f"✅ {label} COMPLETE")
    print(f"Best CV MAE: {study.best_value:.4f}")
    if study.best_value >= TUNING_INVALID_SCORE:
        print(
            f"  ⚠ WARNING: all trials scored {TUNING_INVALID_SCORE:.0f} — "
            "tuning had no valid CV folds (check training data / season count)."
        )
    print("Best parameters:")
    for key, value in study.best_params.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")
    if bounds_spec:
        at_edge = _params_at_bounds(study.best_params, bounds_spec)
        if at_edge:
            print(f"  ⚠ best params at bound edge: {', '.join(at_edge)}")
    print("=" * 50)
    return study


def _suggest_elo_params(trial, bounds: AdaptiveParameterBounds | None) -> dict:
    b = bounds
    eb = ELO_PARAM_BOUNDS
    return {
        "k_off": _suggest_float_param(trial, b, "k_off", *eb["k_off"]),
        "k_def": _suggest_float_param(trial, b, "k_def", *eb["k_def"]),
        "elo_scaling": _suggest_int_param(trial, b, "elo_scaling", *eb["elo_scaling"]),
        "home_boost": _suggest_float_param(trial, b, "home_boost", *eb["home_boost"]),
        "offseason_reversion": _suggest_float_param(
            trial, b, "offseason_reversion", *eb["offseason_reversion"],
        ),
        "usage_floor": _suggest_float_param(trial, b, "usage_floor", *eb["usage_floor"]),
        "assist_split": _suggest_float_param(trial, b, "assist_split", *eb["assist_split"]),
        "k_mult_half_life": _suggest_float_param(
            trial, b, "k_mult_half_life", *eb["k_mult_half_life"],
        ),
        "rd_floor": _suggest_float_param(trial, b, "rd_floor", *eb["rd_floor"]),
        "garbage_time_weight": _suggest_float_param(
            trial, b, "garbage_time_weight", *eb["garbage_time_weight"],
        ),
        "clutch_boost": _suggest_float_param(trial, b, "clutch_boost", *eb["clutch_boost"]),
        "tov_penalty": _suggest_float_param(trial, b, "tov_penalty", *eb["tov_penalty"]),
        "foul_draw_boost": _suggest_float_param(trial, b, "foul_draw_boost", *eb["foul_draw_boost"]),
        "variance_dampen": _suggest_float_param(trial, b, "variance_dampen", *eb["variance_dampen"]),
        "xppp_actual_blend": _suggest_float_param(
            trial, b, "xppp_actual_blend", *eb["xppp_actual_blend"],
        ),
        "k_def_events": _suggest_float_param(trial, b, "k_def_events", *eb["k_def_events"]),
        "tov_rate_threshold": _suggest_float_param(
            trial, b, "tov_rate_threshold", *eb["tov_rate_threshold"],
        ),
        "three_pa_rate_threshold": _suggest_float_param(
            trial, b, "three_pa_rate_threshold", *eb["three_pa_rate_threshold"],
        ),
    }


def _suggest_hier_params(trial, bounds: AdaptiveParameterBounds | None) -> dict:
    b = bounds
    hb = HIER_PARAM_BOUNDS
    return {
        "w1": _suggest_float_param(trial, b, "w1", *hb["w1"]),
        "w2": _suggest_float_param(trial, b, "w2", *hb["w2"]),
        "w3": _suggest_float_param(trial, b, "w3", *hb["w3"]),
        "w5": _suggest_float_param(trial, b, "w5", *hb["w5"]),
        "k_off": _suggest_float_param(trial, b, "k_off", *hb["k_off"]),
        "k_def": _suggest_float_param(trial, b, "k_def", *hb["k_def"]),
        "league_rtg": _suggest_float_param(trial, b, "league_rtg", *hb["league_rtg"]),
    }


def tune_elo_tracker(
    stints_df, league_xppp, n_trials=20, bounds: AdaptiveParameterBounds = None,
):
    """
    Optimized Elo Tuner using Chronological Time-Series Cross-Validation.
    Fits ratings on past historical seasons and evaluates strictly on the following season.
    """
    df = _ensure_season_column(stints_df)
    records = _prepare_stint_records(df)
    by_season = _group_records_by_season(records)
    seasons = sorted(by_season.keys())

    def objective(trial):
        params = _suggest_elo_params(trial, bounds)
        config = {
            "K_OFF": params["k_off"], "K_DEF": params["k_def"],
            "ELO_SCALING_FACTOR": params["elo_scaling"],
            "HOME_PPP_BOOST": params["home_boost"],
            "OFFSEASON_REVERSION": params["offseason_reversion"],
            "USAGE_FLOOR": params["usage_floor"],
            "assist_split": params["assist_split"],
            "k_mult_half_life": params["k_mult_half_life"],
            "rd_floor": params["rd_floor"],
            "garbage_time_weight": params["garbage_time_weight"],
            "clutch_boost": params["clutch_boost"],
            "tov_penalty": params["tov_penalty"],
            "foul_draw_boost": params["foul_draw_boost"],
            "variance_dampen": params["variance_dampen"],
            "xppp_actual_blend": params["xppp_actual_blend"],
            "k_def_events": params["k_def_events"],
            "tov_rate_threshold": params["tov_rate_threshold"],
            "three_pa_rate_threshold": params["three_pa_rate_threshold"],
        }
        home_boost = params["home_boost"]
        elo_scaling = params["elo_scaling"]

        def fold_fn(train_recs, val_recs):
            tracker = PlayerRatingTracker(config=config, league_xppp=league_xppp)
            _elo_warmup(tracker, train_recs)
            mae, ats_miss = _elo_eval_fold(
                tracker, val_recs, league_xppp, home_boost, elo_scaling,
            )
            if mae >= TUNING_INVALID_SCORE:
                return TUNING_INVALID_SCORE
            # ATS-aware blend: lower is better
            return 0.5 * mae + 0.3 * (ats_miss * 30.0) + 0.2 * (ats_miss * 20.0)

        return _season_walkforward_mae(seasons, by_season, fold_fn)

    study = _run_optuna_study(objective, n_trials, "ELO TUNING", ELO_PARAM_BOUNDS)
    if study.best_value >= TUNING_INVALID_SCORE:
        print("  ⚠ Elo tuning invalid — using pipeline defaults (not Optuna trial params).")
        return default_elo_config(), study.best_value
    return map_elo_params(study.best_params), study.best_value


def tune_hierarchical(stints_df, n_trials=30, bounds: AdaptiveParameterBounds = None):
    """
    Optimized Hierarchical Tuner using Chronological Time-Series Cross-Validation.
    Fits ratings on past historical seasons and evaluates strictly on the following season.
    """
    df = _ensure_season_column(stints_df)
    records = _prepare_stint_records(df)
    by_season = _group_records_by_season(records)
    seasons = sorted(by_season.keys())

    def objective(trial):
        params = _suggest_hier_params(trial, bounds)
        total = params["w1"] + params["w2"] + params["w3"] + params["w5"]
        if total <= 0:
            return TUNING_INVALID_SCORE

        def fold_fn(train_recs, val_recs):
            engine = HierarchicalPossessionEngine(
                params["w1"] / total, params["w2"] / total,
                params["w3"] / total, params["w5"] / total,
                k_off=params["k_off"], k_def=params["k_def"],
                league_avg_rtg=params["league_rtg"],
            )
            _hier_warmup(engine, train_recs)
            return _hier_eval_fold(engine, val_recs)

        return _season_walkforward_mae(seasons, by_season, fold_fn)

    study = _run_optuna_study(objective, n_trials, "HIERARCHICAL TUNING", HIER_PARAM_BOUNDS)
    if study.best_value >= TUNING_INVALID_SCORE:
        print("  ⚠ Hierarchical tuning invalid — using pipeline defaults.")
        return default_hier_config(), study.best_value
    best = study.best_params.copy()
    best["league_avg_rtg"] = best.pop("league_rtg")
    return best, study.best_value


print("✅ Tuning functions ready.")


In [ ]:
# ── module: tuning_cache ─────────────────────────────────────────────────────
"""Cache tuned hyperparameters per training-season window."""
from __future__ import annotations

import hashlib
import json
from pathlib import Path


CACHE_DIR = STATE_DIR / "tuning_cache"
# Bump when tuning/feature logic changes so stale caches are not reused.
CACHE_VERSION = "2026-06-21-v2"


def _seasons_hash(train_seasons, stints_df=None) -> str:
    # Task 004: fold in the preprocessing/feature/market-snapshot/validation
    # schema versions so any change to those (score/date repair, stint
    # semantics, market snapshot logic, CV logic) produces a different key
    # instead of reusing a tuning result computed under different semantics.
    h = hashlib.md5()
    h.update(CACHE_VERSION.encode())
    h.update(
        f"prep={PREPROCESSING_SCHEMA_VERSION};feat={FEATURE_SCHEMA_VERSION};"
        f"mkt={MARKET_SNAPSHOT_SCHEMA_VERSION};val={VALIDATION_SCHEMA_VERSION}".encode()
    )
    h.update(",".join(str(s) for s in sorted(train_seasons)).encode())
    if stints_df is not None and not stints_df.empty:
        h.update(str(len(stints_df)).encode())
        if "GAME_ID" in stints_df.columns:
            h.update(str(stints_df["GAME_ID"].nunique()).encode())
    return h.hexdigest()[:16]


def cache_path(kind: str, train_seasons, stints_df=None) -> Path:
    key = _seasons_hash(train_seasons, stints_df)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    return CACHE_DIR / f"{kind}_{key}.json"


def load_cached(kind: str, train_seasons, stints_df=None):
    path = cache_path(kind, train_seasons, stints_df)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text())
    except Exception:
        return None


def save_cached(kind: str, train_seasons, params: dict, stints_df=None):
    path = cache_path(kind, train_seasons, stints_df)
    path.write_text(json.dumps(params, indent=2))


In [ ]:
# ── module: calibration_policy ───────────────────────────────────────────────
"""Central routing for cover/win probability sources (legacy vs simplified)."""
from __future__ import annotations

import numpy as np



def select_cover_prob(
    mode: str | None,
    *,
    raw_cover: float,
    calibrated_cover: float,
    ats_classifier_prob: float | None = None,
    blend_weight: float = 0.5,
) -> float:
    mode = mode or CALIBRATION_MODE
    if mode == "simplified":
        base = float(raw_cover)
    elif mode == "beta_ats" and ats_classifier_prob is not None:
        base = float(ats_classifier_prob)
    else:
        base = float(calibrated_cover)
    if ats_classifier_prob is not None and mode in ("beta_ats",) and CALIBRATION_MODE != "legacy_stack":
        return base
    if ats_classifier_prob is not None and mode == "legacy_stack":
        return base
    return base


def blend_ats_classifier_cover(
    spread_cover: float,
    ats_prob: float | None,
    blend: float = 0.5,
) -> float:
    if ats_prob is None or not np.isfinite(ats_prob):
        return float(spread_cover)
    w = float(np.clip(blend, 0.0, 1.0))
    return float((1.0 - w) * spread_cover + w * ats_prob)


def resolve_win_prob(
    *,
    margin_for_win: float,
    win_model=None,
    calibrator=None,
    meta_calibrate_fn=None,
    pred_total=None,
) -> float:
    source = WIN_PROB_SOURCE
    if source == "auto":
        if win_model is not None and getattr(win_model, "fitted", False) and REQUIRE_META_WIN_WHEN_FITTED:
            source = "meta_win"
        elif calibrator is not None:
            source = "rolling_platt"
        else:
            source = "margin_isotonic"

    if source == "meta_win" and win_model is not None and getattr(win_model, "fitted", False):
        raise RuntimeError("resolve_win_prob expects precomputed win_model output at call site")

    if source == "rolling_platt" and calibrator is not None:
        try:
            return float(calibrator.predict(margin_for_win, total=pred_total))
        except TypeError:
            return float(calibrator.predict(margin_for_win))

    if meta_calibrate_fn is not None:
        return float(meta_calibrate_fn(margin_for_win))

    return float(1.0 / (1.0 + np.exp(-margin_for_win / 12.0)))


def should_use_bet_calibrator(mode: str | None = None) -> bool:
    if BET_SELECTION_MODE == "confidence_only":
        return True
    if CONFIDENCE_MODE == "unified" or CONFIDENCE_SELECTION_MODE == "min_score":
        return True
    mode = mode or CALIBRATION_MODE
    return mode == "legacy_stack"


def should_use_season1_calibrator() -> bool:
    """Fit confidence calibrator on training calib split before the first simulated season."""
    return BET_SELECTION_MODE == "confidence_only" or CONFIDENCE_MODE == "unified"


def should_use_rolling_platt_when_win_model(mode: str | None = None) -> bool:
    mode = mode or CALIBRATION_MODE
    if WIN_PROB_SOURCE == "rolling_platt":
        return True
    if mode == "simplified":
        return False
    return True


In [ ]:
# ── module: calibration_registry ─────────────────────────────────────────────
"""Calibration slice registry (Task 053).

Phase 4B of the plan requires exactly one calibration path per target
(engine margin, final margin, cover probability, ML win probability,
interval width) and forbids fitting Elo isotonic, MetaScore isotonic,
static spread calibration, MetaWin isotonic, and ATS isotonic calibrators
all on the identical tail slice (currently ``calib_features`` in
``pipeline/backtest.py``, reused byte-for-byte at every one of those call
sites).

This module does not decide *how* to split calibration data — that is a
larger backtest.py control-flow change (flagged as remaining risk in this
phase's handoff, since it also feeds the Elo/hierarchical tuning control
flow owned by a different phase). It provides the machinery to (a) declare
one calibration path per target and (b) fail loudly the moment two
different targets are calibrated on the exact same row set, so the defect
can no longer regress silently once the production wiring is repaired.
"""
from __future__ import annotations

import hashlib

import numpy as np

# Phase 4B's five single-purpose calibration targets.
CALIBRATION_TARGETS = (
    "engine_margin",
    "final_margin",
    "cover_probability",
    "ml_win_probability",
    "interval_width",
)


class CalibrationRoleError(ValueError):
    """Raised when a target's calibration path is registered twice, or two
    different targets share the exact same calibration row slice."""


def _slice_fingerprint(row_ids) -> str:
    arr = np.asarray(sorted(np.asarray(row_ids).tolist()))
    return hashlib.sha256(arr.tobytes()).hexdigest()


class CalibrationSliceRegistry:
    """Tracks which row-id slice calibrated which target."""

    def __init__(self):
        self._by_target: dict[str, str] = {}
        self._fingerprint_to_targets: dict[str, list[str]] = {}

    def register(self, target: str, row_ids) -> None:
        if target in self._by_target:
            raise CalibrationRoleError(
                f"Target {target!r} already has a registered calibration "
                f"path (Task 053: exactly one calibration path per target)."
            )
        fp = _slice_fingerprint(row_ids)
        self._by_target[target] = fp
        self._fingerprint_to_targets.setdefault(fp, []).append(target)
        others = self._fingerprint_to_targets[fp]
        if len(others) > 1:
            raise CalibrationRoleError(
                f"Targets {others!r} were all calibrated on the identical "
                "row slice. Phase 4B forbids reusing the same calibration "
                "slice across multiple calibrators (Elo isotonic, "
                "MetaScore isotonic, static spread calibration, MetaWin "
                "isotonic, and ATS isotonic must not all fit on the same "
                "tail slice)."
            )

    def registered_targets(self) -> list:
        return list(self._by_target.keys())

    def assert_all_targets_registered(self, targets=CALIBRATION_TARGETS) -> None:
        missing = [t for t in targets if t not in self._by_target]
        if missing:
            raise CalibrationRoleError(
                f"Missing a declared calibration path for: {missing!r}"
            )


# `pipeline/backtest.py`'s five calibrators that previously all fit on the
# identical tail `calib_features` slice (Task 053's confirmed
# `calibration_slice_reuse` defect).
BACKTEST_CALIBRATION_TARGETS = (
    "elo_isotonic",
    "metascore_isotonic",
    "static_spread_calibration",
    "metawin_isotonic",
    "ats_isotonic",
)


def chronological_game_id_partition(
    calib_df, targets=BACKTEST_CALIBRATION_TARGETS, date_col: str = "game_date",
    id_col: str = "GAME_ID",
) -> dict:
    """Partition ``calib_df``'s ``GAME_ID``s, ordered by ``date_col``, into
    one contiguous, mutually disjoint block per target.

    This is the "give each calibrator its own out-of-sample slice" fix for
    the `calibration_slice_reuse` leak: every target gets a distinct set of
    rows from the same calibration-tail span, so no two calibrators can be
    fit on the identical row set (which the registry would otherwise reject
    the moment both are ``register()``ed), while every slice remains
    strictly within the pre-test-season calibration window (no chronological
    leakage relative to `test_season`).

    Returns ``{target: np.ndarray of GAME_ID}``, in the same iteration order
    as ``targets``, with sizes as close to equal as possible (any remainder
    games go to the earliest targets).
    """
    targets = list(targets)
    if calib_df is None or len(calib_df) == 0:
        return {t: np.array([]) for t in targets}
    ordered_ids = (
        calib_df[[id_col, date_col]]
        .drop_duplicates(id_col)
        .sort_values(date_col)[id_col]
        .to_numpy()
    )
    n = len(ordered_ids)
    k = len(targets)
    sizes = [n // k + (1 if i < n % k else 0) for i in range(k)]
    boundaries = [0]
    for s in sizes:
        boundaries.append(boundaries[-1] + s)
    return {
        target: ordered_ids[boundaries[i]:boundaries[i + 1]]
        for i, target in enumerate(targets)
    }


def slice_by_game_ids(df, game_ids, id_col: str = "GAME_ID"):
    """Return the rows of ``df`` whose ``id_col`` is in ``game_ids``."""
    if df is None or len(df) == 0:
        return df
    return df[df[id_col].isin(set(game_ids))].copy()


def build_disjoint_calibration_slices(
    calib_df, registry: CalibrationSliceRegistry,
    targets=BACKTEST_CALIBRATION_TARGETS, date_col: str = "game_date",
    id_col: str = "GAME_ID",
) -> dict:
    """Partition ``calib_df`` into one disjoint sub-DataFrame per target and
    register each slice's row ids with ``registry`` in one step.

    Raises ``CalibrationRoleError`` (via ``registry.register``) if this is
    ever called twice for the same target, or if a future edit makes two
    targets' slices overlap.
    """
    id_partition = chronological_game_id_partition(
        calib_df, targets=targets, date_col=date_col, id_col=id_col,
    )
    slices = {}
    for target, ids in id_partition.items():
        registry.register(target, ids)
        slices[target] = slice_by_game_ids(calib_df, ids, id_col=id_col)
    return slices


In [ ]:
# ── module: ats_classifier ───────────────────────────────────────────────────
"""Walk-forward ATS cover classifier (direct P(cover) optimization)."""
from __future__ import annotations

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


DERIVED_COLS = [
    "model_edge", "abs_model_edge", "rating_uncertainty_sum",
    "elo_meta_agreement_feat", "spread_quantile_width_feat",
]

EXTRA_FEATURE_DEFAULTS = {
    "model_edge": 0.0,
    "abs_model_edge": 0.0,
    "rating_uncertainty_sum": 700.0,
    "elo_meta_agreement_feat": 1.0,
    "spread_quantile_width_feat": 24.0,
}


class ATSClassifier:
    def __init__(self, feature_cols=None, calibrator: str = "isotonic", C: float = 0.1):
        self.feature_cols = list(feature_cols) if feature_cols else [
            c for c in SAFE_FEATURE_COLS if c in SAFE_FEATURE_COLS
        ][:40]
        self.calibrator_name = calibrator
        self.model = LogisticRegression(C=C, solver="lbfgs", max_iter=300)
        self.scaler = StandardScaler()
        self.iso = None
        self._fit_cols: list[str] = []
        self.fitted = False

    @staticmethod
    def _close_spread(row) -> float:
        for k in ("closing_spread", "CLOSING_SPREAD", "market_spread", "MARKET_SPREAD"):
            v = row.get(k) if hasattr(row, "get") else None
            if v is not None and pd.notna(v):
                return float(v)
        return np.nan

    @staticmethod
    def _pred_spread(row) -> float:
        for k in ("pred_margin", "PRED_SPREAD", "pred_spread"):
            v = row.get(k) if hasattr(row, "get") else None
            if v is not None and pd.notna(v):
                return float(v)
        return 0.0

    def _enrich_row(self, row: dict) -> dict:
        r = dict(row)
        pred = self._pred_spread(r)
        close = self._close_spread(r)
        mkt = r.get("market_spread", r.get("MARKET_SPREAD", close))
        edge = pred + float(mkt) if pd.notna(mkt) else pred + close
        r["model_edge"] = edge
        r["abs_model_edge"] = abs(edge)
        h_unc = float(r.get("h_rating_uncertainty", r.get("H_RATING_UNCERTAINTY", 350)) or 350)
        a_unc = float(r.get("a_rating_uncertainty", r.get("A_RATING_UNCERTAINTY", 350)) or 350)
        r["rating_uncertainty_sum"] = h_unc + a_unc
        r["elo_meta_agreement_feat"] = float(
            r.get("elo_meta_agreement", r.get("ELO_META_AGREEMENT", 1.0)) or 1.0
        )
        r["spread_quantile_width_feat"] = float(
            r.get("spread_quantile_width", r.get("SPREAD_QUANTILE_WIDTH", 24)) or 24
        )
        return r

    def _build_matrix(self, df: pd.DataFrame) -> np.ndarray:
        cols = list(self._fit_cols)
        X = pd.DataFrame(index=df.index)
        for c in cols:
            if c in df.columns:
                X[c] = df[c].fillna(0)
            else:
                X[c] = 0.0
        for _, row in df.iterrows():
            enriched = self._enrich_row(row.to_dict())
            for c in DERIVED_COLS:
                if c not in cols:
                    continue
                idx = row.name
                X.loc[idx, c] = enriched.get(c, EXTRA_FEATURE_DEFAULTS.get(c, 0))
        return X[cols].fillna(0).to_numpy(dtype=float)

    @staticmethod
    def labels_from_df(df: pd.DataFrame) -> np.ndarray:
        close = closing_spread_series(df)
        if close is None:
            close = pd.to_numeric(df.get("market_spread", df.get("MARKET_SPREAD")), errors="coerce")
        margin = df["actual_margin"] if "actual_margin" in df.columns else df["ACTUAL_MARGIN"]
        cover = margin + close
        pred = df.get("pred_margin", df.get("PRED_SPREAD", 0))
        edge = pred + close
        direction = np.where(edge >= 0, "Home", "Away")
        y = np.where(cover > 0, 1, 0)
        y = np.where(direction == "Away", 1 - y, y)
        push = cover == 0
        y = y.astype(float)
        y[push] = np.nan
        return y

    def fit(self, train_df: pd.DataFrame, calib_df: pd.DataFrame | None = None):
        train_df = train_df.dropna(subset=["actual_margin"] if "actual_margin" in train_df.columns else ["ACTUAL_MARGIN"])
        y = self.labels_from_df(train_df)
        mask = np.isfinite(y)
        train_df = train_df.loc[mask].copy()
        y = y[mask].astype(int)
        if len(train_df) < 50 or len(np.unique(y)) < 2:
            self.fitted = False
            return self

        base_cols = [c for c in self.feature_cols if c in train_df.columns]
        self._fit_cols = base_cols + [c for c in DERIVED_COLS if c not in base_cols]
        X = self._build_matrix(train_df)
        Xs = self.scaler.fit_transform(X)
        self.model.fit(Xs, y)

        if calib_df is not None and not calib_df.empty:
            cy = self.labels_from_df(calib_df)
            cm = np.isfinite(cy)
            calib_df = calib_df.loc[cm].copy()
            cy = cy[cm].astype(int)
            if len(calib_df) >= 30 and len(np.unique(cy)) > 1:
                Xc = self.scaler.transform(self._build_matrix(calib_df))
                raw = self.model.predict_proba(Xc)[:, 1]
                if self.calibrator_name == "beta":
                    try:
                        from betacal import BetaCalibration
                        self.iso = BetaCalibration(parameters="ab")
                        self.iso.fit(raw, cy)
                    except ImportError:
                        self.iso = IsotonicRegression(out_of_bounds="clip")
                        self.iso.fit(raw, cy)
                else:
                    self.iso = IsotonicRegression(out_of_bounds="clip")
                    self.iso.fit(raw, cy)
        self.fitted = True
        return self

    def predict_cover_prob(self, feat_dict: dict, direction: str, market_spread: float) -> float:
        if not self.fitted:
            return 0.524
        row = self._enrich_row(dict(feat_dict))
        row["market_spread"] = market_spread
        X = np.array([[row.get(c, EXTRA_FEATURE_DEFAULTS.get(c, 0)) for c in self._fit_cols]], dtype=float)
        Xs = self.scaler.transform(X)
        raw = float(self.model.predict_proba(Xs)[0, 1])
        if direction == "Away":
            raw = 1.0 - raw
        if self.iso is not None:
            try:
                prob = float(self.iso.predict([raw if direction == "Home" else 1 - raw])[0])
            except Exception:
                prob = float(self.iso.predict(np.array([raw]))[0])
            if direction == "Away":
                prob = 1.0 - prob if self.calibrator_name != "beta" else 1.0 - prob
            return float(np.clip(prob if direction == "Home" else 1.0 - (1.0 - prob), 0.01, 0.99))
        return float(np.clip(raw if direction == "Home" else 1.0 - raw, 0.01, 0.99))

    def save(self, path: str | Path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path: str | Path):
        with open(path, "rb") as f:
            return pickle.load(f)


In [ ]:
# ── module: team_volatility ──────────────────────────────────────────────────
"""Rolling team margin-surprise volatility for stake sizing."""
from __future__ import annotations

from collections import defaultdict, deque

import numpy as np


class TeamVolatilityTracker:
    def __init__(self, window: int = 20, min_games: int = 8):
        self.window = window
        self.min_games = min_games
        self.errors = defaultdict(lambda: deque(maxlen=window))
        self._league_sigma = 10.0

    def update(self, team: str, pred_margin: float, actual_margin: float, *, home: bool = True):
        """Record signed margin error from home perspective for one team."""
        if home:
            err = float(actual_margin) - float(pred_margin)
        else:
            err = float(-actual_margin) - float(-pred_margin)
        self.errors[team].append(err)
        all_errs = [e for q in self.errors.values() for e in q]
        if len(all_errs) >= self.min_games:
            self._league_sigma = float(np.std(all_errs, ddof=1)) if len(all_errs) > 1 else 10.0

    def update_matchup(self, home: str, away: str, pred_margin: float, actual_margin: float):
        self.update(home, pred_margin, actual_margin, home=True)
        self.update(away, pred_margin, actual_margin, home=False)

    def sigma(self, team: str) -> float:
        errs = list(self.errors.get(team, []))
        if len(errs) < self.min_games:
            return self._league_sigma
        return float(np.std(errs, ddof=1)) if len(errs) > 1 else self._league_sigma

    def matchup_sigma(self, home: str, away: str) -> float:
        return float(max(self.sigma(home), self.sigma(away)))

    def stake_multiplier(self, home: str, away: str) -> float:
        sig = self.matchup_sigma(home, away)
        lg = max(self._league_sigma, 1e-6)
        raw = 1.0 - 0.3 * (sig / lg - 1.0)
        return float(np.clip(raw, 0.5, 1.5))

    def save_state(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump({
                "window": self.window,
                "min_games": self.min_games,
                "errors": {k: list(v) for k, v in self.errors.items()},
                "_league_sigma": self._league_sigma,
            }, f)

    @classmethod
    def load_state(cls, path):
        import pickle
        with open(path, "rb") as f:
            data = pickle.load(f)
        obj = cls(window=data["window"], min_games=data["min_games"])
        for k, v in data["errors"].items():
            obj.errors[k] = deque(v, maxlen=obj.window)
        obj._league_sigma = data.get("_league_sigma", 10.0)
        return obj


In [ ]:
# ── module: simulate ─────────────────────────────────────────────────────────
# Cell 12 – run_simulation (rewritten with SOS, proper Kelly, Platt rolling calibration)
from __future__ import annotations

from collections import defaultdict, deque

import numpy as np
import pandas as pd
from tqdm.auto import tqdm



def run_simulation(
    season_df,
    hier_engine,
    elo_tracker,
    meta_model,
    pace_tracker,
    odds_dict,
    calibrator=None,               # should be a RollingPlattCalibrator instance
    team_xppp_tracker=None,
    team_form_tracker=None,
    spread_calibrator=None,
    conformal=None,
    rotation_tracker=None,
    inactive_map: dict = None,
    sos_window: int = None,
    lineup_elo_tracker=None,
    chemistry_tracker=None,
    team_elo_tracker=None,
    travel_tracker=None,
    epm_tracker=None,
    ref_tracker=None,
    shot_quality_tracker=None,
    hapm_tracker=None,
    require_rlm: bool = False,
    ou_bet: bool = True,
    min_edge_pts: float = None,
    max_favorite_decimal: float = 1.45,
    ou_min_edge: float = 3.0,
    confidence_calibrator=None,
    win_model=None,
    total_model=None,
    score_pair_model=None,
    ats_classifier=None,
    vol_tracker=None,
    edge_threshold: float = None,
    elo_calibrator=None,
    require_elo_agreement: bool = False,
    elo_agreement_extra_edge: float = None,
    disagreement_model=None,
    min_confidence_threshold: float | None = None,
    max_confidence_threshold: float | None = None,
    ml_calibrator=None,
    min_ml_win_pct: float | None = None,
):
    """
    Walk‑forward simulation for a single season.
    """
    if "game_date" in season_df.columns:
        # Task 015: enforce chronological iteration explicitly by decision
        # timestamp (game_date) then GAME_ID, regardless of input row order.
        season_df = season_df.sort_values(["game_date", "GAME_ID", "stint_id"]).reset_index(drop=True)
        _per_game_dates = season_df.drop_duplicates("GAME_ID")["game_date"]
        assert _per_game_dates.is_monotonic_increasing, (
            "run_simulation: per-game decision timestamps are not monotonic "
            "nondecreasing after sort — chronological iteration is violated"
        )

    if min_edge_pts is None:
        min_edge_pts = edge_threshold if edge_threshold is not None else GOOD_BET_EDGE
    if elo_agreement_extra_edge is None:
        elo_agreement_extra_edge = ELO_AGREEMENT_EXTRA_EDGE
    if min_ml_win_pct is None:
        min_ml_win_pct = float(MIN_ML_WIN_PCT)

    results = []
    resid_hist = []
    last_date = {}
    team_game_dates = defaultdict(lambda: deque(maxlen=20))   # schedule density

    season_start_date = None
    team_games_played = defaultdict(int)
    team_rosters_seen = defaultdict(set)
    lineup_cache = {}
    season_player_ids = set()
    season_year = None

    # Recent form deques (net rating)
    RECENT_WINDOW = 10
    team_recent_net = defaultdict(lambda: deque(maxlen=RECENT_WINDOW))
    # Team-specific home-court: rolling margins split by venue.
    team_home_margin = defaultdict(lambda: deque(maxlen=20))
    team_road_margin = defaultdict(lambda: deque(maxlen=20))

    # --- Strength of Schedule (opponent net rating history) ---
    if sos_window is None:
        sos_window = SOS_WINDOW
    opponent_history = defaultdict(lambda: deque(maxlen=sos_window))

    # Vectorized per-game aggregation (replaces inner per-stint Python loop)
    game_stats_map = precompute_game_team_stats(season_df)

    last_valid_date = None
    skipped_games = 0

    for game_id, group in tqdm(season_df.groupby("GAME_ID", sort=False), desc="Walk-forward simulation"):
        first_row = group.iloc[0]
        raw_gdate = first_row.get("game_date", pd.NaT)
        home_team = TEAM_MAP.get(first_row.get("home_team", "HOME"), first_row.get("home_team", "HOME"))
        away_team = TEAM_MAP.get(first_row.get("away_team", "AWAY"), first_row.get("away_team", "AWAY"))

        season_yr = None
        if is_valid_timestamp(season_start_date):
            season_yr = season_start_date.year
        gdate = coerce_game_date(
            raw_gdate, game_id=game_id, prev_date=last_valid_date, season_start_year=season_yr,
        )
        if gdate is None:
            skipped_games += 1
            continue
        last_valid_date = gdate

        if isinstance(gdate, pd.Timestamp):
            current_season = gdate.year + (1 if gdate.month >= 9 else 0)
            if season_year is None:
                season_year = gdate.year
        else:
            current_season = 2025

        if season_start_date is None or (
            is_valid_timestamp(gdate) and gdate.month > 8
            and is_valid_timestamp(season_start_date)
            and (gdate - season_start_date).days > 150
        ):
            if isinstance(gdate, pd.Timestamp) and season_year is not None and gdate.year > season_year:
                returning_ids = set(season_player_ids) if season_player_ids else None
                hier_engine.offseason_revert()
                elo_tracker.offseason_revert(returning_player_ids=returning_ids)
                season_player_ids = set()
            season_start_date = gdate if is_valid_timestamp(gdate) else pd.Timestamp.now()
            if isinstance(gdate, pd.Timestamp):
                season_year = gdate.year
            team_games_played.clear()
            team_rosters_seen.clear()
            lineup_cache.clear()
            team_recent_net.clear()
            opponent_history.clear()   # also reset SOS at season boundary

        # Task 031: same fix as `pipeline/features.py::generate_features` —
        # `home_starters`/`away_starters` feed T-60 feature generation below
        # and must never come from this game's own actual PBP lineup
        # (Rule 4). `lineup_cache` only ever holds the roster observed in a
        # previously *completed* game (updated post-game below); an unseen
        # team this season has a genuinely unknown pre-game roster (Rule 5),
        # not a fabricated one.
        home_starters = lineup_cache.get(home_team, [])
        away_starters = lineup_cache.get(away_team, [])
        # Actual PBP-observed lineup for *this* game — postgame-only, used
        # to update rosters-seen bookkeeping and `lineup_cache` for future
        # games, never as this game's own T-60 feature input.
        first_stint = group.iloc[0]
        actual_home_starters = _parse_player_string(first_stint.get("HOME_players", ""))
        actual_away_starters = _parse_player_string(first_stint.get("AWAY_players", ""))

        # Aggregate actual outcomes (precomputed, vectorized).
        # Task 014: act_h/act_a prefer the canonical, independently-verified
        # final score (Task 006/007) via precompute_game_team_stats; never
        # re-sum home_pts/away_pts from filtered stints in this loop.
        gs = game_stats_map.get(game_id, {})
        act_h = float(gs.get("act_h", 0.0))
        act_a = float(gs.get("act_a", 0.0))
        tot_poss = float(gs.get("tot_poss", 0.0))
        home_tot_poss = float(gs.get("home_poss", 0.0))
        away_tot_poss = float(gs.get("away_poss", 0.0))
        home_tot_xpts = float(gs.get("home_xpts", 0.0))
        away_tot_xpts = float(gs.get("away_xpts", 0.0))

        if home_tot_poss == 0:
            home_tot_poss = tot_poss / 2
        if away_tot_poss == 0:
            away_tot_poss = tot_poss / 2
        if act_h == 0 and act_a == 0:
            act_h = group.get("HOME_FINAL_SCORE", pd.Series([110.0])).max()
            act_a = group.get("AWAY_FINAL_SCORE", pd.Series([107.0])).max()
        if tot_poss == 0:
            tot_poss = 100.0

        home_xppp_game = home_tot_xpts / home_tot_poss if home_tot_poss > 0 else DEFAULT_LEAGUE_XPPP
        away_xppp_game = away_tot_xpts / away_tot_poss if away_tot_poss > 0 else DEFAULT_LEAGUE_XPPP

        # Rotation-weighted expected lineups
        h_weights = rotation_tracker.expected_weights(home_team, home_starters) if rotation_tracker else []
        a_weights = rotation_tracker.expected_weights(away_team, away_starters) if rotation_tracker else []
        home_lineup = _top_lineup(h_weights) or home_starters
        away_lineup = _top_lineup(a_weights) or away_starters

        inactive_ids = set()
        if inactive_map is not None:
            inactive_ids = {str(x) for x in inactive_map.get(game_id, [])}

        feat = build_game_features(
            game_id=game_id,
            gdate=gdate,
            home_team=home_team,
            away_team=away_team,
            home_starters=home_starters,
            away_starters=away_starters,
            elo_tracker=elo_tracker,
            hier_engine=hier_engine,
            pace_tracker=pace_tracker,
            odds_dict=odds_dict,
            team_xppp_tracker=team_xppp_tracker,
            team_form_tracker=team_form_tracker,
            rotation_tracker=rotation_tracker,
            lineup_elo_tracker=lineup_elo_tracker,
            chemistry_tracker=chemistry_tracker,
            team_elo_tracker=team_elo_tracker,
            travel_tracker=travel_tracker,
            epm_tracker=epm_tracker,
            ref_tracker=ref_tracker,
            shot_quality_tracker=shot_quality_tracker,
            hapm_tracker=hapm_tracker,
            last_date=last_date,
            team_game_dates=team_game_dates,
            team_recent_net=team_recent_net,
            opponent_history=opponent_history,
            team_home_margin=team_home_margin,
            team_road_margin=team_road_margin,
            team_games_played=team_games_played,
            team_rosters_seen=team_rosters_seen,
            season_start_date=season_start_date,
            inactive_ids=inactive_ids,
            elo_calibrator=elo_calibrator,
        )
        market_spread = feat.get("market_spread", np.nan)
        market_ml = feat.get("market_ml", np.nan)
        market_total = feat.get("market_total", np.nan)
        closing_spread = feat.get("closing_spread", market_spread)
        spread_move = feat.get("spread_move", 0.0)
        public_home_pct = feat.get("public_home_pct", 0.5)

        # Prediction (with interaction features for train/serve parity)
        feat_model = build_feature_row(feat)
        if hasattr(meta_model, 'fitted') and meta_model.fitted:
            preds = meta_model.predict(feat_model)
        else:
            preds = {"pred_home": 110, "pred_away": 110, "pred_margin": 0.0,
                     "pred_total": 220, "win_prob": 0.5}

        raw_pred_margin = preds["pred_margin"]

        # Spread bias correction (before win-prob so probabilities match betting spread)
        if spread_calibrator is not None:
            corrected_spread = spread_calibrator.correct(raw_pred_margin)
        else:
            corrected_spread = raw_pred_margin

        if (
            score_pair_model is not None
            and getattr(score_pair_model, "fitted", False)
        ):
            pair_out = score_pair_model.predict_scores(feat_model)
            preds["pred_home"] = pair_out["pred_home"]
            preds["pred_away"] = pair_out["pred_away"]
            preds["sigma_total"] = pair_out.get("sigma_total")
            preds["p_over"] = pair_out.get("p_over")
            if USE_SCORE_PAIR_TOTAL:
                preds["pred_total"] = pair_out["pred_total"]
        elif (
            total_model is not None
            and getattr(total_model, "fitted", False)
        ):
            total_out = total_model._predict_raw_total(feat_model)
            preds["pred_total"] = total_out["pred_total"]
            if "total_quantile_width" in total_out:
                preds["total_quantile_width"] = total_out["total_quantile_width"]
                preds["sigma_total"] = float(total_out["total_quantile_width"]) / 2.0
            if "total_q10" in total_out:
                preds["total_q10"] = total_out["total_q10"]
                preds["total_q90"] = total_out["total_q90"]
            preds["pred_home"] = (preds["pred_total"] + corrected_spread) / 2.0
            preds["pred_away"] = (preds["pred_total"] - corrected_spread) / 2.0
        else:
            preds.setdefault("sigma_total", float(OU_DEFAULT_SIGMA))

        margin_for_win = corrected_spread
        if hasattr(meta_model, 'fitted') and meta_model.fitted:
            use_meta_win = (
                win_model is not None and getattr(win_model, "fitted", False)
                and (
                    WIN_PROB_SOURCE == "meta_win"
                    or (WIN_PROB_SOURCE == "auto" and REQUIRE_META_WIN_WHEN_FITTED)
                )
            )
            use_rolling = (
                calibrator is not None
                and (
                    WIN_PROB_SOURCE == "rolling_platt"
                    or (
                        WIN_PROB_SOURCE == "auto"
                        and not use_meta_win
                        and should_use_rolling_platt_when_win_model()
                    )
                )
            )
            if use_meta_win:
                raw_wp = win_model.predict_proba(
                    feat_model, pred_margin=margin_for_win,
                )
                preds["win_prob_raw"] = raw_wp
                if (
                    ml_calibrator is not None
                    and getattr(ml_calibrator, "_fitted", False)
                ):
                    preds["win_prob"] = ml_calibrator.transform(raw_wp)
                else:
                    preds["win_prob"] = raw_wp
            elif use_rolling:
                try:
                    preds["win_prob"] = calibrator.predict(
                        margin_for_win, total=preds.get("pred_total"),
                    )
                except TypeError:
                    preds["win_prob"] = calibrator.predict(margin_for_win)

        # Keep predicted scores consistent with the bias-corrected spread and the
        # predicted total: home - away == corrected_spread, home + away == total.
        pred_total = preds.get("pred_total", preds.get("pred_home", 0) + preds.get("pred_away", 0))
        cons_home = (pred_total + corrected_spread) / 2.0
        cons_away = (pred_total - corrected_spread) / 2.0
        preds["pred_home"] = cons_home
        preds["pred_away"] = cons_away

        # Conformal interval from spread calibrator residuals (fallback ±12)
        if conformal is not None:
            conf_lower, conf_upper, conf_width = conformal.predict_interval(corrected_spread, alpha=0.10)
        elif spread_calibrator is not None and hasattr(spread_calibrator, "predict_interval"):
            conf_lower, conf_upper, conf_width = spread_calibrator.predict_interval(corrected_spread, alpha=0.10)
        else:
            conf_lower, conf_upper, conf_width = corrected_spread - 12.0, corrected_spread + 12.0, 24.0

        q_width = preds.get("spread_quantile_width")
        if q_width is not None and np.isfinite(q_width):
            conf_width = float(q_width)
            if preds.get("spread_q10") is not None and preds.get("spread_q90") is not None:
                conf_lower = float(preds["spread_q10"])
                conf_upper = float(preds["spread_q90"])

        effective_edge_thr = variance_aware_edge_threshold(min_edge_pts, conf_width)
        disagreement_info = {}
        if disagreement_model is not None and getattr(disagreement_model, "fitted", False):
            disagreement_info = disagreement_model.predict(
                elo_margin_calibrated=feat.get("elo_margin_calibrated"),
                model_spread=corrected_spread,
                closing_spread=closing_spread,
                market_spread=market_spread,
                uncertainty=float(feat.get("h_rating_uncertainty", 350) + feat.get("a_rating_uncertainty", 350)),
                spread_move=spread_move,
                h_star_out=float(feat.get("h_star_out", 0) or 0),
                a_star_out=float(feat.get("a_star_out", 0) or 0),
            )
            effective_edge_thr += float(disagreement_info.get("edge_bump", 0.0))

        ho_off = feat.get("h_elo_off", 1500)
        ho_def = feat.get("h_elo_def", 1500)
        ao_off = feat.get("a_elo_off", 1500)
        ao_def = feat.get("a_elo_def", 1500)
        h_o_rd = feat.get("h_rating_uncertainty", 350) / 2
        h_d_rd = h_o_rd
        a_o_rd = feat.get("a_rating_uncertainty", 350) / 2
        a_d_rd = a_o_rd
        pred_poss = feat.get("exp_poss", 100)

        # --- Proper Kelly Fraction (moneyline) ---
        kelly_fraction = 0.0
        if not pd.isna(market_ml) and "win_prob" in preds:

            side, ev_side, dec = select_ml_bet(
                preds["win_prob"], market_ml, min_ev=ML_MIN_EV,
                max_favorite_decimal=max_favorite_decimal,
                min_win_pct=min_ml_win_pct,
            )
            if side != "Pass":
                p = ml_prob_for_side(preds["win_prob"], market_ml, side)
                b = dec - 1
                kelly = (p * b - (1 - p)) / b
            else:
                kelly = 0.0

            # Fractional Kelly (25%) and uncertainty penalty
            FRACTIONAL_K = 0.25
            kelly = max(0.0, min(1.0, kelly * FRACTIONAL_K))

            rating_uncertainty = (h_o_rd + h_d_rd + a_o_rd + a_d_rd) / 4.0
            uncertainty_penalty = max(0.0, 1.0 - (rating_uncertainty - 150) / 400)
            kelly_fraction = kelly * uncertainty_penalty
        else:
            kelly_fraction = 0.0

        # Betting analysis
        rating_uncertainty = float(feat.get("h_rating_uncertainty", 350) + feat.get("a_rating_uncertainty", 350))

        matchup_vol_sigma = (
            float(vol_tracker.matchup_sigma(home_team, away_team))
            if vol_tracker is not None else np.nan
        )

        draft_direction = (
            "Home" if corrected_spread + float(market_spread or 0) > 0 else "Away"
        ) if not pd.isna(market_spread) else "Home"
        ats_prob_preview = None
        if (
            ats_classifier is not None and getattr(ats_classifier, "fitted", False)
            and not pd.isna(market_spread)
        ):
            ats_feat = {**feat, "pred_margin": corrected_spread, "market_spread": market_spread}
            ats_prob_preview = ats_classifier.predict_cover_prob(
                ats_feat, draft_direction, market_spread,
            )

        ba = bet_analysis(
            model_spread=corrected_spread,
            market_spread=market_spread,
            model_win_prob=preds.get("win_prob"),
            market_ml=market_ml,
            min_edge_pts=analysis_min_edge_pts(effective_edge_thr),
            min_ev=ML_MIN_EV,
            conf_width=conf_width,
            spread_move=spread_move,
            public_home_pct=public_home_pct,
            require_rlm=require_rlm,
            confidence_calibrator=confidence_calibrator if should_use_bet_calibrator() else None,
            rating_uncertainty=rating_uncertainty,
            max_favorite_decimal=max_favorite_decimal,
            require_elo_agreement=require_elo_agreement,
            elo_margin_calibrated=feat.get("elo_margin_calibrated"),
            elo_agreement_extra_edge=elo_agreement_extra_edge,
            disagreement_trust=disagreement_info.get("disagreement_trust", 1.0),
            phantom_injury_flag=disagreement_info.get("phantom_injury_flag", False),
            pred_home=cons_home,
            pred_away=cons_away,
            matchup_vol_sigma=matchup_vol_sigma,
            ats_classifier_prob=ats_prob_preview,
        )
        edge_pts = float(ba.get("spread_edge_pts", 0.0) or 0.0)
        if not np.isfinite(edge_pts):
            edge_pts = 0.0
        model_lean = ba.get("spread_direction", "Pass")
        if model_lean == "Pass":
            model_lean = spread_lean_from_edge(edge_pts)
        direction = model_lean
        conf_score = ba.get("spread_confidence", 0)
        conf_raw = ba.get("confidence_score_raw", np.nan)
        stars = ba.get("spread_stars", "No market")
        ml_ev = ba.get("ml_ev", np.nan)
        ml_dir = ba.get("ml_direction", "Pass")

        if direction != "Pass":
            if (edge_pts > 0 and direction != "Home") or (edge_pts < 0 and direction != "Away"):
                direction = spread_lean_from_edge(edge_pts)

        direction = apply_bet_selection_gates(direction, edge_pts, conf_width)

        cover_prob = ba.get("cover_prob_calibrated", 0.524)
        raw_cover = ba.get("spread_cover_prob_raw", cover_prob)
        ats_prob = None
        lean_for_conf = model_lean if model_lean != "Pass" else spread_lean_from_edge(edge_pts)
        if (
            ats_classifier is not None and getattr(ats_classifier, "fitted", False)
            and lean_for_conf != "Pass" and not pd.isna(market_spread)
        ):
            ats_feat = {**feat, "pred_margin": corrected_spread, "market_spread": market_spread}
            ats_prob = ats_classifier.predict_cover_prob(ats_feat, lean_for_conf, market_spread)
            if BET_SELECTION_MODE == "edge_bucket_ats" and ats_prob < ATS_CLASSIFIER_MIN_PROB:
                direction = "Pass"
        blended_cover = raw_cover
        if ats_prob is not None:
            blended_cover = blend_ats_classifier_cover(raw_cover, ats_prob, ATS_CLASSIFIER_BLEND)
        cover_prob = select_cover_prob(
            None,
            raw_cover=raw_cover,
            calibrated_cover=blended_cover,
            ats_classifier_prob=ats_prob,
        )

        if (
            lean_for_conf != "Pass"
            and confidence_calibrator is not None
            and should_use_bet_calibrator()
        ):
            conf_feats = build_confidence_features(
                spread_edge_pts=edge_pts,
                conf_width=conf_width,
                rating_uncertainty=rating_uncertainty,
                elo_meta_agreement=elo_meta_agreement(
                    corrected_spread, market_spread, feat.get("elo_margin_calibrated"),
                ),
                spread_cover_prob=blended_cover,
                matchup_vol_sigma=matchup_vol_sigma,
                disagreement_trust=disagreement_info.get("disagreement_trust", 1.0),
                phantom_injury_flag=disagreement_info.get("phantom_injury_flag", False),
                ats_classifier_prob=ats_prob if ats_prob is not None else ats_prob_preview,
                win_prob=float(preds.get("win_prob", 0.5) or 0.5),
                elo_margin=float(feat.get("elo_margin_calibrated", 0) or 0),
                lean=lean_for_conf,
            )
            cal = confidence_calibrator.predict("ats", conf_feats, direction=lean_for_conf)
            conf_score = int(round(cal["confidence_score"]))
            conf_raw = cal["confidence_score_raw"]
            cover_prob = float(np.clip(cal["calibrated_prob"], 0.01, 0.99))
            conf_tier = cal["confidence_tier"]
            stars = cal["stars"]
        else:
            conf_tier = ba.get("confidence_tier", 2)

        conf_thr = (
            float(min_confidence_threshold)
            if min_confidence_threshold is not None
            else float(MIN_CONFIDENCE_SCORE)
        )
        conf_max = (
            float(max_confidence_threshold)
            if max_confidence_threshold is not None
            else MAX_CONFIDENCE_SCORE
        )
        elo_agree = elo_meta_agreement(
            corrected_spread, market_spread, feat.get("elo_margin_calibrated"),
        )
        if uses_edge_gates():
            actionable = lean_for_conf != "Pass" and direction != "Pass"
        else:
            actionable = passes_confidence_actionable_gates(
                lean=lean_for_conf,
                edge_pts=edge_pts,
                conf_score=conf_score,
                conf_width=conf_width,
                min_confidence=conf_thr,
                max_confidence=conf_max,
                disagreement_trust=disagreement_info.get("disagreement_trust", 1.0),
                phantom_injury_flag=disagreement_info.get("phantom_injury_flag", False),
                market_spread=market_spread,
                elo_meta_agreement=elo_agree,
            )


        clv_pre = np.nan
        if pd.notna(market_spread) and pd.notna(closing_spread):
            clv_pre = compute_clv(corrected_spread, market_spread, closing_spread)
        if (
            REQUIRE_NONNEGATIVE_CLV
            and actionable
            and pd.notna(clv_pre)
            and float(clv_pre) < float(MIN_CLV_POINTS)
        ):
            actionable = False

        if CONFIDENCE_SELECTION_MODE == "min_score" and lean_for_conf != "Pass" and not actionable:
            direction = "Pass"
        elif lean_for_conf != "Pass" and actionable:
            direction = lean_for_conf
        elif lean_for_conf != "Pass" and not actionable:
            direction = "Pass"

        if (
            USE_VENN_ABERS_FILTER
            and direction != "Pass"
            and confidence_calibrator is not None
            and hasattr(confidence_calibrator, "venn_abers_width")
        ):
            va_score = conf_score
            if isinstance(va_score, (int, float)):
                va_w = confidence_calibrator.venn_abers_width(float(va_score))
                if va_w is not None and va_w > VENN_ABERS_MAX_WIDTH:
                    direction = "Pass"
        actionable = int(direction != "Pass")

        elo_meta_agreement_val = elo_meta_agreement(
            corrected_spread, market_spread, feat.get("elo_margin_calibrated"),
        )

        edge_mult = edge_stake_multiplier(abs(edge_pts)) if uses_edge_gates() else 1.0
        edge_bucket_min = edge_bucket_min_for_stakes()
        vol_mult = 1.0
        if STAKE_SIZING_MODE == "volatility_adjusted" and vol_tracker is not None:
            vol_mult = vol_tracker.stake_multiplier(home_team, away_team)

        # Spread Kelly + profile stakes
        spread_kelly = spread_kelly_fraction(cover_prob)
        stakes = {}
        for profile in PROFILES:
            stakes[profile] = compute_stake(
                profile,
                direction=direction,
                edge_pts=edge_pts,
                edge_threshold=effective_edge_thr,
                cover_prob=cover_prob,
                confidence_tier=conf_tier,
                conf_width=conf_width,
                rating_uncertainty=rating_uncertainty,
                edge_stake_mult=edge_mult,
                edge_bucket_min=edge_bucket_min,
                volatility_mult=vol_mult,
                confidence_score=conf_score,
            )

        ml_kelly = compute_ml_stake(
            "moderate",
            ml_direction=ml_dir,
            win_prob=preds.get("win_prob", 0.5),
            market_ml=market_ml,
            confidence_tier=ba.get("ml_confidence_tier", conf_tier),
            max_favorite_decimal=max_favorite_decimal,
        )

        pred_total_val = preds.get("pred_total", 0)
        total_edge = pred_total_val - float(market_total) if pd.notna(market_total) and market_total else 0.0
        ou_dir = "Pass"
        p_over = 0.5
        sigma_total = preds.get("sigma_total")
        if sigma_total is None or not np.isfinite(float(sigma_total or np.nan)):
            t_width = preds.get("total_quantile_width")
            if t_width is not None and np.isfinite(float(t_width)):
                sigma_total = float(t_width) / 2.0
            else:
                sigma_total = float(OU_DEFAULT_SIGMA)
        if ou_bet:
            if OU_BET_ENABLED and pd.notna(market_total) and market_total > 0:
                ou_dir, p_over, total_edge = select_ou_bet(
                    pred_total_val,
                    market_total,
                    sigma_total,
                    min_edge=ou_min_edge,
                )
                if OU_REQUIRE_POSITIVE_CI and ou_dir != "Pass":
                    t_width = preds.get("total_quantile_width")
                    if t_width is not None and np.isfinite(float(t_width)) and float(t_width) > 18.0:
                        ou_dir = "Pass"
                if (
                    ou_dir != "Pass"
                    and confidence_calibrator is not None
                    and getattr(confidence_calibrator, "_fitted", False)
                    and confidence_calibrator.models.get("ou") is not None
                ):
                    ou_feat = {
                        "total_edge": total_edge,
                        "PRED_TOTAL": pred_total_val,
                        "MARKET_TOTAL": float(market_total),
                        "OU_DIRECTION": ou_dir,
                        "WIN_PROB": preds.get("win_prob", 0.5),
                        "CONF_WIDTH": conf_width,
                        "RATING_UNCERTAINTY": rating_uncertainty,
                        "P_OVER": p_over,
                    }
                    ou_cal = confidence_calibrator.predict("ou", ou_feat, direction=ou_dir)
                    if ou_cal.get("calibrated_prob", 0.5) < OU_CALIBRATOR_MIN_PROB:
                        ou_dir = "Pass"
        ou_win = np.nan
        if ou_dir != "Pass" and pd.notna(market_total) and market_total > 0:
            actual_total = act_h + act_a
            ou_win = int((ou_dir == "Over" and actual_total > market_total) or
                         (ou_dir == "Under" and actual_total < market_total))

        ml_explain = explain_ml_bet(preds.get("win_prob", 0.5), market_ml)
        # Prefer explained side when EV mode selected bet; keep ba ml_dir if already set
        if ml_dir == "Pass" and ml_explain.get("ml_bet") != "Pass":
            ml_dir = ml_explain["ml_bet"]
            ml_ev = ml_explain.get("ml_ev", ml_ev)

        row_out = {
            "GAME_ID": game_id,
            "DATE": gdate.date() if is_valid_timestamp(gdate) else None,
            "HOME": home_team, "AWAY": away_team,
            "PRED_HOME": round(preds.get("pred_home", 0), 1),
            "PRED_AWAY": round(preds.get("pred_away", 0), 1),
            "PRED_SPREAD": round(corrected_spread, 2),
            "RAW_PRED_MARGIN": round(raw_pred_margin, 2),
            "PRED_TOTAL": round(preds.get("pred_total", 0), 2),
            "WIN_PROB": round(preds.get("win_prob", 0), 3),
            "WIN_PROB_RAW": round(preds.get("win_prob_raw", preds.get("win_prob", 0)), 3),
            "ACTUAL_HOME": act_h, "ACTUAL_AWAY": act_a,
            "ACTUAL_MARGIN": act_h - act_a,
            "MARKET_SPREAD": market_spread,
            "CLOSING_SPREAD": closing_spread,
            "MARKET_ML": market_ml,
            "MARKET_TOTAL": market_total,
            "EDGE": round(edge_pts, 2),
            "EDGE_LEAN": model_lean,
            "WIN_PCT": conf_score,
            "ACTIONABLE": int(actionable),
            "DIRECTION": direction,
            "CONFIDENCE": conf_score,
            "CONFIDENCE_RAW": round(float(conf_raw), 3) if pd.notna(conf_raw) else np.nan,
            "MATCHUP_VOL_SIGMA": round(float(matchup_vol_sigma), 3) if pd.notna(matchup_vol_sigma) else np.nan,
            "CALIBRATED_COVER_PROB": round(cover_prob, 3),
            "STARS": stars,
            "ML_EV": ml_ev,
            "ML_DIRECTION": ml_dir,
            "OU_DIRECTION": ou_dir,
            "OU_WIN": ou_win,
            "P_OVER": round(float(p_over), 3),
            "SIGMA_TOTAL": round(float(sigma_total), 2) if sigma_total is not None else np.nan,
            "PREDICTED_WINNER": ml_explain.get("predicted_winner", "Pass"),
            "EV_HOME": round(float(ml_explain["ev_home"]), 4) if pd.notna(ml_explain.get("ev_home")) else np.nan,
            "EV_AWAY": round(float(ml_explain["ev_away"]), 4) if pd.notna(ml_explain.get("ev_away")) else np.nan,
            "ML_BET": ml_explain.get("ml_bet", ml_dir),
            "ML_REASON": ml_explain.get("ml_reason", ""),
            "P_HOME": round(float(ml_explain.get("p_home", preds.get("win_prob", 0.5))), 3),
            "P_AWAY": round(float(ml_explain.get("p_away", 1.0 - preds.get("win_prob", 0.5))), 3),
            "CONF_LOWER": round(conf_lower, 2),
            "CONF_UPPER": round(conf_upper, 2),
            "CONF_WIDTH": round(conf_width, 2),
            "KELLY_FRACTION": round(kelly_fraction, 4),
            "SPREAD_KELLY": round(spread_kelly, 4),
            "COVER_PROB_CALIBRATED": round(cover_prob, 3),
            "SPREAD_COVER_PROB_RAW": round(ba.get("spread_cover_prob_raw", cover_prob), 3),
            "CONFIDENCE_TIER": conf_tier,
            "ELO_META_AGREEMENT": elo_meta_agreement_val,
            "RATING_UNCERTAINTY": round(rating_uncertainty, 1),
            "TOTAL_EDGE": round(total_edge, 2),
            "STAKE_CONSERVATIVE": round(stakes.get("conservative", 0), 4),
            "STAKE_MODERATE": round(stakes.get("moderate", 0), 4),
            "STAKE_AGGRESSIVE": round(stakes.get("aggressive", 0), 4),
            "ML_KELLY": round(ml_kelly, 4),
            "EDGE_THRESHOLD": effective_edge_thr,
            "BASE_EDGE_THRESHOLD": min_edge_pts,
            "SPREAD_QUANTILE_WIDTH": round(float(q_width), 2) if q_width is not None and np.isfinite(q_width) else np.nan,
            "PHANTOM_INJURY_FLAG": int(disagreement_info.get("phantom_injury_flag", False)),
            "DISAGREEMENT_TRUST": round(float(disagreement_info.get("disagreement_trust", 1.0)), 3),
            "ELO_MARGIN_CALIBRATED": round(float(feat.get("elo_margin_calibrated", 0) or 0), 2),
            "H_STAR_OUT": feat.get("h_star_out", 0),
            "A_STAR_OUT": feat.get("a_star_out", 0),
            "ATS_CLASSIFIER_PROB": round(float(ats_prob), 3) if ats_prob is not None else np.nan,
            "VOL_STAKE_MULT": round(vol_mult, 3),
            "MODEL_ML_CORRECT": int((preds.get("win_prob", 0.5) > 0.5) == (act_h - act_a > 0)),
            "TOTAL_ERR": round(preds.get("pred_total", 0) - (act_h + act_a), 2),
        }
        results.append(row_out)

        # --- Post‑game updates ---
        seas_prog = 0.5
        h_form = (
            team_form_tracker.get(home_team, current_season, gdate)
            if team_form_tracker is not None else None
        )
        a_form = (
            team_form_tracker.get(away_team, current_season, gdate)
            if team_form_tracker is not None else None
        )
        update_trackers_after_game(
            group=group,
            gs=gs,
            game_id=game_id,
            gdate=gdate,
            home=home_team,
            away=away_team,
            home_starters=actual_home_starters,
            away_starters=actual_away_starters,
            act_h=act_h,
            act_a=act_a,
            home_tot_poss=home_tot_poss,
            away_tot_poss=away_tot_poss,
            home_xppp_game=home_xppp_game,
            away_xppp_game=away_xppp_game,
            current_season=current_season,
            market_spread=market_spread,
            elo_tracker=elo_tracker,
            hier_engine=hier_engine,
            pace_tracker=pace_tracker,
            team_xppp_tracker=team_xppp_tracker,
            team_form_tracker=team_form_tracker,
            rotation_tracker=rotation_tracker,
            lineup_elo_tracker=lineup_elo_tracker,
            chemistry_tracker=chemistry_tracker,
            team_elo_tracker=team_elo_tracker,
            travel_tracker=travel_tracker,
            ref_tracker=ref_tracker,
            shot_quality_tracker=shot_quality_tracker,
            last_game_date=last_date,
            team_game_dates=team_game_dates,
            team_recent_net=team_recent_net,
            opponent_history=opponent_history,
            team_home_margin=team_home_margin,
            team_road_margin=team_road_margin,
            team_games_played=team_games_played,
            team_rosters_seen=team_rosters_seen,
            lineup_cache=lineup_cache,
            season_player_ids=season_player_ids,
            seas_prog=seas_prog,
            ho_off=ho_off,
            ho_def=ho_def,
            ao_off=ao_off,
            ao_def=ao_def,
            h_form=h_form,
            a_form=a_form,
        )

        if calibrator is not None:
            try:
                calibrator.update(corrected_spread, 1 if act_h > act_a else 0,
                                  total=preds.get("pred_total"))
            except TypeError:
                calibrator.update(corrected_spread, 1 if act_h > act_a else 0)

        if spread_calibrator is not None:
            spread_calibrator.update(raw_pred_margin, act_h - act_a)

        if vol_tracker is not None:
            vol_tracker.update_matchup(home_team, away_team, corrected_spread, act_h - act_a)

        resid_hist.append(abs(corrected_spread - (act_h - act_a)))

    if skipped_games:
        print(f"  ℹ️ simulation: skipped {skipped_games} games with unrecoverable dates")

    out = pd.DataFrame(results)
    if not out.empty:
        out["SPREAD_ERR"] = (out["PRED_SPREAD"] - out["ACTUAL_MARGIN"]).abs()
        out["TOTAL_ERR"] = (out["PRED_TOTAL"] - (out["ACTUAL_HOME"] + out["ACTUAL_AWAY"])).abs()
        out = add_clv_columns(out)
    return out


In [ ]:
# ── module: predict ──────────────────────────────────────────────────────────
"""Single-game live prediction using shared build_game_features path."""
from __future__ import annotations

import json
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd



class PredictionContext:
    """Mutable walk-forward state for live predictions (persist via save/load)."""

    def __init__(self):
        self.last_date = {}
        self.team_game_dates = defaultdict(lambda: deque(maxlen=20))
        self.team_recent_net = defaultdict(lambda: deque(maxlen=10))
        self.opponent_history = defaultdict(lambda: deque(maxlen=15))
        self.team_home_margin = defaultdict(lambda: deque(maxlen=20))
        self.team_road_margin = defaultdict(lambda: deque(maxlen=20))
        self.team_games_played = defaultdict(int)
        self.team_rosters_seen = defaultdict(set)
        self.season_start_date = None
        self.lineup_cache = {}

    def save_state(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump(self.__dict__, f)

    @classmethod
    def load_state(cls, path):
        import pickle
        obj = cls()
        with open(path, "rb") as f:
            data = pickle.load(f)
        for k, v in data.items():
            if k.endswith("_history") or k in ("team_game_dates", "team_recent_net",
                                                "team_home_margin", "team_road_margin"):
                setattr(obj, k, defaultdict(lambda: deque(maxlen=20), {
                    tk: deque(tv) for tk, tv in v.items()
                }))
            else:
                setattr(obj, k, v)
        return obj


def load_tuning_config(path: Path | None = None) -> dict:

    path = path or (STATE_DIR / "tuning_results.json")
    if not path.exists():
        return {
            "walkforward_edge_threshold": GOOD_BET_EDGE,
            "max_favorite_decimal": ML_MAX_FAVORITE_DECIMAL_DEFAULT,
            "ou_min_edge": float(OU_MIN_EDGE),
            "recommended_profile": "moderate",
            "elo_calibration": load_elo_knobs().to_dict(),
        }
    cfg = json.loads(path.read_text())
    if "elo_calibration" not in cfg:
        cfg["elo_calibration"] = load_elo_knobs().to_dict()
    return cfg


def _ou_direction_for_predict(preds: dict, feat: dict, cfg: dict) -> str:

    if not OU_BET_ENABLED:
        return "Pass"
    market_total = feat.get("market_total")
    if market_total is None or not np.isfinite(float(market_total or np.nan)) or float(market_total) <= 0:
        return "Pass"
    ou_min = float(cfg.get("ou_min_edge", OU_MIN_EDGE))
    pred_total = float(preds.get("pred_total", 0) or 0)
    sigma = preds.get("sigma_total", OU_DEFAULT_SIGMA)
    direction, _, _ = select_ou_bet(pred_total, market_total, sigma, min_edge=ou_min)
    return direction


def load_score_pair_model(path: Path | None = None):
    path = path or (STATE_DIR / "score_pair.pkl")
    if not path.exists():
        return None
    return MetaScorePairModel.load(path)


def load_live_calibrator(path: Path | None = None):

    path = path or default_calibrator_path()
    cal = WalkForwardBetCalibrator.load(path) if path.exists() else None
    if cal is None and CONFIDENCE_MODE == "unified":
        cal = WalkForwardBetCalibrator()
    if cal is not None and CONFIDENCE_MODE == "unified":
        cal.confidence_weights = load_latest_confidence_weights()
    return cal


def load_min_confidence_threshold(default: float | None = None) -> float:

    default = float(MIN_CONFIDENCE_SCORE if default is None else default)
    tuning_path = STATE_DIR / "tuning_results.json"
    if tuning_path.exists():
        try:
            payload = json.loads(tuning_path.read_text())
            if "suggested_min_confidence" in payload:
                return float(payload["suggested_min_confidence"])
            if "min_confidence_score" in payload:
                return float(payload["min_confidence_score"])
        except (OSError, json.JSONDecodeError, TypeError, ValueError):
            pass
    path = confidence_weights_path()
    if not path.exists():
        return default
    try:
        with open(path) as f:
            payload = json.load(f)
        return float(payload.get("suggested_min_confidence", default))
    except (OSError, json.JSONDecodeError, TypeError, ValueError):
        return default


def load_max_confidence_threshold(default: float | None = None) -> float | None:

    if default is None:
        default = MAX_CONFIDENCE_SCORE
    tuning_path = STATE_DIR / "tuning_results.json"
    if tuning_path.exists():
        try:
            payload = json.loads(tuning_path.read_text())
            if "max_confidence_score" in payload and payload["max_confidence_score"] is not None:
                return float(payload["max_confidence_score"])
        except (OSError, json.JSONDecodeError, TypeError, ValueError):
            pass
    return default


def load_elo_calibrator(path: Path | None = None):
    path = path or default_elo_calibrator_path()
    if path.exists():
        return WalkForwardEloCalibrator.load(path)
    return None


def load_disagreement_model(path: Path | None = None):
    path = path or default_disagreement_path()
    if path.exists():
        return WalkForwardMarketDisagreementModel.load(path)
    return None


def load_total_model(path: Path | None = None):
    path = path or (STATE_DIR / "latest_total.pkl")
    if not path.exists():
        path = STATE_DIR / "total.pkl"
    if path.exists():
        return MetaTotalModel.load(path)
    return None


def load_win_model(path: Path | None = None):
    path = path or (STATE_DIR / "latest_win.pkl")
    if path.exists():
        return MetaWinModel.load(path)
    return None


def load_ats_classifier(path: Path | None = None):
    path = path or (STATE_DIR / "latest_ats.pkl")
    if not path.exists():
        return None
    return ATSClassifier.load(path)


def load_vol_tracker(path: Path | None = None):
    path = path or (STATE_DIR / "latest_vol.pkl")
    if not path.exists():
        return None
    return TeamVolatilityTracker.load_state(path)


def load_ml_calibrator(path: Path | None = None):

    path = path or default_ml_calibrator_path()
    if path.exists():
        return WalkForwardMLCalibrator.load(path)
    return None


def predict_game(
    home_abbr: str,
    away_abbr: str,
    game_date,
    hier_engine,
    elo_tracker,
    meta_model,
    pace_tracker,
    team_xppp_tracker=None,
    team_form_tracker=None,
    spread_calibrator=None,
    rotation_tracker=None,
    lineup_elo_tracker=None,
    chemistry_tracker=None,
    team_elo_tracker=None,
    travel_tracker=None,
    epm_tracker=None,
    ref_tracker=None,
    shot_quality_tracker=None,
    hapm_tracker=None,
    ctx: PredictionContext | None = None,
    last_game_dates: dict | None = None,
    home_starters: list | None = None,
    away_starters: list | None = None,
    inactive_ids: list | None = None,
    live_market_spread=None,
    live_market_ml=None,
    live_market_total=None,
    odds_dict=None,
    win_model=None,
    total_model=None,
    score_pair_model=None,
    ats_classifier=None,
    vol_tracker=None,
    confidence_calibrator=None,
    tuning_config: dict | None = None,
    elo_calibrator=None,
    auto_load_models: bool = True,
):
    """Predict a single upcoming game with full train/serve feature parity."""
    if not getattr(meta_model, "fitted", False):
        raise RuntimeError("MetaScoreModel must be fitted before predicting games.")

    if auto_load_models:
        if win_model is None:
            win_model = load_win_model()
        if total_model is None:
            total_model = load_total_model()
        if score_pair_model is None:
            score_pair_model = load_score_pair_model()
        if ats_classifier is None:
            ats_classifier = load_ats_classifier()
        if vol_tracker is None:
            vol_tracker = load_vol_tracker()

    cfg = tuning_config or load_tuning_config()
    edge_thr = float(
        cfg.get("walkforward_edge_threshold")
        or cfg.get("optimal_bet_edge")
        or GOOD_BET_EDGE
    )
    max_fav = float(cfg.get("max_favorite_decimal", 1.45))
    recommended = cfg.get("recommended_profile", "moderate")
    if elo_calibrator is None:
        elo_calibrator = load_elo_calibrator()

    home = TEAM_MAP.get(home_abbr, home_abbr)
    away = TEAM_MAP.get(away_abbr, away_abbr)
    gdate = pd.Timestamp(game_date)
    ctx = ctx or PredictionContext()
    if last_game_dates:
        ctx.last_date.update(last_game_dates)

    home_starters = home_starters or ctx.lineup_cache.get(home) or ["0"]
    away_starters = away_starters or ctx.lineup_cache.get(away) or ["0"]

    odds_override = {}
    if odds_dict:
        odds_override = dict(odds_dict)
    if live_market_spread is not None:
        key = (gdate.date(), home)
        odds_override[key] = {
            "spread": live_market_spread,
            "ml": live_market_ml,
            "total": live_market_total,
            "closing_spread": live_market_spread,
        }

    feat = build_game_features(
        game_id=f"live_{home}_{away}_{gdate.date()}",
        gdate=gdate,
        home_team=home,
        away_team=away,
        home_starters=home_starters,
        away_starters=away_starters,
        elo_tracker=elo_tracker,
        hier_engine=hier_engine,
        pace_tracker=pace_tracker,
        odds_dict=odds_override or None,
        team_xppp_tracker=team_xppp_tracker,
        team_form_tracker=team_form_tracker,
        rotation_tracker=rotation_tracker,
        lineup_elo_tracker=lineup_elo_tracker,
        chemistry_tracker=chemistry_tracker,
        team_elo_tracker=team_elo_tracker,
        travel_tracker=travel_tracker,
        epm_tracker=epm_tracker,
        ref_tracker=ref_tracker,
        shot_quality_tracker=shot_quality_tracker,
        hapm_tracker=hapm_tracker,
        last_date=ctx.last_date,
        team_game_dates=ctx.team_game_dates,
        team_recent_net=ctx.team_recent_net,
        opponent_history=ctx.opponent_history,
        team_home_margin=ctx.team_home_margin,
        team_road_margin=ctx.team_road_margin,
        team_games_played=ctx.team_games_played,
        team_rosters_seen=ctx.team_rosters_seen,
        season_start_date=ctx.season_start_date or gdate,
        inactive_ids=inactive_ids or [],
        elo_calibrator=elo_calibrator,
    )

    feat_model = build_feature_row(feat)
    preds = meta_model.predict(feat_model)

    corrected = (
        spread_calibrator.correct(preds["pred_margin"])
        if spread_calibrator is not None
        else preds["pred_margin"]
    )

    if score_pair_model is not None and getattr(score_pair_model, "fitted", False):
        pair_out = score_pair_model.predict_scores(feat_model)
        preds["pred_home"] = pair_out["pred_home"]
        preds["pred_away"] = pair_out["pred_away"]
        preds["pred_home_pts"] = pair_out["pred_home"]
        preds["pred_away_pts"] = pair_out["pred_away"]
        preds["sigma_total"] = pair_out.get("sigma_total")
        preds["p_over"] = pair_out.get("p_over")
        if USE_SCORE_PAIR_TOTAL:
            preds["pred_total"] = pair_out["pred_total"]
            # Soft cross-check vs spread model (uncertainty flag only)
            pair_margin = pair_out["pred_margin"]
            preds["score_pair_margin"] = pair_margin
            preds["margin_disagreement"] = abs(float(pair_margin) - float(corrected))
    elif total_model is not None and getattr(total_model, "fitted", False):
        total_out = total_model._predict_raw_total(feat_model)
        preds["pred_total"] = total_out["pred_total"]
        if "total_quantile_width" in total_out:
            preds["total_quantile_width"] = total_out["total_quantile_width"]
            preds["sigma_total"] = float(total_out["total_quantile_width"]) / 2.0
        preds["pred_home"] = (preds["pred_total"] + corrected) / 2.0
        preds["pred_away"] = (preds["pred_total"] - corrected) / 2.0
        preds["pred_home_pts"] = preds["pred_home"]
        preds["pred_away_pts"] = preds["pred_away"]
    else:
        preds.setdefault("pred_home_pts", preds.get("pred_home"))
        preds.setdefault("pred_away_pts", preds.get("pred_away"))
        preds.setdefault("sigma_total", float(OU_DEFAULT_SIGMA))

    if preds.get("p_over") is None and feat_model.get("market_total"):
        sig = preds.get("sigma_total", OU_DEFAULT_SIGMA)
        preds["p_over"] = total_over_prob_gaussian(
            preds.get("pred_total", 0), feat_model.get("market_total"), sig,
        )


    use_meta_win = (
        win_model is not None and getattr(win_model, "fitted", False)
        and (
            WIN_PROB_SOURCE == "meta_win"
            or (WIN_PROB_SOURCE == "auto" and REQUIRE_META_WIN_WHEN_FITTED)
        )
    )
    if use_meta_win:
        raw_wp = win_model.predict_proba(feat_model, pred_margin=corrected)
        preds["win_prob_raw"] = raw_wp
        ml_cal = load_ml_calibrator()
        preds["win_prob"] = (
            ml_cal.transform(raw_wp)
            if ml_cal is not None and getattr(ml_cal, "_fitted", False)
            else raw_wp
        )
    elif WIN_PROB_SOURCE == "rolling_platt":
        preds["win_prob"] = meta_model.calibrate_prob(corrected)

    if spread_calibrator is not None and hasattr(spread_calibrator, "predict_interval"):
        conf_lower, conf_upper, conf_width = spread_calibrator.predict_interval(corrected, alpha=0.10)
    else:
        conf_lower, conf_upper, conf_width = corrected - 12.0, corrected + 12.0, 24.0

    q_width = preds.get("spread_quantile_width")
    if q_width is not None and np.isfinite(q_width):
        conf_width = float(q_width)
        if preds.get("spread_q10") is not None and preds.get("spread_q90") is not None:
            conf_lower = float(preds["spread_q10"])
            conf_upper = float(preds["spread_q90"])

    market_spread = feat.get("market_spread", live_market_spread)
    market_ml = feat.get("market_ml", live_market_ml)
    rating_uncertainty = float(feat.get("h_rating_uncertainty", 350) + feat.get("a_rating_uncertainty", 350))

    effective_edge_thr = variance_aware_edge_threshold(edge_thr, conf_width)
    disagreement_info = {}
    disagreement_model = load_disagreement_model()
    if disagreement_model is not None and disagreement_model.fitted:
        disagreement_info = disagreement_model.predict(
            elo_margin_calibrated=feat.get("elo_margin_calibrated"),
            model_spread=corrected,
            closing_spread=feat.get("closing_spread", market_spread),
            market_spread=market_spread,
            uncertainty=rating_uncertainty,
            spread_move=feat.get("spread_move", 0),
            h_star_out=float(feat.get("h_star_out", 0) or 0),
            a_star_out=float(feat.get("a_star_out", 0) or 0),
        )
        effective_edge_thr += float(disagreement_info.get("edge_bump", 0.0))



    min_conf = load_min_confidence_threshold()
    max_conf = load_max_confidence_threshold()

    matchup_vol_sigma = (
        float(vol_tracker.matchup_sigma(home, away)) if vol_tracker is not None else np.nan
    )

    calibrator = confidence_calibrator or load_live_calibrator()
    ba = bet_analysis(
        model_spread=corrected,
        market_spread=market_spread,
        model_win_prob=preds.get("win_prob"),
        market_ml=market_ml,
        min_edge_pts=analysis_min_edge_pts(effective_edge_thr),
        spread_move=feat.get("spread_move", 0),
        public_home_pct=feat.get("public_home_pct", 0.5),
        conf_width=conf_width,
        confidence_calibrator=calibrator if should_use_bet_calibrator() else None,
        rating_uncertainty=rating_uncertainty,
        max_favorite_decimal=max_fav,
        elo_margin_calibrated=feat.get("elo_margin_calibrated"),
        disagreement_trust=disagreement_info.get("disagreement_trust", 1.0),
        phantom_injury_flag=disagreement_info.get("phantom_injury_flag", False),
        pred_home=(preds.get("pred_total", 220) + corrected) / 2.0,
        pred_away=(preds.get("pred_total", 220) - corrected) / 2.0,
        matchup_vol_sigma=matchup_vol_sigma,
    )

    edge_pts = float(ba.get("spread_edge_pts", 0.0) or 0.0)
    if not np.isfinite(edge_pts):
        edge_pts = 0.0
    model_lean = ba.get("spread_direction", "Pass")
    if model_lean == "Pass":
        model_lean = spread_lean_from_edge(edge_pts)
    direction = model_lean
    conf_score = ba.get("spread_confidence", 0)
    conf_raw = ba.get("confidence_score_raw", np.nan)
    direction = apply_bet_selection_gates(direction, edge_pts, conf_width)

    lean_for_conf = model_lean if model_lean != "Pass" else spread_lean_from_edge(edge_pts)
    raw_cover = ba.get("spread_cover_prob_raw", ba.get("cover_prob_calibrated", 0.5))
    ats_prob = None
    if ats_classifier is None:
        ats_classifier = load_ats_classifier()
    if (
        ats_classifier is not None and getattr(ats_classifier, "fitted", False)
        and lean_for_conf != "Pass" and market_spread is not None and pd.notna(market_spread)
    ):
        ats_feat = {**feat, "pred_margin": corrected, "market_spread": market_spread}
        ats_prob = ats_classifier.predict_cover_prob(ats_feat, lean_for_conf, market_spread)
        if BET_SELECTION_MODE == "edge_bucket_ats" and ats_prob < ATS_CLASSIFIER_MIN_PROB:
            direction = "Pass"
    blended_cover = raw_cover
    if ats_prob is not None:
        blended_cover = blend_ats_classifier_cover(raw_cover, ats_prob, ATS_CLASSIFIER_BLEND)

    if lean_for_conf != "Pass" and calibrator is not None and should_use_bet_calibrator():
        conf_feats = build_confidence_features(
            spread_edge_pts=edge_pts,
            conf_width=conf_width,
            rating_uncertainty=rating_uncertainty,
            elo_meta_agreement=elo_meta_agreement(
                corrected, market_spread, feat.get("elo_margin_calibrated"),
            ),
            spread_cover_prob=blended_cover,
            matchup_vol_sigma=matchup_vol_sigma,
            disagreement_trust=disagreement_info.get("disagreement_trust", 1.0),
            phantom_injury_flag=disagreement_info.get("phantom_injury_flag", False),
            ats_classifier_prob=ats_prob,
            win_prob=float(preds.get("win_prob", 0.5) or 0.5),
            elo_margin=float(feat.get("elo_margin_calibrated", 0) or 0),
            lean=lean_for_conf,
        )
        cal = calibrator.predict("ats", conf_feats, direction=lean_for_conf)
        conf_score = int(round(cal["confidence_score"]))
        conf_raw = cal["confidence_score_raw"]
        cover_prob = float(np.clip(cal["calibrated_prob"], 0.01, 0.99))
        conf_tier = cal["confidence_tier"]
    else:
        cover_prob = ba.get("cover_prob_calibrated", 0.5)
        conf_tier = ba.get("confidence_tier", 2)

    elo_agree = elo_meta_agreement(
        corrected, market_spread, feat.get("elo_margin_calibrated"),
    )
    if uses_edge_gates():
        actionable = lean_for_conf != "Pass" and direction != "Pass"
    else:
        actionable = passes_confidence_actionable_gates(
            lean=lean_for_conf,
            edge_pts=edge_pts,
            conf_score=conf_score,
            conf_width=conf_width,
            min_confidence=min_conf,
            max_confidence=max_conf,
            disagreement_trust=disagreement_info.get("disagreement_trust", 1.0),
            phantom_injury_flag=disagreement_info.get("phantom_injury_flag", False),
            market_spread=market_spread,
            elo_meta_agreement=elo_agree,
        )

    closing_spread = feat.get("closing_spread", market_spread)

    clv_pre = np.nan
    if pd.notna(market_spread) and pd.notna(closing_spread):
        clv_pre = compute_clv(corrected, market_spread, closing_spread)
    if (
        REQUIRE_NONNEGATIVE_CLV
        and actionable
        and pd.notna(clv_pre)
        and float(clv_pre) < float(MIN_CLV_POINTS)
    ):
        actionable = False

    if CONFIDENCE_SELECTION_MODE == "min_score" and lean_for_conf != "Pass" and not actionable:
        direction = "Pass"
    elif lean_for_conf != "Pass" and actionable:
        direction = lean_for_conf
    elif lean_for_conf != "Pass":
        direction = "Pass"

    edge_mult = edge_stake_multiplier(abs(edge_pts)) if uses_edge_gates() else 1.0
    edge_bucket_min = edge_bucket_min_for_stakes()
    vol_mult = 1.0
    if STAKE_SIZING_MODE == "volatility_adjusted" and vol_tracker is not None:
        vol_mult = vol_tracker.stake_multiplier(home, away)

    if (
        USE_VENN_ABERS_FILTER
        and direction != "Pass"
        and calibrator is not None
        and hasattr(calibrator, "venn_abers_width")
    ):
        va_w = calibrator.venn_abers_width(float(conf_score))
        if va_w is not None and va_w > VENN_ABERS_MAX_WIDTH:
            direction = "Pass"
    actionable = bool(direction != "Pass")

    stakes = {}
    for profile in PROFILES:
        stakes[f"stake_{profile}"] = round(
            compute_stake(
                profile,
                direction=direction,
                edge_pts=edge_pts,
                edge_threshold=effective_edge_thr,
                cover_prob=cover_prob,
                confidence_tier=conf_tier,
                conf_width=conf_width,
                rating_uncertainty=rating_uncertainty,
                edge_stake_mult=edge_mult,
                edge_bucket_min=edge_bucket_min,
                volatility_mult=vol_mult,
                confidence_score=conf_score,
            ),
            4,
        )


    ml_explain = explain_ml_bet(preds.get("win_prob", 0.5), market_ml)
    sigma_t = preds.get("sigma_total", OU_DEFAULT_SIGMA)
    ou_dir, p_over, _ = select_ou_bet(
        preds.get("pred_total", 0),
        feat.get("market_total", live_market_total),
        sigma_t,
        min_edge=float(cfg.get("ou_min_edge", OU_MIN_EDGE)),
    )
    if preds.get("p_over") is not None:
        p_over = preds["p_over"]

    return {
        "date": str(gdate.date()),
        "home": home,
        "away": away,
        "pred_spread": round(corrected, 2),
        "pred_margin": round(corrected, 2),
        "pred_total": round(preds.get("pred_total", 0), 1),
        "pred_home": round(float(preds.get("pred_home", 0) or 0), 1),
        "pred_away": round(float(preds.get("pred_away", 0) or 0), 1),
        "pred_home_pts": round(float(preds.get("pred_home_pts", preds.get("pred_home", 0)) or 0), 1),
        "pred_away_pts": round(float(preds.get("pred_away_pts", preds.get("pred_away", 0)) or 0), 1),
        "sigma_total": round(float(sigma_t), 2) if sigma_t is not None else None,
        "p_over": round(float(p_over), 3) if p_over is not None else None,
        "p_under": round(1.0 - float(p_over), 3) if p_over is not None else None,
        "win_prob": round(preds.get("win_prob", 0.5), 3),
        "p_home": round(float(ml_explain.get("p_home", preds.get("win_prob", 0.5))), 3),
        "p_away": round(float(ml_explain.get("p_away", 1.0 - preds.get("win_prob", 0.5))), 3),
        "predicted_winner": ml_explain.get("predicted_winner", "Pass"),
        "ev_home": round(float(ml_explain["ev_home"]), 4) if pd.notna(ml_explain.get("ev_home")) else None,
        "ev_away": round(float(ml_explain["ev_away"]), 4) if pd.notna(ml_explain.get("ev_away")) else None,
        "ml_bet": ml_explain.get("ml_bet", "Pass"),
        "ml_reason": ml_explain.get("ml_reason", ""),
        "edge": round(edge_pts if np.isfinite(edge_pts) else spread_edge(corrected, market_spread) or 0, 2),
        "edge_lean": model_lean,
        "win_pct": conf_score,
        "actionable": actionable,
        "direction": direction,
        "stars": ba.get("spread_stars", ""),
        "confidence_score": conf_score,
        "win_probability_pct": conf_score,
        "confidence_score_raw": round(float(conf_raw), 3) if pd.notna(conf_raw) else None,
        "matchup_vol_sigma": round(float(matchup_vol_sigma), 3) if pd.notna(matchup_vol_sigma) else None,
        "confidence_tier": conf_tier,
        "cover_prob_calibrated": round(cover_prob, 3),
        "ou_direction": ou_dir,
        "ml_direction": ml_explain.get("ml_bet", ba.get("ml_direction", "Pass")),
        "edge_threshold": effective_edge_thr,
        "base_edge_threshold": edge_thr,
        "spread_quantile_width": round(float(q_width), 2) if q_width is not None and np.isfinite(q_width) else None,
        "phantom_injury_flag": bool(disagreement_info.get("phantom_injury_flag", False)),
        "disagreement_trust": round(float(disagreement_info.get("disagreement_trust", 1.0)), 3),
        "margin_disagreement": round(float(preds.get("margin_disagreement", 0) or 0), 2),
        "recommended_profile": recommended,
        "stake_recommended": stakes.get(f"stake_{recommended}", 0),
        **stakes,
        "features": feat,
    }


In [ ]:
# ── module: backtest ─────────────────────────────────────────────────────────
"""Walk-forward backtest orchestration."""
from __future__ import annotations

import copy
import json

import numpy as np
import pandas as pd



def _series_last_optional_float(df: pd.DataFrame, col: str, default=None):
    if col not in df.columns or df.empty:
        return default
    val = df[col].iloc[-1]
    if val is None or (isinstance(val, float) and (pd.isna(val) or not np.isfinite(val))):
        return default
    try:
        return float(val)
    except (TypeError, ValueError):
        return default



__all__ = [
    "run_multi_year_backtest_walkforward",
    "benchmark_results",
    "grid_search_bet_edge",
    "walkforward_edge_threshold",
    "walkforward_min_confidence",
    "apply_edge_threshold",
]


def _persist_backtest_metadata(extra: dict) -> None:
    """Save run metadata; works in package and flat Colab namespace."""
    fn = globals().get("save_backtest_metadata")
    if callable(fn):
        fn(extra=extra)
        return
    try:
        import importlib
        mod = importlib.import_module("pipeline.artifacts")
        mod.save_backtest_metadata(extra=extra)
    except (ImportError, AttributeError):
        pass


def run_multi_year_backtest_walkforward(
    stints_df: pd.DataFrame,
    odds_dict: dict = None,
    n_tuning_trials_elo: int = 30,
    n_tuning_trials_hier: int = 30,
    n_tuning_trials_meta: int = 40,
    n_tuning_trials_total: int = 15,
    n_tuning_trials_meta_win: int | None = None,
    rolling_window_size: int = 3,
    model_kwargs: dict = None,
    feature_cols: list = None,
    walkforward_edge: bool = True,
    pace_per_team: bool = True,
    spread_calib_mode: str = "linear",
    variance_aware_winprob: bool = True,
    use_tuning_cache: bool = True,
    train_win_model: bool = True,
    tune_total_head: bool = True,
    fast_tuning: bool = False,
    require_elo_agreement: bool = False,
    use_hapm_priors: bool = False,
) -> pd.DataFrame:
    """
    True walk-forward backtest with NO data leakage.
    """
    rolling_window_size = int(max(2, min(3, rolling_window_size)))
    if n_tuning_trials_meta_win is None:
        n_tuning_trials_meta_win = min(n_tuning_trials_meta, 25)
    stints_df = stints_df.copy()
    stints_df["game_date"] = pd.to_datetime(stints_df["game_date"], errors="coerce")
    if "season" not in stints_df.columns:
        stints_df['season'] = stints_df['game_date'].dt.year + (stints_df['game_date'].dt.month >= 9).astype(int)

    all_seasons = sorted(stints_df['season'].dropna().unique())
    print(f"\n🚀 Walk-forward over {len(all_seasons)} seasons: {all_seasons}")
    print(f"   Rolling window size = {rolling_window_size} season(s)")

    compiled_results = []
    phase2a_rows: list[dict] = []
    confidence_weight_center = dict(DEFAULT_CONFIDENCE_WEIGHTS)
    param_bounds = AdaptiveParameterBounds()
    elo_bounds = AdaptiveParameterBounds()
    hier_bounds = AdaptiveParameterBounds()
    bet_calibrator = WalkForwardBetCalibrator(method=CONFIDENCE_CALIB_METHOD)
    ml_calibrator = WalkForwardMLCalibrator(method=ML_CALIB_METHOD)
    disagreement_model = WalkForwardMarketDisagreementModel()

    if model_kwargs is None:
        model_kwargs = {
            "total_mode": "external",
            "use_elo_stack": True,
            "train_target": "close_residual",
            "use_quantile_heads": True,
            "prediction_mode": "absolute",
        }

    full_elo = full_hier = meta_model = win_model = total_model = score_pair_model = ats_classifier = None
    sim_pace = sim_xppp = sim_form = sim_rotation = None
    sim_lineup_elo = sim_chemistry = sim_team_elo = sim_travel = sim_epm = None
    sim_vol = None
    last_good_total_params = None

    for i, test_season in enumerate(all_seasons):
        print(f"\n{'='*65}")
        print(f"🚀 SIMULATING SEASON {int(test_season)-1}-{int(test_season)}")
        print('='*65)

        train_seasons = all_seasons[max(0, i - rolling_window_size):i]
        if not train_seasons:
            print("  [!] No previous seasons – skipping (no training data).")
            continue

        print(f"  Training on seasons: {train_seasons}")
        train_stints = stints_df[stints_df['season'].isin(train_seasons)].copy()
        test_stints = stints_df[stints_df['season'] == test_season].copy()

        # Tune Elo / Hier (with cache)
        best_elo = load_cached("elo", train_seasons, train_stints) if use_tuning_cache else None
        if best_elo is None:
            print("  ⏳ Tuning Elo tracker on past data...")
            best_elo, elo_cv = tune_elo_tracker(
                train_stints, DEFAULT_LEAGUE_XPPP,
                n_trials=n_tuning_trials_elo, bounds=elo_bounds,
            )
            if use_tuning_cache:
                save_cached("elo", train_seasons, best_elo, train_stints)
            if elo_cv < TUNING_INVALID_SCORE:
                elo_bounds.record_best(int(test_season), best_elo)
                elo_bounds.report()
            else:
                print("  ⚠ Skipping Elo adaptive bounds (CV did not run).")
        else:
            print("  ⚡ Using cached Elo params")
            elo_bounds.record_best(int(test_season), best_elo)

        best_hier = load_cached("hier", train_seasons, train_stints) if use_tuning_cache else None
        if best_hier is None:
            print("  ⏳ Tuning Hierarchical engine on past data...")
            best_hier, hier_cv = tune_hierarchical(
                train_stints, n_trials=n_tuning_trials_hier, bounds=hier_bounds,
            )
            if use_tuning_cache:
                save_cached("hier", train_seasons, best_hier, train_stints)
            if hier_cv < TUNING_INVALID_SCORE:
                hier_bounds.record_best(int(test_season), best_hier)
                hier_bounds.report()
            else:
                print("  ⚠ Skipping Hierarchical adaptive bounds (CV did not run).")
        else:
            print("  ⚡ Using cached Hierarchical params")
            hier_bounds.record_best(int(test_season), best_hier)

        game_dates = (
            train_stints[['game_date', 'GAME_ID']]
            .drop_duplicates('GAME_ID')
            .sort_values('game_date')
        )
        split_idx = int(len(game_dates) * 0.8)
        base_game_ids = game_dates.iloc[:split_idx]['GAME_ID'].values
        calib_game_ids = game_dates.iloc[split_idx:]['GAME_ID'].values
        base_stints = train_stints[train_stints['GAME_ID'].isin(base_game_ids)].copy()
        calib_stints = train_stints[train_stints['GAME_ID'].isin(calib_game_ids)].copy()
        print(f"  Base games: {len(base_game_ids)} | Calibration games: {len(calib_game_ids)}")

        base_hier = HierarchicalPossessionEngine(**best_hier)
        base_elo = PlayerRatingTracker(config=best_elo, league_xppp=DEFAULT_LEAGUE_XPPP)
        base_pace = PaceTracker(team_window=10, league_window=100, per_team=pace_per_team)
        base_xppp = TeamXpppTracker(window_size=40, prev_season_weight=0.5)
        base_form = TeamFormTracker(window=15, prev_season_weight=0.4)
        base_rotation = RotationLineupTracker(window_games=10, top_n=8)
        base_lineup_elo = LineupEloTracker()
        base_chemistry = ChemistryTracker()
        base_team_elo = TeamEloTracker()
        base_travel = TravelTracker()
        base_ref = RefTracker()
        base_shot_quality = ShotQualityTracker()
        base_hapm = HapmPriorTracker()
        if use_hapm_priors:
            # `base_hapm` is fit once on the *entire* train_stints span for
            # legitimate use later against `test_season` only (strictly
            # after every game in train_stints — no leak there). It must
            # NOT be used as a static lookup for games *inside* train_stints
            # itself (that was the `hapm_before_split_leak`: an early
            # base/calib-slice game's HAPM feature would then reflect
            # dyad/trio coefficients estimated from later games in the same
            # span). `hapm_train_lookup` below is the leak-free replacement
            # used for base_features/calib_features generation.
            base_hapm.fit(train_stints)
        hapm_train_lookup = hapm_lookup_for_training(train_stints) if use_hapm_priors else None
        base_epm = EpmPriorTracker()

        base_features = generate_features(
            base_stints, base_hier, base_elo, base_pace,
            odds_dict=odds_dict, update_engines=True,
            team_xppp_tracker=base_xppp, team_form_tracker=base_form,
            rotation_tracker=base_rotation, lineup_elo_tracker=base_lineup_elo,
            chemistry_tracker=base_chemistry, team_elo_tracker=base_team_elo,
            travel_tracker=base_travel, epm_tracker=base_epm,
            ref_tracker=base_ref,
            shot_quality_tracker=base_shot_quality,
            hapm_tracker=hapm_train_lookup,
        )
        base_features = engineer_interaction_features(base_features)

        calib_hier = copy.deepcopy(base_hier)
        calib_elo = copy.deepcopy(base_elo)
        calib_pace = copy.deepcopy(base_pace)
        calib_xppp = copy.deepcopy(base_xppp)
        calib_form = copy.deepcopy(base_form)
        calib_rotation = copy.deepcopy(base_rotation)
        calib_lineup_elo = copy.deepcopy(base_lineup_elo)
        calib_chemistry = copy.deepcopy(base_chemistry)
        calib_team_elo = copy.deepcopy(base_team_elo)
        calib_travel = copy.deepcopy(base_travel)
        calib_ref = copy.deepcopy(base_ref)
        calib_shot_quality = copy.deepcopy(base_shot_quality)
        calib_hapm = copy.deepcopy(base_hapm)
        calib_epm = copy.deepcopy(base_epm)

        calib_features = generate_features(
            calib_stints, calib_hier, calib_elo, calib_pace,
            odds_dict=odds_dict, update_engines=True,
            team_xppp_tracker=calib_xppp, team_form_tracker=calib_form,
            rotation_tracker=calib_rotation, lineup_elo_tracker=calib_lineup_elo,
            chemistry_tracker=calib_chemistry, team_elo_tracker=calib_team_elo,
            travel_tracker=calib_travel, epm_tracker=calib_epm,
            ref_tracker=calib_ref,
            shot_quality_tracker=calib_shot_quality,
            # Same past-only lookup as base_features (Task hapm_before_split_leak
            # fix): calib_stints is a later slice of the same train_stints span,
            # so `hapm_train_lookup` still resolves every game to a tracker
            # fit only on strictly-earlier blocks.
            hapm_tracker=hapm_train_lookup,
        )
        calib_features = engineer_interaction_features(calib_features)

        # Task calibration_slice_reuse fix: Elo isotonic, MetaScore isotonic,
        # static spread calibration, MetaWin isotonic, and ATS isotonic
        # previously all fit on the identical tail `calib_features` slice.
        # `CalibrationSliceRegistry` raises immediately if that ever
        # regresses; `chronological_game_id_partition` gives each of the
        # five its own disjoint, chronologically-contiguous sub-slice of the
        # same calibration-tail span (still strictly before test_season, so
        # no chronological leakage — only the cross-target row-reuse is
        # fixed). `_calib_slice(target)` below re-derives each target's rows
        # from the *current* `calib_features` (before/after Elo calibration
        # is applied) using this same fixed GAME_ID partition.
        calib_slice_registry = CalibrationSliceRegistry()
        calib_id_partition = chronological_game_id_partition(
            calib_features, targets=BACKTEST_CALIBRATION_TARGETS,
        )
        for _target, _ids in calib_id_partition.items():
            calib_slice_registry.register(_target, _ids)
        calib_slice_registry.assert_all_targets_registered(BACKTEST_CALIBRATION_TARGETS)

        def _calib_slice(target: str) -> pd.DataFrame:
            """The disjoint calibration sub-slice for `target`, taken from
            the *current* (possibly Elo-calibrated) `calib_features` frame —
            same GAME_ID partition throughout, so re-slicing after
            `apply_elo_calibration_df` picks up its new/overwritten columns
            without re-registering (and therefore without risking a second
            registry entry for the same target)."""
            return slice_by_game_ids(calib_features, calib_id_partition[target])

        elo_knobs = tune_elo_calibrator(base_features, calib_df=_calib_slice("elo_isotonic"))
        elo_calibrator = WalkForwardEloCalibrator(knobs=elo_knobs)
        elo_calibrator.fit(base_features, calib_df=_calib_slice("elo_isotonic"))
        if elo_calibrator.fitted:
            print(
                f"  📐 ELO 3-group calibrator: eps={elo_knobs.huber_epsilon} "
                f"blend=({elo_knobs.hier_blend_elo:.2f},{elo_knobs.hier_blend_hier:.2f}) "
                f"MAE={elo_calibrator.knobs.group_train_mae.get('combined', 0):.2f}"
            )
        base_features = apply_elo_calibration_df(base_features, elo_calibrator)
        calib_features = apply_elo_calibration_df(calib_features, elo_calibrator)

        best_params = load_cached("meta", train_seasons, train_stints) if use_tuning_cache else None
        if best_params is None:
            print("  ⏳ Tuning MetaScoreModel hyperparameters...")
            best_params = tune_margin_model(
                base_features, n_trials=n_tuning_trials_meta, feature_cols=feature_cols,
                bounds=param_bounds, season=int(test_season),
                fast_mode=fast_tuning or n_tuning_trials_meta <= 5,
                train_target=(model_kwargs or {}).get("train_target", "close_residual"),
            )
            if use_tuning_cache:
                save_cached("meta", train_seasons, best_params, train_stints)
            param_bounds.record_best(int(test_season), meta_params_for_bounds(best_params))
        else:
            print("  ⚡ Using cached Meta params")
            param_bounds.record_best(int(test_season), meta_params_for_bounds(best_params))

        elo_knobs.elo_blend_alpha = best_params.get(
            "elo_blend_alpha", elo_knobs.elo_blend_alpha,
        )
        elo_knobs.elo_ridge_alpha = best_params.get(
            "elo_ridge_alpha", elo_knobs.elo_ridge_alpha,
        )

        mk = dict(model_kwargs)
        meta_model = MetaScoreModel(
            ridge_alpha=best_params['ridge_alpha'],
            cb_params=best_params['cb_params'],
            huber_epsilon=best_params['huber_epsilon'],
            margin_cap=best_params.get('margin_cap'),
            use_isotonic_calibration=True,
            feature_cols=feature_cols,
            elo_blend_alpha=best_params.get(
                'elo_blend_alpha', elo_knobs.elo_blend_alpha,
            ),
            elo_ridge_alpha=best_params.get(
                'elo_ridge_alpha', elo_knobs.elo_ridge_alpha,
            ),
            **mk,
        )
        meta_model.fit(
            base_features, base_features['actual_home'], base_features['actual_away'],
            calib_df=_calib_slice("metascore_isotonic"),
        )

        if tune_total_head:
            total_params = load_cached("total", train_seasons, train_stints) if use_tuning_cache else None
            if total_params is None:
                print("  ⏳ Tuning total model...")
                total_params = tune_total_model(
                    base_features,
                    n_trials=n_tuning_trials_total,
                    feature_cols=total_feature_cols(base_features),
                    bounds=param_bounds,
                    season=int(test_season),
                    fast_mode=fast_tuning or n_tuning_trials_total <= 5,
                )
                if use_tuning_cache:
                    save_cached("total", train_seasons, total_params, train_stints)
                param_bounds.record_best(int(test_season), meta_params_for_bounds(total_params))
            else:
                print("  ⚡ Using cached total params")
                param_bounds.record_best(int(test_season), meta_params_for_bounds(total_params))
            total_params = dict(total_params)
            cv_score = float(total_params.pop("_cv_score", total_params.pop("_cv_mae", np.nan)))
            apply_total = True
            if np.isfinite(cv_score) and cv_score > float(TOTAL_HEAD_CV_SANITY_CAP):
                print(
                    f"  ⚠ Total-model CV score {cv_score:.1f} exceeds cap "
                    f"({TOTAL_HEAD_CV_SANITY_CAP}) — skipping total refit"
                )
                apply_total = False
                if last_good_total_params is not None:
                    total_params = dict(last_good_total_params)
                    apply_total = True
            elif np.isfinite(cv_score):
                last_good_total_params = dict(total_params)
            if apply_total and total_params:
                total_model = MetaTotalModel(
                    ridge_alpha=total_params.get("ridge_alpha", 10.0),
                    cb_params=total_params.get("cb_params"),
                    huber_epsilon=total_params.get("huber_epsilon", 1.35),
                    feature_cols=total_params.get("feature_cols") or total_feature_cols(base_features),
                    total_elo_beta=total_params.get("total_elo_beta", 0.25),
                    train_target=total_params.get("train_target", "absolute"),
                )
                total_model.fit(base_features)

            # Dual-head home/away score model (specialized totals path)
            score_pair_model = MetaScorePairModel(
                feature_cols=total_feature_cols(base_features),
                train_target="absolute",
            )
            try:
                score_pair_model.fit(base_features)
                if total_model is not None and getattr(total_model, "fitted", False):
                    # Align sigma with standalone total error when available
                    pass
            except Exception as e:
                print(f"  ⚠ MetaScorePairModel fit skipped: {e}")
                score_pair_model = None

        win_model = None
        ats_classifier = None
        if train_win_model:
            train_feats = base_features.copy()
            # Task calibration_slice_reuse fix: `static_cal` (the "static
            # spread calibration" target) is fit only on its own disjoint
            # slice — not the same rows used below for the MetaWin/ATS
            # isotonic calibrators.
            calib_feats = _calib_slice("static_spread_calibration")
            train_raw = meta_model._predict_raw(train_feats)["pred_margin"]
            calib_raw = meta_model._predict_raw(calib_feats)["pred_margin"]
            # Task 054: replay the rolling calibrator row-by-row over the
            # calib slice (predict, then observe) instead of fitting once on
            # the whole slice and blanket-correcting it with the *final*
            # state — the blanket pattern lets each row's correction see
            # later rows in the same slice, which disagrees with how
            # pipeline/simulate.py calibrates at live inference time.
            calib_actual = (
                calib_feats["actual_margin"].values
                if "actual_margin" in calib_feats.columns
                else np.full(len(calib_feats), np.nan)
            )
            calib_corrected, static_cal = replay_rolling_spread_calibration(
                calib_raw, calib_actual,
                window=max(SPREAD_CALIB_WINDOW, len(calib_feats)),
                min_samples=min(30, max(10, len(calib_feats) // 3)),
                mode=spread_calib_mode,
            )
            calib_feats["pred_margin"] = calib_corrected
            # train_feats predates calib_feats and was never used to fit
            # static_cal, so applying its fully-replayed end state here is
            # not self-referential (no row corrects itself with its own
            # future information).
            train_feats["pred_margin"] = [static_cal.correct(float(p)) for p in train_raw]

            win_params = load_cached("meta_win", train_seasons, train_stints) if use_tuning_cache else None
            if win_params is None and n_tuning_trials_meta_win > 0:
                print("  ⏳ Tuning MetaWin model...")
                win_params = tune_meta_win_model(
                    train_feats,
                    n_trials=n_tuning_trials_meta_win,
                    season=int(test_season),
                    pred_margin_col="pred_margin",
                    fast_mode=fast_tuning or n_tuning_trials_meta_win <= 5,
                )
                if use_tuning_cache:
                    save_cached("meta_win", train_seasons, win_params, train_stints)
            elif win_params is None:
                win_params = {
                    "C": 0.1,
                    "elo_win_blend": None,
                    "calib_method": "isotonic",
                    "feature_cols": win_feature_cols(train_feats),
                }
            else:
                print("  ⚡ Using cached MetaWin params")

            win_model = MetaWinModel(
                C=win_params.get("C", 0.1),
                feature_cols=win_params.get("feature_cols") or win_feature_cols(train_feats),
                elo_win_blend=win_params.get("elo_win_blend"),
                calib_method=win_params.get("calib_method", "isotonic"),
            )
            # Task calibration_slice_reuse fix: MetaWin isotonic gets its own
            # disjoint slice (not `calib_feats`, which fit `static_cal`
            # above). Its `pred_margin` is produced by *applying* (not
            # re-fitting) the already-fit `static_cal` correction — this is
            # not self-referential, since `static_cal` was fit on a
            # completely different set of rows.
            win_calib_df = _calib_slice("metawin_isotonic").copy()
            if not win_calib_df.empty:
                win_raw = meta_model._predict_raw(win_calib_df)["pred_margin"]
                win_calib_df["pred_margin"] = [static_cal.correct(float(p)) for p in win_raw]
            win_model.fit(
                train_feats,
                (train_feats["actual_home"] > train_feats["actual_away"]).astype(int),
                calib_df=win_calib_df,
            )

            ats_classifier = ATSClassifier(feature_cols=feature_cols)
            ats_train = train_feats.copy()
            # Task calibration_slice_reuse fix: ATS isotonic gets its own
            # disjoint slice, distinct from both `calib_feats`
            # (static_spread_calibration) and `win_calib_df` (metawin_isotonic)
            # above.
            ats_calib = _calib_slice("ats_isotonic").copy()
            ats_train["pred_margin"] = train_feats["pred_margin"]
            if not ats_calib.empty:
                ats_raw = meta_model._predict_raw(ats_calib)["pred_margin"]
                ats_calib["pred_margin"] = [static_cal.correct(float(p)) for p in ats_raw]
            if "market_spread" not in ats_train.columns and "closing_spread" in ats_train.columns:
                ats_train["market_spread"] = ats_train["closing_spread"]
                if "closing_spread" in ats_calib.columns:
                    ats_calib["market_spread"] = ats_calib["closing_spread"]
            ats_classifier.fit(ats_train, calib_df=ats_calib)

        # Reuse post-calibration engines (skip redundant third feature pass)
        full_hier = calib_hier
        full_elo = calib_elo
        sim_pace = copy.deepcopy(calib_pace)
        sim_xppp = copy.deepcopy(calib_xppp)
        sim_form = copy.deepcopy(calib_form)
        sim_rotation = copy.deepcopy(calib_rotation)
        sim_lineup_elo = copy.deepcopy(calib_lineup_elo)
        sim_chemistry = copy.deepcopy(calib_chemistry)
        sim_team_elo = copy.deepcopy(calib_team_elo)
        sim_travel = copy.deepcopy(calib_travel)
        sim_ref = copy.deepcopy(calib_ref)
        sim_shot_quality = copy.deepcopy(calib_shot_quality)
        sim_hapm = copy.deepcopy(calib_hapm)
        sim_vol = TeamVolatilityTracker()
        sim_epm = copy.deepcopy(calib_epm)

        if i > 0:
            returning_ids = set(full_elo.players.keys()) if full_elo.players else None
            full_hier.offseason_revert()
            full_elo.offseason_revert(returning_player_ids=returning_ids)

        # Walk-forward thresholds from prior seasons
        edge_thr = 0.0 if not uses_edge_gates() else GOOD_BET_EDGE
        fav_dec = ML_MAX_FAVORITE_DECIMAL_DEFAULT
        ou_thr = float(OU_MIN_EDGE)
        conf_thr = float(MIN_CONFIDENCE_SCORE)
        conf_max = MAX_CONFIDENCE_SCORE
        min_ml_thr = float(MIN_ML_WIN_PCT)
        if not compiled_results and should_use_season1_calibrator():
            calib_for_conf = calib_features.copy()
            calib_raw = meta_model._predict_raw(calib_for_conf)["pred_margin"]
            # Task 054: same rolling replay fix as above — no blanket
            # self-referential correction of the calibration slice.
            calib_for_conf_actual = (
                calib_for_conf["actual_margin"].values
                if "actual_margin" in calib_for_conf.columns
                else np.full(len(calib_for_conf), np.nan)
            )
            calib_for_conf_corrected, _ = replay_rolling_spread_calibration(
                calib_raw, calib_for_conf_actual,
                window=max(SPREAD_CALIB_WINDOW, len(calib_for_conf)),
                min_samples=min(30, max(10, len(calib_for_conf) // 3)),
                mode=spread_calib_mode,
            )
            calib_for_conf["pred_margin"] = calib_for_conf_corrected
            season_lbl = f"{int(test_season)-1}-{int(test_season)}-calib"
            calib_ats_df = build_calib_split_ats_frame(
                calib_for_conf,
                pred_margin_col="pred_margin",
                season_label=season_lbl,
            )
            if len(calib_ats_df) >= 30:
                bet_calibrator.fit(
                    calib_ats_df,
                    method=CONFIDENCE_CALIB_METHOD,
                    scope=CONFIDENCE_CALIB_SCOPE,
                )
                conf_thr, conf_max = walkforward_confidence_gate(
                    calib_ats_df,
                    default_min=MIN_CONFIDENCE_SCORE,
                    default_max=MAX_CONFIDENCE_SCORE,
                    bet_calibrator=bet_calibrator if bet_calibrator._fitted else None,
                )
                print(
                    f"  📐 Season-1 calibrator fit on calib split "
                    f"({len(calib_ats_df)} games, min_confidence={conf_thr:.0f}"
                    f"{f'-{conf_max:.0f}' if conf_max is not None else ''})"
                )
        if i > 0 and compiled_results:
            prior = pd.concat(compiled_results, ignore_index=True)
            prior_year = compiled_results[-1]
            if uses_edge_gates():
                wf_floor = WALKFORWARD_EDGE_MIN_FLOOR
                if wf_floor is None and BET_SELECTION_MODE != "legacy_tiers":
                    wf_floor = 5.5
                edge_thr = walkforward_edge_threshold(
                    prior, default=GOOD_BET_EDGE, optimize="bucket_roi", min_floor=wf_floor,
                )
            else:
                edge_thr = 0.0
            fav_dec = walkforward_favorite_decimal(
                prior, default=ML_MAX_FAVORITE_DECIMAL_DEFAULT,
            )
            ou_thr = walkforward_ou_edge(prior, default=float(OU_MIN_EDGE))
            calib_source = prior_year
            train_lbl = (
                prior_year["simulated_season_window"].iloc[0]
                if "simulated_season_window" in prior_year.columns and not prior_year.empty
                else "prior-year"
            )
            test_lbl = f"{int(test_season)-1}-{int(test_season)}"
            if CONFIDENCE_MODE == "unified":
                bet_calibrator, p2a_meta = fit_walkforward_confidence_calibrator(
                    calib_source,
                    train_season_label=str(train_lbl),
                    test_season_label=test_lbl,
                    season_index=len(compiled_results),
                    weight_center=confidence_weight_center,
                )
                phase2a_rows.append(p2a_meta)
                confidence_weight_center = shrink_weight_center(
                    confidence_weight_center,
                    p2a_meta.get("suggested_weights", confidence_weight_center),
                )
                conf_thr = float(p2a_meta.get("train_min_confidence", MIN_CONFIDENCE_SCORE))
                conf_max = p2a_meta.get("train_max_confidence", MAX_CONFIDENCE_SCORE)
                if conf_max is not None:
                    conf_max = float(conf_max)
                feat_w = p2a_meta.get("feature_weights") or logistic_feature_importance(bet_calibrator)
                if feat_w:
                    top = sorted(feat_w.items(), key=lambda kv: abs(kv[1]), reverse=True)[:5]
                    coefs = ", ".join(f"{k}={v:+.2f}" for k, v in top)
                    print(
                        f"  📐 Phase 2a inline ({p2a_meta.get('calib_method', CONFIDENCE_CALIB_METHOD)}): "
                        f"{coefs}  C={p2a_meta.get('logistic_c', bet_calibrator.logistic_c):.2f}"
                    )
            else:
                bet_calibrator.fit(
                    calib_source,
                    method=CONFIDENCE_CALIB_METHOD,
                    scope=CONFIDENCE_CALIB_SCOPE,
                )
                conf_thr, conf_max = walkforward_confidence_gate(
                    prior_year,
                    default_min=MIN_CONFIDENCE_SCORE,
                    default_max=MAX_CONFIDENCE_SCORE,
                    bet_calibrator=bet_calibrator if bet_calibrator._fitted else None,
                )
            ml_calibrator.fit(prior_year, scope=ML_CALIB_SCOPE)
            disagreement_model.fit(prior)
            ml_tag = f"ml={ML_CALIB_METHOD}" if ml_calibrator._fitted else "ml=raw"
            print(
                f"  📊 Walk-forward edge={edge_thr}  max_fav_dec={fav_dec:.2f}  "
                f"ou_edge={ou_thr}  min_confidence={conf_thr:.0f}"
                f"{f'-{conf_max:.0f}' if conf_max is not None else ''}  "
                f"min_ml_win%={min_ml_thr:.0f}  "
                f"confidence={CONFIDENCE_MODE}/{CONFIDENCE_CALIB_METHOD}  {ml_tag} (fit {train_lbl})"
            )

        rolling_calibrator = RollingPlattCalibrator(
            window_size=300, min_samples=20,
            cold_start_fn=meta_model.calibrate_prob,
            variance_aware=variance_aware_winprob,
        )
        spread_calibrator = SpreadCalibrator(window=SPREAD_CALIB_WINDOW, mode=spread_calib_mode)

        results = run_simulation(
            season_df=test_stints,
            hier_engine=full_hier,
            elo_tracker=full_elo,
            meta_model=meta_model,
            pace_tracker=sim_pace,
            odds_dict=odds_dict,
            calibrator=rolling_calibrator,
            team_xppp_tracker=sim_xppp,
            team_form_tracker=sim_form,
            spread_calibrator=spread_calibrator,
            rotation_tracker=sim_rotation,
            lineup_elo_tracker=sim_lineup_elo,
            chemistry_tracker=sim_chemistry,
            team_elo_tracker=sim_team_elo,
            travel_tracker=sim_travel,
            epm_tracker=sim_epm,
            ref_tracker=sim_ref,
            shot_quality_tracker=sim_shot_quality,
            hapm_tracker=sim_hapm if use_hapm_priors else None,
            min_edge_pts=edge_thr,
            max_favorite_decimal=fav_dec,
            ou_min_edge=ou_thr,
            confidence_calibrator=bet_calibrator if bet_calibrator._fitted else None,
            win_model=win_model,
            total_model=total_model,
            score_pair_model=score_pair_model,
            ats_classifier=ats_classifier,
            vol_tracker=sim_vol,
            elo_calibrator=elo_calibrator if elo_calibrator.fitted else None,
            require_elo_agreement=require_elo_agreement,
            disagreement_model=disagreement_model if disagreement_model.fitted else None,
            min_confidence_threshold=conf_thr,
            max_confidence_threshold=conf_max,
            ml_calibrator=ml_calibrator if ml_calibrator._fitted else None,
            min_ml_win_pct=min_ml_thr,
        )

        print(f"  Season {test_season} produced {len(results)} rows." if results is not None else "  Season returned None.")
        if results is not None and not results.empty:
            results['simulated_season_window'] = f"{int(test_season)-1}-{int(test_season)}"
            results['EDGE_THRESHOLD'] = edge_thr
            results['MAX_FAVORITE_DECIMAL'] = fav_dec
            results['OU_MIN_EDGE'] = ou_thr
            results['MIN_CONFIDENCE_SCORE'] = conf_thr
            results['MAX_CONFIDENCE_SCORE'] = conf_max
            results['MIN_ML_WIN_PCT'] = min_ml_thr
            results['PHASE2A_INLINE'] = int(
                CONFIDENCE_MODE == "unified" and i > 0 and len(compiled_results) > 0
            )
            results['CONFIDENCE_CALIB_METHOD'] = CONFIDENCE_CALIB_METHOD
            if bet_calibrator._fitted:
                results['PHASE2A_LOGISTIC_C'] = float(bet_calibrator.logistic_c)
            compiled_results.append(results)
        else:
            print(f"  ⚠️ No results for season {test_season}")

    if not compiled_results:
        return pd.DataFrame()

    final = pd.concat(compiled_results, ignore_index=True)

    if walkforward_edge and uses_edge_gates():
        print("\n  (Diagnostic) Post-hoc edge threshold comparison:")
        for idx, season_res in enumerate(compiled_results):
            if idx == 0:
                thr = GOOD_BET_EDGE
            else:
                prior = pd.concat(compiled_results[:idx], ignore_index=True)
                thr = walkforward_edge_threshold(prior, default=GOOD_BET_EDGE, optimize="clv_roi")
            win = season_res.get("simulated_season_window", pd.Series(["?"])).iloc[0]
            print(f"    Season {win}: post-hoc threshold = {thr}")

    final = add_all_profile_columns(final)

    if compiled_results and not final.empty:
        prefix = STATE_DIR / "latest"
        try:
            full_elo.save_state(prefix.with_name(prefix.name + "_elo.pkl"))
            full_hier.save_state(prefix.with_name(prefix.name + "_hier.pkl"))
            if hasattr(sim_pace, "save_state"):
                sim_pace.save_state(prefix.with_name(prefix.name + "_pace.pkl"))
            meta_model.save(prefix.with_name(prefix.name + "_meta.pkl"))
            if win_model is not None:
                win_model.save(prefix.with_name(prefix.name + "_win.pkl"))
            if total_model is not None and getattr(total_model, "fitted", False):
                total_model.save(prefix.with_name(prefix.name + "_total.pkl"))
                total_model.save(STATE_DIR / "total.pkl")
            if score_pair_model is not None and getattr(score_pair_model, "fitted", False):
                score_pair_model.save(prefix.with_name(prefix.name + "_score_pair.pkl"))
                score_pair_model.save(STATE_DIR / "score_pair.pkl")
            if ats_classifier is not None and getattr(ats_classifier, "fitted", False):
                ats_classifier.save(prefix.with_name(prefix.name + "_ats.pkl"))
            sim_vol.save_state(prefix.with_name(prefix.name + "_vol.pkl"))
            sim_rotation.save_state(prefix.with_name(prefix.name + "_rotation.pkl"))
            sim_lineup_elo.save_state(prefix.with_name(prefix.name + "_lineup_elo.pkl"))
            sim_chemistry.save_state(prefix.with_name(prefix.name + "_chemistry.pkl"))
            sim_team_elo.save_state(prefix.with_name(prefix.name + "_team_elo.pkl"))
            sim_travel.save_state(prefix.with_name(prefix.name + "_travel.pkl"))
            bet_calibrator.save(default_calibrator_path())
            if ml_calibrator._fitted:
                ml_calibrator.save(default_ml_calibrator_path())
            if disagreement_model.fitted:
                disagreement_model.save(default_disagreement_path())
            if elo_calibrator.fitted:
                elo_calibrator.save(default_elo_calibrator_path())
            save_elo_knobs(elo_knobs)
            elo_knobs_merged = elo_knobs.to_dict()
            elo_knobs_merged["elo_blend_alpha"] = float(meta_model.elo_blend_alpha)
            elo_knobs_merged["elo_ridge_alpha"] = float(meta_model.elo_ridge_alpha)
            edge_val = float(final["EDGE_THRESHOLD"].iloc[-1]) if "EDGE_THRESHOLD" in final.columns else GOOD_BET_EDGE
            import subprocess
            git_hash = ""
            try:
                git_hash = subprocess.check_output(
                    ["git", "rev-parse", "--short", "HEAD"],
                    cwd=str(STATE_DIR.parent),
                    stderr=subprocess.DEVNULL,
                    text=True,
                ).strip()
            except Exception:
                pass
            om = {}
            if not final.empty:
                m = compute_metrics(final)
                overall = m[m["season"] == "ALL"]
                if not overall.empty:
                    om = overall.iloc[0].to_dict()
            tuning_state = {
                "walkforward_edge_threshold": edge_val,
                "optimal_bet_edge": edge_val,
                "max_favorite_decimal": float(final["MAX_FAVORITE_DECIMAL"].iloc[-1]) if "MAX_FAVORITE_DECIMAL" in final.columns else 1.45,
                "ou_min_edge": float(final["OU_MIN_EDGE"].iloc[-1]) if "OU_MIN_EDGE" in final.columns else float(OU_MIN_EDGE),
                "min_confidence_score": _series_last_optional_float(
                    final, "MIN_CONFIDENCE_SCORE", float(MIN_CONFIDENCE_SCORE),
                ),
                "max_confidence_score": _series_last_optional_float(
                    final, "MAX_CONFIDENCE_SCORE", MAX_CONFIDENCE_SCORE,
                ),
                "min_ml_win_pct": float(final["MIN_ML_WIN_PCT"].iloc[-1]) if "MIN_ML_WIN_PCT" in final.columns else float(MIN_ML_WIN_PCT),
                "last_walkforward_season": final["simulated_season_window"].iloc[-1],
                "phase2a_inline": bool(phase2a_rows),
                "confidence_calib_method": CONFIDENCE_CALIB_METHOD,
                "elo_calibration": elo_knobs_merged,
                "config_flags": {
                    "BET_SELECTION_MODE": BET_SELECTION_MODE,
                    "CONFIDENCE_SELECTION_MODE": CONFIDENCE_SELECTION_MODE,
                    "CONFIDENCE_MIN_EDGE": CONFIDENCE_MIN_EDGE,
                    "CALIBRATION_MODE": CALIBRATION_MODE,
                    "STAKE_SIZING_MODE": STAKE_SIZING_MODE,
                    "WIN_PROB_SOURCE": WIN_PROB_SOURCE,
                    "ARTIFACT_SCHEMA_VERSION": ARTIFACT_SCHEMA_VERSION,
                },
                "git_hash": git_hash,
                "seasons": list(final["simulated_season_window"].unique()) if "simulated_season_window" in final.columns else [],
                "overall_metrics": om,
            }
            if "MIN_CONFIDENCE_SCORE" in final.columns:
                tuning_state["min_confidence_by_season"] = (
                    final.groupby("simulated_season_window", dropna=False)["MIN_CONFIDENCE_SCORE"]
                    .first()
                    .astype(float)
                    .to_dict()
                )
            if phase2a_rows:
                tuning_state["unified_weight_tuning"] = phase2a_rows
                latest_p2a = phase2a_rows[-1]
                tuning_state["suggested_weights_latest"] = latest_p2a.get(
                    "suggested_weights", dict(DEFAULT_CONFIDENCE_WEIGHTS),
                )
                tuning_state["feature_weights_latest"] = latest_p2a.get("feature_weights", {})
                tuning_state["suggested_logistic_c"] = latest_p2a.get("logistic_c")
                if latest_p2a.get("train_min_confidence") is not None:
                    tuning_state["suggested_min_confidence"] = float(
                        latest_p2a["train_min_confidence"],
                    )
                save_phase2a_walkforward_state(phase2a_rows, latest_center=confidence_weight_center)
            (STATE_DIR / "tuning_results.json").write_text(json.dumps(tuning_state, indent=2, default=str))
            (STATE_DIR / "run_manifest.json").write_text(json.dumps(tuning_state, indent=2, default=str))
            print(f"  💾 Saved engine state to {STATE_DIR}/latest_*.pkl")
            _persist_backtest_metadata({
                "n_games": int(len(final)),
                "last_walkforward_season": final["simulated_season_window"].iloc[-1],
            })
        except Exception as e:  # noqa: BLE001
            print(f"  ⚠️ Could not save engine state: {e}")

    return final


In [ ]:
# ── module: ablation ─────────────────────────────────────────────────────────
"""Walk-forward ablation harness.

Runs the full walk-forward backtest under several configurations and reports,
per season and overall: spread MAE, total MAE, ATS win%, ROI (-110), and Brier.

Because the market already prices most box-score signal, *more features is not
automatically better* - this harness exists so every change is kept only if it
improves out-of-sample. Intended to run where the PBP data lives (e.g. Colab).

Acceptance gate (pooled ALL season row):
  - ATS ROI improves OR stays within 0.3% while spread MAE improves
  - Never flip default if spread MAE worsens by >0.1 without ROI gain

Example
-------
>>> from pipeline.ablation import run_ablation
>>> summary, detail = run_ablation(all_stints, modern_odds_dict,
...                                n_tuning_trials_meta=10)
>>> summary   # one row per (config, season) + overall
"""
from __future__ import annotations

import numpy as np
import pandas as pd


BREAKEVEN = 110.0 / 210.0

# Walk-forward validation checklist (Phase 6) — compare configs on these metrics.
VALIDATION_CHECKLIST = """
Elo-first validation checklist (run via run_ablation):
  1. Ablation grid: baseline_current vs edge_bucket vs edge_bucket_ats vs simplified_calib
  2. Primary (ATS): spread MAE, ATS win%, ROI, mean CLV, CLV-weighted ROI
  3. Secondary (ML): ML winner accuracy, ML ROI, Brier on win prob
  4. Tertiary (O/U): total MAE, O/U ROI when ou_bet enabled
  5. Elo diagnostics: elo_vs_market distribution, raw_spread_mae vs spread_mae gap
  6. Clear state/tuning_cache when Elo update logic changes materially

Default flip gate (passes_ablation_gate):
  - pooled ROI delta >= -0.003 OR (MAE improves and ROI within 0.3%)
  - spread_mae must not worsen by >0.1 unless ROI improves materially
  - tertiary: ML Brier, total MAE, CRPS — never raw hit-rate
  - use promote_ablation_winners(summary_df) after a grid run; apply to config after review
"""


def _without(cols, drop):
    drop = set(drop)
    return [c for c in cols if c not in drop]


def _config_overrides(**kwargs) -> dict:
    """Build ablation spec with optional config flag overrides (runtime only)."""
    return kwargs


def calibration_ablation_configs() -> dict:
    """Cover-probability / calibration mode experiments (require full re-run)."""
    base = default_configs()
    keys = ("baseline_current", "simplified_calib", "beta_ats_calib", "skellam_cover")
    return {k: base[k] for k in keys if k in base}


def default_configs() -> dict:
    """Named configurations.

    Each value is a dict with optional keys:
      - feature_cols: feature subset (None = full SAFE_FEATURE_COLS)
      - model_kwargs: passed to MetaScoreModel (e.g. margin_cap, total_mode)
      - backtest_kwargs: passed to the backtest (e.g. walkforward_edge)
      - config_overrides: dict of pipeline.config attribute overrides applied at runtime

    Note: behavioural fixes that are now global defaults (per-team pace,
    walk-forward zone PPS, the linear spread calibrator) are present in *every*
    config; toggle-able groups below isolate feature blocks and bet selection.
    """
    legacy_feats = CORE_FEATURE_COLS + FORM_FEATURE_COLS_BASE
    elo_heavy_drop = [
        "hier_net", "hier_margin",
        "h_hier_off", "h_hier_def", "a_hier_off", "a_hier_def",
    ]
    hier_heavy_drop = [
        "elo_net", "elo_margin",
        "h_elo_off", "h_elo_def", "a_elo_off", "a_elo_def",
        "elo_diff_off", "elo_diff_def",
        *ELO_INTERACTION_COLS,
    ]
    form_only_feats = (
        FORM_FEATURE_COLS + CONTEXT_FEATURE_COLS + SCHEDULE_FEATURE_COLS
        + HCA_FEATURE_COLS + MARKET_TOTAL_COLS
    )
    return {
        # Reference snapshot of current production defaults (Phase 0 baseline).
        "baseline_current": {},
        # TRUE legacy baseline: legacy features, fixed 225 total, +/-20 cap,
        # fixed 2.5 edge, AND the global fixes (per-team pace, linear calib,
        # variance-aware win-prob) all turned OFF.
        "baseline_legacy": {"feature_cols": legacy_feats,
                            "model_kwargs": {"margin_cap": 20.0, "total_mode": "fixed"},
                            "backtest_kwargs": {"walkforward_edge": False,
                                                "pace_per_team": False,
                                                "spread_calib_mode": "additive",
                                                "variance_aware_winprob": False}},
        # All changes on.
        "improved_full": {},
        # Bet selection experiments (Phase 1+3) — applied via config_overrides at runtime.
        "edge_bucket": {"config_overrides": {
            "BET_SELECTION_MODE": "edge_bucket",
            "USE_TIER_STAKE_GATES": False,
            "WALKFORWARD_EDGE_MIN_FLOOR": 5.5,
        }},
        "edge_bucket_ats": {"config_overrides": {
            "BET_SELECTION_MODE": "edge_bucket_ats",
            "USE_TIER_STAKE_GATES": False,
            "WALKFORWARD_EDGE_MIN_FLOOR": 5.5,
        }},
        "simplified_calib": {"config_overrides": {
            "CALIBRATION_MODE": "simplified",
            "REQUIRE_META_WIN_WHEN_FITTED": True,
        }},
        "beta_ats_calib": {"config_overrides": {
            "CALIBRATION_MODE": "beta_ats",
            "REQUIRE_META_WIN_WHEN_FITTED": True,
        }},
        "volatility_staking": {"config_overrides": {
            "STAKE_SIZING_MODE": "volatility_adjusted",
        }},
        "skellam_cover": {"config_overrides": {"USE_SKELLAM_COVER_PROB": True}},
        "lgbm_stack": {"model_kwargs": {"use_lightgbm_base": True}},
        "garbage_form": {"config_overrides": {"USE_GARBAGE_WEIGHTED_FORM": True}},
        "decayed_sos": {"config_overrides": {"USE_DECAYED_SOS": True}},
        # Improved minus each new feature group, to isolate incremental value.
        "no_four_factors": {"feature_cols": _without(SAFE_FEATURE_COLS, FOUR_FACTOR_COLS)},
        "no_pythagorean": {"feature_cols": _without(SAFE_FEATURE_COLS, PYTHAG_COLS)},
        "no_opp_adj": {"feature_cols": _without(SAFE_FEATURE_COLS, OPP_ADJ_COLS)},
        "no_hca": {"feature_cols": _without(SAFE_FEATURE_COLS, HCA_FEATURE_COLS)},
        "no_context": {"feature_cols": _without(SAFE_FEATURE_COLS, CONTEXT_FEATURE_COLS)},
        "no_engine_margin": {"feature_cols": _without(SAFE_FEATURE_COLS, ENGINE_MARGIN_COLS)},
        "no_schedule": {"feature_cols": _without(SAFE_FEATURE_COLS, SCHEDULE_FEATURE_COLS)},
        # Isolate the prediction fixes (real total + wider cap) on legacy features.
        "fixes_only": {"feature_cols": legacy_feats},
        # Full features but fixed 2.5 edge (isolates walk-forward edge selection).
        "fixed_edge_2p5": {"backtest_kwargs": {"walkforward_edge": False}},
        # Isolate each global fix (everything else stays improved).
        "legacy_pace": {"backtest_kwargs": {"pace_per_team": False}},
        "additive_calib": {"backtest_kwargs": {"spread_calib_mode": "additive"}},
        "no_variance_winprob": {"backtest_kwargs": {"variance_aware_winprob": False}},
        # Engine / feature isolation configs (Phase 0 measurement).
        "elo_heavy": {"feature_cols": _without(SAFE_FEATURE_COLS, elo_heavy_drop)},
        "hier_heavy": {"feature_cols": _without(SAFE_FEATURE_COLS, hier_heavy_drop)},
        "form_only": {"feature_cols": form_only_feats},
        "no_elo_interactions": {"feature_cols": _without(SAFE_FEATURE_COLS, ELO_INTERACTION_COLS)},
        # Total-head ablations (Phase 1).
        "fixed_total_225": {"model_kwargs": {"total_mode": "fixed", "league_avg_total": 225.0}},
        "model_total_market": {"model_kwargs": {"total_mode": "model"}},
        # Elo-first two-stage stack (Phase 3).
        "elo_heavy_stack": {"model_kwargs": {"use_elo_stack": True}},
        "no_elo_stack": {"model_kwargs": {"use_elo_stack": False}},
        # Market-residual prediction (Phase 4).
        "residual_blend": {"model_kwargs": {"prediction_mode": "blend", "residual_alpha": 0.5}},
        "residual_full": {"model_kwargs": {"prediction_mode": "residual", "residual_alpha": 0.7}},
        "purged_cv": {"model_kwargs": {"use_purged_cv": True}},
        "shuffled_kfold_cv": {"model_kwargs": {"use_purged_cv": False}},
        # Elo-first betting accuracy experiments
        "dynamic_elo_blend": {"model_kwargs": {"dynamic_elo_blend": True}},
        "no_dynamic_elo_blend": {"model_kwargs": {"dynamic_elo_blend": False}},
        "elo_gated_bets": {"backtest_kwargs": {"require_elo_agreement": True}},
        "elo_total_anchor": {"model_kwargs": {"total_elo_beta": 0.35}},
        "standalone_total_model": {
            "model_kwargs": {"total_mode": "external"},
            "backtest_kwargs": {"tune_total_head": True},
        },
        "ml_isotonic_calib": {"config_overrides": {"ML_CALIB_METHOD": "isotonic"}},
        "ml_no_spread_feature": {"config_overrides": {"ML_USE_SPREAD_FEATURES": False}},
        "total_market_residual": {"config_overrides": {"TOTAL_TRAIN_TARGET": "market_residual"}},
        "no_shot_quality": {"feature_cols": _without(SAFE_FEATURE_COLS, SHOT_QUALITY_COLS)},
        "no_lineup_composite": {"feature_cols": _without(SAFE_FEATURE_COLS, LINEUP_COMPOSITE_COLS)},
        "hapm_priors": {"backtest_kwargs": {"use_hapm_priors": True}},
        # Specialized market pipeline v2
        "gaussian_ou_gates": {"config_overrides": {
            "OU_USE_GAUSSIAN_PROB": True,
            "OU_MIN_PROB": 0.55,
        }},
        "score_pair_total": {"config_overrides": {"USE_SCORE_PAIR_TOTAL": True}},
        "ml_both_sides_ev": {"config_overrides": {"ML_BET_PREDICTED_WINNER_ONLY": False}},
        "ml_winner_only": {"config_overrides": {"ML_BET_PREDICTED_WINNER_ONLY": True}},
        "stacked_lgbm_catboost": {"model_kwargs": {"use_lightgbm_base": True}},
        "no_elo_on_totals": {"config_overrides": {"TOTAL_ELO_BETA": 0.0}},
    }


def ml_ablation_configs() -> dict:
    base = default_configs()
    keys = ("baseline_current", "ml_isotonic_calib", "ml_no_spread_feature")
    return {k: base[k] for k in keys if k in base}


def total_ablation_configs() -> dict:
    base = default_configs()
    keys = ("baseline_current", "standalone_total_model", "total_market_residual", "elo_total_anchor")
    return {k: base[k] for k in keys if k in base}


def passes_ablation_gate(
    baseline: dict,
    candidate: dict,
    *,
    roi_tol: float = 0.003,
    mae_tol: float = 0.1,
    ml_brier_tol: float = 0.01,
    total_mae_tol: float = 0.5,
    crps_tol: float = 0.5,
) -> bool:
    """Return True if candidate meets pooled acceptance criteria vs baseline.

    Hit-rate / raw accuracy is never a promotion criterion. Primary gates:
    ATS ROI + spread MAE; tertiary: ML Brier, total MAE/CRPS.
    """
    b_roi = float(baseline.get("roi", np.nan))
    c_roi = float(candidate.get("roi", np.nan))
    b_mae = float(baseline.get("spread_mae", np.nan))
    c_mae = float(candidate.get("spread_mae", np.nan))
    if not np.isfinite(c_roi) or not np.isfinite(c_mae):
        return False
    roi_ok = c_roi >= b_roi - roi_tol or (c_mae < b_mae and c_roi >= b_roi - roi_tol)
    mae_regression = c_mae > b_mae + mae_tol
    roi_gain = c_roi > b_roi + roi_tol
    if mae_regression and not roi_gain:
        return False
    if not roi_ok:
        return False
    b_ml_brier = baseline.get("ml_brier", baseline.get("brier"))
    c_ml_brier = candidate.get("ml_brier", candidate.get("brier"))
    if b_ml_brier is not None and c_ml_brier is not None:
        if np.isfinite(b_ml_brier) and np.isfinite(c_ml_brier):
            if c_ml_brier > float(b_ml_brier) + ml_brier_tol and not roi_gain:
                return False
    b_total_mae = baseline.get("total_mae")
    c_total_mae = candidate.get("total_mae")
    if b_total_mae is not None and c_total_mae is not None:
        if np.isfinite(b_total_mae) and np.isfinite(c_total_mae):
            if c_total_mae > float(b_total_mae) + total_mae_tol and not roi_gain:
                return False
    b_crps = baseline.get("crps")
    c_crps = candidate.get("crps")
    if b_crps is not None and c_crps is not None:
        if np.isfinite(b_crps) and np.isfinite(c_crps):
            if c_crps > float(b_crps) + crps_tol and not roi_gain:
                return False
    return True


def recommend_default_flips(summary_df: pd.DataFrame, baseline_name: str = "baseline_current") -> list[str]:
    """List config names that pass the gate vs baseline (does not mutate config.py)."""
    if summary_df is None or summary_df.empty:
        return []
    pooled = summary_df[summary_df["season"] == "ALL"]
    base_row = pooled[pooled["config"] == baseline_name]
    if base_row.empty:
        return []
    baseline = base_row.iloc[0].to_dict()
    passed = []
    for cfg in pooled["config"].unique():
        if cfg == baseline_name:
            continue
        row = pooled[pooled["config"] == cfg].iloc[0].to_dict()
        if passes_ablation_gate(baseline, row):
            passed.append(cfg)
    return passed


# Config override keys that are safe to promote into pipeline.config when ablation passes.
_PROMOTABLE_OVERRIDES = {
    "gaussian_ou_gates": {"OU_USE_GAUSSIAN_PROB": True, "OU_MIN_PROB": 0.55},
    "ml_isotonic_calib": {"ML_CALIB_METHOD": "isotonic"},
    "ml_both_sides_ev": {"ML_BET_PREDICTED_WINNER_ONLY": False},
    "simplified_calib": {"CALIBRATION_MODE": "simplified"},
    "beta_ats_calib": {"CALIBRATION_MODE": "beta_ats"},
    "skellam_cover": {"USE_SKELLAM_COVER_PROB": True},
    "total_market_residual": {"TOTAL_TRAIN_TARGET": "market_residual"},
    "no_elo_on_totals": {"TOTAL_ELO_BETA": 0.0},
    "score_pair_total": {"USE_SCORE_PAIR_TOTAL": True},
    "lgbm_stack": {"USE_LIGHTGBM_BASE": True},
    "stacked_lgbm_catboost": {"USE_LIGHTGBM_BASE": True},
}


def promote_ablation_winners(
    summary_df: pd.DataFrame,
    baseline_name: str = "baseline_current",
    *,
    apply: bool = False,
) -> dict:
    """Recommend (and optionally apply in-process) config flips from ablation.

    Does not rewrite config.py on disk — returns the mapping and, if apply=True,
    patches pipeline.config for the current process (useful in notebooks).
    """
    winners = recommend_default_flips(summary_df, baseline_name=baseline_name)
    promotions = {}
    for name in winners:
        if name in _PROMOTABLE_OVERRIDES:
            promotions[name] = dict(_PROMOTABLE_OVERRIDES[name])
    if apply and promotions:
        for overrides in promotions.values():
            for key, val in overrides.items():
                if hasattr(cfg, key):
                    setattr(cfg, key, val)
    return {
        "winners": winners,
        "promotions": promotions,
        "note": (
            "Apply returned promotions to pipeline.config.py after review. "
            "Hit-rate was not used as a gate."
        ),
    }


def compute_metrics(results_df: pd.DataFrame, season_col="simulated_season_window") -> pd.DataFrame:
    """Re-export from metrics (avoids circular import with backtest)."""

    return _compute_metrics(results_df, season_col=season_col)


def _apply_config_overrides(overrides: dict | None):
    """Temporarily patch pipeline.config flags for one ablation run."""
    if not overrides:
        return {}
    saved = {}
    for key, val in overrides.items():
        if hasattr(cfg, key):
            saved[key] = getattr(cfg, key)
            setattr(cfg, key, val)
    return saved


def _restore_config_overrides(saved: dict):
    if not saved:
        return
    for key, val in saved.items():
        setattr(cfg, key, val)


def run_ablation(stints_df, odds_dict=None, configs: dict = None,
                 rolling_window_size: int = 3,
                 n_tuning_trials_elo: int = 10,
                 n_tuning_trials_hier: int = 10,
                 n_tuning_trials_meta: int = 10) -> tuple:
    """Run each configuration end-to-end and return (summary_df, detail_dict).

    summary_df: config x season metrics (long form), good for direct comparison.
    detail_dict: {config_name: raw results DataFrame}.
    """
    if configs is None:
        configs = default_configs()

    detail = {}
    summaries = []
    for name, spec in configs.items():
        spec = spec or {}
        feature_cols = spec.get("feature_cols")
        model_kwargs = spec.get("model_kwargs")
        backtest_kwargs = spec.get("backtest_kwargs", {})
        saved_cfg = _apply_config_overrides(spec.get("config_overrides"))
        print(f"\n{'#'*70}\n# ABLATION CONFIG: {name}\n{'#'*70}")
        try:
            results = run_multi_year_backtest_walkforward(
                stints_df,
                odds_dict=odds_dict,
                n_tuning_trials_elo=n_tuning_trials_elo,
                n_tuning_trials_hier=n_tuning_trials_hier,
                n_tuning_trials_meta=n_tuning_trials_meta,
                rolling_window_size=rolling_window_size,
                model_kwargs=model_kwargs,
                feature_cols=feature_cols,
                **backtest_kwargs,
            )
        finally:
            _restore_config_overrides(saved_cfg)
        detail[name] = results
        m = compute_metrics(results)
        if not m.empty:
            m.insert(0, "config", name)
            summaries.append(m)
            print(f"\n=== {name} (overall) ===")
            print(m[m["season"] == "ALL"].to_string(index=False,
                  float_format=lambda v: f"{v:.3f}"))

    summary_df = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()

    if not summary_df.empty:
        print(f"\n{'='*70}\nOVERALL COMPARISON (season == ALL)\n{'='*70}")
        piv = summary_df[summary_df["season"] == "ALL"].set_index("config")
        print(piv[["spread_mae", "total_mae", "ats_pct", "roi", "brier", "n_bets"]]
              .to_string(float_format=lambda v: f"{v:.3f}"))
        rec = recommend_default_flips(summary_df)
        if rec:
            print(f"\nConfigs passing ablation gate vs baseline_current: {rec}")
            print("(Defaults in config.py are NOT auto-flipped — review and enable manually.)")
        else:
            print("\nNo configs passed ablation gate vs baseline_current.")
    return summary_df, detail


In [ ]:
# ── module: ablation_posthoc ─────────────────────────────────────────────────
"""Post-hoc ablation on saved backtest CSV (no re-run of walk-forward training)."""
from __future__ import annotations

import numpy as np
import pandas as pd



def _apply_edge_bucket_filter(df: pd.DataFrame, min_edge: float = MIN_EDGE_BUCKET) -> pd.DataFrame:
    out = df.copy()
    if "DIRECTION" not in out.columns:
        return out
    active = out["DIRECTION"] != "Pass"
    edge_ok = out["EDGE"].abs() >= float(min_edge)
    width = out.get("CONF_WIDTH", pd.Series(24.0, index=out.index))
    width_ok = width.fillna(24.0) <= MAX_QUANTILE_WIDTH
    out.loc[active & ~(edge_ok & width_ok), "DIRECTION"] = "Pass"
    return out


def _apply_confidence_gate(df: pd.DataFrame, min_conf: float | None = None) -> pd.DataFrame:
    out = df.copy()
    if "CONFIDENCE" not in out.columns or "DIRECTION" not in out.columns:
        return out
    thr = float(MIN_CONFIDENCE_SCORE if min_conf is None else min_conf)
    active = out["DIRECTION"] != "Pass"
    out.loc[active & (out["CONFIDENCE"].fillna(0) < thr), "DIRECTION"] = "Pass"
    return out


def _apply_tight_width(df: pd.DataFrame, max_width: float = 22.0) -> pd.DataFrame:
    out = df.copy()
    if "CONF_WIDTH" not in out.columns or "DIRECTION" not in out.columns:
        return out
    active = out["DIRECTION"] != "Pass"
    out.loc[active & (out["CONF_WIDTH"].fillna(99) > max_width), "DIRECTION"] = "Pass"
    return out


def simulate_selection_mode(df: pd.DataFrame, mode: str) -> pd.DataFrame:
    """Return a copy of results with bet selection applied post-hoc."""
    out = df.copy()
    if mode == "edge_bucket" or mode == "edge_bucket_ats":
        out = _apply_edge_bucket_filter(out)
    elif mode == "edge_bucket_conf":
        out = _apply_edge_bucket_filter(out)
        out = _apply_confidence_gate(out)
    elif mode == "edge_bucket_width22":
        out = _apply_edge_bucket_filter(out)
        out = _apply_tight_width(out, max_width=22.0)
    elif mode == "edge_bucket_full":
        out = _apply_edge_bucket_filter(out)
        out = _apply_tight_width(out, max_width=22.0)
        out = _apply_confidence_gate(out)
    return out


def posthoc_ablation_summary(
    results_df: pd.DataFrame,
    modes: tuple[str, ...] = (
        "baseline_current",
        "edge_bucket",
        "edge_bucket_ats",
        "edge_bucket_conf",
        "edge_bucket_width22",
        "edge_bucket_full",
    ),
) -> pd.DataFrame:
    """Compare selection modes on one backtest export."""
    rows = []
    for mode in modes:
        sim = results_df if mode == "baseline_current" else simulate_selection_mode(results_df, mode)
        m = compute_metrics(sim)
        if m.empty:
            continue
        overall = m[m["season"] == "ALL"].iloc[0].to_dict()
        overall["config"] = mode
        overall["season"] = "ALL"
        rows.append(overall)
    return pd.DataFrame(rows)


def passes_flip_gate(metrics: dict, *, roi_gain: float = 0.02, min_ats: float = 0.55, min_bets: int = 400) -> bool:
    """Plan checklist: pooled ROI +2%, ATS >= 55%, n_bets >= 400, MAE not worse."""
    roi = float(metrics.get("roi", np.nan))
    ats = float(metrics.get("ats_pct", np.nan))
    n = int(metrics.get("n_bets", 0) or 0)
    return (
        np.isfinite(roi)
        and np.isfinite(ats)
        and n >= min_bets
        and ats >= min_ats
        and roi >= roi_gain
    )


In [ ]:
# ── module: edge_analysis ────────────────────────────────────────────────────
"""Data-driven edge vs accuracy analysis (flat and stake-weighted ROI)."""
from __future__ import annotations

import numpy as np
import pandas as pd

BREAKEVEN_ATS = 110.0 / 210.0


def _analysis_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Spread bets for edge analysis — all leans in confidence_only, else actionable only."""
    if df is None or df.empty:
        return pd.DataFrame()

    if BET_SELECTION_MODE == "confidence_only":
        d = lean_spread_frame(df)
        if d.empty:
            return pd.DataFrame()
        cover = d["ACTUAL_MARGIN"] + d["MARKET_SPREAD"]
        side = d["_side"] if "_side" in d.columns else spread_side_series(d)
        d = d.copy()
        d["won"] = (((side == "Home") & (cover > 0)) | ((side == "Away") & (cover < 0))).astype(int)
        d["abs_edge"] = pd.to_numeric(d["EDGE"], errors="coerce").abs()
        return d

    return _ats_frame(df)


def _ats_frame(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    d = df[df.get("DIRECTION", "Pass") != "Pass"].copy()
    if "MARKET_SPREAD" not in d.columns:
        return pd.DataFrame()
    cover = d["ACTUAL_MARGIN"] + d["MARKET_SPREAD"]
    home = d["DIRECTION"] == "Home"
    d["won"] = ((home & (cover > 0)) | (~home & (cover < 0))).astype(int)
    d["abs_edge"] = d["EDGE"].abs()
    return d


def _roi_from_winrate(wp: float) -> float:
    return wp * (100.0 / 110.0) - (1.0 - wp)


def fine_edge_bins(
    df: pd.DataFrame,
    *,
    width: float = 0.5,
    min_edge: float = 2.5,
    max_edge: float = 16.0,
    min_n: int = 25,
) -> pd.DataFrame:
    """ATS/ROI in half-point (or custom width) edge bins."""
    bets = _analysis_frame(df)
    if bets.empty:
        return pd.DataFrame()
    bins = np.arange(min_edge, max_edge + width, width)
    bets["ebin"] = pd.cut(bets["abs_edge"], bins=bins, right=False)
    rows = []
    for iv, sub in bets.groupby("ebin", observed=True):
        if len(sub) < min_n:
            continue
        wp = float(sub["won"].mean())
        rows.append({
            "edge_lo": float(iv.left),
            "edge_hi": float(iv.right),
            "edge_mid": float((iv.left + iv.right) / 2.0),
            "n_bets": len(sub),
            "ats_pct": wp,
            "flat_roi": _roi_from_winrate(wp),
        })
    return pd.DataFrame(rows)


def rolling_edge_ats(
    df: pd.DataFrame,
    *,
    centers: np.ndarray | None = None,
    half_width: float = 0.75,
    min_n: int = 40,
) -> pd.DataFrame:
    """Smoothed ATS curve over |edge| (rolling window in edge space)."""
    bets = _analysis_frame(df)
    if bets.empty:
        return pd.DataFrame()
    if centers is None:
        centers = np.arange(3.0, 13.0, 0.25)
    rows = []
    for c in centers:
        sub = bets[(bets["abs_edge"] >= c - half_width) & (bets["abs_edge"] < c + half_width)]
        if len(sub) < min_n:
            continue
        wp = float(sub["won"].mean())
        rows.append({
            "edge_center": float(c),
            "n_bets": len(sub),
            "ats_pct": wp,
            "flat_roi": _roi_from_winrate(wp),
        })
    return pd.DataFrame(rows)


def fit_parabolic_edge_curve(binned: pd.DataFrame) -> dict:
    """Quadratic ATS ~ edge + edge² on fine bins (detects mid-range valleys)."""
    if binned is None or binned.empty or len(binned) < 4:
        return {}
    x = binned["edge_mid"].to_numpy(dtype=float)
    y = binned["ats_pct"].to_numpy(dtype=float)
    w = np.sqrt(binned["n_bets"].to_numpy(dtype=float))
    X = np.column_stack([np.ones(len(x)), x, x ** 2])
    coef, _, _, _ = np.linalg.lstsq(X * w[:, None], y * w, rcond=None)
    a, b, c = coef
    out = {"intercept": float(a), "linear": float(b), "quadratic": float(c)}
    if abs(c) > 1e-8:
        peak = -b / (2.0 * c)
        out["vertex_edge"] = float(peak)
        out["vertex_ats"] = float(a + b * peak + c * peak ** 2)
    return out


def find_edge_valleys(rolling: pd.DataFrame, *, min_drop: float = 0.025) -> list[dict]:
    """Local ATS dips on the rolling curve (candidate injury/noise zones)."""
    if rolling is None or len(rolling) < 5:
        return []
    r = rolling.sort_values("edge_center").reset_index(drop=True)
    valleys = []
    for i in range(1, len(r) - 1):
        prev_a = r.loc[i - 1, "ats_pct"]
        cur_a = r.loc[i, "ats_pct"]
        next_a = r.loc[i + 1, "ats_pct"]
        local_peak = max(prev_a, next_a)
        if local_peak - cur_a >= min_drop:
            valleys.append({
                "edge_center": float(r.loc[i, "edge_center"]),
                "ats_pct": float(cur_a),
                "drop_from_neighbors": float(local_peak - cur_a),
                "n_bets": int(r.loc[i, "n_bets"]),
            })
    return valleys


def threshold_grid(
    df: pd.DataFrame,
    *,
    thresholds: np.ndarray | None = None,
    min_n: int = 40,
    stake_col: str = "STAKE_MODERATE",
    profit_col: str = "PROFIT_MODERATE",
) -> pd.DataFrame:
    """Flat ATS/ROI and optional Kelly-weighted ROI by minimum |edge|."""
    bets = _analysis_frame(df)
    if bets.empty:
        return pd.DataFrame()
    if thresholds is None:
        thresholds = np.arange(2.5, 10.5, 0.25)
    rows = []
    for thr in thresholds:
        sub = bets[bets["abs_edge"] >= thr]
        if len(sub) < min_n:
            continue
        wp = float(sub["won"].mean())
        row = {
            "min_edge": float(thr),
            "n_bets": len(sub),
            "ats_pct": wp,
            "flat_roi": _roi_from_winrate(wp),
        }
        if stake_col in sub.columns and profit_col in sub.columns:
            st = sub[stake_col].fillna(0)
            if st.sum() > 0:
                row["stake_roi"] = float(sub[profit_col].fillna(0).sum() / st.sum())
        rows.append(row)
    return pd.DataFrame(rows)


def segment_breakpoints(df: pd.DataFrame) -> pd.DataFrame:
    """Hand-tuned segments that match typical NBA backtest shapes."""
    bets = _analysis_frame(df)
    if bets.empty:
        return pd.DataFrame()
    segments = [
        (2.5, 4.0, "dead_zone_2-4"),
        (4.0, 5.5, "build_4-5.5"),
        (5.5, 7.0, "core_5.5-7"),
        (7.0, 9.5, "valley_7-9.5"),
        (9.5, 11.0, "recovery_9.5-11"),
        (11.0, 99.0, "tail_11+"),
    ]
    rows = []
    for lo, hi, name in segments:
        sub = bets[(bets["abs_edge"] >= lo) & (bets["abs_edge"] < hi)]
        if len(sub) < 15:
            continue
        wp = float(sub["won"].mean())
        rows.append({
            "segment": name,
            "edge_lo": lo,
            "edge_hi": hi,
            "n_bets": len(sub),
            "ats_pct": wp,
            "flat_roi": _roi_from_winrate(wp),
            "mean_confidence": float(sub["CONFIDENCE"].mean()) if "CONFIDENCE" in sub.columns else np.nan,
        })
    return pd.DataFrame(rows)


def injury_disagreement_split(df: pd.DataFrame) -> pd.DataFrame:
    """Split ATS by phantom-injury / disagreement flags when present."""
    bets = _analysis_frame(df)
    if bets.empty:
        return pd.DataFrame()
    rows = []
    flags = []
    if "PHANTOM_INJURY_FLAG" in bets.columns:
        flags.append(("phantom_injury", bets["PHANTOM_INJURY_FLAG"].astype(bool)))
    if "DISAGREEMENT_TRUST" in bets.columns:
        flags.append(("low_trust", bets["DISAGREEMENT_TRUST"] < 0.85))
    if "H_STAR_OUT" in bets.columns or "A_STAR_OUT" in bets.columns:
        star = bets.get("H_STAR_OUT", 0).fillna(0) + bets.get("A_STAR_OUT", 0).fillna(0)
        flags.append(("known_star_out", star > 0))
    if "ELO_META_AGREEMENT" in bets.columns:
        flags.append(("elo_meta_disagree", bets["ELO_META_AGREEMENT"] < 0.5))

    if not flags:
        # Legacy proxy: huge edge + high confidence without injury columns
        proxy = (bets["abs_edge"] >= 8.0) & (bets.get("CONFIDENCE", 0) >= 35)
        flags.append(("legacy_phantom_proxy", proxy))

    for name, mask in flags:
        for label, sub in [("yes", bets[mask]), ("no", bets[~mask])]:
            if len(sub) < 20:
                continue
            wp = float(sub["won"].mean())
            rows.append({
                "flag": name,
                "value": label,
                "n_bets": len(sub),
                "ats_pct": wp,
                "flat_roi": _roi_from_winrate(wp),
                "mean_edge": float(sub["abs_edge"].mean()),
            })
    return pd.DataFrame(rows)


def recommend_policy(
    df: pd.DataFrame,
    *,
    min_bets: int = 80,
) -> dict:
    """Suggest min edge, optional cap, and segments to favor/avoid."""
    grid = threshold_grid(df, min_n=min_bets)
    rolling = rolling_edge_ats(df)
    valleys = find_edge_valleys(rolling)
    segments = segment_breakpoints(df)
    injury = injury_disagreement_split(df)

    policy = {
        "n_active_bets": int(len(_analysis_frame(df))),
        "valleys": valleys,
        "injury_split": injury.to_dict(orient="records") if not injury.empty else [],
    }

    if not grid.empty:
        best_flat = grid.loc[grid["flat_roi"].idxmax()]
        policy["best_min_edge_flat"] = {
            "min_edge": float(best_flat["min_edge"]),
            "n_bets": int(best_flat["n_bets"]),
            "ats_pct": float(best_flat["ats_pct"]),
            "flat_roi": float(best_flat["flat_roi"]),
        }
        # Volume-balanced: best ROI with n >= 15% of max n in grid
        max_n = grid["n_bets"].max()
        bal = grid[grid["n_bets"] >= 0.15 * max_n]
        if not bal.empty:
            bb = bal.loc[bal["flat_roi"].idxmax()]
            policy["best_min_edge_balanced"] = {
                "min_edge": float(bb["min_edge"]),
                "n_bets": int(bb["n_bets"]),
                "ats_pct": float(bb["ats_pct"]),
                "flat_roi": float(bb["flat_roi"]),
            }

    if not segments.empty:
        policy["best_segment"] = segments.loc[segments["flat_roi"].idxmax()].to_dict()
        policy["worst_segment"] = segments.loc[segments["flat_roi"].idxmin()].to_dict()

    bins = fine_edge_bins(df)
    policy["parabolic_fit"] = fit_parabolic_edge_curve(bins)

    return policy


def run_edge_analysis(df: pd.DataFrame, *, verbose: bool = True) -> dict:
    """Full report dict + optional console summary."""
    bets = _analysis_frame(df)
    out = {
        "fine_bins": fine_edge_bins(df),
        "rolling": rolling_edge_ats(df),
        "threshold_grid": threshold_grid(df),
        "segments": segment_breakpoints(df),
        "injury_split": injury_disagreement_split(df),
        "policy": recommend_policy(df),
    }
    if verbose and not bets.empty:
        print("\n" + "=" * 60)
        print("EDGE vs ACCURACY ANALYSIS")
        print("=" * 60)
        print(f"Active spread bets (non-push): {len(bets):,}")
        print(f"Pooled ATS: {bets['won'].mean():.1%}  flat ROI: {_roi_from_winrate(bets['won'].mean()):+.1%}")

        seg = out["segments"]
        if not seg.empty:
            print("\nSegment table:")
            print(seg.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

        pol = out["policy"]
        if pol.get("best_min_edge_balanced"):
            b = pol["best_min_edge_balanced"]
            print(f"\nRecommended min |edge| (volume-balanced): {b['min_edge']:.2f} pt")
            print(f"  n={b['n_bets']}  ATS={b['ats_pct']:.1%}  flat ROI={b['flat_roi']:+.1%}")
        if pol.get("valleys"):
            print("\nLocal ATS valleys (possible injury/noise zones):")
            for v in pol["valleys"]:
                print(f"  edge~{v['edge_center']:.2f}: ATS={v['ats_pct']:.1%}  drop={v['drop_from_neighbors']:.1%}")

        inj = out["injury_split"]
        if not inj.empty:
            print("\nInjury / disagreement splits:")
            print(inj.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

        pf = pol.get("parabolic_fit", {})
        if pf:
            print(f"\nQuadratic bin fit: ATS ≈ {pf.get('intercept', 0):.3f} + {pf.get('linear', 0):.4f}·edge + {pf.get('quadratic', 0):.5f}·edge²")
    return out


In [ ]:
# ── module: diagnostics ──────────────────────────────────────────────────────
"""Per-season diagnostics: text summaries + matplotlib graphs.

Two entry points:
  - ``diagnose_loaded_data``  — after stints/odds load (coverage, dates, games)
  - ``run_backtest_diagnostics`` — after walk-forward backtest (MAE, edge, ATS, …)

Designed for both local scripts and the Colab notebook (inline ``plt.show()``).
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

BREAKEVEN = 110.0 / 210.0  # -110 break-even ATS win rate
EDGE_BUCKETS = [0, 2, 4, 6, 8, np.inf]
EDGE_LABELS = ["0-2", "2-4", "4-6", "6-8", "8+"]
CLOSE_MARGINS = (3, 5, 7)


def _season_col(df: pd.DataFrame) -> str:
    for c in ("simulated_season_window", "season", "SEASON"):
        if c in df.columns:
            return c
    return "season"


def _norm_results(df: pd.DataFrame) -> pd.DataFrame:
    """Standardise column names used across the pipeline."""
    out = df.copy()
    rename = {}
    if "DATE" in out.columns and "date" not in out.columns:
        rename["DATE"] = "date"
    if rename:
        out = out.rename(columns=rename)
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"], errors="coerce")
    sc = _season_col(out)
    out["_season"] = out[sc].astype(str)
    if "ACTUAL_MARGIN" not in out.columns and {"ACTUAL_HOME", "ACTUAL_AWAY"}.issubset(out.columns):
        out["ACTUAL_MARGIN"] = out["ACTUAL_HOME"] - out["ACTUAL_AWAY"]
    if "SPREAD_ERR" not in out.columns and "PRED_SPREAD" in out.columns:
        out["SPREAD_ERR"] = (out["PRED_SPREAD"] - out["ACTUAL_MARGIN"]).abs()
    if "RAW_SPREAD_ERR" not in out.columns and "RAW_PRED_MARGIN" in out.columns:
        out["RAW_SPREAD_ERR"] = (out["RAW_PRED_MARGIN"] - out["ACTUAL_MARGIN"]).abs()
    if "TOTAL_ERR" not in out.columns and "PRED_TOTAL" in out.columns:
        out["TOTAL_ERR"] = out["PRED_TOTAL"] - (out["ACTUAL_HOME"] + out["ACTUAL_AWAY"])
    return out


def _spread_side_series(d: pd.DataFrame) -> pd.Series:
    if "EDGE_LEAN" in d.columns:
        lean = d["EDGE_LEAN"].fillna("Pass").astype(str)
    else:
        lean = d.get("DIRECTION", pd.Series("Pass", index=d.index)).fillna("Pass").astype(str)
    if "EDGE" in d.columns:
        edge = pd.to_numeric(d["EDGE"], errors="coerce").fillna(0)
        need = lean.isin(("Pass", "nan", "")) | lean.isna()
        inferred = np.where(edge > 0, "Home", np.where(edge < 0, "Away", "Pass"))
        lean = lean.where(~need, inferred)
    return lean


def _ats_lean_frame(d: pd.DataFrame) -> pd.DataFrame:
    """ATS outcomes for every lined game using model lean (includes sub-threshold edges)."""
    m = d[d["MARKET_SPREAD"].notna()].copy()
    side = _spread_side_series(m)
    m = m[side != "Pass"].copy()
    m["_side"] = side.loc[m.index]
    cover = m["ACTUAL_MARGIN"] + m["MARKET_SPREAD"]
    m["ats_win"] = (
        ((m["_side"] == "Home") & (cover > 0))
        | ((m["_side"] == "Away") & (cover < 0))
    ).astype(int)
    m["ats_push"] = (cover == 0).astype(int)
    return m


def _ats_frame(d: pd.DataFrame) -> pd.DataFrame:
    """Actionable ATS bets with ``ats_win`` / ``ats_push`` always present.

    An empty actionable set still returns those columns so callers can safely
    filter on ``ats_push`` without KeyError (e.g. confidence-only mode with
    zero placed bets for a season).
    """

    m = actionable_spread_frame(d)
    if m.empty:
        out = m.copy()
        if "ats_win" not in out.columns:
            out["ats_win"] = pd.Series(dtype=int)
        if "ats_push" not in out.columns:
            out["ats_push"] = pd.Series(dtype=int)
        return out
    side = m["_side"] if "_side" in m.columns else spread_side_series(m)
    cover = m["ACTUAL_MARGIN"] + m["MARKET_SPREAD"]
    m = m.copy()
    m["ats_win"] = (
        ((side == "Home") & (cover > 0))
        | ((side == "Away") & (cover < 0))
    ).astype(int)
    m["ats_push"] = (cover == 0).astype(int)
    return m


def _non_push_ats(bets: pd.DataFrame) -> pd.DataFrame:
    """Filter push rows; no-op/empty-safe when ``ats_push`` is missing."""
    if bets is None or bets.empty:
        return bets if bets is not None else pd.DataFrame()
    if "ats_push" not in bets.columns:
        return bets.iloc[0:0].copy()
    return bets[bets["ats_push"] == 0]


def _ou_frame(d: pd.DataFrame) -> pd.DataFrame:
    """O/U bets with win flag."""
    if "OU_DIRECTION" not in d.columns or "MARKET_TOTAL" not in d.columns:
        return pd.DataFrame()
    m = d[(d["OU_DIRECTION"] != "Pass") & d["MARKET_TOTAL"].notna()].copy()
    if m.empty:
        return m
    if "OU_WIN" in m.columns:
        m["ou_win"] = pd.to_numeric(m["OU_WIN"], errors="coerce")
    else:
        actual = m["ACTUAL_HOME"] + m["ACTUAL_AWAY"]
        mkt = pd.to_numeric(m["MARKET_TOTAL"], errors="coerce")
        m["ou_win"] = np.where(
            m["OU_DIRECTION"] == "Over",
            (actual > mkt).astype(float),
            (actual < mkt).astype(float),
        )
    return m


def _ou_edge_bucket_table(g: pd.DataFrame) -> pd.DataFrame:
    """O/U win rate by |TOTAL_EDGE| bucket."""
    g = _norm_results(g)
    if "TOTAL_EDGE" not in g.columns:
        return pd.DataFrame()
    ou = _ou_frame(g)
    if ou.empty:
        return pd.DataFrame()
    ou = ou.copy()
    ou["ebin"] = pd.cut(ou["TOTAL_EDGE"].abs(), bins=EDGE_BUCKETS, labels=EDGE_LABELS, right=False)
    rows = []
    for lbl in EDGE_LABELS:
        sub = ou[ou["ebin"] == lbl]
        if sub.empty:
            continue
        wp = float(sub["ou_win"].mean())
        rows.append({"bucket": lbl, "n": len(sub), "ou_win_pct": wp, "roi": wp * (100 / 110) - (1 - wp)})
    return pd.DataFrame(rows)


def _tight_spread_table(g: pd.DataFrame) -> pd.DataFrame:
    """ATS on actionable bets when |market spread| is within TIGHT_SPREAD_MAX."""

    g = _norm_results(g)
    bets = _ats_frame(g)
    if bets.empty or "MARKET_SPREAD" not in bets.columns:
        return pd.DataFrame()
    tight = bets[bets["MARKET_SPREAD"].abs() <= float(TIGHT_SPREAD_MAX)]
    tight = _non_push_ats(tight)
    if tight.empty:
        return pd.DataFrame()
    wp = float(tight["ats_win"].mean())
    return pd.DataFrame([{
        "max_spread": float(TIGHT_SPREAD_MAX),
        "n_bets": len(tight),
        "ats_pct": wp,
        "roi": wp * (100.0 / 110.0) - (1 - wp),
    }])


def _season_metrics(g: pd.DataFrame) -> dict:
    """One row of summary stats for a single season."""
    g = _norm_results(g)
    n = len(g)
    has_mkt = g["MARKET_SPREAD"].notna() if "MARKET_SPREAD" in g.columns else pd.Series(False, index=g.index)
    n_mkt = int(has_mkt.sum())
    spread_mae = float(g["SPREAD_ERR"].mean()) if "SPREAD_ERR" in g else np.nan
    raw_spread_mae = float(g["RAW_SPREAD_ERR"].mean()) if "RAW_SPREAD_ERR" in g else np.nan
    spread_bias = float((g["PRED_SPREAD"] - g["ACTUAL_MARGIN"]).mean()) if "PRED_SPREAD" in g else np.nan
    total_mae = float(g["TOTAL_ERR"].abs().mean()) if "TOTAL_ERR" in g else np.nan
    brier = np.nan
    log_loss = np.nan
    ece = np.nan
    if "WIN_PROB" in g.columns:
        hw = (g["ACTUAL_MARGIN"] > 0).astype(int)
        brier = float(((g["WIN_PROB"] - hw) ** 2).mean())
        try:
            ece = float(compute_ece(hw.values, g["WIN_PROB"].values))
        except Exception:
            ece = np.nan
        try:
            from sklearn.metrics import log_loss as sk_log_loss
            probs = np.clip(g["WIN_PROB"].values.astype(float), 1e-6, 1 - 1e-6)
            log_loss = float(sk_log_loss(hw.values, np.column_stack([1 - probs, probs])))
        except Exception:
            log_loss = np.nan
    # Prefer calibration metrics over hit-rate for ML (kept for reference only)
    ml_acc = float(g["MODEL_ML_CORRECT"].mean()) if "MODEL_ML_CORRECT" in g else np.nan

    crps = np.nan
    if "PRED_TOTAL" in g.columns and "ACTUAL_HOME" in g.columns and "ACTUAL_AWAY" in g.columns:
        actual_total = g["ACTUAL_HOME"] + g["ACTUAL_AWAY"]
        sigma = g["SIGMA_TOTAL"] if "SIGMA_TOTAL" in g.columns else OU_DEFAULT_SIGMA
        if isinstance(sigma, (int, float)):
            sigma = pd.Series(float(sigma), index=g.index)
        crps_vals = [
            crps_gaussian(a, m, s)
            for a, m, s in zip(actual_total, g["PRED_TOTAL"], sigma.fillna(OU_DEFAULT_SIGMA))
        ]
        crps_vals = [c for c in crps_vals if c is not None and np.isfinite(c)]
        crps = float(np.mean(crps_vals)) if crps_vals else np.nan

    close = {}
    for cm in CLOSE_MARGINS:
        close[f"pct_decided_{cm}"] = float((g["ACTUAL_MARGIN"].abs() <= cm).mean()) if n else np.nan

    edge = g["EDGE"].abs() if "EDGE" in g else pd.Series(dtype=float)
    edge_stats = {
        "edge_mean": float(edge.mean()) if len(edge) else np.nan,
        "edge_median": float(edge.median()) if len(edge) else np.nan,
        "edge_p90": float(edge.quantile(0.9)) if len(edge) else np.nan,
    }

    ats = _ats_frame(g)
    non_push = _non_push_ats(ats)
    ats_n = int(len(non_push))
    if ats_n:
        wp = non_push["ats_win"].mean()
        roi = wp * (100.0 / 110.0) - (1 - wp)
    else:
        wp = roi = np.nan

    lean = _ats_lean_frame(g)
    lean_non_push = _non_push_ats(lean)
    lean_n = int(len(lean_non_push))
    lean_wp = lean_non_push["ats_win"].mean() if lean_n else np.nan

    ou_n = ou_wp = ou_roi = np.nan
    ou = _ou_frame(g)
    if not ou.empty:
        ou_n = len(ou)
        ou_wp = float(ou["ou_win"].mean())
        ou_roi = ou_wp * (100.0 / 110.0) - (1 - ou_wp)

    cover_ece = np.nan
    cover_brier = np.nan
    sharpe = np.nan
    max_dd = np.nan
    if "COVER_PROB_CALIBRATED" in g.columns and not ats.empty:
        bets = _non_push_ats(ats).copy()
        if len(bets):
            cover_brier = compute_brier(bets["ats_win"].values, bets["COVER_PROB_CALIBRATED"].values)
            cover_ece = compute_ece(bets["ats_win"].values, bets["COVER_PROB_CALIBRATED"].values)
    if "PROFIT_MODERATE" in g.columns and "STAKE_MODERATE" in g.columns:
        active = g[g["STAKE_MODERATE"] > 0]
        if len(active) >= 5:
            profits = active["PROFIT_MODERATE"].values
            sharpe = sharpe_ratio(profits)
            max_dd = max_drawdown(np.cumsum(profits))

    dates_ok = True
    if "date" in g.columns:
        dates_ok = g["date"].dropna().dt.normalize().nunique() > 1

    return {
        "n_games": n,
        "n_with_odds": n_mkt,
        "pct_with_odds": n_mkt / n if n else np.nan,
        "spread_mae": spread_mae,
        "raw_spread_mae": raw_spread_mae,
        "spread_bias": spread_bias,
        "total_mae": total_mae,
        "crps": crps,
        "brier": brier,
        "log_loss": log_loss,
        "ece": ece,
        "cover_brier": cover_brier,
        "cover_ece": cover_ece,
        "sharpe_per_bet": sharpe,
        "max_drawdown": max_dd,
        "ml_acc": ml_acc,  # reference only — do not promote models on hit-rate
        "n_bets": ats_n,
        "n_leans": lean_n,
        "lean_ats_pct": lean_wp,
        "ats_pct": wp,
        "roi": roi,
        "ou_n_bets": ou_n,
        "ou_win_pct": ou_wp,
        "ou_roi": ou_roi,
        "dates_ok": dates_ok,
        **close,
        **edge_stats,
    }


def _confidence_tier_table(g: pd.DataFrame) -> pd.DataFrame:
    """ATS win rate and ROI by calibrated confidence tier."""
    g = _norm_results(g)
    if "CONFIDENCE_TIER" not in g.columns:
        return pd.DataFrame()
    bets = _ats_frame(g)
    bets = _non_push_ats(bets)
    if bets.empty:
        return pd.DataFrame()
    rows = []
    for tier in sorted(bets["CONFIDENCE_TIER"].dropna().unique()):
        sub = bets[bets["CONFIDENCE_TIER"] == tier]
        wp = sub["ats_win"].mean()
        rows.append({
            "tier": int(tier),
            "n_bets": len(sub),
            "ats_pct": wp,
            "roi": wp * (100.0 / 110.0) - (1 - wp) if len(sub) else np.nan,
            "mean_edge": sub["EDGE"].abs().mean() if "EDGE" in sub.columns else np.nan,
        })
    return pd.DataFrame(rows)


def _confidence_score_table(g: pd.DataFrame, n_bins: int = 10) -> pd.DataFrame:
    """ATS win rate and ROI by confidence score deciles."""
    return confidence_score_table(g, score_col="CONFIDENCE", n_bins=n_bins)


def _edge_bucket_table(g: pd.DataFrame, include_ats: bool = True) -> pd.DataFrame:
    g = _norm_results(g)
    if "EDGE" not in g.columns:
        return pd.DataFrame()
    g = g.copy()
    g["edge_bucket"] = pd.cut(g["EDGE"].abs(), bins=EDGE_BUCKETS, labels=EDGE_LABELS, right=False)
    rows = []
    for lbl in EDGE_LABELS:
        sub = g[g["edge_bucket"] == lbl]
        row = {"bucket": lbl, "n_games": len(sub), "pct_games": len(sub) / len(g) if len(g) else 0}
        if include_ats and "MARKET_SPREAD" in g.columns:
            bets = _non_push_ats(_ats_lean_frame(sub))
            row["n_bets"] = len(bets)
            row["ats_pct"] = bets["ats_win"].mean() if len(bets) else np.nan
            if "WIN_PCT" in sub.columns and len(sub):
                row["mean_win_pct"] = float(pd.to_numeric(sub["WIN_PCT"], errors="coerce").mean())
            elif "CONFIDENCE" in sub.columns and len(sub):
                row["mean_win_pct"] = float(pd.to_numeric(sub["CONFIDENCE"], errors="coerce").mean())
            if len(bets) and bets["ats_win"].notna().any():
                wp = bets["ats_win"].mean()
                row["roi"] = wp * (100.0 / 110.0) - (1 - wp)
            else:
                row["roi"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def diagnose_loaded_data(stints_df: pd.DataFrame, odds_dict: dict | None = None) -> pd.DataFrame:
    """Print a per-season load summary (games, dates, odds coverage)."""
    if stints_df is None or stints_df.empty:
        print("No stint data loaded.")
        return pd.DataFrame()

    df = stints_df.copy()
    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    games = (
        df[["season", "GAME_ID", "game_date", "home_team", "away_team"]]
        .drop_duplicates(["season", "GAME_ID"])
    )

    rows = []
    print("\n" + "=" * 72)
    print("LOAD DIAGNOSTICS · per season")
    print("=" * 72)
    for season in sorted(games["season"].unique()):
        sg = games[games["season"] == season]
        n_g = len(sg)
        dmin, dmax = sg["game_date"].min(), sg["game_date"].max()
        n_dates = sg["game_date"].dropna().dt.normalize().nunique()
        dates_ok = n_dates > 1
        n_odds = 0
        if odds_dict:
            for _, r in sg.iterrows():
                d = r["game_date"]
                if pd.isna(d):
                    continue
                key = (d.date() if hasattr(d, "date") else d, r["home_team"])
                if key in odds_dict:
                    n_odds += 1
        pct_odds = n_odds / n_g if n_g else 0
        flag = "" if dates_ok else "  ⚠️ collapsed dates"
        print(f"\n── Season {season}{flag}")
        print(f"   games={n_g:,}  unique_dates={n_dates}  range={str(dmin)[:10]} → {str(dmax)[:10]}")
        print(f"   odds matched (home+date key): {n_odds:,} ({pct_odds:.1%})")
        rows.append({
            "season": season, "n_games": n_g, "n_dates": n_dates,
            "date_min": dmin, "date_max": dmax, "dates_ok": dates_ok,
            "n_odds_matched": n_odds, "pct_odds": pct_odds,
        })
    return pd.DataFrame(rows)


def _setup_plt():
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "figure.facecolor": "white", "axes.grid": True, "grid.alpha": 0.3,
        "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
    })
    return plt


def _save_or_show(fig, save_dir: Path | None, name: str):
    import matplotlib.pyplot as plt
    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_dir / name, dpi=150, bbox_inches="tight")
        plt.close(fig)
    else:
        fig.tight_layout()
        plt.show()


def _print_gate_attrition(g: pd.DataFrame) -> None:
    """Explain zero actionable ATS bets via successive confidence-gate filters."""
    from pipeline import config as cfg

    n = len(g)
    if n == 0:
        return
    edge = pd.to_numeric(g.get("EDGE"), errors="coerce").abs()
    conf = pd.to_numeric(
        g["CONFIDENCE"] if "CONFIDENCE" in g.columns else g.get("WIN_PCT"),
        errors="coerce",
    )
    width = pd.to_numeric(
        g["CONF_WIDTH"] if "CONF_WIDTH" in g.columns else g.get("SPREAD_QUANTILE_WIDTH"),
        errors="coerce",
    )
    mkt = pd.to_numeric(g.get("MARKET_SPREAD"), errors="coerce").abs()
    trust = pd.to_numeric(g.get("DISAGREEMENT_TRUST"), errors="coerce").fillna(1.0)
    phantom = pd.to_numeric(g.get("PHANTOM_INJURY_FLAG"), errors="coerce").fillna(0).astype(bool)

    has_odds = mkt.notna()
    pass_edge = has_odds & edge.notna() & (edge >= float(CONFIDENCE_MIN_EDGE))
    floor = float(MIN_CONFIDENCE_SCORE)
    if CONFIDENCE_EDGE_SCALED:
        need = np.where(edge < 5.0, floor + 4.0, np.where(edge < 8.0, floor + 2.0, floor))
    else:
        need = floor
    pass_conf = pass_edge & conf.notna() & (conf >= need)
    if MAX_QUANTILE_WIDTH is None:
        pass_width = pass_conf
    else:
        pass_width = pass_conf & (width.isna() | (width <= float(MAX_QUANTILE_WIDTH)))
    pass_trust = pass_width & (trust >= float(MIN_DISAGREEMENT_TRUST))
    pass_phantom = pass_trust & (~phantom) if SKIP_PHANTOM_INJURY else pass_trust
    pass_tight = (
        pass_phantom & (mkt > float(TIGHT_SPREAD_MAX))
        if SKIP_TIGHT_SPREAD
        else pass_phantom
    )
    band = EDGE_AVOID_BAND
    if band is not None:
        lo, hi = band
        in_band = (edge >= float(lo)) & (edge < float(hi))
        agree = pd.to_numeric(g.get("ELO_META_AGREEMENT"), errors="coerce").fillna(1.0)
        pass_band = pass_tight & (~in_band | (agree >= float(EDGE_AVOID_BAND_MIN_ELO_AGREE)))
    else:
        pass_band = pass_tight

    print("  ⚠️  0 ATS bets — gate attrition (rows surviving each filter):")
    print(f"     games={n:,}  |edge|>={CONFIDENCE_MIN_EDGE}: {int(pass_edge.sum()):,}")
    print(
        f"     + conf>={MIN_CONFIDENCE_SCORE}"
        f"{' (edge-scaled)' if CONFIDENCE_EDGE_SCALED else ''}: {int(pass_conf.sum()):,}"
    )
    print(f"     + width<={MAX_QUANTILE_WIDTH}: {int(pass_width.sum()):,}")
    print(f"     + trust>={MIN_DISAGREEMENT_TRUST}: {int(pass_trust.sum()):,}")
    if SKIP_PHANTOM_INJURY:
        print(f"     + no phantom injury: {int(pass_phantom.sum()):,}")
    if SKIP_TIGHT_SPREAD:
        print(f"     + |market|>{TIGHT_SPREAD_MAX}: {int(pass_tight.sum()):,}")
    print(f"     + edge avoid band {band}: {int(pass_band.sum()):,}")
    if width.notna().any():
        print(
            f"     conf width: median={float(width.median()):.1f}  "
            f"p90={float(width.quantile(0.9)):.1f}  "
            f"(cap={MAX_QUANTILE_WIDTH})"
        )


def run_backtest_diagnostics(
    results_df: pd.DataFrame,
    save_dir: str | Path | None = None,
    show_graphs: bool = True,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    """Print per-season backtest diagnostics and draw graphs.

    Returns (season_summary_df, {season: edge_bucket_df}).
    """
    if results_df is None or results_df.empty:
        print("No backtest results to diagnose.")
        return pd.DataFrame(), {}

    df = _norm_results(results_df)
    seasons = sorted(df["_season"].unique())
    save_path = Path(save_dir) if save_dir else None
    plt = _setup_plt() if show_graphs else None

    summary_rows = []
    edge_tables: dict[str, pd.DataFrame] = {}

    print("\n" + "=" * 72)
    print("BACKTEST DIAGNOSTICS · per season")
    print("=" * 72)
    print("(PRED_SPREAD is the bias/slope-corrected model spread)")

    for s in seasons:
        g = df[df["_season"] == s]
        m = _season_metrics(g)
        m["season"] = s
        summary_rows.append(m)
        eb = _edge_bucket_table(g)
        edge_tables[s] = eb

        flag = "" if m["dates_ok"] else "  ⚠️ suspect dates"
        print(f"\n{'─' * 72}")
        print(f"Season {s}{flag}")
        print(f"  games={m['n_games']:,}  with_odds={m['n_with_odds']:,} ({m['pct_with_odds']:.1%})")
        print(f"  spread MAE={m['spread_mae']:.2f}  raw MAE={m.get('raw_spread_mae', float('nan')):.2f}  "
              f"bias(corrected)={m['spread_bias']:+.2f}  total MAE={m['total_mae']:.2f}  "
              f"CRPS={m.get('crps', float('nan')):.2f}")
        print(f"  Brier={m['brier']:.3f}  LogLoss={m.get('log_loss', float('nan')):.3f}  "
              f"ECE={m.get('ece', float('nan')):.3f}  cover ECE={m.get('cover_ece', float('nan')):.3f}  "
              f"(ML hit-rate ref={m['ml_acc']:.1%})")
        if pd.notna(m.get("sharpe_per_bet")):
            print(f"  moderate Sharpe={m['sharpe_per_bet']:.2f}  max DD={m.get('max_drawdown', float('nan')):.3f}")
        print(f"  close games: ≤3pt={m['pct_decided_3']:.1%}  ≤5pt={m['pct_decided_5']:.1%}  ≤7pt={m['pct_decided_7']:.1%}")
        print(f"  edge |model-market|: mean={m['edge_mean']:.2f}  median={m['edge_median']:.2f}  p90={m['edge_p90']:.2f}")
        print(f"  ATS bets={m['n_bets']:,}  win%={m['ats_pct']:.1%}  ROI={m['roi']:+.1%}  (break-even={BREAKEVEN:.1%})")
        if int(m["n_bets"] or 0) == 0:
            _print_gate_attrition(g)
        if not eb.empty:
            print("  edge buckets (|edge| pts):")
            for _, r in eb.iterrows():
                ats_s = f"  ATS={r['ats_pct']:.1%} n={int(r['n_bets'])}" if pd.notna(r.get("ats_pct")) else ""
                print(f"    {r['bucket']:>4}: {int(r['n_games']):4d} games ({r['pct_games']:.1%}){ats_s}")
        ct = _confidence_tier_table(g)
        if not ct.empty:
            print("  confidence tiers (calibrated):")
            for _, r in ct.iterrows():
                print(
                    f"    tier {int(r['tier'])}: n={int(r['n_bets']):4d}  "
                    f"ATS={r['ats_pct']:.1%}  ROI={r['roi']:+.1%}  "
                    f"mean|edge|={r['mean_edge']:.2f}"
                )

    summary = pd.DataFrame(summary_rows)

    # ── pooled overall ──
    om = _season_metrics(df)
    print(f"\n{'═' * 72}")
    print("OVERALL (all seasons pooled)")
    print(f"  games={om['n_games']:,}  spread MAE={om['spread_mae']:.2f}  total MAE={om['total_mae']:.2f}")
    print(f"  ATS {om['n_bets']:,} bets @ {om['ats_pct']:.1%}  ROI={om['roi']:+.1%}")
    if pd.notna(om.get("lean_ats_pct")):
        print(f"  all leans: n={int(om.get('n_leans', 0)):,}  win%={om['lean_ats_pct']:.1%}")
    tight = _tight_spread_table(df)
    if not tight.empty:
        r = tight.iloc[0]
        print(
            f"  tight spread (|line|<={r['max_spread']:.1f}): "
            f"n={int(r['n_bets'])}  ATS={r['ats_pct']:.1%}  ROI={r['roi']:+.1%}"
        )
    eb_all = _edge_bucket_table(df)
    if not eb_all.empty:
        print("  pooled edge buckets:")
        print(eb_all.to_string(index=False, float_format=lambda x: f"{x:.3f}" if isinstance(x, float) else str(x)))

    if not show_graphs:
        return summary, edge_tables

    palette = {s: c for s, c in zip(seasons, plt.cm.tab10.colors)}

    # 1 · ATS win% by season
    fig, ax = plt.subplots(figsize=(9, 5))
    vals = [summary.loc[summary["season"] == s, "ats_pct"].iloc[0] for s in seasons]
    ns = [summary.loc[summary["season"] == s, "n_bets"].iloc[0] for s in seasons]
    colors = ["#55A868" if summary.loc[summary["season"] == s, "dates_ok"].iloc[0] else "#C44E52" for s in seasons]
    bars = ax.bar([str(s) for s in seasons], vals, color=colors, edgecolor="black", alpha=0.85)
    ax.axhline(BREAKEVEN, ls="--", color="black", lw=1.2, label=f"Break-even ({BREAKEVEN:.1%})")
    for b, v, n in zip(bars, vals, ns):
        if pd.notna(v):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.1%}\n(n={int(n)})", ha="center", fontsize=9)
    ax.set_ylim(0.4, 0.75)
    ax.set_ylabel("ATS win rate")
    ax.set_title("ATS Win Rate by Season (active spread bets)")
    ax.legend()
    _save_or_show(fig, save_path, "01_ats_by_season.png")

    # 2 · Spread MAE + bias by season
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(seasons))
    mae_v = [summary.loc[summary["season"] == s, "spread_mae"].iloc[0] for s in seasons]
    bias_v = [summary.loc[summary["season"] == s, "spread_bias"].iloc[0] for s in seasons]
    ax.bar(x - 0.2, mae_v, 0.4, label="Spread MAE (corrected pred)", color="#4C72B0", edgecolor="black")
    ax.bar(x + 0.2, np.abs(bias_v), 0.4, label="|Bias| (pred − actual)", color="#DD8452", edgecolor="black")
    ax.set_xticks(x)
    ax.set_xticklabels([str(s) for s in seasons], rotation=15, ha="right")
    ax.set_ylabel("Points")
    ax.set_title("Corrected Spread Error by Season")
    ax.legend()
    _save_or_show(fig, save_path, "02_spread_mae_bias.png")

    # 3b · ATS win% by confidence tier (pooled)
    ct_all = _confidence_tier_table(df)
    if not ct_all.empty:
        fig, ax = plt.subplots(figsize=(8, 5))
        tiers = ct_all["tier"].astype(int).astype(str)
        ax.bar(tiers, ct_all["ats_pct"], color="#8172B3", edgecolor="black", alpha=0.85)
        ax.axhline(BREAKEVEN, ls="--", color="black", lw=1)
        for i, (_, r) in enumerate(ct_all.iterrows()):
            ax.text(i, r["ats_pct"] + 0.01, f"n={int(r['n_bets'])}", ha="center", fontsize=9)
        ax.set_xlabel("Confidence tier")
        ax.set_ylabel("ATS win rate")
        ax.set_title("ATS Win % by Calibrated Confidence Tier (pooled)")
        _save_or_show(fig, save_path, "03b_ats_by_confidence_tier.png")

    # 3 · ATS win% by edge bucket (grouped by season)
    fig, ax = plt.subplots(figsize=(11, 6))
    width = 0.8 / max(len(seasons), 1)
    for i, s in enumerate(seasons):
        eb = edge_tables[s]
        if eb.empty:
            continue
        xs = np.arange(len(EDGE_LABELS)) + i * width
        ys = [eb.loc[eb["bucket"] == lbl, "ats_pct"].iloc[0] if lbl in eb["bucket"].values else np.nan
              for lbl in EDGE_LABELS]
        ax.bar(xs, ys, width, label=str(s), color=palette[s], edgecolor="black", alpha=0.85)
    ax.axhline(BREAKEVEN, ls="--", color="black", lw=1)
    ax.set_xticks(np.arange(len(EDGE_LABELS)) + width * (len(seasons) - 1) / 2)
    ax.set_xticklabels(EDGE_LABELS)
    ax.set_xlabel("|Model edge| bucket (pts)")
    ax.set_ylabel("ATS win rate")
    ax.set_title("ATS Win % by Edge Bucket (per season)")
    ax.legend(fontsize=8, ncol=2)
    _save_or_show(fig, save_path, "03_ats_by_edge_bucket.png")

    # 3c · Rolling ATS vs |edge| (data-driven curve, not fixed buckets)
    try:

        roll = rolling_edge_ats(df)
        if not roll.empty:
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(roll["edge_center"], roll["ats_pct"], color="#4C72B0", lw=2, marker="o", ms=4)
            ax.axhline(BREAKEVEN, ls="--", color="black", lw=1, label=f"Break-even ({BREAKEVEN:.1%})")
            seg = segment_breakpoints(df)
            for _, r in seg.iterrows():
                if r["flat_roi"] < 0:
                    ax.axvspan(r["edge_lo"], min(r["edge_hi"], 14), alpha=0.12, color="red")
            ax.set_xlabel("|Model edge| (pts, rolling ±0.75 pt)")
            ax.set_ylabel("ATS win rate")
            ax.set_title("Rolling ATS vs Edge (valleys ≈ injury/noise zones)", fontweight="bold")
            ax.legend()
            _save_or_show(fig, save_path, "03c_rolling_ats_vs_edge.png")
    except Exception as e:  # noqa: BLE001
        print(f"  ⚠ rolling edge plot skipped: {e}")

    # 4 · Edge distribution (% of games in each bucket) stacked
    fig, ax = plt.subplots(figsize=(10, 6))
    bottom = np.zeros(len(EDGE_LABELS))
    for s in seasons:
        eb = edge_tables[s]
        if eb.empty:
            continue
        pcts = [eb.loc[eb["bucket"] == lbl, "pct_games"].iloc[0] if lbl in eb["bucket"].values else 0
                for lbl in EDGE_LABELS]
        ax.bar(EDGE_LABELS, pcts, bottom=bottom, label=str(s), color=palette[s], edgecolor="white", alpha=0.9)
        bottom += np.array(pcts)
    ax.set_ylabel("Share of games")
    ax.set_xlabel("|Model − market| edge bucket (pts)")
    ax.set_title("Edge Distribution by Season (stacked % of games)")
    ax.legend(fontsize=8, ncol=2)
    _save_or_show(fig, save_path, "04_edge_distribution.png")

    # 5 · Close-game rates
    fig, ax = plt.subplots(figsize=(9, 5))
    w = 0.25
    for j, cm in enumerate(CLOSE_MARGINS):
        vals = [summary.loc[summary["season"] == s, f"pct_decided_{cm}"].iloc[0] for s in seasons]
        ax.bar(np.arange(len(seasons)) + (j - 1) * w, vals, w, label=f"≤{cm} pts", edgecolor="black", alpha=0.85)
    ax.set_xticks(np.arange(len(seasons)))
    ax.set_xticklabels([str(s) for s in seasons], rotation=15, ha="right")
    ax.set_ylabel("% of games")
    ax.set_title("Games Decided by Close Margins")
    ax.legend()
    _save_or_show(fig, save_path, "05_close_games.png")

    # 6 · Total MAE by season
    if summary["total_mae"].notna().any():
        fig, ax = plt.subplots(figsize=(9, 5))
        tm = [summary.loc[summary["season"] == s, "total_mae"].iloc[0] for s in seasons]
        ax.bar([str(s) for s in seasons], tm, color="#8172B3", edgecolor="black", alpha=0.85)
        ax.set_ylabel("Total MAE (pts)")
        ax.set_title("Game Total Prediction Error by Season")
        _save_or_show(fig, save_path, "06_total_mae.png")

    # 7 · Win-probability calibration (one panel per season + pooled)
    n_panels = len(seasons) + 1
    ncols = min(3, n_panels)
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4 * nrows), squeeze=False)
    axes_flat = axes.flatten()
    pb = np.linspace(0, 1, 11)
    for idx, (label, g) in enumerate(list(zip(seasons, [df[df["_season"] == s] for s in seasons])) + [("ALL", df)]):
        ax = axes_flat[idx]
        g = g.copy()
        g["home_win"] = (g["ACTUAL_MARGIN"] > 0).astype(int)
        g["pbin"] = pd.cut(g["WIN_PROB"], pb)
        cal = g.groupby("pbin", observed=True).agg(
            pred=("WIN_PROB", "mean"), obs=("home_win", "mean"), n=("home_win", "size")
        ).dropna()
        ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
        if not cal.empty:
            ax.scatter(cal["pred"], cal["obs"], s=np.clip(cal["n"] / 5, 20, 200),
                       color="#4C72B0", edgecolor="black", zorder=3)
            ax.plot(cal["pred"], cal["obs"], color="#4C72B0", lw=1.2)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_title(str(label), fontsize=10)
        ax.set_xlabel("Pred win prob")
        ax.set_ylabel("Observed")
    for j in range(n_panels, len(axes_flat)):
        axes_flat[j].set_visible(False)
    fig.suptitle("Win-Probability Calibration (corrected spread → win prob)", fontweight="bold")
    _save_or_show(fig, save_path, "07_calibration.png")

    # 8 · Rolling spread MAE through each season
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for s in seasons:
        g = df[df["_season"] == s].copy()
        if g["date"].notna().any() and g["date"].dropna().dt.normalize().nunique() > 1:
            g = g.sort_values("date")
        else:
            g = g.sort_values("GAME_ID")
        err = (g["PRED_SPREAD"] - g["ACTUAL_MARGIN"]).abs()
        roll = err.rolling(50, min_periods=15).mean()
        prog = np.linspace(0, 1, len(g))
        ax.plot(prog, roll, label=str(s), color=palette[s], lw=2 if g["date"].dropna().dt.normalize().nunique() > 1 else 1.2)
    ax.set_xlabel("Season progress (0 → 1)")
    ax.set_ylabel("Rolling spread MAE (50-game window)")
    ax.set_title("Corrected Spread MAE Through the Season")
    ax.legend(fontsize=8, ncol=2)
    _save_or_show(fig, save_path, "08_rolling_mae.png")

    if save_path:
        print(f"\n📊 Diagnostic plots saved → {save_path}/")
        try:
            edge_report = run_edge_analysis(df, verbose=True)
            for key in ("fine_bins", "rolling", "segments", "injury_split"):
                tbl = edge_report.get(key)
                if isinstance(tbl, pd.DataFrame) and not tbl.empty:
                    tbl.to_csv(save_path / f"edge_{key}.csv", index=False)
        except Exception as e:  # noqa: BLE001
            print(f"  ⚠ edge curve analysis skipped: {e}")

    return summary, edge_tables


def generate_betting_plots(df, save_dir=None):
    """Bankroll curves, ROI by confidence tier, ATS reliability diagram."""
    import matplotlib.pyplot as plt

    save_path = Path(save_dir) if save_dir else None
    if df is None or df.empty:
        return

    df = _norm_results(df)

    if "PROFIT_MODERATE" not in df.columns:
        df = add_all_profile_columns(df)

    fig, ax = plt.subplots(figsize=(11, 6))
    df_sorted = df.sort_values("DATE") if "DATE" in df.columns else df
    for profile, color in [("conservative", "#4C72B0"), ("moderate", "#55A868"), ("aggressive", "#C44E52")]:
        pc = f"PROFIT_{profile.upper()}"
        if pc not in df_sorted.columns:
            continue
        cum = df_sorted[pc].fillna(0).cumsum()
        ax.plot(range(len(cum)), cum, label=profile, color=color, lw=2)
    ax.axhline(0, ls="--", color="gray")
    ax.set_xlabel("Bet sequence")
    ax.set_ylabel("Cumulative profit (bankroll units)")
    ax.set_title("Bankroll Curves by Stake Profile", fontweight="bold")
    ax.legend()
    _save_or_show(fig, save_path, "10_bankroll_by_profile.png")

    if "CONFIDENCE_TIER" in df.columns:
        fig, ax = plt.subplots(figsize=(8, 5))
        tiers = sorted(df["CONFIDENCE_TIER"].dropna().unique())
        rois = []
        for t in tiers:
            sub = df[df["CONFIDENCE_TIER"] == t]
            sc, pc = "STAKE_MODERATE", "PROFIT_MODERATE"
            if sc in sub.columns and sub[sc].sum() > 0:
                rois.append(sub[pc].sum() / sub[sc].sum())
            else:
                rois.append(np.nan)
        ax.bar([str(int(t)) for t in tiers], rois, color="#55A868", edgecolor="black")
        ax.axhline(0, ls="--", color="black")
        ax.set_xlabel("Confidence tier")
        ax.set_ylabel("Moderate profile ROI")
        ax.set_title("ROI by Calibrated Confidence Tier", fontweight="bold")
        _save_or_show(fig, save_path, "11_roi_by_confidence_tier.png")

    if "COVER_PROB_CALIBRATED" in df.columns and "DIRECTION" in df.columns:
        fig, ax = plt.subplots(figsize=(7, 7))
        bets = df[df["DIRECTION"] != "Pass"].copy()
        if not bets.empty:
            cover = bets["ACTUAL_MARGIN"] + bets["MARKET_SPREAD"]
            home = bets["DIRECTION"] == "Home"
            bets["won"] = (home & (cover > 0)) | (~home & (cover < 0))
            bins = np.linspace(0.45, 0.75, 8)
            bets["pb"] = pd.cut(bets["COVER_PROB_CALIBRATED"], bins=bins)
            cal = bets.groupby("pb", observed=True)["won"].mean()
            centers = [iv.mid for iv in cal.index]
            ax.plot([0, 1], [0, 1], ls="--", color="gray", label="Perfect")
            ax.scatter(centers, cal.values, s=80, color="#4C72B0", zorder=3)
            ax.set_xlabel("Calibrated cover probability")
            ax.set_ylabel("Actual cover rate")
            ax.set_title("ATS Reliability (Calibrated Confidence)", fontweight="bold")
            ax.legend()
        _save_or_show(fig, save_path, "12_confidence_reliability.png")

    # 12b · Cover-prob reliability by season (pooled panels)
    if "COVER_PROB_CALIBRATED" in df.columns and "DIRECTION" in df.columns:
        seasons = sorted(df["_season"].unique())
        n_panels = len(seasons) + 1
        ncols = min(3, n_panels)
        nrows = int(np.ceil(n_panels / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4 * nrows), squeeze=False)
        panels = list(zip(seasons, [df[df["_season"] == s] for s in seasons])) + [("ALL", df)]
        for idx, (label, g) in enumerate(panels):
            ax = axes.flatten()[idx]
            bets = g[g["DIRECTION"] != "Pass"].copy()
            if bets.empty:
                ax.set_visible(False)
                continue
            cover = bets["ACTUAL_MARGIN"] + bets["MARKET_SPREAD"]
            home = bets["DIRECTION"] == "Home"
            y = ((home & (cover > 0)) | (~home & (cover < 0))).astype(int)
            p = bets["COVER_PROB_CALIBRATED"].astype(float)
            rb = reliability_bins(y, p, n_bins=8)
            ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
            if not rb.empty:
                ax.plot(rb["pred_mean"], rb["obs_rate"], "o-", color="#8172B3", lw=1.5)
            ece = compute_ece(y, p)
            ax.set_title(f"{label}  ECE={ece:.3f}" if np.isfinite(ece) else str(label), fontsize=10)
            ax.set_xlim(0.45, 0.75)
            ax.set_ylim(0.35, 0.75)
        for j in range(len(panels), len(axes.flatten())):
            axes.flatten()[j].set_visible(False)
        fig.suptitle("ATS Cover-Prob Reliability by Season", fontweight="bold")
        _save_or_show(fig, save_path, "12b_cover_prob_reliability_by_season.png")

    if save_path:
        print(f"📊 Betting plots saved → {save_path}/")


In [ ]:
# ── module: ml_diagnostics ───────────────────────────────────────────────────
"""Moneyline diagnostics: calibration, ROI, underdog/favorite value tracking."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd



def _ml_outcome(row, side: str) -> int | float:
    if pd.isna(row.get("ACTUAL_HOME")) or pd.isna(row.get("ACTUAL_AWAY")):
        return np.nan
    home_win = float(row["ACTUAL_HOME"]) > float(row["ACTUAL_AWAY"])
    if side == "Home":
        return int(home_win)
    if side == "Away":
        return int(not home_win)
    return np.nan


def enrich_ml_columns(
    df: pd.DataFrame,
    min_ev: float = ML_MIN_EV,
    max_fav_dec: float | None = None,
    min_win_pct: float | None = None,
) -> pd.DataFrame:
    """Add per-game ML EV, best side, dog/fav tag, and flat ROI."""
    out = _norm_results(df)
    if "WIN_PROB" not in out.columns or "MARKET_ML" not in out.columns:
        return out

    if max_fav_dec is None:
        max_fav_dec = ML_MAX_FAVORITE_DECIMAL_DEFAULT
    if min_win_pct is None:
        min_win_pct = float(MIN_ML_WIN_PCT)

    rows = []
    for r in out.itertuples(index=False):
        rd = r._asdict()
        ml = rd.get("MARKET_ML")
        wp = rd.get("WIN_PROB")
        if pd.isna(ml) or pd.isna(wp):
            rows.append({
                "ml_ev_home": np.nan, "ml_ev_away": np.nan,
                "ml_best_ev": np.nan, "ml_best_side": "Pass",
                "ml_predicted_winner": "Pass",
                "ml_bet_decimal": np.nan, "ml_implied_prob": np.nan,
                "ml_value_type": "none", "ml_flat_profit": 0.0,
            })
            continue
        dec_h = american_to_decimal(float(ml))
        dec_a = american_to_decimal(float(-ml))
        p_h = ml_prob_for_side(float(wp), float(ml), "Home")
        p_a = ml_prob_for_side(float(wp), float(ml), "Away")
        ev_h = (p_h * dec_h) - 1.0
        ev_a = (p_a * dec_a) - 1.0
        side, ev, dec = select_ml_bet(
            float(wp), float(ml),
            min_ev=min_ev,
            max_favorite_decimal=max_fav_dec,
            min_win_pct=min_win_pct,
        )
        if side == "Pass":
            vtype = "none"
        elif dec >= UNDERDOG_DECIMAL:
            vtype = "underdog_value"
        else:
            vtype = "favorite_value"
        won = _ml_outcome(rd, side) if side != "Pass" else np.nan
        profit = 0.0
        if side != "Pass" and pd.notna(won):
            profit = (dec - 1.0) if won else -1.0
        bet_model_p = p_h if side == "Home" else (p_a if side == "Away" else np.nan)
        predicted_winner = ml_predicted_winner_side(float(wp))
        rows.append({
            "ml_ev_home": ev_h,
            "ml_ev_away": ev_a,
            "ml_best_ev": ev if side != "Pass" else max(ev_h, ev_a),
            "ml_best_side": side,
            "ml_predicted_winner": predicted_winner,
            "ml_bet_decimal": dec if side != "Pass" else np.nan,
            "ml_implied_prob": (
                implied_probability(float(ml) if side == "Home" else float(-ml))
                if side != "Pass" else np.nan
            ),
            "ml_bet_model_prob": bet_model_p,
            "ml_value_type": vtype,
            "ml_flat_profit": profit,
            "ml_won": won,
        })
    extra = pd.DataFrame(rows, index=out.index)
    return pd.concat([out, extra], axis=1)


def _prob_metrics(g: pd.DataFrame, prob_col: str) -> tuple[float, float]:

    if prob_col not in g.columns:
        return np.nan, np.nan
    hw = (g["ACTUAL_MARGIN"] > 0).astype(int)
    wp_col = g[prob_col].dropna()
    if wp_col.empty:
        return np.nan, np.nan
    hw_aligned = hw.loc[wp_col.index]
    return (
        float(compute_brier(hw_aligned.values, wp_col.values)),
        float(compute_ece(hw_aligned.values, wp_col.values)),
    )


def _reliability_plot(ax, g: pd.DataFrame, prob_col: str, label: str, color: str):
    sub = g.dropna(subset=[prob_col]).copy()
    if sub.empty:
        return
    sub["home_win"] = (sub["ACTUAL_MARGIN"] > 0).astype(int)
    bins = np.linspace(0.2, 0.85, 14)
    sub["pb"] = pd.cut(sub[prob_col], bins=bins)
    cal = sub.groupby("pb", observed=True)["home_win"].mean()
    centers = [iv.mid for iv in cal.index]
    ax.scatter(centers, cal.values, s=50, color=color, label=label, zorder=3)


def run_ml_diagnostics(
    results_df: pd.DataFrame,
    *,
    min_ev: float = ML_MIN_EV,
    max_fav_dec: float | None = None,
    min_win_pct: float | None = None,
    save_dir: str | Path | None = None,
    show_plots: bool = True,
) -> pd.DataFrame:
    """Win-prob calibration, ML ROI by season, dog vs fav split."""
    if results_df is None or results_df.empty:
        print("No results for ML diagnostics.")
        return pd.DataFrame()

    if max_fav_dec is None:
        max_fav_dec = walkforward_favorite_decimal(results_df)
    if min_win_pct is None:
        min_win_pct = float(MIN_ML_WIN_PCT)

    df = enrich_ml_columns(
        results_df, min_ev=min_ev, max_fav_dec=max_fav_dec, min_win_pct=min_win_pct,
    )
    save_path = Path(save_dir) if save_dir else None
    has_raw = "WIN_PROB_RAW" in df.columns

    print("\n" + "=" * 72)
    print("MONEYLINE DIAGNOSTICS (MetaWin / WIN_PROB vs market)")
    print("=" * 72)
    if ML_BET_PREDICTED_WINNER_ONLY:
        winner_mode = (
            f"predicted winner if EV>{ML_MIN_EV:.0%} and win%>={min_win_pct:.0f}; "
            f"coin-flip underdog={'on' if ML_COIN_FLIP_UNDERDOG_ONLY else 'off'}"
        )
    else:
        winner_mode = f"best calibrated-EV side if EV>{ML_MIN_EV:.0%} and win%>={min_win_pct:.0f}"
    print(f"  bet policy={winner_mode}  min_ev={min_ev:.1%}  min_win%={min_win_pct:.0f}")
    print(f"  max_favorite_decimal={max_fav_dec:.2f}  ml_calib={ML_CALIB_METHOD}")
    print(f"  market_shrink={ML_MARKET_SHRINK:.2f}  underdog_extra_shrink={ML_UNDERDOG_EXTRA_SHRINK:.2f}")

    seasons = sorted(df["_season"].unique())
    summary_rows = []

    for label, g in list(zip(seasons, [df[df["_season"] == s] for s in seasons])) + [("ALL", df)]:
        brier, ece = _prob_metrics(g, "WIN_PROB")
        raw_brier, raw_ece = _prob_metrics(g, "WIN_PROB_RAW") if has_raw else (np.nan, np.nan)
        ml_acc = float(g["MODEL_ML_CORRECT"].mean()) if "MODEL_ML_CORRECT" in g.columns else np.nan
        if has_raw and "WIN_PROB_RAW" in g.columns:
            raw_acc = float(
                ((g["WIN_PROB_RAW"] > 0.5) == (g["ACTUAL_MARGIN"] > 0)).mean()
            )
        else:
            raw_acc = np.nan
        bets = g[g["ml_best_side"] != "Pass"]
        n_ml = len(bets)
        n_games = len(g)
        wp = bets["ml_won"].mean() if n_ml else np.nan
        roi = bets["ml_flat_profit"].mean() if n_ml else np.nan
        dog = bets[bets["ml_value_type"] == "underdog_value"]
        fav = bets[bets["ml_value_type"] == "favorite_value"]
        dog_pred = float(dog["ml_bet_model_prob"].mean()) if len(dog) else np.nan
        dog_obs = float(dog["ml_won"].mean()) if len(dog) else np.nan
        fav_pred = float(fav["ml_bet_model_prob"].mean()) if len(fav) else np.nan
        fav_obs = float(fav["ml_won"].mean()) if len(fav) else np.nan
        summary_rows.append({
            "season": label,
            "games": n_games,
            "raw_brier": raw_brier,
            "brier": brier,
            "raw_ece": raw_ece,
            "ece": ece,
            "raw_winner_acc": raw_acc,
            "ml_winner_acc": ml_acc,
            "ml_bets": n_ml,
            "ml_bet_rate": n_ml / n_games if n_games else np.nan,
            "ml_win_pct": wp,
            "ml_roi": roi,
            "underdog_bets": len(dog),
            "underdog_share": len(dog) / n_ml if n_ml else np.nan,
            "underdog_pred_win": dog_pred,
            "underdog_win_pct": dog_obs,
            "underdog_cal_gap": (dog_pred - dog_obs) if pd.notna(dog_pred) and pd.notna(dog_obs) else np.nan,
            "underdog_roi": dog["ml_flat_profit"].mean() if len(dog) else np.nan,
            "favorite_bets": len(fav),
            "favorite_pred_win": fav_pred,
            "favorite_win_pct": fav_obs,
            "favorite_cal_gap": (fav_pred - fav_obs) if pd.notna(fav_pred) and pd.notna(fav_obs) else np.nan,
            "favorite_roi": fav["ml_flat_profit"].mean() if len(fav) else np.nan,
        })

    summary = pd.DataFrame(summary_rows)
    print(summary.round(3).to_string(index=False))
    print(
        "\n  Note: promote models on Brier/ECE/ROI — raw_winner_acc is reference only."
    )

    # Both-sides EV: predicted winner vs actual bet side
    if "ml_best_side" in df.columns and "WIN_PROB" in df.columns:
        tmp = df.copy()
        tmp["_pred_win"] = tmp["WIN_PROB"].map(ml_predicted_winner_side)
        bets = tmp[tmp["ml_best_side"] != "Pass"]
        if len(bets):
            mismatch = bets[bets["ml_best_side"] != bets["_pred_win"]]
            fav_pass = tmp[
                (tmp["_pred_win"].isin(["Home", "Away"]))
                & (tmp["ml_best_side"] == "Pass")
            ]
            print(
                f"\n  Both-sides EV: {len(mismatch)}/{len(bets)} bets on non-predicted winner "
                f"(dog/value); {len(fav_pass)} predicted-winner Pass (ev too low / short price)."
            )

    if show_plots and "WIN_PROB" in df.columns:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7, 7))
        g = df.dropna(subset=["WIN_PROB"])
        ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect")
        if has_raw:
            _reliability_plot(ax, g, "WIN_PROB_RAW", "Raw WIN_PROB", "#C44E52")
        _reliability_plot(ax, g, "WIN_PROB", "Calibrated WIN_PROB", "#4C72B0")
        ax.set_xlabel("Model P(home win)")
        ax.set_ylabel("Actual home win rate")
        ax.set_title("ML Win-Probability Reliability", fontweight="bold")
        ax.legend()
        if save_path:
            save_path.mkdir(parents=True, exist_ok=True)
            fig.savefig(save_path / "13_ml_winprob_reliability.png", dpi=120, bbox_inches="tight")
            plt.close(fig)
            print(f"  Saved → {save_path / '13_ml_winprob_reliability.png'}")
        else:
            plt.show()

    if save_path:
        summary.to_csv(save_path / "ml_diagnostics_summary.csv", index=False)

    return summary


def ml_value_plays(
    results_df: pd.DataFrame,
    *,
    min_ev: float = ML_MIN_EV,
    max_fav_dec: float | None = None,
    min_win_pct: float | None = None,
    value_types: tuple[str, ...] = ("underdog_value", "favorite_value"),
) -> pd.DataFrame:
    """Games flagged as underdog value or undervalued favorite."""
    if max_fav_dec is None:
        max_fav_dec = ML_MAX_FAVORITE_DECIMAL_DEFAULT
    if "ml_value_type" in results_df.columns:
        df = _norm_results(results_df)
    else:
        df = enrich_ml_columns(
            results_df, min_ev=min_ev, max_fav_dec=max_fav_dec, min_win_pct=min_win_pct,
        )
    plays = df[
        df["ml_value_type"].isin(value_types) & (df["ml_best_side"] != "Pass")
    ].copy()
    cols = [
        "DATE", "date", "GAME_ID", "HOME", "AWAY", "_season",
        "WIN_PROB", "WIN_PROB_RAW", "MARKET_ML", "ml_best_side", "ml_best_ev",
        "ml_bet_decimal", "ml_implied_prob", "ml_bet_model_prob", "ml_value_type",
        "ml_won", "ml_flat_profit",
        "ML_DIRECTION", "ML_EV", "CONFIDENCE", "CONFIDENCE_TIER",
        "ACTUAL_HOME", "ACTUAL_AWAY",
    ]
    keep = [c for c in cols if c in plays.columns]
    sort_col = "DATE" if "DATE" in plays.columns else ("date" if "date" in plays.columns else None)
    out = plays[keep]
    if sort_col:
        out = out.sort_values([sort_col, "ml_best_ev"], ascending=[True, False])
    return out


def export_ml_tracking(
    results_df: pd.DataFrame,
    path: str | Path,
    **kwargs,
) -> pd.DataFrame:
    """Write underdog/favorite value plays to CSV for tracking."""
    plays = ml_value_plays(results_df, **kwargs)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plays.to_csv(path, index=False)
    n_dog = int((plays["ml_value_type"] == "underdog_value").sum()) if not plays.empty else 0
    n_fav = int((plays["ml_value_type"] == "favorite_value").sum()) if not plays.empty else 0
    print(f"ML tracking export → {path}  ({len(plays)} plays: {n_dog} dogs, {n_fav} favorites)")
    return plays


In [ ]:
# ── module: confidence_diagnostics ───────────────────────────────────────────
"""Phase 2a unified confidence diagnostics (prior-year calibration + weight tuning)."""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm



def _roi_from_winrate(wp: float) -> float:
    return float(wp * (100.0 / 110.0) - (1.0 - wp))


def confidence_score_table(df: pd.DataFrame, score_col: str = "CONFIDENCE", n_bins: int = 10) -> pd.DataFrame:
    """ATS win rate and ROI by confidence score deciles."""
    bets = _non_push_ats(_ats_frame(_norm_results(df)))
    if bets.empty or score_col not in bets.columns:
        return pd.DataFrame()
    bets = bets.dropna(subset=[score_col])
    if bets.empty:
        return pd.DataFrame()
    try:
        bets = bets.copy()
        bets["score_bin"] = pd.qcut(bets[score_col], q=min(n_bins, bets[score_col].nunique()), duplicates="drop")
    except ValueError:
        edges = np.linspace(bets[score_col].min(), bets[score_col].max(), n_bins + 1)
        bets["score_bin"] = pd.cut(bets[score_col], bins=edges)
    rows = []
    for label, sub in bets.groupby("score_bin", observed=True):
        wp = float(sub["ats_win"].mean())
        rows.append({
            "bin": str(label),
            "score_lo": float(sub[score_col].min()),
            "score_hi": float(sub[score_col].max()),
            "score_mean": float(sub[score_col].mean()),
            "n_bets": len(sub),
            "ats_pct": wp,
            "roi": _roi_from_winrate(wp),
        })
    return pd.DataFrame(rows)


def confidence_threshold_grid(
    df: pd.DataFrame,
    thresholds=range(50, 66),
    score_col: str = "CONFIDENCE",
) -> pd.DataFrame:
    bets = _non_push_ats(_ats_frame(_norm_results(df)))
    if bets.empty or score_col not in bets.columns:
        return pd.DataFrame()
    rows = []
    for thr in thresholds:
        sub = bets[bets[score_col] >= thr]
        if len(sub) < 20:
            continue
        wp = float(sub["ats_win"].mean())
        rows.append({
            "min_confidence": thr,
            "n_bets": len(sub),
            "ats_pct": wp,
            "flat_roi": _roi_from_winrate(wp),
        })
    return pd.DataFrame(rows)


def prior_year_calibration_ablation(
    results_df: pd.DataFrame,
    methods: tuple[str, ...] = ("logistic_isotonic", "logistic_features", "isotonic", "platt", "none"),
    season_col: str = "simulated_season_window",
    show_progress: bool = True,
) -> pd.DataFrame:
    """Calibrate each test season from the immediately prior season only."""
    df = _norm_results(results_df)
    if season_col not in df.columns:
        return pd.DataFrame()

    seasons = sorted(df[season_col].dropna().unique())
    tasks = [
        (seasons[i - 1], test_season, method)
        for i, test_season in enumerate(seasons)
        if i > 0
        for method in methods
    ]
    rows = []
    for train_season, test_season, method in tqdm(
        tasks,
        desc="2a · calibration ablation",
        disable=not show_progress,
        leave=False,
    ):
        prior = df[df[season_col] == train_season]
        test = df[df[season_col] == test_season]
        cal = WalkForwardBetCalibrator(method=method)
        cal.fit(prior, method=method, scope="all_prior")
        outcomes, probs = [], []
        for _, row in test.iterrows():
            y = WalkForwardBetCalibrator._ats_outcome(row)
            if y is None:
                continue
            row_dict = cal._enrich_ats_row(row)
            raw = cal._build_score(row_dict, "ats")
            feat = cal._row_features(row_dict, "ats")
            prob = cal._predict_prob("ats", raw, feat)
            probs.append(prob)
            outcomes.append(y)
        if len(outcomes) < 20:
            continue
        y = np.asarray(outcomes, dtype=float)
        p = np.asarray(probs, dtype=float)
        wp = float(y.mean())
        rows.append({
            "season": test_season,
            "train_season": train_season,
            "method": method,
            "n_bets": len(y),
            "ats_pct": wp,
            "roi": _roi_from_winrate(wp),
            "brier": compute_brier(y, p),
            "ece": compute_ece(y, p),
        })
    return pd.DataFrame(rows)


def run_phase_2a_unified_confidence(
    results_df: pd.DataFrame,
    *,
    save_dir: str | Path | None = None,
    show_plots: bool = True,
    show_progress: bool = True,
) -> dict:
    """
    Phase 2a diagnostics (plots + tables).
    Weight tuning runs **inline during walk-forward backtest** when CONFIDENCE_MODE='unified'.
    This cell replays diagnostics only unless backtest was run without inline Phase 2a.
    """
    if results_df is None or results_df.empty:
        print("No results for Phase 2a confidence diagnostics.")
        return {}

    df = _norm_results(results_df)
    save_path = Path(save_dir) if save_dir else None
    out: dict = {
        "mode": CONFIDENCE_MODE,
        "calib_method": CONFIDENCE_CALIB_METHOD,
        "calib_scope": "prior_year",
        "weight_shrink": CONFIDENCE_WEIGHT_SHRINK,
        "min_confidence_default": MIN_CONFIDENCE_SCORE,
        "inline_phase2a": bool(
            "PHASE2A_INLINE" in df.columns and df["PHASE2A_INLINE"].fillna(0).astype(int).max() > 0
        ),
    }

    # Phase 2 default (from backtest CSV)
    phase_steps = tqdm(
        total=7,
        desc="Phase 2a",
        disable=not show_progress,
        unit="step",
    )

    phase_steps.set_description("2a · default deciles")
    default_table = confidence_score_table(df, "CONFIDENCE")
    default_raw = confidence_score_table(df, "CONFIDENCE_RAW") if "CONFIDENCE_RAW" in df.columns else pd.DataFrame()
    phase_steps.update(1)

    phase_steps.set_description("2a · weight tuning")
    weight_tuning = yearly_confidence_weight_tuning(
        df, method=CONFIDENCE_CALIB_METHOD, show_progress=show_progress,
    )
    phase_steps.update(1)

    phase_steps.set_description("2a · threshold grid")
    thr_grid = confidence_threshold_grid(df)
    phase_steps.update(1)

    phase_steps.set_description("2a · calibration ablation")
    ablation = prior_year_calibration_ablation(df, show_progress=show_progress)
    phase_steps.update(1)

    phase_steps.set_description("2a · summarize")

    out["default_confidence_deciles"] = default_table.to_dict(orient="records")
    out["default_raw_deciles"] = default_raw.to_dict(orient="records") if not default_raw.empty else []
    out["unified_weight_tuning"] = weight_tuning.to_dict(orient="records")
    out["threshold_grid"] = thr_grid.to_dict(orient="records")
    out["prior_year_calibration_ablation"] = ablation.to_dict(orient="records")

    if not weight_tuning.empty:
        latest = weight_tuning.iloc[-1]
        suggested = {
            k.replace("w_", ""): float(latest[k])
            for k in weight_tuning.columns if k.startswith("w_")
        }
        out["suggested_weights_latest"] = suggested
        if "train_min_confidence" in latest:
            out["suggested_min_confidence"] = int(latest["train_min_confidence"])
        elif not thr_grid.empty:
            out["suggested_min_confidence"] = int(
                thr_grid.loc[thr_grid["flat_roi"].idxmax()]["min_confidence"]
            )
        else:
            out["suggested_min_confidence"] = MIN_CONFIDENCE_SCORE
        out["weights_changed_latest"] = bool(latest.get("weights_changed", False))

    phase_steps.update(1)

    phase_steps.set_description("2a · plots")
    if show_plots:
        import matplotlib.pyplot as plt

        if not default_table.empty:
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.plot(default_table["score_mean"], default_table["ats_pct"], "o-", color="#4C72B0", lw=2, label="Default")
            ax.axhline(BREAKEVEN_ATS, ls="--", color="gray", label="Break-even")
            ax.set_xlabel("Mean confidence score")
            ax.set_ylabel("ATS win rate")
            ax.set_title("Phase 2 Default Confidence Reliability", fontweight="bold")
            ax.legend()
            _save_or_show(fig, save_path, "14_confidence_reliability.png")

        if not default_raw.empty:
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.plot(default_raw["score_mean"], default_raw["ats_pct"], "o-", color="#C44E52", lw=2)
            ax.axhline(BREAKEVEN_ATS, ls="--", color="gray")
            ax.set_xlabel("Mean raw confidence score")
            ax.set_ylabel("ATS win rate")
            ax.set_title("Raw Confidence Score Reliability", fontweight="bold")
            _save_or_show(fig, save_path, "14b_raw_confidence_reliability.png")

        if not thr_grid.empty:
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.plot(thr_grid["min_confidence"], thr_grid["flat_roi"], "o-", color="#55A868", lw=2)
            ax.axhline(0, ls="--", color="black")
            ax.set_xlabel("Minimum confidence threshold")
            ax.set_ylabel("Flat ROI")
            ax.set_title("ROI vs Minimum Confidence Gate", fontweight="bold")
            _save_or_show(fig, save_path, "15_roi_vs_confidence_threshold.png")

        if not ablation.empty:
            fig, ax = plt.subplots(figsize=(9, 5))
            for method, g in ablation.groupby("method"):
                ax.plot(g["season"].astype(str), g["ece"], "o-", label=method, lw=1.8)
            ax.set_xlabel("Test season")
            ax.set_ylabel("ECE (lower is better)")
            ax.set_title("Prior-Year Calibration Method Ablation (ECE)", fontweight="bold")
            ax.legend(fontsize=8)
            plt.xticks(rotation=20)
            _save_or_show(fig, save_path, "16_confidence_calibration_ablation.png")

            fig, ax = plt.subplots(figsize=(9, 5))
            for method, g in ablation.groupby("method"):
                ax.plot(g["season"].astype(str), g["brier"], "s--", label=method, lw=1.5)
            ax.set_xlabel("Test season")
            ax.set_ylabel("Brier score")
            ax.set_title("Prior-Year Calibration Method Ablation (Brier)", fontweight="bold")
            ax.legend(fontsize=8)
            plt.xticks(rotation=20)
            _save_or_show(fig, save_path, "16c_confidence_calibration_brier.png")

        if not weight_tuning.empty and "test_roi" in weight_tuning.columns:
            fig, ax = plt.subplots(figsize=(9, 5))
            ax.plot(
                weight_tuning["test_season"].astype(str),
                weight_tuning["test_roi"],
                "o-",
                color="#8172B2",
                lw=2,
                label="Unified tuned weights",
            )
            ax.plot(
                weight_tuning["test_season"].astype(str),
                weight_tuning["train_roi"],
                "s--",
                color="#CCB974",
                lw=1.5,
                label="In-sample (train season)",
            )
            ax.axhline(0, ls="--", color="gray")
            ax.set_xlabel("Test season")
            ax.set_ylabel("ATS ROI")
            ax.set_title("Unified Weight Tuning — Prior Year Fit", fontweight="bold")
            ax.legend()
            plt.xticks(rotation=20)
            _save_or_show(fig, save_path, "16b_unified_weight_tuning_roi.png")

        bets = _non_push_ats(_ats_frame(df))
        if not bets.empty and "CONFIDENCE" in bets.columns:
            fig, ax = plt.subplots(figsize=(7, 7))
            rb = reliability_bins(bets["ats_win"].values, bets["CONFIDENCE"].values / 100.0, n_bins=8)
            if not rb.empty:
                ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect")
                ax.scatter(rb["pred_mean"], rb["obs_rate"], s=rb["n"] * 3, color="#4C72B0", zorder=3)
                ax.set_xlabel("Default calibrated confidence / 100")
                ax.set_ylabel("Observed ATS win rate")
                ax.set_title("Default Confidence Reliability Diagram", fontweight="bold")
                ax.legend()
            _save_or_show(fig, save_path, "12_confidence_reliability.png")

    phase_steps.update(1)

    phase_steps.set_description("2a · save")
    if save_path:
        save_path.mkdir(parents=True, exist_ok=True)
        if not default_table.empty:
            default_table.to_csv(save_path / "confidence_deciles_default.csv", index=False)
        if not weight_tuning.empty:
            weight_tuning.to_csv(save_path / "unified_confidence_weights_by_season.csv", index=False)
        if not thr_grid.empty:
            thr_grid.to_csv(save_path / "confidence_threshold_grid.csv", index=False)
        if not ablation.empty:
            ablation.to_csv(save_path / "confidence_calibration_ablation.csv", index=False)

    summary_path = confidence_weights_path()
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    with open(summary_path, "w") as f:
        json.dump(out, f, indent=2, default=str)
    ablation_path = STATE_DIR / "confidence_calibration_ablation.json"
    with open(ablation_path, "w") as f:
        json.dump(out, f, indent=2, default=str)

    phase_steps.update(1)
    phase_steps.close()

    print("\n" + "=" * 72)
    print("PHASE 2a · UNIFIED CONFIDENCE (inline walk-forward + diagnostics)")
    print("=" * 72)
    inline = (
        "PHASE2A_INLINE" in df.columns
        and df["PHASE2A_INLINE"].fillna(0).astype(int).max() > 0
    )
    if inline:
        print("Weights/calibration were tuned each season inside the walk-forward backtest.")
        print("This section shows reliability plots and threshold grids on those scores.\n")
    else:
        print("Replaying Phase 2a tuning post-hoc (re-run backtest with CONFIDENCE_MODE='unified' for inline).\n")

    if not default_table.empty:
        print("Default CONFIDENCE deciles (from Phase 2 backtest):")
        print(default_table.round(3).to_string(index=False))

    if not weight_tuning.empty:
        print("\nSuggested per-variable weights (latest season):")
        wcols = [c for c in weight_tuning.columns if c.startswith("w_")]
        display_cols = [
            "test_season", "train_season", "train_roi", "test_roi",
            "train_min_confidence", "train_gated_n_bets", "test_gated_n_bets",
        ] + wcols[:4]
        display_cols = [c for c in display_cols if c in weight_tuning.columns]
        print(weight_tuning[display_cols].round(4).tail(3).to_string(index=False))
        if not out.get("weights_changed_latest"):
            print("\n⚠️  Weight tuner returned defaults for latest season — "
                  "try increasing CONFIDENCE_WEIGHT_SEARCH_SAMPLES or check train bet count.")
        if "suggested_weights_latest" in out:
            print("\nApply for live/unified mode (pipeline/config.py CONFIDENCE_MODE='unified'):")
            for k, v in out["suggested_weights_latest"].items():
                rng_col = f"range_{k}"
                rng = weight_tuning.iloc[-1].get(rng_col, "")
                print(f"  {k}: {v:.4f}  {rng}")

    print(f"\n  Weight suggestions → {summary_path}")
    if save_path:
        print(f"  Plots/tables → {save_path}/")

    return out


# Backward-compatible alias
def run_confidence_diagnostics(results_df, **kwargs):
    return run_phase_2a_unified_confidence(results_df, **kwargs)


In [ ]:
# ── module: monitoring ───────────────────────────────────────────────────────
"""Model monitoring and drift detection for live deployment."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd



def load_prediction_log(path) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        return pd.DataFrame()
    return pd.read_csv(p)


def weekly_report(results_df: pd.DataFrame, window: int = 30) -> dict:
    """Summary metrics for the last ``window`` bets/games."""
    if results_df is None or results_df.empty:
        return {"status": "no_data"}
    df = add_clv_columns(results_df.tail(window))
    wins = ats_win_series(df)
    mae = (df["PRED_SPREAD"] - df["ACTUAL_MARGIN"]).abs().mean() if "ACTUAL_MARGIN" in df else np.nan
    _, lo, hi = bootstrap_ci(wins.astype(float)) if len(wins) else (np.nan, np.nan, np.nan)
    clv = df["CLV"].mean() if "CLV" in df else np.nan
    alert = bool(len(wins) >= 20 and wins.mean() < 0.48)
    return {
        "n_games": len(df),
        "n_bets": len(wins),
        "ats_pct": wins.mean() if len(wins) else np.nan,
        "ats_ci_lo": lo,
        "ats_ci_hi": hi,
        "spread_mae": mae,
        "mean_clv": clv,
        "alert_underperformance": alert,
    }


def feature_drift(train_features: pd.DataFrame, live_features: pd.DataFrame,
                  cols=None, threshold: float = 2.0) -> pd.DataFrame:
    """Flag features whose live mean deviates > threshold std from training."""
    cols = cols or [c for c in train_features.columns if c in live_features.columns]
    rows = []
    for c in cols:
        if not np.issubdtype(train_features[c].dtype, np.number):
            continue
        tm, ts = train_features[c].mean(), train_features[c].std() or 1e-6
        lm = live_features[c].mean()
        z = abs(lm - tm) / ts
        if z >= threshold:
            rows.append({"feature": c, "train_mean": tm, "live_mean": lm, "z_score": z})
    return pd.DataFrame(rows)


In [ ]:
# ── module: shap_prune ───────────────────────────────────────────────────────
"""SHAP-guided feature pruning with walk-forward stability gate.

Workflow per market (spread / ML / total):
  1. Fit a baseline tree model
  2. Rank features by mean |SHAP|
  3. Drop bottom ~30%
  4. Retrain
  5. Keep only features in the top 50% of importance in >80% of folds
"""
from __future__ import annotations

from typing import Iterable

import numpy as np
import pandas as pd


def _mean_abs_shap(model, X: pd.DataFrame) -> pd.Series:
    """Return mean |SHAP| per feature. Falls back to tree feature_importances_."""
    try:
        import shap
        explainer = shap.TreeExplainer(model)
        values = explainer.shap_values(X)
        if isinstance(values, list):
            values = values[1] if len(values) > 1 else values[0]
        vals = np.abs(np.asarray(values)).mean(axis=0)
        return pd.Series(vals, index=list(X.columns))
    except Exception:
        if hasattr(model, "feature_importances_"):
            return pd.Series(model.feature_importances_, index=list(X.columns))
        raise RuntimeError("Could not compute SHAP or feature_importances_")


def prune_bottom_fraction(importance: pd.Series, drop_frac: float = 0.30) -> list[str]:
    """Keep top (1 - drop_frac) features by importance."""
    if importance.empty:
        return []
    keep_n = max(1, int(np.ceil(len(importance) * (1.0 - drop_frac))))
    return list(importance.sort_values(ascending=False).head(keep_n).index)


def stable_top_features(
    fold_rankings: Iterable[pd.Series],
    *,
    top_frac: float = 0.50,
    min_fold_frac: float = 0.80,
) -> list[str]:
    """Keep features that appear in the top_frac of importance in >= min_fold_frac folds."""
    fold_sets = []
    for imp in fold_rankings:
        if imp is None or len(imp) == 0:
            continue
        k = max(1, int(np.ceil(len(imp) * top_frac)))
        fold_sets.append(set(imp.sort_values(ascending=False).head(k).index))
    if not fold_sets:
        return []
    n = len(fold_sets)
    threshold = min_fold_frac * n
    counts: dict[str, int] = {}
    for s in fold_sets:
        for f in s:
            counts[f] = counts.get(f, 0) + 1
    return [f for f, c in counts.items() if c >= threshold]


def shap_prune_loop(
    model_factory,
    X: pd.DataFrame,
    y,
    *,
    drop_frac: float = 0.30,
    n_rounds: int = 2,
    target_size: tuple[int, int] = (40, 60),
    fold_importances: list[pd.Series] | None = None,
) -> list[str]:
    """Iterative SHAP prune toward target feature count, with optional stability gate.

    model_factory(X_cols) -> fitted model with predict / feature_importances_ or TreeExplainer support.
    """
    cols = list(X.columns)
    for _ in range(max(1, n_rounds)):
        if len(cols) <= target_size[0]:
            break
        model = model_factory(X[cols])
        imp = _mean_abs_shap(model, X[cols])
        cols = prune_bottom_fraction(imp, drop_frac=drop_frac)
        if fold_importances:
            stable = stable_top_features(fold_importances + [imp])
            if stable:
                cols = [c for c in cols if c in stable] or cols
        if len(cols) <= target_size[1]:
            break
    # Cap at upper target if still too many
    if len(cols) > target_size[1]:
        model = model_factory(X[cols])
        imp = _mean_abs_shap(model, X[cols])
        cols = list(imp.sort_values(ascending=False).head(target_size[1]).index)
    return cols


---

## Phase 0 — Runtime toggles

Saves `checkpoints/runtime_toggles.json`. Set `USE_SAVED_ARTIFACTS = False` to force a full retrain.


In [ ]:
# ============================================================================
#  PHASE 0 · Runtime toggles (speed vs accuracy)
# ============================================================================
USE_SAVED_ARTIFACTS = False  # False: always recompute (recommended after model changes)
QUICK = False
FAST_BACKTEST = False
USE_STINTS_CACHE = True
USE_TUNING_CACHE = True
WALKFORWARD_ZONE_PPS = True

if FAST_BACKTEST:
    ELO_TRIALS, HIER_TRIALS, META_TRIALS, TOTAL_TRIALS, META_WIN_TRIALS, WINDOW = 12, 15, 15, 8, 12, 2
else:
    ELO_TRIALS, HIER_TRIALS, META_TRIALS, TOTAL_TRIALS, META_WIN_TRIALS, WINDOW = 150, 140, 50, 20, 25, 4

toggle_payload = {
    "USE_SAVED_ARTIFACTS": USE_SAVED_ARTIFACTS,
    "QUICK": QUICK,
    "FAST_BACKTEST": FAST_BACKTEST,
    "USE_STINTS_CACHE": USE_STINTS_CACHE,
    "USE_TUNING_CACHE": USE_TUNING_CACHE,
    "WALKFORWARD_ZONE_PPS": WALKFORWARD_ZONE_PPS,
    "ELO_TRIALS": ELO_TRIALS,
    "HIER_TRIALS": HIER_TRIALS,
    "META_TRIALS": META_TRIALS,
    "TOTAL_TRIALS": TOTAL_TRIALS,
    "META_WIN_TRIALS": META_WIN_TRIALS,
    "WINDOW": WINDOW,
    "BET_SELECTION_MODE": BET_SELECTION_MODE,
    "MIN_EDGE_BUCKET": MIN_EDGE_BUCKET,
    "MIN_CONFIDENCE_SCORE": MIN_CONFIDENCE_SCORE,
    "CONFIDENCE_MIN_EDGE": CONFIDENCE_MIN_EDGE,
    "CONFIDENCE_EDGE_SCALED": CONFIDENCE_EDGE_SCALED,
    "MIN_DISAGREEMENT_TRUST": MIN_DISAGREEMENT_TRUST,
    "SKIP_PHANTOM_INJURY": SKIP_PHANTOM_INJURY,
    "SKIP_TIGHT_SPREAD": SKIP_TIGHT_SPREAD,
    "EDGE_AVOID_BAND": EDGE_AVOID_BAND,
    "TOTAL_HEAD_CV_SANITY_CAP": TOTAL_HEAD_CV_SANITY_CAP,
    "MAX_QUANTILE_WIDTH": MAX_QUANTILE_WIDTH,
    "STAKE_SIZING_MODE": STAKE_SIZING_MODE,
    "CONFIDENCE_MODE": CONFIDENCE_MODE,
}
save_json_artifact("toggles", toggle_payload)
print(
    f"Toggles: USE_SAVED={USE_SAVED_ARTIFACTS}  QUICK={QUICK}  FAST_BACKTEST={FAST_BACKTEST}  "
    f"stints_cache={USE_STINTS_CACHE}  tuning_cache={USE_TUNING_CACHE}"
)
print(f"Trials: elo={ELO_TRIALS} hier={HIER_TRIALS} meta={META_TRIALS} total={TOTAL_TRIALS} meta_win={META_WIN_TRIALS} window={WINDOW}")


---

## Phase 1 — Load data

Loads from checkpoint when `USE_SAVED_ARTIFACTS=True` and files exist; otherwise builds stints + odds.


In [ ]:
# ============================================================================
#  PHASE 1 · Load odds + build leak-free stint timelines
# ============================================================================
import hashlib

def read_csv_fast(path):
    try:
        return pd.read_csv(path, engine="pyarrow")
    except Exception:
        return pd.read_csv(path, low_memory=False)

def _stints_cache_key(paths, quick, walkforward_zone_pps):
    h = hashlib.md5()
    for season, path in sorted(paths, key=lambda kv: kv[0]):
        p = Path(path)
        if p.exists():
            stt = p.stat()
            h.update(f"{season}:{p.name}:{int(stt.st_mtime)}:{stt.st_size}".encode())
    h.update(f"q={quick};wzpps={walkforward_zone_pps};as={ASSIST_SPLIT}".encode())
    return h.hexdigest()[:16]

def load_all_stints(quick=False, walkforward_zone_pps=True, use_cache=True) -> pd.DataFrame:
    paths = list(V3_DATA_PATHS.items())
    if PBP_2026_PATH.exists():
        paths.append((2025, PBP_2026_PATH))
    cache_path = None
    if use_cache:
        cache_path = STATE_DIR / f"stints_cache_{_stints_cache_key(paths, quick, walkforward_zone_pps)}.pkl"
        if cache_path.exists():
            try:
                print(f"  ⚡ loading cached stints from {cache_path.name} ...")
                cached = pd.read_pickle(cache_path)
                print(f"  ✅ cache hit: {len(cached):,} stint rows")
                return cached
            except Exception as e:
                print(f"  ⚠️ cache read failed ({e}); rebuilding from source")
    sched = None
    try:
        if MODERN_ODDS_PATH.exists():
            sched = load_odds_schedule(str(MODERN_ODDS_PATH))
            print(f"  schedule for date recovery: {len(sched)} games")
    except Exception as e:
        print(f"  ⚠️ schedule load failed ({e})")
        sched = None
    frames = []
    prev_zone_pps = None
    for season, path in sorted(paths, key=lambda kv: kv[0]):
        if not Path(path).exists():
            print(f"  skip missing {path}")
            continue
        print(f"  loading {Path(path).name} ...")
        raw = read_csv_fast(path)
        df = convert_new_pbp(raw, name_to_id=name_to_id) if season >= 2025 else convert_v3_pbp(raw)
        df = preprocess_pbp(df, compute_xpoints=True,
                            zone_pps=prev_zone_pps if walkforward_zone_pps else None)
        if walkforward_zone_pps:
            _zp = compute_zone_pps(df)
            if _zp:
                prev_zone_pps = _zp
        st = build_stints(df, assist_split=ASSIST_SPLIT)
        st["season"] = season + 1
        if season < 2025 and sched is not None and not st.empty:
            _gd = pd.to_datetime(st["game_date"], errors="coerce")
            if _gd.dt.normalize().nunique() <= 1:
                st = attach_real_dates(st, sched, season_start_year=season)
        frames.append(st)
        del raw, df
        gc.collect()
        if quick and len(frames) >= 2:
            break
    if not frames:
        raise FileNotFoundError("No PBP data found under basketballData/")
    allst = pd.concat(frames, ignore_index=True)
    allst["game_date"] = pd.to_datetime(allst["game_date"], errors="coerce")
    allst = allst.sort_values(["game_date", "GAME_ID", "stint_id"]).reset_index(drop=True)
    if use_cache and cache_path is not None:
        try:
            allst.to_pickle(cache_path)
            print(f"  💾 cached stints -> {cache_path.name} ({len(allst):,} rows)")
        except Exception as e:
            print(f"  ⚠️ cache write failed: {e}")
    return allst

phase1_loaded = False
if use_saved_artifacts("1") and _artifact_ready("odds", "phase1_manifest"):
    print("⚡ Loading Phase 1 artifacts from disk ...")
    manifest = load_json_artifact("phase1_manifest")
    modern_odds_dict = load_pickle_artifact("odds")
    all_stints = load_all_stints(
        quick=manifest.get("QUICK", QUICK),
        walkforward_zone_pps=manifest.get("WALKFORWARD_ZONE_PPS", WALKFORWARD_ZONE_PPS),
        use_cache=True,
    )
    if ARTIFACTS["load_summary"].exists():
        load_summary = pd.read_csv(ARTIFACTS["load_summary"])
        display(load_summary.round(3))
    phase1_loaded = True
    print(f"✅ loaded stints={len(all_stints):,}  odds={len(modern_odds_dict):,}")

if not phase1_loaded:
    print("⏳ Building stint timelines...")
    all_stints = load_all_stints(
        quick=QUICK,
        walkforward_zone_pps=WALKFORWARD_ZONE_PPS,
        use_cache=USE_STINTS_CACHE,
    )
    print(f"✅ stints={len(all_stints):,}  games={all_stints['GAME_ID'].nunique():,}  seasons={sorted(all_stints['season'].unique())}")

    print("⏳ Loading modern Vegas lines...")
    modern_odds_dict = load_modern_odds(str(MODERN_ODDS_PATH)) if MODERN_ODDS_PATH.exists() else {}
    print(f"   all_odds entries: {len(modern_odds_dict):,}")

    if PINNACLE_LINES_PATH.exists():
        _sched_2026 = (
            all_stints[all_stints["season"] == 2026][["GAME_ID", "game_date", "home_team", "away_team"]]
            .drop_duplicates("GAME_ID")
            .rename(columns={"game_date": "date", "home_team": "home", "away_team": "away"})
        )
        _pin = load_pinnacle_lines(
            str(PINNACLE_LINES_PATH),
            schedule_df=_sched_2026 if not _sched_2026.empty else None,
        )
        modern_odds_dict.update(_pin)
        print(f"   merged {len(_pin):,} Pinnacle 2026 entries -> total {len(modern_odds_dict):,}")
    else:
        print("   (no nba_main_lines.csv found — using all_odds.csv only)")

    load_summary = diagnose_loaded_data(all_stints, modern_odds_dict)
    display(load_summary.round(3))

    save_pickle_artifact("odds", modern_odds_dict)
    load_summary.to_csv(ARTIFACTS["load_summary"], index=False)
    save_json_artifact("phase1_manifest", {
        "QUICK": QUICK,
        "WALKFORWARD_ZONE_PPS": WALKFORWARD_ZONE_PPS,
        "n_stint_rows": int(len(all_stints)),
        "n_games": int(all_stints["GAME_ID"].nunique()),
        "seasons": [int(s) for s in sorted(all_stints["season"].unique())],
        "n_odds": int(len(modern_odds_dict)),
    })


---

## Phase 2 — Walk-forward backtest

Loads `backtest_results.csv` when `USE_SAVED_ARTIFACTS=True`; otherwise runs full walk-forward.


In [ ]:
# ============================================================================
#  PHASE 2 · Walk-forward backtest (no leakage) + ROI-first stake profiles
# ============================================================================
import numpy as np

results = pd.DataFrame()
if use_saved_artifacts("2") and ARTIFACTS["backtest"].exists():
    print(f"⚡ Loading backtest results from {ARTIFACTS['backtest']} ...")
    results = load_results_csv()
    if not results.empty:
        if "STAKE_MODERATE" not in results.columns:
            results = add_all_profile_columns(results)
        print(f"✅ loaded {len(results):,} games from CSV")
        if ARTIFACTS["tuning"].exists():
            print(f"   tuning config: {ARTIFACTS['tuning']}")

if results.empty:
    print("⏳ Running full walk-forward backtest (no saved results or USE_SAVED_ARTIFACTS=False) ...")
    results = run_multi_year_backtest_walkforward(
        all_stints,
        odds_dict=modern_odds_dict,
        n_tuning_trials_elo=ELO_TRIALS,
        n_tuning_trials_hier=HIER_TRIALS,
        n_tuning_trials_meta=META_TRIALS,
        n_tuning_trials_total=TOTAL_TRIALS,
        rolling_window_size=WINDOW,
        tune_total_head=True,
        train_win_model=True,
        use_tuning_cache=USE_TUNING_CACHE,
        fast_tuning=FAST_BACKTEST,
    )

if not results.empty:
    if "STAKE_MODERATE" not in results.columns:
        results = add_all_profile_columns(results)

    benchmark_results(results)
    print_accuracy_layers(results)

    best_edge, grid = grid_search_bet_edge(results)
    print("\nGOOD_BET_EDGE grid search:")
    print(grid.to_string(index=False) if not grid.empty else "  insufficient bets")
    print(f"Recommended OPTIMAL_BET_EDGE = {best_edge}")

    min_conf_last = (
        float(results["MIN_CONFIDENCE_SCORE"].iloc[-1])
        if "MIN_CONFIDENCE_SCORE" in results.columns
        else float(MIN_CONFIDENCE_SCORE)
    )
    strat = compare_selection_strategies(
        results,
        min_conf=min_conf_last,
        confidence_min_edge=CONFIDENCE_MIN_EDGE,
    )
    print("\nSelection strategy comparison (same walk-forward export):")
    print(strat.to_string(index=False) if not strat.empty else "  insufficient data")

    profile_stats = {}
    recommended = "moderate"
    best_roi = -1e9
    for profile in ("conservative", "moderate", "aggressive"):
        stats = benchmark_betting_roi(results, profile=profile)
        profile_stats[profile] = stats
        roi = stats.get("roi", stats.get("roi_pct", float("nan")))
        ci_lo = stats.get("ci_lo", float("nan"))
        print(f"  {profile}: ROI={roi:.2f}%  bets={stats.get('n_bets', 0)}  CI=[{ci_lo:.2f}, {stats.get('ci_hi', float('nan')):.2f}]")
        if np.isfinite(roi) and np.isfinite(ci_lo) and ci_lo > 0 and roi > best_roi:
            best_roi = roi
            recommended = profile
    print(f"\nRecommended stake profile (blind walk-forward): {recommended}")

    edge_thr = float(results["EDGE_THRESHOLD"].iloc[-1]) if "EDGE_THRESHOLD" in results.columns else best_edge
    fav_dec = float(results["MAX_FAVORITE_DECIMAL"].iloc[-1]) if "MAX_FAVORITE_DECIMAL" in results.columns else 1.45
    ou_thr = float(results["OU_MIN_EDGE"].iloc[-1]) if "OU_MIN_EDGE" in results.columns else 3.0
    elo_cal = {}
    knobs_path = STATE_DIR / "elo_calibration_knobs.json"
    if knobs_path.exists():
        elo_cal = json.loads(knobs_path.read_text())

    tuning_payload = {
        "optimal_bet_edge": best_edge,
        "walkforward_edge_threshold": edge_thr,
        "max_favorite_decimal": fav_dec,
        "ou_min_edge": ou_thr,
        "recommended_profile": recommended,
        "min_confidence_score": float(results["MIN_CONFIDENCE_SCORE"].iloc[-1]) if "MIN_CONFIDENCE_SCORE" in results.columns else MIN_CONFIDENCE_SCORE,
        "suggested_min_confidence": float(results["MIN_CONFIDENCE_SCORE"].iloc[-1]) if "MIN_CONFIDENCE_SCORE" in results.columns else MIN_CONFIDENCE_SCORE,
        "confidence_mode": CONFIDENCE_MODE,
        "confidence_min_edge": CONFIDENCE_MIN_EDGE,
        "confidence_edge_scaled": CONFIDENCE_EDGE_SCALED,
        "spread_calib_window": SPREAD_CALIB_WINDOW,
        "last_walkforward_season": results["simulated_season_window"].iloc[-1] if "simulated_season_window" in results.columns else None,
        "blind_ats_roi": {p: profile_stats.get(p, {}).get("roi") for p in profile_stats},
        "blind_roi_ci": {
            p: {"lo": profile_stats.get(p, {}).get("ci_lo"), "hi": profile_stats.get(p, {}).get("ci_hi")}
            for p in profile_stats
        },
        "elo_calibration": elo_cal,
        "grid": grid.to_dict(orient="records") if not grid.empty else [],
    }
    ARTIFACTS["tuning"].parent.mkdir(parents=True, exist_ok=True)
    ARTIFACTS["tuning"].write_text(json.dumps(tuning_payload, indent=2))
    results.to_csv(ARTIFACTS["backtest"], index=False)
    print(f"💾 Saved → {ARTIFACTS['backtest']}")
    print(f"💾 Tuning config → {ARTIFACTS['tuning']}")
else:
    print("No backtest results produced.")


---

## Phase 2a — Unified confidence diagnostics

**Tuning runs inline** during the walk-forward backtest (`CONFIDENCE_MODE='unified'`, `isotonic` on composite score).
This cell plots reliability / threshold grids from saved results — it does **not** re-tune unless you re-run backtest without inline Phase 2a.


In [ ]:
# ============================================================================
#  PHASE 2a · Unified confidence diagnostics (tuning is inline in walk-forward)
# ============================================================================
%matplotlib inline

results = ensure_results()
conf_diag = {}
conf_plots = (
    "14_confidence_reliability.png",
    "14b_raw_confidence_reliability.png",
    "15_roi_vs_confidence_threshold.png",
    "16_confidence_calibration_ablation.png",
    "16b_unified_weight_tuning_roi.png",
    "16c_confidence_calibration_brier.png",
)
plot_dir = ROOT / "analysis_plots"

inline_p2a = (
    not results.empty
    and "PHASE2A_INLINE" in results.columns
    and results["PHASE2A_INLINE"].fillna(0).astype(int).max() > 0
)

if use_saved_artifacts("2a") and ARTIFACTS["confidence_weights"].exists() and all((plot_dir / n).exists() for n in conf_plots):
    print("⚡ Phase 2a confidence artifacts already on disk (skipping recompute)")
    import json
    with open(ARTIFACTS["confidence_weights"]) as f:
        conf_diag = json.load(f)
elif not results.empty:
    if inline_p2a:
        print("✅ Phase 2a weights were tuned inline during walk-forward — diagnostics/plots only")
    conf_diag = run_phase_2a_unified_confidence(
        results,
        save_dir=plot_dir,
        show_plots=True,
    )
else:
    print("Skip Phase 2a — no backtest results.")

from IPython.display import Image, display
for name in conf_plots:
    p = plot_dir / name
    if p.exists():
        display(Image(filename=str(p)))


---

## Phase 2b — Spread diagnostics + ablation

Saves `checkpoints/diag_summary.csv`, `diag_edges.pkl`, `state/ablation_posthoc.json`.


In [ ]:
# ============================================================================
#  PHASE 2b · Backtest diagnostics (per season + graphs) + spread ablation
# ============================================================================
%matplotlib inline
from IPython.display import display

results = ensure_results()
diag_summary = pd.DataFrame()
diag_edges = {}

if use_saved_artifacts("2b") and _artifact_ready("diag_summary", "diag_edges"):
    print("⚡ Loading Phase 2b artifacts from disk ...")
    diag_summary = pd.read_csv(ARTIFACTS["diag_summary"])
    diag_edges = load_pickle_artifact("diag_edges")
    display(diag_summary.round(3))
    for season, tbl in diag_edges.items():
        print(f"\nEdge buckets · {season}")
        display(tbl.round(3))
    if ARTIFACTS["ablation"].exists():
        ab_summary = pd.read_json(ARTIFACTS["ablation"])
        print("\n--- Post-hoc spread ablation (loaded) ---")
        display(ab_summary.round(4))

elif not results.empty:
    diag_summary, diag_edges = run_backtest_diagnostics(
        results,
        save_dir=ROOT / "analysis_plots",
        show_graphs=True,
    )
    display(diag_summary.round(3))
    for season, tbl in diag_edges.items():
        print(f"\nEdge buckets · {season}")
        display(tbl.round(3))

    diag_summary.to_csv(ARTIFACTS["diag_summary"], index=False)
    save_pickle_artifact("diag_edges", diag_edges)
    print(f"💾 Saved → {ARTIFACTS['diag_summary']}")

    print("\n--- Post-hoc spread ablation ---")
    ab_summary = posthoc_ablation_summary(results)
    display(ab_summary.round(4))
    if not ab_summary.empty:
        base = ab_summary[ab_summary["config"] == "baseline_current"]
        full = ab_summary[ab_summary["config"] == "edge_bucket_full"]
        if not base.empty and not full.empty:
            print("Ablation gate (baseline vs edge_bucket_full):",
                  passes_ablation_gate(base.iloc[0].to_dict(), full.iloc[0].to_dict()))
    print("\n--- Calibration mode ablation configs (full re-run required) ---")
    for name in calibration_ablation_configs():
        print(f"  · {name}")
    ARTIFACTS["ablation"].parent.mkdir(parents=True, exist_ok=True)
    ARTIFACTS["ablation"].write_text(ab_summary.to_json(orient="records", indent=2))
    print(f"💾 Saved → {ARTIFACTS['ablation']}")
else:
    print("Skip diagnostics — no backtest results.")


---

## Phase 2c — Bankroll & confidence plots

Saves PNGs under `analysis_plots/`. Skips if plots exist and `FORCE_RECOMPUTE["2c"]` is False.


In [ ]:
# ============================================================================
#  PHASE 2c · ROI / bankroll / confidence plots
# ============================================================================
%matplotlib inline

results = ensure_results()
plot_dir = ROOT / "analysis_plots"
key_plots = (
    "10_bankroll_by_profile.png",
    "11_roi_by_confidence_tier.png",
    "12_confidence_reliability.png",
    "12b_cover_prob_reliability_by_season.png",
)

if results.empty:
    print("Skip betting plots — no backtest results.")
elif use_saved_artifacts("2c") and all((plot_dir / n).exists() for n in key_plots):
    print(f"⚡ Loading existing plots from {plot_dir} ...")
else:
    generate_betting_plots(results, save_dir=plot_dir)
    print(f"💾 Plots saved → {plot_dir}/")

from IPython.display import Image, display
for name in key_plots + ("13_ml_winprob_reliability.png",):
    p = plot_dir / name
    if p.exists():
        display(Image(filename=str(p)))


---

## Phase 2d — Moneyline diagnostics (MetaWin calibration)

Saves `analysis_plots/ml_diagnostics_summary.csv` and reliability plot.


In [ ]:
# ============================================================================
#  PHASE 2d · Moneyline training diagnostics (WIN_PROB / MetaWin)
# ============================================================================
results = ensure_results()
MIN_ML_EV = 0.03
MAX_FAV_DEC = float(results["MAX_FAVORITE_DECIMAL"].iloc[-1]) if (
    not results.empty and "MAX_FAVORITE_DECIMAL" in results.columns
) else 1.45

ml_summary = pd.DataFrame()
if use_saved_artifacts("2d") and ARTIFACTS["ml_summary"].exists():
    print(f"⚡ Loading ML diagnostics from {ARTIFACTS['ml_summary']} ...")
    ml_summary = pd.read_csv(ARTIFACTS["ml_summary"])
    display(ml_summary.round(3))
elif not results.empty:
    ml_summary = run_ml_diagnostics(
        results,
        min_ev=MIN_ML_EV,
        max_fav_dec=MAX_FAV_DEC,
        save_dir=ROOT / "analysis_plots",
        show_plots=True,
    )
    display(ml_summary.round(3))
    print(f"Using walk-forward config: edge≥{WALKFORWARD_EDGE_MIN_FLOOR}, "
          f"ML min EV={MIN_ML_EV:.0%}, max favorite decimal={MAX_FAV_DEC:.2f}")
else:
    print("Skip ML diagnostics — no backtest results.")


---

## Phase 2e — ML underdog / favorite value tracker

Saves `ml_value_plays.csv` for tracking underdog value and undervalued favorites.


In [ ]:
# ============================================================================
#  PHASE 2e · ML value plays (underdog / undervalued favorite tracker)
# ============================================================================
results = ensure_results()
MIN_ML_EV = globals().get("MIN_ML_EV", 0.03)
MAX_FAV_DEC = globals().get(
    "MAX_FAV_DEC",
    float(results["MAX_FAVORITE_DECIMAL"].iloc[-1]) if (
        not results.empty and "MAX_FAVORITE_DECIMAL" in results.columns
    ) else 1.45,
)
ml_plays = pd.DataFrame()
if use_saved_artifacts("2e") and ARTIFACTS["ml_plays"].exists():
    print(f"⚡ Loading ML value plays from {ARTIFACTS['ml_plays']} ...")
    ml_plays = pd.read_csv(ARTIFACTS["ml_plays"], low_memory=False)
elif not results.empty:
    ml_plays = export_ml_tracking(
        results,
        ARTIFACTS["ml_plays"],
        min_ev=MIN_ML_EV,
        max_fav_dec=MAX_FAV_DEC,
    )
else:
    print("Skip ML tracker — no backtest results.")

if not ml_plays.empty:
    display(ml_plays.head(20))
    by_type = ml_plays.groupby("ml_value_type").agg(
        n=("ml_value_type", "count"),
        win_pct=("ml_won", "mean"),
        avg_ev=("ml_best_ev", "mean"),
        roi=("ml_flat_profit", "mean"),
    )
    print("\nML value plays by type:")
    display(by_type.round(3))
elif use_saved_artifacts("2e"):
    print("No ML value plays at current thresholds.")


---

## Phase 3 — Dedicated ML + Total + Score-pair models

Trains tuned `MetaWinModel`, `MetaTotalModel`, and `MetaScorePairModel`; saves `state/win.pkl`, `state/total.pkl`, `state/score_pair.pkl`, and `checkpoints/final_feats.parquet` for Phase 4.


In [ ]:
# ============================================================================
#  PHASE 3 · Dedicated Moneyline + Total + Score-pair training
# ============================================================================
ml_total_loaded = False
if (
    use_saved_artifacts("3")
    and (STATE_DIR / "win.pkl").exists()
    and (STATE_DIR / "total.pkl").exists()
):
    print(f"⚡ Loading ML/Total models from {STATE_DIR} ...")
    final_win = MetaWinModel.load(STATE_DIR / "win.pkl")
    final_total = MetaTotalModel.load(STATE_DIR / "total.pkl")
    final_score_pair = (
        MetaScorePairModel.load(STATE_DIR / "score_pair.pkl")
        if (STATE_DIR / "score_pair.pkl").exists()
        else None
    )
    if final_score_pair is not None:
        print("   loaded score_pair.pkl")
    if ARTIFACTS["final_feats"].exists():
        final_feats = pd.read_parquet(ARTIFACTS["final_feats"])
        print(f"   loaded feature matrix: {len(final_feats):,} rows")
    else:
        final_feats = None
    ml_total_loaded = True

if not ml_total_loaded:
    train_df = all_stints.copy()
    best_elo_raw = tune_elo_tracker(train_df, DEFAULT_LEAGUE_XPPP, n_trials=ELO_TRIALS)
    best_elo = map_elo_params(best_elo_raw)
    best_hier = tune_hierarchical(train_df, n_trials=HIER_TRIALS)

    feat_hier = HierarchicalPossessionEngine(**best_hier)
    feat_elo = PlayerRatingTracker(config=best_elo, league_xppp=DEFAULT_LEAGUE_XPPP)
    feat_pace = PaceTracker(team_window=10, league_window=100)
    feat_xppp = TeamXpppTracker(window_size=40, prev_season_weight=0.5)
    feat_form = TeamFormTracker(window=15, prev_season_weight=0.4)
    feat_rotation = RotationLineupTracker()
    feat_lineup_elo = LineupEloTracker()
    feat_chemistry = ChemistryTracker()
    feat_team_elo = TeamEloTracker()
    feat_travel = TravelTracker()

    final_feats = generate_features(
        train_df, feat_hier, feat_elo, feat_pace,
        odds_dict=modern_odds_dict, update_engines=True,
        team_xppp_tracker=feat_xppp, team_form_tracker=feat_form,
        rotation_tracker=feat_rotation, lineup_elo_tracker=feat_lineup_elo,
        chemistry_tracker=feat_chemistry, team_elo_tracker=feat_team_elo,
        travel_tracker=feat_travel,
    )
    final_feats = engineer_interaction_features(final_feats)
    final_feats.to_parquet(ARTIFACTS["final_feats"], index=False)
    print(f"💾 Saved feature matrix → {ARTIFACTS['final_feats']}")

    calib_cut = max(len(final_feats) // 5, 80)
    meta_train = final_feats.iloc[:-calib_cut]
    meta_calib = final_feats.iloc[-calib_cut:]

    margin_params = tune_margin_model(
        final_feats, n_trials=max(8, META_TRIALS // 2),
        fast_mode=FAST_BACKTEST or META_TRIALS <= 5,
    )
    margin_model = MetaScoreModel(
        ridge_alpha=margin_params["ridge_alpha"],
        cb_params=margin_params["cb_params"],
        huber_epsilon=margin_params["huber_epsilon"],
        use_isotonic_calibration=True,
        use_elo_stack=True,
        total_mode="external",
    )
    margin_model.fit(
        meta_train, meta_train["actual_home"], meta_train["actual_away"],
        calib_df=meta_calib,
    )
    meta_train = meta_train.copy()
    meta_calib = meta_calib.copy()
    meta_train["pred_margin"] = margin_model._predict_raw(meta_train)["pred_margin"]
    meta_calib["pred_margin"] = margin_model._predict_raw(meta_calib)["pred_margin"]

    best_win = tune_meta_win_model(
        meta_train, n_trials=META_WIN_TRIALS,
        pred_margin_col="pred_margin",
        fast_mode=FAST_BACKTEST or META_WIN_TRIALS <= 5,
    )
    save_json_artifact("meta_win_tuning", best_win)
    final_win = MetaWinModel(
        C=best_win.get("C", 0.1),
        feature_cols=best_win.get("feature_cols") or win_feature_cols(meta_train),
        elo_win_blend=best_win.get("elo_win_blend"),
        calib_method=best_win.get("calib_method", "isotonic"),
    )
    final_win.fit(
        meta_train,
        (meta_train["actual_home"] > meta_train["actual_away"]).astype(int),
        calib_df=meta_calib,
    )
    final_win.save(STATE_DIR / "win.pkl")
    print(f"💾 Saved → {STATE_DIR / 'win.pkl'}")

    best_total = tune_total_model(
        final_feats, n_trials=TOTAL_TRIALS,
        feature_cols=total_feature_cols(final_feats),
        fast_mode=FAST_BACKTEST or TOTAL_TRIALS <= 5,
    )
    save_json_artifact("meta_total_tuning", best_total)
    best_total = dict(best_total)
    cv_score = float(best_total.pop("_cv_score", best_total.pop("_cv_mae", np.nan)))
    if np.isfinite(cv_score) and cv_score > float(TOTAL_HEAD_CV_SANITY_CAP):
        print(f"  ⚠ Total-model CV score {cv_score:.1f} exceeds cap — using defaults")
        best_total.setdefault("ridge_alpha", 10.0)
        best_total.setdefault("huber_epsilon", 1.35)
        best_total.setdefault("total_elo_beta", TOTAL_ELO_BETA)
        best_total.setdefault("train_target", TOTAL_TRAIN_TARGET)
        best_total.setdefault("feature_cols", total_feature_cols(final_feats))
    final_total = MetaTotalModel(
        ridge_alpha=best_total.get("ridge_alpha", 10.0),
        cb_params=best_total.get("cb_params"),
        huber_epsilon=best_total.get("huber_epsilon", 1.35),
        feature_cols=best_total.get("feature_cols") or total_feature_cols(final_feats),
        total_elo_beta=best_total.get("total_elo_beta", TOTAL_ELO_BETA),
        train_target=best_total.get("train_target", TOTAL_TRAIN_TARGET),
    )
    final_total.fit(final_feats)
    final_total.save(STATE_DIR / "total.pkl")
    print(f"💾 Saved → {STATE_DIR / 'total.pkl'}")

    # Dual home/away score heads (specialized totals path)
    final_score_pair = MetaScorePairModel(
        feature_cols=total_feature_cols(final_feats),
        train_target="absolute",
    )
    final_score_pair.fit(final_feats)
    final_score_pair.save(STATE_DIR / "score_pair.pkl")
    print(f"💾 Saved → {STATE_DIR / 'score_pair.pkl'}")
    print("✅ Phase 3 ML + Total + Score-pair models trained")


---

## Phase 4 — Spread / final engines

Trains margin `MetaScoreModel` and persists engines. Reloads `final_feats` from Phase 3 when available.


In [ ]:
# ============================================================================
#  PHASE 4 · Train spread engines + MetaScoreModel (margin-only)
# ============================================================================
engines_loaded = False
if use_saved_artifacts("4") and (STATE_DIR / "meta.pkl").exists():
    print(f"⚡ Loading spread engines from {STATE_DIR} ...")
    final_hier = HierarchicalPossessionEngine.load_state(STATE_DIR / "hier.pkl")
    final_elo = PlayerRatingTracker.load_state(STATE_DIR / "elo.pkl")
    final_pace = PaceTracker.load_state(STATE_DIR / "pace.pkl") if (STATE_DIR / "pace.pkl").exists() else PaceTracker()
    final_xppp = TeamXpppTracker()
    final_form = TeamFormTracker.load_state(STATE_DIR / "form.pkl") if (STATE_DIR / "form.pkl").exists() else TeamFormTracker()
    final_rotation = RotationLineupTracker.load_state(STATE_DIR / "rotation.pkl") if (STATE_DIR / "rotation.pkl").exists() else RotationLineupTracker()
    final_lineup_elo = LineupEloTracker.load_state(STATE_DIR / "lineup_elo.pkl") if (STATE_DIR / "lineup_elo.pkl").exists() else LineupEloTracker()
    final_chemistry = ChemistryTracker.load_state(STATE_DIR / "chemistry.pkl") if (STATE_DIR / "chemistry.pkl").exists() else ChemistryTracker()
    final_team_elo = TeamEloTracker.load_state(STATE_DIR / "team_elo.pkl") if (STATE_DIR / "team_elo.pkl").exists() else TeamEloTracker()
    final_travel = TravelTracker.load_state(STATE_DIR / "travel.pkl") if (STATE_DIR / "travel.pkl").exists() else TravelTracker()
    final_meta = MetaScoreModel.load(STATE_DIR / "meta.pkl")
    if "final_win" not in dir() and (STATE_DIR / "win.pkl").exists():
        final_win = MetaWinModel.load(STATE_DIR / "win.pkl")
    if "final_total" not in dir() and (STATE_DIR / "total.pkl").exists():
        final_total = MetaTotalModel.load(STATE_DIR / "total.pkl")
    if "final_score_pair" not in dir() and (STATE_DIR / "score_pair.pkl").exists():
        final_score_pair = MetaScorePairModel.load(STATE_DIR / "score_pair.pkl")
    if (STATE_DIR / "last_dates.pkl").exists():
        with open(STATE_DIR / "last_dates.pkl", "rb") as f:
            last_game_dates = pickle.load(f)
    else:
        last_game_dates = {}
    engines_loaded = True
    print("✅ Spread engines loaded from disk")

if not engines_loaded:
    train_df = all_stints.copy()
    if "final_feats" not in dir() or final_feats is None:
        if ARTIFACTS["final_feats"].exists():
            final_feats = pd.read_parquet(ARTIFACTS["final_feats"])
            print(f"⚡ Loaded feature matrix from {ARTIFACTS['final_feats']}")
        else:
            best_elo_raw = tune_elo_tracker(train_df, DEFAULT_LEAGUE_XPPP, n_trials=ELO_TRIALS)
            best_elo = map_elo_params(best_elo_raw)
            best_hier = tune_hierarchical(train_df, n_trials=HIER_TRIALS)
            final_hier = HierarchicalPossessionEngine(**best_hier)
            final_elo = PlayerRatingTracker(config=best_elo, league_xppp=DEFAULT_LEAGUE_XPPP)
            final_pace = PaceTracker(team_window=10, league_window=100)
            final_xppp = TeamXpppTracker(window_size=40, prev_season_weight=0.5)
            final_form = TeamFormTracker(window=15, prev_season_weight=0.4)
            final_rotation = RotationLineupTracker()
            final_lineup_elo = LineupEloTracker()
            final_chemistry = ChemistryTracker()
            final_team_elo = TeamEloTracker()
            final_travel = TravelTracker()
            final_feats = generate_features(
                train_df, final_hier, final_elo, final_pace,
                odds_dict=modern_odds_dict, update_engines=True,
                team_xppp_tracker=final_xppp, team_form_tracker=final_form,
                rotation_tracker=final_rotation, lineup_elo_tracker=final_lineup_elo,
                chemistry_tracker=final_chemistry, team_elo_tracker=final_team_elo,
                travel_tracker=final_travel,
            )
            final_feats = engineer_interaction_features(final_feats)
    else:
        best_elo_raw = tune_elo_tracker(train_df, DEFAULT_LEAGUE_XPPP, n_trials=ELO_TRIALS)
        best_elo = map_elo_params(best_elo_raw)
        best_hier = tune_hierarchical(train_df, n_trials=HIER_TRIALS)
        final_hier = HierarchicalPossessionEngine(**best_hier)
        final_elo = PlayerRatingTracker(config=best_elo, league_xppp=DEFAULT_LEAGUE_XPPP)
        final_pace = PaceTracker(team_window=10, league_window=100)
        final_xppp = TeamXpppTracker(window_size=40, prev_season_weight=0.5)
        final_form = TeamFormTracker(window=15, prev_season_weight=0.4)
        final_rotation = RotationLineupTracker()
        final_lineup_elo = LineupEloTracker()
        final_chemistry = ChemistryTracker()
        final_team_elo = TeamEloTracker()
        final_travel = TravelTracker()

    calib_cut = max(len(final_feats) // 5, 80)
    meta_train = final_feats.iloc[:-calib_cut]
    meta_calib = final_feats.iloc[-calib_cut:]
    elo_knobs = tune_elo_calibrator(meta_train, calib_df=meta_calib)
    elo_calibrator = WalkForwardEloCalibrator(knobs=elo_knobs)
    elo_calibrator.fit(meta_train, calib_df=meta_calib)
    final_feats = apply_elo_calibration_df(final_feats, elo_calibrator)
    meta_train = final_feats.iloc[:-calib_cut]
    meta_calib = final_feats.iloc[-calib_cut:]

    best_meta = tune_margin_model(
        final_feats, n_trials=META_TRIALS, fast_mode=FAST_BACKTEST or META_TRIALS <= 5,
    )
    elo_knobs.elo_blend_alpha = best_meta.get("elo_blend_alpha", elo_knobs.elo_blend_alpha)
    elo_knobs.elo_ridge_alpha = best_meta.get("elo_ridge_alpha", elo_knobs.elo_ridge_alpha)
    final_meta = MetaScoreModel(
        ridge_alpha=best_meta["ridge_alpha"], cb_params=best_meta["cb_params"],
        huber_epsilon=best_meta["huber_epsilon"], use_isotonic_calibration=True,
        use_elo_stack=True, total_mode="external",
        elo_blend_alpha=elo_knobs.elo_blend_alpha,
        elo_ridge_alpha=elo_knobs.elo_ridge_alpha,
    )
    final_meta.fit(
        meta_train, meta_train["actual_home"], meta_train["actual_away"],
        calib_df=meta_calib,
    )

    final_elo.save_state(STATE_DIR / "elo.pkl")
    final_hier.save_state(STATE_DIR / "hier.pkl")
    final_pace.save_state(STATE_DIR / "pace.pkl")
    final_form.save_state(STATE_DIR / "form.pkl")
    final_meta.save(STATE_DIR / "meta.pkl")
    final_rotation.save_state(STATE_DIR / "rotation.pkl")
    final_lineup_elo.save_state(STATE_DIR / "lineup_elo.pkl")
    final_chemistry.save_state(STATE_DIR / "chemistry.pkl")
    final_team_elo.save_state(STATE_DIR / "team_elo.pkl")
    final_travel.save_state(STATE_DIR / "travel.pkl")
    if elo_calibrator.fitted:
        elo_calibrator.save(STATE_DIR / "elo_calibrator.pkl")
    save_elo_knobs(elo_knobs, STATE_DIR / "elo_calibration_knobs.json")
    last_game_dates = train_df.groupby("home_team")["game_date"].max().to_dict()
    for t, d in train_df.groupby("away_team")["game_date"].max().to_dict().items():
        last_game_dates[t] = max(last_game_dates.get(t, d), d)
    with open(STATE_DIR / "last_dates.pkl", "wb") as f:
        pickle.dump(last_game_dates, f)
    print(f"✅ Spread engines saved to {STATE_DIR}")

final_ats = None
final_vol = TeamVolatilityTracker()
try:
    ats_path = STATE_DIR / "latest_ats.pkl"
    if ats_path.exists():
        final_ats = ATSClassifier.load(ats_path)
        print("Loaded ATS classifier from", ats_path)
except Exception as e:
    print("ATS classifier load skipped:", e)


---

## Phase 5 — Daily prediction

Loads spread (`meta`), moneyline (`win`), total, and score-pair models for live picks.

Outputs for each game: `pred_margin`, `p_home`/`p_away`, `ml_bet` (Home/Away/Pass), `pred_home_pts`/`pred_away_pts`, `pred_total`, `sigma_total`, `p_over`.


In [ ]:
# ============================================================================
#  PHASE 5 · Daily prediction for an upcoming game
# ============================================================================
if "final_score_pair" not in dir() and (STATE_DIR / "score_pair.pkl").exists():
    final_score_pair = MetaScorePairModel.load(STATE_DIR / "score_pair.pkl")
if "final_win" not in dir() and (STATE_DIR / "win.pkl").exists():
    final_win = MetaWinModel.load(STATE_DIR / "win.pkl")
if "final_total" not in dir() and (STATE_DIR / "total.pkl").exists():
    final_total = MetaTotalModel.load(STATE_DIR / "total.pkl")

elo_cal = WalkForwardEloCalibrator.load(STATE_DIR / "elo_calibrator.pkl") \
    if (STATE_DIR / "elo_calibrator.pkl").exists() else None
bet_cal = WalkForwardBetCalibrator.load(STATE_DIR / "bet_calibrator.pkl") \
    if (STATE_DIR / "bet_calibrator.pkl").exists() else None
tuning_cfg = load_tuning_config()
print(f"Using profile: {tuning_cfg.get('recommended_profile', 'moderate')}")
print(f"Confidence: mode={CONFIDENCE_MODE}  min_score={load_min_confidence_threshold()}  "
      f"selection={CONFIDENCE_SELECTION_MODE}")
print(f"OU Gaussian={OU_USE_GAUSSIAN_PROB}  OU_MIN_PROB={OU_MIN_PROB}  "
      f"USE_SCORE_PAIR_TOTAL={USE_SCORE_PAIR_TOTAL}  ML_BET_PREDICTED_WINNER_ONLY={ML_BET_PREDICTED_WINNER_ONLY}")
if tuning_cfg.get("elo_calibration"):
    ec = tuning_cfg["elo_calibration"]
    print(f"ELO knobs: blend_alpha={ec.get('elo_blend_alpha')} ridge_alpha={ec.get('elo_ridge_alpha')} huber={ec.get('huber_epsilon')}")

pred = predict_game(
    home_abbr="BOS",
    away_abbr="LAL",
    game_date="2026-10-28",
    hier_engine=final_hier,
    elo_tracker=final_elo,
    meta_model=final_meta,
    pace_tracker=final_pace,
    team_xppp_tracker=final_xppp,
    team_form_tracker=final_form,
    rotation_tracker=final_rotation,
    lineup_elo_tracker=final_lineup_elo,
    chemistry_tracker=final_chemistry,
    team_elo_tracker=final_team_elo,
    travel_tracker=final_travel,
    win_model=final_win if "final_win" in dir() else None,
    total_model=final_total if "final_total" in dir() else None,
    score_pair_model=final_score_pair if "final_score_pair" in dir() else None,
    ats_classifier=final_ats if "final_ats" in dir() else None,
    vol_tracker=final_vol if "final_vol" in dir() else None,
    elo_calibrator=elo_cal,
    confidence_calibrator=bet_cal,
    tuning_config=tuning_cfg,
    last_game_dates=last_game_dates,
    home_starters=None,
    away_starters=None,
    live_market_spread=None,
    live_market_ml=None,
    auto_load_models=False,
)

# Highlight the four specialized outputs + ML decision
keys_first = [
    "pred_spread", "pred_margin", "cover_prob_calibrated",
    "p_home", "p_away", "predicted_winner", "ev_home", "ev_away", "ml_bet", "ml_reason",
    "pred_home_pts", "pred_away_pts",
    "pred_total", "sigma_total", "p_over", "p_under", "ou_direction",
    "direction", "actionable", "confidence_score",
]
print("\n── Specialized market outputs ──")
for k in keys_first:
    if k in pred:
        print(f"{k:>22}: {pred[k]}")
print("\n── Full prediction dict ──")
for k, v in pred.items():
    if k == "features":
        print(f"{k:>22}: <{len(v) if isinstance(v, dict) else '?'} features>")
        continue
    print(f"{k:>22}: {v}")

payload = {k: v for k, v in pred.items() if k != "features"}
save_json_artifact("prediction", {
    "home": "BOS",
    "away": "LAL",
    "game_date": "2026-10-28",
    "prediction": {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in payload.items()},
})
